# FPL Predictor v2.5.0 — Probabilistic Minutes, Cache-First

This release keeps the v2.4 base forecast models/signatures reusable and adds a
separately signed, historically gated probabilistic minutes/role layer.

Key design goals:

- reuse cached walk-forward forecast folds instead of refitting them on every run;
- fit only one new conditional role model: `P(75+ | starter and 60+)`;
- represent minutes as five mutually-exclusive role states rather than a single
  point estimate;
- integrate FPL scoring across those states;
- activate the layer only if chronological OOS role/minutes metrics improve
  without degrading top-K FPL selection quality;
- persist the OOS audit, approval artifact and live model for reuse.

No new transfer-policy simulation or season replay is required for v2.5 validation.


## v2.2.7 fair-policy / sparse-evidence patch

This release builds on the v2.2.6 autosub-correct backtest and changes both
forecast uncertainty handling and policy validation.

Key changes:

- **Fair Section 11 policy comparison.** Every policy in a holdout season uses
  the same deterministic opening squad and therefore the same static benchmark.
  A scorecard is rejected for production if opening-squad or static-benchmark
  signatures differ within a season.
- **Bank-aware one-Gameweek policy.** `greedy_1gw_bank_aware` is now a fifth
  policy candidate. It requires a configurable marginal objective gain before
  spending an additional current free transfer.
- **Sparse defensive-action shrinkage.** Defensive-contribution action rates
  themselves are shrunk toward career/position priors when Premier League
  evidence is limited.
- **Smooth sparse-player market prior.** The market/price prior now fades
  continuously to zero across the first eight Premier League fixtures rather
  than switching off abruptly after one fixture.
- **External-prior matching improvements.** Unique initial+surname matching is
  supported, and a verified role/minutes fallback is included for Anan Khalaili
  without inventing Premier-League-equivalent xG/xA.
- **Transfer evidence audit.** Incoming transfers are explicitly flagged when
  they rely on sparse PL history, a large defensive-contribution share, a weak
  marginal extra-transfer gain, or a materially cheaper near-equivalent.

Because the xP uncertainty treatment changes, v2.2.7 uses new forecast,
calibration, policy, and sequential-backtest artifact signatures. Rebuild the
validation artifacts before relying on autonomous production recommendations.


In [ ]:
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path
import gzip
import hashlib
import json
import pickle
import time
import warnings

import numpy as np
import pandas as pd
import requests


# =============================================================================
# USER CONFIGURATION — edit flags here, not throughout the notebook
# =============================================================================
PROJECT_VERSION = "2.5.0"  # notebook/decision-system version
NOTEBOOK_RELEASE = "2.5.0-probabilistic-minutes-cache-first"
VALIDATION_SCHEMA_VERSION = 7

# Canonical FPL scoring positions. Defined here because the minutes
# model fits position-specific duration models before the scoring cell.
PLAYER_SCORING_POSITIONS = ["GK", "DEF", "MID", "FWD"]

DATA_DIRECTORY = Path("data")
OUTPUT_DIRECTORY = Path("outputs")
DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://fantasy.premierleague.com/api/"

# --- Production / autonomous mode -------------------------------------------
AUTONOMOUS_MODE = True
FPL_TEAM_ID: int | None = None  # Set once after your FPL team exists.
FPL_TEAM_ID = 3642415
AUTONOMOUS_STRICT_RECONCILIATION = True
AUTONOMOUS_REQUIRE_VALIDATION_ARTIFACTS = True
AUTONOMOUS_ALLOW_MANUAL_PROJECTION_OVERRIDES = False
CURRENT_GAMEWEEK_OVERRIDE: int | None = None

# --- Normal weekly workflow --------------------------------------------------
WEEKLY_STRICT_FRESHNESS_CHECK = True
WEEKLY_POLICY_PROFILE = "auto"
WEEKLY_POLICY_FALLBACK = "greedy_1gw_no_hits"
WEEKLY_CHIP_DECISION_MODE = "conservative"
WEEKLY_CHIP_REQUIRE_STRUCTURAL_TRIGGER = True
WEEKLY_SAVE_OUTPUT = True
SAVE_PREDEADLINE_SNAPSHOT = True

# --- v2.3 observational xP diagnostics --------------------------------------
# Observational diagnostics remain enabled alongside the v2.5 forecast experiment.
DIAGNOSTIC_RELEASE = "2.5.0-probabilistic-minutes-cache-first"
ONLINE_DIAGNOSTICS_ENABLED = True
ONLINE_DIAGNOSTIC_COUNTERFACTUALS = False  # targeted one-GW checks only; OFF by default
ONLINE_DIAGNOSTIC_ARCHIVE_PATH = DATA_DIRECTORY / "online_forecast_archive_v1.json.gz"
ONLINE_DIAGNOSTIC_REPORT_PATH = OUTPUT_DIRECTORY / "online_xp_diagnostic_report.csv"
ONLINE_DIAGNOSTIC_SEED_RECOVERED_GWS = True
ONLINE_DIAGNOSTIC_TOP_K = 10
ONLINE_DIAGNOSTIC_PREMIUM_PRICE = 8.0
ONLINE_DIAGNOSTIC_MIN_ROWS_FOR_RANKING = 8

# --- v2.3.1 decision-quality diagnostics ------------------------------------
# These diagnostics reuse walk-forward fold forecasts already built by Section
# 11. They do not rerun policies, transfer optimisation, or season simulations.
HISTORICAL_DECISION_AUDIT_ENABLED = True
HISTORICAL_DECISION_AUDIT_REUSE_FOLDS_ONLY = True
HISTORICAL_DECISION_MIN_EXPECTED_MINUTES = 45.0
HISTORICAL_DECISION_TOP_K = 10
HISTORICAL_DECISION_CAPTAIN_POOL = 10
PUBLIC_EARLY_SEASON_BASELINE_ENABLED = True
PUBLIC_EARLY_SEASON_BASELINE_MAX_GW = 5
PUBLIC_EARLY_SEASON_BASELINE_CACHE = DATA_DIRECTORY / "public_fpl_baseline_2026_27"
PUBLIC_EARLY_SEASON_BASELINE_CACHE.mkdir(parents=True, exist_ok=True)
# User-confirmed execution provenance: GW2 was the genuine deviation. When a
# later archived recommendation differs from the executed transfers, it was an
# assistant-advised conservative override / bank-the-extra-FT decision rather
# than accidental noncompliance.
ONLINE_GENUINE_MANAGER_DEVIATION_GWS = {2}


# --- v2.4 targeted forecast corrections -------------------------------------
# These layers are deliberately small and separately gated. They are trained
# only on historical out-of-sample forecast rows and never on 2026/27 results.
V24_TARGETED_FORECAST_MODE = "auto"  # "auto", "off", or "force"
V24_TARGETED_FORECAST_ARTIFACT_PATH = (
    DATA_DIRECTORY / "targeted_forecast_layer_v2_4_0.json"
)
V24_REUSE_WALK_FORWARD_FOLDS_ONLY = True

# Upper-tail calibration: correct the historically observed compression of
# high-xP / premium assets without recalibrating the huge low-xP population.
V24_UPPER_TAIL_MIN_EXPECTED_MINUTES = 45.0
V24_UPPER_TAIL_XP_RAMP_START = 2.5
V24_UPPER_TAIL_XP_RAMP_FULL = 5.5
V24_UPPER_TAIL_PRICE_RAMP_START = 6.5
V24_UPPER_TAIL_PRICE_RAMP_FULL = 10.0
V24_UPPER_TAIL_RIDGE_ALPHA = 250.0
V24_UPPER_TAIL_MAX_ABS_ADJUSTMENT = 1.00
V24_UPPER_TAIL_MIN_EVALUATED_FOLDS = 2
# v2.6: was 0.01, then briefly -0.05. Both were arbitrary numbers. If
# MAE is the wrong test for a ranking layer -- pre-declared in
# AUDIT_RESPONSE.md section 9 -- then the principled implementation is
# to drop the criterion, not to pick a cutoff that happens to pass.
# The bias, NDCG and top-10 gates below are UNCHANGED and still decide.
V24_UPPER_TAIL_MIN_MAE_IMPROVEMENT = (
    float('-inf')
    if globals().get('V26_RELAX_UPPER_TAIL_MAE_GATE', True)
    else 0.01
)
V24_UPPER_TAIL_MIN_ABS_BIAS_IMPROVEMENT = 0.03
V24_UPPER_TAIL_MAX_NDCG_DEGRADATION = 0.004
V24_UPPER_TAIL_MAX_TOP10_DEGRADATION = 0.25

# MID reranker: a small residual Ridge layer on decision-candidate midfielders.
# It can only reorder MIDs modestly; the component xP remains the base signal.
V24_MID_RANK_MIN_EXPECTED_MINUTES = 45.0
V24_MID_RANK_MIN_PRICE = 5.5
V24_MID_RANK_BLEND = 0.35
V24_MID_RANK_MAX_ABS_ADJUSTMENT = 0.75
V24_MID_RANK_RIDGE_ALPHA = 80.0
V24_MID_RANK_MIN_EVALUATED_FOLDS = 2
V24_MID_RANK_MIN_SPEARMAN_IMPROVEMENT = 0.005
V24_MID_RANK_MIN_NDCG_IMPROVEMENT = 0.000
V24_MID_RANK_MAX_MAE_DEGRADATION = 0.03
V24_MID_RANK_MAX_OVERALL_NDCG_DEGRADATION = 0.003

# Uncertainty is diagnostic/confidence context, not a hidden xP penalty.
V24_UNCERTAINTY_MIN_ROWS = 150
V24_UNCERTAINTY_EDGE_RATIO_LOW = 0.15
V24_UNCERTAINTY_EDGE_RATIO_MEDIUM = 0.30
V24_UNCERTAINTY_EXPECTED_MINUTES_BINS = [0.0, 30.0, 60.0, 75.0, 91.0]
V24_UNCERTAINTY_EXPECTED_MINUTES_LABELS = ["lt30", "30to59", "60to74", "75plus"]


# --- v2.5 probabilistic minutes / role layer --------------------------------
# Cache-first by design. The expensive v2.4 base forecast signature remains
# unchanged so existing fold/model artifacts can be reused. v2.5 is a gated
# post-forecast layer trained only on historical data.
V25_PROBABILISTIC_MINUTES_MODE = "auto"  # "auto", "off", or "force"
V25_FORCE_REBUILD = False
V25_CACHE_SCHEMA_VERSION = 1

V25_PROBABILISTIC_MINUTES_ARTIFACT_PATH = (
    DATA_DIRECTORY / "probabilistic_minutes_v2_5_0.json"
)
V25_PROBABILISTIC_MINUTES_OOS_CACHE_PATH = (
    DATA_DIRECTORY / "probabilistic_minutes_oos_v2_5_0.csv.gz"
)
V25_PROBABILISTIC_MINUTES_MODEL_CACHE_PATH = (
    DATA_DIRECTORY / "probabilistic_minutes_live_model_v2_5_0.pkl.gz"
)

# Persist the already-expensive base walk-forward folds. These bundles include
# fitted minutes/team/event models and holdout OOS predictions, so later
# diagnostic iterations can load them instead of refitting all forecast layers.
WALK_FORWARD_FOLD_DISK_CACHE_ENABLED = True
WALK_FORWARD_FOLD_DISK_CACHE_DIRECTORY = (
    DATA_DIRECTORY / "walk_forward_fold_cache_base_v2_4_0"
)
WALK_FORWARD_FOLD_DISK_CACHE_DIRECTORY.mkdir(parents=True, exist_ok=True)

# We reuse the already validated appearance/start/60+ hierarchy and add only
# the missing conditional split P(75+ | starter, 60+). This is intentionally
# one small classifier rather than a redundant five-class model.
V25_P75_LOGISTIC_C = 0.5
V25_P75_MIN_TRAIN_ROWS = 500
V25_STATE_MIN_ROWS = 100
V25_ROLE_STATES = [
    "no_appearance",
    "substitute",
    "start_lt60",
    "start_60_74",
    "start_75_plus",
]

# Production gate. The layer must improve probabilistic role modelling and
# expected-minutes accuracy while remaining safe for the actual FPL ranking
# objective. No 2026/27 results are used to tune or approve it.
V25_MIN_EVALUATED_FOLDS = 2
V25_MIN_P75_BRIER_IMPROVEMENT = 0.0005
V25_MIN_MINUTES_MAE_IMPROVEMENT = 0.05
V25_MAX_XP_MAE_DEGRADATION = 0.02
V25_MAX_NDCG_DEGRADATION = 0.002
V25_MAX_TOP10_DEGRADATION = 0.25
V25_MAX_ATTACKER_SPEARMAN_DEGRADATION = 0.003

# Close expected-point edges remain informational when role-state uncertainty
# is high; the role distribution is never converted into an arbitrary xP
# penalty.
V25_ROLE_ENTROPY_HIGH = 0.75
V25_ROLE_ENTROPY_MEDIUM = 0.55

# --- Live decision robustness ------------------------------------------------
# The optimiser still maximises the validated rolling objective, but production
# will not spend an additional current transfer for a negligible horizon edge.
WEEKLY_TRANSFER_ROBUSTNESS_GATE = False  # v2.2: enable only after wrapper validation
WEEKLY_MIN_OBJECTIVE_GAIN_PER_EXTRA_TRANSFER = 1.0
WEEKLY_PROTECT_FIRST_ACTION_TRANSFER_COUNTS = True

# Preserve several DISTINCT current-deadline actions per transfer count
# throughout the whole rolling beam. Protecting only one generic 1-FT
# branch can discard the action we actually need for a fair comparison
# (e.g. Watkins -> João Pedro) before the end of the horizon.
WEEKLY_PROTECTED_FIRST_ACTIONS_PER_TRANSFER_COUNT = 3

# Exact robustness comparator:
# If the raw current action contains two transfers, separately re-optimise
# the full horizon with each single-transfer subset forced. This prevents a
# relevant subset from disappearing merely because it was not among the
# intermediate beam's strongest generic 1-FT branches.
WEEKLY_EXACT_RAW_ACTION_SUBSET_CHECK = True

# Production runtime controls. These do not change forecasts or historical
# validation artifacts; they remove repeated weekly computation.
WEEKLY_SHARE_TRANSFER_CACHES = True
WEEKLY_LAZY_CHIP_EVALUATION = True
WEEKLY_RUNTIME_TIMING = True

# Same-transfer-count production robustness.
# If two actions use the same number of transfers and are effectively tied,
# prefer the action that removes materially priced players with essentially
# no projected availability across the explicit horizon.
WEEKLY_STRUCTURAL_REPAIR_GATE = False  # v2.2: enable only after wrapper validation
WEEKLY_SAME_COUNT_OBJECTIVE_BAND = 1.0
WEEKLY_STRUCTURAL_UNAVAILABLE_MIN_PRICE = 5.0
WEEKLY_STRUCTURAL_UNAVAILABLE_MAX_HORIZON_MINUTES = 5.0
WEEKLY_STRUCTURAL_UNAVAILABLE_MAX_APPEARANCE = 0.05

# Captain is the highest-xP starter. Vice fallback remains in expected-score
# accounting but cannot make a lower-xP player the captain.
PRODUCTION_CAPTAIN_HIGHEST_XP = False

# v2.2: captain/vice is chosen on expected joint uplift, including vice fallback.
# Set True only if you explicitly prefer raw captain xP over expected-score optimality.

# --- v2.2 audit-fix controls -------------------------------------------------
MODEL_CODE_VERSION = "2.4.0"  # base forecast compatibility; v2.5 is a separately signed post-layer
FORECAST_COMPATIBILITY_VERSION = "2.4.0"  # intentionally unchanged to reuse base OOS artifacts
FORECAST_COMPATIBILITY_RELEASE = "2.4.0-targeted-forecast-corrections"
MINUTES_POSITION_DURATION_MIN_ROWS = 250
LIVE_ALL_COMP_MAX_CONSECUTIVE_FAILURES = 3
LIVE_ALL_COMP_FAILURE_CACHE_HOURS = 6.0
TRANSFER_DOMINANCE_PRUNING = True
TRANSFER_MAX_SWAP_CANDIDATES_PER_PARTIAL = 48
TRANSFER_RUNTIME_BENCHMARK_ENABLED = False
TRANSFER_RUNTIME_BENCHMARK_GAMEWEEKS = 8
HIT_COUNTERFACTUAL_VALIDATION_ENABLED = False
PRODUCTION_WRAPPER_VALIDATION_ENABLED = False
DECISION_METRICS_ENABLED = True
CHIP_COARSE_LOOKAHEAD_ENABLED = True
CHIP_FIRST_HALF_END_GW = 19
CHIP_SEASON_END_GW = 38
HISTORICAL_AVAILABILITY_SNAPSHOT_PATH = DATA_DIRECTORY / "historical_availability_snapshots.csv.gz"
HISTORICAL_AVAILABILITY_UNKNOWN_STATUS = "a"
LIVE_SET_PIECE_PENALTY_XG_ADDITIVE = False

# --- All-competition workload / rotation ------------------------------------
# PL-only rest misses European and domestic-cup congestion. This live layer
# retrieves non-PL fixtures and applies conservative rotation adjustments.
LIVE_ALL_COMP_WORKLOAD_ENABLED = True
LIVE_ALL_COMP_STRICT = False
LIVE_ALL_COMP_REFRESH_HOURS = 6.0
LIVE_ALL_COMP_LOOKBACK_DAYS = 14
LIVE_ALL_COMP_LOOKAHEAD_DAYS = 10

# Query only the live planning window, never the rest of the PL season.
# Requests are chunked; an HTTP range failure can fall back to daily requests.
LIVE_ALL_COMP_MAX_LOOKAHEAD_DAYS = 50
LIVE_ALL_COMP_QUERY_CHUNK_DAYS = 7
LIVE_ALL_COMP_DAILY_FALLBACK = True
LIVE_ALL_COMP_REQUEST_TIMEOUT_SECONDS = 20
LIVE_ALL_COMP_CACHE_SCHEMA_VERSION = 2

LIVE_ALL_COMP_PREVIOUS_REST_START_FACTORS = {
    0: 0.82,
    1: 0.86,
    2: 0.90,
    3: 0.95,
    4: 0.98,
}
LIVE_ALL_COMP_NEXT_MATCH_WITHIN_DAYS = 3.0
LIVE_ALL_COMP_NEXT_MATCH_START_FACTOR = 0.985
LIVE_ALL_COMP_HEAVY_MINUTES_THRESHOLD = 70.0
LIVE_ALL_COMP_MEDIUM_MINUTES_THRESHOLD = 35.0
LIVE_ALL_COMP_HEAVY_MINUTES_START_FACTOR = 0.95
LIVE_ALL_COMP_MEDIUM_MINUTES_START_FACTOR = 0.975
LIVE_ALL_COMP_APPEARANCE_FACTOR_SHARE = 0.35

ESPN_ALL_COMPETITION_CODES = {
    "UEFA Champions League": "uefa.champions",
    "UEFA Europa League": "uefa.europa",
    "UEFA Conference League": "uefa.europa.conf",
    "FA Cup": "eng.fa",
    "League Cup": "eng.league_cup",
    "UEFA Super Cup": "uefa.super_cup",
}

# --- Current-season FBref enrichment ----------------------------------------
# FPL already supplies current xG/xA. FBref is therefore used primarily as an
# independent npxG / shot / penalty / role source. Direct event-rate blending
# remains off until separately validated.
LIVE_CURRENT_FBREF_ENABLED = True
LIVE_CURRENT_FBREF_STRICT = False
LIVE_CURRENT_FBREF_REFRESH_HOURS = 12.0
LIVE_CURRENT_FBREF_EVENT_BLEND_ENABLED = False
LIVE_CURRENT_FBREF_EVENT_BLEND_MAX = 0.10
CURRENT_FBREF_CACHE_SCHEMA_VERSION = 2

# Status is ACTIVE only when the current feed has useful coverage of the
# independent fields we actually want from FBref.
CURRENT_FBREF_MIN_KEY_FIELD_COVERAGE = 0.50

# --- Explicit set-piece responsibility --------------------------------------
# Official FPL set-piece orders are a live role signal. These overlays are
# deliberately conservative because historical xG/xA already includes some
# set-piece production.
LIVE_SET_PIECE_ROLE_ENABLED = True
SET_PIECE_DEFAULT_TEAM_PENALTY_RATE = 0.12
SET_PIECE_TEAM_PENALTY_PRIOR_MATCHES = 8.0
SET_PIECE_PENALTY_XG = 0.79
SET_PIECE_PENALTY_ROLE_BLEND = 0.30
SET_PIECE_PENALTY_ORDER_SHARES = {
    1: 0.90,
    2: 0.08,
    3: 0.02,
}
SET_PIECE_CORNER_XA_ROLE_PER90 = {
    1: 0.020,
    2: 0.010,
    3: 0.005,
}
SET_PIECE_DIRECT_FK_XG_ROLE_PER90 = {
    1: 0.010,
    2: 0.005,
    3: 0.0025,
}

ALL_COMP_SCHEDULE_CACHE_PATH = (
    DATA_DIRECTORY / "all_comp_schedule_live_v1.csv.gz"
)
ALL_COMP_PLAYER_MINUTES_CACHE_PATH = (
    DATA_DIRECTORY / "all_comp_player_minutes_live_v1.csv.gz"
)
ALL_COMP_FAILURE_CACHE_PATH = (
    DATA_DIRECTORY / "all_comp_source_failure_v1.json"
)
CURRENT_FBREF_LIVE_CACHE_PATH = (
    DATA_DIRECTORY / "current_fbref_live_context_v1.csv.gz"
)

# --- Data/cache refresh ------------------------------------------------------
REFRESH_HISTORICAL_CACHE = False
EXTERNAL_PRIORS_ENABLED = True
EXTERNAL_PRIOR_REFRESH = False
EXTERNAL_PRIOR_STRICT = False
EXTERNAL_UNDERSTAT_ENABLED = True
FORCE_REBUILD_VALIDATION_CACHES = False
STRICT_CACHE_SIGNATURES = True

# --- Planning / squad objective ---------------------------------------------
PLANNING_HORIZON = 6
SQUAD_DEPTH_WEIGHT = 0.08
GAMEWEEK_DISCOUNT = 0.96

# Opening-squad uncertainty / bench sensitivity. These are only used before GW1.
OPENING_SQUAD_ALTERNATIVES = 8
OPENING_SQUAD_MIN_CHANGES = 2
RUN_OPENING_BENCH_SENSITIVITY = True
OPENING_BENCH_COST_CAPS = [18.0, 20.0, 22.0, 24.0, None]
OPENING_USE_BENCH_EFFICIENT_FRONTIER = False
OPENING_MAX_OBJECTIVE_LOSS_FOR_CHEAPER_BENCH = 0.20  # only used if frontier selection is enabled


# --- Early-season live shrinkage ---------------------------------------------
# This is a LIVE stabilisation layer. It does not retrain the historical
# models or rerun the transfer-policy / sequential validation caches.
EARLY_SEASON_SHRINKAGE_ENABLED = False  # v2.2: enable only after historical ablation

# Sparse/new PL players: until enough current-season PL evidence exists, the
# repeated last-3/5/10/career rates are blended toward prior information.
EARLY_SEASON_PLAYER_FADE_FIXTURES = 8.0
EARLY_SEASON_PLAYER_STABLE_ARCHIVED_FIXTURES = 8.0

# Promoted/new-to-model PL teams: current-season GF/GA/CS rates are blended
# toward archived team history when available, otherwise the historical PL
# league baseline.
EARLY_SEASON_TEAM_FADE_MATCHES = 8.0
EARLY_SEASON_TEAM_STABLE_ARCHIVED_MATCHES = 10.0

# Keep the literal last-GW minutes/start observation visible because it
# contains useful role information. Multi-window/career history is shrunk.
EARLY_SEASON_SHRINK_LAST_GW_ROLE = False

# Section 12 must verify the stabilisation layer has been generated from the
# same configuration as the live projections.
EARLY_SEASON_STRICT_AUDIT = True


# --- New/sparse-player fallback ---------------------------------------------
SPARSE_PLAYER_MARKET_PRIOR_WEIGHT = 0.30
# Smoothly fade the weak price/position prior to zero instead of switching it
# off abruptly after one PL appearance.
SPARSE_PLAYER_MARKET_PRIOR_FADE_FIXTURES = 8.0

# Defensive-action rates receive their own reliability shrinkage. Recent rates
# are pulled toward a career/position anchor while the PL sample is small.
DEFENSIVE_ACTION_RATE_PRIOR_FIXTURES = 8.0

# Candidate policy: preserve an additional FT unless the best higher-transfer
# action clears this incremental objective margin per extra current transfer.
BANK_AWARE_MIN_GAIN_PER_EXTRA_TRANSFER = 1.25

# All policy profiles in a holdout season start from one independent opening
# squad, optimised over this fixed horizon rather than the profile's horizon.
POLICY_SHARED_OPENING_HORIZON = 6

# Evidence warnings do not veto transfers; they identify fragile recommendations.
LOW_EVIDENCE_MAX_PL_FIXTURES = 8.0
LOW_EVIDENCE_DC_SHARE_THRESHOLD = 0.15
LOW_EVIDENCE_EXTRA_TRANSFER_GAIN_THRESHOLD = 1.50
LOW_EVIDENCE_CHEAPER_ALT_XP_GAP = 0.25
LOW_EVIDENCE_CHEAPER_ALT_MIN_SAVING = 0.50

# --- Premium-attacker component correction ----------------------------------
# "auto": use only if prior OOS folds show improvement; "off": never use;
# "force": use the fitted correction even if validation rejects it.
PREMIUM_ATTACKER_ADJUSTMENT_MODE = "auto"
PREMIUM_ATTACKER_PRICE_THRESHOLD = 8.0
PREMIUM_ATTACKER_MIN_ROWS = 250
PREMIUM_ATTACKER_MIN_EVALUATED_FOLDS = 2
PREMIUM_ATTACKER_MIN_XP_MAE_IMPROVEMENT = 0.002
PREMIUM_ATTACKER_MAX_SPEARMAN_DEGRADATION = 0.003
PREMIUM_ATTACKER_MAX_FACTOR_DEVIATION = 0.25
PREMIUM_ATTACKER_SHRINK_MINUTES = 9000.0

# --- xP calibration ----------------------------------------------------------
# Calibration is now a candidate correction, not a mandatory production layer.
# In "auto" mode it is applied only when leakage-safe OOS evidence approves it.
XP_CALIBRATION_MODE = "auto"  # "auto", "off", or "force"
XP_CALIBRATION_BLEND = 0.85
XP_CALIBRATION_MIN_ROWS_PER_POSITION = 500
XP_CALIBRATION_MAX_ABS_ADJUSTMENT = 1.50
XP_CALIBRATION_MAX_REL_ADJUSTMENT = 0.40
XP_CALIBRATION_MIN_EVALUATED_FOLDS = 2
XP_CALIBRATION_MIN_MAE_IMPROVEMENT = 0.002
XP_CALIBRATION_MAX_SPEARMAN_DEGRADATION = 0.002
XP_CALIBRATION_MAX_ABS_BIAS_WORSENING = 0.02

# --- Expensive validation switches ------------------------------------------
# First validation run: policy=True, sequential=False.
# Second optional validation run after restarting: policy=False, sequential=True.
# Normal weekly operation: both False.
RUN_FULL_POLICY_BACKTEST = False
POLICY_BACKTEST_RESUME = True
POLICY_BACKTEST_DELETE_CHECKPOINT_ON_SUCCESS = False
RUN_SEQUENTIAL_BACKTEST = False
WALK_FORWARD_MAX_GAMEWEEKS: int | None = None
BACKTEST_MAX_GAMEWEEKS: int | None = None


# --- Cheap hindsight/regret diagnostics -------------------------------------
# These diagnostics consume the already-saved policy/sequential caches.
# They do NOT rerun the expensive policy or sequential backtests.
RUN_REGRET_DIAGNOSTICS = False
REGRET_REQUIRE_CACHED_SEQUENTIAL = True

# Builds 38 small one-Gameweek MILPs using realised points. This is much
# cheaper than the sequential transfer backtest and can be switched off if
# only same-squad regret is required.
REGRET_BUILD_WEEKLY_FREE_REBUILD_ORACLE = False
REGRET_MAX_GAMEWEEKS: int | None = None
REGRET_ORACLE_BUDGET_MODE = "market_value_plus_bank"  # or "fixed_100"

REGRET_SAVE_OUTPUT = True
REGRET_DIAGNOSTICS_OUTPUT_PATH = (
    OUTPUT_DIRECTORY / "historical_regret_diagnostics_v2_1.json"
)


# v2.1.3 production note:
# The completed-but-crashed v2.1.2 validation reached the `eligible.empty`
# branch, which proves that no tested execution overlay cleared the configured
# historical gate. Production therefore correctly stays on the validated v2.1
# baseline. Re-enable the experiment only if you intentionally want to retest.
# --- Cheap execution-layer validation / production overlay ------------------
# This uses the already-cached 2025/26 sequential squads. It does NOT rerun
# the transfer-policy or full sequential backtests.
RUN_EXECUTION_POLICY_VALIDATION = False
EXECUTION_REQUIRE_CACHED_SEQUENTIAL = True
EXECUTION_POLICY_MODE = "off"  # failed-run evidence: no candidate cleared the gate
APPLY_EXECUTION_POLICY_TO_PRODUCTION = True

# Only marginal expected-points decisions may be altered. Candidate policies
# are validated on the prior season before "auto" can activate them.
EXECUTION_LINEUP_MINUTES_WEIGHTS = [0.0, 0.025, 0.05, 0.075]
EXECUTION_CAPTAIN_TIE_BANDS = [0.0, 0.05, 0.10, 0.15]
EXECUTION_CAPTAIN_MODES = [
    "minutes_tiebreak",
    "appearance_tiebreak",
    "ceiling_tiebreak",
]
EXECUTION_VICE_XP_TIE_BAND = 0.10

# Conservative activation gate. 2025/26 is prior data for 2026/27, but the
# policy still has to improve the total and avoid a large half-season failure.
EXECUTION_MIN_HISTORICAL_GAIN = 8.0
EXECUTION_MAX_HALF_SEASON_DEGRADATION = 3.0
EXECUTION_MAX_BASELINE_RECONSTRUCTION_ERROR = 2.0

# Captain-confidence labels shown in the weekly production output.
CAPTAIN_CONFIDENCE_VERY_LOW_GAP = 0.05
CAPTAIN_CONFIDENCE_LOW_GAP = 0.15
CAPTAIN_CONFIDENCE_MEDIUM_GAP = 0.35
CAPTAIN_ALTERNATIVES_TO_SHOW = 8

# Cheap realistic transfer-opportunity diagnostic. It uses perfect hindsight
# but allows only one or two same-position transfers from the squad actually
# held before that deadline. Budget uses current market value + bank and is
# therefore an optimistic approximation to exact FPL selling value.
RUN_REALISTIC_TRANSFER_REGRET = False
TRANSFER_REGRET_MAX_TRANSFERS = 2
TRANSFER_REGRET_MAX_GAMEWEEKS: int | None = None

EXECUTION_POLICY_OUTPUT_PATH = (
    OUTPUT_DIRECTORY / "execution_policy_validation_v2_1.json"
)
TRANSFER_REGRET_OUTPUT_PATH = (
    OUTPUT_DIRECTORY / "realistic_transfer_regret_v2_1.json"
)

# --- Versioned artifacts -----------------------------------------------------
AUTONOMOUS_STATE_PATH = DATA_DIRECTORY / "autonomous_fpl_state_v2.json"
HISTORICAL_CACHE_PATH = DATA_DIRECTORY / "historical_fpl.csv.gz"
LIVE_SNAPSHOT_PATH = DATA_DIRECTORY / "fpl_weekly_snapshots.json.gz"
EXTERNAL_PRIOR_CACHE_PATH = DATA_DIRECTORY / "external_player_priors_v1_2.csv.gz"
XP_CALIBRATION_CACHE_PATH = DATA_DIRECTORY / "xp_calibration_v2_4_0.json"
PREMIUM_ATTACKER_CACHE_PATH = DATA_DIRECTORY / "premium_attacker_adjustment_v2_4_0.json"
WALK_FORWARD_POLICY_SCORECARD_CACHE = DATA_DIRECTORY / "walk_forward_policy_scorecard_v2_4_0.csv.gz"
POLICY_BACKTEST_CHECKPOINT_PATH = DATA_DIRECTORY / "walk_forward_policy_checkpoint_v2_2_7.csv.gz"
SEQUENTIAL_BACKTEST_CACHE_PATH = DATA_DIRECTORY / "sequential_backtest_v2_4_0.pkl.gz"
WEEKLY_OUTPUT_PATH = OUTPUT_DIRECTORY / "latest_weekly_recommendation.json"
POLICY_RECOMMENDATION_FALLBACK = "greedy_1gw_bank_aware"


def stable_configuration_signature(payload: dict) -> str:
    """Deterministic hash used to reject stale validation caches."""
    serialised = json.dumps(
        payload,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )
    return hashlib.sha256(serialised.encode("utf-8")).hexdigest()[:20]


# Separate from FORECAST_CACHE_SIGNATURE: signed historical validation
# artifacts remain valid, while Section 12 can still detect stale live
# shrinkage settings.

# ---------------------------------------------------------------------------
# Live-config normalisation
# ---------------------------------------------------------------------------
# These values are repeated defensively immediately before the live signature.
# This protects the notebook from accidental cell/source merge ordering issues
# when a newer ingestion layer is pasted into an older notebook.
#
# Existing values are preserved; defaults are used only if a symbol is absent.
LIVE_ALL_COMP_QUERY_CHUNK_DAYS = int(
    globals().get("LIVE_ALL_COMP_QUERY_CHUNK_DAYS", 7)
)
LIVE_ALL_COMP_MAX_LOOKAHEAD_DAYS = int(
    globals().get("LIVE_ALL_COMP_MAX_LOOKAHEAD_DAYS", 50)
)
LIVE_ALL_COMP_CACHE_SCHEMA_VERSION = int(
    globals().get("LIVE_ALL_COMP_CACHE_SCHEMA_VERSION", 2)
)

# Compatibility defaults for experimental variants that may contain these
# newer source/Understat signature keys. They are harmless when unused.
LIVE_ALL_COMP_SOURCE = str(
    globals().get("LIVE_ALL_COMP_SOURCE", "espn")
)
LIVE_CURRENT_UNDERSTAT_ENABLED = bool(
    globals().get("LIVE_CURRENT_UNDERSTAT_ENABLED", False)
)
CURRENT_UNDERSTAT_CACHE_SCHEMA_VERSION = int(
    globals().get("CURRENT_UNDERSTAT_CACHE_SCHEMA_VERSION", 1)
)

# Fail here with a readable message if a required production config value is
# somehow invalid, rather than later inside the live pipeline.
if LIVE_ALL_COMP_QUERY_CHUNK_DAYS < 1:
    raise ValueError(
        "LIVE_ALL_COMP_QUERY_CHUNK_DAYS must be >= 1."
    )
if LIVE_ALL_COMP_MAX_LOOKAHEAD_DAYS < 1:
    raise ValueError(
        "LIVE_ALL_COMP_MAX_LOOKAHEAD_DAYS must be >= 1."
    )
if LIVE_ALL_COMP_MAX_CONSECUTIVE_FAILURES < 1:
    raise ValueError(
        "LIVE_ALL_COMP_MAX_CONSECUTIVE_FAILURES must be >= 1."
    )
if LIVE_ALL_COMP_FAILURE_CACHE_HOURS < 0:
    raise ValueError(
        "LIVE_ALL_COMP_FAILURE_CACHE_HOURS must be >= 0."
    )

LIVE_FEATURE_SIGNATURE = stable_configuration_signature({
    "notebook_release": NOTEBOOK_RELEASE,
    "early_season_shrinkage_enabled": EARLY_SEASON_SHRINKAGE_ENABLED,
    "player_fade_fixtures": EARLY_SEASON_PLAYER_FADE_FIXTURES,
    "player_stable_archived_fixtures": EARLY_SEASON_PLAYER_STABLE_ARCHIVED_FIXTURES,
    "team_fade_matches": EARLY_SEASON_TEAM_FADE_MATCHES,
    "team_stable_archived_matches": EARLY_SEASON_TEAM_STABLE_ARCHIVED_MATCHES,
    "shrink_last_gw_role": EARLY_SEASON_SHRINK_LAST_GW_ROLE,
    "transfer_robustness_gate": WEEKLY_TRANSFER_ROBUSTNESS_GATE,
    "min_objective_gain_per_extra_transfer": WEEKLY_MIN_OBJECTIVE_GAIN_PER_EXTRA_TRANSFER,
    "protect_first_action_transfer_counts": WEEKLY_PROTECT_FIRST_ACTION_TRANSFER_COUNTS,
    "protected_first_actions_per_count": WEEKLY_PROTECTED_FIRST_ACTIONS_PER_TRANSFER_COUNT,
    "exact_subset_check": WEEKLY_EXACT_RAW_ACTION_SUBSET_CHECK,
    "lazy_chip_evaluation": WEEKLY_LAZY_CHIP_EVALUATION,
    "captain_highest_xp": PRODUCTION_CAPTAIN_HIGHEST_XP,
    "all_comp_enabled": LIVE_ALL_COMP_WORKLOAD_ENABLED,
    "all_comp_source": LIVE_ALL_COMP_SOURCE,
    "all_comp_cache_schema": LIVE_ALL_COMP_CACHE_SCHEMA_VERSION,
    "all_comp_max_lookahead": LIVE_ALL_COMP_MAX_LOOKAHEAD_DAYS,
    "all_comp_query_chunk_days": LIVE_ALL_COMP_QUERY_CHUNK_DAYS,
    "all_comp_max_consecutive_failures": LIVE_ALL_COMP_MAX_CONSECUTIVE_FAILURES,
    "all_comp_failure_cache_hours": LIVE_ALL_COMP_FAILURE_CACHE_HOURS,
    "all_comp_rest_factors": LIVE_ALL_COMP_PREVIOUS_REST_START_FACTORS,
    "all_comp_next_factor": LIVE_ALL_COMP_NEXT_MATCH_START_FACTOR,
    "all_comp_heavy_factor": LIVE_ALL_COMP_HEAVY_MINUTES_START_FACTOR,
    "current_fbref_enabled": LIVE_CURRENT_FBREF_ENABLED,
    "current_fbref_cache_schema": CURRENT_FBREF_CACHE_SCHEMA_VERSION,
    "current_understat_enabled": LIVE_CURRENT_UNDERSTAT_ENABLED,
    "current_understat_cache_schema": CURRENT_UNDERSTAT_CACHE_SCHEMA_VERSION,
    "current_fbref_event_blend": LIVE_CURRENT_FBREF_EVENT_BLEND_ENABLED,
    "set_piece_enabled": LIVE_SET_PIECE_ROLE_ENABLED,
    "set_piece_penalty_blend": SET_PIECE_PENALTY_ROLE_BLEND,
    "set_piece_corner_xa": SET_PIECE_CORNER_XA_ROLE_PER90,
    "set_piece_direct_fk_xg": SET_PIECE_DIRECT_FK_XG_ROLE_PER90,
})


print(
    "Notebook startup:",
    NOTEBOOK_RELEASE,
    "| live-config chunk days:",
    LIVE_ALL_COMP_QUERY_CHUNK_DAYS,
    "| all-comp source:",
    LIVE_ALL_COMP_SOURCE,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)


In [ ]:
# =============================================================================
# v2.6 MECHANISM FIXES — setup
# =============================================================================
# Diagnosed against 4 seasons of archived FPL data. See fixes/INTEGRATION.md.
#
# The corrections live in fixes/fpl_xp_fixes.py. Conversion factors are held in
# module-level globals that calculate_component_expected_points consults, so the
# v2.5 per-role-state integration inherits them automatically.

import sys as _sys
from pathlib import Path as _Path

for _candidate in (_Path.cwd(), _Path.cwd() / "fixes", _Path.cwd().parent / "fixes"):
    if (_candidate / "fpl_xp_fixes.py").exists():
        _sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError(
        "fpl_xp_fixes.py not found. Keep the fixes/ directory next to the "
        "notebook, or add its location to sys.path before this cell."
    )

import fpl_xp_fixes as fx

V26_ENABLED = True            # set False to run the notebook unchanged
V26_WILDCARD_THRESHOLD_ONLY = False  # WITHDRAWN: did not survive re-validation
V26_APPLY_ASSIST_CONVERSION = False  # FAILED its gate on the real pipeline
V26_APPLY_COVERAGE_MASK = True       # F2: mask 2022-23's fabricated zero xG/xA
V26_RELAX_UPPER_TAIL_MAE_GATE = True # v2.4 upper tail: judge on ranking, not MAE
V26_WILDCARD_MIN_HORIZON = 3         # below this, a wildcard cannot be judged
V26_CHIP_EXPIRY_SLACK = 1            # spare weeks before the expiry cascade
V26_APPLY_GOAL_CONVERSION = fx.APPLY_GOAL_CONVERSION_BY_DEFAULT  # False: goals/xG is ~1.0

# Identity factors until v26_set_conversion() is called.
V26_ASSIST_FACTORS: dict = {}
V26_GOAL_FACTORS: dict = {}
_V26_FACTOR_CACHE: dict = {}


def _v26_position_factor(position, factors):
    """Map a position Series to its conversion factor, defaulting to 1.0."""
    import pandas as _pd

    if not factors:
        return _pd.Series(1.0, index=position.index, dtype=float)
    lookup = {k: v for k, v in factors.items() if k != "__pooled__"}
    return (
        position.map(lookup)
        .astype(float)
        .fillna(float(factors.get("__pooled__", 1.0)))
    )


def v26_set_conversion(training_frame, *, label="default"):
    """
    Fit (and cache) realised/expected conversion factors on a TRAINING frame.

    Must be called with development/fold training rows only -- never with the
    holdout or live rows -- so the walk-forward validation stays leakage-safe.
    """
    global V26_ASSIST_FACTORS, V26_GOAL_FACTORS

    if not V26_ENABLED:
        V26_ASSIST_FACTORS, V26_GOAL_FACTORS = {}, {}
        return {}, {}

    key = (label, len(training_frame))
    if key not in _V26_FACTOR_CACHE:
        assists = fx.fit_event_conversion_factors(
            training_frame,
            realised_col="assists",
            expected_col="expected_assists",
            bounds=fx.ASSIST_CONVERSION_BOUNDS,
        ) if V26_APPLY_ASSIST_CONVERSION else {}
        goals = fx.fit_event_conversion_factors(
            training_frame,
            realised_col="goals_scored",
            expected_col="expected_goals",
            bounds=fx.GOAL_CONVERSION_BOUNDS,
        ) if V26_APPLY_GOAL_CONVERSION else {}
        _V26_FACTOR_CACHE[key] = (assists, goals)
        print(
            f"[v2.6] conversion factors ({label}, {len(training_frame):,} rows)"
            f"\n       assists/xA: "
            f"{ {k: round(v, 3) for k, v in assists.items()} }"
            f"\n       goals/xG  : "
            f"{ {k: round(v, 3) for k, v in goals.items()} }"
        )

    V26_ASSIST_FACTORS, V26_GOAL_FACTORS = _V26_FACTOR_CACHE[key]
    return V26_ASSIST_FACTORS, V26_GOAL_FACTORS


def v26_clear_conversion():
    """Reset to identity factors (used by the A/B comparison cell)."""
    global V26_ASSIST_FACTORS, V26_GOAL_FACTORS
    V26_ASSIST_FACTORS, V26_GOAL_FACTORS = {}, {}


print("[v2.6] mechanism fixes loaded; V26_ENABLED =", V26_ENABLED)


### Environment preflight

The notebook targets Python 3.10+ and requires:

```bash
pip install numpy pandas requests scipy scikit-learn soccerdata
```

The next cell fails early with a clear message when a required package is missing.

External new-signing priors are downloaded through SoccerData's FBref interface and cached locally. If FBref is temporarily unavailable, the notebook can fall back to its FPL-only priors unless strict external-prior mode is enabled.


In [ ]:
import importlib.util
import platform
import sys

MINIMUM_PYTHON = (3, 10)

if sys.version_info < MINIMUM_PYTHON:
    raise RuntimeError(
        "FPL Predictor requires Python 3.10 or newer. "
        f"Detected: {platform.python_version()}"
    )

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "requests": "requests",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "soccerdata": "soccerdata",
}
missing_packages = [
    install_name
    for import_name, install_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]
if missing_packages:
    raise RuntimeError(
        "Missing required packages. Install them with: pip install "
        + " ".join(missing_packages)
    )

import scipy
import sklearn

try:
    import soccerdata as sd
    _soccerdata_version = getattr(sd, "__version__", "unknown")
except Exception:
    _soccerdata_version = "unavailable"

required_runtime_objects = {
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "requests": requests.__version__,
    "scipy": scipy.__version__,
    "scikit-learn": sklearn.__version__,
    "soccerdata": _soccerdata_version,
}

Path("data").mkdir(parents=True, exist_ok=True)
Path("outputs").mkdir(parents=True, exist_ok=True)

print(f"FPL Predictor v{PROJECT_VERSION}")
print(f"Python: {platform.python_version()}")
print("Runtime packages:", required_runtime_objects)


## Autonomous production configuration

All user-facing flags are centralised in the **first code cell**. Do not edit duplicate switches later in the notebook.

The notebook keeps a small local ledger so the model can manage the same FPL team across the season without re-entering squad state every week. The public FPL entry endpoints are the source of truth after each published deadline; the local ledger preserves purchase prices, banked free transfers and pending model decisions. If you do not execute a recommendation exactly, the next run reconciles to what you actually did and continues from there.

Before Gameweek 1, `FPL_TEAM_ID` may remain `None`. After your FPL entry exists, set it once and leave autonomous mode enabled.

### v2.1 validation sequence

**Run A — required after this model update**

```python
RUN_FULL_POLICY_BACKTEST = True
RUN_SEQUENTIAL_BACKTEST = False
```

Run through Section 11. This creates signed v2.1 premium-attacker, xP-validation and policy artifacts. Then restart the kernel.

**Run B — recommended once**

```python
RUN_FULL_POLICY_BACKTEST = False
RUN_SEQUENTIAL_BACKTEST = True
```

Run the full sequential validation once. Then restart the kernel.

**Normal deadline use**

```python
RUN_FULL_POLICY_BACKTEST = False
RUN_SEQUENTIAL_BACKTEST = False
```

Run Sections 1–10 and Section 12. Stale validation caches are rejected automatically.


In [ ]:
# Derived autonomous configuration --------------------------------------------
# All user-facing flags are intentionally centralised in the first code cell.

print("Autonomous mode:", AUTONOMOUS_MODE)
print("FPL team ID:", FPL_TEAM_ID)
print("State ledger:", AUTONOMOUS_STATE_PATH)
print("xP validation artifact:", XP_CALIBRATION_CACHE_PATH)
print("Premium-attacker artifact:", PREMIUM_ATTACKER_CACHE_PATH)
print("Policy cache:", WALK_FORWARD_POLICY_SCORECARD_CACHE)
print("Sequential cache:", SEQUENTIAL_BACKTEST_CACHE_PATH)


### Data


In [ ]:
session = requests.Session()

session.headers.update({
    "User-Agent": "FPL-Predictor/2.0"
})


def get_fpl_data(endpoint: str) -> dict | list:
    """
    Download JSON data from an FPL API endpoint.

    Parameters
    ----------
    endpoint:
        Endpoint relative to the FPL API base URL,
        for example 'bootstrap-static/'.

    Returns
    -------
    dict or list
        Parsed JSON response.
    """
    url = f"{BASE_URL}{endpoint}"

    try:
        response = session.get(url, timeout=30)
        response.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(
            f"Unable to download FPL data from {url}"
        ) from exc

    try:
        return response.json()
    except requests.JSONDecodeError as exc:
        raise RuntimeError(
            f"FPL returned an invalid JSON response from {url}"
        ) from exc

In [ ]:
bootstrap = get_fpl_data("bootstrap-static/")
fixtures_raw = get_fpl_data("fixtures/")

print("Bootstrap sections:")
print(list(bootstrap.keys()))

print(f"\nNumber of fixtures: {len(fixtures_raw)}")

In [ ]:
teams = (
    pd.DataFrame(bootstrap["teams"])
    .rename(columns={
        "id": "team_id",
        "name": "team_name",
        "short_name": "team_short_name",
    })
)

positions = (
    pd.DataFrame(bootstrap["element_types"])
    .rename(columns={
        "id": "element_type",
        "singular_name": "position_name",
        "singular_name_short": "position",
    })
)

display(
    teams[
        [
            "team_id",
            "team_name",
            "team_short_name",
            "strength",
            "strength_attack_home",
            "strength_attack_away",
            "strength_defence_home",
            "strength_defence_away",
        ]
    ]
)

display(
    positions[
        [
            "element_type",
            "position_name",
            "position",
            "squad_select",
            "squad_min_play",
            "squad_max_play",
        ]
    ]
)

In [ ]:
players = pd.DataFrame(bootstrap["elements"])

players = (
    players
    .merge(
        teams[
            [
                "team_id",
                "team_name",
                "team_short_name",
            ]
        ],
        left_on="team",
        right_on="team_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        positions[
            [
                "element_type",
                "position_name",
                "position",
            ]
        ],
        on="element_type",
        how="left",
        validate="many_to_one",
    )
)

# FPL stores prices as integer tenths of a million:
# 45 means £4.5m, 100 means £10.0m, etc.
players["price"] = players["now_cost"] / 10

players["full_name"] = (
    players["first_name"].str.strip()
    + " "
    + players["second_name"].str.strip()
)
# FPL's player code is substantially more stable across seasons than the
# season-specific element ID. It is used to connect current players to the
# historical archive.
if "code" in players.columns:
    players["player_code"] = pd.to_numeric(
        players["code"],
        errors="coerce",
    ).astype("Int64")
else:
    players["player_code"] = pd.Series(
        pd.NA,
        index=players.index,
        dtype="Int64",
    )

players["player_key"] = (
    "current:element:"
    + players["id"].astype("Int64").astype(str)
)

has_player_code = players["player_code"].notna()
players.loc[has_player_code, "player_key"] = (
    "code:"
    + players.loc[
        has_player_code,
        "player_code",
    ].astype(str)
)


In [ ]:
numeric_columns = [
    "selected_by_percent",
    "form",
    "points_per_game",
    "value_form",
    "value_season",
    "influence",
    "creativity",
    "threat",
    "ict_index",
    "expected_goals",
    "expected_assists",
    "expected_goal_involvements",
    "expected_goals_conceded",
]

for column in numeric_columns:
    if column in players.columns:
        players[column] = pd.to_numeric(
            players[column],
            errors="coerce",
        )

In [ ]:
preferred_player_columns = [
    # Stable identifiers retained for later joins
    "id",
    "player_code",
    "player_key",
    "team_id",
    "element_type",

    # Player information
    "full_name",
    "web_name",
    "team_name",
    "team_short_name",
    "position",
    "price",

    # Availability information
    "status",
    "chance_of_playing_next_round",
    "news",

    # Official current set-piece responsibility
    "penalties_order",
    "penalties_text",
    "corners_and_indirect_freekicks_order",
    "corners_and_indirect_freekicks_text",
    "direct_freekicks_order",
    "direct_freekicks_text",

    # Current-season FPL information
    "selected_by_percent",
    "total_points",
    "points_per_game",
    "form",
    "minutes",
    "starts",
    "goals_scored",
    "assists",
    "clean_sheets",
    "goals_conceded",
    "saves",
    "bonus",
    "bps",

    # Current-season underlying statistics
    "expected_goals",
    "expected_assists",
    "expected_goal_involvements",
    "expected_goals_conceded",
]

available_player_columns = [
    column
    for column in preferred_player_columns
    if column in players.columns
]

players = players[available_player_columns].copy()

display(players.head(10))


### Fixtures

In [ ]:
fixtures = pd.DataFrame(fixtures_raw)

home_teams = teams[
    ["team_id", "team_name", "team_short_name"]
].rename(columns={
    "team_id": "team_h",
    "team_name": "home_team",
    "team_short_name": "home_team_short",
})

away_teams = teams[
    ["team_id", "team_name", "team_short_name"]
].rename(columns={
    "team_id": "team_a",
    "team_name": "away_team",
    "team_short_name": "away_team_short",
})

fixtures = (
    fixtures
    .merge(
        home_teams,
        on="team_h",
        how="left",
        validate="many_to_one",
    )
    .merge(
        away_teams,
        on="team_a",
        how="left",
        validate="many_to_one",
    )
)

fixtures["kickoff_time"] = pd.to_datetime(
    fixtures["kickoff_time"],
    utc=True,
    errors="coerce",
)

# Convert UTC fixture times to mainland Spain time.
fixtures["kickoff_time_spain"] = (
    fixtures["kickoff_time"]
    .dt.tz_convert("Europe/Madrid")
)

In [ ]:
preferred_fixture_columns = [
    # Stable identifiers retained for later joins
    "id",
    "event",
    "team_h",
    "team_a",

    # Fixture information
    "kickoff_time",
    "kickoff_time_spain",
    "home_team",
    "home_team_short",
    "away_team",
    "away_team_short",
    "team_h_score",
    "team_a_score",
    "team_h_difficulty",
    "team_a_difficulty",
    "started",
    "finished",
]

available_fixture_columns = [
    column
    for column in preferred_fixture_columns
    if column in fixtures.columns
]

fixtures = fixtures[available_fixture_columns].copy()

display(
    fixtures
    .sort_values("kickoff_time_spain")
    .head(20)
)


### Gameweeks

In [ ]:
gameweeks = pd.DataFrame(bootstrap["events"])

gameweeks["deadline_time"] = pd.to_datetime(
    gameweeks["deadline_time"],
    utc=True,
    errors="coerce",
)

gameweeks["deadline_time_spain"] = (
    gameweeks["deadline_time"]
    .dt.tz_convert("Europe/Madrid")
)

gameweek_columns = [
    "id",
    "name",
    "deadline_time_spain",
    "finished",
    "is_previous",
    "is_current",
    "is_next",
    "average_entry_score",
    "highest_score",
]

gameweek_columns = [
    column
    for column in gameweek_columns
    if column in gameweeks.columns
]

display(gameweeks[gameweek_columns])

### Validation

In [ ]:
assert not players.empty, "The player table is empty."
assert not fixtures.empty, "The fixture table is empty."
assert players["id"].is_unique, "Player IDs are not unique."
assert players["team_name"].notna().all(), "Some players have no mapped team."
assert players["position"].notna().all(), "Some players have no mapped position."

assert players["price"].between(3.0, 20.0).all(), (
    "At least one player price appears invalid."
)

print(f"Players loaded: {len(players):,}")
print(f"Teams loaded: {len(teams):,}")
print(f"Fixtures loaded: {len(fixtures):,}")
print(f"Gameweeks loaded: {len(gameweeks):,}")

print("\nPlayers by position:")
print(players["position"].value_counts())

print("\nPlayers by team:")
print(
    players["team_short_name"]
    .value_counts()
    .sort_index()
)

## 2. Player–fixture modelling table

The forecasting unit is **one player in one fixture**.

This is preferable to one row per player per Gameweek because a player may have:

- one fixture in a normal Gameweek;
- no fixture in a blank Gameweek;
- multiple fixtures in a double Gameweek.

The table built below is the common input for the minutes model, expected-points model, line-up selection, transfers and chip evaluation.

### Planning horizon

For live use, the first Gameweek in the analysis is taken from FPL's `is_next` flag.  
`CURRENT_GAMEWEEK_OVERRIDE` can later be used for historical backtesting or manual testing.

In [ ]:

def get_first_planning_gameweek(
    gameweek_table: pd.DataFrame,
    override: int | None = None,
) -> int:
    """
    Return the first Gameweek to include in the current analysis.

    For a live pre-deadline run, FPL marks the upcoming Gameweek with
    `is_next=True`. An explicit override is useful for historical tests.
    """
    if override is not None:
        valid_gameweeks = set(gameweek_table["id"].astype(int))
        if override not in valid_gameweeks:
            raise ValueError(
                f"Gameweek {override} is outside the available schedule."
            )
        return int(override)

    next_gameweeks = gameweek_table.loc[
        gameweek_table["is_next"].fillna(False),
        "id",
    ]

    if len(next_gameweeks) == 1:
        return int(next_gameweeks.iloc[0])

    if len(next_gameweeks) > 1:
        raise ValueError("More than one Gameweek is marked as next.")

    unfinished_gameweeks = gameweek_table.loc[
        ~gameweek_table["finished"].fillna(False),
        "id",
    ]

    if unfinished_gameweeks.empty:
        raise RuntimeError("The FPL season appears to be finished.")

    # Fallback for unusual API states.
    return int(unfinished_gameweeks.min())


first_planning_gameweek = get_first_planning_gameweek(
    gameweeks,
    CURRENT_GAMEWEEK_OVERRIDE,
)

last_planning_gameweek = min(
    int(gameweeks["id"].max()),
    first_planning_gameweek + PLANNING_HORIZON - 1,
)

planning_gameweeks = list(
    range(first_planning_gameweek, last_planning_gameweek + 1)
)

print(
    f"Planning Gameweeks {first_planning_gameweek}–"
    f"{last_planning_gameweek}"
)


### Convert fixtures to the perspective of each team

The official fixture table stores separate home-team and away-team fields.  
For modelling, each fixture is converted into two rows:

- one from the home team's perspective;
- one from the away team's perspective.

This gives every team–fixture row a consistent `team_id`, `opponent_id`, venue and fixture difficulty.

In [ ]:
home_fixtures = (
    fixtures
    .rename(columns={
        "id": "fixture_id",
        "event": "gameweek",
        "team_h": "team_id",
        "team_a": "opponent_id",
        "away_team": "opponent_name",
        "away_team_short": "opponent_short_name",
        "team_h_difficulty": "fixture_difficulty",
    })
    [
        [
            "fixture_id",
            "gameweek",
            "team_id",
            "opponent_id",
            "opponent_name",
            "opponent_short_name",
            "kickoff_time",
            "kickoff_time_spain",
            "fixture_difficulty",
            "started",
            "finished",
        ]
    ]
    .copy()
)

home_fixtures["is_home"] = True
home_fixtures["venue"] = "H"


away_fixtures = (
    fixtures
    .rename(columns={
        "id": "fixture_id",
        "event": "gameweek",
        "team_a": "team_id",
        "team_h": "opponent_id",
        "home_team": "opponent_name",
        "home_team_short": "opponent_short_name",
        "team_a_difficulty": "fixture_difficulty",
    })
    [
        [
            "fixture_id",
            "gameweek",
            "team_id",
            "opponent_id",
            "opponent_name",
            "opponent_short_name",
            "kickoff_time",
            "kickoff_time_spain",
            "fixture_difficulty",
            "started",
            "finished",
        ]
    ]
    .copy()
)

away_fixtures["is_home"] = False
away_fixtures["venue"] = "A"


team_fixtures = (
    pd.concat(
        [home_fixtures, away_fixtures],
        ignore_index=True,
    )
    .sort_values(
        ["team_id", "kickoff_time", "fixture_id"]
    )
    .reset_index(drop=True)
)

# This uses Premier League fixtures only. Cup and European matches will
# later be added separately when modelling rotation and recovery time.
team_fixtures["previous_league_kickoff"] = (
    team_fixtures
    .groupby("team_id")["kickoff_time"]
    .shift(1)
)

team_fixtures["days_since_previous_league_fixture"] = (
    (
        team_fixtures["kickoff_time"]
        - team_fixtures["previous_league_kickoff"]
    )
    .dt.total_seconds()
    .div(86_400)
)

display(team_fixtures.head(10))


### Restrict the schedule to the current planning horizon

Fixtures without an assigned Gameweek are excluded for now. They can be restored automatically after FPL assigns them to a later Gameweek.

In [ ]:
planning_team_fixtures = (
    team_fixtures.loc[
        team_fixtures["gameweek"].isin(planning_gameweeks)
    ]
    .copy()
    .sort_values(
        ["gameweek", "kickoff_time", "team_id"]
    )
    .reset_index(drop=True)
)

print(
    f"Team–fixture rows in horizon: "
    f"{len(planning_team_fixtures):,}"
)
print(
    f"Unique Premier League fixtures in horizon: "
    f"{planning_team_fixtures['fixture_id'].nunique():,}"
)


### Join every player to their team's fixtures

The result contains one row for each FPL-listed player in each fixture played by that player's club.

In [ ]:
player_feature_columns = [
    "id",
    "player_code",
    "player_key",
    "team_id",
    "element_type",
    "full_name",
    "web_name",
    "team_name",
    "team_short_name",
    "position",
    "price",
    "status",
    "chance_of_playing_next_round",
    "news",
    "selected_by_percent",
    "total_points",
    "points_per_game",
    "form",
    "minutes",
    "starts",
    "goals_scored",
    "assists",
    "clean_sheets",
    "goals_conceded",
    "saves",
    "bonus",
    "bps",
    "expected_goals",
    "expected_assists",
    "expected_goal_involvements",
    "expected_goals_conceded",
]

player_feature_columns = [
    column
    for column in player_feature_columns
    if column in players.columns
]

player_base = (
    players[player_feature_columns]
    .rename(columns={"id": "player_id"})
    .copy()
)

player_fixtures = (
    player_base
    .merge(
        planning_team_fixtures,
        on="team_id",
        how="inner",
        validate="many_to_many",
    )
    .sort_values(
        [
            "gameweek",
            "kickoff_time",
            "team_id",
            "position",
            "player_id",
        ]
    )
    .reset_index(drop=True)
)

display(player_fixtures.head(20))


### Validate the modelling table

These checks catch common data problems early, especially accidental duplicate joins and missing team mappings.

In [ ]:
assert not player_fixtures.empty, (
    "The player–fixture table is empty."
)

assert not player_fixtures.duplicated(
    ["player_id", "fixture_id"]
).any(), (
    "A player appears more than once in the same fixture."
)

assert player_fixtures["gameweek"].notna().all(), (
    "Some planning fixtures have no Gameweek."
)

assert player_fixtures["opponent_id"].notna().all(), (
    "Some player–fixture rows have no opponent."
)

assert player_fixtures["position"].notna().all(), (
    "Some player–fixture rows have no FPL position."
)

players_per_team = (
    players
    .groupby("team_id")
    .size()
    .rename("listed_players")
)

expected_rows = (
    planning_team_fixtures
    .merge(
        players_per_team,
        left_on="team_id",
        right_index=True,
        how="left",
        validate="many_to_one",
    )["listed_players"]
    .sum()
)

assert len(player_fixtures) == expected_rows, (
    "The player–fixture row count does not match the schedule."
)

print(f"Player–fixture rows: {len(player_fixtures):,}")
print(
    "Unique players represented: "
    f"{player_fixtures['player_id'].nunique():,}"
)
print(
    "Unique fixtures represented: "
    f"{player_fixtures['fixture_id'].nunique():,}"
)


### Inspect blanks and doubles

A count of zero represents a blank Gameweek; a count above one represents a double or larger Gameweek.

In [ ]:
schedule_index = pd.MultiIndex.from_product(
    [
        sorted(teams["team_id"].unique()),
        planning_gameweeks,
    ],
    names=["team_id", "gameweek"],
)

team_gameweek_counts = (
    planning_team_fixtures
    .groupby(["team_id", "gameweek"])
    .size()
    .reindex(schedule_index, fill_value=0)
    .rename("fixture_count")
    .reset_index()
    .merge(
        teams[
            [
                "team_id",
                "team_name",
                "team_short_name",
            ]
        ],
        on="team_id",
        how="left",
        validate="many_to_one",
    )
)

unusual_gameweeks = (
    team_gameweek_counts.loc[
        team_gameweek_counts["fixture_count"] != 1
    ]
    [
        [
            "gameweek",
            "team_short_name",
            "team_name",
            "fixture_count",
        ]
    ]
    .sort_values(["gameweek", "team_short_name"])
)

if unusual_gameweeks.empty:
    print(
        "No blank or double Gameweeks are currently assigned "
        "inside the planning horizon."
    )
else:
    display(unusual_gameweeks)


### Summary by Gameweek

The number of player–fixture rows can exceed the number of unique players when a club has multiple fixtures in the same Gameweek.

In [ ]:
gameweek_summary = (
    player_fixtures
    .groupby("gameweek")
    .agg(
        fixtures=("fixture_id", "nunique"),
        teams_with_fixture=("team_id", "nunique"),
        unique_players=("player_id", "nunique"),
        player_fixture_rows=("player_id", "size"),
    )
    .reset_index()
)

display(gameweek_summary)


### Current live-table limitations

The current player–fixture table correctly represents the official 2026/27 player list and Premier League schedule, but it does **not** yet produce forecasts.

In particular:

- FPL's pre-season player statistics contain little or no current-season information;
- the official team-strength fields are currently uninformative;
- European and domestic cup fixtures are not yet included;
- injury news has not yet been converted into probabilities;
- rotation, expected minutes and expected points have not yet been modelled.

The historical section below provides the training outcomes and leakage-safe rolling features needed for the first expected-minutes baseline.

## 3. Historical data and leakage-safe features

The live 2026/27 analysis uses the official FPL API. Historical match outcomes are loaded from the public
[`vaastav/Fantasy-Premier-League`](https://github.com/vaastav/Fantasy-Premier-League) archive.

### Design choices

- **Seasons:** start with 2022/23–2025/26. These seasons reflect the modern five-substitution environment and provide enough recent observations without giving very old seasons equal influence.
- **Unit of observation:** retain one row per player per fixture.
- **Stable player identity:** use FPL's player `code` where available because the season-specific `element` ID changes between seasons.
- **No same-Gameweek leakage:** historical rolling features are calculated at player–Gameweek level and shifted by one complete Gameweek before being merged back to fixture rows. Both fixtures in a double Gameweek therefore receive the same pre-deadline player-history features.
- **No archived `xP`:** the source repository warns that this field may have been scraped after matches and can therefore contain information unavailable at the deadline. It is deliberately discarded.
- **Market variables:** same-row price, ownership and transfer counts are not treated as pre-deadline features. Only values observed in a previous archived Gameweek are retained.
- **Scoring changes:** historical `total_points` is kept as a useful benchmark target, but the final model will predict scoring components separately and apply the current 2026/27 scoring rules.
- **Current-season history:** one compressed archive stores the official pre-deadline bootstrap and fixture snapshots for each Gameweek. This will give us exact point-in-time live data without creating dozens of separate files.

In [ ]:
CURRENT_SEASON = "2026-27"

# Recent seasons are deliberately prioritised. Older seasons can be added
# later when testing whether the extra sample size improves validation results.
HISTORICAL_SEASONS = [
    "2022-23",
    "2023-24",
    "2024-25",
    "2025-26",
]

HISTORICAL_GITHUB_ROOT = (
    "https://raw.githubusercontent.com/"
    "vaastav/Fantasy-Premier-League/master/data"
)


# Set this to True only when the source data has changed or the list of
# historical seasons has been modified.

POSITION_ID_TO_NAME = {
    1: "GK",
    2: "DEF",
    3: "MID",
    4: "FWD",
    5: "AM",
}

## 3A. Previous-season external priors for new signings

FPL history is strongest for established Premier League players. A brand-new overseas signing can otherwise look like a generic positional prior because there are no earlier FPL rows for that player.

This section adds a leakage-safe prior from the **immediately previous season** for new-to-PL players, overseas/returning signings and sparse-history players:

- player appearances, starts and minutes;
- goals and assists;
- xG and xA;
- goalkeeper saves where available;
- a source-league-to-Premier-League translation based on same-position league reference rates;
- a reliability score based on sample minutes.

Automatic ingestion uses SoccerData's FBref (playing time/role) and Understat (xG/xA) interface for the top five European leagues. A manual override hook is included for signings from unsupported leagues.

The external prior is not treated as permanent truth. Its explicit blend is reserved for players whose latest prior came from outside the Premier League (plus genuinely sparse-PL-history players), and it fades to zero across the first eight current-season Premier League fixtures. Established PL players still expose prior-season fields to the fitted model, but they are not given a second explicit blend.


In [ ]:

from difflib import SequenceMatcher
import re
import unicodedata
import warnings

# External prior configuration -------------------------------------------------
#
# The current live season is 2026/27. To avoid leakage, external priors use
# only the immediately PREVIOUS season for each FPL season being modelled.
#
# SoccerData's FBref interface has built-in support for the top five European
# leagues. Additional leagues can be added through SoccerData's own custom
# league configuration if desired. The manual-prior hook below covers players
# whose previous league is not available from the automatic source.


EXTERNAL_SOCCERDATA_CACHE_DIR = (
    DATA_DIRECTORY / "soccerdata_fbref"
)
EXTERNAL_UNDERSTAT_CACHE_DIR = (
    DATA_DIRECTORY / "soccerdata_understat"
)

# Understat is used as a robust source for previous-season xG/xA.
# FBref remains the primary source for playing time, role and goalkeeper saves.

EXTERNAL_FBREF_LEAGUES = [
    "ENG-Premier League",
    "ESP-La Liga",
    "GER-Bundesliga",
    "ITA-Serie A",
    "FRA-Ligue 1",
]

# Optional extra leagues can be added here AFTER they have been configured
# in SoccerData. The automatic loader skips any source that fails rather than
# invalidating the rest of the FPL notebook.
EXTERNAL_FBREF_EXTRA_LEAGUES = [
    "ENG-Championship",
    "NED-Eredivisie",
    "POR-Primeira Liga",
    "BEL-Pro League",
]

# High-confidence manual name overrides. Keys and values are normalised
# automatically, so accents/punctuation/case do not matter.
EXTERNAL_NAME_OVERRIDES = {
    # "FPL name": "FBref name",
}

# Manual prior rows are useful for signings from leagues not covered by the
# automatic FBref scrape. Values should describe the player's PREVIOUS season.
# `pl_equivalent_*` can be supplied directly; if omitted, raw rates are used.
EXTERNAL_MANUAL_PRIORS = [
    # Verified role/minutes fallback for a high-priority 2026/27 new-to-PL
    # player whose Belgian source is not guaranteed to be configured locally.
    # 2025/26 Union SG league record: 38 appearances, 36 starts, 3,087 minutes,
    # 3 goals and 5 assists. No PL-equivalent xG/xA is hard-coded here.
    {
        "season": "2025-26",
        "player": "Anan Khalaili",
        "position": "DEF",
        "source_leagues": "BEL-Pro League / Union SG (manual verified fallback)",
        "minutes": 3087,
        "matches": 38,
        "starts": 36,
        "goals_per90": 3.0 / (3087.0 / 90.0),
        "assists_per90": 5.0 / (3087.0 / 90.0),
        "source_note": (
            "Verified 2025/26 Union SG role/minutes fallback; "
            "no hard-coded PL-equivalent xG/xA."
        ),
    },
    # Add other unsupported-league signings only after verifying the
    # immediately preceding season. Prefer automatic SoccerData ingestion.
]

EXTERNAL_MIN_REFERENCE_MINUTES = 450.0
EXTERNAL_FULL_RELIABILITY_MINUTES = 1800.0
EXTERNAL_FUZZY_MATCH_THRESHOLD = 0.94
EXTERNAL_FUZZY_MATCH_MARGIN = 0.03
EXTERNAL_INITIAL_SURNAME_MATCH_THRESHOLD = 0.70

# The explicit blend is intentionally strongest only for players with little
# or no Premier League history. It fades out after several PL fixtures.
NEW_PLAYER_PRIOR_FADE_OUT_PL_FIXTURES = 8.0
EXTERNAL_MAX_MINUTES_BLEND = 0.35
EXTERNAL_MAX_EVENT_BLEND = 0.60


def previous_season_label(season: str) -> str:
    """Return the season immediately preceding a canonical YYYY-YY label."""
    start = int(str(season).split("-")[0])
    previous_start = start - 1
    return f"{previous_start}-{str(start)[-2:]}"


def canonical_external_season(value: object) -> str:
    """Normalise SoccerData season values to YYYY-YY."""
    if pd.isna(value):
        return ""

    text = str(value).strip()
    four_digit_years = re.findall(r"(?:19|20)\d{2}", text)
    if four_digit_years:
        start = int(four_digit_years[0])
        return f"{start}-{str(start + 1)[-2:]}"

    digits = re.sub(r"\D", "", text)
    if len(digits) == 4 and int(digits[:2]) <= 99:
        start = 2000 + int(digits[:2])
        return f"{start}-{digits[2:]}"
    if len(digits) == 2:
        start = 2000 + int(digits)
        return f"{start}-{str(start + 1)[-2:]}"

    raise ValueError(f"Could not interpret external season value: {value!r}")


def normalise_player_name(value: object) -> str:
    """Create a conservative accent/punctuation-insensitive player key."""
    if pd.isna(value):
        return ""

    text = str(value).replace("_", " ").strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        character
        for character in text
        if not unicodedata.combining(character)
    )
    text = re.sub(r"[^a-z0-9 ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def _snake_external_column(value: object) -> str:
    if isinstance(value, tuple):
        parts = [
            str(part).strip()
            for part in value
            if str(part).strip()
            and not str(part).startswith("Unnamed")
        ]
        text = "_".join(parts)
    else:
        text = str(value).strip()

    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        character
        for character in text
        if not unicodedata.combining(character)
    )
    text = text.lower()
    text = text.replace("%", " pct ")
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return re.sub(r"_+", "_", text).strip("_")


def _make_unique_column_names(columns: list[str]) -> list[str]:
    counts: dict[str, int] = {}
    result: list[str] = []
    for column in columns:
        base = column or "column"
        count = counts.get(base, 0)
        counts[base] = count + 1
        result.append(
            base if count == 0 else f"{base}_{count + 1}"
        )
    return result


def flatten_soccerdata_frame(data: pd.DataFrame) -> pd.DataFrame:
    """Flatten SoccerData's MultiIndex columns and index."""
    frame = data.copy()

    if isinstance(frame.columns, pd.MultiIndex):
        frame.columns = _make_unique_column_names(
            [_snake_external_column(column) for column in frame.columns]
        )
    else:
        frame.columns = _make_unique_column_names(
            [_snake_external_column(column) for column in frame.columns]
        )

    frame = frame.reset_index()
    frame.columns = _make_unique_column_names(
        [_snake_external_column(column) for column in frame.columns]
    )
    return frame


def _find_external_column(
    frame: pd.DataFrame,
    exact: tuple[str, ...] = (),
    required_tokens: tuple[str, ...] = (),
    forbidden_tokens: tuple[str, ...] = (),
) -> str | None:
    """Find one semantic column without depending on one FBref layout."""
    columns = list(frame.columns)

    for candidate in exact:
        if candidate in columns:
            return candidate

    for column in columns:
        if (
            all(token in column for token in required_tokens)
            and not any(token in column for token in forbidden_tokens)
        ):
            return column

    return None


def _numeric_external_column(
    frame: pd.DataFrame,
    column: str | None,
) -> pd.Series:
    if column is None:
        return pd.Series(np.nan, index=frame.index, dtype=float)
    return pd.to_numeric(frame[column], errors="coerce")


def _normalise_external_position(value: object) -> str | None:
    text = str(value).upper() if pd.notna(value) else ""
    if text in {"GK", "GKP"} or "GK" in text:
        return "GK"
    if text in {"FWD", "FW"} or "FW" in text:
        return "FWD"
    if text in {"MID", "MF"} or "MF" in text:
        return "MID"
    if text in {"DEF", "DF"} or "DF" in text:
        return "DEF"
    return None



def tidy_fbref_standard(data: pd.DataFrame) -> pd.DataFrame:
    """
    Normalise FBref standard player-season data.

    FBref/SoccerData layouts have changed over time, so the extractor searches
    both total and per-90 variants of xG/xAG instead of assuming one exact
    column layout.
    """
    frame = flatten_soccerdata_frame(data)

    required_identity = ["league", "season", "team", "player"]
    missing_identity = [
        column for column in required_identity
        if column not in frame.columns
    ]
    if missing_identity:
        raise ValueError(
            "FBref standard table is missing identity columns: "
            f"{missing_identity}"
        )

    position_col = _find_external_column(
        frame,
        exact=("pos",),
        required_tokens=("pos",),
    )
    appearances_col = _find_external_column(
        frame,
        exact=("playing_time_mp", "mp"),
        required_tokens=("playing_time", "mp"),
    )
    starts_col = _find_external_column(
        frame,
        exact=("playing_time_starts", "starts"),
        required_tokens=("starts",),
        forbidden_tokens=("per_90", "pct"),
    )
    minutes_col = _find_external_column(
        frame,
        exact=("playing_time_min", "min", "minutes"),
        required_tokens=("playing_time", "min"),
        forbidden_tokens=("pct",),
    )
    goals_col = _find_external_column(
        frame,
        exact=("performance_gls", "gls"),
        required_tokens=("gls",),
        forbidden_tokens=("per_90",),
    )
    assists_col = _find_external_column(
        frame,
        exact=("performance_ast", "ast"),
        required_tokens=("ast",),
        forbidden_tokens=("per_90",),
    )

    xg_col = _find_external_column(
        frame,
        exact=("expected_xg", "standard_expected_xg"),
        required_tokens=("xg",),
        forbidden_tokens=("npxg", "per_90", "xg_plus", "xg_sh"),
    )
    xa_col = _find_external_column(
        frame,
        exact=(
            "expected_xag",
            "expected_xa",
            "standard_expected_xag",
            "standard_expected_xa",
            "xag",
            "xa",
        ),
        required_tokens=("xa",),
        forbidden_tokens=("per_90", "xag_plus"),
    )

    xg_per90_col = _find_external_column(
        frame,
        exact=(
            "per_90_minutes_xg",
            "per_90_xg",
            "expected_xg_per_90",
        ),
        required_tokens=("xg", "90"),
        forbidden_tokens=("npxg",),
    )
    xa_per90_col = _find_external_column(
        frame,
        exact=(
            "per_90_minutes_xag",
            "per_90_minutes_xa",
            "per_90_xag",
            "per_90_xa",
        ),
        required_tokens=("xa", "90"),
    )

    output = frame[required_identity].copy()
    output["external_position_raw"] = (
        frame[position_col] if position_col is not None else pd.NA
    )
    output["external_position"] = output[
        "external_position_raw"
    ].map(_normalise_external_position)
    output["external_appearances"] = _numeric_external_column(
        frame, appearances_col
    )
    output["external_starts"] = _numeric_external_column(
        frame, starts_col
    )
    output["external_minutes"] = _numeric_external_column(
        frame, minutes_col
    )
    output["external_goals"] = _numeric_external_column(
        frame, goals_col
    )
    output["external_assists"] = _numeric_external_column(
        frame, assists_col
    )
    output["external_xg"] = _numeric_external_column(
        frame, xg_col
    )
    output["external_xa"] = _numeric_external_column(
        frame, xa_col
    )
    output["fbref_xg_per90_direct"] = _numeric_external_column(
        frame, xg_per90_col
    )
    output["fbref_xa_per90_direct"] = _numeric_external_column(
        frame, xa_per90_col
    )
    return output


def tidy_fbref_shooting(data: pd.DataFrame) -> pd.DataFrame:
    """Normalise FBref shooting data and preserve both xG totals and per-90."""
    frame = flatten_soccerdata_frame(data)
    identity = ["league", "season", "team", "player"]

    if any(column not in frame.columns for column in identity):
        return pd.DataFrame(
            columns=identity + ["shooting_xg", "shooting_xg_per90"]
        )

    xg_col = _find_external_column(
        frame,
        exact=("expected_xg", "standard_xg"),
        required_tokens=("xg",),
        forbidden_tokens=("npxg", "per_90", "xg_sh", "xg_plus"),
    )
    xg_per90_col = _find_external_column(
        frame,
        exact=(
            "per_90_minutes_xg",
            "per_90_xg",
            "expected_xg_per_90",
        ),
        required_tokens=("xg", "90"),
        forbidden_tokens=("npxg",),
    )

    output = frame[identity].copy()
    output["shooting_xg"] = _numeric_external_column(
        frame, xg_col
    )
    output["shooting_xg_per90"] = _numeric_external_column(
        frame, xg_per90_col
    )
    return output

def tidy_fbref_keeper(data: pd.DataFrame) -> pd.DataFrame:
    frame = flatten_soccerdata_frame(data)
    identity = ["league", "season", "team", "player"]

    if any(column not in frame.columns for column in identity):
        return pd.DataFrame(columns=identity + ["external_saves"])

    saves_col = _find_external_column(
        frame,
        exact=("performance_saves", "saves"),
        required_tokens=("saves",),
        forbidden_tokens=("pct", "per_90"),
    )
    output = frame[identity].copy()
    output["external_saves"] = _numeric_external_column(
        frame, saves_col
    )
    return output




def tidy_understat_player_season(data: pd.DataFrame) -> pd.DataFrame:
    """
    Normalise SoccerData Understat player-season xG/xA.

    Understat provides player-season xG and xA directly. The data are
    aggregated to one player/league/season row before joining to FBref so
    intra-season club transfers cannot duplicate the expected-stat totals.
    """
    if data is None or data.empty:
        return pd.DataFrame(
            columns=[
                "external_season",
                "league",
                "external_name_key",
                "understat_minutes",
                "understat_xg_per90",
                "understat_xa_per90",
            ]
        )

    frame = flatten_soccerdata_frame(data)

    required = ["league", "season", "player"]
    if any(column not in frame.columns for column in required):
        raise ValueError(
            "Understat player-season data are missing one of: "
            f"{required}"
        )

    minutes_col = _find_external_column(
        frame,
        exact=("minutes",),
        required_tokens=("minutes",),
    )
    xg_col = _find_external_column(
        frame,
        exact=("xg",),
        required_tokens=("xg",),
        forbidden_tokens=("chain", "buildup", "np_"),
    )
    xa_col = _find_external_column(
        frame,
        exact=("xa",),
        required_tokens=("xa",),
    )

    tidy = frame[required].copy()
    tidy["external_season"] = tidy["season"].map(
        canonical_external_season
    )
    tidy["external_name_key"] = tidy["player"].map(
        normalise_player_name
    )
    tidy["understat_minutes"] = _numeric_external_column(
        frame, minutes_col
    )
    tidy["understat_xg"] = _numeric_external_column(
        frame, xg_col
    )
    tidy["understat_xa"] = _numeric_external_column(
        frame, xa_col
    )

    grouped = (
        tidy.groupby(
            [
                "external_season",
                "league",
                "external_name_key",
            ],
            as_index=False,
            dropna=False,
        )
        .agg(
            understat_minutes=("understat_minutes", "sum"),
            understat_xg=("understat_xg", "sum"),
            understat_xa=("understat_xa", "sum"),
        )
    )

    exposure = grouped["understat_minutes"].div(90.0)
    grouped["understat_xg_per90"] = (
        grouped["understat_xg"] / exposure
    ).where(exposure.gt(0))
    grouped["understat_xa_per90"] = (
        grouped["understat_xa"] / exposure
    ).where(exposure.gt(0))

    return grouped[
        [
            "external_season",
            "league",
            "external_name_key",
            "understat_minutes",
            "understat_xg_per90",
            "understat_xa_per90",
        ]
    ]

def _weighted_average(
    values: pd.Series,
    weights: pd.Series,
) -> float:
    numeric_values = pd.to_numeric(values, errors="coerce")
    numeric_weights = pd.to_numeric(weights, errors="coerce").fillna(0.0)
    valid = numeric_values.notna() & numeric_weights.gt(0)
    if valid.any():
        return float(
            np.average(
                numeric_values.loc[valid],
                weights=numeric_weights.loc[valid],
            )
        )
    valid = numeric_values.notna()
    if valid.any():
        return float(numeric_values.loc[valid].mean())
    return np.nan



def build_external_player_prior_table(
    standard: pd.DataFrame,
    shooting: pd.DataFrame,
    keeper: pd.DataFrame,
    expected_stats: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """
    Create one previous-season performance prior per player.

    Playing time/role comes primarily from FBref. Expected attacking metrics
    use the following precedence:

        Understat xG/xA
        -> direct FBref per-90 xG/xAG
        -> FBref shooting per-90 xG
        -> FBref total xG/xA divided by minutes

    The final rates are translated to Premier League equivalents relative to
    same-position league baselines.
    """
    identity = ["league", "season", "team", "player"]

    data = standard.merge(
        shooting,
        on=identity,
        how="left",
        validate="one_to_one",
    )
    data = data.merge(
        keeper,
        on=identity,
        how="left",
        validate="one_to_one",
    )

    data["external_season"] = data["season"].map(
        canonical_external_season
    )
    data["external_name_key"] = data["player"].map(
        normalise_player_name
    )

    if expected_stats is not None and not expected_stats.empty:
        data = data.merge(
            expected_stats,
            on=[
                "external_season",
                "league",
                "external_name_key",
            ],
            how="left",
            validate="many_to_one",
        )
    else:
        for column in [
            "understat_minutes",
            "understat_xg_per90",
            "understat_xa_per90",
        ]:
            data[column] = np.nan

    numeric_columns = [
        "external_appearances",
        "external_starts",
        "external_minutes",
        "external_goals",
        "external_assists",
        "external_xg",
        "external_xa",
        "fbref_xg_per90_direct",
        "fbref_xa_per90_direct",
        "shooting_xg",
        "shooting_xg_per90",
        "external_saves",
        "understat_minutes",
        "understat_xg_per90",
        "understat_xa_per90",
    ]
    for column in numeric_columns:
        if column not in data.columns:
            data[column] = np.nan
        data[column] = pd.to_numeric(
            data[column],
            errors="coerce",
        )

    data["external_xg"] = data["external_xg"].fillna(
        data["shooting_xg"]
    )

    exposure_90 = data["external_minutes"].div(90.0)
    xg_from_totals = (
        data["external_xg"] / exposure_90
    ).where(exposure_90.gt(0))
    xa_from_totals = (
        data["external_xa"] / exposure_90
    ).where(exposure_90.gt(0))

    data["external_goals_per90"] = (
        data["external_goals"] / exposure_90
    ).where(exposure_90.gt(0))
    data["external_assists_per90"] = (
        data["external_assists"] / exposure_90
    ).where(exposure_90.gt(0))

    data["external_xg_per90"] = (
        data["understat_xg_per90"]
        .combine_first(data["fbref_xg_per90_direct"])
        .combine_first(data["shooting_xg_per90"])
        .combine_first(xg_from_totals)
    )
    data["external_xa_per90"] = (
        data["understat_xa_per90"]
        .combine_first(data["fbref_xa_per90_direct"])
        .combine_first(xa_from_totals)
    )
    data["external_saves_per90"] = (
        data["external_saves"] / exposure_90
    ).where(exposure_90.gt(0))

    data["external_expected_source"] = np.select(
        [
            data["understat_xg_per90"].notna()
            | data["understat_xa_per90"].notna(),
            data["fbref_xg_per90_direct"].notna()
            | data["fbref_xa_per90_direct"].notna()
            | data["shooting_xg_per90"].notna(),
            xg_from_totals.notna() | xa_from_totals.notna(),
        ],
        [
            "Understat",
            "FBref per90",
            "FBref totals",
        ],
        default="none",
    )

    data["external_reliability_row"] = (
        data["external_minutes"]
        .div(EXTERNAL_FULL_RELIABILITY_MINUTES)
        .clip(0.0, 1.0)
    )

    reference_pool = data.loc[
        data["external_minutes"].ge(
            EXTERNAL_MIN_REFERENCE_MINUTES
        )
        & data["external_position"].notna()
    ].copy()

    translated_metrics = {
        "external_xg_per90": "external_pl_equivalent_xg_per90",
        "external_xa_per90": "external_pl_equivalent_xa_per90",
        "external_saves_per90": "external_pl_equivalent_saves_per90",
    }

    for source_metric, translated_metric in translated_metrics.items():
        source_reference = (
            reference_pool.groupby(
                ["external_season", "league", "external_position"],
                dropna=False,
            )[source_metric]
            .mean()
            .rename("source_reference")
            .reset_index()
        )
        pl_reference = (
            reference_pool.loc[
                reference_pool["league"].eq(
                    "ENG-Premier League"
                )
            ]
            .groupby(
                ["external_season", "external_position"],
                dropna=False,
            )[source_metric]
            .mean()
            .rename("pl_reference")
            .reset_index()
        )
        overall_reference = (
            reference_pool.groupby(
                ["external_season", "external_position"],
                dropna=False,
            )[source_metric]
            .mean()
            .rename("overall_reference")
            .reset_index()
        )

        data = data.merge(
            source_reference,
            on=[
                "external_season",
                "league",
                "external_position",
            ],
            how="left",
            validate="many_to_one",
        )
        data = data.merge(
            pl_reference,
            on=["external_season", "external_position"],
            how="left",
            validate="many_to_one",
        )
        data = data.merge(
            overall_reference,
            on=["external_season", "external_position"],
            how="left",
            validate="many_to_one",
        )

        source_ref = data["source_reference"].fillna(
            data["overall_reference"]
        )
        pl_ref = data["pl_reference"].fillna(
            data["overall_reference"]
        )
        raw_rate = pd.to_numeric(
            data[source_metric],
            errors="coerce",
        )

        relative_rate = (
            raw_rate / source_ref.replace(0.0, np.nan)
        ).clip(0.0, 4.0)

        reliability = data[
            "external_reliability_row"
        ].fillna(0.0)

        shrunk_relative = (
            1.0
            + reliability
            * (relative_rate - 1.0)
        )
        translated = pl_ref * shrunk_relative
        translated = translated.where(raw_rate.notna())

        translated = translated.where(
            ~data["league"].eq("ENG-Premier League"),
            raw_rate,
        )

        data[translated_metric] = translated.clip(lower=0.0)
        data = data.drop(
            columns=[
                "source_reference",
                "pl_reference",
                "overall_reference",
            ]
        )

    def aggregate_player(group: pd.DataFrame) -> pd.Series:
        weights = group["external_minutes"].fillna(0.0)
        highest_minutes_index = (
            weights.fillna(-1).idxmax()
            if len(group)
            else None
        )
        representative = (
            group.loc[highest_minutes_index]
            if highest_minutes_index is not None
            else group.iloc[0]
        )

        total_minutes = float(
            group["external_minutes"].fillna(0.0).sum()
        )
        total_appearances = float(
            group["external_appearances"].fillna(0.0).sum()
        )
        total_starts = float(
            group["external_starts"].fillna(0.0).sum()
        )

        expected_sources = [
            value
            for value in group[
                "external_expected_source"
            ].dropna().astype(str).unique().tolist()
            if value != "none"
        ]

        return pd.Series({
            "external_player": representative["player"],
            "external_source_leagues": ", ".join(
                sorted(
                    set(
                        group["league"].dropna().astype(str)
                    )
                )
            ),
            "external_expected_source": (
                ", ".join(sorted(expected_sources))
                if expected_sources
                else "none"
            ),
            "external_minutes": total_minutes,
            "external_appearances": total_appearances,
            "external_starts": total_starts,
            "external_start_share": (
                total_starts / total_appearances
                if total_appearances > 0
                else np.nan
            ),
            "external_minutes_per_appearance": (
                total_minutes / total_appearances
                if total_appearances > 0
                else np.nan
            ),
            "external_goals_per90": _weighted_average(
                group["external_goals_per90"],
                weights,
            ),
            "external_assists_per90": _weighted_average(
                group["external_assists_per90"],
                weights,
            ),
            "external_xg_per90": _weighted_average(
                group["external_xg_per90"],
                weights,
            ),
            "external_xa_per90": _weighted_average(
                group["external_xa_per90"],
                weights,
            ),
            "external_saves_per90": _weighted_average(
                group["external_saves_per90"],
                weights,
            ),
            "external_pl_equivalent_xg_per90": _weighted_average(
                group["external_pl_equivalent_xg_per90"],
                weights,
            ),
            "external_pl_equivalent_xa_per90": _weighted_average(
                group["external_pl_equivalent_xa_per90"],
                weights,
            ),
            "external_pl_equivalent_saves_per90": _weighted_average(
                group["external_pl_equivalent_saves_per90"],
                weights,
            ),
            "external_prior_reliability": float(
                np.clip(
                    total_minutes
                    / EXTERNAL_FULL_RELIABILITY_MINUTES,
                    0.0,
                    1.0,
                )
            ),
            "external_prior_non_pl": bool(
                any(
                    league != "ENG-Premier League"
                    for league in group[
                        "league"
                    ].dropna().astype(str)
                )
            ),
        })

    priors = (
        data.loc[
            data["external_name_key"].ne("")
        ]
        .groupby(
            [
                "external_season",
                "external_name_key",
                "external_position",
            ],
            sort=False,
            dropna=False,
        )
        .apply(aggregate_player)
        .reset_index()
    )

    priors["external_role_score"] = (
        0.65
        * pd.to_numeric(
            priors["external_start_share"],
            errors="coerce",
        ).clip(0.0, 1.0)
        + 0.35
        * (
            pd.to_numeric(
                priors[
                    "external_minutes_per_appearance"
                ],
                errors="coerce",
            )
            .div(90.0)
            .clip(0.0, 1.0)
        )
    )

    if "external_prior_source_note" not in priors.columns:
        priors["external_prior_source_note"] = pd.NA

    return priors

def manual_external_prior_frame() -> pd.DataFrame:
    """Normalise optional hand-entered priors to the automatic schema."""
    if not EXTERNAL_MANUAL_PRIORS:
        return pd.DataFrame()

    manual = pd.DataFrame(EXTERNAL_MANUAL_PRIORS).copy()
    manual["external_season"] = manual["season"].map(
        canonical_external_season
    )
    manual["external_player"] = manual["player"].astype(str)
    manual["external_name_key"] = manual[
        "external_player"
    ].map(normalise_player_name)
    manual["external_position"] = manual.get(
        "position",
        pd.Series(pd.NA, index=manual.index),
    ).map(_normalise_external_position)
    manual["external_source_leagues"] = manual.get(
        "source_leagues",
        "Manual",
    )
    manual["external_expected_source"] = "Manual"
    manual["external_prior_source_note"] = manual.get(
        "source_note",
        pd.Series(pd.NA, index=manual.index),
    )
    manual["external_minutes"] = pd.to_numeric(
        manual.get("minutes"),
        errors="coerce",
    )
    manual["external_appearances"] = pd.to_numeric(
        manual.get("matches"),
        errors="coerce",
    )
    manual["external_starts"] = pd.to_numeric(
        manual.get("starts"),
        errors="coerce",
    )
    manual["external_start_share"] = (
        manual["external_starts"]
        / manual["external_appearances"]
    ).where(manual["external_appearances"].gt(0))
    manual["external_minutes_per_appearance"] = (
        manual["external_minutes"]
        / manual["external_appearances"]
    ).where(manual["external_appearances"].gt(0))

    source_to_output = {
        "xg_per90": "external_xg_per90",
        "xa_per90": "external_xa_per90",
        "saves_per90": "external_saves_per90",
        "goals_per90": "external_goals_per90",
        "assists_per90": "external_assists_per90",
        "pl_equivalent_xg_per90": (
            "external_pl_equivalent_xg_per90"
        ),
        "pl_equivalent_xa_per90": (
            "external_pl_equivalent_xa_per90"
        ),
        "pl_equivalent_saves_per90": (
            "external_pl_equivalent_saves_per90"
        ),
    }
    for source, output in source_to_output.items():
        manual[output] = pd.to_numeric(
            manual.get(source),
            errors="coerce",
        )

    for raw_column, translated_column in [
        (
            "external_xg_per90",
            "external_pl_equivalent_xg_per90",
        ),
        (
            "external_xa_per90",
            "external_pl_equivalent_xa_per90",
        ),
        (
            "external_saves_per90",
            "external_pl_equivalent_saves_per90",
        ),
    ]:
        manual[translated_column] = manual[
            translated_column
        ].fillna(manual[raw_column])

    manual["external_prior_reliability"] = (
        manual["external_minutes"]
        .div(EXTERNAL_FULL_RELIABILITY_MINUTES)
        .clip(0.0, 1.0)
    )
    manual["external_prior_non_pl"] = True
    manual["external_role_score"] = (
        0.65
        * manual["external_start_share"].clip(0.0, 1.0)
        + 0.35
        * manual[
            "external_minutes_per_appearance"
        ].div(90.0).clip(0.0, 1.0)
    )

    required_output = [
        "external_season",
        "external_name_key",
        "external_player",
        "external_position",
        "external_source_leagues",
        "external_expected_source",
        "external_prior_source_note",
        "external_minutes",
        "external_appearances",
        "external_starts",
        "external_start_share",
        "external_minutes_per_appearance",
        "external_goals_per90",
        "external_assists_per90",
        "external_xg_per90",
        "external_xa_per90",
        "external_saves_per90",
        "external_pl_equivalent_xg_per90",
        "external_pl_equivalent_xa_per90",
        "external_pl_equivalent_saves_per90",
        "external_prior_reliability",
        "external_prior_non_pl",
        "external_role_score",
    ]
    for column in required_output:
        if column not in manual.columns:
            manual[column] = np.nan
    return manual[required_output]



def load_external_player_priors() -> pd.DataFrame:
    """
    Load previous-season player performance priors.

    FBref provides playing time, role, goals/assists and goalkeeper saves.
    Understat is additionally queried for robust xG/xA player-season totals.
    A versioned processed cache avoids silently reusing the earlier v1.1
    cache in which xG/xA coverage could be empty.
    """
    manual = manual_external_prior_frame()

    if (
        EXTERNAL_PRIOR_CACHE_PATH.exists()
        and not EXTERNAL_PRIOR_REFRESH
    ):
        automatic = pd.read_csv(
            EXTERNAL_PRIOR_CACHE_PATH,
            compression="gzip",
        )
        automatic["external_prior_non_pl"] = (
            automatic["external_prior_non_pl"]
            .astype("string")
            .str.lower()
            .map({"true": True, "false": False})
            .fillna(False)
            .astype(bool)
        )
    elif not EXTERNAL_PRIORS_ENABLED:
        automatic = pd.DataFrame()
    else:
        try:
            import soccerdata as sd

            prior_season_labels = sorted({
                previous_season_label(season)
                for season in HISTORICAL_SEASONS + [CURRENT_SEASON]
            })
            soccerdata_seasons = [
                season.split("-")[0]
                for season in prior_season_labels
            ]
            leagues = (
                EXTERNAL_FBREF_LEAGUES
                + EXTERNAL_FBREF_EXTRA_LEAGUES
            )

            fbref = sd.FBref(
                leagues=leagues,
                seasons=soccerdata_seasons,
                data_dir=EXTERNAL_SOCCERDATA_CACHE_DIR,
            )

            standard_raw = fbref.read_player_season_stats(
                stat_type="standard"
            )
            shooting_raw = fbref.read_player_season_stats(
                stat_type="shooting"
            )
            try:
                keeper_raw = fbref.read_player_season_stats(
                    stat_type="keeper"
                )
            except Exception as keeper_exc:
                warnings.warn(
                    "FBref goalkeeper prior data could not be loaded; "
                    f"goalkeeper saves will use model priors. {keeper_exc}"
                )
                keeper_raw = pd.DataFrame()

            standard = tidy_fbref_standard(standard_raw)
            shooting = tidy_fbref_shooting(shooting_raw)
            keeper = (
                tidy_fbref_keeper(keeper_raw)
                if not keeper_raw.empty
                else pd.DataFrame(
                    columns=[
                        "league",
                        "season",
                        "team",
                        "player",
                        "external_saves",
                    ]
                )
            )

            expected_stats = pd.DataFrame()
            if EXTERNAL_UNDERSTAT_ENABLED:
                try:
                    available_understat = set(
                        sd.Understat.available_leagues()
                    )
                    understat_leagues = [
                        league
                        for league in leagues
                        if league in available_understat
                    ]
                    if understat_leagues:
                        understat = sd.Understat(
                            leagues=understat_leagues,
                            seasons=soccerdata_seasons,
                            data_dir=EXTERNAL_UNDERSTAT_CACHE_DIR,
                        )
                        understat_raw = (
                            understat.read_player_season_stats()
                        )
                        expected_stats = (
                            tidy_understat_player_season(
                                understat_raw
                            )
                        )
                except Exception as understat_exc:
                    warnings.warn(
                        "Understat xG/xA prior data could not be loaded. "
                        "FBref expected-stat fields will be used as fallback. "
                        f"{understat_exc}"
                    )

            automatic = build_external_player_prior_table(
                standard,
                shooting,
                keeper,
                expected_stats=expected_stats,
            )

            EXTERNAL_PRIOR_CACHE_PATH.parent.mkdir(
                parents=True,
                exist_ok=True,
            )
            automatic.to_csv(
                EXTERNAL_PRIOR_CACHE_PATH,
                index=False,
                compression="gzip",
            )
        except Exception as exc:
            if EXTERNAL_PRIOR_STRICT:
                raise
            warnings.warn(
                "External player-prior ingestion failed. "
                "The notebook will fall back to FPL-only priors unless "
                "manual rows are supplied. "
                f"Reason: {exc}"
            )
            automatic = pd.DataFrame()

    frames = [
        frame
        for frame in [automatic, manual]
        if frame is not None and not frame.empty
    ]
    if not frames:
        return pd.DataFrame()

    priors = pd.concat(frames, ignore_index=True, sort=False)

    priors = (
        priors
        .drop_duplicates(
            [
                "external_season",
                "external_name_key",
                "external_position",
            ],
            keep="last",
        )
        .reset_index(drop=True)
    )

    return priors

EXTERNAL_PRIOR_COLUMNS = [
    "external_prior_available",
    "external_prior_match_method",
    "external_prior_match_score",
    "external_prior_season",
    "external_player",
    "external_position",
    "external_source_leagues",
    "external_expected_source",
    "external_prior_source_note",
    "external_prior_non_pl",
    "external_minutes",
    "external_appearances",
    "external_starts",
    "external_start_share",
    "external_minutes_per_appearance",
    "external_role_score",
    "external_goals_per90",
    "external_assists_per90",
    "external_xg_per90",
    "external_xa_per90",
    "external_saves_per90",
    "external_pl_equivalent_xg_per90",
    "external_pl_equivalent_xa_per90",
    "external_pl_equivalent_saves_per90",
    "external_prior_reliability",
]


def empty_external_prior_columns(
    index: pd.Index,
) -> pd.DataFrame:
    result = pd.DataFrame(index=index)
    for column in EXTERNAL_PRIOR_COLUMNS:
        if column in {
            "external_prior_match_method",
            "external_prior_season",
            "external_player",
            "external_position",
            "external_source_leagues",
            "external_expected_source",
            "external_prior_source_note",
        }:
            result[column] = pd.NA
        elif column == "external_prior_non_pl":
            result[column] = False
        elif column == "external_prior_available":
            result[column] = 0.0
        else:
            result[column] = np.nan
    return result


def _name_similarity(left: str, right: str) -> float:
    return SequenceMatcher(None, left, right).ratio()


def _fuzzy_external_match(
    target_key: str,
    target_position: str | None,
    candidates: pd.DataFrame,
) -> tuple[pd.Series | None, float]:
    if candidates.empty or not target_key:
        return None, 0.0

    compatible = candidates
    if target_position:
        same_position = candidates.loc[
            candidates["external_position"].eq(
                target_position
            )
            | candidates["external_position"].isna()
        ]
        if not same_position.empty:
            compatible = same_position

    scored = compatible.copy()
    scored["_match_score"] = scored[
        "external_name_key"
    ].map(lambda value: _name_similarity(target_key, str(value)))
    scored = scored.sort_values(
        "_match_score",
        ascending=False,
    )
    best = scored.iloc[0]
    best_score = float(best["_match_score"])
    second_score = (
        float(scored.iloc[1]["_match_score"])
        if len(scored) > 1
        else 0.0
    )

    target_tokens = target_key.split()
    best_tokens = str(best["external_name_key"]).split()
    surname_agrees = bool(
        target_tokens
        and best_tokens
        and target_tokens[-1] == best_tokens[-1]
    )

    if (
        best_score >= EXTERNAL_FUZZY_MATCH_THRESHOLD
        and best_score - second_score >= EXTERNAL_FUZZY_MATCH_MARGIN
        and surname_agrees
    ):
        return best.drop(labels=["_match_score"]), best_score

    return None, best_score


def _initial_surname_external_match(
    target_key: str,
    target_position: str | None,
    candidates: pd.DataFrame,
) -> tuple[pd.Series | None, float]:
    """
    Match abbreviations such as "A Khalaili" to "Anan Khalaili" only when
    surname + first initial identify exactly one position-compatible player.
    """
    if candidates.empty or not target_key:
        return None, 0.0

    target_tokens = str(target_key).split()
    if len(target_tokens) < 2:
        return None, 0.0
    target_surname = target_tokens[-1]
    target_initial = target_tokens[0][:1]
    if not target_surname or not target_initial:
        return None, 0.0

    compatible = candidates.copy()
    if target_position:
        same_position = compatible.loc[
            compatible["external_position"].eq(target_position)
            | compatible["external_position"].isna()
        ]
        if not same_position.empty:
            compatible = same_position

    def same_identity(value: object) -> bool:
        tokens = str(value).split()
        return bool(
            len(tokens) >= 2
            and tokens[-1] == target_surname
            and tokens[0][:1] == target_initial
        )

    matched = compatible.loc[
        compatible["external_name_key"].map(same_identity)
    ].copy()
    if len(matched) != 1:
        return None, 0.0

    row = matched.iloc[0]
    score = _name_similarity(target_key, str(row["external_name_key"]))
    if score < float(EXTERNAL_INITIAL_SURNAME_MATCH_THRESHOLD):
        return None, float(score)
    return row, float(score)


def match_players_to_external_priors(
    target_players: pd.DataFrame,
    external_priors: pd.DataFrame,
    season_column: str,
    name_column: str,
    position_column: str,
    key_columns: list[str],
) -> pd.DataFrame:
    """Match FPL players to only the preceding season's external data."""
    selection_columns = list(dict.fromkeys(
        key_columns
        + [season_column, name_column, position_column]
    ))
    target = target_players[
        selection_columns
    ].drop_duplicates(key_columns).copy()

    result = target[key_columns].copy()
    empty = empty_external_prior_columns(target.index)
    result = pd.concat(
        [
            result.reset_index(drop=True),
            empty.reset_index(drop=True),
        ],
        axis=1,
    )

    if external_priors.empty:
        return result

    priors = external_priors.copy()
    priors["external_name_key"] = priors[
        "external_name_key"
    ].astype(str)

    override_lookup = {
        normalise_player_name(key): normalise_player_name(value)
        for key, value in EXTERNAL_NAME_OVERRIDES.items()
    }

    for output_index, row in target.reset_index(drop=True).iterrows():
        prior_season = previous_season_label(
            str(row[season_column])
        )
        target_key = normalise_player_name(row[name_column])
        target_position = _normalise_external_position(
            row[position_column]
        )

        season_candidates = priors.loc[
            priors["external_season"].eq(prior_season)
        ]
        if season_candidates.empty:
            continue

        override_key = override_lookup.get(target_key)
        lookup_key = override_key or target_key

        exact = season_candidates.loc[
            season_candidates["external_name_key"].eq(
                lookup_key
            )
        ]

        matched = None
        method = None
        score = np.nan

        if len(exact) == 1:
            matched = exact.iloc[0]
            method = "manual_name_override" if override_key else "exact_name"
            score = 1.0
        elif len(exact) > 1:
            position_exact = exact.loc[
                exact["external_position"].eq(
                    target_position
                )
            ]
            if len(position_exact) == 1:
                matched = position_exact.iloc[0]
                method = "exact_name_position"
                score = 1.0
        elif override_key is None:
            matched, abbreviated_score = _initial_surname_external_match(
                target_key,
                target_position,
                season_candidates,
            )
            if matched is not None:
                method = "initial_surname_unique"
                score = abbreviated_score
            else:
                matched, fuzzy_score = _fuzzy_external_match(
                    target_key,
                    target_position,
                    season_candidates,
                )
                if matched is not None:
                    method = "fuzzy_name"
                    score = fuzzy_score

        if matched is None:
            continue

        result.loc[
            output_index,
            "external_prior_available",
        ] = 1.0
        result.loc[
            output_index,
            "external_prior_match_method",
        ] = method
        result.loc[
            output_index,
            "external_prior_match_score",
        ] = float(score)
        result.loc[
            output_index,
            "external_prior_season",
        ] = prior_season

        for column in [
            column
            for column in EXTERNAL_PRIOR_COLUMNS
            if column not in {
                "external_prior_available",
                "external_prior_match_method",
                "external_prior_match_score",
                "external_prior_season",
            }
            and column in matched.index
        ]:
            result.loc[output_index, column] = matched[column]

    result["external_prior_available"] = pd.to_numeric(
        result["external_prior_available"],
        errors="coerce",
    ).fillna(0.0)
    result["external_prior_match_score"] = pd.to_numeric(
        result["external_prior_match_score"],
        errors="coerce",
    )
    result["external_prior_reliability"] = pd.to_numeric(
        result["external_prior_reliability"],
        errors="coerce",
    ).clip(0.0, 1.0)

    return result


def attach_external_priors_to_historical(
    historical_model: pd.DataFrame,
    external_priors: pd.DataFrame,
) -> pd.DataFrame:
    """Attach only prior-season external data to each historical FPL player."""
    player_rows = historical_model[
        ["season", "player_key", "player_name", "position"]
    ].drop_duplicates(
        ["season", "player_key"]
    )

    matches = match_players_to_external_priors(
        player_rows,
        external_priors,
        season_column="season",
        name_column="player_name",
        position_column="position",
        key_columns=["season", "player_key"],
    )

    enriched = historical_model.merge(
        matches,
        on=["season", "player_key"],
        how="left",
        validate="many_to_one",
    )

    for column in EXTERNAL_PRIOR_COLUMNS:
        if column not in enriched.columns:
            enriched[column] = (
                0.0
                if column == "external_prior_available"
                else False
                if column == "external_prior_non_pl"
                else np.nan
            )

    matched = enriched["external_prior_available"].eq(1.0)
    if matched.any():
        prior_start = pd.to_numeric(
            enriched.loc[
                matched,
                "external_prior_season",
            ].astype(str).str.slice(0, 4),
            errors="coerce",
        )
        current_start = pd.to_numeric(
            enriched.loc[
                matched,
                "season",
            ].astype(str).str.slice(0, 4),
            errors="coerce",
        )
        assert prior_start.lt(current_start).all(), (
            "External prior leakage detected: a prior is not from an "
            "earlier season."
        )

    return enriched


def build_current_external_player_priors(
    current_players: pd.DataFrame,
    external_priors: pd.DataFrame,
) -> pd.DataFrame:
    current_rows = current_players[
        ["player_key", "full_name", "position"]
    ].drop_duplicates("player_key").copy()
    current_rows["season"] = CURRENT_SEASON

    return match_players_to_external_priors(
        current_rows,
        external_priors,
        season_column="season",
        name_column="full_name",
        position_column="position",
        key_columns=["player_key"],
    )


def external_new_player_weight(
    feature_data: pd.DataFrame,
    maximum_weight: float,
) -> pd.Series:
    """
    Weight the previous-season external prior for new/returning players.

    The explicit blend is reserved for players whose most recent prior came
    from outside the Premier League, plus genuinely sparse-PL-history players.
    It fades with CURRENT-season Premier League evidence rather than lifetime
    PL appearances. This means a returning former PL player can still benefit
    from last season's overseas data, while an established PL player is not
    double-counted simply because a new season has begun.
    """
    current_season_fixtures = pd.to_numeric(
        feature_data.get(
            "prior_season_fixture_count",
            pd.Series(0.0, index=feature_data.index),
        ),
        errors="coerce",
    ).fillna(0.0)
    career_pl_fixtures = pd.to_numeric(
        feature_data.get(
            "prior_career_fixture_count",
            pd.Series(0.0, index=feature_data.index),
        ),
        errors="coerce",
    ).fillna(0.0)

    evidence_fade = (
        1.0
        - current_season_fixtures.div(
            NEW_PLAYER_PRIOR_FADE_OUT_PL_FIXTURES
        )
    ).clip(0.0, 1.0)

    non_pl_prior = (
        feature_data.get(
            "external_prior_non_pl",
            pd.Series(False, index=feature_data.index),
        )
        .fillna(False)
        .astype(bool)
    )
    sparse_pl_history = career_pl_fixtures.lt(
        NEW_PLAYER_PRIOR_FADE_OUT_PL_FIXTURES
    )
    relevant_player = (
        non_pl_prior | sparse_pl_history
    ).astype(float)

    reliability = pd.to_numeric(
        feature_data.get(
            "external_prior_reliability",
            pd.Series(0.0, index=feature_data.index),
        ),
        errors="coerce",
    ).fillna(0.0).clip(0.0, 1.0)
    match_score = pd.to_numeric(
        feature_data.get(
            "external_prior_match_score",
            pd.Series(0.0, index=feature_data.index),
        ),
        errors="coerce",
    ).fillna(0.0).clip(0.0, 1.0)
    available = pd.to_numeric(
        feature_data.get(
            "external_prior_available",
            pd.Series(0.0, index=feature_data.index),
        ),
        errors="coerce",
    ).fillna(0.0).clip(0.0, 1.0)

    return (
        float(maximum_weight)
        * evidence_fade
        * relevant_player
        * reliability
        * match_score
        * available
    ).clip(0.0, float(maximum_weight))


external_player_priors = load_external_player_priors()
current_external_player_priors = (
    build_current_external_player_priors(
        players,
        external_player_priors,
    )
)

print(
    "External prior rows:",
    f"{len(external_player_priors):,}",
)
print(
    "Current FPL players matched to a previous-season external prior:",
    f"{int(current_external_player_priors['external_prior_available'].sum()):,}"
)


EXTERNAL_MINUTES_MODEL_FEATURES = [
    "external_prior_available",
    "external_prior_reliability",
    "external_start_share",
    "external_minutes_per_appearance",
    "external_role_score",
]

EXTERNAL_EVENT_MODEL_FEATURES = [
    "external_prior_available",
    "external_prior_reliability",
    "external_pl_equivalent_xg_per90",
    "external_pl_equivalent_xa_per90",
    "external_pl_equivalent_saves_per90",
]

print("External prior source coverage by season")
if not external_player_priors.empty:
    display(
        external_player_priors.groupby("external_season")
        .agg(
            players=("external_name_key", "nunique"),
            non_pl=(
                "external_prior_non_pl",
                lambda values: int(pd.Series(values).fillna(False).sum()),
            ),
        )
        .reset_index()
        .sort_values("external_season")
    )
else:
    print(
        "No automatic external priors are currently available. "
        "FPL-only priors will be used until the cache/source/manual rows are available."
    )


In [ ]:
def external_expected_stat_coverage(
    priors: pd.DataFrame,
) -> pd.DataFrame:
    """Audit whether new-signing xG/xA/save priors are genuinely populated."""
    if priors is None or priors.empty:
        return pd.DataFrame()

    rows = []
    for position in ["GK", "DEF", "MID", "FWD"]:
        subset = priors.loc[
            priors["external_position"].eq(position)
        ].copy()
        if subset.empty:
            continue
        rows.append({
            "position": position,
            "players": len(subset),
            "xg_raw_coverage": float(
                subset["external_xg_per90"].notna().mean()
            ),
            "xa_raw_coverage": float(
                subset["external_xa_per90"].notna().mean()
            ),
            "xg_pl_equivalent_coverage": float(
                subset[
                    "external_pl_equivalent_xg_per90"
                ].notna().mean()
            ),
            "xa_pl_equivalent_coverage": float(
                subset[
                    "external_pl_equivalent_xa_per90"
                ].notna().mean()
            ),
            "saves_pl_equivalent_coverage": float(
                subset[
                    "external_pl_equivalent_saves_per90"
                ].notna().mean()
            ),
        })
    return pd.DataFrame(rows)


external_expected_stat_coverage_table = (
    external_expected_stat_coverage(
        external_player_priors
    )
)

print("External expected-stat coverage")
display(external_expected_stat_coverage_table)

if not external_expected_stat_coverage_table.empty:
    outfield = external_expected_stat_coverage_table.loc[
        external_expected_stat_coverage_table[
            "position"
        ].isin(["DEF", "MID", "FWD"])
    ]
    xg_ok = (
        outfield["xg_pl_equivalent_coverage"].mean()
        if not outfield.empty
        else np.nan
    )
    xa_ok = (
        outfield["xa_pl_equivalent_coverage"].mean()
        if not outfield.empty
        else np.nan
    )
    if pd.notna(xg_ok) and xg_ok < 0.25:
        warnings.warn(
            "External xG coverage is still low. Check Understat/FBref "
            "ingestion before relying heavily on new-signing attacking priors."
        )
    if pd.notna(xa_ok) and xa_ok < 0.25:
        warnings.warn(
            "External xA coverage is still low. Check Understat/FBref "
            "ingestion before relying heavily on new-signing attacking priors."
        )


### Download and normalise completed seasons

For each season, the notebook downloads only three files:

- `gws/merged_gw.csv`: one player–fixture outcome row;
- `players_raw.csv`: stable player codes and position metadata;
- `teams.csv`: team IDs, codes and names.

They are combined into one local compressed cache, `data/historical_fpl.csv.gz`. The source files do not need to be
kept separately.

### Archive-quality handling

The merged historical files are external archives rather than formally versioned tables. The loader therefore removes rows without a fixture ID and resolves repeated player–fixture records by preferring completed, more complete match records. It still validates uniqueness after cleaning and reports any unresolved records with a diagnostic sample.


In [ ]:
def get_remote_csv(url: str) -> pd.DataFrame:
    """Download a CSV with the shared HTTP session."""
    try:
        response = session.get(url, timeout=90)
        response.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(
            f"Unable to download historical FPL data from {url}"
        ) from exc

    return pd.read_csv(
        StringIO(response.text),
        low_memory=False,
    )


def normalise_boolean(series: pd.Series) -> pd.Series:
    """Convert common boolean encodings into pandas' nullable Boolean type."""
    mapping = {
        True: True,
        False: False,
        1: True,
        0: False,
        "1": True,
        "0": False,
        "True": True,
        "False": False,
        "true": True,
        "false": False,
    }
    return series.map(mapping).astype("boolean")


def resolve_historical_player_fixture_duplicates(
    gameweeks: pd.DataFrame,
    season: str,
) -> pd.DataFrame:
    """
    Return one reliable row per player and Premier League fixture.

    Some archived merged Gameweek files contain rows with no fixture and,
    occasionally, repeated records for the same player–fixture pair. Rows
    without a fixture are not valid observations for a fixture-level model.

    For repeated fixture records, prefer rows that contain completed-match
    scores and the greatest amount of recorded match data. Source order is
    used only as the final tie-breaker because rescheduled fixtures can appear
    again in a later Gameweek extract.
    """
    data = gameweeks.copy()

    data["player_id_season"] = pd.to_numeric(
        data["player_id_season"],
        errors="coerce",
    ).astype("Int64")
    data["fixture_id"] = pd.to_numeric(
        data["fixture_id"],
        errors="coerce",
    ).astype("Int64")

    missing_fixture = data["fixture_id"].isna()
    if missing_fixture.any():
        print(
            f"Season {season}: excluding "
            f"{int(missing_fixture.sum()):,} rows without a fixture ID."
        )
        data = data.loc[~missing_fixture].copy()

    key_columns = ["player_id_season", "fixture_id"]
    duplicate_mask = data.duplicated(key_columns, keep=False)

    if not duplicate_mask.any():
        return data.reset_index(drop=True)

    duplicate_groups = (
        data.loc[duplicate_mask, key_columns]
        .drop_duplicates()
        .shape[0]
    )

    data["_source_order"] = np.arange(len(data))

    if {"team_h_score", "team_a_score"}.issubset(data.columns):
        data["_completed_fixture"] = (
            data["team_h_score"].notna()
            & data["team_a_score"].notna()
        )
    else:
        data["_completed_fixture"] = False

    outcome_columns = [
        column
        for column in [
            "kickoff_time",
            "team_h_score",
            "team_a_score",
            "total_points",
            "minutes",
            "starts",
            "goals_scored",
            "assists",
            "clean_sheets",
            "goals_conceded",
            "saves",
            "bonus",
            "bps",
            "expected_goals",
            "expected_assists",
            "expected_goal_involvements",
            "expected_goals_conceded",
        ]
        if column in data.columns
    ]

    data["_outcome_fields_present"] = (
        data[outcome_columns].notna().sum(axis=1)
        if outcome_columns
        else 0
    )

    # Prefer the most complete completed-match record. The later source row
    # is only a final tie-breaker for otherwise equivalent archive records.
    data = (
        data
        .sort_values(
            key_columns
            + [
                "_completed_fixture",
                "_outcome_fields_present",
                "_source_order",
            ],
            ascending=[True, True, False, False, False],
        )
        .drop_duplicates(key_columns, keep="first")
        .drop(
            columns=[
                "_source_order",
                "_completed_fixture",
                "_outcome_fields_present",
            ]
        )
        .reset_index(drop=True)
    )

    if data.duplicated(key_columns).any():
        raise ValueError(
            f"Season {season} still contains ambiguous duplicate "
            "player–fixture rows after archive cleaning."
        )

    print(
        f"Season {season}: resolved "
        f"{duplicate_groups:,} duplicated player–fixture groups."
    )

    return data


def load_historical_season(season: str) -> pd.DataFrame:
    """
    Download and normalise one completed FPL season.

    The output is one row per player per Premier League fixture.
    """
    season_root = f"{HISTORICAL_GITHUB_ROOT}/{season}"

    gameweeks = get_remote_csv(
        f"{season_root}/gws/merged_gw.csv"
    )
    player_metadata = get_remote_csv(
        f"{season_root}/players_raw.csv"
    )
    team_metadata = get_remote_csv(
        f"{season_root}/teams.csv"
    )

    gameweeks = gameweeks.rename(columns={
        "element": "player_id_season",
        "fixture": "fixture_id",
        "opponent_team": "opponent_id",
    })

    if "GW" in gameweeks.columns:
        gameweeks["gameweek"] = pd.to_numeric(
            gameweeks["GW"],
            errors="coerce",
        )
    elif "round" in gameweeks.columns:
        gameweeks["gameweek"] = pd.to_numeric(
            gameweeks["round"],
            errors="coerce",
        )
    else:
        raise KeyError(
            f"No Gameweek column was found for season {season}."
        )

    required_gameweek_columns = {
        "player_id_season",
        "fixture_id",
        "opponent_id",
        "gameweek",
    }
    missing_gameweek_columns = (
        required_gameweek_columns - set(gameweeks.columns)
    )
    if missing_gameweek_columns:
        raise KeyError(
            f"Season {season} is missing required columns: "
            f"{sorted(missing_gameweek_columns)}"
        )

    gameweeks = resolve_historical_player_fixture_duplicates(
        gameweeks,
        season,
    )

    player_lookup_columns = [
        column
        for column in [
            "id",
            "code",
            "element_type",
            "team",
            "web_name",
            "first_name",
            "second_name",
        ]
        if column in player_metadata.columns
    ]

    player_lookup = (
        player_metadata[player_lookup_columns]
        .rename(columns={
            "id": "player_id_season",
            "code": "player_code",
            "element_type": "position_id_metadata",
            "team": "team_id_end_season",
            "web_name": "web_name_metadata",
            "first_name": "first_name_metadata",
            "second_name": "second_name_metadata",
        })
        .drop_duplicates("player_id_season")
    )

    gameweeks = gameweeks.merge(
        player_lookup,
        on="player_id_season",
        how="left",
        validate="many_to_one",
    )

    team_lookup_columns = [
        column
        for column in ["id", "code", "name", "short_name"]
        if column in team_metadata.columns
    ]

    team_lookup = (
        team_metadata[team_lookup_columns]
        .rename(columns={
            "id": "team_id",
            "code": "team_code",
            "name": "team_name",
            "short_name": "team_short_name",
        })
        .drop_duplicates("team_id")
    )

    # The historical GW file contains the player's team name at that
    # particular Gameweek. This is safer than using the end-of-season team
    # from players_raw.csv for players who transferred during the season.
    if "team" in gameweeks.columns:
        if pd.api.types.is_numeric_dtype(gameweeks["team"]):
            gameweeks = gameweeks.merge(
                team_lookup,
                left_on="team",
                right_on="team_id",
                how="left",
                validate="many_to_one",
            )
        else:
            gameweeks = gameweeks.merge(
                team_lookup,
                left_on="team",
                right_on="team_name",
                how="left",
                validate="many_to_one",
            )
    else:
        gameweeks = gameweeks.merge(
            team_lookup,
            left_on="team_id_end_season",
            right_on="team_id",
            how="left",
            validate="many_to_one",
        )

    # Fill rare unmapped team rows from the end-of-season player metadata.
    fallback_team_lookup = team_lookup.set_index("team_id")
    for column in [
        "team_code",
        "team_name",
        "team_short_name",
    ]:
        if column not in gameweeks.columns:
            gameweeks[column] = pd.NA
        gameweeks[column] = gameweeks[column].fillna(
            gameweeks["team_id_end_season"].map(
                fallback_team_lookup[column]
            )
        )

    if "team_id" not in gameweeks.columns:
        gameweeks["team_id"] = gameweeks["team_id_end_season"]
    else:
        gameweeks["team_id"] = gameweeks["team_id"].fillna(
            gameweeks["team_id_end_season"]
        )

    opponent_lookup = team_lookup.rename(columns={
        "team_id": "opponent_id",
        "team_code": "opponent_code",
        "team_name": "opponent_name",
        "team_short_name": "opponent_short_name",
    })

    gameweeks = gameweeks.merge(
        opponent_lookup,
        on="opponent_id",
        how="left",
        validate="many_to_one",
    )

    gameweeks["season"] = season
    gameweeks["season_start_year"] = int(season[:4])

    gameweeks["kickoff_time"] = pd.to_datetime(
        gameweeks.get("kickoff_time"),
        utc=True,
        errors="coerce",
    )

    if "was_home" in gameweeks.columns:
        gameweeks["was_home"] = normalise_boolean(
            gameweeks["was_home"]
        )
    else:
        gameweeks["was_home"] = pd.Series(
            pd.NA,
            index=gameweeks.index,
            dtype="boolean",
        )

    if "name" in gameweeks.columns:
        gameweeks["player_name"] = gameweeks["name"]
    else:
        full_name = (
            gameweeks.get(
                "first_name_metadata",
                pd.Series("", index=gameweeks.index),
            )
            .fillna("")
            .astype(str)
            .str.strip()
            + " "
            + gameweeks.get(
                "second_name_metadata",
                pd.Series("", index=gameweeks.index),
            )
            .fillna("")
            .astype(str)
            .str.strip()
        ).str.strip()

        gameweeks["player_name"] = (
            gameweeks.get(
                "web_name_metadata",
                pd.Series(pd.NA, index=gameweeks.index),
            )
            .fillna(full_name)
        )

    if "position" not in gameweeks.columns:
        gameweeks["position"] = pd.NA

    gameweeks["position"] = gameweeks["position"].fillna(
        pd.to_numeric(
            gameweeks.get("position_id_metadata"),
            errors="coerce",
        ).map(POSITION_ID_TO_NAME)
    )

    gameweeks["player_code"] = pd.to_numeric(
        gameweeks.get("player_code"),
        errors="coerce",
    ).astype("Int64")

    gameweeks["player_id_season"] = pd.to_numeric(
        gameweeks["player_id_season"],
        errors="coerce",
    ).astype("Int64")

    # Season-specific fallback is used only when a stable FPL player code
    # is unavailable.
    gameweeks["player_key"] = (
        season
        + ":element:"
        + gameweeks["player_id_season"].astype(str)
    )

    has_player_code = gameweeks["player_code"].notna()
    gameweeks.loc[has_player_code, "player_key"] = (
        "code:"
        + gameweeks.loc[
            has_player_code,
            "player_code",
        ].astype(str)
    )

    gameweeks["team_code"] = pd.to_numeric(
        gameweeks.get("team_code"),
        errors="coerce",
    ).astype("Int64")

    gameweeks["opponent_code"] = pd.to_numeric(
        gameweeks.get("opponent_code"),
        errors="coerce",
    ).astype("Int64")

    gameweeks["team_key"] = (
        season
        + ":team:"
        + pd.to_numeric(
            gameweeks["team_id"],
            errors="coerce",
        )
        .astype("Int64")
        .astype(str)
    )

    has_team_code = gameweeks["team_code"].notna()
    gameweeks.loc[has_team_code, "team_key"] = (
        "code:"
        + gameweeks.loc[
            has_team_code,
            "team_code",
        ].astype(str)
    )

    # Archived xP is deliberately removed because it may contain
    # post-match information according to the source repository.
    gameweeks = gameweeks.drop(
        columns=["xP"],
        errors="ignore",
    )

    numeric_columns = [
        "gameweek",
        "fixture_id",
        "opponent_id",
        "team_id",
        "total_points",
        "minutes",
        "starts",
        "goals_scored",
        "assists",
        "clean_sheets",
        "goals_conceded",
        "own_goals",
        "penalties_saved",
        "penalties_missed",
        "yellow_cards",
        "red_cards",
        "saves",
        "bonus",
        "bps",
        "influence",
        "creativity",
        "threat",
        "ict_index",
        "expected_goals",
        "expected_assists",
        "expected_goal_involvements",
        "expected_goals_conceded",
        "tackles",
        "recoveries",
        "clearances_blocks_interceptions",
        "defensive_contribution",
        "value",
        "selected",
        "transfers_in",
        "transfers_out",
        "transfers_balance",
        "team_h_score",
        "team_a_score",
    ]

    for column in numeric_columns:
        if column in gameweeks.columns:
            gameweeks[column] = pd.to_numeric(
                gameweeks[column],
                errors="coerce",
            )

    if "value" in gameweeks.columns:
        gameweeks["price_at_collection"] = (
            gameweeks["value"] / 10
        )
    else:
        gameweeks["price_at_collection"] = np.nan

    if {
        "team_h_score",
        "team_a_score",
    }.issubset(gameweeks.columns):
        gameweeks["team_goals"] = np.where(
            gameweeks["was_home"].eq(True),
            gameweeks["team_h_score"],
            gameweeks["team_a_score"],
        )
        gameweeks["opponent_goals"] = np.where(
            gameweeks["was_home"].eq(True),
            gameweeks["team_a_score"],
            gameweeks["team_h_score"],
        )

    preferred_columns = [
        "season",
        "season_start_year",
        "gameweek",
        "fixture_id",
        "kickoff_time",
        "player_key",
        "player_code",
        "player_id_season",
        "player_name",
        "position",
        "team_key",
        "team_id",
        "team_code",
        "team_name",
        "team_short_name",
        "opponent_id",
        "opponent_code",
        "opponent_name",
        "opponent_short_name",
        "was_home",
        "price_at_collection",
        "selected",
        "transfers_in",
        "transfers_out",
        "transfers_balance",
        "total_points",
        "minutes",
        "starts",
        "goals_scored",
        "assists",
        "clean_sheets",
        "goals_conceded",
        "own_goals",
        "penalties_saved",
        "penalties_missed",
        "yellow_cards",
        "red_cards",
        "saves",
        "bonus",
        "bps",
        "influence",
        "creativity",
        "threat",
        "ict_index",
        "expected_goals",
        "expected_assists",
        "expected_goal_involvements",
        "expected_goals_conceded",
        "tackles",
        "recoveries",
        "clearances_blocks_interceptions",
        "defensive_contribution",
        "team_h_score",
        "team_a_score",
        "team_goals",
        "opponent_goals",
    ]

    available_columns = [
        column
        for column in preferred_columns
        if column in gameweeks.columns
    ]

    season_data = (
        gameweeks[available_columns]
        .copy()
        .sort_values(
            [
                "kickoff_time",
                "fixture_id",
                "player_key",
            ]
        )
        .reset_index(drop=True)
    )

    duplicate_key = [
        "season",
        "player_id_season",
        "fixture_id",
    ]
    duplicates = season_data.duplicated(
        duplicate_key,
        keep=False,
    )
    if duplicates.any():
        duplicate_preview = (
            season_data.loc[
                duplicates,
                [
                    column
                    for column in [
                        "season",
                        "gameweek",
                        "fixture_id",
                        "player_id_season",
                        "player_name",
                        "team_name",
                        "opponent_name",
                        "kickoff_time",
                        "minutes",
                        "total_points",
                    ]
                    if column in season_data.columns
                ],
            ]
            .head(10)
            .to_dict("records")
        )
        raise ValueError(
            f"Season {season} contains unresolved duplicate "
            f"player–fixture rows. Sample: {duplicate_preview}"
        )

    return season_data


def load_historical_fpl_data(
    seasons: list[str],
    cache_path: Path = HISTORICAL_CACHE_PATH,
    refresh: bool = REFRESH_HISTORICAL_CACHE,
) -> pd.DataFrame:
    """Load the requested seasons from one cache or rebuild it."""
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    if cache_path.exists() and not refresh:
        cached = pd.read_csv(
            cache_path,
            compression="gzip",
            low_memory=False,
        )
        cached["kickoff_time"] = pd.to_datetime(
            cached["kickoff_time"],
            utc=True,
            errors="coerce",
        )

        cached_seasons = set(cached["season"].dropna())
        requested_seasons = set(seasons)

        if requested_seasons.issubset(cached_seasons):
            print(
                "Loaded historical data from "
                f"{cache_path}."
            )
            return (
                cached.loc[cached["season"].isin(seasons)]
                .copy()
                .reset_index(drop=True)
            )

        print(
            "The historical cache does not contain every requested "
            "season, so it will be rebuilt."
        )

    season_frames = []

    for season in seasons:
        print(f"Downloading historical season {season}...")
        season_frames.append(
            load_historical_season(season)
        )

    historical = (
        pd.concat(
            season_frames,
            ignore_index=True,
            sort=False,
        )
        .sort_values(
            [
                "kickoff_time",
                "fixture_id",
                "player_key",
            ]
        )
        .reset_index(drop=True)
    )

    historical.to_csv(
        cache_path,
        index=False,
        compression="gzip",
    )

    print(
        f"Saved {len(historical):,} historical rows to "
        f"{cache_path}."
    )

    return historical

In [ ]:
historical_fpl = load_historical_fpl_data(
    HISTORICAL_SEASONS
)

print(f"Historical player–fixture rows: {len(historical_fpl):,}")
print(
    "Historical players: "
    f"{historical_fpl['player_key'].nunique():,}"
)
print(
    "Historical fixtures: "
    f"{historical_fpl[['season', 'fixture_id']].drop_duplicates().shape[0]:,}"
)

### Historical-data audit

The table below checks schema coverage before any model is trained. A missing value does not necessarily mean the row is
unusable: some statistics were introduced by FPL only in later seasons.

In [ ]:
def non_missing_rate(
    data: pd.DataFrame,
    column: str,
) -> float:
    if column not in data.columns:
        return 0.0
    return float(data[column].notna().mean())


historical_summary_rows = []

for season, season_data in historical_fpl.groupby("season"):
    historical_summary_rows.append({
        "season": season,
        "gameweeks": season_data["gameweek"].nunique(),
        "fixtures": season_data["fixture_id"].nunique(),
        "players": season_data["player_key"].nunique(),
        "player_fixture_rows": len(season_data),
        "starts_coverage": non_missing_rate(
            season_data,
            "starts",
        ),
        "xg_coverage": non_missing_rate(
            season_data,
            "expected_goals",
        ),
        "xa_coverage": non_missing_rate(
            season_data,
            "expected_assists",
        ),
        "defensive_data_coverage": non_missing_rate(
            season_data,
            "defensive_contribution",
        ),
    })

historical_summary = pd.DataFrame(
    historical_summary_rows
).sort_values("season")

display(historical_summary)

assert "xP" not in historical_fpl.columns, (
    "The archived xP field must not enter the modelling table."
)

assert not historical_fpl.duplicated(
    [
        "season",
        "player_id_season",
        "fixture_id",
    ]
).any(), (
    "Historical data contains duplicate player–fixture rows."
)

assert historical_fpl["kickoff_time"].notna().all(), (
    "Some historical player–fixture rows have no kickoff time."
)

### Save one pre-deadline archive for the live season

Historical archives tell us what happened. For exact future backtests, we also need to preserve what the official API
showed **before** each deadline.

The function below stores all weekly snapshots inside one compressed JSON file. It refuses to save after the relevant
deadline, preventing accidental post-deadline contamination.

In [ ]:


def save_predeadline_snapshot(
    season: str,
    gameweek: int,
    bootstrap_data: dict,
    fixture_data: list,
    event_table: pd.DataFrame,
    path: Path = LIVE_SNAPSHOT_PATH,
) -> bool:
    """Save or refresh one Gameweek's snapshot before its deadline."""
    event_row = event_table.loc[
        event_table["id"].eq(gameweek)
    ]

    if len(event_row) != 1:
        raise ValueError(
            f"Could not identify Gameweek {gameweek} uniquely."
        )

    deadline = pd.to_datetime(
        event_row.iloc[0]["deadline_time"],
        utc=True,
        errors="coerce",
    )

    if pd.isna(deadline):
        raise ValueError(
            f"Gameweek {gameweek} has no valid deadline."
        )

    captured_at = datetime.now(timezone.utc)

    if captured_at >= deadline.to_pydatetime():
        print(
            "Snapshot not saved: the Gameweek "
            f"{gameweek} deadline has already passed."
        )
        return False

    path.parent.mkdir(parents=True, exist_ok=True)

    if path.exists():
        with gzip.open(
            path,
            "rt",
            encoding="utf-8",
        ) as file:
            archive = json.load(file)
    else:
        archive = {
            "format_version": 1,
            "snapshots": {},
        }

    snapshot_key = f"{season}_GW{gameweek:02d}"

    archive["snapshots"][snapshot_key] = {
        "season": season,
        "gameweek": gameweek,
        "captured_at_utc": captured_at.isoformat(),
        "deadline_time_utc": deadline.isoformat(),
        "bootstrap": bootstrap_data,
        "fixtures": fixture_data,
    }

    with gzip.open(
        path,
        "wt",
        encoding="utf-8",
    ) as file:
        json.dump(archive, file)

    print(
        f"Saved pre-deadline snapshot {snapshot_key} "
        f"to {path}."
    )

    return True


if SAVE_PREDEADLINE_SNAPSHOT:
    save_predeadline_snapshot(
        season=CURRENT_SEASON,
        gameweek=first_planning_gameweek,
        bootstrap_data=bootstrap,
        fixture_data=fixtures_raw,
        event_table=gameweeks,
    )

### Build leakage-safe player-history features

The historical outcome table is fixture-level, but all rolling player features are calculated from **completed prior
Gameweeks**.

This distinction matters in double Gameweeks. Suppose a player has two fixtures in Gameweek 25. The second fixture must
not use minutes, goals or points from the first fixture because both line-up decisions were made at the same deadline.
The aggregation-and-shift procedure below prevents that leakage.

In [ ]:
ROLLING_GAMEWEEK_WINDOWS = [3, 5, 10]

PLAYER_OUTCOME_COLUMNS = [
    "minutes",
    "starts",
    "total_points",
    "goals_scored",
    "assists",
    "clean_sheets",
    "goals_conceded",
    "saves",
    "bonus",
    "bps",
    "expected_goals",
    "expected_assists",
    "expected_goal_involvements",
    "expected_goals_conceded",
    "influence",
    "creativity",
    "threat",
    "ict_index",
    "yellow_cards",
    "red_cards",
    "tackles",
    "recoveries",
    "clearances_blocks_interceptions",
    "defensive_contribution",
    "defensive_return",
]

MARKET_COLUMNS = [
    "price_at_collection",
    "selected",
    "transfers_in",
    "transfers_out",
    "transfers_balance",
]


def add_historical_player_features(
    historical_data: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Create shifted player-history features and merge them to fixture rows.

    Returns
    -------
    historical_model_data:
        One row per player per fixture with targets and pre-Gameweek
        features.
    player_gameweek_features:
        One row per player per Gameweek containing the intermediate
        aggregated history.
    """
    data = historical_data.copy()

    # Defensive-contribution points are threshold events, not a linear
    # function of the action total. Derive the historical award indicator
    # before aggregating to Gameweek level so all rolling rates remain shifted.
    raw_position = (
        data.get("position", pd.Series(pd.NA, index=data.index))
        .astype("string")
        .replace({"GKP": "GK"})
    )
    tackles = pd.to_numeric(
        data.get(
            "tackles",
            pd.Series(np.nan, index=data.index),
        ),
        errors="coerce",
    )
    recoveries = pd.to_numeric(
        data.get(
            "recoveries",
            pd.Series(np.nan, index=data.index),
        ),
        errors="coerce",
    )
    cbi = pd.to_numeric(
        data.get(
            "clearances_blocks_interceptions",
            pd.Series(np.nan, index=data.index),
        ),
        errors="coerce",
    )
    has_defensive_data = tackles.notna() | recoveries.notna() | cbi.notna()
    defensive_actions = tackles.fillna(0.0) + cbi.fillna(0.0)
    defensive_actions = defensive_actions + np.where(
        raw_position.isin(["MID", "FWD"]),
        recoveries.fillna(0.0),
        0.0,
    )
    defensive_threshold = raw_position.map({
        "DEF": 10.0,
        "MID": 12.0,
        "FWD": 12.0,
    })
    data["defensive_return"] = np.where(
        has_defensive_data & defensive_threshold.notna(),
        defensive_actions.ge(defensive_threshold).astype(float),
        np.nan,
    )

    for column in PLAYER_OUTCOME_COLUMNS + MARKET_COLUMNS:
        if column not in data.columns:
            data[column] = np.nan

    for column in PLAYER_OUTCOME_COLUMNS + MARKET_COLUMNS:
        data[column] = pd.to_numeric(
            data[column],
            errors="coerce",
        )

    data["target_minutes"] = data["minutes"]
    data["target_started"] = data["starts"]
    data["target_appeared"] = (
        data["minutes"].fillna(0).gt(0).astype(int)
    )
    data["target_60_plus"] = (
        data["minutes"].fillna(0).ge(60).astype(int)
    )
    data["target_fpl_points"] = data["total_points"]

    player_gameweek_keys = [
        "season",
        "season_start_year",
        "player_key",
        "gameweek",
    ]

    grouped = data.groupby(
        player_gameweek_keys,
        dropna=False,
        sort=False,
    )

    fixture_counts = (
        grouped["fixture_id"]
        .nunique()
        .rename("gw_fixture_count")
    )

    outcome_sums = (
        grouped[PLAYER_OUTCOME_COLUMNS]
        .sum(min_count=1)
        .add_prefix("gw_")
    )

    derived_counts = (
        grouped[
            [
                "target_appeared",
                "target_60_plus",
            ]
        ]
        .sum(min_count=1)
        .rename(columns={
            "target_appeared": "gw_appearance_count",
            "target_60_plus": "gw_60_plus_count",
        })
    )

    market_last = (
        grouped[MARKET_COLUMNS]
        .last()
        .add_prefix("observed_")
    )

    kickoff_bounds = grouped["kickoff_time"].agg(
        gw_first_kickoff="min",
        gw_last_kickoff="max",
    )

    player_gameweeks = (
        pd.concat(
            [
                fixture_counts,
                outcome_sums,
                derived_counts,
                market_last,
                kickoff_bounds,
            ],
            axis=1,
        )
        .reset_index()
        .sort_values(
            [
                "player_key",
                "season_start_year",
                "gameweek",
            ]
        )
        .reset_index(drop=True)
    )

    player_group = player_gameweeks.groupby(
        "player_key",
        group_keys=False,
        sort=False,
    )

    player_season_group = player_gameweeks.groupby(
        ["season", "player_key"],
        group_keys=False,
        sort=False,
    )

    feature_series: dict[str, pd.Series] = {}

    feature_series["prior_gameweek_count"] = (
        player_group.cumcount()
    )

    prior_season_fixture_count = (
        player_season_group["gw_fixture_count"].cumsum()
        - player_gameweeks["gw_fixture_count"]
    )
    feature_series[
        "prior_season_fixture_count"
    ] = prior_season_fixture_count

    prior_career_fixture_count = (
        player_group["gw_fixture_count"].cumsum()
        - player_gameweeks["gw_fixture_count"]
    )
    feature_series[
        "prior_career_fixture_count"
    ] = prior_career_fixture_count

    last_fixture_count = player_group[
        "gw_fixture_count"
    ].shift(1)

    for source_column, output_name in {
        "gw_minutes": "minutes_per_fixture_last_gw",
        "gw_starts": "start_rate_last_gw",
        "gw_appearance_count": "appearance_rate_last_gw",
        "gw_60_plus_count": "sixty_plus_rate_last_gw",
        "gw_total_points": "points_per_fixture_last_gw",
        "gw_expected_goals": "xg_per_fixture_last_gw",
        "gw_expected_assists": "xa_per_fixture_last_gw",
    }.items():
        feature_series[output_name] = (
            player_group[source_column].shift(1)
            / last_fixture_count
        )

    rate_sources = {
        "gw_minutes": "minutes_per_fixture",
        "gw_starts": "start_rate",
        "gw_appearance_count": "appearance_rate",
        "gw_60_plus_count": "sixty_plus_rate",
        "gw_total_points": "points_per_fixture",
        "gw_goals_scored": "goals_per_fixture",
        "gw_assists": "assists_per_fixture",
        "gw_clean_sheets": "clean_sheets_per_fixture",
        "gw_goals_conceded": "goals_conceded_per_fixture",
        "gw_saves": "saves_per_fixture",
        "gw_bonus": "bonus_per_fixture",
        "gw_bps": "bps_per_fixture",
        "gw_expected_goals": "xg_per_fixture",
        "gw_expected_assists": "xa_per_fixture",
        "gw_expected_goal_involvements": "xgi_per_fixture",
        "gw_expected_goals_conceded": "xgc_per_fixture",
        "gw_influence": "influence_per_fixture",
        "gw_creativity": "creativity_per_fixture",
        "gw_threat": "threat_per_fixture",
        "gw_ict_index": "ict_per_fixture",
        "gw_yellow_cards": "yellow_cards_per_fixture",
        "gw_red_cards": "red_cards_per_fixture",
        "gw_tackles": "tackles_per_fixture",
        "gw_recoveries": "recoveries_per_fixture",
        "gw_clearances_blocks_interceptions": (
            "cbi_per_fixture"
        ),
        "gw_defensive_contribution": (
            "defensive_contribution_per_fixture"
        ),
        "gw_defensive_return": "defensive_return_rate",
    }

    for window in ROLLING_GAMEWEEK_WINDOWS:
        prior_fixture_count = player_group[
            "gw_fixture_count"
        ].transform(
            lambda series: (
                series.shift(1)
                .rolling(
                    window=window,
                    min_periods=1,
                )
                .sum()
            )
        )

        for source_column, feature_name in rate_sources.items():
            prior_sum = player_group[
                source_column
            ].transform(
                lambda series: (
                    series.shift(1)
                    .rolling(
                        window=window,
                        min_periods=1,
                    )
                    .sum()
                )
            )

            feature_series[
                f"{feature_name}_last_{window}_gws"
            ] = prior_sum / prior_fixture_count

    # Career-to-date rates provide a stable fallback when recent windows
    # contain little information.
    prior_career_fixtures = (
        prior_career_fixture_count.replace(0, np.nan)
    )

    for source_column, feature_name in rate_sources.items():
        prior_career_total = (
            player_group[source_column].cumsum()
            - player_gameweeks[source_column]
        )
        feature_series[
            f"career_{feature_name}"
        ] = (
            prior_career_total
            / prior_career_fixtures
        )

    # These are explicitly previous archived observations, not same-row
    # pre-deadline values.
    for column in MARKET_COLUMNS:
        observed_column = f"observed_{column}"
        feature_series[
            f"previous_{column}"
        ] = player_group[observed_column].shift(1)

    feature_series[
        "days_since_previous_player_gameweek"
    ] = (
        player_gameweeks["gw_first_kickoff"]
        - player_group["gw_last_kickoff"].shift(1)
    ).dt.total_seconds().div(86_400)

    feature_frame = pd.DataFrame(
        feature_series,
        index=player_gameweeks.index,
    )

    player_gameweeks = pd.concat(
        [
            player_gameweeks,
            feature_frame,
        ],
        axis=1,
    )

    feature_columns = list(feature_frame.columns)

    fixture_features = data.merge(
        player_gameweeks[
            player_gameweek_keys + feature_columns
        ],
        on=player_gameweek_keys,
        how="left",
        validate="many_to_one",
    )

    # Premier League rest information is fixture-specific and known before
    # kickoff. It does not yet include cups or European competitions.
    team_schedule = (
        fixture_features[
            [
                "season",
                "team_key",
                "fixture_id",
                "kickoff_time",
            ]
        ]
        .drop_duplicates(
            [
                "season",
                "team_key",
                "fixture_id",
            ]
        )
        .sort_values(
            [
                "team_key",
                "kickoff_time",
                "fixture_id",
            ]
        )
        .reset_index(drop=True)
    )

    team_schedule["previous_league_kickoff"] = (
        team_schedule
        .groupby("team_key")["kickoff_time"]
        .shift(1)
    )

    team_schedule[
        "days_since_previous_league_fixture"
    ] = (
        (
            team_schedule["kickoff_time"]
            - team_schedule["previous_league_kickoff"]
        )
        .dt.total_seconds()
        .div(86_400)
    )

    fixture_features = fixture_features.merge(
        team_schedule[
            [
                "season",
                "team_key",
                "fixture_id",
                "days_since_previous_league_fixture",
            ]
        ],
        on=[
            "season",
            "team_key",
            "fixture_id",
        ],
        how="left",
        validate="many_to_one",
    )

    fixture_features["short_league_rest"] = (
        fixture_features[
            "days_since_previous_league_fixture"
        ].le(4)
    ).astype("boolean")

    return fixture_features, player_gameweeks

In [ ]:
# v2.6: 2022-23 gameweeks 1-15 carry expected_goals/expected_assists of
# exactly 0.0 (not null) for every player -- 4,036 rows containing 370 real
# goals and 331 real assists. Those pass the `.notna()` training filter and are
# learned as genuine zero-xG observations. They must be flagged as missing
# BEFORE add_historical_player_features(), because that function builds the
# shifted rolling/career xG/xA features that every later model consumes.
if V26_ENABLED and V26_APPLY_COVERAGE_MASK:
    historical_fpl = fx.mask_fabricated_event_coverage(
        historical_fpl,
        season_col="season",
        gameweek_col="gameweek",
        minutes_col="minutes",
    )

historical_model_data, historical_player_gameweeks = (
    add_historical_player_features(
        historical_fpl
    )
)

print(
    "Historical modelling rows: "
    f"{len(historical_model_data):,}"
)
print(
    "Player–Gameweek history rows: "
    f"{len(historical_player_gameweeks):,}"
)

historical_preview_columns = [
    "season",
    "gameweek",
    "player_name",
    "team_short_name",
    "opponent_short_name",
    "was_home",
    "target_minutes",
    "target_started",
    "minutes_per_fixture_last_3_gws",
    "start_rate_last_5_gws",
    "xg_per_fixture_last_5_gws",
    "days_since_previous_league_fixture",
]

historical_preview = (
    historical_model_data
    .sort_values(
        [
            "kickoff_time",
            "player_name",
        ]
    )
    .loc[:, historical_preview_columns]
    .head(20)
)

display(historical_preview)


### Attach external priors to historical folds

For every historical FPL season, only external statistics from the immediately preceding season are attached. This lets the walk-forward tests learn how prior-season performance translates into subsequent Premier League minutes and attacking returns without using future information.


In [ ]:

# Attach previous-season external priors to the historical modelling table.
# Every matched row is checked to ensure that the external season predates the
# FPL season being predicted.
historical_model_data = attach_external_priors_to_historical(
    historical_model_data,
    external_player_priors,
)

historical_external_player_audit = (
    historical_model_data[
        [
            "season",
            "player_key",
            "external_prior_available",
            "external_prior_non_pl",
        ]
    ]
    .drop_duplicates(["season", "player_key"])
    .groupby("season", dropna=False)
    .agg(
        players=("player_key", "nunique"),
        matched_external_priors=(
            "external_prior_available",
            "sum",
        ),
        non_pl_external_priors=(
            "external_prior_non_pl",
            "sum",
        ),
    )
    .reset_index()
)

print("Historical external-prior coverage")
display(historical_external_player_audit)


### Leakage audit and chronological holdout

The first modelling experiments will use 2022/23–2024/25 for development and reserve the entire 2025/26 season as an
untouched chronological holdout. Random row splits would allow the same players, teams and adjacent Gameweeks to appear
on both sides of the split and would give an unrealistically optimistic estimate.

After model selection, the final live model can be refitted on all four seasons.

In [ ]:
DEVELOPMENT_SEASONS = [
    "2022-23",
    "2023-24",
    "2024-25",
]
HOLDOUT_SEASON = "2025-26"

TARGET_COLUMNS = [
    "target_minutes",
    "target_started",
    "target_appeared",
    "target_60_plus",
    "target_fpl_points",
]

# These are initial, understandable features. Team-strength and competition
# congestion features will be added before the final minutes model.
MINUTES_FEATURE_COLUMNS = [
    "gameweek",
    "was_home",
    "position",
    "days_since_previous_league_fixture",
    "short_league_rest",
    "prior_season_fixture_count",
    "prior_career_fixture_count",
    "minutes_per_fixture_last_gw",
    "start_rate_last_gw",
    "appearance_rate_last_gw",
    "sixty_plus_rate_last_gw",
    "minutes_per_fixture_last_3_gws",
    "minutes_per_fixture_last_5_gws",
    "minutes_per_fixture_last_10_gws",
    "start_rate_last_3_gws",
    "start_rate_last_5_gws",
    "start_rate_last_10_gws",
    "appearance_rate_last_3_gws",
    "appearance_rate_last_5_gws",
    "appearance_rate_last_10_gws",
    "sixty_plus_rate_last_3_gws",
    "sixty_plus_rate_last_5_gws",
    "sixty_plus_rate_last_10_gws",
    "career_minutes_per_fixture",
    "career_start_rate",
    "career_appearance_rate",
    "career_sixty_plus_rate",
] + EXTERNAL_MINUTES_MODEL_FEATURES

forbidden_model_features = {
    "xP",
    "minutes",
    "starts",
    "total_points",
    "goals_scored",
    "assists",
    "clean_sheets",
    "expected_goals",
    "expected_assists",
    "price_at_collection",
    "selected",
    "transfers_in",
    "transfers_out",
    "transfers_balance",
}

assert not (
    set(MINUTES_FEATURE_COLUMNS)
    & forbidden_model_features
), (
    "A same-row outcome or market field entered the minutes feature list."
)

missing_minutes_features = (
    set(MINUTES_FEATURE_COLUMNS)
    - set(historical_model_data.columns)
)

assert not missing_minutes_features, (
    "The minutes feature table is missing: "
    f"{sorted(missing_minutes_features)}"
)

development_data = (
    historical_model_data.loc[
        historical_model_data["season"].isin(
            DEVELOPMENT_SEASONS
        )
    ]
    .copy()
)

holdout_data = (
    historical_model_data.loc[
        historical_model_data["season"].eq(
            HOLDOUT_SEASON
        )
    ]
    .copy()
)

print(
    f"Development rows: {len(development_data):,} "
    f"({', '.join(DEVELOPMENT_SEASONS)})"
)
print(
    f"Holdout rows: {len(holdout_data):,} "
    f"({HOLDOUT_SEASON})"
)

target_coverage = (
    historical_model_data[
        ["season"] + TARGET_COLUMNS
    ]
    .groupby("season")
    .agg(
        rows=("season", "size"),
        minutes_coverage=(
            "target_minutes",
            lambda values: values.notna().mean(),
        ),
        starts_coverage=(
            "target_started",
            lambda values: values.notna().mean(),
        ),
        points_coverage=(
            "target_fpl_points",
            lambda values: values.notna().mean(),
        ),
    )
    .reset_index()
)

display(target_coverage)

### Result of the historical-data stage

The notebook now has two complementary datasets:

- `player_fixtures`: current players and upcoming fixtures;
- `historical_model_data`: completed historical player–fixture outcomes with leakage-safe prior-Gameweek features.

The next section trains and evaluates the first expected-minutes and rotation baseline.


## 4. Expected minutes and rotation baseline

Minutes are modelled as a sequence of related events rather than one opaque regression:

1. probability of making any appearance;
2. probability of starting;
3. probability of reaching 60 minutes;
4. expected minutes conditional on starting;
5. expected minutes conditional on appearing as a substitute.

The final expectation is:

\[
E[\text{minutes}] =
P(\text{start})E[\text{minutes}\mid\text{start}]
+
P(\text{sub appearance})E[\text{minutes}\mid\text{sub appearance}]
\]

The probabilities are reconciled after prediction so that:

```text
P(start) <= P(appearance)
P(60+ minutes) <= P(start)
P(no appearance) + P(start) + P(sub appearance) = 1
```

This is an initial, transparent baseline. It uses only the leakage-safe features defined above and does not yet include European or domestic-cup congestion, press-conference information, or squad-depth features.

### Modelling imports and configuration

The development seasons are used for fitting. The entire 2025/26 season remains untouched until the final holdout evaluation.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


RANDOM_STATE = 42
PROBABILITY_EPSILON = 1e-6

MINUTES_CATEGORICAL_FEATURES = ["position"]
MINUTES_NUMERIC_FEATURES = [
    column
    for column in MINUTES_FEATURE_COLUMNS
    if column not in MINUTES_CATEGORICAL_FEATURES
]


# ---------------------------------------------------------------------------
# Model feature-schema contracts
# ---------------------------------------------------------------------------
# All model families use these helpers instead of direct frame[FEATURE_COLUMNS]
# indexing. This prevents live/training schema drift from becoming a KeyError,
# while keeping a visible audit trail whenever a repair was necessary.

MODEL_SCHEMA_REPAIR_LOG: list[dict[str, object]] = []
_MODEL_SCHEMA_WARNED_SIGNATURES: set[tuple] = set()

MODEL_FEATURE_ALIASES: dict[str, tuple[str, ...]] = {
    "gameweek": ("event", "fixture_gameweek"),
    "was_home": ("is_home", "future_was_home"),
    "position": ("model_position", "element_type", "position_id"),
    "model_position": ("position", "element_type", "position_id"),
}


def _normalise_position_feature(
    values: pd.Series,
    target_column: str,
) -> pd.Series:
    """Normalise common FPL position encodings for a requested feature."""
    raw = values.astype("string").str.strip().str.upper()
    numeric = pd.to_numeric(values, errors="coerce")

    if target_column == "position":
        mapping = {
            "GK": "GKP",
            "GKP": "GKP",
            "GOALKEEPER": "GKP",
            "KEEPER": "GKP",
            "DEFENDER": "DEF",
            "MIDFIELDER": "MID",
            "FORWARD": "FWD",
            "STRIKER": "FWD",
            "1": "GKP",
            "2": "DEF",
            "3": "MID",
            "4": "FWD",
        }
        result = raw.replace(mapping)
        numeric_map = {1: "GKP", 2: "DEF", 3: "MID", 4: "FWD"}
    else:
        mapping = {
            "GK": "GK",
            "GKP": "GK",
            "GOALKEEPER": "GK",
            "KEEPER": "GK",
            "DEFENDER": "DEF",
            "MIDFIELDER": "MID",
            "FORWARD": "FWD",
            "STRIKER": "FWD",
            "1": "GK",
            "2": "DEF",
            "3": "MID",
            "4": "FWD",
        }
        result = raw.replace(mapping)
        numeric_map = {1: "GK", 2: "DEF", 3: "MID", 4: "FWD"}

    numeric_values = numeric.map(numeric_map).astype("string")
    return result.where(numeric_values.isna(), numeric_values)


def _collapse_exact_duplicate_columns(
    frame: pd.DataFrame,
    *,
    context: str,
) -> tuple[pd.DataFrame, list[str]]:
    """
    Collapse exact duplicate column labels left-to-right.

    Duplicate labels can arise from manually concatenated frames. Non-null
    values are coalesced; conflicting non-null values preserve the left-most
    value and are recorded in the schema audit.
    """
    if not frame.columns.duplicated().any():
        return frame.copy(), []

    repairs: list[str] = []
    result = pd.DataFrame(index=frame.index)

    for column in dict.fromkeys(frame.columns.tolist()):
        block = frame.loc[:, frame.columns == column]
        series = block.iloc[:, 0].copy()

        if block.shape[1] > 1:
            repairs.append(f"collapsed_duplicate:{column}")
            for j in range(1, block.shape[1]):
                other = block.iloc[:, j]
                series = series.combine_first(other)

        result[column] = series

    return result, repairs


def _coalesce_available_columns(
    frame: pd.DataFrame,
    candidates: list[str],
) -> pd.Series | None:
    """Return the first non-null value across available candidate columns."""
    available = [column for column in candidates if column in frame.columns]
    if not available:
        return None

    series = frame[available[0]].copy()
    for column in available[1:]:
        series = series.combine_first(frame[column])
    return series


def prepare_model_feature_frame(
    frame: pd.DataFrame,
    feature_columns: list[str],
    *,
    context: str,
    categorical_columns: list[str] | tuple[str, ...] = (),
    emit_warning: bool = True,
) -> pd.DataFrame:
    """
    Materialise a stable model-input schema in the exact trained column order.

    Recovery order for a missing expected feature:
      1. pandas merge suffixes (`feature_x` / `feature_y`);
      2. known semantic aliases (`event` -> `gameweek`, etc.);
      3. a missing-value placeholder handled by the fitted preprocessor.

    This is deliberately prediction-safe: optional/source-specific columns may
    disappear without producing a KeyError. Repairs are logged and warned once.
    """
    if not isinstance(frame, pd.DataFrame):
        raise TypeError(
            f"{context}: expected a pandas DataFrame, got {type(frame).__name__}."
        )

    working, repairs = _collapse_exact_duplicate_columns(
        frame,
        context=context,
    )
    categorical = set(categorical_columns)

    for column in feature_columns:
        if column in working.columns:
            continue

        # Recover pandas merge suffixes. For opponent features prefer the right
        # side (`_y`), which is normally the explicitly merged opponent state.
        suffix_candidates = (
            [f"{column}_y", f"{column}_x"]
            if column.startswith("opponent_")
            else [f"{column}_x", f"{column}_y"]
        )
        recovered = _coalesce_available_columns(
            working,
            suffix_candidates,
        )
        recovery_source = [
            candidate
            for candidate in suffix_candidates
            if candidate in working.columns
        ]

        if recovered is None:
            alias_candidates = list(
                MODEL_FEATURE_ALIASES.get(column, ())
            )
            recovered = _coalesce_available_columns(
                working,
                alias_candidates,
            )
            recovery_source = [
                candidate
                for candidate in alias_candidates
                if candidate in working.columns
            ]

        if recovered is not None:
            if column in {"position", "model_position"}:
                recovered = _normalise_position_feature(
                    recovered,
                    column,
                )
            elif column == "was_home":
                recovered = normalise_boolean(recovered)

            working[column] = recovered
            repairs.append(
                f"recovered:{column}<-{','.join(recovery_source)}"
            )
        else:
            working[column] = (
                "__MISSING__" if column in categorical else np.nan
            )
            repairs.append(f"materialised_missing:{column}")

    prepared = working.reindex(columns=feature_columns).copy()

    for column in feature_columns:
        if column in {"position", "model_position"}:
            prepared[column] = _normalise_position_feature(
                prepared[column],
                column,
            )

        if column in categorical:
            prepared[column] = (
                prepared[column]
                .astype("string")
                .fillna("__MISSING__")
                .astype(object)
            )
        else:
            prepared[column] = (
                pd.to_numeric(
                    prepared[column],
                    errors="coerce",
                )
                .replace([np.inf, -np.inf], np.nan)
            )

    if repairs:
        record = {
            "context": str(context),
            "repairs": tuple(repairs),
            "rows": int(len(frame)),
        }
        MODEL_SCHEMA_REPAIR_LOG.append(record)

        signature = (str(context), tuple(repairs))
        if emit_warning and signature not in _MODEL_SCHEMA_WARNED_SIGNATURES:
            warnings.warn(
                f"{context}: model feature schema repaired automatically: "
                + "; ".join(repairs),
                RuntimeWarning,
                stacklevel=2,
            )
            _MODEL_SCHEMA_WARNED_SIGNATURES.add(signature)

    return prepared


def materialise_model_feature_columns(
    frame: pd.DataFrame,
    feature_columns: list[str],
    *,
    context: str,
    categorical_columns: list[str] | tuple[str, ...] = (),
    emit_warning: bool = True,
) -> pd.DataFrame:
    """Return the original frame with every expected model feature materialised."""
    result = frame.copy()
    prepared = prepare_model_feature_frame(
        result,
        feature_columns,
        context=context,
        categorical_columns=categorical_columns,
        emit_warning=emit_warning,
    )
    for column in feature_columns:
        result[column] = prepared[column]
    return result


def model_feature_contract_report(
    frame: pd.DataFrame,
    feature_columns: list[str],
) -> dict[str, object]:
    """Summarise schema health before any automatic repair is applied."""
    columns = list(frame.columns)
    missing = [
        column
        for column in feature_columns
        if column not in columns
    ]
    suffixed = {
        column: [
            candidate
            for candidate in (f"{column}_x", f"{column}_y")
            if candidate in columns
        ]
        for column in missing
    }
    suffixed = {
        column: candidates
        for column, candidates in suffixed.items()
        if candidates
    }
    duplicates = sorted(
        set(
            frame.columns[
                frame.columns.duplicated(keep=False)
            ].tolist()
        )
    )
    return {
        "missing": missing,
        "recoverable_from_suffix": suffixed,
        "duplicate_columns": duplicates,
    }


def _run_model_schema_contract_self_tests() -> None:
    """Regression tests for missing/suffixed/aliased model features."""
    synthetic = pd.DataFrame(
        {
            "event": [4, 4],
            "is_home": [True, False],
            "opponent_goals_for_last_5_x": [np.nan, np.nan],
            "opponent_goals_for_last_5_y": [1.2, 0.8],
            "position": ["GKP", "MID"],
        }
    )

    team_features = prepare_model_feature_frame(
        synthetic,
        [
            "gameweek",
            "was_home",
            "opponent_goals_for_last_5",
            "opponent_xg_for_last_5",
        ],
        context="schema_self_test.team",
        emit_warning=False,
    )
    assert team_features.columns.tolist() == [
        "gameweek",
        "was_home",
        "opponent_goals_for_last_5",
        "opponent_xg_for_last_5",
    ]
    assert team_features["gameweek"].eq(4).all()
    assert team_features["opponent_goals_for_last_5"].tolist() == [1.2, 0.8]
    assert team_features["opponent_xg_for_last_5"].isna().all()

    event_features = prepare_model_feature_frame(
        synthetic,
        ["model_position"],
        context="schema_self_test.player_event",
        categorical_columns=["model_position"],
        emit_warning=False,
    )
    assert event_features["model_position"].tolist() == ["GK", "MID"]


_run_model_schema_contract_self_tests()
# Self-test repairs are not production diagnostics.
MODEL_SCHEMA_REPAIR_LOG.clear()


def make_dense_one_hot_encoder() -> OneHotEncoder:
    """Create a dense encoder across recent and older scikit-learn versions."""
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


def make_minutes_preprocessor() -> ColumnTransformer:
    """Build a fresh preprocessing pipeline for one model."""
    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
        ),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        ("one_hot", make_dense_one_hot_encoder()),
    ])

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                MINUTES_NUMERIC_FEATURES,
            ),
            (
                "categorical",
                categorical_pipeline,
                MINUTES_CATEGORICAL_FEATURES,
            ),
        ],
        remainder="drop",
        sparse_threshold=0,
    )


def make_binary_minutes_model() -> Pipeline:
    """Return a regularised probability model."""
    return Pipeline([
        ("preprocess", make_minutes_preprocessor()),
        (
            "model",
            LogisticRegression(
                C=0.5,
                max_iter=2_000,
                solver="lbfgs",
                random_state=RANDOM_STATE,
            ),
        ),
    ])


def make_conditional_minutes_model() -> Pipeline:
    """Return a regularised conditional-minutes regression model."""
    return Pipeline([
        ("preprocess", make_minutes_preprocessor()),
        ("model", Ridge(alpha=10.0)),
    ])


### Fit the component models

No hyperparameters are selected using the holdout season. The initial regularisation values are deliberately fixed; tuning can later be performed with chronological validation inside the development seasons.

In [ ]:
def resolve_minutes_model_position(
    frame: pd.DataFrame,
) -> pd.Series:
    """
    Return canonical GK/DEF/MID/FWD labels for the minutes layer.

    The historical feature table carries `position` (usually GKP/DEF/MID/FWD),
    while later scoring/live tables can carry `model_position`. Raw FPL tables
    may instead expose the numeric element type 1..4. Position is not itself a
    minutes-model feature here; it is used only to choose an optional
    position-specific conditional-duration model.

    Missing position metadata is deliberately non-fatal: callers then fall
    back to the globally fitted starter/substitute duration models.
    """
    source_column = next(
        (
            column
            for column in (
                "model_position",
                "position",
                "element_type",
                "position_id",
            )
            if column in frame.columns
        ),
        None,
    )

    if source_column is None:
        return pd.Series(
            pd.NA,
            index=frame.index,
            dtype="string",
            name="_minutes_model_position",
        )

    raw = frame[source_column]
    numeric = pd.to_numeric(raw, errors="coerce")

    canonical = (
        raw.astype("string")
        .str.strip()
        .str.upper()
        .replace(
            {
                "GKP": "GK",
                "GOALKEEPER": "GK",
                "KEEPER": "GK",
                "DEFENDER": "DEF",
                "MIDFIELDER": "MID",
                "FORWARD": "FWD",
                "STRIKER": "FWD",
                "1": "GK",
                "2": "DEF",
                "3": "MID",
                "4": "FWD",
            }
        )
    )

    numeric_mapping = {1: "GK", 2: "DEF", 3: "MID", 4: "FWD"}
    numeric_position = numeric.map(numeric_mapping).astype("string")
    canonical = canonical.where(numeric_position.isna(), numeric_position)
    canonical = canonical.where(
        canonical.isin(PLAYER_SCORING_POSITIONS),
        pd.NA,
    )
    canonical.name = "_minutes_model_position"
    return canonical


def fit_minutes_model_bundle(
    training_data: pd.DataFrame,
) -> dict[str, object]:
    """
    Fit a coherent hierarchical minutes model.

    v2.2 models:
      P(appear)
      P(start | appear)
      P(60+ | start)

    This guarantees probability coherence by construction instead of clipping
    three independently fitted unconditional classifiers. Duration models are
    fitted globally and, where sample size allows, separately by position.
    """
    models: dict[str, object] = {}

    # `historical_model_data` still exposes the archived FPL `position`
    # column at this stage. Resolve it here rather than assuming the later
    # expected-points `model_position` column already exists.
    training_positions = resolve_minutes_model_position(training_data)

    # Appearance is unconditional.
    appearance_rows = training_data.loc[
        training_data["target_appeared"].notna()
    ].copy()
    appearance_target = (
        pd.to_numeric(appearance_rows["target_appeared"], errors="coerce")
        .clip(0, 1)
        .astype(int)
    )
    if appearance_target.nunique() < 2:
        raise ValueError("Appearance target has fewer than two classes.")
    appearance_model = make_binary_minutes_model()
    appearance_model.fit(
        prepare_model_feature_frame(
            appearance_rows,
            MINUTES_FEATURE_COLUMNS,
            context="minutes.fit.appearance",
            categorical_columns=MINUTES_CATEGORICAL_FEATURES,
        ),
        appearance_target,
    )
    models["appearance"] = appearance_model

    # Conditional start probability among players who appeared.
    start_rows = training_data.loc[
        training_data["target_appeared"].eq(1)
        & training_data["target_started"].notna()
    ].copy()
    start_target = (
        pd.to_numeric(start_rows["target_started"], errors="coerce")
        .clip(0, 1)
        .astype(int)
    )
    if start_target.nunique() < 2:
        raise ValueError("Conditional start target has fewer than two classes.")
    start_model = make_binary_minutes_model()
    start_model.fit(
        prepare_model_feature_frame(
            start_rows,
            MINUTES_FEATURE_COLUMNS,
            context="minutes.fit.start_given_appearance",
            categorical_columns=MINUTES_CATEGORICAL_FEATURES,
        ),
        start_target,
    )
    # Preserve the public key used elsewhere in the notebook.
    models["start"] = start_model

    # Conditional 60+ probability among starters.
    sixty_rows = training_data.loc[
        training_data["target_started"].eq(1)
        & training_data["target_60_plus"].notna()
    ].copy()
    sixty_target = (
        pd.to_numeric(sixty_rows["target_60_plus"], errors="coerce")
        .clip(0, 1)
        .astype(int)
    )
    if sixty_target.nunique() < 2:
        raise ValueError("Conditional 60+ target has fewer than two classes.")
    sixty_model = make_binary_minutes_model()
    sixty_model.fit(
        prepare_model_feature_frame(
            sixty_rows,
            MINUTES_FEATURE_COLUMNS,
            context="minutes.fit.sixty_given_start",
            categorical_columns=MINUTES_CATEGORICAL_FEATURES,
        ),
        sixty_target,
    )
    models["sixty_plus"] = sixty_model

    starter_rows = training_data.loc[
        training_data["target_started"].eq(1)
        & training_data["target_minutes"].notna()
    ].copy()

    substitute_rows = training_data.loc[
        training_data["target_started"].eq(0)
        & training_data["target_appeared"].eq(1)
        & training_data["target_minutes"].notna()
    ].copy()

    if starter_rows.empty or substitute_rows.empty:
        raise ValueError(
            "Conditional minutes models require both starter and substitute rows."
        )

    starter_model = make_conditional_minutes_model()
    starter_model.fit(
        prepare_model_feature_frame(
            starter_rows,
            MINUTES_FEATURE_COLUMNS,
            context="minutes.fit.duration_start",
            categorical_columns=MINUTES_CATEGORICAL_FEATURES,
        ),
        starter_rows["target_minutes"],
    )
    models["minutes_if_start"] = starter_model

    substitute_model = make_conditional_minutes_model()
    substitute_model.fit(
        prepare_model_feature_frame(
            substitute_rows,
            MINUTES_FEATURE_COLUMNS,
            context="minutes.fit.duration_sub",
            categorical_columns=MINUTES_CATEGORICAL_FEATURES,
        ),
        substitute_rows["target_minutes"],
    )
    models["minutes_if_sub"] = substitute_model

    # Position-specific duration models reduce the bias from fitting the same
    # substitution/starter duration process to GKs, defenders and attackers.
    models["minutes_if_start_by_position"] = {}
    models["minutes_if_sub_by_position"] = {}
    for position in PLAYER_SCORING_POSITIONS:
        pos_starters = starter_rows.loc[
            training_positions.reindex(starter_rows.index).eq(position)
        ]
        if len(pos_starters) >= int(MINUTES_POSITION_DURATION_MIN_ROWS):
            model = make_conditional_minutes_model()
            model.fit(
                prepare_model_feature_frame(
                    pos_starters,
                    MINUTES_FEATURE_COLUMNS,
                    context=f"minutes.fit.duration_start.{position}",
                    categorical_columns=MINUTES_CATEGORICAL_FEATURES,
                ),
                pos_starters["target_minutes"],
            )
            models["minutes_if_start_by_position"][position] = model

        pos_subs = substitute_rows.loc[
            training_positions.reindex(substitute_rows.index).eq(position)
        ]
        if len(pos_subs) >= int(MINUTES_POSITION_DURATION_MIN_ROWS):
            model = make_conditional_minutes_model()
            model.fit(
                prepare_model_feature_frame(
                    pos_subs,
                    MINUTES_FEATURE_COLUMNS,
                    context=f"minutes.fit.duration_sub.{position}",
                    categorical_columns=MINUTES_CATEGORICAL_FEATURES,
                ),
                pos_subs["target_minutes"],
            )
            models["minutes_if_sub_by_position"][position] = model

    return models


minutes_models = fit_minutes_model_bundle(development_data)

print("Fitted expected-minutes components:")
for model_name in minutes_models:
    print(f"- {model_name}")

### Generate internally consistent minutes predictions

The three binary models are fitted independently. Their raw probabilities are therefore reconciled into a valid probability decomposition before expected minutes are calculated.

In [ ]:

def predict_minutes_components(
    model_bundle: dict[str, object],
    feature_data: pd.DataFrame,
) -> pd.DataFrame:
    """
    Predict coherent appearance states and expected minutes.

    Hierarchy:
        P(appear)
        P(start) = P(appear) * P(start | appear)
        P(60+)   = P(start)  * P(60+ | start)

    Sparse/new-player external role priors are blended only after the coherent
    base probabilities are formed and the hierarchy is re-enforced afterwards.
    """
    features = prepare_model_feature_frame(
        feature_data,
        MINUTES_FEATURE_COLUMNS,
        context="minutes.predict",
        categorical_columns=MINUTES_CATEGORICAL_FEATURES,
    )

    appearance_probability = (
        model_bundle["appearance"].predict_proba(features)[:, 1]
    )
    conditional_start_probability = (
        model_bundle["start"].predict_proba(features)[:, 1]
    )
    conditional_sixty_probability = (
        model_bundle["sixty_plus"].predict_proba(features)[:, 1]
    )

    start_probability = (
        appearance_probability * conditional_start_probability
    )
    sixty_plus_probability = (
        start_probability * conditional_sixty_probability
    )

    external_weight = external_new_player_weight(
        feature_data,
        EXTERNAL_MAX_MINUTES_BLEND,
    ).to_numpy(dtype=float)

    external_role = pd.to_numeric(
        feature_data.get(
            "external_role_score",
            pd.Series(np.nan, index=feature_data.index),
        ),
        errors="coerce",
    ).to_numpy(dtype=float)
    external_role = np.nan_to_num(
        external_role,
        nan=0.0,
        posinf=1.0,
        neginf=0.0,
    )
    external_role = np.clip(external_role, 0.0, 1.0)

    external_minutes_per_appearance = pd.to_numeric(
        feature_data.get(
            "external_minutes_per_appearance",
            pd.Series(np.nan, index=feature_data.index),
        ),
        errors="coerce",
    ).to_numpy(dtype=float)
    external_minutes_per_appearance = np.nan_to_num(
        external_minutes_per_appearance,
        nan=45.0,
        posinf=90.0,
        neginf=0.0,
    )
    external_sixty_role = np.clip(
        external_role
        * np.clip(external_minutes_per_appearance / 60.0, 0.0, 1.0),
        0.0,
        1.0,
    )

    start_probability = (
        (1.0 - external_weight) * start_probability
        + external_weight * external_role
    )
    appearance_probability = np.maximum(
        appearance_probability,
        start_probability,
    )
    sixty_plus_probability = (
        (1.0 - external_weight) * sixty_plus_probability
        + external_weight * external_sixty_role
    )

    start_probability = np.minimum(
        start_probability,
        appearance_probability,
    )
    sixty_plus_probability = np.minimum(
        sixty_plus_probability,
        start_probability,
    )

    # Recompute conditional diagnostics after the sparse-player blend.
    conditional_start_probability = np.divide(
        start_probability,
        appearance_probability,
        out=np.zeros_like(start_probability, dtype=float),
        where=appearance_probability > 1e-12,
    )
    conditional_sixty_probability = np.divide(
        sixty_plus_probability,
        start_probability,
        out=np.zeros_like(start_probability, dtype=float),
        where=start_probability > 1e-12,
    )

    substitute_probability = np.maximum(
        appearance_probability - start_probability,
        0.0,
    )
    no_appearance_probability = np.maximum(
        1.0 - appearance_probability,
        0.0,
    )

    # Global duration predictions.
    minutes_if_start = np.clip(
        model_bundle["minutes_if_start"].predict(features),
        1.0,
        90.0,
    )
    minutes_if_sub = np.clip(
        model_bundle["minutes_if_sub"].predict(features),
        1.0,
        90.0,
    )

    # Position-specific overrides when they were trained with enough data.
    positions = resolve_minutes_model_position(feature_data)
    # If position is unavailable, no row matches the optional position-specific
    # dictionaries and the global duration predictions remain in force.
    for position, model in model_bundle.get(
        "minutes_if_start_by_position", {}
    ).items():
        mask = positions.eq(str(position)).to_numpy()
        if mask.any():
            minutes_if_start[mask] = np.clip(
                model.predict(features.loc[mask]),
                1.0,
                90.0,
            )
    for position, model in model_bundle.get(
        "minutes_if_sub_by_position", {}
    ).items():
        mask = positions.eq(str(position)).to_numpy()
        if mask.any():
            minutes_if_sub[mask] = np.clip(
                model.predict(features.loc[mask]),
                1.0,
                90.0,
            )

    expected_minutes = (
        start_probability * minutes_if_start
        + substitute_probability * minutes_if_sub
    )

    predictions = pd.DataFrame(
        {
            "appearance_probability": appearance_probability,
            "conditional_start_given_appearance_probability": (
                conditional_start_probability
            ),
            "start_probability": start_probability,
            "substitute_appearance_probability": substitute_probability,
            "no_appearance_probability": no_appearance_probability,
            "conditional_sixty_given_start_probability": (
                conditional_sixty_probability
            ),
            "sixty_plus_probability": sixty_plus_probability,
            "minutes_if_start": minutes_if_start,
            "minutes_if_sub": minutes_if_sub,
            "expected_minutes": np.clip(expected_minutes, 0.0, 90.0),
            "external_minutes_prior_weight": external_weight,
        },
        index=feature_data.index,
    )

    probability_total = (
        predictions["start_probability"]
        + predictions["substitute_appearance_probability"]
        + predictions["no_appearance_probability"]
    )

    if not np.allclose(probability_total, 1.0, atol=1e-8):
        raise AssertionError(
            "Appearance-state probabilities do not sum to one."
        )
    if (
        predictions["sixty_plus_probability"]
        > predictions["start_probability"] + 1e-10
    ).any():
        raise AssertionError("P(60+) exceeded P(start).")
    if (
        predictions["start_probability"]
        > predictions["appearance_probability"] + 1e-10
    ).any():
        raise AssertionError("P(start) exceeded P(appearance).")

    return predictions


### Holdout evaluation

The learned model is compared with simple rolling-history baselines. This matters because a more complex model is only useful if it improves on recent average minutes and recent appearance rates.

In [ ]:
def coalesced_history_baseline(
    data: pd.DataFrame,
    candidate_columns: list[str],
    fallback_by_position: pd.Series,
) -> pd.Series:
    """Use the first available historical rate and then a position prior."""
    available_columns = [
        column
        for column in candidate_columns
        if column in data.columns
    ]

    if not available_columns:
        baseline = pd.Series(
            np.nan,
            index=data.index,
            dtype=float,
        )
    else:
        baseline = data[available_columns].bfill(axis=1).iloc[:, 0]

    position_fallback = data["position"].map(
        fallback_by_position
    )

    return baseline.fillna(position_fallback)


def safe_binary_metrics(
    target: pd.Series,
    probability: pd.Series | np.ndarray,
) -> dict[str, float]:
    """Calculate probability metrics without failing on a single class."""
    target_array = np.asarray(target, dtype=int)
    probability_array = np.clip(
        np.asarray(probability, dtype=float),
        PROBABILITY_EPSILON,
        1.0 - PROBABILITY_EPSILON,
    )

    metrics = {
        "brier": brier_score_loss(
            target_array,
            probability_array,
        ),
        "log_loss": log_loss(
            target_array,
            probability_array,
            labels=[0, 1],
        ),
        "roc_auc": np.nan,
    }

    if np.unique(target_array).size == 2:
        metrics["roc_auc"] = roc_auc_score(
            target_array,
            probability_array,
        )

    return metrics


def evaluate_minutes_holdout(
    training_data: pd.DataFrame,
    evaluation_data: pd.DataFrame,
    model_bundle: dict[str, Pipeline],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return holdout predictions and a model-versus-baseline scorecard."""
    evaluation = evaluation_data.copy()
    component_predictions = predict_minutes_components(
        model_bundle,
        evaluation,
    )

    evaluation = pd.concat(
        [evaluation, component_predictions],
        axis=1,
    )

    binary_specifications = {
        "appearance": {
            "target": "target_appeared",
            "prediction": "appearance_probability",
            "history": [
                "appearance_rate_last_5_gws",
                "appearance_rate_last_10_gws",
                "career_appearance_rate",
            ],
        },
        "start": {
            "target": "target_started",
            "prediction": "start_probability",
            "history": [
                "start_rate_last_5_gws",
                "start_rate_last_10_gws",
                "career_start_rate",
            ],
        },
        "sixty_plus": {
            "target": "target_60_plus",
            "prediction": "sixty_plus_probability",
            "history": [
                "sixty_plus_rate_last_5_gws",
                "sixty_plus_rate_last_10_gws",
                "career_sixty_plus_rate",
            ],
        },
    }

    metric_rows: list[dict[str, object]] = []

    for outcome, specification in binary_specifications.items():
        target_column = specification["target"]
        valid = evaluation[target_column].notna()

        position_prior = (
            training_data.loc[
                training_data[target_column].notna()
            ]
            .groupby("position")[target_column]
            .mean()
        )

        global_prior = training_data[target_column].mean()
        position_prior = position_prior.fillna(global_prior)

        baseline_probability = coalesced_history_baseline(
            evaluation,
            specification["history"],
            position_prior,
        ).fillna(global_prior).clip(0.0, 1.0)

        evaluation[
            f"baseline_{outcome}_probability"
        ] = baseline_probability

        for method, probability_column in [
            ("rolling baseline", f"baseline_{outcome}_probability"),
            ("component model", specification["prediction"]),
        ]:
            metrics = safe_binary_metrics(
                evaluation.loc[valid, target_column],
                evaluation.loc[valid, probability_column],
            )
            metric_rows.append({
                "outcome": outcome,
                "method": method,
                **metrics,
            })

    position_minutes_prior = (
        training_data
        .groupby("position")["target_minutes"]
        .mean()
    )
    global_minutes_prior = training_data["target_minutes"].mean()

    evaluation["baseline_expected_minutes"] = (
        coalesced_history_baseline(
            evaluation,
            [
                "minutes_per_fixture_last_5_gws",
                "minutes_per_fixture_last_10_gws",
                "career_minutes_per_fixture",
            ],
            position_minutes_prior,
        )
        .fillna(global_minutes_prior)
        .clip(0.0, 90.0)
    )

    valid_minutes = evaluation["target_minutes"].notna()

    for method, prediction_column in [
        ("rolling baseline", "baseline_expected_minutes"),
        ("component model", "expected_minutes"),
    ]:
        target = evaluation.loc[valid_minutes, "target_minutes"]
        prediction = evaluation.loc[
            valid_minutes,
            prediction_column,
        ]

        metric_rows.append({
            "outcome": "minutes",
            "method": method,
            "mae": mean_absolute_error(target, prediction),
            "rmse": np.sqrt(
                mean_squared_error(target, prediction)
            ),
        })

    scorecard = pd.DataFrame(metric_rows)

    return evaluation, scorecard


holdout_minutes_predictions, minutes_scorecard = (
    evaluate_minutes_holdout(
        development_data,
        holdout_data,
        minutes_models,
    )
)

display(minutes_scorecard)


### Diagnostics by position and probability calibration

The overall score can hide systematic weaknesses. The following tables show expected-minutes error by FPL position and whether predicted starting probabilities correspond to observed start rates.

In [ ]:
minutes_error_by_position = (
    holdout_minutes_predictions.loc[
        holdout_minutes_predictions["target_minutes"].notna()
    ]
    .assign(
        absolute_error=lambda frame: (
            frame["target_minutes"]
            - frame["expected_minutes"]
        ).abs(),
        baseline_absolute_error=lambda frame: (
            frame["target_minutes"]
            - frame["baseline_expected_minutes"]
        ).abs(),
    )
    .groupby("position")
    .agg(
        rows=("target_minutes", "size"),
        model_mae=("absolute_error", "mean"),
        baseline_mae=("baseline_absolute_error", "mean"),
        observed_minutes=("target_minutes", "mean"),
        predicted_minutes=("expected_minutes", "mean"),
    )
    .reset_index()
)

display(minutes_error_by_position)


start_calibration = (
    holdout_minutes_predictions.loc[
        holdout_minutes_predictions["target_started"].notna()
    ]
    .assign(
        probability_bin=lambda frame: pd.cut(
            frame["start_probability"],
            bins=np.linspace(0.0, 1.0, 11),
            include_lowest=True,
        )
    )
    .groupby(
        "probability_bin",
        observed=False,
    )
    .agg(
        rows=("target_started", "size"),
        mean_predicted_probability=("start_probability", "mean"),
        observed_start_rate=("target_started", "mean"),
    )
    .reset_index()
)

display(start_calibration)


### Interpret the start model

Because the baseline uses regularised logistic regression, its coefficients can be inspected. Coefficients describe associations after preprocessing; they should not be interpreted as causal effects.

In [ ]:
def get_linear_model_coefficients(
    fitted_pipeline: Pipeline,
) -> pd.DataFrame:
    """Return transformed feature names and fitted linear coefficients."""
    preprocessor = fitted_pipeline.named_steps["preprocess"]
    estimator = fitted_pipeline.named_steps["model"]

    feature_names = preprocessor.get_feature_names_out()
    coefficients = np.asarray(estimator.coef_).reshape(-1)

    return (
        pd.DataFrame({
            "feature": feature_names,
            "coefficient": coefficients,
            "absolute_coefficient": np.abs(coefficients),
        })
        .sort_values(
            "absolute_coefficient",
            ascending=False,
        )
        .reset_index(drop=True)
    )


start_model_coefficients = get_linear_model_coefficients(
    minutes_models["start"]
)

display(start_model_coefficients.head(20))


## 5. Apply the minutes model to upcoming fixtures

Historical model evaluation and live prediction are separate operations.

For live use, the notebook builds each current player's latest history from:

- all completed archived seasons;
- completed Gameweeks from the current FPL season, obtained from the official `event/{gameweek}/live/` endpoint;
- the current official injury and availability fields.

The official live endpoint is aggregated at Gameweek level. Its fixture-level `explain` entries are used when available to count appearances in double Gameweeks. If those entries are unavailable, the code uses a conservative fallback.

### Load completed current-season Gameweeks

At the start of the season this function returns an empty table, so the opening projections rely on prior-season history and position-level priors.

In [ ]:
CURRENT_SEASON_START_YEAR = int(CURRENT_SEASON.split("-")[0])

CURRENT_GAMEWEEK_RAW_COLUMNS = [
    "gw_fixture_count",
    "gw_minutes",
    "gw_starts",
    "gw_appearance_count",
    "gw_60_plus_count",
    "gw_first_kickoff",
    "gw_last_kickoff",
]


_COMPLETED_GAMEWEEK_RESOLUTION_CACHE: dict[str, object] = {}


def resolved_completed_gameweek_ids(
    gameweek_table: pd.DataFrame,
    fixture_table: pd.DataFrame | None = None,
) -> list[int]:
    """
    Resolve completed/published Gameweeks from multiple FPL signals.

    This fixes the short propagation window where entry history/live data are
    available before bootstrap-static has flipped events.finished to True.
    """
    cached = _COMPLETED_GAMEWEEK_RESOLUTION_CACHE.get("resolved")
    if isinstance(cached, list):
        return list(cached)

    completed: set[int] = set()

    if "finished" in gameweek_table.columns and "id" in gameweek_table.columns:
        completed.update(
            gameweek_table.loc[
                gameweek_table["finished"].fillna(False),
                "id",
            ]
            .dropna()
            .astype(int)
            .tolist()
        )

    if fixture_table is not None and not fixture_table.empty:
        event_col = (
            "event"
            if "event" in fixture_table.columns
            else "gameweek"
            if "gameweek" in fixture_table.columns
            else None
        )
        if event_col is not None:
            work = fixture_table.loc[
                fixture_table[event_col].notna()
            ].copy()

            if (
                "team_h_score" in work.columns
                and "team_a_score" in work.columns
            ):
                work["_score_final"] = (
                    work["team_h_score"].notna()
                    & work["team_a_score"].notna()
                )
            else:
                work["_score_final"] = False

            if "finished" in work.columns:
                work["_fixture_final"] = (
                    work["finished"].fillna(False)
                    | work["_score_final"]
                )
            else:
                work["_fixture_final"] = work["_score_final"]

            completion = (
                work.groupby(event_col)["_fixture_final"]
                .agg(["size", "all"])
            )
            completed.update(
                int(v)
                for v in completion.loc[
                    completion["size"].gt(0)
                    & completion["all"]
                ].index
            )

    if FPL_TEAM_ID is not None:
        try:
            history_payload = get_fpl_data(
                f"entry/{int(FPL_TEAM_ID)}/history/"
            )
            completed.update(
                int(row["event"])
                for row in history_payload.get("current", [])
                if row.get("event") is not None
            )
        except Exception as exc:
            warnings.warn(
                "Could not use FPL entry history as completed-GW fallback: "
                f"{exc}"
            )

    valid_ids = set(
        pd.to_numeric(
            gameweek_table["id"], errors="coerce"
        )
        .dropna()
        .astype(int)
        .tolist()
    )
    resolved = sorted(completed & valid_ids)
    _COMPLETED_GAMEWEEK_RESOLUTION_CACHE["resolved"] = list(resolved)
    return resolved


def extract_explain_minutes(
    explain: object,
) -> list[float]:
    """Extract per-fixture minutes from an FPL live `explain` field."""
    if not isinstance(explain, list):
        return []

    fixture_minutes: list[float] = []

    for fixture_explanation in explain:
        if not isinstance(fixture_explanation, dict):
            continue

        minutes_value = None
        for statistic in fixture_explanation.get("stats", []):
            if (
                isinstance(statistic, dict)
                and statistic.get("identifier") == "minutes"
            ):
                minutes_value = pd.to_numeric(
                    statistic.get("value"),
                    errors="coerce",
                )
                break

        if pd.notna(minutes_value):
            fixture_minutes.append(float(minutes_value))

    return fixture_minutes


def load_current_season_player_gameweeks(
    gameweek_table: pd.DataFrame,
    current_players: pd.DataFrame,
    current_team_fixtures: pd.DataFrame,
) -> pd.DataFrame:
    """Load completed current-season player–Gameweek aggregates."""
    completed_gameweeks = resolved_completed_gameweek_ids(
        gameweek_table,
        current_team_fixtures,
    )

    if not completed_gameweeks:
        return pd.DataFrame(
            columns=[
                "season",
                "season_start_year",
                "player_key",
                "gameweek",
            ] + CURRENT_GAMEWEEK_RAW_COLUMNS
        )

    player_lookup = (
        current_players[
            [
                "id",
                "player_key",
                "team_id",
            ]
        ]
        .rename(columns={"id": "player_id"})
        .copy()
    )

    completed_schedule = (
        current_team_fixtures.loc[
            current_team_fixtures["gameweek"].isin(
                completed_gameweeks
            )
            & current_team_fixtures["finished"].fillna(False)
        ]
        .groupby(["gameweek", "team_id"])
        .agg(
            gw_fixture_count=("fixture_id", "nunique"),
            gw_first_kickoff=("kickoff_time", "min"),
            gw_last_kickoff=("kickoff_time", "max"),
        )
        .reset_index()
    )

    rows: list[dict[str, object]] = []

    for gameweek in completed_gameweeks:
        live_payload = get_fpl_data(
            f"event/{gameweek}/live/"
        )

        for element in live_payload.get("elements", []):
            statistics = element.get("stats", {})
            explain_minutes = extract_explain_minutes(
                element.get("explain")
            )

            total_minutes = pd.to_numeric(
                statistics.get("minutes"),
                errors="coerce",
            )
            starts = pd.to_numeric(
                statistics.get("starts"),
                errors="coerce",
            )

            rows.append({
                "player_id": element.get("id"),
                "gameweek": gameweek,
                "gw_minutes": total_minutes,
                "gw_starts": starts,
                "explain_appearance_count": sum(
                    value > 0 for value in explain_minutes
                ),
                "explain_60_plus_count": sum(
                    value >= 60 for value in explain_minutes
                ),
            })

    live = pd.DataFrame(rows)

    if live.empty:
        return pd.DataFrame(
            columns=[
                "season",
                "season_start_year",
                "player_key",
                "gameweek",
            ] + CURRENT_GAMEWEEK_RAW_COLUMNS
        )

    live["player_id"] = pd.to_numeric(
        live["player_id"],
        errors="coerce",
    ).astype("Int64")

    live = (
        live
        .merge(
            player_lookup,
            on="player_id",
            how="left",
            validate="many_to_one",
        )
        .merge(
            completed_schedule,
            on=["gameweek", "team_id"],
            how="left",
            validate="many_to_one",
        )
    )

    # A current-team lookup can be imperfect for players transferred within
    # the league. Keep a usable fallback and flag this limitation for later
    # snapshot-based reconstruction.
    live["gw_fixture_count"] = (
        pd.to_numeric(
            live["gw_fixture_count"],
            errors="coerce",
        )
        .fillna(1)
        .clip(lower=1)
    )

    fallback_appearance_count = (
        pd.to_numeric(
            live["gw_minutes"],
            errors="coerce",
        )
        .fillna(0)
        .gt(0)
        .astype(int)
    )

    live["gw_appearance_count"] = np.where(
        live["explain_appearance_count"].gt(0),
        live["explain_appearance_count"],
        fallback_appearance_count,
    )

    fallback_60_plus_count = (
        pd.to_numeric(
            live["gw_minutes"],
            errors="coerce",
        )
        .fillna(0)
        .ge(60)
        .astype(int)
    )

    live["gw_60_plus_count"] = np.where(
        live["explain_60_plus_count"].gt(0),
        live["explain_60_plus_count"],
        fallback_60_plus_count,
    )

    live["season"] = CURRENT_SEASON
    live["season_start_year"] = CURRENT_SEASON_START_YEAR

    return (
        live[
            [
                "season",
                "season_start_year",
                "player_key",
                "gameweek",
            ] + CURRENT_GAMEWEEK_RAW_COLUMNS
        ]
        .dropna(subset=["player_key"])
        .sort_values(["player_key", "gameweek"])
        .reset_index(drop=True)
    )


current_season_player_gameweeks = (
    load_current_season_player_gameweeks(
        gameweeks,
        players,
        team_fixtures,
    )
)

print(
    "Completed current-season player–Gameweek rows: "
    f"{len(current_season_player_gameweeks):,}"
)


### Convert all past Gameweeks into current as-of features

For upcoming fixtures, every historical observation is in the past. The function below therefore calculates the latest rolling and career rates without shifting away the most recent completed Gameweek.

In [ ]:
LIVE_HISTORY_RATE_COLUMNS = {
    "gw_minutes": "minutes_per_fixture",
    "gw_starts": "start_rate",
    "gw_appearance_count": "appearance_rate",
    "gw_60_plus_count": "sixty_plus_rate",
}



def _canonical_fpl_position(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.upper()
        .replace({
            "GKP": "GK",
            "GOALKEEPER": "GK",
            "DEFENDER": "DEF",
            "MIDFIELDER": "MID",
            "FORWARD": "FWD",
        })
    )


def _safe_group_rate(
    frame: pd.DataFrame | None,
    raw_column: str,
    *,
    window: int | None = None,
) -> tuple[float, float]:
    """Return (rate per fixture, contributing fixture count)."""
    if (
        frame is None
        or frame.empty
        or raw_column not in frame.columns
        or "gw_fixture_count" not in frame.columns
    ):
        return np.nan, 0.0

    usable = frame.loc[frame[raw_column].notna()].copy()
    if window is not None:
        usable = usable.tail(int(window))
    if usable.empty:
        return np.nan, 0.0

    fixtures = pd.to_numeric(
        usable["gw_fixture_count"], errors="coerce"
    ).fillna(0.0)
    denominator = float(fixtures.sum())
    if denominator <= 0:
        return np.nan, 0.0

    numerator = float(
        pd.to_numeric(
            usable[raw_column], errors="coerce"
        ).fillna(0.0).sum()
    )
    return numerator / denominator, denominator


def _historical_position_feature_priors(
    feature_columns: list[str],
) -> dict[str, dict[str, float]]:
    """Historical mean model-input rate by position for fallback priors."""
    data = historical_model_data.copy()
    if "position" not in data.columns:
        return {}

    data["_shrink_position"] = _canonical_fpl_position(
        data["position"]
    )
    result: dict[str, dict[str, float]] = {}

    for feature in feature_columns:
        if feature not in data.columns:
            continue
        working = pd.DataFrame({
            "position": data["_shrink_position"],
            "value": pd.to_numeric(
                data[feature], errors="coerce"
            ),
        }).dropna()
        if working.empty:
            continue
        by_position = working.groupby("position")["value"].mean()
        result[feature] = {
            str(position): float(value)
            for position, value in by_position.items()
            if np.isfinite(value)
        }
    return result


def apply_early_season_player_history_shrinkage(
    feature_frame: pd.DataFrame,
    archived_player_gameweeks: pd.DataFrame,
    current_player_gameweeks: pd.DataFrame,
    current_players: pd.DataFrame,
    rate_map: dict[str, str],
    *,
    shrink_last_gw: bool = False,
) -> pd.DataFrame:
    """
    Stabilise sparse live player histories before model prediction.

    Established PL players are untouched. For a new/sparse PL player, current
    observations receive weight n/fade. The rest goes to the player's archived
    PL rate (itself shrunk if sparse) or a historical position prior.
    """
    result = feature_frame.copy()
    result["early_season_player_shrinkage_active"] = False
    result["early_season_player_current_fixtures"] = 0.0
    result["early_season_player_archived_pl_fixtures"] = 0.0
    result["early_season_player_current_weight"] = 1.0
    result["early_season_player_prior_source"] = "none"

    if not EARLY_SEASON_SHRINKAGE_ENABLED or result.empty:
        return result

    context = (
        current_players[["player_key", "position"]]
        .drop_duplicates("player_key")
        .copy()
    )
    context["model_position"] = _canonical_fpl_position(
        context["position"]
    )
    position_by_key = context.set_index(
        "player_key"
    )["model_position"].to_dict()

    archived_groups = {
        key: group.sort_values(
            ["season_start_year", "gameweek"]
        )
        for key, group in archived_player_gameweeks.groupby(
            "player_key", sort=False
        )
    }
    current_groups = {
        key: group.sort_values(
            ["season_start_year", "gameweek"]
        )
        for key, group in current_player_gameweeks.groupby(
            "player_key", sort=False
        )
    }

    feature_names: list[str] = []
    for feature_name in rate_map.values():
        if shrink_last_gw:
            feature_names.append(f"{feature_name}_last_gw")
        for window in ROLLING_GAMEWEEK_WINDOWS:
            feature_names.append(
                f"{feature_name}_last_{window}_gws"
            )
        feature_names.append(f"career_{feature_name}")

    position_priors = _historical_position_feature_priors(
        feature_names
    )
    fade = max(float(EARLY_SEASON_PLAYER_FADE_FIXTURES), 1.0)
    stable_archived = max(
        float(EARLY_SEASON_PLAYER_STABLE_ARCHIVED_FIXTURES),
        1.0,
    )

    for idx, player_key in result["player_key"].items():
        archived = archived_groups.get(player_key)
        current = current_groups.get(player_key)

        archived_fixtures = (
            float(
                pd.to_numeric(
                    archived["gw_fixture_count"],
                    errors="coerce",
                ).fillna(0.0).sum()
            )
            if archived is not None and not archived.empty
            else 0.0
        )
        current_fixtures = (
            float(
                pd.to_numeric(
                    current["gw_fixture_count"],
                    errors="coerce",
                ).fillna(0.0).sum()
            )
            if current is not None and not current.empty
            else 0.0
        )

        result.at[
            idx, "early_season_player_current_fixtures"
        ] = current_fixtures
        result.at[
            idx, "early_season_player_archived_pl_fixtures"
        ] = archived_fixtures

        active = (
            current_fixtures > 0.0
            and current_fixtures < fade
            and archived_fixtures < stable_archived
        )
        if not active:
            continue

        result.at[
            idx, "early_season_player_shrinkage_active"
        ] = True
        result.at[
            idx, "early_season_player_current_weight"
        ] = min(1.0, current_fixtures / fade)

        position = str(position_by_key.get(player_key, ""))
        archived_prior_weight = min(
            1.0, archived_fixtures / stable_archived
        )

        for raw_column, feature_name in rate_map.items():
            targets: list[tuple[str, int | None]] = []
            if shrink_last_gw:
                targets.append(
                    (f"{feature_name}_last_gw", 1)
                )
            targets.extend(
                (
                    f"{feature_name}_last_{window}_gws",
                    int(window),
                )
                for window in ROLLING_GAMEWEEK_WINDOWS
            )
            targets.append((f"career_{feature_name}", None))

            for target_column, window in targets:
                if target_column not in result.columns:
                    continue

                current_rate, current_evidence = _safe_group_rate(
                    current,
                    raw_column,
                    window=window,
                )
                archived_rate, _ = _safe_group_rate(
                    archived,
                    raw_column,
                    window=window,
                )
                position_prior = (
                    position_priors
                    .get(target_column, {})
                    .get(position, np.nan)
                )

                if np.isfinite(archived_rate) and np.isfinite(position_prior):
                    prior = (
                        archived_prior_weight * archived_rate
                        + (1.0 - archived_prior_weight) * position_prior
                    )
                    prior_source = (
                        "archived_pl"
                        if archived_prior_weight >= 0.999
                        else "archived_pl+position"
                    )
                elif np.isfinite(archived_rate):
                    prior = archived_rate
                    prior_source = "archived_pl"
                elif np.isfinite(position_prior):
                    prior = position_prior
                    prior_source = "position"
                else:
                    prior = np.nan
                    prior_source = "none"

                if np.isfinite(current_rate) and np.isfinite(prior):
                    evidence = min(
                        current_fixtures,
                        max(float(current_evidence), 0.0),
                    )
                    current_weight = min(1.0, evidence / fade)
                    shrunk = (
                        current_weight * current_rate
                        + (1.0 - current_weight) * prior
                    )
                elif np.isfinite(current_rate):
                    shrunk = current_rate
                else:
                    shrunk = prior

                if np.isfinite(shrunk):
                    result.at[idx, target_column] = float(shrunk)
                    result.at[
                        idx, "early_season_player_prior_source"
                    ] = prior_source

    return result


def build_current_as_of_history_features(
    archived_player_gameweeks: pd.DataFrame,
    current_player_gameweeks: pd.DataFrame,
    current_players: pd.DataFrame,
) -> pd.DataFrame:
    """Create one latest pre-deadline history row per current player."""
    archived_columns = [
        "season",
        "season_start_year",
        "player_key",
        "gameweek",
    ] + CURRENT_GAMEWEEK_RAW_COLUMNS

    archived = archived_player_gameweeks.copy()
    for column in archived_columns:
        if column not in archived.columns:
            archived[column] = np.nan
    archived = archived[archived_columns]

    current = current_player_gameweeks.copy()
    for column in archived_columns:
        if column not in current.columns:
            current[column] = np.nan
    current = current[archived_columns]

    history_frames = [archived]
    if not current.empty:
        history_frames.append(current)

    history = (
        pd.concat(
            history_frames,
            ignore_index=True,
        )
        .sort_values(
            [
                "player_key",
                "season_start_year",
                "gameweek",
            ]
        )
        .reset_index(drop=True)
    )

    for column in [
        "gw_fixture_count",
        "gw_minutes",
        "gw_starts",
        "gw_appearance_count",
        "gw_60_plus_count",
    ]:
        history[column] = pd.to_numeric(
            history[column],
            errors="coerce",
        )

    current_keys = current_players[
        ["player_key", "position"]
    ].drop_duplicates("player_key")

    feature_rows: list[dict[str, object]] = []

    history_groups = {
        player_key: group.sort_values(
            ["season_start_year", "gameweek"]
        )
        for player_key, group in history.groupby(
            "player_key",
            sort=False,
        )
    }

    for player_record in current_keys.itertuples(index=False):
        player_key = player_record.player_key
        player_history = history_groups.get(player_key)

        features: dict[str, object] = {
            "player_key": player_key,
            "prior_gameweek_count": 0,
            "prior_season_fixture_count": 0.0,
            "prior_career_fixture_count": 0.0,
        }

        for suffix in [
            "minutes_per_fixture",
            "start_rate",
            "appearance_rate",
            "sixty_plus_rate",
        ]:
            features[f"{suffix}_last_gw"] = np.nan
            for window in ROLLING_GAMEWEEK_WINDOWS:
                features[f"{suffix}_last_{window}_gws"] = np.nan
            features[f"career_{suffix}"] = np.nan

        if player_history is not None and not player_history.empty:
            player_history = player_history.copy()
            fixture_count = player_history[
                "gw_fixture_count"
            ].fillna(0)

            features["prior_gameweek_count"] = len(player_history)
            features["prior_career_fixture_count"] = fixture_count.sum()
            features["prior_season_fixture_count"] = (
                player_history.loc[
                    player_history["season"].eq(CURRENT_SEASON),
                    "gw_fixture_count",
                ]
                .fillna(0)
                .sum()
            )

            last_fixture_count = fixture_count.iloc[-1]

            for raw_column, feature_name in (
                LIVE_HISTORY_RATE_COLUMNS.items()
            ):
                raw_values = player_history[raw_column].fillna(0)

                if last_fixture_count > 0:
                    features[
                        f"{feature_name}_last_gw"
                    ] = raw_values.iloc[-1] / last_fixture_count

                for window in ROLLING_GAMEWEEK_WINDOWS:
                    recent = player_history.tail(window)
                    recent_fixtures = recent[
                        "gw_fixture_count"
                    ].fillna(0).sum()

                    if recent_fixtures > 0:
                        features[
                            f"{feature_name}_last_{window}_gws"
                        ] = (
                            recent[raw_column].fillna(0).sum()
                            / recent_fixtures
                        )

                career_fixtures = fixture_count.sum()
                if career_fixtures > 0:
                    features[f"career_{feature_name}"] = (
                        raw_values.sum() / career_fixtures
                    )

        feature_rows.append(features)

    return pd.DataFrame(feature_rows)


current_as_of_history_unshrunk = build_current_as_of_history_features(
    historical_player_gameweeks,
    current_season_player_gameweeks,
    players,
)

current_as_of_history = apply_early_season_player_history_shrinkage(
    current_as_of_history_unshrunk,
    historical_player_gameweeks,
    current_season_player_gameweeks,
    players,
    LIVE_HISTORY_RATE_COLUMNS,
    shrink_last_gw=bool(EARLY_SEASON_SHRINK_LAST_GW_ROLE),
)

print(
    "Current players with as-of history rows: "
    f"{len(current_as_of_history):,}"
)
print(
    "Sparse-player early-season shrinkage active for:",
    int(
        current_as_of_history[
            "early_season_player_shrinkage_active"
        ].sum()
    ),
    "players",
)


### Build live feature rows and apply availability information

The statistical model estimates normal selection risk. Official FPL availability information is then applied as a separate, visible adjustment for the next Gameweek.

For later Gameweeks in the six-week horizon, injury recovery is not yet modelled; only players marked permanently unavailable remain at zero. This limitation is preferable to silently assuming a recovery date.

In [ ]:
def next_gameweek_availability_multiplier(
    data: pd.DataFrame,
    first_gameweek: int,
) -> pd.Series:
    """Translate official availability fields into a next-GW multiplier."""
    multiplier = pd.Series(
        1.0,
        index=data.index,
        dtype=float,
    )

    status = data["status"].fillna("a").astype(str)
    chance = pd.to_numeric(
        data["chance_of_playing_next_round"],
        errors="coerce",
    ).div(100).clip(0.0, 1.0)

    next_gameweek_rows = data["gameweek"].eq(first_gameweek)

    explicit_chance = next_gameweek_rows & chance.notna()
    multiplier.loc[explicit_chance] = chance.loc[explicit_chance]

    no_chance_and_unavailable = (
        next_gameweek_rows
        & chance.isna()
        & status.isin(["i", "s", "u"])
    )
    multiplier.loc[no_chance_and_unavailable] = 0.0

    # FPL's unavailable status is treated as persistent throughout the
    # planning horizon until the source data changes.
    multiplier.loc[status.eq("u")] = 0.0

    return multiplier


live_minutes_feature_data = (
    player_fixtures
    .merge(
        current_as_of_history,
        on="player_key",
        how="left",
        validate="many_to_one",
    )
    .merge(
        current_external_player_priors,
        on="player_key",
        how="left",
        validate="many_to_one",
    )
    .rename(columns={"is_home": "was_home"})
    .copy()
)

live_minutes_feature_data["short_league_rest"] = (
    live_minutes_feature_data[
        "days_since_previous_league_fixture"
    ].le(4)
)

missing_live_features = (
    set(MINUTES_FEATURE_COLUMNS)
    - set(live_minutes_feature_data.columns)
)
assert not missing_live_features, (
    "Live minutes data is missing model features: "
    f"{sorted(missing_live_features)}"
)

live_component_predictions = predict_minutes_components(
    minutes_models,
    live_minutes_feature_data,
)

live_minutes_predictions = pd.concat(
    [
        live_minutes_feature_data.reset_index(drop=True),
        live_component_predictions.reset_index(drop=True),
    ],
    axis=1,
)

live_minutes_predictions["availability_multiplier"] = (
    next_gameweek_availability_multiplier(
        live_minutes_predictions,
        first_planning_gameweek,
    )
)

live_minutes_predictions["model_appearance_probability"] = (
    live_minutes_predictions["appearance_probability"]
)

live_minutes_predictions["appearance_probability"] = (
    live_minutes_predictions["appearance_probability"]
    * live_minutes_predictions["availability_multiplier"]
)

live_minutes_predictions["start_probability"] = np.minimum(
    live_minutes_predictions["start_probability"]
    * live_minutes_predictions["availability_multiplier"],
    live_minutes_predictions["appearance_probability"],
)

live_minutes_predictions[
    "substitute_appearance_probability"
] = np.maximum(
    live_minutes_predictions["appearance_probability"]
    - live_minutes_predictions["start_probability"],
    0.0,
)

live_minutes_predictions["no_appearance_probability"] = (
    1.0 - live_minutes_predictions["appearance_probability"]
)

live_minutes_predictions["sixty_plus_probability"] = np.minimum(
    live_minutes_predictions["sixty_plus_probability"]
    * live_minutes_predictions["availability_multiplier"],
    live_minutes_predictions["start_probability"],
)

live_minutes_predictions["expected_minutes"] = (
    live_minutes_predictions["start_probability"]
    * live_minutes_predictions["minutes_if_start"]
    + live_minutes_predictions[
        "substitute_appearance_probability"
    ]
    * live_minutes_predictions["minutes_if_sub"]
).clip(0.0, 90.0)

live_minutes_predictions["rotation_risk"] = (
    1.0 - live_minutes_predictions["start_probability"]
)

assert live_minutes_predictions["expected_minutes"].between(
    0.0,
    90.0,
).all()

assert (
    live_minutes_predictions["start_probability"]
    <= live_minutes_predictions["appearance_probability"] + 1e-9
).all()

assert (
    live_minutes_predictions["sixty_plus_probability"]
    <= live_minutes_predictions["start_probability"] + 1e-9
).all()


### Inspect upcoming minutes and rotation projections

`rotation_risk` is currently defined as `1 - P(start)`. It is retained as a continuous value rather than converted into arbitrary labels.

In [ ]:
live_minutes_display_columns = [
    "gameweek",
    "kickoff_time_spain",
    "web_name",
    "team_short_name",
    "opponent_short_name",
    "venue",
    "position",
    "status",
    "chance_of_playing_next_round",
    "start_probability",
    "substitute_appearance_probability",
    "appearance_probability",
    "sixty_plus_probability",
    "expected_minutes",
    "rotation_risk",
]

live_minutes_preview = (
    live_minutes_predictions
    .sort_values(
        [
            "gameweek",
            "expected_minutes",
            "web_name",
        ],
        ascending=[True, False, True],
    )
    .loc[:, live_minutes_display_columns]
    .head(40)
)

display(live_minutes_preview)


### Result of the minutes stage

The notebook now produces, for every player and upcoming fixture:

```text
P(appearance)
P(start)
P(substitute appearance)
P(no appearance)
P(60+ minutes)
Expected minutes if starting
Expected minutes if used as a substitute
Expected minutes
Rotation risk
```

The principal object for the next stage is:

```python
live_minutes_predictions
```

The next stage is the expected-points model. It will combine these minutes projections with player attacking rates, team scoring and clean-sheet probabilities, goalkeeper saves, defensive contributions, bonus potential, and expected deductions.

## 6. Component-level expected-points baseline

This section converts the minutes forecasts into fixture-level expected FPL points.

The first version remains deliberately transparent. It estimates the principal scoring components separately:

1. appearance points;
2. goals;
3. assists;
4. clean sheets;
5. goalkeeper save points;
6. goals-conceded deductions;
7. defensive-contribution points;
8. bonus points;
9. card and rare-event deductions.

The component structure mirrors the official FPL scoring rules. In 2026/27, defenders earn two defensive-contribution points after 10 qualifying actions, while midfielders and forwards require 12. Bonus points remain worth one to three points, although the underlying Bonus Points System changed for 2026/27. The historical bonus model below is therefore a baseline that should be recalibrated as current-season matches accumulate.

Official references:

- https://www.premierleague.com/en/news/2174909/fpl-basics-scoring
- https://www.premierleague.com/en/news/4361991/whats-happening-with-defensive-contribution-points-in-202627-fantasy
- https://www.premierleague.com/en/news/4679946/whats-new-in-202627-fantasy-changes-to-bonus-points-system

The output is produced at player–fixture level and then aggregated to player–Gameweek level, which preserves blank and double Gameweeks.

### Scoring configuration and modelling imports

The model uses Poisson regressions for non-negative event rates. These models are interpretable and ensure that predicted rates cannot become negative.

In [ ]:
from scipy.stats import poisson, spearmanr
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_poisson_deviance



GOAL_POINTS_BY_POSITION = {
    "GK": 10.0,
    "DEF": 6.0,
    "MID": 5.0,
    "FWD": 4.0,
}

CLEAN_SHEET_POINTS_BY_POSITION = {
    "GK": 4.0,
    "DEF": 4.0,
    "MID": 1.0,
    "FWD": 0.0,
}

DEFENSIVE_CONTRIBUTION_THRESHOLD = {
    "GK": np.inf,
    "DEF": 10,
    "MID": 12,
    "FWD": 12,
}


def normalise_scoring_position(series: pd.Series) -> pd.Series:
    """Use one position encoding across archived and live FPL data."""
    return (
        series
        .astype("string")
        .replace({"GKP": "GK"})
    )


### Leakage-safe team attack and defence features

Player history alone does not describe the opponent. The following table creates pre-fixture team-strength features from completed scores:

- goals scored per match;
- goals conceded per match;
- clean-sheet rate;
- recent five- and ten-match windows;
- a longer-run career rate.

Every statistic is shifted before rolling, so the fixture being predicted never contributes to its own features. The opponent's corresponding features are then joined to the same row.

In [ ]:
TEAM_ROLLING_WINDOWS = [5, 10]


def historical_opponent_team_key(data: pd.DataFrame) -> pd.Series:
    """Construct a stable opponent key from archived team codes."""
    opponent_code = pd.to_numeric(
        data.get("opponent_code"),
        errors="coerce",
    ).astype("Int64")

    fallback = (
        data["season"].astype(str)
        + ":team:"
        + pd.to_numeric(
            data.get("opponent_id"),
            errors="coerce",
        ).astype("Int64").astype(str)
    )

    result = fallback.astype("string")
    has_code = opponent_code.notna()
    result.loc[has_code] = (
        "code:" + opponent_code.loc[has_code].astype(str)
    )
    return result


def build_historical_team_fixture_features(
    historical_data: pd.DataFrame,
) -> pd.DataFrame:
    """
    Create one leakage-safe team-strength row per team and fixture.

    v2.2 augments goals/clean-sheet history with team xG/xGA whenever the FPL
    archive supplies player expected_goals. The xG inputs are aggregated from
    completed fixture outcomes and then shifted before rolling, so they remain
    deadline-safe.
    """
    required = [
        "season",
        "season_start_year",
        "gameweek",
        "fixture_id",
        "kickoff_time",
        "team_key",
        "team_code",
        "opponent_id",
        "opponent_code",
        "was_home",
        "team_goals",
        "opponent_goals",
    ]

    missing = set(required) - set(historical_data.columns)
    if missing:
        raise KeyError(
            "Historical data is missing team-result columns: "
            f"{sorted(missing)}"
        )

    team_results = (
        historical_data[required]
        .drop_duplicates(["season", "fixture_id", "team_key"])
        .copy()
    )

    for column in ["gameweek", "team_goals", "opponent_goals"]:
        team_results[column] = pd.to_numeric(
            team_results[column], errors="coerce"
        )

    team_results["was_home"] = normalise_boolean(
        team_results["was_home"]
    )
    team_results["opponent_team_key"] = (
        historical_opponent_team_key(team_results)
    )
    team_results["team_clean_sheet"] = (
        team_results["opponent_goals"].eq(0).astype(float)
    )

    # Aggregate player xG into team xG for archives that contain the metric.
    if "expected_goals" in historical_data.columns:
        xg = historical_data[
            ["season", "fixture_id", "team_key", "expected_goals"]
        ].copy()
        xg["expected_goals"] = pd.to_numeric(
            xg["expected_goals"], errors="coerce"
        )
        xg = (
            xg.groupby(
                ["season", "fixture_id", "team_key"],
                as_index=False,
                dropna=False,
            )
            .agg(team_xg=("expected_goals", lambda s: s.sum(min_count=1)))
        )
        team_results = team_results.merge(
            xg,
            on=["season", "fixture_id", "team_key"],
            how="left",
            validate="one_to_one",
        )
        opponent_xg = xg.rename(
            columns={
                "team_key": "opponent_team_key",
                "team_xg": "opponent_xg",
            }
        )
        team_results = team_results.merge(
            opponent_xg,
            on=["season", "fixture_id", "opponent_team_key"],
            how="left",
            validate="one_to_one",
        )
    else:
        team_results["team_xg"] = np.nan
        team_results["opponent_xg"] = np.nan

    team_results = (
        team_results
        .sort_values(["team_key", "kickoff_time", "fixture_id"])
        .reset_index(drop=True)
    )

    grouped = team_results.groupby(
        "team_key",
        group_keys=False,
        sort=False,
    )

    source_columns = {
        "team_goals": "goals_for",
        "opponent_goals": "goals_against",
        "team_clean_sheet": "clean_sheet_rate",
    }
    if team_results["team_xg"].notna().any():
        source_columns["team_xg"] = "xg_for"
        source_columns["opponent_xg"] = "xg_against"

    team_feature_columns: list[str] = []

    for window in TEAM_ROLLING_WINDOWS:
        for source, label in source_columns.items():
            output = f"team_{label}_last_{window}"
            team_results[output] = grouped[source].transform(
                lambda values: (
                    values.shift(1)
                    .rolling(window=window, min_periods=1)
                    .mean()
                )
            )
            team_feature_columns.append(output)

    for source, label in source_columns.items():
        output = f"team_career_{label}"
        team_results[output] = grouped[source].transform(
            lambda values: (
                values.shift(1)
                .expanding(min_periods=1)
                .mean()
            )
        )
        team_feature_columns.append(output)

    opponent_features = (
        team_results[
            ["season", "fixture_id", "team_key"] + team_feature_columns
        ]
        .rename(columns={
            "team_key": "opponent_team_key",
            **{
                column: column.replace("team_", "opponent_", 1)
                for column in team_feature_columns
            },
        })
    )

    team_results = team_results.merge(
        opponent_features,
        on=["season", "fixture_id", "opponent_team_key"],
        how="left",
        validate="one_to_one",
    )

    return team_results


historical_team_fixtures = (
    build_historical_team_fixture_features(historical_fpl)
)

print(
    "Historical team–fixture rows: "
    f"{len(historical_team_fixtures):,}"
)

### Team scoring models

Two related models are fitted:

- expected goals scored by the player's team;
- expected goals conceded by the player's team.

The holdout season remains 2025/26. A home/away league-average forecast is included as a baseline.

In [ ]:
TEAM_FEATURE_COLUMNS = [
    "gameweek",
    "was_home",
    "team_goals_for_last_5",
    "team_goals_against_last_5",
    "team_clean_sheet_rate_last_5",
    "team_goals_for_last_10",
    "team_goals_against_last_10",
    "team_clean_sheet_rate_last_10",
    "team_career_goals_for",
    "team_career_goals_against",
    "team_career_clean_sheet_rate",
    "opponent_goals_for_last_5",
    "opponent_goals_against_last_5",
    "opponent_clean_sheet_rate_last_5",
    "opponent_goals_for_last_10",
    "opponent_goals_against_last_10",
    "opponent_clean_sheet_rate_last_10",
    "opponent_career_goals_for",
    "opponent_career_goals_against",
    "opponent_career_clean_sheet_rate",
]

# Prefer xG-based strength signals when archive coverage exists.
_XG_TEAM_FEATURE_COLUMNS = [
    *[
        f"{side}_{metric}_{suffix}"
        for side in ("team", "opponent")
        for metric in ("xg_for", "xg_against")
        for suffix in ("last_5", "last_10")
    ],
    *[
        f"{side}_career_{metric}"
        for side in ("team", "opponent")
        for metric in ("xg_for", "xg_against")
    ],
]
TEAM_FEATURE_COLUMNS += [
    column
    for column in _XG_TEAM_FEATURE_COLUMNS
    if column in historical_team_fixtures.columns
]


def make_nonnegative_numeric_preprocessor(
    feature_columns: list[str],
) -> ColumnTransformer:
    """Preprocess an all-numeric feature table."""
    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline([
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median",
                            add_indicator=True,
                        ),
                    ),
                    ("scaler", StandardScaler()),
                ]),
                feature_columns,
            )
        ],
        remainder="drop",
        sparse_threshold=0,
    )


def make_team_goal_model() -> Pipeline:
    """Return a regularised Poisson model for team goal counts."""
    return Pipeline([
        (
            "preprocess",
            make_nonnegative_numeric_preprocessor(
                TEAM_FEATURE_COLUMNS
            ),
        ),
        (
            "model",
            PoissonRegressor(
                alpha=1.0,
                max_iter=2_000,
            ),
        ),
    ])


def fit_team_goal_models(
    training_team_fixtures: pd.DataFrame,
) -> dict[str, Pipeline]:
    """Fit expected goals-for and goals-against models."""
    training = training_team_fixtures.loc[
        training_team_fixtures["team_goals"].notna()
        & training_team_fixtures["opponent_goals"].notna()
    ].copy()

    if training.empty:
        raise ValueError("No completed team fixtures are available.")

    models: dict[str, Pipeline] = {}
    training_features = prepare_model_feature_frame(
        training,
        TEAM_FEATURE_COLUMNS,
        context="team_goals.fit",
    )

    for name, target in {
        "goals_for": "team_goals",
        "goals_against": "opponent_goals",
    }.items():
        model = make_team_goal_model()
        model.fit(
            training_features,
            training[target].clip(lower=0),
        )
        models[name] = model

    return models


def predict_team_goal_components(
    model_bundle: dict[str, Pipeline],
    feature_data: pd.DataFrame,
) -> pd.DataFrame:
    """Predict coherent non-negative team scoring expectations."""
    features = prepare_model_feature_frame(
        feature_data,
        TEAM_FEATURE_COLUMNS,
        context="team_goals.predict",
    )

    expected_for = np.clip(
        model_bundle["goals_for"].predict(
            features
        ),
        0.05,
        6.0,
    )
    expected_against = np.clip(
        model_bundle["goals_against"].predict(
            features
        ),
        0.05,
        6.0,
    )

    return pd.DataFrame(
        {
            "expected_team_goals": expected_for,
            "expected_team_goals_conceded": expected_against,
            "team_clean_sheet_probability": np.exp(
                -expected_against
            ),
        },
        index=feature_data.index,
    )


team_development = historical_team_fixtures.loc[
    historical_team_fixtures["season"].isin(
        DEVELOPMENT_SEASONS
    )
].copy()

team_holdout = historical_team_fixtures.loc[
    historical_team_fixtures["season"].eq(HOLDOUT_SEASON)
].copy()

team_goal_models = fit_team_goal_models(team_development)

team_holdout_predictions = pd.concat(
    [
        team_holdout.reset_index(drop=True),
        predict_team_goal_components(
            team_goal_models,
            team_holdout,
        ).reset_index(drop=True),
    ],
    axis=1,
)

home_away_goal_priors = (
    team_development
    .groupby("was_home")
    .agg(
        baseline_team_goals=("team_goals", "mean"),
        baseline_goals_conceded=("opponent_goals", "mean"),
    )
)

team_holdout_predictions = team_holdout_predictions.join(
    home_away_goal_priors,
    on="was_home",
)

team_metric_rows = []
for target, model_column, baseline_column in [
    (
        "team_goals",
        "expected_team_goals",
        "baseline_team_goals",
    ),
    (
        "opponent_goals",
        "expected_team_goals_conceded",
        "baseline_goals_conceded",
    ),
]:
    valid = team_holdout_predictions[target].notna()
    actual = team_holdout_predictions.loc[valid, target]

    for method, prediction_column in [
        ("home/away baseline", baseline_column),
        ("team-strength model", model_column),
    ]:
        predicted = team_holdout_predictions.loc[
            valid,
            prediction_column,
        ].clip(lower=PROBABILITY_EPSILON)

        team_metric_rows.append({
            "target": target,
            "method": method,
            "mae": mean_absolute_error(actual, predicted),
            "poisson_deviance": mean_poisson_deviance(
                actual,
                predicted,
            ),
        })

team_goal_scorecard = pd.DataFrame(team_metric_rows)
display(team_goal_scorecard)

### Player event-rate features

Attacking and secondary event models predict rates per 90 minutes. The final expected event count is the predicted rate multiplied by expected minutes.

Historical per-90 features use only shifted rolling totals. Very short appearances receive lower training weights, which limits the influence of unstable per-90 values from one- or two-minute cameos.

In [ ]:
PLAYER_RATE_SOURCES = {
    "points": "points_per_fixture",
    "goals": "goals_per_fixture",
    "assists": "assists_per_fixture",
    "xg": "xg_per_fixture",
    "xa": "xa_per_fixture",
    "saves": "saves_per_fixture",
    "bonus": "bonus_per_fixture",
    "bps": "bps_per_fixture",
    "yellow_cards": "yellow_cards_per_fixture",
    "red_cards": "red_cards_per_fixture",
    "threat": "threat_per_fixture",
    "creativity": "creativity_per_fixture",
    "influence": "influence_per_fixture",
    "tackles": "tackles_per_fixture",
    "recoveries": "recoveries_per_fixture",
    "cbi": "cbi_per_fixture",
    "defensive_contribution": (
        "defensive_contribution_per_fixture"
    ),
}


def safe_per90(
    event_rate_per_fixture: pd.Series,
    minutes_per_fixture: pd.Series,
) -> pd.Series:
    """Convert a per-fixture rate to a stable per-90 rate."""
    event_rate = pd.to_numeric(
        event_rate_per_fixture,
        errors="coerce",
    )
    minutes_rate = pd.to_numeric(
        minutes_per_fixture,
        errors="coerce",
    )

    result = 90.0 * event_rate / minutes_rate
    return result.where(minutes_rate.gt(0))


def add_player_per90_history_features(
    data: pd.DataFrame,
) -> pd.DataFrame:
    """Derive rolling and career per-90 features without using targets."""
    result = data.copy()

    for output_prefix, source_prefix in PLAYER_RATE_SOURCES.items():
        for window in ROLLING_GAMEWEEK_WINDOWS:
            source = f"{source_prefix}_last_{window}_gws"
            minutes = f"minutes_per_fixture_last_{window}_gws"
            output = f"{output_prefix}_per90_last_{window}_gws"

            if source not in result.columns:
                result[source] = np.nan
            if minutes not in result.columns:
                result[minutes] = np.nan

            result[output] = safe_per90(
                result[source],
                result[minutes],
            )

        career_source = f"career_{source_prefix}"
        career_minutes = "career_minutes_per_fixture"
        career_output = f"career_{output_prefix}_per90"

        if career_source not in result.columns:
            result[career_source] = np.nan
        if career_minutes not in result.columns:
            result[career_minutes] = np.nan

        result[career_output] = safe_per90(
            result[career_source],
            result[career_minutes],
        )

    result["model_position"] = normalise_scoring_position(
        result["position"]
    )

    return result


def add_player_event_targets(
    data: pd.DataFrame,
) -> pd.DataFrame:
    """Create per-90 training targets from completed fixture outcomes."""
    result = data.copy()
    minutes = pd.to_numeric(
        result["target_minutes"],
        errors="coerce",
    )

    event_columns = {
        "xg": "expected_goals",
        "xa": "expected_assists",
        "saves": "saves",
        "bonus": "bonus",
        "yellow_cards": "yellow_cards",
        "red_cards": "red_cards",
    }

    for label, source in event_columns.items():
        if source not in result.columns:
            result[source] = np.nan

        values = pd.to_numeric(
            result[source],
            errors="coerce",
        )
        result[f"target_{label}_per90"] = (
            90.0 * values / minutes
        ).where(minutes.gt(0))

    tackles = pd.to_numeric(
        result.get("tackles"),
        errors="coerce",
    )
    recoveries = pd.to_numeric(
        result.get("recoveries"),
        errors="coerce",
    )
    cbi = pd.to_numeric(
        result.get("clearances_blocks_interceptions"),
        errors="coerce",
    )

    base_actions = tackles.fillna(0) + cbi.fillna(0)
    midfield_forward_actions = (
        base_actions + recoveries.fillna(0)
    )

    has_action_data = (
        tackles.notna() | recoveries.notna() | cbi.notna()
    )

    result["target_defensive_actions"] = np.where(
        result["model_position"].isin(["MID", "FWD"]),
        midfield_forward_actions,
        base_actions,
    )
    result.loc[
        ~has_action_data,
        "target_defensive_actions",
    ] = np.nan

    result["target_defensive_actions_per90"] = (
        90.0 * result["target_defensive_actions"] / minutes
    ).where(minutes.gt(0))

    return result


### Fit player event-rate models

The initial model predicts expected goals, expected assists, saves, bonus and card rates. Expected goals and assists are used instead of realised goals and assists because they provide less noisy training targets.

The bonus model is intentionally labelled a baseline: historical bonus outcomes were produced under older BPS rules.

In [ ]:
PLAYER_EVENT_FEATURE_COLUMNS = [
    "gameweek",
    "was_home",
    "model_position",
    "prior_season_fixture_count",
    "prior_career_fixture_count",
    "expected_team_goals",
    "expected_team_goals_conceded",
    "points_per90_last_5_gws",
    "points_per90_last_10_gws",
    "career_points_per90",
    "goals_per90_last_5_gws",
    "goals_per90_last_10_gws",
    "career_goals_per90",
    "assists_per90_last_5_gws",
    "assists_per90_last_10_gws",
    "career_assists_per90",
    "xg_per90_last_3_gws",
    "xg_per90_last_5_gws",
    "xg_per90_last_10_gws",
    "career_xg_per90",
    "xa_per90_last_3_gws",
    "xa_per90_last_5_gws",
    "xa_per90_last_10_gws",
    "career_xa_per90",
    "threat_per90_last_5_gws",
    "threat_per90_last_10_gws",
    "career_threat_per90",
    "creativity_per90_last_5_gws",
    "creativity_per90_last_10_gws",
    "career_creativity_per90",
    "saves_per90_last_5_gws",
    "saves_per90_last_10_gws",
    "career_saves_per90",
    "bonus_per90_last_5_gws",
    "bonus_per90_last_10_gws",
    "career_bonus_per90",
    "bps_per90_last_5_gws",
    "bps_per90_last_10_gws",
    "career_bps_per90",
    "yellow_cards_per90_last_10_gws",
    "career_yellow_cards_per90",
    "red_cards_per90_last_10_gws",
    "career_red_cards_per90",
] + EXTERNAL_EVENT_MODEL_FEATURES

PLAYER_EVENT_CATEGORICAL_FEATURES = ["model_position"]
PLAYER_EVENT_NUMERIC_FEATURES = [
    column
    for column in PLAYER_EVENT_FEATURE_COLUMNS
    if column not in PLAYER_EVENT_CATEGORICAL_FEATURES
]

PLAYER_RATE_MODEL_SPECIFICATIONS = {
    "xg": {
        "target": "target_xg_per90",
        # Goalkeeper attacking returns are too rare to estimate reliably.
        "positions": ["DEF", "MID", "FWD"],
        "alpha": 0.4,
        "upper_clip": 2.0,
        "position_specific": True,
    },
    "xa": {
        "target": "target_xa_per90",
        # Goalkeeper attacking returns are treated as zero in the baseline.
        "positions": ["DEF", "MID", "FWD"],
        "alpha": 0.4,
        "upper_clip": 2.0,
        "position_specific": True,
    },
    "saves": {
        "target": "target_saves_per90",
        "positions": ["GK"],
        "alpha": 2.0,
        "upper_clip": 12.0,
    },
    "bonus": {
        "target": "target_bonus_per90",
        "positions": PLAYER_SCORING_POSITIONS,
        "alpha": 2.0,
        "upper_clip": 3.0,
    },
    "yellow_cards": {
        "target": "target_yellow_cards_per90",
        "positions": PLAYER_SCORING_POSITIONS,
        "alpha": 4.0,
        "upper_clip": 1.5,
    },
    "red_cards": {
        "target": "target_red_cards_per90",
        "positions": PLAYER_SCORING_POSITIONS,
        "alpha": 10.0,
        "upper_clip": 0.5,
    },
}


def observed_numeric_features(
    frame: pd.DataFrame,
    candidate_features: list[str],
) -> tuple[list[str], list[str]]:
    """
    Split numeric features into usable and entirely-missing columns.

    Median imputation cannot learn a value for a feature that is NaN for every
    training row. Rather than letting scikit-learn silently drop that feature,
    each fitted model explicitly excludes such columns. This is especially
    important for external new-signing priors because early walk-forward folds
    or position-specific subsets may legitimately have no external observations.
    """
    active = []
    all_missing = []

    for column in candidate_features:
        if column not in frame.columns:
            all_missing.append(column)
            continue

        values = pd.to_numeric(
            frame[column],
            errors="coerce",
        )
        if values.notna().any():
            active.append(column)
        else:
            all_missing.append(column)

    return active, all_missing


def make_player_event_preprocessor(
    numeric_features: list[str],
) -> ColumnTransformer:
    """Build a fresh mixed-type preprocessing pipeline."""
    if not numeric_features:
        raise ValueError(
            "No observed numeric features are available for this event model."
        )

    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
        ),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        ("one_hot", make_dense_one_hot_encoder()),
    ])

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_features,
            ),
            (
                "categorical",
                categorical_pipeline,
                PLAYER_EVENT_CATEGORICAL_FEATURES,
            ),
        ],
        remainder="drop",
        sparse_threshold=0,
    )


def _fit_single_player_rate_model(
    model_data: pd.DataFrame,
    target_column: str,
    alpha: float,
    upper_clip: float,
) -> dict[str, object]:
    """Fit one weighted Poisson rate model or return a stable constant."""
    if model_data.empty:
        return {
            "model": None,
            "constant": 0.0,
            "active_numeric_features": [],
            "dropped_all_missing_features": list(
                PLAYER_EVENT_NUMERIC_FEATURES
            ),
        }

    active_numeric_features, dropped_all_missing_features = (
        observed_numeric_features(
            model_data,
            PLAYER_EVENT_NUMERIC_FEATURES,
        )
    )

    target = pd.to_numeric(
        model_data[target_column],
        errors="coerce",
    ).clip(lower=0.0, upper=float(upper_clip))

    sample_weight = (
        pd.to_numeric(
            model_data["target_minutes"],
            errors="coerce",
        )
        .div(90.0)
        .clip(lower=0.05, upper=1.0)
    )

    weighted_mean = float(np.average(target, weights=sample_weight))
    if np.isclose(target.sum(), 0.0):
        return {
            "model": None,
            "constant": weighted_mean,
            "active_numeric_features": active_numeric_features,
            "dropped_all_missing_features": dropped_all_missing_features,
        }

    model = Pipeline([
        (
            "preprocess",
            make_player_event_preprocessor(
                active_numeric_features,
            ),
        ),
        (
            "model",
            PoissonRegressor(
                alpha=float(alpha),
                max_iter=2_000,
            ),
        ),
    ])
    model.fit(
        prepare_model_feature_frame(
            model_data,
            PLAYER_EVENT_FEATURE_COLUMNS,
            context=f"player_events.fit.{target_column}",
            categorical_columns=PLAYER_EVENT_CATEGORICAL_FEATURES,
        ),
        target,
        model__sample_weight=sample_weight,
    )
    return {
        "model": model,
        "constant": weighted_mean,
        "active_numeric_features": active_numeric_features,
        "dropped_all_missing_features": dropped_all_missing_features,
    }


def fit_player_event_rate_models(
    training_data: pd.DataFrame,
) -> dict[str, dict[str, object]]:
    """
    Fit non-negative event-rate models.

    xG and xA are fitted separately by FPL position. A single strongly
    regularised pooled model was shrinking defenders and forwards toward the
    same rate, which inflated defender attacking returns and suppressed
    forwards. Other low-frequency components retain pooled models.
    """
    model_bundle: dict[str, dict[str, object]] = {}

    for name, specification in PLAYER_RATE_MODEL_SPECIFICATIONS.items():
        target_column = specification["target"]
        positions = list(specification["positions"])
        upper_clip = float(specification["upper_clip"])
        alpha = float(specification["alpha"])
        position_specific = bool(
            specification.get("position_specific", False)
        )

        valid = training_data.loc[
            training_data["model_position"].isin(positions)
            & training_data[target_column].notna()
            & training_data["target_minutes"].gt(0)
        ].copy()

        entry: dict[str, object] = {
            "positions": positions,
            "upper_clip": upper_clip,
            "position_specific": position_specific,
        }

        if position_specific:
            models_by_position = {}
            constants_by_position = {}
            for scoring_position in positions:
                position_data = valid.loc[
                    valid["model_position"].eq(scoring_position)
                ].copy()
                fitted = _fit_single_player_rate_model(
                    position_data,
                    target_column,
                    alpha,
                    upper_clip,
                )
                models_by_position[scoring_position] = fitted["model"]
                constants_by_position[scoring_position] = float(
                    fitted["constant"]
                )
                entry.setdefault(
                    "active_numeric_features_by_position",
                    {},
                )[scoring_position] = fitted[
                    "active_numeric_features"
                ]
                entry.setdefault(
                    "dropped_all_missing_features_by_position",
                    {},
                )[scoring_position] = fitted[
                    "dropped_all_missing_features"
                ]
            entry["models_by_position"] = models_by_position
            entry["constants_by_position"] = constants_by_position
        else:
            fitted = _fit_single_player_rate_model(
                valid,
                target_column,
                alpha,
                upper_clip,
            )
            entry["model"] = fitted["model"]
            entry["constant"] = float(fitted["constant"])
            entry["active_numeric_features"] = fitted[
                "active_numeric_features"
            ]
            entry["dropped_all_missing_features"] = fitted[
                "dropped_all_missing_features"
            ]

        model_bundle[name] = entry

    return model_bundle



def predict_player_event_rates(
    model_bundle: dict[str, dict[str, object]],
    feature_data: pd.DataFrame,
) -> pd.DataFrame:
    """
    Predict finite position-aware per-90 rates.

    Previous-season external xG/xA/save priors are blended only for players
    with little Premier League history. Their influence fades as real PL
    observations accumulate.
    """
    predictions = pd.DataFrame(index=feature_data.index)
    model_features = prepare_model_feature_frame(
        feature_data,
        PLAYER_EVENT_FEATURE_COLUMNS,
        context="player_events.predict",
        categorical_columns=PLAYER_EVENT_CATEGORICAL_FEATURES,
    )
    external_weight = external_new_player_weight(
        feature_data,
        EXTERNAL_MAX_EVENT_BLEND,
    )

    external_targets = {
        "xg": "external_pl_equivalent_xg_per90",
        "xa": "external_pl_equivalent_xa_per90",
        "saves": "external_pl_equivalent_saves_per90",
    }

    for name, specification in model_bundle.items():
        positions = list(specification["positions"])
        upper_clip = float(specification["upper_clip"])
        values = np.zeros(len(feature_data), dtype=float)

        if specification.get("position_specific", False):
            for scoring_position in positions:
                allowed = model_features["model_position"].eq(
                    scoring_position
                )
                if not allowed.any():
                    continue
                model = specification["models_by_position"].get(
                    scoring_position
                )
                constant = float(
                    specification["constants_by_position"].get(
                        scoring_position,
                        0.0,
                    )
                )
                if model is None:
                    raw_prediction = np.full(
                        int(allowed.sum()),
                        constant,
                        dtype=float,
                    )
                else:
                    raw_prediction = model.predict(
                        model_features.loc[allowed]
                    )
                safe_prediction = np.nan_to_num(
                    raw_prediction,
                    nan=constant,
                    posinf=upper_clip,
                    neginf=0.0,
                )
                values[allowed.to_numpy()] = np.clip(
                    safe_prediction,
                    0.0,
                    upper_clip,
                )
        else:
            allowed = model_features["model_position"].isin(positions)
            model = specification.get("model")
            constant = float(specification.get("constant", 0.0))
            if model is None:
                values[allowed.to_numpy()] = np.clip(
                    constant,
                    0.0,
                    upper_clip,
                )
            elif allowed.any():
                raw_prediction = model.predict(
                    model_features.loc[allowed]
                )
                safe_prediction = np.nan_to_num(
                    raw_prediction,
                    nan=constant,
                    posinf=upper_clip,
                    neginf=0.0,
                )
                values[allowed.to_numpy()] = np.clip(
                    safe_prediction,
                    0.0,
                    upper_clip,
                )

        # External priors must obey the same positional domain as the
        # fitted event model. For example, xG/xA are deliberately modelled
        # only for DEF/MID/FWD, while saves are modelled only for GK.
        #
        # Without this mask, an external source can reintroduce a tiny xG/xA
        # rate for a goalkeeper after the base model has correctly set it to
        # zero.
        eligible_position = (
            model_features["model_position"]
            .isin(positions)
            .to_numpy(dtype=bool)
        )

        target_column = external_targets.get(name)
        if (
            target_column is not None
            and target_column in feature_data.columns
        ):
            external_target = pd.to_numeric(
                feature_data[target_column],
                errors="coerce",
            ).to_numpy(dtype=float)
            weight = external_weight.to_numpy(dtype=float)
            valid_external = np.isfinite(external_target)

            effective_weight = np.where(
                valid_external & eligible_position,
                weight,
                0.0,
            )

            values = (
                (1.0 - effective_weight) * values
                + effective_weight
                * np.nan_to_num(
                    external_target,
                    nan=0.0,
                    posinf=upper_clip,
                    neginf=0.0,
                )
            )

        # Structural safeguard: positions outside the event specification
        # always receive an exact zero, even after any external-prior blend.
        values[~eligible_position] = 0.0

        predictions[f"predicted_{name}_per90"] = np.clip(
            values,
            0.0,
            upper_clip,
        )

    predictions["external_event_prior_weight"] = (
        external_weight.to_numpy(dtype=float)
    )
    return predictions



In [ ]:
def external_event_feature_availability_audit(
    model_bundle: dict[str, dict[str, object]],
) -> pd.DataFrame:
    """
    Show which external event features were usable in each fitted model.

    A dropped feature is not an error: it means the relevant training subset
    contained no observed values for that feature. The explicit live
    new-signing blend still operates independently when a matched external
    prior is available.
    """
    rows = []

    for model_name, specification in model_bundle.items():
        if specification.get("position_specific", False):
            positions = specification.get("positions", [])
            for position in positions:
                active = specification.get(
                    "active_numeric_features_by_position",
                    {},
                ).get(position, [])
                dropped = specification.get(
                    "dropped_all_missing_features_by_position",
                    {},
                ).get(position, [])
                rows.append({
                    "model": model_name,
                    "position": position,
                    "external_xg_used": (
                        "external_pl_equivalent_xg_per90" in active
                    ),
                    "external_xa_used": (
                        "external_pl_equivalent_xa_per90" in active
                    ),
                    "external_saves_used": (
                        "external_pl_equivalent_saves_per90" in active
                    ),
                    "all_missing_external_features": ", ".join(
                        feature
                        for feature in dropped
                        if feature.startswith("external_")
                    ),
                })
        else:
            active = specification.get(
                "active_numeric_features",
                [],
            )
            dropped = specification.get(
                "dropped_all_missing_features",
                [],
            )
            rows.append({
                "model": model_name,
                "position": "pooled",
                "external_xg_used": (
                    "external_pl_equivalent_xg_per90" in active
                ),
                "external_xa_used": (
                    "external_pl_equivalent_xa_per90" in active
                ),
                "external_saves_used": (
                    "external_pl_equivalent_saves_per90" in active
                ),
                "all_missing_external_features": ", ".join(
                    feature
                    for feature in dropped
                    if feature.startswith("external_")
                ),
            })

    return pd.DataFrame(rows)


### Construct the holdout expected-points dataset

Team forecasts are joined to the player rows before fitting the player event models. Expected minutes are generated by the already-fitted minutes model, so the holdout xP calculation does not use realised minutes as an input.

In [ ]:
historical_team_model_predictions = pd.concat(
    [
        historical_team_fixtures[
            ["season", "fixture_id", "team_key"]
        ].reset_index(drop=True),
        predict_team_goal_components(
            team_goal_models,
            historical_team_fixtures,
        ).reset_index(drop=True),
    ],
    axis=1,
)

historical_points_data = (
    historical_model_data
    .merge(
        historical_team_model_predictions,
        on=["season", "fixture_id", "team_key"],
        how="left",
        validate="many_to_one",
    )
)

historical_points_data = add_player_per90_history_features(
    historical_points_data
)
historical_points_data = add_player_event_targets(
    historical_points_data
)

historical_points_data = historical_points_data.loc[
    historical_points_data["model_position"].isin(
        PLAYER_SCORING_POSITIONS
    )
].copy()

historical_event_schema_before = model_feature_contract_report(
    historical_points_data,
    PLAYER_EVENT_FEATURE_COLUMNS,
)
historical_points_data = materialise_model_feature_columns(
    historical_points_data,
    PLAYER_EVENT_FEATURE_COLUMNS,
    context="historical.player_event_features",
    categorical_columns=PLAYER_EVENT_CATEGORICAL_FEATURES,
)
if historical_event_schema_before["missing"]:
    print(
        "Historical player-event schema repaired:",
        historical_event_schema_before["missing"],
    )

development_points_data = historical_points_data.loc[
    historical_points_data["season"].isin(
        DEVELOPMENT_SEASONS
    )
].copy()

holdout_points_data = historical_points_data.loc[
    historical_points_data["season"].eq(HOLDOUT_SEASON)
].copy()

player_event_models = fit_player_event_rate_models(
    development_points_data
)

holdout_minutes_for_points = predict_minutes_components(
    minutes_models,
    holdout_points_data,
)

holdout_event_rates = predict_player_event_rates(
    player_event_models,
    holdout_points_data,
)

holdout_xp_inputs = pd.concat(
    [
        holdout_points_data.reset_index(drop=True),
        holdout_minutes_for_points.reset_index(drop=True),
        holdout_event_rates.reset_index(drop=True),
    ],
    axis=1,
)

print(
    "Holdout expected-points rows: "
    f"{len(holdout_xp_inputs):,}"
)


### Convert component forecasts into FPL points

Some scoring events are nonlinear:

- one save point is awarded for each complete set of three saves;
- one point is deducted for each complete pair of goals conceded;
- defensive-contribution points are awarded only after reaching a threshold.

For these components, the notebook treats the event count as Poisson-distributed and calculates the expected FPL score under the scoring rule rather than applying the rule to a rounded mean.

In [ ]:
def expected_poisson_floor_groups(
    expected_count: pd.Series | np.ndarray,
    group_size: int,
    max_count: int = 40,
) -> np.ndarray:
    """Return E[floor(N / group_size)] for Poisson N."""
    rate = np.clip(
        np.asarray(expected_count, dtype=float),
        0.0,
        20.0,
    )
    counts = np.arange(max_count + 1)
    probabilities = poisson.pmf(
        counts[:, None],
        rate[None, :],
    )
    grouped_points = np.floor_divide(
        counts,
        group_size,
    ).astype(float)

    expectation = (
        grouped_points[:, None] * probabilities
    ).sum(axis=0)

    # The remaining tail is tiny for the rates used here. Assigning it the
    # next complete group prevents systematic undercounting.
    tail = poisson.sf(max_count, rate)
    expectation += (
        (max_count + 1) // group_size
    ) * tail

    return expectation


def coalesced_numeric_rate(
    data: pd.DataFrame,
    candidate_columns: list[str],
    fallback_by_position: pd.Series,
) -> pd.Series:
    """Use the first available rate and then a position-level prior."""
    available = [
        column
        for column in candidate_columns
        if column in data.columns
    ]

    if available:
        result = data[available].bfill(axis=1).iloc[:, 0]
    else:
        result = pd.Series(np.nan, index=data.index)

    return result.fillna(
        data["model_position"].map(fallback_by_position)
    )


def defensive_action_position_priors(
    training_data: pd.DataFrame,
) -> pd.Series:
    """Estimate action rates by position from available archived data."""
    valid = training_data.loc[
        training_data["target_defensive_actions"].notna()
        & training_data["target_minutes"].gt(0)
    ].copy()

    if valid.empty:
        return pd.Series(
            0.0,
            index=PLAYER_SCORING_POSITIONS,
            dtype=float,
        )

    totals = valid.groupby("model_position").agg(
        actions=("target_defensive_actions", "sum"),
        minutes=("target_minutes", "sum"),
    )

    rate = 90.0 * totals["actions"] / totals["minutes"]
    return rate.reindex(PLAYER_SCORING_POSITIONS).fillna(0.0)


def defensive_return_position_priors(
    training_data: pd.DataFrame,
) -> pd.Series:
    """Estimate the historical probability of earning DC points by position."""
    if "target_defensive_actions" not in training_data.columns:
        return pd.Series(
            0.0,
            index=PLAYER_SCORING_POSITIONS,
            dtype=float,
        )

    valid = training_data.loc[
        training_data["target_defensive_actions"].notna()
    ].copy()
    if valid.empty:
        return pd.Series(
            0.0,
            index=PLAYER_SCORING_POSITIONS,
            dtype=float,
        )

    threshold = valid["model_position"].map(
        DEFENSIVE_CONTRIBUTION_THRESHOLD
    )
    valid["earned_defensive_return"] = (
        valid["target_defensive_actions"].ge(threshold)
        & threshold.notna()
    ).astype(float)

    return (
        valid.groupby("model_position")[
            "earned_defensive_return"
        ]
        .mean()
        .reindex(PLAYER_SCORING_POSITIONS)
        .fillna(0.0)
    )


def estimate_defensive_return_probability(
    data: pd.DataFrame,
    expected_defensive_actions: pd.Series,
    position_priors: pd.Series,
) -> pd.DataFrame:
    """
    Blend an action-count threshold model with observed award frequencies.

    The previous pure-Poisson calculation treated a stable action mean as if
    match actions were independent Poisson events. That made high-volume
    centre-backs approach a two-point return almost every match. Empirical
    shifted return rates provide calibration while the action model preserves
    fixture/minutes sensitivity.
    """
    position = data["model_position"]
    poisson_probability = pd.Series(0.0, index=data.index, dtype=float)

    for scoring_position, threshold in (
        DEFENSIVE_CONTRIBUTION_THRESHOLD.items()
    ):
        mask = position.eq(scoring_position)
        if not mask.any() or not np.isfinite(threshold):
            continue
        poisson_probability.loc[mask] = poisson.sf(
            int(threshold) - 1,
            expected_defensive_actions.loc[mask],
        )

    empirical_probability = coalesced_numeric_rate(
        data,
        [
            "defensive_return_rate_last_5_gws",
            "defensive_return_rate_last_10_gws",
            "career_defensive_return_rate",
        ],
        position_priors,
    ).clip(0.0, 1.0)

    prior = position.map(position_priors).fillna(0.0)
    history_count = pd.to_numeric(
        data.get(
            "prior_season_fixture_count",
            pd.Series(0.0, index=data.index),
        ),
        errors="coerce",
    ).fillna(0.0).clip(lower=0.0)

    # Shrink sparse empirical records toward the position prior.
    reliability = history_count / (history_count + 8.0)
    empirical_probability = (
        reliability * empirical_probability
        + (1.0 - reliability) * prior
    )

    # The award is practically unreachable without substantial minutes.
    minute_scale = (
        pd.to_numeric(data["expected_minutes"], errors="coerce")
        .fillna(0.0)
        .div(90.0)
        .clip(0.0, 1.0)
        .pow(0.65)
    )

    # v2.6: `expected_defensive_actions` already carries expected_minutes/90,
    # so the Poisson branch must not be scaled by minutes a second time. Only
    # the empirical branch (a raw award frequency) needs `minute_scale`.
    blended = (
        0.30 * poisson_probability
        + 0.70 * empirical_probability * minute_scale
    ).clip(0.0, 1.0)

    return pd.DataFrame({
        "poisson_defensive_return_probability": poisson_probability,
        "empirical_defensive_return_probability": empirical_probability,
        "defensive_return_probability": blended,
    })


def estimate_defensive_action_rate(
    data: pd.DataFrame,
    position_priors: pd.Series,
) -> pd.Series:
    """
    Estimate CBIT/CBIRT actions per 90 with explicit evidence shrinkage.

    v2.2.6 selected the first available last-5/last-10/career rate almost
    directly. That can overstate a new player's DC probability after only a
    handful of PL fixtures. v2.2.7 uses a two-level empirical-Bayes style
    shrinkage:
      1) career PL rate -> position prior when total PL history is sparse;
      2) recent rate -> that career/position anchor while current-season
         evidence is sparse.
    """
    recent_estimates = []

    for window in [5, 10]:
        tackles = pd.to_numeric(
            data.get(
                f"tackles_per90_last_{window}_gws",
                pd.Series(np.nan, index=data.index),
            ),
            errors="coerce",
        )
        cbi = pd.to_numeric(
            data.get(
                f"cbi_per90_last_{window}_gws",
                pd.Series(np.nan, index=data.index),
            ),
            errors="coerce",
        )
        recoveries = pd.to_numeric(
            data.get(
                f"recoveries_per90_last_{window}_gws",
                pd.Series(np.nan, index=data.index),
            ),
            errors="coerce",
        )

        has_data = tackles.notna() | cbi.notna() | recoveries.notna()
        base = tackles.fillna(0.0) + cbi.fillna(0.0)
        rate = pd.Series(
            np.where(
                data["model_position"].isin(["MID", "FWD"]),
                base + recoveries.fillna(0.0),
                base,
            ),
            index=data.index,
            dtype=float,
        ).where(has_data)
        recent_estimates.append(rate)

    recent_rate = (
        pd.concat(recent_estimates, axis=1)
        .bfill(axis=1)
        .iloc[:, 0]
    )

    career_tackles = pd.to_numeric(
        data.get(
            "career_tackles_per90",
            pd.Series(np.nan, index=data.index),
        ),
        errors="coerce",
    )
    career_cbi = pd.to_numeric(
        data.get(
            "career_cbi_per90",
            pd.Series(np.nan, index=data.index),
        ),
        errors="coerce",
    )
    career_recoveries = pd.to_numeric(
        data.get(
            "career_recoveries_per90",
            pd.Series(np.nan, index=data.index),
        ),
        errors="coerce",
    )
    career_has_data = (
        career_tackles.notna()
        | career_cbi.notna()
        | career_recoveries.notna()
    )
    career_base = career_tackles.fillna(0.0) + career_cbi.fillna(0.0)
    career_rate = pd.Series(
        np.where(
            data["model_position"].isin(["MID", "FWD"]),
            career_base + career_recoveries.fillna(0.0),
            career_base,
        ),
        index=data.index,
        dtype=float,
    ).where(career_has_data)

    position_prior = (
        data["model_position"]
        .map(position_priors)
        .fillna(0.0)
        .astype(float)
    )
    career_count = pd.to_numeric(
        data.get(
            "prior_career_fixture_count",
            pd.Series(0.0, index=data.index),
        ),
        errors="coerce",
    ).fillna(0.0).clip(lower=0.0)
    season_count = pd.to_numeric(
        data.get(
            "prior_season_fixture_count",
            career_count,
        ),
        errors="coerce",
    ).fillna(0.0).clip(lower=0.0)

    prior_strength = float(DEFENSIVE_ACTION_RATE_PRIOR_FIXTURES)
    career_reliability = (
        career_count / (career_count + prior_strength)
    ).clip(0.0, 1.0)
    recent_reliability = (
        season_count / (season_count + prior_strength)
    ).clip(0.0, 1.0)

    career_observed = career_rate.notna()
    career_anchor = pd.Series(
        np.where(
            career_observed,
            career_reliability * career_rate.fillna(position_prior)
            + (1.0 - career_reliability) * position_prior,
            position_prior,
        ),
        index=data.index,
        dtype=float,
    )

    recent_observed = recent_rate.notna()
    combined = pd.Series(
        np.where(
            recent_observed,
            recent_reliability * recent_rate.fillna(career_anchor)
            + (1.0 - recent_reliability) * career_anchor,
            career_anchor,
        ),
        index=data.index,
        dtype=float,
    )

    return combined.fillna(position_prior).fillna(0.0).clip(0.0, 30.0)


def rare_event_position_rates(
    training_data: pd.DataFrame,
) -> dict[str, pd.Series]:
    """Estimate very rare event rates rather than fitting unstable models."""
    rates: dict[str, pd.Series] = {}

    for event in [
        "own_goals",
        "penalties_saved",
        "penalties_missed",
    ]:
        if event not in training_data.columns:
            rates[event] = pd.Series(
                0.0,
                index=PLAYER_SCORING_POSITIONS,
                dtype=float,
            )
            continue

        valid = training_data.loc[
            training_data["target_minutes"].gt(0)
            & training_data[event].notna()
        ]

        if valid.empty:
            rates[event] = pd.Series(
                0.0,
                index=PLAYER_SCORING_POSITIONS,
                dtype=float,
            )
            continue

        totals = valid.groupby("model_position").agg(
            events=(event, "sum"),
            minutes=("target_minutes", "sum"),
        )
        rate = 90.0 * totals["events"] / totals["minutes"]
        rates[event] = (
            rate
            .reindex(PLAYER_SCORING_POSITIONS)
            .fillna(0.0)
        )

    return rates


def calculate_component_expected_points(
    data: pd.DataFrame,
    defensive_action_priors: pd.Series,
    rare_event_rates: dict[str, pd.Series],
    defensive_return_priors: pd.Series | None = None,
) -> pd.DataFrame:
    """Calculate expected FPL points from minutes and event forecasts."""
    result = pd.DataFrame(index=data.index)

    position = data["model_position"]
    expected_minutes = pd.to_numeric(
        data["expected_minutes"],
        errors="coerce",
    ).fillna(0.0).clip(0.0, 90.0)
    minute_fraction = expected_minutes / 90.0

    result["xp_appearance"] = (
        data["appearance_probability"].fillna(0.0)
        + data["sixty_plus_probability"].fillna(0.0)
    )

    result["projected_goals"] = (
        data["predicted_xg_per90"].fillna(0.0)
        * minute_fraction
    )
    result["projected_assists"] = (
        data["predicted_xa_per90"].fillna(0.0)
        * minute_fraction
    )

    # v2.6: FPL `expected_assists` is not on the realised-assist scale
    # (pooled 2023-26: DEF 1.31, MID 1.47, FWD 2.39). `expected_goals` is
    # close to 1.0 but is corrected too so the assumption stays monitored.
    # Factors are set by v26_set_conversion() on TRAINING rows only; when
    # unset both default to 1.0 and this is a no-op.
    _v26_assists = globals().get("V26_ASSIST_FACTORS") or {}
    _v26_goals = globals().get("V26_GOAL_FACTORS") or {}
    if _v26_goals:
        result["projected_goals"] = (
            result["projected_goals"]
            * _v26_position_factor(position, _v26_goals)
        )
    if _v26_assists:
        result["projected_assists"] = (
            result["projected_assists"]
            * _v26_position_factor(position, _v26_assists)
        )

    result["xp_goals"] = (
        result["projected_goals"]
        * position.map(GOAL_POINTS_BY_POSITION).fillna(0.0)
    )
    result["xp_assists"] = 3.0 * result["projected_assists"]

    # v2.2: FPL clean sheets are determined by goals conceded while the
    # player is on the pitch. A player subbed after 60 minutes keeps the CS
    # even if the team concedes later. Therefore use player exposure rather
    # than the full-match team CS probability.
    full_match_clean_sheet_probability = pd.to_numeric(
        data["team_clean_sheet_probability"],
        errors="coerce",
    ).fillna(0.0).clip(0.0, 1.0)

    expected_goals_conceded_90 = pd.to_numeric(
        data.get(
            "expected_team_goals_conceded",
            -np.log(
                full_match_clean_sheet_probability.clip(
                    lower=PROBABILITY_EPSILON
                )
            ),
        ),
        errors="coerce",
    ).fillna(0.0).clip(lower=0.0)

    sixty_exposure_minutes = pd.to_numeric(
        data.get(
            "minutes_if_start",
            expected_minutes,
        ),
        errors="coerce",
    ).fillna(expected_minutes).clip(60.0, 90.0)

    player_clean_sheet_probability_if_60 = np.exp(
        -expected_goals_conceded_90
        * sixty_exposure_minutes
        / 90.0
    ).clip(0.0, 1.0)

    result["clean_sheet_probability"] = (
        full_match_clean_sheet_probability
    )
    result["clean_sheet_exposure_minutes_if_60"] = (
        sixty_exposure_minutes
    )
    result["player_clean_sheet_probability_if_60"] = (
        player_clean_sheet_probability_if_60
    )
    result["xp_clean_sheet"] = (
        data["sixty_plus_probability"].fillna(0.0)
        * player_clean_sheet_probability_if_60
        * position.map(
            CLEAN_SHEET_POINTS_BY_POSITION
        ).fillna(0.0)
    )

    expected_saves = (
        data["predicted_saves_per90"].fillna(0.0)
        * minute_fraction
    )
    expected_saves = expected_saves.where(
        position.eq("GK"),
        0.0,
    )
    result["projected_saves"] = expected_saves
    result["xp_saves"] = expected_poisson_floor_groups(
        expected_saves,
        group_size=3,
    )

    expected_goals_conceded_on_pitch = (
        pd.to_numeric(
            data["expected_team_goals_conceded"],
            errors="coerce",
        ).fillna(0.0)
        * minute_fraction
    )
    result[
        "projected_goals_conceded_on_pitch"
    ] = expected_goals_conceded_on_pitch

    conceded_deductions = expected_poisson_floor_groups(
        expected_goals_conceded_on_pitch,
        group_size=2,
    )
    result["xp_goals_conceded"] = np.where(
        position.isin(["GK", "DEF"]),
        -conceded_deductions,
        0.0,
    )

    defensive_actions_per90 = estimate_defensive_action_rate(
        data,
        defensive_action_priors,
    )
    expected_defensive_actions = (
        defensive_actions_per90 * minute_fraction
    )
    result[
        "projected_defensive_actions"
    ] = expected_defensive_actions

    if defensive_return_priors is None:
        defensive_return_priors = pd.Series(
            0.0,
            index=PLAYER_SCORING_POSITIONS,
            dtype=float,
        )

    defensive_probability = estimate_defensive_return_probability(
        data,
        expected_defensive_actions,
        defensive_return_priors,
    )
    for column in defensive_probability.columns:
        result[column] = defensive_probability[column]
    result["xp_defensive_contribution"] = (
        2.0 * result["defensive_return_probability"]
    )

    result["xp_bonus"] = (
        data["predicted_bonus_per90"].fillna(0.0)
        * minute_fraction
    ).clip(0.0, 3.0)

    result["xp_yellow_cards"] = -(
        data["predicted_yellow_cards_per90"].fillna(0.0)
        * minute_fraction
    )
    result["xp_red_cards"] = -3.0 * (
        data["predicted_red_cards_per90"].fillna(0.0)
        * minute_fraction
    )

    own_goal_rate = position.map(
        rare_event_rates["own_goals"]
    ).fillna(0.0)
    penalty_save_rate = position.map(
        rare_event_rates["penalties_saved"]
    ).fillna(0.0)
    penalty_miss_rate = position.map(
        rare_event_rates["penalties_missed"]
    ).fillna(0.0)

    result["xp_own_goals"] = -2.0 * own_goal_rate * minute_fraction
    result["xp_penalty_saves"] = (
        5.0 * penalty_save_rate * minute_fraction
    )
    result["xp_penalty_misses"] = (
        -2.0 * penalty_miss_rate * minute_fraction
    )

    point_components = [
        "xp_appearance",
        "xp_goals",
        "xp_assists",
        "xp_clean_sheet",
        "xp_saves",
        "xp_goals_conceded",
        "xp_defensive_contribution",
        "xp_bonus",
        "xp_yellow_cards",
        "xp_red_cards",
        "xp_own_goals",
        "xp_penalty_saves",
        "xp_penalty_misses",
    ]

    result["expected_points"] = result[point_components].sum(axis=1)

    return result


development_defensive_action_priors = (
    defensive_action_position_priors(
        development_points_data
    )
)
development_defensive_return_priors = (
    defensive_return_position_priors(
        development_points_data
    )
)
development_rare_event_rates = rare_event_position_rates(
    development_points_data
)

v26_set_conversion(development_points_data, label="development")
holdout_xp_components = calculate_component_expected_points(
    holdout_xp_inputs,
    development_defensive_action_priors,
    development_rare_event_rates,
    development_defensive_return_priors,
)

holdout_expected_points = pd.concat(
    [
        holdout_xp_inputs.reset_index(drop=True),
        holdout_xp_components.reset_index(drop=True),
    ],
    axis=1,
)

### Holdout evaluation

The component model is compared with a direct rolling-points baseline. Evaluation includes absolute error, rank correlation and calibration by predicted-xP decile.

Player-level point forecasts are intrinsically noisy; ranking players well is often as important as minimising raw error.

In [ ]:
position_points_prior = (
    development_points_data
    .groupby("model_position")["target_fpl_points"]
    .mean()
)
global_points_prior = development_points_data[
    "target_fpl_points"
].mean()

holdout_expected_points["baseline_expected_points"] = (
    coalesced_history_baseline(
        holdout_expected_points,
        [
            "points_per_fixture_last_5_gws",
            "points_per_fixture_last_10_gws",
            "career_points_per_fixture",
        ],
        position_points_prior,
    )
    .fillna(global_points_prior)
    .clip(lower=-3.0, upper=20.0)
)

valid_points = holdout_expected_points[
    "target_fpl_points"
].notna()
actual_points = holdout_expected_points.loc[
    valid_points,
    "target_fpl_points",
]

points_metric_rows = []
for method, prediction_column in [
    ("rolling-points baseline", "baseline_expected_points"),
    ("component xP model", "expected_points"),
]:
    predicted = holdout_expected_points.loc[
        valid_points,
        prediction_column,
    ]

    rank_correlation = spearmanr(
        actual_points,
        predicted,
        nan_policy="omit",
    ).statistic

    points_metric_rows.append({
        "method": method,
        "mae": mean_absolute_error(actual_points, predicted),
        "rmse": np.sqrt(
            mean_squared_error(actual_points, predicted)
        ),
        "spearman": rank_correlation,
        "mean_actual": actual_points.mean(),
        "mean_predicted": predicted.mean(),
    })

expected_points_scorecard = pd.DataFrame(points_metric_rows)
display(expected_points_scorecard)

points_error_by_position = (
    holdout_expected_points.loc[valid_points]
    .assign(
        absolute_error=lambda frame: (
            frame["target_fpl_points"]
            - frame["expected_points"]
        ).abs(),
        signed_error=lambda frame: (
            frame["expected_points"]
            - frame["target_fpl_points"]
        ),
    )
    .groupby("model_position")
    .agg(
        rows=("target_fpl_points", "size"),
        actual_points=("target_fpl_points", "mean"),
        predicted_points=("expected_points", "mean"),
        mae=("absolute_error", "mean"),
        bias=("signed_error", "mean"),
    )
    .reset_index()
)

display(points_error_by_position)

calibration_data = holdout_expected_points.loc[
    valid_points,
    ["expected_points", "target_fpl_points"],
].copy()

calibration_data["xp_decile"] = pd.qcut(
    calibration_data["expected_points"],
    q=10,
    duplicates="drop",
)

expected_points_calibration = (
    calibration_data
    .groupby("xp_decile", observed=True)
    .agg(
        rows=("target_fpl_points", "size"),
        mean_xp=("expected_points", "mean"),
        mean_actual_points=("target_fpl_points", "mean"),
    )
    .reset_index()
)

display(expected_points_calibration)


### Expected-points stability audit

The first executed version exposed a numerical failure in the event-rate layer: a few unbounded Poisson predictions for midfielders produced impossible expected-point values. The safeguards below are deliberately strict:

- every event-rate prediction is capped at the same plausible ceiling used for its training target;
- non-finite predictions are replaced safely;
- goalkeeper goal and assist rates are set to zero in this baseline;
- fixture-level xP must remain finite and within a broad plausibility range;
- diagnostics show the extreme quantiles before the values are used by the optimiser.

These are model guardrails, not substitutes for calibration. The holdout scorecard remains the test of whether the repaired component model is useful.

In [ ]:
PREDICTED_EVENT_RATE_CAPS = {
    f"predicted_{name}_per90": float(specification["upper_clip"])
    for name, specification in PLAYER_RATE_MODEL_SPECIFICATIONS.items()
}

event_rate_diagnostics = (
    holdout_expected_points[
        list(PREDICTED_EVENT_RATE_CAPS)
    ]
    .describe(
        percentiles=[0.50, 0.90, 0.95, 0.99, 0.999]
    )
    .T
)

display(event_rate_diagnostics)

for column, upper_cap in PREDICTED_EVENT_RATE_CAPS.items():
    values = pd.to_numeric(
        holdout_expected_points[column],
        errors="coerce",
    )
    assert np.isfinite(values).all(), (
        f"{column} contains a non-finite prediction."
    )
    assert values.ge(-1e-12).all(), (
        f"{column} contains a negative prediction."
    )
    assert values.le(upper_cap + 1e-9).all(), (
        f"{column} exceeded its configured prediction cap."
    )

goalkeeper_rows = holdout_expected_points[
    "model_position"
].eq("GK")
assert holdout_expected_points.loc[
    goalkeeper_rows,
    ["predicted_xg_per90", "predicted_xa_per90"],
].abs().to_numpy().max(initial=0.0) < 1e-12, (
    "Goalkeepers received ordinary attacking-rate projections."
)

fixture_xp = pd.to_numeric(
    holdout_expected_points["expected_points"],
    errors="coerce",
)
assert np.isfinite(fixture_xp).all(), (
    "Expected points contain a non-finite value."
)
assert fixture_xp.between(-5.0, 35.0).all(), (
    "Fixture-level expected points breached the broad sanity range."
)

xp_stability_summary = fixture_xp.describe(
    percentiles=[0.50, 0.90, 0.95, 0.99, 0.999]
)
display(xp_stability_summary.to_frame("expected_points"))

component_audit_by_position = (
    holdout_expected_points
    .groupby("model_position")
    [
        [
            "xp_appearance",
            "xp_goals",
            "xp_assists",
            "xp_clean_sheet",
            "xp_saves",
            "xp_goals_conceded",
            "xp_defensive_contribution",
            "xp_bonus",
            "expected_points",
        ]
    ]
    .mean()
)

display(component_audit_by_position)

display(
    holdout_expected_points[
        [
            "season",
            "gameweek",
            "player_name",
            "team_short_name",
            "model_position",
            "expected_minutes",
            "predicted_xg_per90",
            "predicted_xa_per90",
            "predicted_bonus_per90",
            "expected_points",
            "target_fpl_points",
        ]
    ]
    .sort_values("expected_points", ascending=False)
    .head(20)
)

### Production xP calibration

The component model intentionally remains the interpretable source of goals, assists, clean sheets, saves, bonus and other point components. A second-stage calibration is applied only to the final xP total. It is learned from genuinely out-of-sample historical forecasts using expanding chronological folds.

This addresses systematic compression of premium/high-xP players without manually inflating specific names. The raw component xP is retained alongside the calibrated production xP for auditability. Captaincy, transfers, hits and chips use the calibrated value.


### v2.2 decision-focused xP validation

Global MAE/RMSE are retained, but the production task is ranking and selecting FPL assets. These diagnostics add top-K ranking quality, starter-only error and captain regret.

In [ ]:
def _dcg_at_k(relevance: np.ndarray, k: int) -> float:
    values = np.asarray(relevance, dtype=float)[:int(k)]
    if values.size == 0:
        return np.nan
    discounts = np.log2(np.arange(2, values.size + 2))
    return float(np.sum(values / discounts))


def decision_focused_xp_metrics(
    fixture_rows: pd.DataFrame,
    *,
    prediction_column: str = "expected_points",
    actual_column: str = "target_fpl_points",
) -> pd.DataFrame:
    """
    Evaluate the player-GW decisions the optimiser actually cares about.

    NDCG uses non-negative realised points as relevance. Captain regret is
    hindsight-only and is reported as a diagnostic, not a training target.
    """
    required = {
        "season",
        "gameweek",
        "player_id_season",
        prediction_column,
        actual_column,
        "target_minutes",
    }
    missing = required - set(fixture_rows.columns)
    if missing:
        raise KeyError(
            "Decision-focused validation missing columns: "
            f"{sorted(missing)}"
        )

    gw = (
        fixture_rows.groupby(
            ["season", "gameweek", "player_id_season"],
            as_index=False,
            dropna=False,
        )
        .agg(
            predicted=(prediction_column, "sum"),
            actual=(actual_column, "sum"),
            minutes=("target_minutes", "sum"),
        )
    )
    gw["predicted"] = pd.to_numeric(gw["predicted"], errors="coerce").fillna(0.0)
    gw["actual"] = pd.to_numeric(gw["actual"], errors="coerce").fillna(0.0)
    gw["minutes"] = pd.to_numeric(gw["minutes"], errors="coerce").fillna(0.0)

    rows = []
    for (season, gameweek), group in gw.groupby(["season", "gameweek"]):
        ranked = group.sort_values("predicted", ascending=False)
        ideal = group.sort_values("actual", ascending=False)
        for k in (10, 25, 50):
            actual_relevance = ranked["actual"].clip(lower=0).to_numpy()
            ideal_relevance = ideal["actual"].clip(lower=0).to_numpy()
            dcg = _dcg_at_k(actual_relevance, k)
            idcg = _dcg_at_k(ideal_relevance, k)
            ndcg = dcg / idcg if np.isfinite(idcg) and idcg > 1e-12 else np.nan
            rows.append({
                "season": season,
                "gameweek": int(gameweek),
                "metric": f"ndcg_at_{k}",
                "value": ndcg,
            })

        predicted_captain = ranked.iloc[0]
        oracle_points = float(group["actual"].max())
        rows.append({
            "season": season,
            "gameweek": int(gameweek),
            "metric": "captain_regret",
            "value": oracle_points - float(predicted_captain["actual"]),
        })

        starters = group.loc[group["minutes"].ge(60)]
        if not starters.empty:
            rows.append({
                "season": season,
                "gameweek": int(gameweek),
                "metric": "starter_mae",
                "value": float(
                    np.mean(np.abs(starters["predicted"] - starters["actual"]))
                ),
            })

    detail = pd.DataFrame(rows)
    if detail.empty:
        return detail
    return (
        detail.groupby(["season", "metric"], as_index=False)
        .agg(
            mean=("value", "mean"),
            median=("value", "median"),
            gameweeks=("value", "count"),
        )
    )


if DECISION_METRICS_ENABLED and "holdout_expected_points" in globals():
    decision_focused_holdout_scorecard = decision_focused_xp_metrics(
        holdout_expected_points
    )
    display(decision_focused_holdout_scorecard)

## 7. Apply the expected-points model to upcoming fixtures

After the untouched holdout evaluation, the live models are refitted on all completed historical seasons. Current-season completed Gameweeks are added to each player's rolling history before forecasting the planning horizon.

### Refit the minutes and team models for live use

In [ ]:

# =============================================================================
# LIVE ALL-COMPETITION WORKLOAD CONTEXT
# =============================================================================
# Live-only production overlay. The signed historical minutes model is left
# unchanged so the v2.1 OOS validation caches remain comparable.

def _normalise_external_team_name(value: object) -> str:
    if pd.isna(value):
        return ""
    text = str(value).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace("&", " and ")
    text = re.sub(r"\bfc\b", " ", text)
    text = re.sub(r"\bafc\b", " ", text)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def _fpl_team_alias_lookup(team_table: pd.DataFrame) -> dict[str, int]:
    aliases: dict[str, int] = {}
    canonical: dict[str, int] = {}

    for row in team_table.itertuples(index=False):
        team_id = int(row.team_id)
        full_key = _normalise_external_team_name(row.team_name)
        short_key = _normalise_external_team_name(row.team_short_name)
        if full_key:
            aliases[full_key] = team_id
            canonical[full_key] = team_id
        if short_key:
            aliases[short_key] = team_id

    common_aliases = {
        "man city": "manchester city",
        "man united": "manchester united",
        "man utd": "manchester united",
        "spurs": "tottenham hotspur",
        "tottenham": "tottenham hotspur",
        "newcastle": "newcastle united",
        "west ham": "west ham united",
        "brighton": "brighton and hove albion",
        "brighton hove albion": "brighton and hove albion",
        "wolves": "wolverhampton wanderers",
        "wolverhampton": "wolverhampton wanderers",
        "nottm forest": "nottingham forest",
        "leeds": "leeds united",
        "sunderland afc": "sunderland",
    }
    for alias, target in common_aliases.items():
        target_id = canonical.get(_normalise_external_team_name(target))
        if target_id is not None:
            aliases[_normalise_external_team_name(alias)] = target_id

    return aliases


def _fresh_live_cache(path: Path, max_hours: float) -> bool:
    if not path.exists():
        return False
    return (
        time.time() - path.stat().st_mtime
        <= float(max_hours) * 3600.0
    )


def _all_comp_date_window(
    fixture_table: pd.DataFrame,
) -> tuple[pd.Timestamp, pd.Timestamp]:
    """
    Return only the live window needed by the rolling FPL horizon.

    v2.1.11 accidentally used the maximum kickoff in the full PL fixture list,
    which could produce a season-long ESPN request. Here the end date is tied
    to the next PLANNING_HORIZON gameweeks and capped absolutely.
    """
    now = pd.Timestamp.now(tz="UTC")
    start = (
        now.normalize()
        - pd.Timedelta(days=int(LIVE_ALL_COMP_LOOKBACK_DAYS))
    )

    default_end = (
        now.normalize()
        + pd.Timedelta(
            days=max(
                int(LIVE_ALL_COMP_LOOKAHEAD_DAYS),
                35,
            )
        )
    )

    fixture_copy = fixture_table.copy()
    fixture_copy["_kickoff_utc"] = pd.to_datetime(
        fixture_copy.get("kickoff_time"),
        utc=True,
        errors="coerce",
    )

    gameweek_column = next(
        (
            column
            for column in ["gameweek", "event"]
            if column in fixture_copy.columns
        ),
        None,
    )

    horizon_end = default_end
    if gameweek_column is not None:
        upcoming = fixture_copy.loc[
            fixture_copy["_kickoff_utc"].ge(now)
            & pd.to_numeric(
                fixture_copy[gameweek_column],
                errors="coerce",
            ).notna()
        ].copy()

        if not upcoming.empty:
            upcoming["_gw_numeric"] = pd.to_numeric(
                upcoming[gameweek_column],
                errors="coerce",
            )
            future_gameweeks = sorted(
                int(value)
                for value in upcoming["_gw_numeric"].dropna().unique()
            )[: int(globals().get("PLANNING_HORIZON", 6))]

            if future_gameweeks:
                selected = upcoming.loc[
                    upcoming["_gw_numeric"].isin(future_gameweeks)
                ]
                selected_max = selected["_kickoff_utc"].max()
                if pd.notna(selected_max):
                    horizon_end = max(
                        horizon_end,
                        selected_max
                        + pd.Timedelta(
                            days=float(
                                LIVE_ALL_COMP_NEXT_MATCH_WITHIN_DAYS
                            )
                            + 1.0
                        ),
                    )

    absolute_cap = (
        now.normalize()
        + pd.Timedelta(days=int(LIVE_ALL_COMP_MAX_LOOKAHEAD_DAYS))
    )
    end = min(horizon_end, absolute_cap)

    if end < start:
        end = start

    return start, end


def _iter_all_comp_date_chunks(
    start: pd.Timestamp,
    end: pd.Timestamp,
    chunk_days: int | None = None,
):
    chunk_days = int(
        chunk_days
        if chunk_days is not None
        else LIVE_ALL_COMP_QUERY_CHUNK_DAYS
    )
    if chunk_days < 1:
        raise ValueError("LIVE_ALL_COMP_QUERY_CHUNK_DAYS must be >= 1.")

    cursor = pd.Timestamp(start).normalize()
    final = pd.Timestamp(end).normalize()

    while cursor <= final:
        chunk_end = min(
            cursor + pd.Timedelta(days=chunk_days - 1),
            final,
        )
        yield cursor, chunk_end
        cursor = chunk_end + pd.Timedelta(days=1)


def _espn_scoreboard_endpoints(
    competition_code: str,
) -> list[str]:
    return [
        (
            "https://site.api.espn.com/apis/site/v2/sports/soccer/"
            f"{competition_code}/scoreboard"
        ),
        (
            "https://site.web.api.espn.com/apis/v2/sports/soccer/"
            f"{competition_code}/scoreboard"
        ),
    ]



def _active_all_comp_failure_cache() -> dict[str, object] | None:
    """
    Return a still-active negative cache entry for the all-competition source.

    This cache is deliberately short lived: it exists only to stop repeated notebook
    reruns from hammering a source that has just failed its circuit breaker.
    """
    if float(LIVE_ALL_COMP_FAILURE_CACHE_HOURS) <= 0:
        return None
    if not ALL_COMP_FAILURE_CACHE_PATH.exists():
        return None

    try:
        payload = json.loads(
            ALL_COMP_FAILURE_CACHE_PATH.read_text(encoding="utf-8")
        )
        if str(payload.get("source")) != str(LIVE_ALL_COMP_SOURCE):
            try:
                ALL_COMP_FAILURE_CACHE_PATH.unlink()
            except FileNotFoundError:
                pass
            return None

        expires_at = pd.to_datetime(
            payload.get("expires_at_utc"),
            utc=True,
            errors="coerce",
        )
        if pd.isna(expires_at) or pd.Timestamp.now(tz="UTC") >= expires_at:
            try:
                ALL_COMP_FAILURE_CACHE_PATH.unlink()
            except FileNotFoundError:
                pass
            return None
        return payload
    except Exception as exc:
        warnings.warn(
            f"Could not read all-competition failure cache: {exc}"
        )
        return None


def _write_all_comp_failure_cache(
    *,
    reason: str,
    consecutive_failures: int,
    attempted_requests: int,
) -> dict[str, object]:
    now = pd.Timestamp.now(tz="UTC")
    expires = now + pd.Timedelta(
        hours=float(LIVE_ALL_COMP_FAILURE_CACHE_HOURS)
    )
    payload = {
        "schema_version": 1,
        "source": str(LIVE_ALL_COMP_SOURCE),
        "failed_at_utc": now.isoformat(),
        "expires_at_utc": expires.isoformat(),
        "consecutive_failures": int(consecutive_failures),
        "attempted_requests": int(attempted_requests),
        "reason": str(reason)[:2000],
    }
    try:
        ALL_COMP_FAILURE_CACHE_PATH.parent.mkdir(
            parents=True,
            exist_ok=True,
        )
        ALL_COMP_FAILURE_CACHE_PATH.write_text(
            json.dumps(payload, indent=2, default=str),
            encoding="utf-8",
        )
    except Exception as exc:
        warnings.warn(
            f"Could not write all-competition failure cache: {exc}"
        )
    return payload


def _clear_all_comp_failure_cache() -> None:
    try:
        ALL_COMP_FAILURE_CACHE_PATH.unlink()
    except FileNotFoundError:
        pass
    except Exception as exc:
        warnings.warn(
            f"Could not clear all-competition failure cache: {exc}"
        )


def _request_espn_scoreboard_window(
    session,
    competition_code: str,
    start: pd.Timestamp,
    end: pd.Timestamp,
    headers: dict[str, str],
) -> tuple[dict | None, dict[str, object]]:
    start = pd.Timestamp(start).normalize()
    end = pd.Timestamp(end).normalize()

    date_token = (
        start.strftime("%Y%m%d")
        if start == end
        else f"{start.strftime('%Y%m%d')}-{end.strftime('%Y%m%d')}"
    )

    errors = []
    for endpoint in _espn_scoreboard_endpoints(competition_code):
        params = {
            "dates": date_token,
            "limit": 200,
            "region": "us",
            "lang": "en",
        }
        if "site.web.api.espn.com" in endpoint:
            params["contentorigin"] = "espn"

        try:
            response = session.get(
                endpoint,
                params=params,
                headers=headers,
                timeout=float(LIVE_ALL_COMP_REQUEST_TIMEOUT_SECONDS),
            )
            status_code = int(response.status_code)
            if status_code >= 400:
                errors.append(
                    f"{endpoint.split('/apis/')[0]} HTTP {status_code}"
                )
                continue

            payload = response.json()
            if not isinstance(payload, dict):
                errors.append("non-dict JSON payload")
                continue

            return payload, {
                "success": True,
                "http_status": status_code,
                "endpoint": endpoint,
                "date_token": date_token,
                "error": None,
            }
        except Exception as exc:
            errors.append(str(exc))

    return None, {
        "success": False,
        "http_status": None,
        "endpoint": None,
        "date_token": date_token,
        "error": " | ".join(errors),
    }


def _extract_espn_schedule_rows(
    payload: dict,
    competition: str,
    competition_code: str,
    aliases: dict[str, int],
) -> list[dict]:
    rows = []

    for event in payload.get("events", []) or []:
        kickoff = pd.to_datetime(
            event.get("date"),
            utc=True,
            errors="coerce",
        )
        if pd.isna(kickoff):
            continue

        competition_rows = event.get("competitions", []) or []
        if not competition_rows:
            continue
        competitors = competition_rows[0].get("competitors", []) or []
        if len(competitors) < 2:
            continue

        parsed = []
        for competitor in competitors:
            team_data = competitor.get("team", {}) or {}
            name = (
                team_data.get("displayName")
                or team_data.get("name")
                or team_data.get("shortDisplayName")
                or ""
            )
            parsed.append({
                "name": str(name),
                "key": _normalise_external_team_name(name),
                "home_away": competitor.get("homeAway"),
            })

        completed = bool(
            event.get("status", {})
            .get("type", {})
            .get("completed", False)
        )

        for current in parsed:
            team_id = aliases.get(current["key"])
            if team_id is None:
                continue

            opponent = next(
                (
                    other["name"]
                    for other in parsed
                    if other["key"] != current["key"]
                ),
                "",
            )
            rows.append({
                "event_id": str(event.get("id", "")),
                "competition": competition,
                "competition_code": competition_code,
                "kickoff_time": kickoff,
                "completed": completed,
                "team_id": int(team_id),
                "team_name_external": current["name"],
                "opponent_external": opponent,
                "home_away": current["home_away"],
            })

    return rows


def load_live_all_comp_schedule(
    team_table: pd.DataFrame,
    fixture_table: pd.DataFrame,
) -> pd.DataFrame:
    """
    Load the live non-PL schedule with bounded failure behaviour.

    The positive schedule cache is preferred. If no usable positive cache exists,
    a short negative cache can suppress network activity after a recent source-wide
    failure. During a fresh request run, every failed logical scoreboard request
    contributes to one global consecutive-failure counter; daily fallback requests
    are included in the same counter. Once the configured threshold is reached,
    all remaining source requests are skipped for this execution.
    """
    global live_all_comp_fetch_audit
    global live_all_comp_schedule_status

    base_columns = [
        "event_id",
        "competition",
        "competition_code",
        "kickoff_time",
        "completed",
        "team_id",
        "team_name_external",
        "opponent_external",
        "home_away",
    ]

    start, end = _all_comp_date_window(fixture_table)

    def empty_status(
        status: str,
        *,
        source: str,
        failure_cache: dict[str, object] | None = None,
    ) -> pd.DataFrame:
        live_all_comp_schedule_status.update({
            "status": status,
            "window_start": start,
            "window_end": end,
            "unique_events": 0,
            "team_fixture_rows": 0,
            "successful_requests": 0,
            "failed_requests": 0,
            "total_requests": 0,
            "source": source,
            "circuit_open": status == "CIRCUIT_OPEN",
            "consecutive_failures_at_stop": int(
                (failure_cache or {}).get("consecutive_failures", 0)
            ),
            "remaining_requests_skipped": None,
            "failure_cache_active": bool(failure_cache),
            "failure_cache_expires_at_utc": (
                (failure_cache or {}).get("expires_at_utc")
            ),
        })
        return pd.DataFrame(columns=base_columns)

    live_all_comp_schedule_status = {}

    if not LIVE_ALL_COMP_WORKLOAD_ENABLED:
        live_all_comp_fetch_audit = pd.DataFrame()
        return empty_status("OFF", source="disabled")

    # Prefer a fresh positive cache over a negative source cache.
    if _fresh_live_cache(
        ALL_COMP_SCHEDULE_CACHE_PATH,
        LIVE_ALL_COMP_REFRESH_HOURS,
    ):
        try:
            cached = pd.read_csv(
                ALL_COMP_SCHEDULE_CACHE_PATH,
                compression="gzip",
            )
            cached["kickoff_time"] = pd.to_datetime(
                cached["kickoff_time"],
                utc=True,
                errors="coerce",
            )

            schema_ok = (
                "all_comp_cache_schema_version" in cached.columns
                and pd.to_numeric(
                    cached["all_comp_cache_schema_version"],
                    errors="coerce",
                ).eq(
                    int(LIVE_ALL_COMP_CACHE_SCHEMA_VERSION)
                ).all()
            )

            coverage_ok = False
            if (
                schema_ok
                and "query_window_start" in cached.columns
                and "query_window_end" in cached.columns
                and not cached.empty
            ):
                cached_start = pd.to_datetime(
                    cached["query_window_start"],
                    utc=True,
                    errors="coerce",
                ).min()
                cached_end = pd.to_datetime(
                    cached["query_window_end"],
                    utc=True,
                    errors="coerce",
                ).max()
                coverage_ok = (
                    pd.notna(cached_start)
                    and pd.notna(cached_end)
                    and cached_start <= start
                    and cached_end >= end
                )

            if schema_ok and coverage_ok:
                _clear_all_comp_failure_cache()
                live_all_comp_fetch_audit = pd.DataFrame([{
                    "competition": "CACHE",
                    "competition_code": "cache",
                    "query_start": start,
                    "query_end": end,
                    "request_mode": "cache",
                    "success": True,
                    "http_status": np.nan,
                    "events_returned": int(
                        cached["event_id"].nunique()
                    ),
                    "epl_team_rows": int(len(cached)),
                    "error": None,
                }])
                live_all_comp_schedule_status = {
                    "status": (
                        "ACTIVE"
                        if len(cached) > 0
                        else "EMPTY"
                    ),
                    "window_start": start,
                    "window_end": end,
                    "unique_events": int(
                        cached["event_id"].nunique()
                    ),
                    "team_fixture_rows": int(len(cached)),
                    "successful_requests": 1,
                    "failed_requests": 0,
                    "total_requests": 1,
                    "source": "fresh_cache",
                    "circuit_open": False,
                    "consecutive_failures_at_stop": 0,
                    "remaining_requests_skipped": 0,
                    "failure_cache_active": False,
                    "failure_cache_expires_at_utc": None,
                }
                return cached
        except Exception as exc:
            warnings.warn(
                f"Could not reuse non-PL schedule cache: {exc}"
            )

    failure_cache = _active_all_comp_failure_cache()
    if failure_cache is not None:
        message = (
            "All-competition source is temporarily suppressed by the "
            "negative failure cache until "
            f"{failure_cache.get('expires_at_utc')}."
        )
        if LIVE_ALL_COMP_STRICT:
            raise RuntimeError(message)
        warnings.warn(message)
        live_all_comp_fetch_audit = pd.DataFrame([{
            "competition": "FAILURE_CACHE",
            "competition_code": "failure_cache",
            "query_start": start,
            "query_end": end,
            "request_mode": "negative_cache",
            "success": False,
            "http_status": np.nan,
            "events_returned": 0,
            "epl_team_rows": 0,
            "error": failure_cache.get("reason"),
        }])
        return empty_status(
            "CIRCUIT_OPEN",
            source="negative_failure_cache",
            failure_cache=failure_cache,
        )

    aliases = _fpl_team_alias_lookup(team_table)
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 Chrome/152 Safari/537.36"
        ),
        "Accept": "application/json,text/plain,*/*",
        "Referer": "https://www.espn.com/",
    }

    rows = []
    audit_rows = []
    consecutive_failures = 0
    circuit_open = False
    attempted_requests = 0
    successful_requests = 0
    last_failure_reason = ""

    work_items = [
        (competition, code, chunk_start, chunk_end)
        for competition, code in ESPN_ALL_COMPETITION_CODES.items()
        for chunk_start, chunk_end in _iter_all_comp_date_chunks(
            start,
            end,
        )
    ]
    work_index = -1

    def register_request_result(
        *,
        competition: str,
        code: str,
        query_start: pd.Timestamp,
        query_end: pd.Timestamp,
        request_mode: str,
        payload: dict | None,
        audit: dict[str, object],
        extracted: list[dict],
    ) -> None:
        nonlocal consecutive_failures
        nonlocal circuit_open
        nonlocal attempted_requests
        nonlocal successful_requests
        nonlocal last_failure_reason

        attempted_requests += 1
        success = bool(audit.get("success", False))
        if success:
            successful_requests += 1
            consecutive_failures = 0
        else:
            consecutive_failures += 1
            last_failure_reason = str(
                audit.get("error") or "unknown ESPN scoreboard failure"
            )

        audit_rows.append({
            "competition": competition,
            "competition_code": code,
            "query_start": query_start,
            "query_end": query_end,
            "request_mode": request_mode,
            "success": success,
            "http_status": audit.get("http_status"),
            "events_returned": int(
                len(payload.get("events", []) or [])
                if payload is not None
                else 0
            ),
            "epl_team_rows": int(len(extracted)),
            "error": audit.get("error"),
            "consecutive_failures_after": int(consecutive_failures),
        })

        if (
            consecutive_failures
            >= int(LIVE_ALL_COMP_MAX_CONSECUTIVE_FAILURES)
        ):
            circuit_open = True

    with requests.Session() as session:
        for work_index, (
            competition,
            code,
            chunk_start,
            chunk_end,
        ) in enumerate(work_items):
            if circuit_open:
                break

            payload, audit = _request_espn_scoreboard_window(
                session,
                code,
                chunk_start,
                chunk_end,
                headers,
            )

            extracted = []
            if payload is not None:
                extracted = _extract_espn_schedule_rows(
                    payload,
                    competition,
                    code,
                    aliases,
                )
                rows.extend(extracted)

            register_request_result(
                competition=competition,
                code=code,
                query_start=chunk_start,
                query_end=chunk_end,
                request_mode="chunk",
                payload=payload,
                audit=audit,
                extracted=extracted,
            )

            if circuit_open:
                break

            # A failed range request may still work day-by-day. Those fallback
            # calls share the SAME failure counter and are therefore bounded.
            if (
                payload is None
                and LIVE_ALL_COMP_DAILY_FALLBACK
                and chunk_end > chunk_start
            ):
                day = chunk_start
                while day <= chunk_end and not circuit_open:
                    daily_payload, daily_audit = (
                        _request_espn_scoreboard_window(
                            session,
                            code,
                            day,
                            day,
                            headers,
                        )
                    )
                    daily_extracted = []
                    if daily_payload is not None:
                        daily_extracted = _extract_espn_schedule_rows(
                            daily_payload,
                            competition,
                            code,
                            aliases,
                        )
                        rows.extend(daily_extracted)

                    register_request_result(
                        competition=competition,
                        code=code,
                        query_start=day,
                        query_end=day,
                        request_mode="daily_fallback",
                        payload=daily_payload,
                        audit=daily_audit,
                        extracted=daily_extracted,
                    )
                    day += pd.Timedelta(days=1)

    live_all_comp_fetch_audit = pd.DataFrame(audit_rows)

    frame = pd.DataFrame(rows, columns=base_columns)
    if not frame.empty:
        frame = (
            frame
            .drop_duplicates(
                ["event_id", "team_id"],
                keep="last",
            )
            .sort_values(["kickoff_time", "team_id"])
            .reset_index(drop=True)
        )

        frame["all_comp_cache_schema_version"] = int(
            LIVE_ALL_COMP_CACHE_SCHEMA_VERSION
        )
        frame["query_window_start"] = start
        frame["query_window_end"] = end
        frame["cache_fetched_at_utc"] = pd.Timestamp.now(
            tz="UTC"
        )
        frame.to_csv(
            ALL_COMP_SCHEDULE_CACHE_PATH,
            index=False,
            compression="gzip",
        )

    successful = int(successful_requests)
    failed = int(max(0, attempted_requests - successful_requests))
    total = int(attempted_requests)
    remaining_skipped = int(
        max(0, len(work_items) - (work_index + 1))
        if circuit_open
        else 0
    )

    failure_cache_payload = None
    if circuit_open and successful == 0:
        failure_cache_payload = _write_all_comp_failure_cache(
            reason=last_failure_reason,
            consecutive_failures=consecutive_failures,
            attempted_requests=attempted_requests,
        )
    elif successful > 0:
        _clear_all_comp_failure_cache()

    if circuit_open:
        status = "DEGRADED" if len(frame) > 0 else "CIRCUIT_OPEN"
    elif len(frame) > 0:
        status = "ACTIVE"
    elif successful > 0:
        status = "EMPTY"
    else:
        status = "UNAVAILABLE"

    live_all_comp_schedule_status = {
        "status": status,
        "window_start": start,
        "window_end": end,
        "unique_events": int(
            frame["event_id"].nunique()
            if not frame.empty
            else 0
        ),
        "team_fixture_rows": int(len(frame)),
        "successful_requests": successful,
        "failed_requests": failed,
        "total_requests": total,
        "source": (
            "espn_chunked_circuit_breaker"
            if circuit_open
            else "espn_chunked"
        ),
        "circuit_open": bool(circuit_open),
        "consecutive_failures_at_stop": int(consecutive_failures),
        "remaining_requests_skipped": remaining_skipped,
        "failure_cache_active": bool(failure_cache_payload),
        "failure_cache_expires_at_utc": (
            failure_cache_payload.get("expires_at_utc")
            if failure_cache_payload
            else None
        ),
    }

    if circuit_open:
        message = (
            "All-competition workload circuit breaker opened after "
            f"{consecutive_failures} consecutive failed logical requests; "
            f"{remaining_skipped} remaining chunk request(s) were skipped."
        )
        if LIVE_ALL_COMP_STRICT:
            raise RuntimeError(message)
        warnings.warn(message)
    elif status == "UNAVAILABLE":
        message = (
            "All-competition workload feed unavailable across the "
            f"{start.date()}–{end.date()} live window."
        )
        if LIVE_ALL_COMP_STRICT:
            raise RuntimeError(message)
        warnings.warn(message)
    elif status == "EMPTY":
        warnings.warn(
            "All-competition requests succeeded but no current PL-club "
            "non-PL fixtures were matched in the live window."
        )

    return frame

def _espn_minute_value(
    value: object,
    default: float | None = None,
) -> float | None:
    if value is None:
        return default
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    text = str(value).strip().lower()
    if text in {"start", "ko", "kickoff"}:
        return 0.0
    if text in {"end", "ft", "full time", "full-time"}:
        return 90.0
    match = re.search(r"(\d+)", text)
    return float(match.group(1)) if match else default


def extract_espn_summary_player_minutes(
    payload: dict,
    team_aliases: dict[str, int],
) -> list[dict]:
    rows = []

    # Preferred box-score minutes.
    for block in payload.get("boxscore", {}).get("players", []) or []:
        team_data = block.get("team", {})
        team_name = (
            team_data.get("displayName")
            or team_data.get("name")
            or ""
        )
        team_id = team_aliases.get(
            _normalise_external_team_name(team_name)
        )
        if team_id is None:
            continue

        for category in block.get("statistics", []) or []:
            labels = [str(x).strip().lower() for x in category.get("labels", [])]
            minute_index = next(
                (
                    i for i,label in enumerate(labels)
                    if label in {"min", "mins", "minutes"}
                ),
                None,
            )
            if minute_index is None:
                continue
            for athlete_row in category.get("athletes", []) or []:
                stats = athlete_row.get("stats", []) or []
                if minute_index >= len(stats):
                    continue
                minutes = _espn_minute_value(stats[minute_index])
                athlete = athlete_row.get("athlete", {})
                player = (
                    athlete.get("displayName")
                    or athlete.get("fullName")
                    or athlete.get("shortName")
                    or ""
                )
                if player and minutes is not None:
                    rows.append({
                        "team_id": int(team_id),
                        "player_name_external": str(player),
                        "minutes": float(np.clip(minutes, 0.0, 130.0)),
                        "minutes_source": "espn_boxscore",
                    })

    # Roster/substitution fallback.
    if not rows:
        for block in payload.get("rosters", []) or []:
            team_data = block.get("team", {})
            team_name = (
                team_data.get("displayName")
                or team_data.get("name")
                or ""
            )
            team_id = team_aliases.get(
                _normalise_external_team_name(team_name)
            )
            if team_id is None:
                continue

            for entry in block.get("roster", []) or []:
                athlete = entry.get("athlete", {})
                player = (
                    athlete.get("displayName")
                    or athlete.get("fullName")
                    or athlete.get("shortName")
                    or ""
                )
                if not player:
                    continue

                minutes = _espn_minute_value(entry.get("minutes"))
                if minutes is None:
                    starter = bool(entry.get("starter", False))
                    sub_in = _espn_minute_value(
                        entry.get("subbedIn"),
                        0.0 if starter else None,
                    )
                    sub_out = _espn_minute_value(
                        entry.get("subbedOut"),
                        90.0 if starter else None,
                    )
                    if sub_in is not None and sub_out is not None:
                        minutes = max(0.0, sub_out - sub_in)

                if minutes is not None:
                    rows.append({
                        "team_id": int(team_id),
                        "player_name_external": str(player),
                        "minutes": float(np.clip(minutes, 0.0, 130.0)),
                        "minutes_source": "espn_roster_fallback",
                    })

    return rows


def _match_non_pl_minutes_to_fpl(
    raw: pd.DataFrame,
    current_players: pd.DataFrame,
) -> pd.DataFrame:
    columns = [
        "event_id",
        "team_id",
        "player_key",
        "player_name_external",
        "minutes",
        "minutes_source",
    ]
    if raw.empty:
        return pd.DataFrame(columns=columns)

    fpl = current_players[
        ["player_key", "team_id", "full_name", "web_name"]
    ].copy()
    fpl["full_key"] = fpl["full_name"].map(normalise_player_name)
    fpl["web_key"] = fpl["web_name"].map(normalise_player_name)

    output = []
    for row in raw.itertuples(index=False):
        target = normalise_player_name(row.player_name_external)
        candidates = fpl.loc[
            fpl["team_id"].eq(int(row.team_id))
        ].copy()
        if candidates.empty or not target:
            continue

        exact = candidates.loc[
            candidates["full_key"].eq(target)
            | candidates["web_key"].eq(target)
        ]
        if len(exact) == 1:
            match = exact.iloc[0]
        else:
            candidates["_score"] = candidates["full_key"].map(
                lambda name: SequenceMatcher(
                    None, target, str(name)
                ).ratio()
            )
            candidates = candidates.sort_values("_score", ascending=False)
            if (
                candidates.empty
                or float(candidates.iloc[0]["_score"]) < 0.90
            ):
                continue
            match = candidates.iloc[0]

        output.append({
            "event_id": str(row.event_id),
            "team_id": int(row.team_id),
            "player_key": match["player_key"],
            "player_name_external": row.player_name_external,
            "minutes": float(row.minutes),
            "minutes_source": row.minutes_source,
        })

    return pd.DataFrame(output, columns=columns)


def load_live_all_comp_player_minutes(
    schedule: pd.DataFrame,
    current_players: pd.DataFrame,
    team_table: pd.DataFrame,
) -> pd.DataFrame:
    """
    Load non-PL player minutes with a circuit breaker.

    A dead upstream source must not trigger hundreds of failing HTTP requests.
    After LIVE_ALL_COMP_MAX_CONSECUTIVE_FAILURES consecutive failures the source
    is disabled for this execution and the PL-only minutes model remains active.
    """
    columns = [
        "event_id",
        "team_id",
        "player_key",
        "player_name_external",
        "minutes",
        "minutes_source",
    ]
    if not LIVE_ALL_COMP_WORKLOAD_ENABLED or schedule.empty:
        return pd.DataFrame(columns=columns)

    cached = pd.DataFrame(columns=columns)
    if ALL_COMP_PLAYER_MINUTES_CACHE_PATH.exists():
        try:
            cached = pd.read_csv(
                ALL_COMP_PLAYER_MINUTES_CACHE_PATH,
                compression="gzip",
            )
            cached["event_id"] = cached["event_id"].astype(str)
        except Exception as exc:
            warnings.warn(
                f"Could not read non-PL player-minute cache: {exc}"
            )

    completed = (
        schedule.loc[schedule["completed"].fillna(False)]
        [["event_id", "competition_code"]]
        .drop_duplicates("event_id")
    )
    known = set(cached["event_id"].astype(str)) if not cached.empty else set()
    to_fetch = completed.loc[
        ~completed["event_id"].astype(str).isin(known)
    ]

    aliases = _fpl_team_alias_lookup(team_table)
    headers = {"User-Agent": f"Mozilla/5.0 FPL-Predictor/{PROJECT_VERSION}"}
    raw_rows = []
    consecutive_failures = 0
    attempted = 0
    successful = 0
    circuit_open = False

    for row in to_fetch.itertuples(index=False):
        if circuit_open:
            break

        attempted += 1
        url = (
            "https://site.api.espn.com/apis/site/v2/sports/soccer/"
            f"{row.competition_code}/summary?event={row.event_id}"
        )
        try:
            response = requests.get(
                url,
                headers=headers,
                timeout=float(LIVE_ALL_COMP_REQUEST_TIMEOUT_SECONDS),
            )
            response.raise_for_status()
            parsed = extract_espn_summary_player_minutes(
                response.json(),
                aliases,
            )
            for item in parsed:
                item["event_id"] = str(row.event_id)
                raw_rows.append(item)
            successful += 1
            consecutive_failures = 0
        except Exception as exc:
            consecutive_failures += 1
            if LIVE_ALL_COMP_STRICT:
                raise
            warnings.warn(
                f"Could not load non-PL minutes for event {row.event_id}: {exc}"
            )
            if (
                consecutive_failures
                >= int(LIVE_ALL_COMP_MAX_CONSECUTIVE_FAILURES)
            ):
                circuit_open = True
                warnings.warn(
                    "Non-PL workload source circuit breaker opened after "
                    f"{consecutive_failures} consecutive failures. "
                    "Skipping remaining requests and using the PL-only "
                    "minutes baseline for unavailable workload context."
                )

    if raw_rows:
        matched = _match_non_pl_minutes_to_fpl(
            pd.DataFrame(raw_rows),
            current_players,
        )
        cached = (
            pd.concat([cached, matched], ignore_index=True)
            .drop_duplicates(["event_id", "player_key"], keep="last")
        )
        cached.to_csv(
            ALL_COMP_PLAYER_MINUTES_CACHE_PATH,
            index=False,
            compression="gzip",
        )

    # Expose a lightweight audit payload without changing the return schema.
    globals()["LIVE_ALL_COMP_PLAYER_MINUTES_AUDIT"] = {
        "attempted_requests": int(attempted),
        "successful_requests": int(successful),
        "consecutive_failures_at_stop": int(consecutive_failures),
        "circuit_open": bool(circuit_open),
        "remaining_skipped": int(max(0, len(to_fetch) - attempted)),
    }

    return cached[columns] if not cached.empty else pd.DataFrame(columns=columns)


def attach_all_comp_workload_features(
    feature_data: pd.DataFrame,
    schedule: pd.DataFrame,
    player_minutes: pd.DataFrame,
) -> pd.DataFrame:
    result = feature_data.copy()

    defaults = {
        "previous_non_pl_event_id": pd.NA,
        "previous_non_pl_competition": pd.NA,
        "days_since_previous_all_comp_fixture": np.nan,
        "days_until_next_all_comp_fixture": np.nan,
        "non_pl_matches_previous_7d": 0.0,
        "non_pl_matches_next_7d": 0.0,
        "previous_non_pl_player_minutes": np.nan,
        "previous_non_pl_player_minutes_source": pd.NA,
        "all_comp_workload_data_available": False,
    }
    for column, default in defaults.items():
        result[column] = default

    if not LIVE_ALL_COMP_WORKLOAD_ENABLED or schedule.empty:
        return result

    by_team = {
        int(team_id): group.sort_values("kickoff_time")
        for team_id, group in schedule.groupby("team_id")
    }
    minute_lookup = {}
    if not player_minutes.empty:
        for row in player_minutes.itertuples(index=False):
            minute_lookup[
                (str(row.event_id), str(row.player_key))
            ] = (float(row.minutes), row.minutes_source)

    kickoffs = pd.to_datetime(
        result["kickoff_time"],
        utc=True,
        errors="coerce",
    )

    for idx in result.index:
        kickoff = kickoffs.loc[idx]
        if pd.isna(kickoff):
            continue
        team_schedule = by_team.get(int(result.at[idx, "team_id"]))
        if team_schedule is None or team_schedule.empty:
            continue

        before = team_schedule.loc[
            team_schedule["kickoff_time"].lt(kickoff)
        ]
        after = team_schedule.loc[
            team_schedule["kickoff_time"].gt(kickoff)
        ]

        if not before.empty:
            prev = before.iloc[-1]
            days = (
                kickoff - prev["kickoff_time"]
            ).total_seconds() / 86400.0
            result.at[idx, "previous_non_pl_event_id"] = str(prev["event_id"])
            result.at[idx, "previous_non_pl_competition"] = prev["competition"]
            result.at[idx, "days_since_previous_all_comp_fixture"] = days

            key = (
                str(prev["event_id"]),
                str(result.at[idx, "player_key"]),
            )
            if key in minute_lookup:
                mins, source = minute_lookup[key]
                result.at[idx, "previous_non_pl_player_minutes"] = mins
                result.at[
                    idx,
                    "previous_non_pl_player_minutes_source",
                ] = source

        if not after.empty:
            next_row = after.iloc[0]
            result.at[idx, "days_until_next_all_comp_fixture"] = (
                next_row["kickoff_time"] - kickoff
            ).total_seconds() / 86400.0

        result.at[idx, "non_pl_matches_previous_7d"] = float(
            team_schedule["kickoff_time"].between(
                kickoff - pd.Timedelta(days=7),
                kickoff,
                inclusive="left",
            ).sum()
        )
        result.at[idx, "non_pl_matches_next_7d"] = float(
            team_schedule["kickoff_time"].between(
                kickoff,
                kickoff + pd.Timedelta(days=7),
                inclusive="right",
            ).sum()
        )
        result.at[idx, "all_comp_workload_data_available"] = True

    return result


def apply_all_comp_minutes_overlay(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    result = frame.copy()
    result["pre_all_comp_start_probability"] = result["start_probability"]
    result["pre_all_comp_expected_minutes"] = result["expected_minutes"]
    result["all_comp_start_multiplier"] = 1.0

    if not LIVE_ALL_COMP_WORKLOAD_ENABLED:
        result["all_comp_minutes_adjustment"] = 0.0
        return result

    rest = pd.to_numeric(
        result.get("days_since_previous_all_comp_fixture"),
        errors="coerce",
    )
    previous_minutes = pd.to_numeric(
        result.get("previous_non_pl_player_minutes"),
        errors="coerce",
    )
    next_days = pd.to_numeric(
        result.get("days_until_next_all_comp_fixture"),
        errors="coerce",
    )

    multiplier = pd.Series(1.0, index=result.index, dtype=float)

    for day_bucket, factor in LIVE_ALL_COMP_PREVIOUS_REST_START_FACTORS.items():
        mask = (
            rest.notna()
            & np.floor(rest.clip(lower=0.0)).eq(float(day_bucket))
        )
        multiplier.loc[mask] = np.minimum(
            multiplier.loc[mask],
            float(factor),
        )

    heavy = previous_minutes.ge(LIVE_ALL_COMP_HEAVY_MINUTES_THRESHOLD)
    medium = (
        previous_minutes.ge(LIVE_ALL_COMP_MEDIUM_MINUTES_THRESHOLD)
        & ~heavy
    )
    multiplier.loc[heavy] *= LIVE_ALL_COMP_HEAVY_MINUTES_START_FACTOR
    multiplier.loc[medium] *= LIVE_ALL_COMP_MEDIUM_MINUTES_START_FACTOR

    future = (
        next_days.notna()
        & next_days.ge(0.0)
        & next_days.le(LIVE_ALL_COMP_NEXT_MATCH_WITHIN_DAYS)
    )
    multiplier.loc[future] *= LIVE_ALL_COMP_NEXT_MATCH_START_FACTOR
    multiplier = multiplier.clip(0.72, 1.0)
    result["all_comp_start_multiplier"] = multiplier

    appearance_factor = (
        1.0
        - (1.0 - multiplier) * LIVE_ALL_COMP_APPEARANCE_FACTOR_SHARE
    ).clip(0.85, 1.0)

    result["appearance_probability"] = (
        result["appearance_probability"] * appearance_factor
    ).clip(0.0, 1.0)
    result["start_probability"] = np.minimum(
        result["start_probability"] * multiplier,
        result["appearance_probability"],
    )
    result["substitute_appearance_probability"] = np.maximum(
        result["appearance_probability"] - result["start_probability"],
        0.0,
    )
    result["no_appearance_probability"] = (
        1.0 - result["appearance_probability"]
    )
    result["sixty_plus_probability"] = np.minimum(
        result["sixty_plus_probability"] * multiplier,
        result["start_probability"],
    )
    result["expected_minutes"] = (
        result["start_probability"] * result["minutes_if_start"]
        + result["substitute_appearance_probability"] * result["minutes_if_sub"]
    ).clip(0.0, 90.0)
    result["rotation_risk"] = 1.0 - result["start_probability"]
    result["all_comp_minutes_adjustment"] = (
        result["expected_minutes"] - result["pre_all_comp_expected_minutes"]
    )
    return result


live_all_comp_schedule = load_live_all_comp_schedule(
    teams,
    fixtures,
)
live_all_comp_player_minutes = load_live_all_comp_player_minutes(
    live_all_comp_schedule,
    players,
    teams,
)

print(
    "All-competition schedule status:",
    live_all_comp_schedule_status.get("status", "UNKNOWN"),
)
print(
    "All-competition live window:",
    str(live_all_comp_schedule_status.get("window_start", "")),
    "→",
    str(live_all_comp_schedule_status.get("window_end", "")),
)
print(
    "Live non-PL fixtures loaded:",
    live_all_comp_schedule_status.get("unique_events", 0),
)
print(
    "All-competition requests:",
    f"{live_all_comp_schedule_status.get('successful_requests', 0)}/"
    f"{live_all_comp_schedule_status.get('total_requests', 0)} successful",
)
if live_all_comp_schedule_status.get("circuit_open", False):
    print(
        "All-competition circuit breaker:",
        "OPEN",
        "| consecutive failures:",
        live_all_comp_schedule_status.get(
            "consecutive_failures_at_stop", 0
        ),
        "| remaining chunk requests skipped:",
        live_all_comp_schedule_status.get(
            "remaining_requests_skipped", 0
        ),
    )
if live_all_comp_schedule_status.get("failure_cache_active", False):
    print(
        "All-competition negative cache active until:",
        live_all_comp_schedule_status.get(
            "failure_cache_expires_at_utc"
        ),
    )
print(
    "Completed non-PL player-minute records:",
    len(live_all_comp_player_minutes),
)

In [ ]:
final_minutes_training_data = historical_model_data.copy()
final_minutes_training_data["position"] = (
    normalise_scoring_position(
        final_minutes_training_data["position"]
    )
)
final_minutes_training_data = final_minutes_training_data.loc[
    final_minutes_training_data["position"].isin(
        PLAYER_SCORING_POSITIONS
    )
].copy()

final_minutes_models = fit_minutes_model_bundle(
    final_minutes_training_data
)

final_team_goal_models = fit_team_goal_models(
    historical_team_fixtures
)


def apply_live_minutes_adjustments(
    feature_data: pd.DataFrame,
    model_bundle: dict[str, Pipeline],
    first_gameweek: int,
) -> pd.DataFrame:
    """Predict live minutes and apply official availability information."""
    raw_predictions = predict_minutes_components(
        model_bundle,
        feature_data,
    )

    result = pd.concat(
        [
            feature_data.reset_index(drop=True),
            raw_predictions.reset_index(drop=True),
        ],
        axis=1,
    )

    result["availability_multiplier"] = (
        next_gameweek_availability_multiplier(
            result,
            first_gameweek,
        )
    )

    result["model_appearance_probability"] = result[
        "appearance_probability"
    ]

    result["appearance_probability"] *= result[
        "availability_multiplier"
    ]

    result["start_probability"] = np.minimum(
        result["start_probability"]
        * result["availability_multiplier"],
        result["appearance_probability"],
    )

    result["substitute_appearance_probability"] = np.maximum(
        result["appearance_probability"]
        - result["start_probability"],
        0.0,
    )
    result["no_appearance_probability"] = (
        1.0 - result["appearance_probability"]
    )

    result["sixty_plus_probability"] = np.minimum(
        result["sixty_plus_probability"]
        * result["availability_multiplier"],
        result["start_probability"],
    )

    result["expected_minutes"] = (
        result["start_probability"]
        * result["minutes_if_start"]
        + result["substitute_appearance_probability"]
        * result["minutes_if_sub"]
    ).clip(0.0, 90.0)

    result["rotation_risk"] = 1.0 - result["start_probability"]

    result = apply_all_comp_minutes_overlay(result)

    return result


final_live_minutes_feature_data = live_minutes_feature_data.copy()
final_live_minutes_feature_data = attach_all_comp_workload_features(
    final_live_minutes_feature_data,
    live_all_comp_schedule,
    live_all_comp_player_minutes,
)
final_live_minutes_feature_data["position"] = (
    normalise_scoring_position(
        final_live_minutes_feature_data["position"]
    )
)

final_live_minutes_predictions = apply_live_minutes_adjustments(
    final_live_minutes_feature_data,
    final_minutes_models,
    first_planning_gameweek,
)


### Load richer current-season player outcomes

The minutes section only required starts and minutes. Expected points also needs attacking, goalkeeping, bonus, card and defensive-action history.

In [ ]:
CURRENT_POINTS_STAT_COLUMNS = [
    "total_points",
    "goals_scored",
    "assists",
    "clean_sheets",
    "goals_conceded",
    "own_goals",
    "penalties_saved",
    "penalties_missed",
    "yellow_cards",
    "red_cards",
    "saves",
    "bonus",
    "bps",
    "influence",
    "creativity",
    "threat",
    "ict_index",
    "expected_goals",
    "expected_assists",
    "expected_goal_involvements",
    "expected_goals_conceded",
    "tackles",
    "recoveries",
    "clearances_blocks_interceptions",
    "defensive_contribution",
]

CURRENT_POINTS_RATE_COLUMNS = {
    "gw_total_points": "points_per_fixture",
    "gw_goals_scored": "goals_per_fixture",
    "gw_assists": "assists_per_fixture",
    "gw_saves": "saves_per_fixture",
    "gw_bonus": "bonus_per_fixture",
    "gw_bps": "bps_per_fixture",
    "gw_expected_goals": "xg_per_fixture",
    "gw_expected_assists": "xa_per_fixture",
    "gw_influence": "influence_per_fixture",
    "gw_creativity": "creativity_per_fixture",
    "gw_threat": "threat_per_fixture",
    "gw_yellow_cards": "yellow_cards_per_fixture",
    "gw_red_cards": "red_cards_per_fixture",
    "gw_tackles": "tackles_per_fixture",
    "gw_recoveries": "recoveries_per_fixture",
    "gw_clearances_blocks_interceptions": "cbi_per_fixture",
    "gw_defensive_contribution": (
        "defensive_contribution_per_fixture"
    ),
    "gw_defensive_return": "defensive_return_rate",
}


def load_current_season_points_gameweeks(
    gameweek_table: pd.DataFrame,
    current_players: pd.DataFrame,
    current_team_fixtures: pd.DataFrame,
) -> pd.DataFrame:
    """Load completed current-season FPL outcomes by player and Gameweek."""
    base_minutes = load_current_season_player_gameweeks(
        gameweek_table,
        current_players,
        current_team_fixtures,
    )

    completed_gameweeks = resolved_completed_gameweek_ids(
        gameweek_table,
        current_team_fixtures,
    )

    output_columns = [
        "season",
        "season_start_year",
        "player_key",
        "gameweek",
    ] + CURRENT_GAMEWEEK_RAW_COLUMNS + [
        f"gw_{column}"
        for column in CURRENT_POINTS_STAT_COLUMNS
    ] + ["gw_defensive_return"]

    if not completed_gameweeks:
        return pd.DataFrame(columns=output_columns)

    player_lookup = (
        current_players[
            ["id", "player_key", "position"]
        ]
        .rename(columns={"id": "player_id"})
    )
    player_lookup["position"] = (
        player_lookup["position"]
        .astype("string")
        .replace({"GKP": "GK"})
    )

    rows = []
    for gameweek in completed_gameweeks:
        payload = get_fpl_data(f"event/{gameweek}/live/")
        for element in payload.get("elements", []):
            row = {
                "player_id": element.get("id"),
                "gameweek": gameweek,
            }
            stats = element.get("stats", {})
            for column in CURRENT_POINTS_STAT_COLUMNS:
                row[f"gw_{column}"] = pd.to_numeric(
                    stats.get(column),
                    errors="coerce",
                )
            rows.append(row)

    current_points = pd.DataFrame(rows)
    current_points["player_id"] = pd.to_numeric(
        current_points["player_id"],
        errors="coerce",
    ).astype("Int64")

    current_points = current_points.merge(
        player_lookup,
        on="player_id",
        how="left",
        validate="many_to_one",
    )

    current_points = base_minutes.merge(
        current_points.drop(columns=["player_id"]),
        on=["player_key", "gameweek"],
        how="left",
        validate="one_to_one",
    )

    current_tackles = pd.to_numeric(
        current_points.get("gw_tackles"),
        errors="coerce",
    )
    current_recoveries = pd.to_numeric(
        current_points.get("gw_recoveries"),
        errors="coerce",
    )
    current_cbi = pd.to_numeric(
        current_points.get(
            "gw_clearances_blocks_interceptions"
        ),
        errors="coerce",
    )
    current_has_actions = (
        current_tackles.notna()
        | current_recoveries.notna()
        | current_cbi.notna()
    )
    current_actions = (
        current_tackles.fillna(0.0)
        + current_cbi.fillna(0.0)
        + np.where(
            current_points["position"].isin(["MID", "FWD"]),
            current_recoveries.fillna(0.0),
            0.0,
        )
    )
    current_threshold = current_points["position"].map({
        "DEF": 10.0,
        "MID": 12.0,
        "FWD": 12.0,
    })
    current_points["gw_defensive_return"] = np.where(
        current_has_actions & current_threshold.notna(),
        current_actions.ge(current_threshold).astype(float),
        np.nan,
    )

    for column in output_columns:
        if column not in current_points.columns:
            current_points[column] = np.nan

    return current_points[output_columns]


current_season_points_gameweeks = (
    load_current_season_points_gameweeks(
        gameweeks,
        players,
        team_fixtures,
    )
)

print(
    "Completed current-season points-history rows: "
    f"{len(current_season_points_gameweeks):,}"
)


### Build current as-of event-rate features

In [ ]:
def build_current_event_as_of_features(
    archived_player_gameweeks: pd.DataFrame,
    current_player_gameweeks: pd.DataFrame,
    current_players: pd.DataFrame,
) -> pd.DataFrame:
    """Create latest rolling event rates for every current player."""
    key_columns = [
        "season",
        "season_start_year",
        "player_key",
        "gameweek",
    ]

    raw_columns = [
        "gw_fixture_count",
        "gw_minutes",
    ] + list(CURRENT_POINTS_RATE_COLUMNS)

    archived = archived_player_gameweeks.copy()
    current = current_player_gameweeks.copy()

    for frame in [archived, current]:
        for column in key_columns + raw_columns:
            if column not in frame.columns:
                frame[column] = np.nan

    history = (
        pd.concat(
            [
                archived[key_columns + raw_columns],
                current[key_columns + raw_columns],
            ],
            ignore_index=True,
        )
        .sort_values(
            ["player_key", "season_start_year", "gameweek"]
        )
        .reset_index(drop=True)
    )

    for column in raw_columns:
        history[column] = pd.to_numeric(
            history[column],
            errors="coerce",
        )

    history_groups = {
        player_key: group.sort_values(
            ["season_start_year", "gameweek"]
        )
        for player_key, group in history.groupby(
            "player_key",
            sort=False,
        )
    }

    rows = []
    for player_key in current_players["player_key"].drop_duplicates():
        player_history = history_groups.get(player_key)
        features: dict[str, object] = {"player_key": player_key}

        for feature_name in CURRENT_POINTS_RATE_COLUMNS.values():
            for window in ROLLING_GAMEWEEK_WINDOWS:
                features[
                    f"{feature_name}_last_{window}_gws"
                ] = np.nan
            features[f"career_{feature_name}"] = np.nan

        if player_history is not None and not player_history.empty:
            fixture_count = player_history[
                "gw_fixture_count"
            ].fillna(0.0)

            for raw_column, feature_name in (
                CURRENT_POINTS_RATE_COLUMNS.items()
            ):
                for window in ROLLING_GAMEWEEK_WINDOWS:
                    recent = player_history.tail(window)
                    available_recent = recent[raw_column].notna()
                    recent_fixtures = recent.loc[
                        available_recent,
                        "gw_fixture_count",
                    ].fillna(0.0).sum()

                    if recent_fixtures > 0:
                        features[
                            f"{feature_name}_last_{window}_gws"
                        ] = (
                            recent.loc[
                                available_recent,
                                raw_column,
                            ].sum()
                            / recent_fixtures
                        )

                available_career = player_history[raw_column].notna()
                career_fixtures = player_history.loc[
                    available_career,
                    "gw_fixture_count",
                ].fillna(0.0).sum()
                if career_fixtures > 0:
                    features[f"career_{feature_name}"] = (
                        player_history.loc[
                            available_career,
                            raw_column,
                        ].sum()
                        / career_fixtures
                    )

        rows.append(features)

    return pd.DataFrame(rows)


current_event_as_of_history_unshrunk = (
    build_current_event_as_of_features(
        historical_player_gameweeks,
        current_season_points_gameweeks,
        players,
    )
)

current_event_as_of_history = (
    apply_early_season_player_history_shrinkage(
        current_event_as_of_history_unshrunk,
        historical_player_gameweeks,
        current_season_points_gameweeks,
        players,
        CURRENT_POINTS_RATE_COLUMNS,
        shrink_last_gw=False,
    )
)


### Build live team-strength features

Completed 2026/27 fixtures are appended to the archive. Upcoming fixtures then receive the latest pre-deadline team and opponent rates.

In [ ]:
def current_team_key_lookup(
    current_teams: pd.DataFrame,
) -> pd.DataFrame:
    """Create stable current team keys from official team codes."""
    lookup = current_teams[
        ["team_id", "code"]
    ].copy()
    lookup["code"] = pd.to_numeric(
        lookup["code"],
        errors="coerce",
    ).astype("Int64")
    lookup["team_key"] = (
        "current:team:"
        + lookup["team_id"].astype("Int64").astype(str)
    )
    has_code = lookup["code"].notna()
    lookup.loc[has_code, "team_key"] = (
        "code:" + lookup.loc[has_code, "code"].astype(str)
    )
    return lookup


def build_current_completed_team_results(
    current_fixtures: pd.DataFrame,
    current_teams: pd.DataFrame,
    gameweek_table: pd.DataFrame,
) -> pd.DataFrame:
    """Convert completed current fixtures to two team-perspective rows."""
    lookup = current_team_key_lookup(current_teams)
    key_by_id = lookup.set_index("team_id")["team_key"]
    code_by_id = lookup.set_index("team_id")["code"]

    completed_gameweeks = set(
        resolved_completed_gameweek_ids(
            gameweek_table,
            current_fixtures,
        )
    )
    completed = current_fixtures.loc[
        current_fixtures["event"].isin(completed_gameweeks)
        & current_fixtures["team_h_score"].notna()
        & current_fixtures["team_a_score"].notna()
    ].copy()

    rows = []
    for fixture in completed.itertuples(index=False):
        rows.extend([
            {
                "season": CURRENT_SEASON,
                "season_start_year": CURRENT_SEASON_START_YEAR,
                "gameweek": fixture.event,
                "fixture_id": fixture.id,
                "kickoff_time": fixture.kickoff_time,
                "team_key": key_by_id.get(fixture.team_h),
                "team_code": code_by_id.get(fixture.team_h),
                "opponent_team_key": key_by_id.get(fixture.team_a),
                "was_home": True,
                "team_goals": fixture.team_h_score,
                "opponent_goals": fixture.team_a_score,
            },
            {
                "season": CURRENT_SEASON,
                "season_start_year": CURRENT_SEASON_START_YEAR,
                "gameweek": fixture.event,
                "fixture_id": fixture.id,
                "kickoff_time": fixture.kickoff_time,
                "team_key": key_by_id.get(fixture.team_a),
                "team_code": code_by_id.get(fixture.team_a),
                "opponent_team_key": key_by_id.get(fixture.team_h),
                "was_home": False,
                "team_goals": fixture.team_a_score,
                "opponent_goals": fixture.team_h_score,
            },
        ])

    return pd.DataFrame(rows)


def build_live_team_as_of_features(
    archived_team_fixtures: pd.DataFrame,
    current_completed_results: pd.DataFrame,
    current_teams: pd.DataFrame,
) -> pd.DataFrame:
    """Calculate latest team rates, including xG/xGA when coverage exists."""
    history_columns = [
        "team_key",
        "kickoff_time",
        "team_goals",
        "opponent_goals",
    ]
    if "team_xg" in archived_team_fixtures.columns:
        history_columns += ["team_xg", "opponent_xg"]

    archived = archived_team_fixtures[history_columns].copy()
    current = current_completed_results.copy()
    for column in history_columns:
        if column not in current.columns:
            current[column] = np.nan

    history = (
        pd.concat(
            [archived, current[history_columns]],
            ignore_index=True,
        )
        .sort_values(["team_key", "kickoff_time"])
    )
    history["team_clean_sheet"] = (
        history["opponent_goals"].eq(0).astype(float)
    )

    lookup = current_team_key_lookup(current_teams)
    rows = []

    for team_record in lookup.itertuples(index=False):
        team_history = history.loc[
            history["team_key"].eq(team_record.team_key)
        ].copy()

        features = {
            "team_id": team_record.team_id,
            "team_key": team_record.team_key,
        }

        for window in TEAM_ROLLING_WINDOWS:
            recent = team_history.tail(window)
            features[f"team_goals_for_last_{window}"] = recent["team_goals"].mean()
            features[f"team_goals_against_last_{window}"] = recent["opponent_goals"].mean()
            features[f"team_clean_sheet_rate_last_{window}"] = recent["team_clean_sheet"].mean()
            if "team_xg" in team_history.columns:
                features[f"team_xg_for_last_{window}"] = recent["team_xg"].mean()
                features[f"team_xg_against_last_{window}"] = recent["opponent_xg"].mean()

        features["team_career_goals_for"] = team_history["team_goals"].mean()
        features["team_career_goals_against"] = team_history["opponent_goals"].mean()
        features["team_career_clean_sheet_rate"] = team_history["team_clean_sheet"].mean()
        if "team_xg" in team_history.columns:
            features["team_career_xg_for"] = team_history["team_xg"].mean()
            features["team_career_xg_against"] = team_history["opponent_xg"].mean()

        rows.append(features)

    result = pd.DataFrame(rows)
    # This table represents the subject team only. Do NOT materialise
    # opponent_* model columns here: doing so causes pandas to suffix the real
    # opponent columns to _x/_y during the later opponent merge.
    own_team_model_features = [
        column
        for column in TEAM_FEATURE_COLUMNS
        if column.startswith("team_")
    ]
    for column in own_team_model_features:
        if column not in result.columns:
            result[column] = np.nan
    return result




def apply_early_season_team_shrinkage(
    team_features: pd.DataFrame,
    archived_team_fixtures: pd.DataFrame,
    current_completed_results: pd.DataFrame,
) -> pd.DataFrame:
    """Shrink sparse current team GF/GA/CS rates toward prior information."""
    result = team_features.copy()
    result["team_early_season_shrinkage_active"] = False
    result["team_early_season_current_matches"] = 0.0
    result["team_early_season_archived_pl_matches"] = 0.0
    result["team_early_season_current_weight"] = 1.0
    result["team_early_season_prior_source"] = "none"

    if not EARLY_SEASON_SHRINKAGE_ENABLED or result.empty:
        return result

    archived = archived_team_fixtures.copy()
    current = current_completed_results.copy()

    for column in ["team_goals", "opponent_goals"]:
        archived[column] = pd.to_numeric(
            archived[column], errors="coerce"
        )
        current[column] = pd.to_numeric(
            current[column], errors="coerce"
        )

    archived["_clean_sheet"] = (
        archived["opponent_goals"].eq(0).astype(float)
    )
    current["_clean_sheet"] = (
        current["opponent_goals"].eq(0).astype(float)
    )

    league_priors = {
        "goals_for": float(archived["team_goals"].mean()),
        "goals_against": float(archived["opponent_goals"].mean()),
        "clean_sheet_rate": float(archived["_clean_sheet"].mean()),
    }

    archived_groups = {
        key: group.sort_values("kickoff_time")
        for key, group in archived.groupby("team_key", sort=False)
    }
    current_groups = {
        key: group.sort_values("kickoff_time")
        for key, group in current.groupby("team_key", sort=False)
    }

    fade = max(float(EARLY_SEASON_TEAM_FADE_MATCHES), 1.0)
    stable_archived = max(
        float(EARLY_SEASON_TEAM_STABLE_ARCHIVED_MATCHES),
        1.0,
    )
    metric_sources = {
        "goals_for": "team_goals",
        "goals_against": "opponent_goals",
        "clean_sheet_rate": "_clean_sheet",
    }

    for idx, row in result.iterrows():
        team_key = row["team_key"]
        archived_team = archived_groups.get(team_key)
        current_team = current_groups.get(team_key)

        archived_matches = (
            float(len(archived_team))
            if archived_team is not None else 0.0
        )
        current_matches = (
            float(len(current_team))
            if current_team is not None else 0.0
        )

        result.at[
            idx, "team_early_season_current_matches"
        ] = current_matches
        result.at[
            idx, "team_early_season_archived_pl_matches"
        ] = archived_matches

        active = (
            current_matches > 0.0
            and current_matches < fade
            and archived_matches < stable_archived
        )
        if not active:
            continue

        result.at[idx, "team_early_season_shrinkage_active"] = True
        result.at[idx, "team_early_season_current_weight"] = min(
            1.0, current_matches / fade
        )

        archived_prior_weight = min(
            1.0, archived_matches / stable_archived
        )

        for metric, source_column in metric_sources.items():
            league_prior = league_priors[metric]

            archived_rate = (
                float(
                    pd.to_numeric(
                        archived_team[source_column],
                        errors="coerce",
                    ).mean()
                )
                if archived_team is not None
                and not archived_team.empty
                else np.nan
            )

            if np.isfinite(archived_rate):
                prior = (
                    archived_prior_weight * archived_rate
                    + (1.0 - archived_prior_weight) * league_prior
                )
                prior_source = (
                    "archived_pl"
                    if archived_prior_weight >= 0.999
                    else "archived_pl+league"
                )
            else:
                prior = league_prior
                prior_source = "league"

            for window in TEAM_ROLLING_WINDOWS:
                target = f"team_{metric}_last_{window}"
                if target not in result.columns:
                    continue

                recent_current = (
                    current_team.tail(int(window))
                    if current_team is not None
                    else pd.DataFrame()
                )
                if recent_current.empty:
                    observed = np.nan
                    evidence = 0.0
                else:
                    observed = float(
                        pd.to_numeric(
                            recent_current[source_column],
                            errors="coerce",
                        ).mean()
                    )
                    evidence = float(len(recent_current))

                current_weight = min(1.0, evidence / fade)
                if np.isfinite(observed) and np.isfinite(prior):
                    value = (
                        current_weight * observed
                        + (1.0 - current_weight) * prior
                    )
                elif np.isfinite(observed):
                    value = observed
                else:
                    value = prior

                if np.isfinite(value):
                    result.at[idx, target] = float(value)

            career_target = f"team_career_{metric}"
            if career_target in result.columns:
                observed = (
                    float(
                        pd.to_numeric(
                            current_team[source_column],
                            errors="coerce",
                        ).mean()
                    )
                    if current_team is not None
                    and not current_team.empty
                    else np.nan
                )
                current_weight = min(1.0, current_matches / fade)
                if np.isfinite(observed) and np.isfinite(prior):
                    value = (
                        current_weight * observed
                        + (1.0 - current_weight) * prior
                    )
                elif np.isfinite(observed):
                    value = observed
                else:
                    value = prior

                if np.isfinite(value):
                    result.at[idx, career_target] = float(value)

            result.at[
                idx, "team_early_season_prior_source"
            ] = prior_source

    return result



def attach_current_team_xg_context(
    completed_team_results: pd.DataFrame,
    player_gameweeks: pd.DataFrame,
    current_players: pd.DataFrame,
    current_teams: pd.DataFrame,
) -> pd.DataFrame:
    """
    Attach current-season team xG/xGA when a team has exactly one fixture in
    that Gameweek. FPL live player xG is Gameweek-aggregated, so DGWs are left
    missing rather than duplicating a GW total across fixtures.
    """
    result = completed_team_results.copy()
    result["team_xg"] = np.nan
    result["opponent_xg"] = np.nan

    if (
        player_gameweeks is None
        or player_gameweeks.empty
        or "gw_expected_goals" not in player_gameweeks.columns
    ):
        return result

    player_lookup = current_players[
        ["player_key", "team_id"]
    ].drop_duplicates("player_key")
    team_lookup = current_team_key_lookup(
        current_teams
    )[["team_id", "team_key"]]

    xg = player_gameweeks[
        ["player_key", "gameweek", "gw_expected_goals"]
    ].copy()
    xg["gw_expected_goals"] = pd.to_numeric(
        xg["gw_expected_goals"], errors="coerce"
    )
    xg = (
        xg.merge(
            player_lookup,
            on="player_key",
            how="left",
            validate="many_to_one",
        )
        .merge(
            team_lookup,
            on="team_id",
            how="left",
            validate="many_to_one",
        )
    )
    xg = (
        xg.groupby(
            ["gameweek", "team_key"],
            as_index=False,
            dropna=False,
        )
        .agg(team_xg=("gw_expected_goals", lambda s: s.sum(min_count=1)))
    )

    fixture_counts = (
        result.groupby(["gameweek", "team_key"])
        ["fixture_id"]
        .nunique()
        .rename("_fixture_count")
        .reset_index()
    )
    xg = xg.merge(
        fixture_counts,
        on=["gameweek", "team_key"],
        how="left",
    )
    xg.loc[xg["_fixture_count"].ne(1), "team_xg"] = np.nan
    xg = xg.drop(columns="_fixture_count")

    result = result.drop(columns=["team_xg", "opponent_xg"]).merge(
        xg,
        on=["gameweek", "team_key"],
        how="left",
        validate="many_to_one",
    )
    opponent = xg.rename(
        columns={
            "team_key": "opponent_team_key",
            "team_xg": "opponent_xg",
        }
    )
    result = result.merge(
        opponent,
        on=["gameweek", "opponent_team_key"],
        how="left",
        validate="many_to_one",
    )
    return result


current_completed_team_results = (
    build_current_completed_team_results(
        fixtures,
        teams,
        gameweeks,
    )
)
current_completed_team_results = attach_current_team_xg_context(
    current_completed_team_results,
    current_season_points_gameweeks,
    players,
    teams,
)

live_team_as_of_unshrunk = build_live_team_as_of_features(
    historical_team_fixtures,
    current_completed_team_results,
    teams,
)

live_team_as_of = apply_early_season_team_shrinkage(
    live_team_as_of_unshrunk,
    historical_team_fixtures,
    current_completed_team_results,
)

# Regression guard for the v2.2.2 opponent-feature collision: the subject-team
# snapshot must never carry trained opponent_* features before the opponent
# table is created.
unexpected_subject_opponent_features = sorted(
    set(live_team_as_of.columns)
    & {
        column
        for column in TEAM_FEATURE_COLUMNS
        if column.startswith("opponent_")
    }
)
if unexpected_subject_opponent_features:
    warnings.warn(
        "Removing opponent-model columns from the subject-team snapshot: "
        f"{unexpected_subject_opponent_features}",
        RuntimeWarning,
    )
    live_team_as_of = live_team_as_of.drop(
        columns=unexpected_subject_opponent_features,
        errors="ignore",
    )

opponent_as_of = live_team_as_of.rename(columns={
    "team_id": "opponent_id",
    "team_key": "opponent_team_key",
    **{
        column: column.replace("team_", "opponent_", 1)
        for column in live_team_as_of.columns
        if column.startswith("team_")
        and column not in ["team_id", "team_key"]
    },
})

live_team_fixture_features = (
    planning_team_fixtures
    .merge(
        live_team_as_of,
        on="team_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        opponent_as_of,
        on="opponent_id",
        how="left",
        validate="many_to_one",
    )
    .copy()
)

live_team_fixture_features["was_home"] = (
    live_team_fixture_features["is_home"]
)

# Production schema contract: materialise the exact trained input schema before
# prediction. Under normal operation this should require no repairs; if an
# upstream source/merge changes, the repair is logged instead of raising a
# pandas KeyError at model.predict().
live_team_schema_before = model_feature_contract_report(
    live_team_fixture_features,
    TEAM_FEATURE_COLUMNS,
)
live_team_fixture_features = materialise_model_feature_columns(
    live_team_fixture_features,
    TEAM_FEATURE_COLUMNS,
    context="live.team_fixture_features",
)
live_team_schema_after = model_feature_contract_report(
    live_team_fixture_features,
    TEAM_FEATURE_COLUMNS,
)
assert not live_team_schema_after["missing"], (
    "Live team schema contract failed after automatic materialisation: "
    f"{live_team_schema_after}"
)

if (
    live_team_schema_before["missing"]
    or live_team_schema_before["duplicate_columns"]
):
    print(
        "Live team model schema: REPAIRED",
        live_team_schema_before,
    )
else:
    print("Live team model schema: PASS")

live_team_diagnostic_columns = [
    column
    for column in live_team_fixture_features.columns
    if (
        column.startswith("team_early_season_")
        or column.startswith("opponent_early_season_")
    )
]

live_team_goal_predictions = pd.concat(
    [
        live_team_fixture_features[
            ["fixture_id", "team_id"] + live_team_diagnostic_columns
        ].reset_index(drop=True),
        predict_team_goal_components(
            final_team_goal_models,
            live_team_fixture_features,
        ).reset_index(drop=True),
    ],
    axis=1,
)

print(
    "Sparse/new-team early-season shrinkage active for:",
    int(
        live_team_as_of[
            "team_early_season_shrinkage_active"
        ].sum()
    ),
    "teams",
)

### Generate live fixture-level and Gameweek-level xP

### Autonomous xP calibration and sparse-player fallback

Two production corrections are applied after the transparent component model:

- **Sparse-player market prior:** if a player has essentially no Premier League history and no usable external prior, a weak empirical prior based on historical FPL price and position is blended into the raw projection.
- **Out-of-sample calibration:** a versioned calibration cache from Section 11 corrects systematic compression/bias by position and price band. Raw component xP is retained for auditability.

Calibration never uses 2026/27 outcomes.


In [ ]:
def normalise_fpl_model_position(
    frame: pd.DataFrame,
) -> pd.Series:
    """
    Return a canonical GK/DEF/MID/FWD position Series.

    Historical Vaastav/FPL tables generally expose `position`, with
    goalkeepers often encoded as `GKP`. Later modelling tables expose
    `model_position`. The autonomous prior layer must work on both because the
    historical price prior is built before the expected-points modelling table
    is created.
    """
    if "model_position" in frame.columns:
        raw = frame["model_position"]
    elif "position" in frame.columns:
        raw = frame["position"]
    else:
        raise KeyError(
            "No FPL position column is available. Expected either "
            "'model_position' or 'position'."
        )

    position = (
        raw.astype("string")
        .str.strip()
        .str.upper()
        .replace({
            "GKP": "GK",
            "GOALKEEPER": "GK",
            "DEFENDER": "DEF",
            "MIDFIELDER": "MID",
            "FORWARD": "FWD",
        })
    )

    # Some raw FPL tables can expose element-type IDs rather than labels.
    numeric = pd.to_numeric(raw, errors="coerce")
    if numeric.notna().any():
        position = position.mask(numeric.eq(1), "GK")
        position = position.mask(numeric.eq(2), "DEF")
        position = position.mask(numeric.eq(3), "MID")
        position = position.mask(numeric.eq(4), "FWD")

    return position


def build_position_price_prior_table(
    historical_rows: pd.DataFrame,
) -> pd.DataFrame:
    """
    Learn unconditional FPL points by position and half-million price band.

    This deliberately derives `model_position` locally so the function is safe
    on the raw historical modelling table as well as on later prediction
    tables. That removes an execution-order dependency introduced in v2.0.1.
    """
    data = historical_rows.copy()
    data["model_position"] = normalise_fpl_model_position(data)

    price_col = (
        "price_at_collection"
        if "price_at_collection" in data.columns
        else "price"
    )
    if price_col not in data.columns:
        raise KeyError(
            "The historical table has neither 'price_at_collection' nor 'price'."
        )
    if "target_fpl_points" not in data.columns:
        raise KeyError(
            "The historical table is missing 'target_fpl_points'."
        )

    data["_price"] = pd.to_numeric(
        data[price_col],
        errors="coerce",
    )
    data["_actual"] = pd.to_numeric(
        data["target_fpl_points"],
        errors="coerce",
    )

    data = data.loc[
        data["model_position"].isin(PLAYER_SCORING_POSITIONS)
        & data["_price"].notna()
        & data["_actual"].notna()
    ].copy()

    if data.empty:
        raise ValueError(
            "No usable historical player rows remained after position, "
            "price and points validation."
        )

    data["price_bin"] = (
        data["_price"] * 2
    ).round() / 2

    prior_table = (
        data.groupby(
            ["model_position", "price_bin"],
            as_index=False,
        )
        .agg(
            rows=("_actual", "size"),
            market_prior_points=("_actual", "mean"),
        )
        .loc[lambda frame: frame["rows"].ge(50)]
        .reset_index(drop=True)
    )

    missing_positions = (
        set(PLAYER_SCORING_POSITIONS)
        - set(prior_table["model_position"].dropna())
    )
    if missing_positions:
        # Do not silently produce a malformed autonomous prior. This should
        # normally never happen with several completed historical seasons.
        raise ValueError(
            "Historical price-prior coverage is missing FPL positions: "
            f"{sorted(missing_positions)}"
        )

    return prior_table


position_price_prior_table = build_position_price_prior_table(
    historical_model_data
)


def market_prior_for_players(
    frame: pd.DataFrame,
    prior_table: pd.DataFrame,
) -> pd.Series:
    """Interpolate the empirical price prior separately within each position."""
    frame = frame.copy()
    frame["model_position"] = normalise_fpl_model_position(frame)
    result = pd.Series(np.nan, index=frame.index, dtype=float)
    price_col = "price" if "price" in frame.columns else "price_at_collection"
    for position in PLAYER_SCORING_POSITIONS:
        mask = frame["model_position"].eq(position)
        priors = prior_table.loc[
            prior_table["model_position"].eq(position)
        ].sort_values("price_bin")
        if not mask.any() or priors.empty:
            continue
        x = priors["price_bin"].to_numpy(dtype=float)
        y = priors["market_prior_points"].to_numpy(dtype=float)
        prices = pd.to_numeric(frame.loc[mask, price_col], errors="coerce").to_numpy(dtype=float)
        result.loc[mask] = np.interp(prices, x, y, left=y[0], right=y[-1])
    return result


def apply_sparse_player_market_prior(
    frame: pd.DataFrame,
    prior_table: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """
    Blend a weak FPL price/position prior with a smooth evidence fade.

    The old <=1-fixture switch made a player jump from "sparse" to fully
    trusted after only two PL appearances. v2.2.7 instead decays the market
    prior linearly to zero by SPARSE_PLAYER_MARKET_PRIOR_FADE_FIXTURES.

    A usable external EVENT prior suppresses the market fallback to avoid
    double-counting. A role/minutes-only external prior (for example a manual
    unsupported-league fallback) does not suppress the market xP shrinkage.
    """
    result = frame.copy()
    if prior_table is None:
        prior_table = position_price_prior_table

    result["market_fallback_prior_points"] = market_prior_for_players(
        result,
        prior_table,
    )

    career_count = pd.to_numeric(
        result.get(
            "prior_career_fixture_count",
            pd.Series(0.0, index=result.index),
        ),
        errors="coerce",
    ).fillna(0.0).clip(lower=0.0)

    fade_horizon = max(
        float(SPARSE_PLAYER_MARKET_PRIOR_FADE_FIXTURES),
        1e-9,
    )
    evidence_fade = (
        1.0 - career_count.div(fade_horizon)
    ).clip(0.0, 1.0)

    position = normalise_fpl_model_position(result)
    outfield_event_prior = (
        pd.to_numeric(
            result.get(
                "external_pl_equivalent_xg_per90",
                pd.Series(np.nan, index=result.index),
            ),
            errors="coerce",
        ).notna()
        | pd.to_numeric(
            result.get(
                "external_pl_equivalent_xa_per90",
                pd.Series(np.nan, index=result.index),
            ),
            errors="coerce",
        ).notna()
    )
    goalkeeper_event_prior = pd.to_numeric(
        result.get(
            "external_pl_equivalent_saves_per90",
            pd.Series(np.nan, index=result.index),
        ),
        errors="coerce",
    ).notna()
    usable_external_event_prior = (
        (position.eq("GK") & goalkeeper_event_prior)
        | (position.isin(["DEF", "MID", "FWD"]) & outfield_event_prior)
    )

    prior_available = result["market_fallback_prior_points"].notna()
    weight = (
        float(SPARSE_PLAYER_MARKET_PRIOR_WEIGHT)
        * evidence_fade
        * prior_available.astype(float)
        * (~usable_external_event_prior).astype(float)
    ).clip(
        0.0,
        float(SPARSE_PLAYER_MARKET_PRIOR_WEIGHT),
    )

    result["market_fallback_weight"] = weight
    result["market_fallback_evidence_fade"] = evidence_fade

    raw = pd.to_numeric(
        result["expected_points"],
        errors="coerce",
    )
    prior = pd.to_numeric(
        result["market_fallback_prior_points"],
        errors="coerce",
    )
    result["expected_points"] = (
        (1.0 - weight) * raw
        + weight * prior.fillna(raw)
    )
    return result


FORECAST_CACHE_SIGNATURE = stable_configuration_signature({
    "project_version": FORECAST_COMPATIBILITY_VERSION,
    "model_code_version": MODEL_CODE_VERSION,
    "notebook_release": FORECAST_COMPATIBILITY_RELEASE,
    "validation_schema": VALIDATION_SCHEMA_VERSION,
    "runtime_packages": globals().get("required_runtime_objects", {}),
    "player_event_features": PLAYER_EVENT_FEATURE_COLUMNS,
    "player_rate_specs": PLAYER_RATE_MODEL_SPECIFICATIONS,
    "event_rate_caps": PREDICTED_EVENT_RATE_CAPS,
    "minutes_features": MINUTES_FEATURE_COLUMNS,
    "minutes_probability_model": "hierarchical_appear_start60_v2",
    "minutes_position_duration_min_rows": MINUTES_POSITION_DURATION_MIN_ROWS,
    "team_feature_columns": TEAM_FEATURE_COLUMNS,
    "team_goal_model": "PoissonRegressor(alpha=1.0,max_iter=2000)",
    "goal_points": GOAL_POINTS_BY_POSITION,
    "clean_sheet_points": CLEAN_SHEET_POINTS_BY_POSITION,
    # v2.6: these entries force the stored xP-calibration, premium-attacker and
    # v2.4 artifacts to be rejected as stale. They were fitted against the
    # uncompressed-assist xP distribution and would otherwise double-correct.
    "v26_enabled": V26_ENABLED,
    "v26_assist_conversion": V26_APPLY_ASSIST_CONVERSION,
    "v26_goal_conversion": V26_APPLY_GOAL_CONVERSION,
    "v26_assist_bounds": fx.ASSIST_CONVERSION_BOUNDS,
    "v26_goal_bounds": fx.GOAL_CONVERSION_BOUNDS,
    "v26_conversion_shrinkage": fx.CONVERSION_SHRINKAGE_EVENTS,
    "v26_coverage_mask": True,
    "v26_dc_minutes_fix": True,
    "v26_wildcard_threshold_only": V26_WILDCARD_THRESHOLD_ONLY,
    "v26_coverage_mask_applied": V26_APPLY_COVERAGE_MASK,
    "clean_sheet_rule": "player_exposure_minutes_v2",
    "external_priors_enabled": EXTERNAL_PRIORS_ENABLED,
    "external_understat_enabled": EXTERNAL_UNDERSTAT_ENABLED,
    "sparse_prior_weight": SPARSE_PLAYER_MARKET_PRIOR_WEIGHT,
    "sparse_prior_fade_fixtures": SPARSE_PLAYER_MARKET_PRIOR_FADE_FIXTURES,
    "defensive_action_rate_prior_fixtures": DEFENSIVE_ACTION_RATE_PRIOR_FIXTURES,
    "external_manual_priors": EXTERNAL_MANUAL_PRIORS,
    "premium_price_threshold": PREMIUM_ATTACKER_PRICE_THRESHOLD,
    "premium_mode": PREMIUM_ATTACKER_ADJUSTMENT_MODE,
    "xp_calibration_mode": XP_CALIBRATION_MODE,
    "xp_calibration_blend": XP_CALIBRATION_BLEND,
    "set_piece_penalty_xg_additive": LIVE_SET_PIECE_PENALTY_XG_ADDITIVE,
    "captain_highest_xp": PRODUCTION_CAPTAIN_HIGHEST_XP,
    "v24_targeted_mode": V24_TARGETED_FORECAST_MODE,
    "v24_upper_tail_xp_start": V24_UPPER_TAIL_XP_RAMP_START,
    "v24_upper_tail_xp_full": V24_UPPER_TAIL_XP_RAMP_FULL,
    "v24_upper_tail_price_start": V24_UPPER_TAIL_PRICE_RAMP_START,
    "v24_upper_tail_price_full": V24_UPPER_TAIL_PRICE_RAMP_FULL,
    "v24_mid_rank_blend": V24_MID_RANK_BLEND,
    "v24_mid_rank_min_minutes": V24_MID_RANK_MIN_EXPECTED_MINUTES,
    "v24_mid_rank_min_price": V24_MID_RANK_MIN_PRICE,
})


def load_signed_json_artifact(
    path: Path,
    *,
    artifact_type: str,
) -> dict | None:
    if FORCE_REBUILD_VALIDATION_CACHES or not path.exists():
        return None
    try:
        artifact = json.loads(path.read_text(encoding="utf-8"))
    except Exception as exc:
        warnings.warn(f"Could not load {artifact_type} artifact: {exc}")
        return None

    if STRICT_CACHE_SIGNATURES:
        if artifact.get("forecast_signature") != FORECAST_CACHE_SIGNATURE:
            warnings.warn(
                f"Ignoring stale {artifact_type} artifact because its model/config "
                "signature does not match the current notebook. Run Section 11."
            )
            return None
    return artifact


def load_xp_calibration_payload(
    path: Path = XP_CALIBRATION_CACHE_PATH,
) -> dict | None:
    return load_signed_json_artifact(path, artifact_type="xP calibration")


def load_premium_attacker_payload(
    path: Path = PREMIUM_ATTACKER_CACHE_PATH,
) -> dict | None:
    return load_signed_json_artifact(path, artifact_type="premium-attacker")


def _price_band(value: float) -> str:
    if not np.isfinite(value):
        return "unknown"
    if value < 5.0:
        return "lt5"
    if value < 6.5:
        return "5to6_5"
    if value < 8.0:
        return "6_5to8"
    if value < 10.0:
        return "8to10"
    return "ge10"


def apply_xp_calibration_payload(
    frame: pd.DataFrame,
    payload: dict | None,
    *,
    position_col: str = "model_position",
    price_col: str = "price",
    value_col: str = "expected_points",
) -> pd.DataFrame:
    """Apply stored monotone calibration and price-band residual correction."""
    result = frame.copy()
    result["raw_expected_points"] = pd.to_numeric(result[value_col], errors="coerce")
    result["xp_calibration_adjustment"] = 0.0
    result["xp_calibration_applied"] = False
    if not payload or "positions" not in payload:
        return result

    calibrated = result["raw_expected_points"].to_numpy(dtype=float).copy()
    prices = pd.to_numeric(result.get(price_col, np.nan), errors="coerce")

    for position, spec in payload.get("positions", {}).items():
        mask = result[position_col].eq(position).to_numpy(dtype=bool)
        if not mask.any():
            continue
        x = np.asarray(spec.get("x", []), dtype=float)
        y = np.asarray(spec.get("y", []), dtype=float)
        if len(x) < 2 or len(y) != len(x):
            continue
        raw = result.loc[mask, "raw_expected_points"].to_numpy(dtype=float)
        iso = np.interp(raw, x, y, left=y[0], right=y[-1])
        candidate = (
            (1.0 - float(XP_CALIBRATION_BLEND)) * raw
            + float(XP_CALIBRATION_BLEND) * iso
        )

        residuals = spec.get("price_band_residual", {})
        position_prices = prices.loc[mask].to_numpy(dtype=float)
        residual = np.array([
            float(residuals.get(_price_band(v), 0.0))
            for v in position_prices
        ])
        candidate = candidate + residual

        delta = candidate - raw
        max_delta = np.minimum(
            float(XP_CALIBRATION_MAX_ABS_ADJUSTMENT),
            float(XP_CALIBRATION_MAX_REL_ADJUSTMENT) * np.maximum(np.abs(raw), 1.0),
        )
        delta = np.clip(delta, -max_delta, max_delta)
        calibrated[mask] = raw + delta

    result[value_col] = calibrated
    result["xp_calibration_adjustment"] = (
        result[value_col] - result["raw_expected_points"]
    )
    result["xp_calibration_applied"] = result["xp_calibration_adjustment"].abs().gt(1e-12)
    return result



def _validation_mode_allows(mode: str, approved: bool) -> bool:
    mode = str(mode).strip().lower()
    if mode == "off":
        return False
    if mode == "force":
        return True
    if mode != "auto":
        raise ValueError(f"Unknown validation mode: {mode!r}")
    return bool(approved)


def active_xp_calibration_payload(artifact: dict | None) -> dict | None:
    if not artifact:
        return None
    decision = artifact.get("production_decision", {})
    if not _validation_mode_allows(
        XP_CALIBRATION_MODE,
        bool(decision.get("approved", False)),
    ):
        return None
    return artifact.get("live_candidate")


def fit_premium_attacker_adjustment_payload(rows: pd.DataFrame) -> dict | None:
    """Fit shrunk xG/xA multipliers for premium MID/FWD players.

    This is intentionally simple and interpretable. It corrects the event-rate
    layer only when historical out-of-sample evidence shows systematic
    compression. The production gate is decided separately.
    """
    if rows is None or rows.empty:
        return None
    data = rows.copy()
    price_col = "price_at_collection" if "price_at_collection" in data.columns else "price"
    data["_price"] = pd.to_numeric(data[price_col], errors="coerce")
    data["_minutes"] = pd.to_numeric(data.get("target_minutes"), errors="coerce").fillna(0.0)
    data = data.loc[
        data["model_position"].isin(["MID", "FWD"])
        & data["_price"].ge(float(PREMIUM_ATTACKER_PRICE_THRESHOLD))
        & data["_minutes"].gt(0)
    ].copy()
    if len(data) < int(PREMIUM_ATTACKER_MIN_ROWS):
        return None

    payload = {"rows": int(len(data)), "positions": {}}
    for position in ["MID", "FWD"]:
        subset = data.loc[data["model_position"].eq(position)].copy()
        if len(subset) < max(80, int(PREMIUM_ATTACKER_MIN_ROWS // 4)):
            continue
        position_payload = {}
        for event in ["xg", "xa"]:
            pred_col = f"predicted_{event}_per90"
            target_col = f"target_{event}_per90"
            valid = subset[pred_col].notna() & subset[target_col].notna()
            sample = subset.loc[valid].copy()
            if len(sample) < 60:
                continue
            weights = sample["_minutes"].clip(lower=1.0, upper=90.0).to_numpy(dtype=float)
            pred = pd.to_numeric(sample[pred_col], errors="coerce").to_numpy(dtype=float)
            actual = pd.to_numeric(sample[target_col], errors="coerce").to_numpy(dtype=float)
            pred_mean = float(np.average(pred, weights=weights))
            actual_mean = float(np.average(actual, weights=weights))
            if not np.isfinite(pred_mean) or pred_mean <= 1e-6:
                continue
            raw_factor = actual_mean / pred_mean
            total_minutes = float(sample["_minutes"].sum())
            shrink = total_minutes / (total_minutes + float(PREMIUM_ATTACKER_SHRINK_MINUTES))
            factor = 1.0 + shrink * (raw_factor - 1.0)
            lo = 1.0 - float(PREMIUM_ATTACKER_MAX_FACTOR_DEVIATION)
            hi = 1.0 + float(PREMIUM_ATTACKER_MAX_FACTOR_DEVIATION)
            position_payload[event] = {
                "factor": float(np.clip(factor, lo, hi)),
                "raw_factor": float(raw_factor),
                "rows": int(len(sample)),
                "minutes": total_minutes,
                "predicted_mean": pred_mean,
                "actual_mean": actual_mean,
            }
        if position_payload:
            payload["positions"][position] = position_payload
    return payload if payload["positions"] else None


def apply_premium_attacker_adjustment(
    frame: pd.DataFrame,
    payload: dict | None,
    *,
    position_col: str = "model_position",
    price_col: str = "price",
) -> pd.DataFrame:
    """Apply validated premium xG/xA factors and recompute attacking xP only."""
    result = frame.copy()
    result["pre_premium_expected_points"] = pd.to_numeric(
        result["expected_points"], errors="coerce"
    )
    result["premium_attacker_adjustment"] = 0.0
    result["premium_attacker_adjustment_applied"] = False
    if not payload or "positions" not in payload:
        return result

    price = pd.to_numeric(result.get(price_col, np.nan), errors="coerce")
    minutes_fraction = pd.to_numeric(
        result.get("expected_minutes", 0.0), errors="coerce"
    ).fillna(0.0).clip(0.0, 90.0) / 90.0

    old_xp_goals = pd.to_numeric(result["xp_goals"], errors="coerce").fillna(0.0)
    old_xp_assists = pd.to_numeric(result["xp_assists"], errors="coerce").fillna(0.0)

    for position, spec in payload.get("positions", {}).items():
        mask = (
            result[position_col].eq(position)
            & price.ge(float(PREMIUM_ATTACKER_PRICE_THRESHOLD))
        )
        if not mask.any():
            continue
        if "xg" in spec:
            factor = float(spec["xg"].get("factor", 1.0))
            result.loc[mask, "predicted_xg_per90"] = (
                pd.to_numeric(result.loc[mask, "predicted_xg_per90"], errors="coerce")
                .fillna(0.0) * factor
            ).clip(lower=0.0, upper=PREDICTED_EVENT_RATE_CAPS["predicted_xg_per90"])
        if "xa" in spec:
            factor = float(spec["xa"].get("factor", 1.0))
            result.loc[mask, "predicted_xa_per90"] = (
                pd.to_numeric(result.loc[mask, "predicted_xa_per90"], errors="coerce")
                .fillna(0.0) * factor
            ).clip(lower=0.0, upper=PREDICTED_EVENT_RATE_CAPS["predicted_xa_per90"])

    # v2.6: this rebuild must go through the same xA -> Fantasy-assist
    # conversion the component model uses, otherwise an approved premium
    # payload silently reverts F1 for premium attackers.
    _v26_pos = result[position_col]
    result["projected_goals"] = (
        pd.to_numeric(result["predicted_xg_per90"], errors="coerce").fillna(0.0)
        * minutes_fraction
        * _v26_position_factor(_v26_pos, globals().get("V26_GOAL_FACTORS") or {})
    )
    result["projected_assists"] = (
        pd.to_numeric(result["predicted_xa_per90"], errors="coerce").fillna(0.0)
        * minutes_fraction
        * _v26_position_factor(_v26_pos, globals().get("V26_ASSIST_FACTORS") or {})
    )
    goal_points = result[position_col].map(GOAL_POINTS_BY_POSITION).fillna(0.0)
    result["xp_goals"] = result["projected_goals"] * goal_points
    result["xp_assists"] = 3.0 * result["projected_assists"]
    result["expected_points"] = (
        result["pre_premium_expected_points"]
        + (result["xp_goals"] - old_xp_goals)
        + (result["xp_assists"] - old_xp_assists)
    )
    result["premium_attacker_adjustment"] = (
        result["expected_points"] - result["pre_premium_expected_points"]
    )
    result["premium_attacker_adjustment_applied"] = (
        result["premium_attacker_adjustment"].abs().gt(1e-12)
    )
    return result


def active_premium_attacker_payload(artifact: dict | None) -> dict | None:
    if not artifact:
        return None
    decision = artifact.get("production_decision", {})
    if not _validation_mode_allows(
        PREMIUM_ATTACKER_ADJUSTMENT_MODE,
        bool(decision.get("approved", False)),
    ):
        return None
    return artifact.get("live_candidate")


xp_calibration_cache = load_xp_calibration_payload()
premium_attacker_cache = load_premium_attacker_payload()
print("Forecast cache signature:", FORECAST_CACHE_SIGNATURE)
print("xP validation artifact loaded:", xp_calibration_cache is not None)
print("Premium-attacker validation artifact loaded:", premium_attacker_cache is not None)

### v2.4 targeted forecast correction layer

The base component model remains the source of xP. v2.4 adds only OOS-gated
corrections for the two issues identified by v2.3.1: upper-tail compression and
MID decision-candidate ordering. A separate historical error table attaches
minutes/xP uncertainty for confidence reporting; uncertainty does **not**
secretly subtract from expected value.

In [ ]:

# =============================================================================
# v2.4 targeted forecast layer — application + uncertainty helpers
# =============================================================================

V24_MID_RANK_FEATURES = [
    "expected_points",
    "projected_goals",
    "projected_assists",
    "expected_minutes",
    "price",
    "start_probability",
    "sixty_plus_probability",
    "xp_bonus",
    "xp_clean_sheet",
    "xp_defensive_contribution",
]


def load_v24_targeted_forecast_artifact() -> dict | None:
    return load_signed_json_artifact(
        V24_TARGETED_FORECAST_ARTIFACT_PATH,
        artifact_type="v2.4 targeted forecast layer",
    )


def _v24_numeric(frame: pd.DataFrame, column: str, default=np.nan) -> pd.Series:
    if column in frame.columns:
        return pd.to_numeric(frame[column], errors="coerce")
    return pd.Series(default, index=frame.index, dtype=float)


def _v24_price(frame: pd.DataFrame, price_col: str) -> pd.Series:
    if price_col in frame.columns:
        return pd.to_numeric(frame[price_col], errors="coerce")
    fallback = "price_at_collection" if "price_at_collection" in frame.columns else "price"
    return _v24_numeric(frame, fallback)


def _v24_upper_tail_design(
    frame: pd.DataFrame,
    *,
    price_col: str,
    value_col: str,
) -> np.ndarray:
    raw = _v24_numeric(frame, value_col).fillna(0.0)
    price = _v24_price(frame, price_col).fillna(0.0)
    xp_span = max(
        float(V24_UPPER_TAIL_XP_RAMP_FULL - V24_UPPER_TAIL_XP_RAMP_START),
        1e-9,
    )
    price_span = max(
        float(V24_UPPER_TAIL_PRICE_RAMP_FULL - V24_UPPER_TAIL_PRICE_RAMP_START),
        1e-9,
    )
    xp_weight = (
        (raw - float(V24_UPPER_TAIL_XP_RAMP_START)) / xp_span
    ).clip(0.0, 1.0)
    price_weight = (
        (price - float(V24_UPPER_TAIL_PRICE_RAMP_START)) / price_span
    ).clip(0.0, 1.0)
    return np.column_stack([xp_weight.to_numpy(), price_weight.to_numpy()])


def _v24_apply_upper_tail_payload(
    frame: pd.DataFrame,
    payload: dict | None,
    *,
    price_col: str,
    value_col: str,
) -> pd.DataFrame:
    result = frame.copy()
    result["v24_upper_tail_adjustment"] = 0.0
    result["v24_upper_tail_applied"] = False
    if not payload:
        return result

    minutes = _v24_numeric(result, "expected_minutes", 0.0).fillna(0.0)
    eligible = minutes.ge(float(V24_UPPER_TAIL_MIN_EXPECTED_MINUTES))
    X = _v24_upper_tail_design(result, price_col=price_col, value_col=value_col)
    coef = np.asarray(payload.get("coefficients", [0.0, 0.0]), dtype=float)
    if coef.shape != (2,):
        return result
    delta = X @ coef
    delta = np.clip(
        delta,
        -float(V24_UPPER_TAIL_MAX_ABS_ADJUSTMENT),
        float(V24_UPPER_TAIL_MAX_ABS_ADJUSTMENT),
    )
    delta = np.where(eligible.to_numpy(), delta, 0.0)
    base = _v24_numeric(result, value_col).fillna(0.0).to_numpy(dtype=float)
    result[value_col] = base + delta
    result["v24_upper_tail_adjustment"] = delta
    result["v24_upper_tail_applied"] = np.abs(delta) > 1e-12
    return result


def _v24_mid_feature_frame(
    frame: pd.DataFrame,
    *,
    price_col: str,
) -> pd.DataFrame:
    features = pd.DataFrame(index=frame.index)
    for column in V24_MID_RANK_FEATURES:
        if column == "price":
            values = _v24_price(frame, price_col)
        elif column == "expected_points" and "pre_v24_expected_points" in frame.columns:
            # Keep the reranker independent of the upper-tail correction.
            values = _v24_numeric(frame, "pre_v24_expected_points")
        else:
            values = _v24_numeric(frame, column)
        features[column] = values
    return features


def _v24_apply_midfield_payload(
    frame: pd.DataFrame,
    payload: dict | None,
    *,
    position_col: str,
    price_col: str,
    value_col: str,
) -> pd.DataFrame:
    result = frame.copy()
    result["v24_midfield_rank_adjustment"] = 0.0
    result["v24_midfield_rank_applied"] = False
    if not payload:
        return result

    feature_names = list(payload.get("features", []))
    if not feature_names:
        return result
    features = _v24_mid_feature_frame(result, price_col=price_col)
    for name in feature_names:
        if name not in features.columns:
            features[name] = np.nan
    features = features[feature_names]

    means = pd.Series(payload.get("means", {}), dtype=float).reindex(feature_names)
    scales = pd.Series(payload.get("scales", {}), dtype=float).reindex(feature_names)
    fills = pd.Series(payload.get("fills", {}), dtype=float).reindex(feature_names)
    X = features.copy()
    for name in feature_names:
        X[name] = pd.to_numeric(X[name], errors="coerce").fillna(float(fills.get(name, 0.0)))
    X = (X - means) / scales.replace(0.0, 1.0)

    coef = np.asarray(payload.get("coefficients", []), dtype=float)
    if coef.shape != (len(feature_names),):
        return result
    residual = float(payload.get("intercept", 0.0)) + X.to_numpy(dtype=float) @ coef
    residual *= float(V24_MID_RANK_BLEND)
    residual = np.clip(
        residual,
        -float(V24_MID_RANK_MAX_ABS_ADJUSTMENT),
        float(V24_MID_RANK_MAX_ABS_ADJUSTMENT),
    )

    position = result[position_col].astype(str)
    price = _v24_price(result, price_col)
    minutes = _v24_numeric(result, "expected_minutes", 0.0)
    eligible = (
        position.eq("MID")
        & price.ge(float(V24_MID_RANK_MIN_PRICE))
        & minutes.ge(float(V24_MID_RANK_MIN_EXPECTED_MINUTES))
    )
    delta = np.where(eligible.to_numpy(), residual, 0.0)
    base = _v24_numeric(result, value_col).fillna(0.0).to_numpy(dtype=float)
    result[value_col] = base + delta
    result["v24_midfield_rank_adjustment"] = delta
    result["v24_midfield_rank_applied"] = np.abs(delta) > 1e-12
    return result


def active_v24_targeted_bundle(artifact: dict | None) -> dict | None:
    if not artifact:
        return None
    if not _validation_mode_allows(
        V24_TARGETED_FORECAST_MODE,
        bool(artifact.get("production_decision", {}).get("approved", False)),
    ):
        return None
    bundle = artifact.get("live_bundle", {})
    return bundle if bundle else None


def apply_v24_targeted_forecast_layer(
    frame: pd.DataFrame,
    artifact_or_bundle: dict | None,
    *,
    position_col: str = "model_position",
    price_col: str = "price",
    value_col: str = "expected_points",
) -> pd.DataFrame:
    """Idempotently apply the validated v2.4 upper-tail/MID corrections."""
    result = frame.copy()
    if "pre_v24_expected_points" in result.columns:
        result[value_col] = pd.to_numeric(
            result["pre_v24_expected_points"], errors="coerce"
        )
    else:
        result["pre_v24_expected_points"] = pd.to_numeric(
            result[value_col], errors="coerce"
        )

    # artifact -> approved live bundle; direct bundle -> fold/debug use.
    if artifact_or_bundle and "production_decision" in artifact_or_bundle:
        bundle = active_v24_targeted_bundle(artifact_or_bundle)
    else:
        bundle = artifact_or_bundle
    bundle = bundle or {}

    result = _v24_apply_upper_tail_payload(
        result,
        bundle.get("upper_tail"),
        price_col=price_col,
        value_col=value_col,
    )
    result = _v24_apply_midfield_payload(
        result,
        bundle.get("midfield"),
        position_col=position_col,
        price_col=price_col,
        value_col=value_col,
    )
    result["v24_total_adjustment"] = (
        pd.to_numeric(result[value_col], errors="coerce")
        - pd.to_numeric(result["pre_v24_expected_points"], errors="coerce")
    )
    result["v24_targeted_layer_applied"] = result["v24_total_adjustment"].abs().gt(1e-12)
    return result


def _v24_minutes_bucket(values: pd.Series) -> pd.Series:
    return pd.cut(
        pd.to_numeric(values, errors="coerce").fillna(0.0),
        bins=V24_UNCERTAINTY_EXPECTED_MINUTES_BINS,
        labels=V24_UNCERTAINTY_EXPECTED_MINUTES_LABELS,
        right=False,
        include_lowest=True,
    ).astype("string")


def apply_v24_uncertainty_payload(
    frame: pd.DataFrame,
    artifact: dict | None,
    *,
    position_col: str = "model_position",
) -> pd.DataFrame:
    result = frame.copy()
    for column in [
        "minutes_uncertainty_mae",
        "minutes_uncertainty_p80",
        "forecast_uncertainty_mae",
        "forecast_uncertainty_p80",
    ]:
        result[column] = np.nan
    if not artifact:
        return result
    payload = artifact.get("uncertainty")
    if not payload:
        return result

    buckets = _v24_minutes_bucket(_v24_numeric(result, "expected_minutes", 0.0))
    positions = result[position_col].astype(str)
    groups = payload.get("groups", {})
    global_groups = payload.get("global_groups", {})
    fallback = payload.get("fallback", {})

    outputs = {name: [] for name in [
        "minutes_uncertainty_mae",
        "minutes_uncertainty_p80",
        "forecast_uncertainty_mae",
        "forecast_uncertainty_p80",
    ]}
    for pos, bucket in zip(positions, buckets):
        spec = groups.get(f"{pos}|{bucket}") or global_groups.get(str(bucket)) or fallback
        for name in outputs:
            outputs[name].append(float(spec.get(name, np.nan)) if spec else np.nan)
    for name, values in outputs.items():
        result[name] = values
    return result


v24_targeted_forecast_cache = load_v24_targeted_forecast_artifact()
print("v2.4 targeted forecast artifact loaded:", v24_targeted_forecast_cache is not None)


In [ ]:

# =============================================================================
# v2.5 probabilistic minutes / role layer — cache-first helpers
# =============================================================================
#
# Design:
#   * Reuse the already validated P(appear), P(start), P(60+) hierarchy.
#   * Fit only the missing conditional P(75+ | starter and 60+).
#   * Convert that hierarchy into five mutually-exclusive role states.
#   * Integrate FPL scoring across those states instead of evaluating every
#     component at one point-estimate of expected minutes.
#
# This is deliberately a post-layer. It does not mutate the signed v2.4 base
# forecast cache signature, so the expensive OOS forecast folds remain reusable.

V25_POINT_COMPONENT_COLUMNS = [
    "xp_appearance",
    "xp_goals",
    "xp_assists",
    "xp_clean_sheet",
    "xp_saves",
    "xp_goals_conceded",
    "xp_defensive_contribution",
    "xp_bonus",
    "xp_yellow_cards",
    "xp_red_cards",
    "xp_own_goals",
    "xp_penalty_saves",
    "xp_penalty_misses",
]

V25_INTEGRATED_DIAGNOSTIC_COLUMNS = [
    "projected_goals",
    "projected_assists",
    "projected_saves",
    "projected_goals_conceded_on_pitch",
    "projected_defensive_actions",
    "defensive_return_probability",
]


def v25_minutes_layer_signature() -> str:
    """Separate signature for the cheap v2.5 post-layer."""
    season_counts = (
        historical_model_data.groupby("season").size().astype(int).to_dict()
        if isinstance(globals().get("historical_model_data"), pd.DataFrame)
        else {}
    )
    target_minutes_sum = (
        float(pd.to_numeric(
            historical_model_data.get("target_minutes"),
            errors="coerce",
        ).fillna(0.0).sum())
        if isinstance(globals().get("historical_model_data"), pd.DataFrame)
        else np.nan
    )
    return stable_configuration_signature({
        "schema_version": V25_CACHE_SCHEMA_VERSION,
        "base_forecast_signature": FORECAST_CACHE_SIGNATURE,
        "minutes_features": MINUTES_FEATURE_COLUMNS,
        "p75_logistic_c": V25_P75_LOGISTIC_C,
        "p75_min_rows": V25_P75_MIN_TRAIN_ROWS,
        "state_min_rows": V25_STATE_MIN_ROWS,
        "role_states": V25_ROLE_STATES,
        "validation_gates": {
            "min_folds": V25_MIN_EVALUATED_FOLDS,
            "p75_brier_improvement": V25_MIN_P75_BRIER_IMPROVEMENT,
            "minutes_mae_improvement": V25_MIN_MINUTES_MAE_IMPROVEMENT,
            "max_xp_mae_degradation": V25_MAX_XP_MAE_DEGRADATION,
            "max_ndcg_degradation": V25_MAX_NDCG_DEGRADATION,
            "max_top10_degradation": V25_MAX_TOP10_DEGRADATION,
            "max_attacker_spearman_degradation": (
                V25_MAX_ATTACKER_SPEARMAN_DEGRADATION
            ),
        },
        "season_counts": season_counts,
        "target_minutes_sum": round(target_minutes_sum, 6)
        if np.isfinite(target_minutes_sum) else None,
    })


def _v25_role_state_from_targets(frame: pd.DataFrame) -> pd.Series:
    """Map realised FPL minutes/start outcomes to mutually-exclusive states."""
    minutes = pd.to_numeric(frame["target_minutes"], errors="coerce")
    appeared = pd.to_numeric(frame["target_appeared"], errors="coerce")
    started = pd.to_numeric(frame["target_started"], errors="coerce")

    state = pd.Series(pd.NA, index=frame.index, dtype="string")
    state.loc[appeared.eq(0)] = "no_appearance"
    state.loc[appeared.eq(1) & started.eq(0)] = "substitute"
    state.loc[started.eq(1) & minutes.lt(60)] = "start_lt60"
    state.loc[started.eq(1) & minutes.ge(60) & minutes.lt(75)] = "start_60_74"
    state.loc[started.eq(1) & minutes.ge(75)] = "start_75_plus"
    return state


def make_v25_p75_model() -> Pipeline:
    """One additional classifier: P(75+ | starter and 60+)."""
    return Pipeline([
        ("preprocess", make_minutes_preprocessor()),
        (
            "model",
            LogisticRegression(
                C=float(V25_P75_LOGISTIC_C),
                max_iter=2_000,
                solver="lbfgs",
                random_state=RANDOM_STATE,
            ),
        ),
    ])


def fit_v25_probabilistic_minutes_bundle(
    training_data: pd.DataFrame,
) -> dict[str, object]:
    """
    Fit the minimal v2.5 role extension.

    The existing hierarchy already estimates appear/start/60+. We therefore
    fit only the conditional 75+ split and empirical within-state minute
    expectations. This avoids a redundant multiclass model.
    """
    data = training_data.copy()
    positions = resolve_minutes_model_position(data)
    state = _v25_role_state_from_targets(data)

    eligible = (
        pd.to_numeric(data["target_started"], errors="coerce").eq(1)
        & pd.to_numeric(data["target_minutes"], errors="coerce").ge(60)
        & data["target_minutes"].notna()
    )
    train = data.loc[eligible].copy()
    target = (
        pd.to_numeric(train["target_minutes"], errors="coerce")
        .ge(75)
        .astype(int)
    )
    if len(train) < int(V25_P75_MIN_TRAIN_ROWS) or target.nunique() < 2:
        raise ValueError(
            "v2.5 P(75+|60+) model does not have enough two-class training rows."
        )

    model = make_v25_p75_model()
    model.fit(
        prepare_model_feature_frame(
            train,
            MINUTES_FEATURE_COLUMNS,
            context="v25.minutes.fit.p75_given_60",
            categorical_columns=MINUTES_CATEGORICAL_FEATURES,
        ),
        target,
    )

    state_minutes: dict[str, float] = {"no_appearance": 0.0}
    state_priors: dict[str, float] = {}

    realised = pd.DataFrame({
        "position": positions,
        "state": state,
        "minutes": pd.to_numeric(data["target_minutes"], errors="coerce"),
    }).dropna(subset=["state", "minutes"])

    # Global state means are the fallback. Means (rather than medians) preserve
    # expected exposure when the state probabilities are averaged.
    global_means = (
        realised.groupby("state")["minutes"].mean().to_dict()
    )
    hard_fallback = {
        "no_appearance": 0.0,
        "substitute": 20.0,
        "start_lt60": 45.0,
        "start_60_74": 67.0,
        "start_75_plus": 84.0,
    }
    for role_state in V25_ROLE_STATES:
        state_minutes[f"GLOBAL|{role_state}"] = float(
            global_means.get(role_state, hard_fallback[role_state])
        )

    for (position, role_state), group in realised.groupby(
        ["position", "state"],
        dropna=False,
    ):
        if (
            pd.notna(position)
            and str(role_state) in V25_ROLE_STATES
            and len(group) >= int(V25_STATE_MIN_ROWS)
        ):
            state_minutes[f"{position}|{role_state}"] = float(
                group["minutes"].mean()
            )

    # Simple conditional 75+ baseline by position for Brier validation.
    p75_rows = realised.loc[
        realised["state"].isin(["start_60_74", "start_75_plus"])
    ].copy()
    p75_rows["_is_75"] = p75_rows["state"].eq("start_75_plus").astype(float)
    global_p75 = float(p75_rows["_is_75"].mean())
    for position in PLAYER_SCORING_POSITIONS:
        group = p75_rows.loc[p75_rows["position"].eq(position)]
        state_priors[position] = (
            float(group["_is_75"].mean())
            if len(group) >= int(V25_STATE_MIN_ROWS)
            else global_p75
        )
    state_priors["GLOBAL"] = global_p75

    return {
        "p75_model": model,
        "state_minutes": state_minutes,
        "p75_position_prior": state_priors,
        "training_rows": int(len(data)),
        "p75_training_rows": int(len(train)),
    }


def _v25_state_minutes_for_rows(
    frame: pd.DataFrame,
    state: str,
    state_minutes: dict[str, float],
) -> np.ndarray:
    positions = resolve_minutes_model_position(frame).astype("string")
    global_value = float(
        state_minutes.get(
            f"GLOBAL|{state}",
            {
                "no_appearance": 0.0,
                "substitute": 20.0,
                "start_lt60": 45.0,
                "start_60_74": 67.0,
                "start_75_plus": 84.0,
            }[state],
        )
    )
    values = np.full(len(frame), global_value, dtype=float)
    for position in PLAYER_SCORING_POSITIONS:
        mask = positions.eq(position).fillna(False).to_numpy()
        if mask.any():
            values[mask] = float(
                state_minutes.get(f"{position}|{state}", global_value)
            )

    bounds = {
        "no_appearance": (0.0, 0.0),
        "substitute": (1.0, 59.0),
        "start_lt60": (1.0, 59.0),
        "start_60_74": (60.0, 74.0),
        "start_75_plus": (75.0, 90.0),
    }[state]
    return np.clip(values, bounds[0], bounds[1])


def predict_v25_role_distribution(
    model_bundle: dict[str, object],
    frame: pd.DataFrame,
) -> pd.DataFrame:
    """
    Produce five coherent role-state probabilities.

    P(appear), P(start) and P(60+) come directly from the existing production
    hierarchy (including live availability/workload overlays when present).
    Only the 60+ split into 60–74 and 75+ comes from the new classifier.
    """
    features = prepare_model_feature_frame(
        frame,
        MINUTES_FEATURE_COLUMNS,
        context="v25.minutes.predict.p75_given_60",
        categorical_columns=MINUTES_CATEGORICAL_FEATURES,
    )
    p75_given_60 = model_bundle["p75_model"].predict_proba(features)[:, 1]
    p75_given_60 = np.clip(p75_given_60, 0.0, 1.0)

    p_appear = pd.to_numeric(
        frame["appearance_probability"], errors="coerce"
    ).fillna(0.0).clip(0.0, 1.0).to_numpy(dtype=float)
    p_start = pd.to_numeric(
        frame["start_probability"], errors="coerce"
    ).fillna(0.0).clip(0.0, 1.0).to_numpy(dtype=float)
    p_start = np.minimum(p_start, p_appear)
    p60 = pd.to_numeric(
        frame["sixty_plus_probability"], errors="coerce"
    ).fillna(0.0).clip(0.0, 1.0).to_numpy(dtype=float)
    p60 = np.minimum(p60, p_start)

    distribution = pd.DataFrame(
        {
            "v25_p_no_appearance": 1.0 - p_appear,
            "v25_p_substitute": np.maximum(p_appear - p_start, 0.0),
            "v25_p_start_lt60": np.maximum(p_start - p60, 0.0),
            "v25_p_start_60_74": p60 * (1.0 - p75_given_60),
            "v25_p_start_75_plus": p60 * p75_given_60,
            "v25_p75_given_60": p75_given_60,
        },
        index=frame.index,
    )

    state_cols = [
        "v25_p_no_appearance",
        "v25_p_substitute",
        "v25_p_start_lt60",
        "v25_p_start_60_74",
        "v25_p_start_75_plus",
    ]
    total = distribution[state_cols].sum(axis=1)
    if not np.allclose(total, 1.0, atol=1e-8):
        raise AssertionError("v2.5 role-state probabilities do not sum to one.")

    expected_minutes = np.zeros(len(frame), dtype=float)
    for state in V25_ROLE_STATES:
        p = distribution[f"v25_p_{state}"].to_numpy(dtype=float)
        mins = _v25_state_minutes_for_rows(
            frame,
            state,
            model_bundle["state_minutes"],
        )
        expected_minutes += p * mins
        distribution[f"v25_minutes_{state}"] = mins

    distribution["v25_probabilistic_expected_minutes"] = np.clip(
        expected_minutes, 0.0, 90.0
    )

    probability_matrix = np.clip(
        distribution[state_cols].to_numpy(dtype=float),
        1e-12,
        1.0,
    )
    entropy = -np.sum(
        probability_matrix * np.log(probability_matrix),
        axis=1,
    ) / np.log(len(state_cols))
    distribution["v25_role_entropy"] = np.clip(entropy, 0.0, 1.0)
    distribution["v25_most_likely_role_probability"] = probability_matrix.max(axis=1)
    distribution["v25_most_likely_role_state"] = np.array(V25_ROLE_STATES, dtype=object)[
        np.argmax(probability_matrix, axis=1)
    ]

    return distribution


def _v25_conditional_state_frame(
    frame: pd.DataFrame,
    state: str,
    state_minutes: np.ndarray,
) -> pd.DataFrame:
    conditional = frame.copy()
    n = len(conditional)

    if state == "no_appearance":
        appear, start, sixty = 0.0, 0.0, 0.0
    elif state == "substitute":
        appear, start, sixty = 1.0, 0.0, 0.0
    elif state == "start_lt60":
        appear, start, sixty = 1.0, 1.0, 0.0
    else:
        appear, start, sixty = 1.0, 1.0, 1.0

    conditional["appearance_probability"] = appear
    conditional["start_probability"] = start
    conditional["substitute_appearance_probability"] = max(appear - start, 0.0)
    conditional["no_appearance_probability"] = 1.0 - appear
    conditional["sixty_plus_probability"] = sixty
    conditional["expected_minutes"] = state_minutes
    conditional["minutes_if_start"] = state_minutes
    conditional["minutes_if_sub"] = state_minutes
    return conditional


def integrate_v25_probabilistic_minutes_xp(
    frame: pd.DataFrame,
    model_bundle: dict[str, object],
    defensive_action_priors: pd.Series,
    rare_event_rates: dict[str, pd.Series],
    defensive_return_priors: pd.Series | None,
    *,
    scoring_season: str | None = None,
) -> pd.DataFrame:
    """Integrate the existing component model over mutually-exclusive roles."""
    result = frame.copy()
    result["pre_v25_expected_points"] = pd.to_numeric(
        result["expected_points"], errors="coerce"
    )
    result["pre_v25_expected_minutes"] = pd.to_numeric(
        result["expected_minutes"], errors="coerce"
    )

    role = predict_v25_role_distribution(model_bundle, result)
    for column in role.columns:
        result[column] = role[column]

    integrated = {
        column: np.zeros(len(result), dtype=float)
        for column in (
            V25_POINT_COMPONENT_COLUMNS
            + V25_INTEGRATED_DIAGNOSTIC_COLUMNS
            + ["expected_points"]
        )
    }

    for state in V25_ROLE_STATES:
        probability = role[f"v25_p_{state}"].to_numpy(dtype=float)
        state_minutes = role[f"v25_minutes_{state}"].to_numpy(dtype=float)
        conditional = _v25_conditional_state_frame(
            result,
            state,
            state_minutes,
        )
        components = calculate_component_expected_points(
            conditional,
            defensive_action_priors,
            rare_event_rates,
            defensive_return_priors,
        )

        # Defensive-contribution points did not exist before 2025/26.
        if (
            scoring_season is not None
            and season_start_year(str(scoring_season)) < 2025
        ):
            components["expected_points"] = (
                components["expected_points"]
                - components["xp_defensive_contribution"]
            )
            components["xp_defensive_contribution"] = 0.0

        for column in integrated:
            if column in components.columns:
                integrated[column] += (
                    probability
                    * pd.to_numeric(
                        components[column], errors="coerce"
                    ).fillna(0.0).to_numpy(dtype=float)
                )

    result["expected_minutes"] = role[
        "v25_probabilistic_expected_minutes"
    ].to_numpy(dtype=float)
    for column, values in integrated.items():
        result[column] = values

    result["v25_minutes_adjustment"] = (
        result["expected_minutes"] - result["pre_v25_expected_minutes"]
    )
    result["v25_probabilistic_xp_adjustment"] = (
        result["expected_points"] - result["pre_v25_expected_points"]
    )

    # v2.6: this function has just recomputed `expected_points` from the role
    # states. Any stored v2.4/premium idempotence baseline is now stale, and a
    # later apply_v24_targeted_forecast_layer() would reset `expected_points`
    # to its pre-v2.5 value, silently discarding this integration.
    if globals().get("V26_ENABLED"):
        result = fx.clear_stale_layer_baselines(result)
    return result


def _v25_prepare_fold_for_integration(
    frame: pd.DataFrame,
    bundle: dict[str, object],
) -> pd.DataFrame:
    """
    Apply any validated premium EVENT-RATE correction before scenario scoring.

    This preserves the probabilistic minutes integration for attacking returns:
    premium xG/xA rates are corrected first, then each role state is scored.
    """
    season = str(bundle["holdout_season"])
    result = frame.copy()
    premium_payload = None
    if isinstance(premium_attacker_cache, dict):
        premium_payload = premium_attacker_cache.get(
            "fold_strategy_payloads", {}
        ).get(season)
    return apply_premium_attacker_adjustment(
        result,
        premium_payload,
        position_col="model_position",
        price_col="price_at_collection",
    )


def _v25_apply_existing_post_layers_to_fold(
    frame: pd.DataFrame,
    bundle: dict[str, object],
) -> pd.DataFrame:
    """Apply scalar post-xP layers after scenario integration."""
    season = str(bundle["holdout_season"])
    result = frame.copy()
    result = apply_sparse_player_market_prior(
        result,
        prior_table=build_position_price_prior_table(
            bundle["development_points_data"]
        ),
    )
    calibration_payload = None
    if isinstance(xp_calibration_cache, dict):
        calibration_payload = xp_calibration_cache.get(
            "fold_strategy_payloads", {}
        ).get(season)
    result = apply_xp_calibration_payload(
        result,
        calibration_payload,
        position_col="model_position",
        price_col="price_at_collection",
        value_col="expected_points",
    )
    # The v2.4 upper-tail/MID layer was rejected in OOS validation and is not
    # stacked on top of v2.5. Keeping it out also avoids resetting the newly
    # integrated xP through its idempotence baseline column.
    return result


def _v25_fold_metrics(
    base: pd.DataFrame,
    candidate: pd.DataFrame,
    model_bundle: dict[str, object],
) -> dict[str, float]:
    """Cheap forecast/decision metrics; no transfer or season simulation."""
    valid_minutes = (
        pd.to_numeric(base["target_minutes"], errors="coerce").notna()
    )
    actual_minutes = pd.to_numeric(
        base.loc[valid_minutes, "target_minutes"], errors="coerce"
    )
    base_minutes = pd.to_numeric(
        base.loc[valid_minutes, "expected_minutes"], errors="coerce"
    )
    candidate_minutes = pd.to_numeric(
        candidate.loc[valid_minutes, "expected_minutes"], errors="coerce"
    )

    valid_points = pd.to_numeric(
        base["target_fpl_points"], errors="coerce"
    ).notna()
    actual_points = pd.to_numeric(
        base.loc[valid_points, "target_fpl_points"], errors="coerce"
    )
    base_xp = pd.to_numeric(
        base.loc[valid_points, "expected_points"], errors="coerce"
    )
    candidate_xp = pd.to_numeric(
        candidate.loc[valid_points, "expected_points"], errors="coerce"
    )

    # Conditional 75+ probability quality among actual 60+ starters.
    role = predict_v25_role_distribution(model_bundle, base)
    p75_mask = (
        pd.to_numeric(base["target_started"], errors="coerce").eq(1)
        & pd.to_numeric(base["target_minutes"], errors="coerce").ge(60)
    )
    p75_target = (
        pd.to_numeric(base.loc[p75_mask, "target_minutes"], errors="coerce")
        .ge(75).astype(int)
    )
    p75_prediction = role.loc[p75_mask, "v25_p75_given_60"]
    positions = resolve_minutes_model_position(base.loc[p75_mask])
    priors = model_bundle["p75_position_prior"]
    p75_baseline = positions.map(priors).fillna(
        float(priors.get("GLOBAL", 0.5))
    ).astype(float)

    model_brier = (
        brier_score_loss(p75_target, p75_prediction)
        if len(p75_target) else np.nan
    )
    baseline_brier = (
        brier_score_loss(p75_target, p75_baseline)
        if len(p75_target) else np.nan
    )

    work = candidate.copy()
    work["v25_base_eval"] = pd.to_numeric(
        base["expected_points"], errors="coerce"
    ).to_numpy(dtype=float)
    work["v25_candidate_eval"] = pd.to_numeric(
        candidate["expected_points"], errors="coerce"
    ).to_numpy(dtype=float)
    pgw = _v24_player_gw(
        work,
        ["v25_base_eval", "v25_candidate_eval"],
    )
    base_rank = _v24_rank_summary(pgw, "v25_base_eval", k=10)
    candidate_rank = _v24_rank_summary(
        pgw, "v25_candidate_eval", k=10
    )

    decision_pool = pgw.loc[
        pgw["position"].isin(["MID", "FWD"])
        & pd.to_numeric(
            pgw["expected_minutes"], errors="coerce"
        ).ge(45.0)
        & pd.to_numeric(pgw["price"], errors="coerce").ge(5.5)
    ].copy()
    base_attack = _v24_rank_summary(
        decision_pool, "v25_base_eval", k=5
    )
    candidate_attack = _v24_rank_summary(
        decision_pool, "v25_candidate_eval", k=5
    )

    return {
        "rows": int(len(base)),
        "base_minutes_mae": float(
            mean_absolute_error(actual_minutes, base_minutes)
        ),
        "candidate_minutes_mae": float(
            mean_absolute_error(actual_minutes, candidate_minutes)
        ),
        "base_xp_mae": float(mean_absolute_error(actual_points, base_xp)),
        "candidate_xp_mae": float(
            mean_absolute_error(actual_points, candidate_xp)
        ),
        "p75_model_brier": float(model_brier),
        "p75_baseline_brier": float(baseline_brier),
        "base_ndcg10": base_rank["ndcg"],
        "candidate_ndcg10": candidate_rank["ndcg"],
        "base_top10_actual": base_rank["top_actual"],
        "candidate_top10_actual": candidate_rank["top_actual"],
        "base_attacker_spearman": base_attack["spearman"],
        "candidate_attacker_spearman": candidate_attack["spearman"],
    }


def _v25_decision_from_scorecard(scorecard: pd.DataFrame) -> dict[str, object]:
    if scorecard is None or scorecard.empty:
        return {
            "approved": False,
            "reason": "no chronological OOS scorecard",
            "evaluated_folds": 0,
        }
    usable = scorecard.loc[scorecard["candidate_evaluated"].fillna(False)].copy()
    if usable.empty:
        return {
            "approved": False,
            "reason": "no candidate-evaluated folds",
            "evaluated_folds": 0,
        }

    def mean_delta(candidate_col: str, base_col: str) -> float:
        return float(
            pd.to_numeric(usable[candidate_col], errors="coerce").mean()
            - pd.to_numeric(usable[base_col], errors="coerce").mean()
        )

    p75_brier_improvement = -mean_delta(
        "p75_model_brier", "p75_baseline_brier"
    )
    minutes_mae_improvement = -mean_delta(
        "candidate_minutes_mae", "base_minutes_mae"
    )
    xp_mae_change = mean_delta(
        "candidate_xp_mae", "base_xp_mae"
    )
    ndcg_change = mean_delta(
        "candidate_ndcg10", "base_ndcg10"
    )
    top10_change = mean_delta(
        "candidate_top10_actual", "base_top10_actual"
    )
    attacker_spearman_change = mean_delta(
        "candidate_attacker_spearman", "base_attacker_spearman"
    )

    approved = bool(
        len(usable) >= int(V25_MIN_EVALUATED_FOLDS)
        and p75_brier_improvement >= float(V25_MIN_P75_BRIER_IMPROVEMENT)
        and minutes_mae_improvement >= float(V25_MIN_MINUTES_MAE_IMPROVEMENT)
        and xp_mae_change <= float(V25_MAX_XP_MAE_DEGRADATION)
        and ndcg_change >= -float(V25_MAX_NDCG_DEGRADATION)
        and top10_change >= -float(V25_MAX_TOP10_DEGRADATION)
        and attacker_spearman_change
            >= -float(V25_MAX_ATTACKER_SPEARMAN_DEGRADATION)
    )

    return {
        "approved": approved,
        "evaluated_folds": int(len(usable)),
        "p75_brier_improvement": p75_brier_improvement,
        "minutes_mae_improvement": minutes_mae_improvement,
        "xp_mae_change": xp_mae_change,
        "ndcg10_change": ndcg_change,
        "top10_actual_change": top10_change,
        "attacker_spearman_change": attacker_spearman_change,
        "reason": (
            "all probabilistic-minutes gates passed"
            if approved
            else "one or more probabilistic-minutes gates failed"
        ),
    }


def load_v25_probabilistic_minutes_artifact() -> dict | None:
    if V25_FORCE_REBUILD or not V25_PROBABILISTIC_MINUTES_ARTIFACT_PATH.exists():
        return None
    try:
        payload = json.loads(
            V25_PROBABILISTIC_MINUTES_ARTIFACT_PATH.read_text(
                encoding="utf-8"
            )
        )
    except Exception:
        return None
    if payload.get("layer_signature") != v25_minutes_layer_signature():
        return None
    return payload


def load_v25_live_model_bundle() -> dict | None:
    if V25_FORCE_REBUILD or not V25_PROBABILISTIC_MINUTES_MODEL_CACHE_PATH.exists():
        return None
    try:
        with gzip.open(
            V25_PROBABILISTIC_MINUTES_MODEL_CACHE_PATH, "rb"
        ) as handle:
            payload = pickle.load(handle)
    except Exception:
        return None
    if (
        not isinstance(payload, dict)
        or payload.get("layer_signature") != v25_minutes_layer_signature()
        or not isinstance(payload.get("model_bundle"), dict)
    ):
        return None
    return payload["model_bundle"]


def save_v25_live_model_bundle(model_bundle: dict[str, object]) -> None:
    V25_PROBABILISTIC_MINUTES_MODEL_CACHE_PATH.parent.mkdir(
        parents=True, exist_ok=True
    )
    with gzip.open(
        V25_PROBABILISTIC_MINUTES_MODEL_CACHE_PATH, "wb"
    ) as handle:
        pickle.dump(
            {
                "layer_signature": v25_minutes_layer_signature(),
                "generated_at_utc": datetime.now(timezone.utc).isoformat(),
                "model_bundle": model_bundle,
            },
            handle,
            protocol=pickle.HIGHEST_PROTOCOL,
        )


def active_v25_probabilistic_minutes(artifact: dict | None) -> bool:
    if not artifact:
        return False
    return _validation_mode_allows(
        V25_PROBABILISTIC_MINUTES_MODE,
        bool(artifact.get("production_decision", {}).get("approved", False)),
    )


def attach_or_apply_v25_live_minutes(
    frame: pd.DataFrame,
    artifact: dict | None,
    model_bundle: dict[str, object],
) -> pd.DataFrame:
    """
    Attach role probabilities always; modify production xP only if approved.

    This makes a rejected experiment still useful for diagnostics without
    silently changing decisions.
    """
    if active_v25_probabilistic_minutes(artifact):
        # v2.6: Section 16 leaves the last historical fold's conversion factors
        # in the globals. Restore the live ones before re-scoring live xP.
        if globals().get("V26_ENABLED"):
            v26_set_conversion(
                globals()["final_historical_points_data"],
                label="live",
            )
        result = integrate_v25_probabilistic_minutes_xp(
            frame,
            model_bundle,
            final_defensive_action_priors,
            final_rare_event_rates,
            final_defensive_return_priors,
            scoring_season=None,
        )
        # Scenario integration recomputes the transparent component total.
        # Reapply only the already-approved scalar post-layers. Premium xG/xA
        # is already embedded in the event rates carried by `frame`.
        result = apply_sparse_player_market_prior(result)
        result = apply_xp_calibration_payload(
            result,
            globals().get("live_xp_payload"),
            position_col="model_position",
            price_col="price",
            value_col="expected_points",
        )
        result["v25_probabilistic_xp_adjustment"] = (
            pd.to_numeric(result["expected_points"], errors="coerce")
            - pd.to_numeric(result["pre_v25_expected_points"], errors="coerce")
        )
        result["v25_probabilistic_minutes_applied"] = True
    else:
        result = frame.copy()
        result["pre_v25_expected_points"] = pd.to_numeric(
            result["expected_points"], errors="coerce"
        )
        result["pre_v25_expected_minutes"] = pd.to_numeric(
            result["expected_minutes"], errors="coerce"
        )
        role = predict_v25_role_distribution(model_bundle, result)
        for column in role.columns:
            result[column] = role[column]
        result["v25_minutes_adjustment"] = (
            role["v25_probabilistic_expected_minutes"]
            - result["pre_v25_expected_minutes"]
        )
        result["v25_probabilistic_xp_adjustment"] = 0.0
        result["v25_probabilistic_minutes_applied"] = False
    return result


def _run_v25_probability_self_tests() -> None:
    # Pure algebraic guard; no model fitting.
    synthetic = pd.DataFrame(
        {
            "appearance_probability": [0.9],
            "start_probability": [0.7],
            "sixty_plus_probability": [0.5],
        }
    )
    q75 = 0.6
    probs = np.array([
        1 - 0.9,
        0.9 - 0.7,
        0.7 - 0.5,
        0.5 * (1 - q75),
        0.5 * q75,
    ])
    assert np.isclose(probs.sum(), 1.0)
    assert (probs >= 0).all()


_run_v25_probability_self_tests()


In [ ]:

# v2.4 lightweight regression guards (no fitting / simulation)
_v24_test = pd.DataFrame({
    "model_position": ["MID", "DEF"],
    "price": [10.0, 4.5],
    "expected_points": [5.0, 2.0],
    "expected_minutes": [80.0, 80.0],
    "projected_goals": [0.4, 0.05],
    "projected_assists": [0.3, 0.03],
    "start_probability": [0.95, 0.95],
    "sixty_plus_probability": [0.85, 0.85],
    "xp_bonus": [0.4, 0.1],
    "xp_clean_sheet": [0.7, 1.5],
    "xp_defensive_contribution": [0.2, 0.5],
})
_v24_upper_test = {"coefficients": [0.5, 0.2]}
_v24_a = apply_v24_targeted_forecast_layer(
    _v24_test, {"upper_tail": _v24_upper_test},
    position_col="model_position", price_col="price",
)
assert _v24_a.loc[0, "expected_points"] > 5.0
assert abs(_v24_a.loc[1, "expected_points"] - 2.0) < 1e-12
_v24_b = apply_v24_targeted_forecast_layer(
    _v24_a, {"upper_tail": _v24_upper_test},
    position_col="model_position", price_col="price",
)
assert np.allclose(_v24_a["expected_points"], _v24_b["expected_points"]), "v2.4 layer is not idempotent"


In [ ]:

# =============================================================================
# CURRENT-SEASON FBREF + EXPLICIT SET-PIECE RESPONSIBILITY
# =============================================================================

def _tidy_current_fbref_player_table(
    standard_raw: pd.DataFrame,
    shooting_raw: pd.DataFrame | None = None,
    passing_raw: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """
    Build a current-season FBref context using multiple stat categories.

    Standard supplies identity/minutes and often xG/xAG.
    Shooting is the preferred fallback for npxG, shots, SoT and penalties.
    Passing is a fallback for xAG when the standard layout omits it.
    """
    standard = flatten_soccerdata_frame(standard_raw)

    player_col = _find_external_column(
        standard,
        exact=("player",),
        required_tokens=("player",),
    )
    team_col = _find_external_column(
        standard,
        exact=("team", "squad"),
    )
    if player_col is None or team_col is None:
        raise ValueError(
            "Current FBref standard table lacks player/team identity."
        )

    def semantic_numeric(
        frame: pd.DataFrame,
        *,
        exact=(),
        required=(),
        forbidden=(),
    ) -> pd.Series:
        column = _find_external_column(
            frame,
            exact=tuple(exact),
            required_tokens=tuple(required),
            forbidden_tokens=tuple(forbidden),
        )
        return _numeric_external_column(frame, column)

    def xag_numeric(frame: pd.DataFrame) -> pd.Series:
        column = _find_external_column(
            frame,
            exact=(
                "expected_xag",
                "standard_expected_xag",
                "xag",
            ),
            required_tokens=("xag",),
            forbidden_tokens=("per_90", "xag_plus"),
        )
        if column is None:
            column = _find_external_column(
                frame,
                exact=(
                    "expected_xa",
                    "standard_expected_xa",
                    "xa",
                ),
                required_tokens=("xa",),
                forbidden_tokens=("per_90", "xag_plus"),
            )
        return _numeric_external_column(frame, column)

    def xag_per90_numeric(
        frame: pd.DataFrame,
    ) -> pd.Series:
        column = _find_external_column(
            frame,
            exact=(
                "per_90_minutes_xag",
                "per_90_xag",
                "expected_xag_per_90",
            ),
            required_tokens=("xag", "90"),
            forbidden_tokens=("plus",),
        )
        if column is None:
            column = _find_external_column(
                frame,
                exact=(
                    "per_90_minutes_xa",
                    "per_90_xa",
                    "expected_xa_per_90",
                ),
                required_tokens=("xa", "90"),
                forbidden_tokens=("plus",),
            )
        return _numeric_external_column(frame, column)

    minutes = semantic_numeric(
        standard,
        exact=("playing_time_min", "min", "minutes"),
        required=("min",),
        forbidden=("per_90", "pct"),
    )
    standard_xg = semantic_numeric(
        standard,
        exact=("expected_xg", "standard_expected_xg"),
        required=("xg",),
        forbidden=("npxg", "per_90", "plus", "xg_sh"),
    )
    standard_npxg = semantic_numeric(
        standard,
        exact=("expected_npxg", "standard_expected_npxg"),
        required=("npxg",),
        forbidden=("per_90", "plus"),
    )
    standard_xag = xag_numeric(standard)

    standard_xg_per90 = semantic_numeric(
        standard,
        exact=(
            "per_90_minutes_xg",
            "per_90_xg",
            "expected_xg_per_90",
        ),
        required=("xg", "90"),
        forbidden=("npxg",),
    )
    standard_npxg_per90 = semantic_numeric(
        standard,
        exact=(
            "per_90_minutes_npxg",
            "per_90_npxg",
            "expected_npxg_per_90",
        ),
        required=("npxg", "90"),
        forbidden=("plus",),
    )
    standard_xag_per90 = xag_per90_numeric(standard)

    standard_pk = semantic_numeric(
        standard,
        exact=("performance_pk", "pk"),
        required=("pk",),
        forbidden=("pkatt", "pct", "per_90"),
    )
    standard_pkatt = semantic_numeric(
        standard,
        exact=("performance_pkatt", "pkatt"),
        required=("pkatt",),
        forbidden=("per_90",),
    )

    base = pd.DataFrame({
        "fbref_player": standard[player_col].astype(str),
        "fbref_team": standard[team_col].astype(str),
        "fbref_minutes": minutes,
        "_standard_xg": standard_xg,
        "_standard_npxg": standard_npxg,
        "_standard_xag": standard_xag,
        "_standard_xg_per90": standard_xg_per90,
        "_standard_npxg_per90": standard_npxg_per90,
        "_standard_xag_per90": standard_xag_per90,
        "_standard_pk": standard_pk,
        "_standard_pkatt": standard_pkatt,
    })
    base["_name_key"] = base["fbref_player"].map(
        normalise_player_name
    )
    base["_team_key"] = base["fbref_team"].map(
        _normalise_external_team_name
    )

    def tidy_secondary(
        raw: pd.DataFrame | None,
        source_name: str,
    ) -> pd.DataFrame:
        if raw is None or raw.empty:
            return pd.DataFrame(
                columns=["_name_key", "_team_key"]
            )

        frame = flatten_soccerdata_frame(raw)
        pcol = _find_external_column(
            frame,
            exact=("player",),
            required_tokens=("player",),
        )
        tcol = _find_external_column(
            frame,
            exact=("team", "squad"),
        )
        if pcol is None or tcol is None:
            return pd.DataFrame(
                columns=["_name_key", "_team_key"]
            )

        out = pd.DataFrame({
            "_name_key": frame[pcol].map(
                normalise_player_name
            ),
            "_team_key": frame[tcol].map(
                _normalise_external_team_name
            ),
        })

        if source_name == "shooting":
            out["_shoot_xg"] = semantic_numeric(
                frame,
                exact=("expected_xg", "standard_xg"),
                required=("xg",),
                forbidden=("npxg", "per_90", "plus", "xg_sh"),
            )
            out["_shoot_npxg"] = semantic_numeric(
                frame,
                exact=("expected_npxg", "standard_npxg", "npxg"),
                required=("npxg",),
                forbidden=("per_90", "plus"),
            )
            out["_shoot_xg_per90"] = semantic_numeric(
                frame,
                exact=(
                    "per_90_minutes_xg",
                    "per_90_xg",
                    "expected_xg_per_90",
                ),
                required=("xg", "90"),
                forbidden=("npxg",),
            )
            out["_shoot_npxg_per90"] = semantic_numeric(
                frame,
                exact=(
                    "per_90_minutes_npxg",
                    "per_90_npxg",
                    "expected_npxg_per_90",
                ),
                required=("npxg", "90"),
                forbidden=("plus",),
            )
            out["fbref_shots"] = semantic_numeric(
                frame,
                exact=("standard_sh", "sh", "shots"),
                required=("sh",),
                forbidden=("sot", "per_90", "dist"),
            )
            out["fbref_shots_on_target"] = semantic_numeric(
                frame,
                exact=("standard_sot", "sot"),
                required=("sot",),
                forbidden=("pct", "per_90"),
            )
            out["_shoot_pk"] = semantic_numeric(
                frame,
                exact=("standard_pk", "performance_pk", "pk"),
                required=("pk",),
                forbidden=("pkatt", "pct", "per_90"),
            )
            out["_shoot_pkatt"] = semantic_numeric(
                frame,
                exact=(
                    "standard_pkatt",
                    "performance_pkatt",
                    "pkatt",
                ),
                required=("pkatt",),
                forbidden=("per_90",),
            )

        elif source_name == "passing":
            out["_passing_xag"] = xag_numeric(frame)
            out["_passing_xag_per90"] = (
                xag_per90_numeric(frame)
            )

        return (
            out
            .drop_duplicates(
                ["_name_key", "_team_key"],
                keep="last",
            )
        )

    shooting = tidy_secondary(
        shooting_raw,
        "shooting",
    )
    passing = tidy_secondary(
        passing_raw,
        "passing",
    )

    frame = (
        base
        .merge(
            shooting,
            on=["_name_key", "_team_key"],
            how="left",
            validate="many_to_one",
        )
        .merge(
            passing,
            on=["_name_key", "_team_key"],
            how="left",
            validate="many_to_one",
        )
    )

    frame["fbref_xg"] = frame[
        "_standard_xg"
    ].combine_first(
        frame.get(
            "_shoot_xg",
            pd.Series(np.nan, index=frame.index),
        )
    )
    frame["fbref_npxg"] = frame[
        "_standard_npxg"
    ].combine_first(
        frame.get(
            "_shoot_npxg",
            pd.Series(np.nan, index=frame.index),
        )
    )
    frame["fbref_xag"] = frame[
        "_standard_xag"
    ].combine_first(
        frame.get(
            "_passing_xag",
            pd.Series(np.nan, index=frame.index),
        )
    )
    frame["fbref_pk"] = frame[
        "_standard_pk"
    ].combine_first(
        frame.get(
            "_shoot_pk",
            pd.Series(np.nan, index=frame.index),
        )
    )
    frame["fbref_pkatt"] = frame[
        "_standard_pkatt"
    ].combine_first(
        frame.get(
            "_shoot_pkatt",
            pd.Series(np.nan, index=frame.index),
        )
    )

    nineties = (
        pd.to_numeric(
            frame["fbref_minutes"],
            errors="coerce",
        )
        .div(90.0)
        .replace(0.0, np.nan)
    )

    frame["fbref_xg_per90"] = (
        frame["_standard_xg_per90"]
        .combine_first(
            frame.get(
                "_shoot_xg_per90",
                pd.Series(np.nan, index=frame.index),
            )
        )
        .combine_first(
            pd.to_numeric(
                frame["fbref_xg"],
                errors="coerce",
            ).div(nineties)
        )
    )
    frame["fbref_npxg_per90"] = (
        frame["_standard_npxg_per90"]
        .combine_first(
            frame.get(
                "_shoot_npxg_per90",
                pd.Series(np.nan, index=frame.index),
            )
        )
        .combine_first(
            pd.to_numeric(
                frame["fbref_npxg"],
                errors="coerce",
            ).div(nineties)
        )
    )
    frame["fbref_xag_per90"] = (
        frame["_standard_xag_per90"]
        .combine_first(
            frame.get(
                "_passing_xag_per90",
                pd.Series(np.nan, index=frame.index),
            )
        )
        .combine_first(
            pd.to_numeric(
                frame["fbref_xag"],
                errors="coerce",
            ).div(nineties)
        )
    )

    for total, rate in [
        ("fbref_shots", "fbref_shots_per90"),
        ("fbref_shots_on_target", "fbref_sot_per90"),
        ("fbref_pkatt", "fbref_pkatt_per90"),
    ]:
        if total in frame.columns:
            frame[rate] = pd.to_numeric(
                frame[total],
                errors="coerce",
            ).div(nineties)
        else:
            frame[rate] = np.nan

    frame["fbref_name_key"] = frame["_name_key"]
    frame["fbref_team_key"] = frame["_team_key"]

    keep = [
        "fbref_player",
        "fbref_team",
        "fbref_minutes",
        "fbref_xg",
        "fbref_npxg",
        "fbref_xag",
        "fbref_pk",
        "fbref_pkatt",
        "fbref_xg_per90",
        "fbref_npxg_per90",
        "fbref_xag_per90",
        "fbref_pkatt_per90",
        "fbref_shots",
        "fbref_shots_on_target",
        "fbref_shots_per90",
        "fbref_sot_per90",
        "fbref_name_key",
        "fbref_team_key",
    ]
    return frame[keep]


def _match_current_fbref_to_fpl(
    fbref: pd.DataFrame,
    current_players: pd.DataFrame,
    team_table: pd.DataFrame,
) -> pd.DataFrame:
    if fbref.empty:
        return pd.DataFrame()

    aliases = _fpl_team_alias_lookup(team_table)
    fbref = fbref.copy()
    fbref["team_id"] = fbref["fbref_team_key"].map(aliases)

    fpl = current_players[
        ["player_key", "team_id", "full_name", "web_name"]
    ].copy()
    fpl["full_key"] = fpl["full_name"].map(normalise_player_name)
    fpl["web_key"] = fpl["web_name"].map(normalise_player_name)

    rows = []
    for row in fbref.itertuples(index=False):
        if pd.isna(row.team_id):
            continue
        target = str(row.fbref_name_key)
        candidates = fpl.loc[
            fpl["team_id"].eq(int(row.team_id))
        ].copy()
        exact = candidates.loc[
            candidates["full_key"].eq(target)
            | candidates["web_key"].eq(target)
        ]
        if len(exact) == 1:
            match = exact.iloc[0]
            method = "exact"
            score = 1.0
        else:
            if candidates.empty:
                continue
            candidates["_score"] = candidates["full_key"].map(
                lambda name: SequenceMatcher(
                    None, target, str(name)
                ).ratio()
            )
            candidates = candidates.sort_values("_score", ascending=False)
            if float(candidates.iloc[0]["_score"]) < 0.90:
                continue
            match = candidates.iloc[0]
            method = "fuzzy_same_team"
            score = float(match["_score"])

        payload = row._asdict()
        payload["player_key"] = match["player_key"]
        payload["fbref_match_method"] = method
        payload["fbref_match_score"] = score
        rows.append(payload)

    matched = pd.DataFrame(rows)
    if matched.empty:
        return matched

    keep = [
        "player_key",
        "fbref_player",
        "fbref_team",
        "fbref_minutes",
        "fbref_xg",
        "fbref_npxg",
        "fbref_xag",
        "fbref_pk",
        "fbref_pkatt",
        "fbref_xg_per90",
        "fbref_npxg_per90",
        "fbref_xag_per90",
        "fbref_pkatt_per90",
        "fbref_shots_per90",
        "fbref_sot_per90",
        "fbref_match_method",
        "fbref_match_score",
    ]
    keep = [column for column in keep if column in matched.columns]
    return (
        matched[keep]
        .drop_duplicates("player_key", keep="last")
        .reset_index(drop=True)
    )



def current_fbref_raw_schema_report(
    tables: dict[str, pd.DataFrame | None],
) -> pd.DataFrame:
    """
    Surface the raw flattened FBref schema around the fields this project wants.

    This is diagnostic only: the production forecast never requires FBref. When
    npxG/xAG coverage disappears after an upstream schema/scraper change, this
    report shows whether useful columns still exist under different names.
    """
    rows = []
    for table_name, raw in tables.items():
        if raw is None or not isinstance(raw, pd.DataFrame) or raw.empty:
            rows.append({
                "table": str(table_name),
                "rows": 0,
                "column_count": 0,
                "npxg_candidates": [],
                "xag_candidates": [],
                "xa_candidates": [],
                "shots_candidates": [],
                "sot_candidates": [],
                "pkatt_candidates": [],
            })
            continue

        flat = flatten_soccerdata_frame(raw)
        columns = [str(column) for column in flat.columns]

        def candidates(*tokens: str) -> list[str]:
            result = []
            for column in columns:
                key = re.sub(r"[^a-z0-9]+", "_", column.lower())
                if any(token in key for token in tokens):
                    result.append(column)
            return result[:25]

        rows.append({
            "table": str(table_name),
            "rows": int(len(flat)),
            "column_count": int(len(columns)),
            "npxg_candidates": candidates("npxg"),
            "xag_candidates": candidates("xag"),
            "xa_candidates": [
                column
                for column in candidates("_xa", "expected_xa")
                if "xag" not in column.lower()
            ][:25],
            "shots_candidates": candidates("shots", "_sh", "standard_sh"),
            "sot_candidates": candidates("sot", "shots_on_target"),
            "pkatt_candidates": candidates("pkatt", "pk_att"),
        })

    return pd.DataFrame(rows)


def load_current_fbref_context(
    current_players: pd.DataFrame,
    team_table: pd.DataFrame,
) -> pd.DataFrame:
    global current_fbref_raw_schema_audit
    global current_fbref_refresh_audit

    legacy_cache = pd.DataFrame()
    current_fbref_raw_schema_audit = pd.DataFrame()
    current_fbref_refresh_audit = {
        "source": "not_started",
        "fresh_refresh_attempted": False,
        "matched_players": 0,
        "fallback_used": False,
        "error": None,
    }

    if not LIVE_CURRENT_FBREF_ENABLED:
        current_fbref_refresh_audit.update({
            "source": "disabled",
        })
        return pd.DataFrame()

    if CURRENT_FBREF_LIVE_CACHE_PATH.exists():
        try:
            legacy_cache = pd.read_csv(
                CURRENT_FBREF_LIVE_CACHE_PATH,
                compression="gzip",
            )
        except Exception:
            legacy_cache = pd.DataFrame()

    if (
        _fresh_live_cache(
            CURRENT_FBREF_LIVE_CACHE_PATH,
            LIVE_CURRENT_FBREF_REFRESH_HOURS,
        )
        and not legacy_cache.empty
        and "fbref_schema_version" in legacy_cache.columns
        and pd.to_numeric(
            legacy_cache["fbref_schema_version"],
            errors="coerce",
        ).eq(
            int(CURRENT_FBREF_CACHE_SCHEMA_VERSION)
        ).all()
    ):
        current_fbref_refresh_audit.update({
            "source": "fresh_cache",
            "matched_players": int(len(legacy_cache)),
        })
        return legacy_cache

    try:
        import soccerdata as sd

        reader = sd.FBref(
            leagues="ENG-Premier League",
            seasons=CURRENT_SEASON_START_YEAR,
            data_dir=EXTERNAL_SOCCERDATA_CACHE_DIR,
        )

        standard = reader.read_player_season_stats(
            stat_type="standard"
        )

        try:
            shooting = reader.read_player_season_stats(
                stat_type="shooting"
            )
        except Exception as exc:
            warnings.warn(
                f"Current FBref shooting table unavailable: {exc}"
            )
            shooting = pd.DataFrame()

        # soccerdata's current FBref player-season API exposes standard,
        # keeper, shooting, playing_time and misc; "passing" is not a valid
        # stat_type in the versions targeted by v2.2. Standard already carries
        # xAG in supported layouts, so do not make a guaranteed failing request.
        passing = pd.DataFrame()

        current_fbref_raw_schema_audit = current_fbref_raw_schema_report({
            "standard": standard,
            "shooting": shooting,
        })
        current_fbref_refresh_audit.update({
            "source": "soccerdata_refresh",
            "fresh_refresh_attempted": True,
        })

        tidy = _tidy_current_fbref_player_table(
            standard,
            shooting,
            passing,
        )
        matched = _match_current_fbref_to_fpl(
            tidy,
            current_players,
            team_table,
        )

        if not matched.empty:
            matched["fbref_schema_version"] = int(
                CURRENT_FBREF_CACHE_SCHEMA_VERSION
            )
            matched.to_csv(
                CURRENT_FBREF_LIVE_CACHE_PATH,
                index=False,
                compression="gzip",
            )

        current_fbref_refresh_audit.update({
            "matched_players": int(len(matched)),
            "fallback_used": False,
        })
        return matched

    except Exception as exc:
        message = (
            "Current-season FBref refresh failed. "
            f"Reason: {exc}"
        )
        if LIVE_CURRENT_FBREF_STRICT:
            raise RuntimeError(message) from exc

        current_fbref_refresh_audit.update({
            "source": "refresh_failed",
            "fresh_refresh_attempted": True,
            "error": str(exc),
        })

        if not legacy_cache.empty:
            current_fbref_refresh_audit.update({
                "source": "legacy_cache_fallback",
                "matched_players": int(len(legacy_cache)),
                "fallback_used": True,
            })
            warnings.warn(
                message
                + " Falling back to the existing partial cache."
            )
            return legacy_cache

        current_fbref_refresh_audit.update({
            "source": "fpl_underlying_fallback",
            "matched_players": 0,
            "fallback_used": True,
        })
        warnings.warn(
            message
            + " FPL current-season underlying data remains the fallback."
        )
        return pd.DataFrame()


def build_team_penalty_rate_context(
    current_players: pd.DataFrame,
    current_fbref: pd.DataFrame,
    current_team_fixtures: pd.DataFrame,
) -> pd.DataFrame:
    team_matches = (
        current_team_fixtures.loc[
            current_team_fixtures["finished"].fillna(False)
        ]
        .groupby("team_id")["fixture_id"]
        .nunique()
        .rename("current_pl_matches")
        .reset_index()
    )

    team_pkatt = pd.DataFrame(
        columns=["team_id", "current_fbref_pkatt"]
    )
    if (
        current_fbref is not None
        and not current_fbref.empty
        and "fbref_pkatt" in current_fbref.columns
    ):
        lookup = current_players[
            ["player_key", "team_id"]
        ].drop_duplicates("player_key")
        team_pkatt = (
            current_fbref
            .merge(
                lookup,
                on="player_key",
                how="left",
                validate="one_to_one",
            )
            .groupby("team_id")["fbref_pkatt"]
            .sum(min_count=1)
            .rename("current_fbref_pkatt")
            .reset_index()
        )

    context = (
        teams[["team_id", "team_short_name"]]
        .merge(team_matches, on="team_id", how="left")
        .merge(team_pkatt, on="team_id", how="left")
    )
    context["current_pl_matches"] = pd.to_numeric(
        context["current_pl_matches"],
        errors="coerce",
    ).fillna(0.0)
    context["current_fbref_pkatt"] = pd.to_numeric(
        context["current_fbref_pkatt"],
        errors="coerce",
    ).fillna(0.0)

    prior_matches = float(SET_PIECE_TEAM_PENALTY_PRIOR_MATCHES)
    prior_rate = float(SET_PIECE_DEFAULT_TEAM_PENALTY_RATE)
    context["team_penalty_attempt_rate"] = (
        context["current_fbref_pkatt"]
        + prior_matches * prior_rate
    ) / (
        context["current_pl_matches"] + prior_matches
    )
    context["team_penalty_attempt_rate"] = (
        context["team_penalty_attempt_rate"].clip(0.04, 0.25)
    )
    context["team_penalty_rate_source"] = np.where(
        context["current_fbref_pkatt"].gt(0.0),
        "FBref current PL + league prior",
        "league prior",
    )
    return context


def _set_piece_order_share(
    values: pd.Series,
    mapping: dict[int, float],
) -> pd.Series:
    return (
        pd.to_numeric(values, errors="coerce")
        .map(mapping)
        .fillna(0.0)
    )


def attach_live_fbref_and_set_piece_context(
    frame: pd.DataFrame,
    current_players: pd.DataFrame,
    current_fbref: pd.DataFrame,
    penalty_context: pd.DataFrame,
) -> pd.DataFrame:
    result = frame.copy()

    role_columns = [
        column
        for column in [
            "player_key",
            "penalties_order",
            "penalties_text",
            "corners_and_indirect_freekicks_order",
            "corners_and_indirect_freekicks_text",
            "direct_freekicks_order",
            "direct_freekicks_text",
        ]
        if column in current_players.columns
    ]
    result = result.merge(
        current_players[role_columns].drop_duplicates("player_key"),
        on="player_key",
        how="left",
        validate="many_to_one",
    )

    if current_fbref is not None and not current_fbref.empty:
        columns = [
            column
            for column in current_fbref.columns
            if column != "player_key"
            and column not in result.columns
        ]
        result = result.merge(
            current_fbref[["player_key"] + columns],
            on="player_key",
            how="left",
            validate="many_to_one",
        )

    return result.merge(
        penalty_context[
            [
                "team_id",
                "team_penalty_attempt_rate",
                "team_penalty_rate_source",
            ]
        ],
        on="team_id",
        how="left",
        validate="many_to_one",
    )


def apply_live_set_piece_role_overlay(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    result = frame.copy()

    result["pre_set_piece_predicted_xg_per90"] = pd.to_numeric(
        result["predicted_xg_per90"],
        errors="coerce",
    ).fillna(0.0)
    result["pre_set_piece_predicted_xa_per90"] = pd.to_numeric(
        result["predicted_xa_per90"],
        errors="coerce",
    ).fillna(0.0)

    for column in [
        "set_piece_penalty_xg_overlay_per90",
        "set_piece_direct_fk_xg_overlay_per90",
        "set_piece_corner_xa_overlay_per90",
    ]:
        result[column] = 0.0

    if not LIVE_SET_PIECE_ROLE_ENABLED:
        result["set_piece_role_score"] = 0.0
        return result

    non_gk = ~result["model_position"].eq("GK")

    penalty_order = pd.to_numeric(
        result.get(
            "penalties_order",
            pd.Series(np.nan, index=result.index),
        ),
        errors="coerce",
    )
    corner_order = pd.to_numeric(
        result.get(
            "corners_and_indirect_freekicks_order",
            pd.Series(np.nan, index=result.index),
        ),
        errors="coerce",
    )
    direct_order = pd.to_numeric(
        result.get(
            "direct_freekicks_order",
            pd.Series(np.nan, index=result.index),
        ),
        errors="coerce",
    )

    penalty_share = penalty_order.map(
        SET_PIECE_PENALTY_ORDER_SHARES
    ).fillna(0.0)
    team_penalty_rate = pd.to_numeric(
        result.get("team_penalty_attempt_rate"),
        errors="coerce",
    ).fillna(SET_PIECE_DEFAULT_TEAM_PENALTY_RATE)

    penalty_overlay = (
        team_penalty_rate
        * penalty_share
        * SET_PIECE_PENALTY_XG
        * SET_PIECE_PENALTY_ROLE_BLEND
    )
    direct_overlay = direct_order.map(
        SET_PIECE_DIRECT_FK_XG_ROLE_PER90
    ).fillna(0.0)
    corner_overlay = corner_order.map(
        SET_PIECE_CORNER_XA_ROLE_PER90
    ).fillna(0.0)

    result.loc[
        non_gk, "set_piece_penalty_xg_overlay_per90"
    ] = penalty_overlay.loc[non_gk]
    result.loc[
        non_gk, "set_piece_direct_fk_xg_overlay_per90"
    ] = direct_overlay.loc[non_gk]
    result.loc[
        non_gk, "set_piece_corner_xa_overlay_per90"
    ] = corner_overlay.loc[non_gk]

    # The base xG model is trained on total xG, which already contains
    # penalties. Adding the full penalty-taker overlay would double-count that
    # component. Keep the role estimate for diagnostics, but only add it when
    # the base model is explicitly configured as non-penalty xG.
    additive_penalty_overlay = (
        result.loc[non_gk, "set_piece_penalty_xg_overlay_per90"]
        if LIVE_SET_PIECE_PENALTY_XG_ADDITIVE
        else 0.0
    )
    result.loc[non_gk, "predicted_xg_per90"] = (
        result.loc[non_gk, "pre_set_piece_predicted_xg_per90"]
        + additive_penalty_overlay
        + result.loc[non_gk, "set_piece_direct_fk_xg_overlay_per90"]
    ).clip(
        0.0,
        PREDICTED_EVENT_RATE_CAPS["predicted_xg_per90"],
    )

    result.loc[non_gk, "predicted_xa_per90"] = (
        result.loc[non_gk, "pre_set_piece_predicted_xa_per90"]
        + result.loc[non_gk, "set_piece_corner_xa_overlay_per90"]
    ).clip(
        0.0,
        PREDICTED_EVENT_RATE_CAPS["predicted_xa_per90"],
    )

    # Optional independent current-FBref xG/xAG blend. Deliberately OFF by
    # default because FPL already supplies live xG/xA.
    if LIVE_CURRENT_FBREF_EVENT_BLEND_ENABLED:
        minutes = pd.to_numeric(
            result.get("fbref_minutes"),
            errors="coerce",
        ).fillna(0.0)
        weight = (
            minutes.div(900.0).clip(0.0, 1.0)
            * LIVE_CURRENT_FBREF_EVENT_BLEND_MAX
        )

        for prediction_column, external_column in [
            ("predicted_xg_per90", "fbref_xg_per90"),
            ("predicted_xa_per90", "fbref_xag_per90"),
        ]:
            if external_column not in result.columns:
                continue
            external = pd.to_numeric(
                result[external_column],
                errors="coerce",
            )
            valid = external.notna() & non_gk
            result.loc[valid, prediction_column] = (
                (1.0 - weight.loc[valid])
                * result.loc[valid, prediction_column]
                + weight.loc[valid] * external.loc[valid]
            )

    result["set_piece_role_score"] = (
        penalty_order.map({1: 3.0, 2: 1.0, 3: 0.4}).fillna(0.0)
        + corner_order.map({1: 1.0, 2: 0.5, 3: 0.25}).fillna(0.0)
        + direct_order.map({1: 1.5, 2: 0.75, 3: 0.35}).fillna(0.0)
    )
    return result



def current_fbref_coverage_report(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    """Return field-level coverage and readiness instead of a single opaque status."""
    fields = [
        "fbref_minutes",
        "fbref_npxg_per90",
        "fbref_xag_per90",
        "fbref_shots_per90",
        "fbref_sot_per90",
        "fbref_pkatt_per90",
    ]
    total = int(len(frame))

    rows = []
    for field in fields:
        if field in frame.columns:
            values = pd.to_numeric(frame[field], errors="coerce")
            populated = int(values.notna().sum())
        else:
            populated = 0

        coverage = float(populated / total) if total > 0 else 0.0
        rows.append({
            "field": field,
            "populated_players": populated,
            "matched_players": total,
            "coverage": coverage,
            "field_status": (
                "ACTIVE"
                if coverage >= float(CURRENT_FBREF_MIN_KEY_FIELD_COVERAGE)
                else "PARTIAL"
                if populated > 0
                else "UNAVAILABLE"
            ),
        })

    return pd.DataFrame(rows)


def current_fbref_status_from_coverage(
    matched: pd.DataFrame,
    coverage: pd.DataFrame,
) -> str:
    if not LIVE_CURRENT_FBREF_ENABLED:
        return "OFF"
    if matched is None or matched.empty:
        return "UNAVAILABLE"

    lookup = (
        coverage.set_index("field")["coverage"].to_dict()
        if coverage is not None and not coverage.empty
        else {}
    )
    key_fields = [
        "fbref_npxg_per90",
        "fbref_xag_per90",
        "fbref_shots_per90",
    ]
    if all(
        float(lookup.get(field, 0.0))
        >= float(CURRENT_FBREF_MIN_KEY_FIELD_COVERAGE)
        for field in key_fields
    ):
        return "ACTIVE"

    return "PARTIAL"


current_fbref_live_context = load_current_fbref_context(
    players,
    teams,
)
current_team_penalty_context = build_team_penalty_rate_context(
    players,
    current_fbref_live_context,
    team_fixtures,
)

current_fbref_live_coverage = current_fbref_coverage_report(
    current_fbref_live_context
)
current_fbref_live_status = current_fbref_status_from_coverage(
    current_fbref_live_context,
    current_fbref_live_coverage,
)
current_fbref_live_coverage_dict = {
    str(row.field): {
        "populated_players": int(row.populated_players),
        "matched_players": int(row.matched_players),
        "coverage": float(row.coverage),
    }
    for row in current_fbref_live_coverage.itertuples(index=False)
}

current_fbref_key_field_audit = pd.DataFrame([
    {
        "field": field,
        "coverage": float(
            current_fbref_live_coverage_dict.get(field, {}).get(
                "coverage", 0.0
            )
        ),
        "production_source": (
            "FBref optional context"
            if float(
                current_fbref_live_coverage_dict.get(field, {}).get(
                    "coverage", 0.0
                )
            ) >= float(CURRENT_FBREF_MIN_KEY_FIELD_COVERAGE)
            else "FPL fallback / field disabled"
        ),
    }
    for field in (
        "fbref_npxg_per90",
        "fbref_xag_per90",
        "fbref_shots_per90",
    )
])

print(
    "Current FBref status:",
    current_fbref_live_status,
)
print(
    "Current FBref players matched:",
    len(current_fbref_live_context),
)
if not current_fbref_live_coverage.empty:
    print("Current FBref field coverage")
    display(
        current_fbref_live_coverage.assign(
            coverage_pct=lambda frame:
                (100.0 * frame["coverage"]).round(1)
        )
    )

_fbref_key_fields_degraded = bool(
    (
        current_fbref_key_field_audit["coverage"]
        < float(CURRENT_FBREF_MIN_KEY_FIELD_COVERAGE)
    ).any()
)
if _fbref_key_fields_degraded:
    print(
        "FBref key-field degradation detected; affected fields remain optional "
        "and FPL current-season xG/xA stays authoritative."
    )
    display(current_fbref_key_field_audit)
    if (
        isinstance(
            globals().get("current_fbref_raw_schema_audit"),
            pd.DataFrame,
        )
        and not current_fbref_raw_schema_audit.empty
    ):
        print("FBref raw-schema candidates")
        display(current_fbref_raw_schema_audit)

print(
    "Official primary penalty takers:",
    int(
        pd.to_numeric(
            players.get("penalties_order"),
            errors="coerce",
        ).eq(1).sum()
    ),
)

In [ ]:
final_historical_team_predictions = pd.concat(
    [
        historical_team_fixtures[
            ["season", "fixture_id", "team_key"]
        ].reset_index(drop=True),
        predict_team_goal_components(
            final_team_goal_models,
            historical_team_fixtures,
        ).reset_index(drop=True),
    ],
    axis=1,
)

final_historical_points_data = (
    historical_model_data
    .merge(
        final_historical_team_predictions,
        on=["season", "fixture_id", "team_key"],
        how="left",
        validate="many_to_one",
    )
)
final_historical_points_data = add_player_per90_history_features(
    final_historical_points_data
)
final_historical_points_data = add_player_event_targets(
    final_historical_points_data
)
final_historical_points_data = final_historical_points_data.loc[
    final_historical_points_data["model_position"].isin(
        PLAYER_SCORING_POSITIONS
    )
].copy()

final_player_event_models = fit_player_event_rate_models(
    final_historical_points_data
)
final_defensive_action_priors = (
    defensive_action_position_priors(
        final_historical_points_data
    )
)
final_defensive_return_priors = (
    defensive_return_position_priors(
        final_historical_points_data
    )
)
final_rare_event_rates = rare_event_position_rates(
    final_historical_points_data
)

additional_event_columns = [
    column
    for column in current_event_as_of_history.columns
    if column != "player_key"
    and column not in final_live_minutes_predictions.columns
]

live_expected_points_inputs = (
    final_live_minutes_predictions
    .merge(
        current_event_as_of_history[
            ["player_key"] + additional_event_columns
        ],
        on="player_key",
        how="left",
        validate="many_to_one",
    )
    .merge(
        live_team_goal_predictions,
        on=["fixture_id", "team_id"],
        how="left",
        validate="many_to_one",
    )
)

live_expected_points_inputs = add_player_per90_history_features(
    live_expected_points_inputs
)

live_event_schema_before = model_feature_contract_report(
    live_expected_points_inputs,
    PLAYER_EVENT_FEATURE_COLUMNS,
)
live_expected_points_inputs = materialise_model_feature_columns(
    live_expected_points_inputs,
    PLAYER_EVENT_FEATURE_COLUMNS,
    context="live.player_event_features",
    categorical_columns=PLAYER_EVENT_CATEGORICAL_FEATURES,
)
if live_event_schema_before["missing"]:
    print(
        "Live player-event schema repaired:",
        live_event_schema_before["missing"],
    )

live_event_rate_predictions = predict_player_event_rates(
    final_player_event_models,
    live_expected_points_inputs,
)

live_expected_points_inputs = pd.concat(
    [
        live_expected_points_inputs.reset_index(drop=True),
        live_event_rate_predictions.reset_index(drop=True),
    ],
    axis=1,
)

live_expected_points_inputs = attach_live_fbref_and_set_piece_context(
    live_expected_points_inputs,
    players,
    current_fbref_live_context,
    current_team_penalty_context,
)
live_expected_points_inputs = apply_live_set_piece_role_overlay(
    live_expected_points_inputs
)

v26_set_conversion(final_historical_points_data, label="live")
live_xp_components = calculate_component_expected_points(
    live_expected_points_inputs,
    final_defensive_action_priors,
    final_rare_event_rates,
    final_defensive_return_priors,
)

live_expected_points = pd.concat(
    [
        live_expected_points_inputs.reset_index(drop=True),
        live_xp_components.reset_index(drop=True),
    ],
    axis=1,
)

# Autonomous production corrections. Premium attacking-rate correction and
# xP calibration are both gated by leakage-safe OOS validation artifacts.
# Rejected corrections remain visible in Section 11 but are not applied live.
live_premium_payload = active_premium_attacker_payload(premium_attacker_cache)
live_expected_points = apply_premium_attacker_adjustment(
    live_expected_points,
    live_premium_payload,
    position_col="model_position",
    price_col="price",
)
live_expected_points = apply_sparse_player_market_prior(
    live_expected_points
)
live_xp_payload = active_xp_calibration_payload(xp_calibration_cache)
live_expected_points = apply_xp_calibration_payload(
    live_expected_points,
    live_xp_payload,
    position_col="model_position",
    price_col="price",
    value_col="expected_points",
)

# v2.4 targeted corrections are loaded from a signed historical-OOS artifact.
live_expected_points = apply_v24_targeted_forecast_layer(
    live_expected_points,
    v24_targeted_forecast_cache,
    position_col="model_position",
    price_col="price",
    value_col="expected_points",
)
live_expected_points = apply_v24_uncertainty_payload(
    live_expected_points,
    v24_targeted_forecast_cache,
    position_col="model_position",
)

LIVE_VALIDATION_ARTIFACT_STATE = {
    "forecast_signature": FORECAST_CACHE_SIGNATURE,
    "live_feature_signature": LIVE_FEATURE_SIGNATURE,
    "xp_artifact_id": (
        xp_calibration_cache.get("artifact_id") if xp_calibration_cache else None
    ),
    "premium_artifact_id": (
        premium_attacker_cache.get("artifact_id") if premium_attacker_cache else None
    ),
    "xp_applied": live_xp_payload is not None,
    "premium_applied": live_premium_payload is not None,
    "v24_artifact_id": (v24_targeted_forecast_cache.get("artifact_id") if v24_targeted_forecast_cache else None),
    "v24_applied": bool(live_expected_points.get("v24_targeted_layer_applied", pd.Series(False, index=live_expected_points.index)).any()),
}

assert live_expected_points["expected_points"].notna().all()

for column, upper_cap in PREDICTED_EVENT_RATE_CAPS.items():
    values = pd.to_numeric(
        live_expected_points[column],
        errors="coerce",
    )
    assert np.isfinite(values).all(), (
        f"Live {column} contains a non-finite prediction."
    )
    assert values.between(0.0, upper_cap).all(), (
        f"Live {column} breached its configured prediction cap."
    )

live_goalkeeper_rows = live_expected_points[
    "model_position"
].eq("GK")
assert live_expected_points.loc[
    live_goalkeeper_rows,
    ["predicted_xg_per90", "predicted_xa_per90"],
].abs().to_numpy().max(initial=0.0) < 1e-12, (
    "Live goalkeepers received ordinary attacking projections."
)

assert np.isfinite(
    live_expected_points["expected_points"]
).all()
assert live_expected_points[
    "expected_points"
].between(-5.0, 35.0).all(), (
    "Live fixture-level expected points breached the sanity range."
)

fixture_xp_display_columns = [
    "gameweek",
    "kickoff_time_spain",
    "web_name",
    "team_short_name",
    "opponent_short_name",
    "venue",
    "position",
    "price",
    "start_probability",
    "expected_minutes",
    "all_comp_start_multiplier",
    "previous_non_pl_competition",
    "days_since_previous_all_comp_fixture",
    "previous_non_pl_player_minutes",
    "penalties_order",
    "corners_and_indirect_freekicks_order",
    "direct_freekicks_order",
    "set_piece_role_score",
    "set_piece_penalty_xg_overlay_per90",
    "projected_goals",
    "projected_assists",
    "clean_sheet_probability",
    "raw_expected_points",
    "xp_calibration_adjustment",
    "v24_upper_tail_adjustment",
    "v24_midfield_rank_adjustment",
    "v24_total_adjustment",
    "minutes_uncertainty_p80",
    "forecast_uncertainty_p80",
    "expected_points",
]

live_fixture_xp_preview = (
    live_expected_points
    .sort_values(
        ["gameweek", "expected_points", "web_name"],
        ascending=[True, False, True],
    )
    .loc[:, fixture_xp_display_columns]
    .groupby("gameweek", group_keys=False)
    .head(20)
)

display(live_fixture_xp_preview)


# Schema-repair audit ---------------------------------------------------------
# A healthy run should normally show no repairs here. Any entry means an
# upstream source/merge changed shape but the model contract recovered it
# without a KeyError; inspect repeated repairs before trusting a long-term run.
if MODEL_SCHEMA_REPAIR_LOG:
    schema_repair_audit = pd.DataFrame(MODEL_SCHEMA_REPAIR_LOG)
    schema_repair_summary = (
        schema_repair_audit
        .groupby("context", as_index=False)
        .agg(
            calls_repaired=("context", "size"),
            latest_repairs=("repairs", "last"),
            max_rows=("rows", "max"),
        )
        .sort_values(
            ["calls_repaired", "context"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )
    print(
        "Model schema contracts: automatic repairs were required. "
        "Review the summary below."
    )
    display(schema_repair_summary)
else:
    print("Model schema contracts: PASS — no automatic repairs required.")


In [ ]:

XP_COMPONENT_COLUMNS = [
    "xp_appearance",
    "xp_goals",
    "xp_assists",
    "xp_clean_sheet",
    "xp_saves",
    "xp_goals_conceded",
    "xp_defensive_contribution",
    "xp_bonus",
    "xp_yellow_cards",
    "xp_red_cards",
    "xp_own_goals",
    "xp_penalty_saves",
    "xp_penalty_misses",
]

player_gameweek_keys = [
    "player_id",
    "player_key",
    "web_name",
    "team_id",
    "team_short_name",
    "position",
    "price",
    "gameweek",
]

live_gameweek_aggregation = {
    "fixture_count": ("fixture_id", "nunique"),
    "expected_minutes": ("expected_minutes", "sum"),
    "projected_goals": ("projected_goals", "sum"),
    "projected_assists": ("projected_assists", "sum"),
    "no_appearance_probability": (
        "no_appearance_probability",
        "prod",
    ),
    "pre_premium_expected_points": ("pre_premium_expected_points", "sum"),
    "premium_attacker_adjustment": ("premium_attacker_adjustment", "sum"),
    "raw_expected_points": ("raw_expected_points", "sum"),
    "xp_calibration_adjustment": ("xp_calibration_adjustment", "sum"),
    "expected_points": ("expected_points", "sum"),
    **{
        component: (component, "sum")
        for component in XP_COMPONENT_COLUMNS
    },
}

# External-prior diagnostics are constant or near-constant for a player over
# the Gameweek. Preserve them so Section 12 can flag recommendations that
# still depend materially on non-PL evidence.
for source_column, output_name, reducer in [
    (
        "external_prior_available",
        "external_prior_available",
        "max",
    ),
    (
        "external_prior_reliability",
        "external_prior_reliability",
        "max",
    ),
    (
        "external_prior_match_score",
        "external_prior_match_score",
        "max",
    ),
    (
        "external_minutes_prior_weight",
        "external_minutes_prior_weight",
        "max",
    ),
    (
        "external_event_prior_weight",
        "external_event_prior_weight",
        "max",
    ),
    (
        "external_prior_non_pl",
        "external_prior_non_pl",
        "max",
    ),
    (
        "external_source_leagues",
        "external_source_leagues",
        "first",
    ),
    (
        "external_prior_match_method",
        "external_prior_match_method",
        "first",
    ),
    (
        "external_prior_source_note",
        "external_prior_source_note",
        "first",
    ),
    (
        "market_fallback_weight",
        "market_fallback_weight",
        "max",
    ),
    (
        "market_fallback_evidence_fade",
        "market_fallback_evidence_fade",
        "max",
    ),
    (
        "market_fallback_prior_points",
        "market_fallback_prior_points",
        "max",
    ),
]:
    if source_column in live_expected_points.columns:
        live_gameweek_aggregation[output_name] = (
            source_column,
            reducer,
        )


# Preserve all-competition, set-piece and current-FBref diagnostics.
for source_column, reducer in [
    ("all_comp_start_multiplier", "mean"),
    ("all_comp_minutes_adjustment", "sum"),
    ("previous_non_pl_competition", "first"),
    ("days_since_previous_all_comp_fixture", "min"),
    ("days_until_next_all_comp_fixture", "min"),
    ("previous_non_pl_player_minutes", "max"),
    ("penalties_order", "first"),
    ("corners_and_indirect_freekicks_order", "first"),
    ("direct_freekicks_order", "first"),
    ("set_piece_role_score", "max"),
    ("set_piece_penalty_xg_overlay_per90", "mean"),
    ("set_piece_direct_fk_xg_overlay_per90", "mean"),
    ("set_piece_corner_xa_overlay_per90", "mean"),
    ("fbref_minutes", "max"),
    ("fbref_npxg_per90", "max"),
    ("fbref_xag_per90", "max"),
    ("fbref_shots_per90", "max"),
    ("fbref_sot_per90", "max"),
    ("fbref_pkatt_per90", "max"),
]:
    if source_column in live_expected_points.columns:
        live_gameweek_aggregation[source_column] = (
            source_column,
            reducer,
        )


# Preserve early-season stabilisation diagnostics through the weekly table.
for source_column, reducer in [
    ("early_season_player_shrinkage_active", "max"),
    ("early_season_player_current_fixtures", "max"),
    ("early_season_player_archived_pl_fixtures", "max"),
    ("early_season_player_current_weight", "max"),
    ("early_season_player_prior_source", "first"),
    ("team_early_season_shrinkage_active", "max"),
    ("team_early_season_current_matches", "max"),
    ("team_early_season_archived_pl_matches", "max"),
    ("team_early_season_current_weight", "max"),
    ("team_early_season_prior_source", "first"),
    ("opponent_early_season_shrinkage_active", "max"),
    ("opponent_early_season_current_matches", "max"),
    ("opponent_early_season_archived_pl_matches", "max"),
    ("opponent_early_season_current_weight", "max"),
    ("opponent_early_season_prior_source", "first"),
]:
    if source_column in live_expected_points.columns:
        live_gameweek_aggregation[source_column] = (
            source_column,
            reducer,
        )

live_gameweek_expected_points = (
    live_expected_points
    .groupby(player_gameweek_keys, dropna=False)
    .agg(**live_gameweek_aggregation)
    .reset_index()
)

# At Gameweek level, a player appears if they appear in at least one fixture.
live_gameweek_expected_points["no_appearance_probability"] = (
    pd.to_numeric(
        live_gameweek_expected_points["no_appearance_probability"],
        errors="coerce",
    )
    .fillna(1.0)
    .clip(0.0, 1.0)
)
live_gameweek_expected_points["appearance_probability"] = (
    1.0
    - live_gameweek_expected_points["no_appearance_probability"]
).clip(0.0, 1.0)

planning_horizon_aggregation = {
    "projected_fixtures": ("fixture_count", "sum"),
    "horizon_expected_minutes": ("expected_minutes", "sum"),
    "horizon_projected_goals": ("projected_goals", "sum"),
    "horizon_projected_assists": ("projected_assists", "sum"),
    "horizon_expected_points": ("expected_points", "sum"),
}
for column in [
    "external_prior_available",
    "external_prior_reliability",
    "external_minutes_prior_weight",
    "external_event_prior_weight",
    "early_season_player_shrinkage_active",
    "early_season_player_current_weight",
    "team_early_season_shrinkage_active",
    "team_early_season_current_weight",
]:
    if column in live_gameweek_expected_points.columns:
        planning_horizon_aggregation[column] = (
            column,
            "max",
        )

planning_horizon_expected_points = (
    live_gameweek_expected_points
    .groupby(
        [
            "player_id",
            "player_key",
            "web_name",
            "team_short_name",
            "position",
            "price",
        ],
        dropna=False,
    )
    .agg(**planning_horizon_aggregation)
    .reset_index()
)

print("Top players in the next Gameweek")
display(
    live_gameweek_expected_points.loc[
        live_gameweek_expected_points["gameweek"].eq(
            first_planning_gameweek
        )
    ]
    .sort_values("expected_points", ascending=False)
    .head(30)
)

print(f"Top players across the {PLANNING_HORIZON}-Gameweek horizon")
display(
    planning_horizon_expected_points
    .sort_values("horizon_expected_points", ascending=False)
    .head(40)
)


## 6A. New-signing prior audit

This audit shows which current players still depend materially on previous-season external evidence, which source league was matched, and how much the external prior changes minutes and event-rate forecasts. High-priced unmatched players are surfaced for manual review.


In [ ]:

# New-signing / new-to-PL prior audit -----------------------------------------
current_new_player_context = (
    players[
        [
            "player_key",
            "id",
            "web_name",
            "full_name",
            "team_short_name",
            "position",
            "price",
        ]
    ]
    .merge(
        current_as_of_history[
            [
                "player_key",
                "prior_career_fixture_count",
                "prior_season_fixture_count",
            ]
        ],
        on="player_key",
        how="left",
        validate="one_to_one",
    )
    .merge(
        current_external_player_priors,
        on="player_key",
        how="left",
        validate="one_to_one",
    )
)

current_new_player_context[
    "prior_career_fixture_count"
] = pd.to_numeric(
    current_new_player_context[
        "prior_career_fixture_count"
    ],
    errors="coerce",
).fillna(0.0)

current_new_player_context["new_to_pl_model"] = (
    current_new_player_context[
        "prior_career_fixture_count"
    ].lt(NEW_PLAYER_PRIOR_FADE_OUT_PL_FIXTURES)
)

current_new_player_context["new_or_returning_signing_model"] = (
    current_new_player_context["new_to_pl_model"]
    | current_new_player_context[
        "external_prior_non_pl"
    ].fillna(False).astype(bool)
)

current_first_gw_prior_usage = (
    live_gameweek_expected_points.loc[
        live_gameweek_expected_points["gameweek"].eq(
            first_planning_gameweek
        ),
        [
            column
            for column in [
                "player_key",
                "expected_minutes",
                "expected_points",
                "external_minutes_prior_weight",
                "external_event_prior_weight",
            ]
            if column in live_gameweek_expected_points.columns
        ],
    ]
)

current_new_player_audit = (
    current_new_player_context
    .merge(
        current_first_gw_prior_usage,
        on="player_key",
        how="left",
        validate="one_to_one",
    )
    .loc[
        lambda frame: (
            frame["new_or_returning_signing_model"]
            | frame["external_prior_available"].fillna(0).eq(1)
        )
    ]
    .sort_values(
        [
            "external_event_prior_weight",
            "price",
        ],
        ascending=[False, False],
        na_position="last",
    )
)

new_player_audit_columns = [
    "web_name",
    "team_short_name",
    "position",
    "price",
    "prior_career_fixture_count",
    "prior_season_fixture_count",
    "external_prior_available",
    "external_prior_non_pl",
    "external_player",
    "external_source_leagues",
    "external_prior_match_method",
    "external_prior_match_score",
    "external_prior_source_note",
    "external_prior_reliability",
    "external_role_score",
    "external_pl_equivalent_xg_per90",
    "external_pl_equivalent_xa_per90",
    "external_pl_equivalent_saves_per90",
    "external_minutes_prior_weight",
    "external_event_prior_weight",
    "market_fallback_weight",
    "market_fallback_evidence_fade",
    "market_fallback_prior_points",
    "expected_minutes",
    "expected_points",
]
new_player_audit_columns = [
    column
    for column in new_player_audit_columns
    if column in current_new_player_audit.columns
]

print("New-signing / sparse-PL-history prior audit")
display(
    current_new_player_audit[
        new_player_audit_columns
    ].head(60)
)

unmatched_sparse_players = current_new_player_audit.loc[
    current_new_player_audit[
        "new_or_returning_signing_model"
    ]
    & ~current_new_player_audit[
        "external_prior_available"
    ].fillna(0).eq(1)
]

if not unmatched_sparse_players.empty:
    print(
        "Sparse-PL-history players without an automatic external prior. "
        "For important signings from unsupported leagues, add a row to "
        "EXTERNAL_MANUAL_PRIORS and rerun from the external-prior section."
    )
    display(
        unmatched_sparse_players[
            [
                "web_name",
                "team_short_name",
                "position",
                "price",
                "prior_career_fixture_count",
            ]
        ]
        .sort_values("price", ascending=False)
        .head(40)
    )


## 6B. Defender-rating audit

The previous live output ranked defenders disproportionately highly. The audit separates three possible causes:

1. **Legitimate scoring effects:** defenders receive four clean-sheet points and can earn two defensive-contribution points.
2. **Attacking-rate shrinkage:** a pooled xG/xA model was pulling defenders and forwards toward a similar average.
3. **Threshold calibration:** a pure Poisson action model was making defensive-contribution returns too close to automatic for some centre-backs.

The revised model fits xG and xA separately by position and blends the action-threshold estimate with leakage-safe historical award frequencies. The tables below show whether the correction has restored a plausible positional balance without removing genuine defensive value.


In [ ]:
first_gameweek_audit = (
    live_gameweek_expected_points.loc[
        live_gameweek_expected_points["gameweek"].eq(
            first_planning_gameweek
        )
    ]
    .copy()
)

position_projection_audit = (
    first_gameweek_audit
    .groupby("position")
    .agg(
        players=("player_id", "size"),
        mean_xp=("expected_points", "mean"),
        median_xp=("expected_points", "median"),
        p90_xp=("expected_points", lambda values: values.quantile(0.90)),
        top_xp=("expected_points", "max"),
        mean_projected_goals=("projected_goals", "mean"),
        mean_projected_assists=("projected_assists", "mean"),
        mean_dc_xp=("xp_defensive_contribution", "mean"),
        mean_clean_sheet_xp=("xp_clean_sheet", "mean"),
    )
    .reset_index()
)

top_projection_component_audit = (
    first_gameweek_audit
    .nlargest(25, "expected_points")
    .assign(
        attacking_xp=lambda frame: (
            frame["xp_goals"] + frame["xp_assists"]
        ),
        defensive_xp=lambda frame: (
            frame["xp_clean_sheet"]
            + frame["xp_goals_conceded"]
            + frame["xp_defensive_contribution"]
            + frame["xp_saves"]
        ),
    )
    [
        [
            "web_name",
            "team_short_name",
            "position",
            "price",
            "expected_minutes",
            "expected_points",
            "xp_appearance",
            "attacking_xp",
            "xp_clean_sheet",
            "xp_goals_conceded",
            "xp_defensive_contribution",
            "xp_bonus",
        ]
    ]
)

position_event_rate_audit = (
    live_expected_points.loc[
        live_expected_points["gameweek"].eq(
            first_planning_gameweek
        )
    ]
    .groupby("model_position")
    .agg(
        mean_xg_per90=("predicted_xg_per90", "mean"),
        p90_xg_per90=(
            "predicted_xg_per90",
            lambda values: values.quantile(0.90),
        ),
        mean_xa_per90=("predicted_xa_per90", "mean"),
        p90_xa_per90=(
            "predicted_xa_per90",
            lambda values: values.quantile(0.90),
        ),
        mean_defensive_return_probability=(
            "defensive_return_probability",
            "mean",
        ),
        p90_defensive_return_probability=(
            "defensive_return_probability",
            lambda values: values.quantile(0.90),
        ),
    )
    .reset_index()
)

top_20_position_share = (
    first_gameweek_audit
    .nlargest(20, "expected_points")["position"]
    .value_counts(normalize=True)
    .rename("top_20_share")
    .rename_axis("position")
    .reset_index()
)

print("Position-level projection audit")
display(position_projection_audit)

print("Top projected players and their point components")
display(top_projection_component_audit)

print("Position-specific event-rate audit")
display(position_event_rate_audit)

print("Composition of the top 20")
display(top_20_position_share)

assert (
    position_event_rate_audit
    .set_index("model_position")
    .loc["FWD", "mean_xg_per90"]
    >
    position_event_rate_audit
    .set_index("model_position")
    .loc["DEF", "mean_xg_per90"]
), (
    "Forward xG remains below defender xG after position-specific fitting."
)


premium_attacker_live_audit = (
    live_expected_points.loc[
        live_expected_points["gameweek"].eq(first_planning_gameweek)
        & live_expected_points["model_position"].isin(["MID", "FWD"])
        & pd.to_numeric(live_expected_points["price"], errors="coerce").ge(
            float(PREMIUM_ATTACKER_PRICE_THRESHOLD)
        )
    ]
    .groupby(
        ["player_id", "web_name", "team_short_name", "model_position", "price"],
        as_index=False,
    )
    .agg(
        expected_minutes=("expected_minutes", "sum"),
        predicted_xg_per90=("predicted_xg_per90", "mean"),
        predicted_xa_per90=("predicted_xa_per90", "mean"),
        premium_adjustment=("premium_attacker_adjustment", "sum"),
        raw_component_xp=("pre_premium_expected_points", "sum"),
        final_xp=("expected_points", "sum"),
    )
    .sort_values("final_xp", ascending=False)
)

print("Premium-attacker compression/correction audit")
print(
    "Premium component correction active:",
    active_premium_attacker_payload(premium_attacker_cache) is not None,
)
display(premium_attacker_live_audit.head(30))


### Result of the expected-points stage

The principal outputs are now:

- `live_expected_points`: one row per player and fixture, with every xP component;
- `live_gameweek_expected_points`: one row per player and Gameweek;
- `planning_horizon_expected_points`: total projection across the rolling horizon.

The model is intentionally modular. Later improvements can replace an individual component without rewriting the optimisation layer. The main known limitations are:

- bonus forecasts are trained on historical BPS rules and need 2026/27 recalibration;
- European and domestic-cup congestion is included through a conservative live workload overlay; actual non-PL player minutes are used when the external match feed exposes them;
- official FPL penalty, corner/indirect-free-kick and direct-free-kick orders are included as transparent live role priors;
- team strength currently uses Premier League results only;
- new signings with prior-season FBref (playing time/role) and Understat (xG/xA) coverage use external role/xG/xA/save priors that fade as Premier League evidence accumulates;
- signings from unsupported leagues still require `EXTERNAL_MANUAL_PRIORS` or fall back toward the original league/position priors.

The next stage is the one-Gameweek decision optimiser: legal starting XI, bench order, captain and vice-captain.

The all-competition workload and explicit set-piece overlays are conservative live production layers. They have not yet been independently retrained/backtested across historical all-competition data, so every adjustment is exposed in the audit tables.


## 8. Initial squad, line-up, bench and captain optimisation

This stage converts the repaired expected-points projections into legal FPL decisions.

Before Gameweek 1, `CURRENT_SQUAD_PLAYER_IDS` can remain empty. The optimiser then selects an initial 15-player squad under the £100.0m budget, positional quotas and three-player-per-club limit.

After the season starts, populate `CURRENT_SQUAD_PLAYER_IDS` with the 15 official FPL player IDs in the current squad. The same optimiser will keep that squad fixed and optimise only the projected line-ups and captains.

The mixed-integer objective values:

- expected points from the starting XI in each Gameweek;
- the captain's additional copy of expected points;
- a small squad-depth term so that the four substitutes are not treated as worthless;
- nearer Gameweeks slightly more strongly than distant Gameweeks.

The current Gameweek bench order is then evaluated using appearance probabilities and automatic-substitution scenarios. Captain and vice-captain are selected jointly, accounting for the chance that the captain plays zero minutes.

Automatic-substitution scenarios assume independent player appearance events. This is a transparent baseline; correlated rotation and team-level late news can be added later.

In [ ]:
from itertools import combinations, permutations, product

from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.sparse import coo_matrix


# Leave empty before Gameweek 1 to optimise the initial 15-player squad.
# During the season, replace this with the 15 official FPL player IDs.
CURRENT_SQUAD_PLAYER_IDS: list[int] = []

INITIAL_SQUAD_BUDGET = 100.0
MAX_PLAYERS_PER_CLUB = 3

SQUAD_POSITION_REQUIREMENTS = {
    "GK": 2,
    "DEF": 5,
    "MID": 5,
    "FWD": 3,
}

LINEUP_POSITION_BOUNDS = {
    "GK": (1, 1),
    "DEF": (3, 5),
    "MID": (2, 5),
    "FWD": (1, 3),
}


def optimiser_position(position: pd.Series) -> pd.Series:
    """Normalise official FPL position labels for optimisation."""
    return (
        position.astype("string")
        .replace({"GKP": "GK"})
    )


In [ ]:
def prepare_optimisation_inputs(
    gameweek_xp: pd.DataFrame,
    player_table: pd.DataFrame,
    gameweeks_to_use: list[int],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Create one candidate row per player and a complete player–GW matrix."""
    candidates = (
        player_table[
            [
                "id",
                "web_name",
                "team_id",
                "team_short_name",
                "position",
                "price",
                "status",
            ]
        ]
        .rename(columns={"id": "player_id"})
        .drop_duplicates("player_id")
        .copy()
    )
    candidates["position"] = optimiser_position(
        candidates["position"]
    )
    candidates["price"] = pd.to_numeric(
        candidates["price"],
        errors="coerce",
    )

    gameweek_xp = gameweek_xp.copy()

    # Older runs of the aggregation cell retained only the probability
    # of missing every fixture. Reconstruct the complementary Gameweek
    # appearance probability when necessary.
    if "appearance_probability" not in gameweek_xp.columns:
        if "no_appearance_probability" not in gameweek_xp.columns:
            raise KeyError(
                "Gameweek projections require either "
                "'appearance_probability' or "
                "'no_appearance_probability'."
            )
        gameweek_xp["appearance_probability"] = (
            1.0
            - pd.to_numeric(
                gameweek_xp["no_appearance_probability"],
                errors="coerce",
            ).fillna(1.0)
        ).clip(0.0, 1.0)

    if "no_appearance_probability" not in gameweek_xp.columns:
        gameweek_xp["no_appearance_probability"] = (
            1.0
            - pd.to_numeric(
                gameweek_xp["appearance_probability"],
                errors="coerce",
            ).fillna(0.0)
        ).clip(0.0, 1.0)

    required_projection_columns = {
        "player_id",
        "gameweek",
        "expected_points",
        "expected_minutes",
        "appearance_probability",
        "no_appearance_probability",
    }
    missing_projection_columns = (
        required_projection_columns - set(gameweek_xp.columns)
    )
    if missing_projection_columns:
        raise KeyError(
            "Gameweek projections are missing optimiser columns: "
            f"{sorted(missing_projection_columns)}"
        )

    projections = gameweek_xp.loc[
        gameweek_xp["gameweek"].isin(gameweeks_to_use),
        [
            "player_id",
            "gameweek",
            "expected_points",
            "expected_minutes",
            "appearance_probability",
            "no_appearance_probability",
        ],
    ].copy()

    full_index = pd.MultiIndex.from_product(
        [
            candidates["player_id"].tolist(),
            gameweeks_to_use,
        ],
        names=["player_id", "gameweek"],
    )

    projections = (
        projections
        .set_index(["player_id", "gameweek"])
        .reindex(full_index)
        .reset_index()
    )

    for column in [
        "expected_points",
        "expected_minutes",
        "appearance_probability",
    ]:
        projections[column] = pd.to_numeric(
            projections[column],
            errors="coerce",
        ).fillna(0.0)

    projections["no_appearance_probability"] = (
        pd.to_numeric(
            projections["no_appearance_probability"],
            errors="coerce",
        )
        .fillna(1.0)
        .clip(0.0, 1.0)
    )
    projections["appearance_probability"] = (
        projections["appearance_probability"]
        .clip(0.0, 1.0)
    )

    candidates = candidates.loc[
        candidates["position"].isin(
            SQUAD_POSITION_REQUIREMENTS
        )
        & candidates["price"].notna()
    ].reset_index(drop=True)

    projections = projections.loc[
        projections["player_id"].isin(
            candidates["player_id"]
        )
    ].copy()

    return candidates, projections


def optimise_squad_lineups_and_captains(
    candidates: pd.DataFrame,
    projections: pd.DataFrame,
    gameweeks_to_use: list[int],
    fixed_squad_player_ids: list[int] | None = None,
    budget: float = INITIAL_SQUAD_BUDGET,
    *,
    excluded_squad_sets: list[set[int]] | None = None,
    minimum_changes_from_excluded: int = 1,
    max_first_gameweek_bench_cost: float | None = None,
    squad_depth_weight: float | None = None,
) -> dict[str, object]:
    """Solve the squad, line-up and captain problem as one MILP."""
    candidates = candidates.reset_index(drop=True).copy()
    player_ids = candidates["player_id"].astype(int).tolist()
    player_index = {
        player_id: index
        for index, player_id in enumerate(player_ids)
    }

    n_players = len(candidates)
    n_gameweeks = len(gameweeks_to_use)

    projection_lookup = (
        projections
        .pivot(
            index="player_id",
            columns="gameweek",
            values="expected_points",
        )
        .reindex(
            index=player_ids,
            columns=gameweeks_to_use,
        )
        .fillna(0.0)
    )

    # Variable blocks: squad x, starter y, captain c.
    x_start = 0
    y_start = n_players
    c_start = n_players + n_players * n_gameweeks
    n_variables = n_players * (1 + 2 * n_gameweeks)

    def x_var(player_i: int) -> int:
        return x_start + player_i

    def y_var(player_i: int, gw_i: int) -> int:
        return y_start + gw_i * n_players + player_i

    def c_var(player_i: int, gw_i: int) -> int:
        return c_start + gw_i * n_players + player_i

    objective = np.zeros(n_variables, dtype=float)

    depth_weight = (
        float(SQUAD_DEPTH_WEIGHT)
        if squad_depth_weight is None
        else float(squad_depth_weight)
    )
    horizon_points = projection_lookup.sum(axis=1).to_numpy()
    for player_i in range(n_players):
        objective[x_var(player_i)] = (
            -depth_weight * horizon_points[player_i]
        )

    for gw_i, gameweek in enumerate(gameweeks_to_use):
        weight = GAMEWEEK_DISCOUNT ** gw_i
        gw_points = projection_lookup[gameweek].to_numpy()

        for player_i in range(n_players):
            objective[y_var(player_i, gw_i)] = (
                -weight * gw_points[player_i]
            )
            objective[c_var(player_i, gw_i)] = (
                -weight * gw_points[player_i]
            )

    lower_bounds = np.zeros(n_variables, dtype=float)
    upper_bounds = np.ones(n_variables, dtype=float)

    fixed_ids = {
        int(player_id)
        for player_id in (fixed_squad_player_ids or [])
    }
    if fixed_ids:
        if len(fixed_ids) != 15:
            raise ValueError(
                "CURRENT_SQUAD_PLAYER_IDS must contain exactly 15 unique IDs."
            )
        missing_ids = fixed_ids - set(player_ids)
        if missing_ids:
            raise ValueError(
                "Current squad IDs are absent from the player table: "
                f"{sorted(missing_ids)}"
            )

        for player_i, player_id in enumerate(player_ids):
            fixed_value = 1.0 if player_id in fixed_ids else 0.0
            lower_bounds[x_var(player_i)] = fixed_value
            upper_bounds[x_var(player_i)] = fixed_value

    row_indices: list[int] = []
    column_indices: list[int] = []
    coefficients: list[float] = []
    constraint_lower: list[float] = []
    constraint_upper: list[float] = []

    def add_constraint(
        terms: dict[int, float],
        lower: float = -np.inf,
        upper: float = np.inf,
    ) -> None:
        row = len(constraint_lower)
        for variable, coefficient in terms.items():
            row_indices.append(row)
            column_indices.append(variable)
            coefficients.append(float(coefficient))
        constraint_lower.append(float(lower))
        constraint_upper.append(float(upper))

    add_constraint(
        {x_var(i): 1.0 for i in range(n_players)},
        lower=15.0,
        upper=15.0,
    )

    for position, required in SQUAD_POSITION_REQUIREMENTS.items():
        mask = candidates["position"].eq(position).to_numpy()
        add_constraint(
            {
                x_var(i): 1.0
                for i in np.flatnonzero(mask)
            },
            lower=float(required),
            upper=float(required),
        )

    if not fixed_ids:
        add_constraint(
            {
                x_var(i): float(candidates.loc[i, "price"])
                for i in range(n_players)
            },
            upper=float(budget),
        )

    # Enumerating alternative opening squads uses no-good constraints. Each
    # new squad must differ from every earlier solution by at least the
    # configured number of players.
    excluded_squad_sets = excluded_squad_sets or []
    min_changes = max(1, int(minimum_changes_from_excluded))
    for excluded in excluded_squad_sets:
        indices = [
            player_index[int(pid)]
            for pid in excluded
            if int(pid) in player_index
        ]
        if indices:
            add_constraint(
                {x_var(i): 1.0 for i in indices},
                upper=float(15 - min_changes),
            )

    # Optional GW1 bench-cost cap for sensitivity/frontier analysis. Bench
    # cost is linear because bench membership is squad minus GW1 starter.
    if max_first_gameweek_bench_cost is not None and gameweeks_to_use:
        add_constraint(
            {
                **{
                    x_var(i): float(candidates.loc[i, "price"])
                    for i in range(n_players)
                },
                **{
                    y_var(i, 0): -float(candidates.loc[i, "price"])
                    for i in range(n_players)
                },
            },
            upper=float(max_first_gameweek_bench_cost),
        )

    for team_id in candidates["team_id"].dropna().unique():
        mask = candidates["team_id"].eq(team_id).to_numpy()
        add_constraint(
            {
                x_var(i): 1.0
                for i in np.flatnonzero(mask)
            },
            upper=float(MAX_PLAYERS_PER_CLUB),
        )

    for gw_i, _gameweek in enumerate(gameweeks_to_use):
        add_constraint(
            {
                y_var(i, gw_i): 1.0
                for i in range(n_players)
            },
            lower=11.0,
            upper=11.0,
        )

        for position, (minimum, maximum) in (
            LINEUP_POSITION_BOUNDS.items()
        ):
            mask = candidates["position"].eq(position).to_numpy()
            add_constraint(
                {
                    y_var(i, gw_i): 1.0
                    for i in np.flatnonzero(mask)
                },
                lower=float(minimum),
                upper=float(maximum),
            )

        add_constraint(
            {
                c_var(i, gw_i): 1.0
                for i in range(n_players)
            },
            lower=1.0,
            upper=1.0,
        )

        for player_i in range(n_players):
            add_constraint(
                {
                    y_var(player_i, gw_i): 1.0,
                    x_var(player_i): -1.0,
                },
                upper=0.0,
            )
            add_constraint(
                {
                    c_var(player_i, gw_i): 1.0,
                    y_var(player_i, gw_i): -1.0,
                },
                upper=0.0,
            )

    constraint_matrix = coo_matrix(
        (
            coefficients,
            (row_indices, column_indices),
        ),
        shape=(len(constraint_lower), n_variables),
    ).tocsr()

    result = milp(
        c=objective,
        integrality=np.ones(n_variables, dtype=int),
        bounds=Bounds(lower_bounds, upper_bounds),
        constraints=LinearConstraint(
            constraint_matrix,
            np.asarray(constraint_lower),
            np.asarray(constraint_upper),
        ),
        options={"time_limit": 120},
    )

    if not result.success or result.x is None:
        raise RuntimeError(
            "Squad optimisation failed: "
            f"{result.message}"
        )

    solution = result.x
    squad_mask = np.array([
        solution[x_var(i)] > 0.5
        for i in range(n_players)
    ])

    squad = candidates.loc[squad_mask].copy()
    squad["squad_cost"] = squad["price"]

    lineup_rows = []
    for gw_i, gameweek in enumerate(gameweeks_to_use):
        for player_i, player_id in enumerate(player_ids):
            if solution[y_var(player_i, gw_i)] <= 0.5:
                continue

            lineup_rows.append({
                "gameweek": gameweek,
                "player_id": player_id,
                "is_captain_milp": (
                    solution[c_var(player_i, gw_i)] > 0.5
                ),
            })

    lineups = (
        pd.DataFrame(lineup_rows)
        .merge(
            candidates,
            on="player_id",
            how="left",
            validate="many_to_one",
        )
        .merge(
            projections,
            on=["player_id", "gameweek"],
            how="left",
            validate="many_to_one",
        )
    )

    first_gameweek = gameweeks_to_use[0] if gameweeks_to_use else None
    if first_gameweek is not None:
        first_starter_ids = set(
            lineups.loc[lineups["gameweek"].eq(first_gameweek), "player_id"].astype(int)
        )
        first_bench_cost = float(
            squad.loc[~squad["player_id"].astype(int).isin(first_starter_ids), "price"].sum()
        )
    else:
        first_bench_cost = np.nan

    return {
        "solver_result": result,
        "squad": squad,
        "lineups": lineups,
        "projection_lookup": projection_lookup,
        "objective_value": float(-result.fun),
        "first_gameweek_bench_cost": first_bench_cost,
        "squad_cost": float(squad["price"].sum()),
    }


def generate_opening_squad_alternatives(
    candidates: pd.DataFrame,
    projections: pd.DataFrame,
    gameweeks_to_use: list[int],
    *,
    budget: float = INITIAL_SQUAD_BUDGET,
    count: int = OPENING_SQUAD_ALTERNATIVES,
    minimum_changes: int = OPENING_SQUAD_MIN_CHANGES,
    max_first_gameweek_bench_cost: float | None = None,
) -> tuple[list[dict[str, object]], pd.DataFrame]:
    """Enumerate materially different near-optimal opening squads."""
    results: list[dict[str, object]] = []
    excluded: list[set[int]] = []
    rows = []
    for rank in range(1, max(1, int(count)) + 1):
        try:
            result = optimise_squad_lineups_and_captains(
                candidates,
                projections,
                gameweeks_to_use,
                budget=budget,
                excluded_squad_sets=excluded,
                minimum_changes_from_excluded=minimum_changes,
                max_first_gameweek_bench_cost=max_first_gameweek_bench_cost,
            )
        except RuntimeError:
            break
        squad_ids = set(result["squad"]["player_id"].astype(int))
        excluded.append(squad_ids)
        results.append(result)
        rows.append({
            "rank": rank,
            "objective": float(result["objective_value"]),
            "squad_cost": float(result["squad_cost"]),
            "bank": float(budget - result["squad_cost"]),
            "gw1_bench_cost": float(result["first_gameweek_bench_cost"]),
            "squad": ", ".join(
                result["squad"].sort_values(["position", "price"], ascending=[True, False])["web_name"].astype(str)
            ),
        })
    table = pd.DataFrame(rows)
    if not table.empty:
        table["objective_gap_to_best"] = table["objective"].max() - table["objective"]
    return results, table


def opening_bench_sensitivity_frontier(
    candidates: pd.DataFrame,
    projections: pd.DataFrame,
    gameweeks_to_use: list[int],
    *,
    budget: float = INITIAL_SQUAD_BUDGET,
) -> tuple[list[dict[str, object]], pd.DataFrame]:
    """Solve GW1 under several bench-cost caps and expose the efficiency frontier."""
    results = []
    rows = []
    seen = set()
    for cap in OPENING_BENCH_COST_CAPS:
        try:
            result = optimise_squad_lineups_and_captains(
                candidates,
                projections,
                gameweeks_to_use,
                budget=budget,
                max_first_gameweek_bench_cost=cap,
            )
        except RuntimeError:
            continue
        key = tuple(sorted(result["squad"]["player_id"].astype(int)))
        if key in seen:
            continue
        seen.add(key)
        result["bench_cap"] = cap
        results.append(result)
        rows.append({
            "bench_cap": cap if cap is not None else np.inf,
            "objective": float(result["objective_value"]),
            "squad_cost": float(result["squad_cost"]),
            "bank": float(budget - result["squad_cost"]),
            "gw1_bench_cost": float(result["first_gameweek_bench_cost"]),
            "squad": ", ".join(result["squad"]["web_name"].astype(str)),
        })
    table = pd.DataFrame(rows)
    if not table.empty:
        best = float(table["objective"].max())
        table["objective_gap_to_best"] = best - table["objective"]
        table = table.sort_values(["objective_gap_to_best", "gw1_bench_cost"]).reset_index(drop=True)
    return results, table


def choose_opening_squad_solution(
    candidates: pd.DataFrame,
    projections: pd.DataFrame,
    gameweeks_to_use: list[int],
    *,
    budget: float = INITIAL_SQUAD_BUDGET,
) -> tuple[dict[str, object], pd.DataFrame, pd.DataFrame]:
    """Return the autonomous opening squad plus uncertainty/bench audits."""
    alternatives, alternative_table = generate_opening_squad_alternatives(
        candidates,
        projections,
        gameweeks_to_use,
        budget=budget,
    )
    if not alternatives:
        raise RuntimeError("No legal opening-squad solution was found.")
    chosen = alternatives[0]

    bench_table = pd.DataFrame()
    if RUN_OPENING_BENCH_SENSITIVITY:
        bench_results, bench_table = opening_bench_sensitivity_frontier(
            candidates,
            projections,
            gameweeks_to_use,
            budget=budget,
        )
        if OPENING_USE_BENCH_EFFICIENT_FRONTIER and bench_results:
            best_objective = max(float(r["objective_value"]) for r in bench_results)
            eligible = [
                r for r in bench_results
                if best_objective - float(r["objective_value"])
                <= float(OPENING_MAX_OBJECTIVE_LOSS_FOR_CHEAPER_BENCH) + 1e-12
            ]
            if eligible:
                chosen = sorted(
                    eligible,
                    key=lambda r: (
                        float(r["first_gameweek_bench_cost"]),
                        -float(r["objective_value"]),
                        float(r["squad_cost"]),
                    ),
                )[0]
    return chosen, alternative_table, bench_table


In [ ]:
def legal_partial_outfield_formation(
    position_counts: dict[str, int],
) -> bool:
    """
    Check a nominal ten-outfield-player FPL formation.

    Kept under the historical public name for notebook compatibility. v2.2 no
    longer tests the reduced set of players who actually appeared; formation
    legality is evaluated on the nominal XI after each legal substitution.
    """
    total = sum(int(position_counts.get(p, 0)) for p in ("DEF", "MID", "FWD"))
    return (
        total == 10
        and 3 <= int(position_counts.get("DEF", 0)) <= 5
        and 2 <= int(position_counts.get("MID", 0)) <= 5
        and 1 <= int(position_counts.get("FWD", 0)) <= 3
    )


def apply_outfield_auto_substitutions(
    appearing_starters: list[dict],
    appearing_bench_in_order: list[dict],
    nominal_starters: list[dict] | None = None,
) -> list[dict]:
    """
    Apply FPL outfield auto-subs without treating unrelated no-shows as an
    illegal formation.

    Formation legality is based on the nominal starting XI. For each appearing
    bench player, we replace one still-absent nominal starter if at least one
    such replacement keeps a legal full formation. A short look-ahead resolves
    rare ambiguous slot assignments while preserving bench priority.
    """
    active = list(appearing_starters)
    if nominal_starters is None:
        raise ValueError(
            "nominal_starters is required for FPL autosub evaluation. "
            "Inferring the starting formation from appearing players is unsafe "
            "because no-shows remove the information needed to reconstruct the "
            "original XI."
        )

    nominal_counts = {"DEF": 0, "MID": 0, "FWD": 0}
    active_counts = {"DEF": 0, "MID": 0, "FWD": 0}
    for player in nominal_starters:
        if player["position"] in nominal_counts:
            nominal_counts[player["position"]] += 1
    for player in appearing_starters:
        if player["position"] in active_counts:
            active_counts[player["position"]] += 1
        else:
            raise ValueError(
                "Appearing starter has unsupported outfield position: "
                f"{player.get('position')!r}"
            )

    if not legal_partial_outfield_formation(nominal_counts):
        raise ValueError(
            f"Nominal starter formation is not legal: {nominal_counts}"
        )

    if any(
        active_counts[position] > nominal_counts[position]
        for position in nominal_counts
    ):
        raise ValueError(
            "Appearing starters are inconsistent with the nominal XI: "
            f"active={active_counts}, nominal={nominal_counts}"
        )

    absent_counts = {
        pos: nominal_counts[pos] - active_counts[pos]
        for pos in nominal_counts
    }

    bench = list(appearing_bench_in_order)
    for player in bench:
        if str(player.get("position")) not in nominal_counts:
            raise ValueError(
                "Appearing bench player has unsupported outfield position: "
                f"{player.get('position')!r}"
            )

    from functools import lru_cache

    @lru_cache(maxsize=None)
    def solve(
        bench_index: int,
        def_count: int,
        mid_count: int,
        fwd_count: int,
        absent_def: int,
        absent_mid: int,
        absent_fwd: int,
    ):
        if bench_index >= len(bench):
            return ()

        formation = {
            "DEF": def_count,
            "MID": mid_count,
            "FWD": fwd_count,
        }
        absent = {
            "DEF": absent_def,
            "MID": absent_mid,
            "FWD": absent_fwd,
        }
        player = bench[bench_index]
        bench_pos = str(player["position"])

        legal_replacements = []
        for outgoing_pos in ("DEF", "MID", "FWD"):
            if absent[outgoing_pos] <= 0:
                continue
            new_counts = dict(formation)
            new_counts[outgoing_pos] -= 1
            new_counts[bench_pos] += 1
            if legal_partial_outfield_formation(new_counts):
                new_absent = dict(absent)
                new_absent[outgoing_pos] -= 1
                suffix = solve(
                    bench_index + 1,
                    new_counts["DEF"],
                    new_counts["MID"],
                    new_counts["FWD"],
                    new_absent["DEF"],
                    new_absent["MID"],
                    new_absent["FWD"],
                )
                legal_replacements.append(
                    (int(player["player_id"]),) + tuple(suffix)
                )

        # FPL bench priority means an appearing bench player comes on whenever
        # a legal substitution exists. Only skip if no legal replacement exists.
        if legal_replacements:
            return max(
                legal_replacements,
                key=lambda ids: (len(ids), tuple(-bench_index for _ in ids)),
            )

        return solve(
            bench_index + 1,
            def_count,
            mid_count,
            fwd_count,
            absent_def,
            absent_mid,
            absent_fwd,
        )

    selected_ids = set(
        solve(
            0,
            nominal_counts["DEF"],
            nominal_counts["MID"],
            nominal_counts["FWD"],
            absent_counts["DEF"],
            absent_counts["MID"],
            absent_counts["FWD"],
        )
    )

    selected = [
        player
        for player in bench
        if int(player["player_id"]) in selected_ids
    ]
    return active + selected


def _poisson_binomial_count_distribution(probabilities: list[float]) -> np.ndarray:
    """Probability distribution for the number of independent appearances."""
    distribution = np.array([1.0], dtype=float)
    for probability in probabilities:
        p = float(np.clip(probability, 0.0, 1.0))
        updated = np.zeros(len(distribution) + 1, dtype=float)
        updated[:-1] += distribution * (1.0 - p)
        updated[1:] += distribution * p
        distribution = updated
    return distribution


def expected_score_with_bench_order(
    squad_gameweek: pd.DataFrame,
    starter_ids: set[int],
    outfield_bench_order: tuple[int, ...],
) -> float:
    """
    Exact expected score under independent appearances and FPL auto-subs.

    v2.2 keeps the efficient position-count dynamic program, but formation
    legality is evaluated against the *nominal* starting formation rather than
    the subset of starters who happened to appear.
    """
    data = squad_gameweek.copy()
    data["player_id"] = data["player_id"].astype(int)
    data["position"] = optimiser_position(data["position"])
    data["appearance_probability"] = (
        pd.to_numeric(data["appearance_probability"], errors="coerce")
        .fillna(0.0)
        .clip(0.0, 1.0)
    )
    data["expected_points"] = pd.to_numeric(
        data["expected_points"], errors="coerce"
    ).fillna(0.0)

    rows = data.set_index("player_id").to_dict("index")
    starter_ids = {int(player_id) for player_id in starter_ids}
    bench_order = tuple(int(player_id) for player_id in outfield_bench_order)

    starting_gk = next(
        player_id for player_id in starter_ids
        if rows[player_id]["position"] == "GK"
    )
    bench_gk = next(
        int(player_id) for player_id in data["player_id"]
        if int(player_id) not in starter_ids
        and rows[int(player_id)]["position"] == "GK"
    )

    expected_score = float(sum(
        float(rows[player_id]["expected_points"])
        for player_id in starter_ids
    ))

    start_gk_p = float(rows[starting_gk]["appearance_probability"])
    bench_gk_p = float(rows[bench_gk]["appearance_probability"])
    bench_gk_conditional_xp = (
        float(rows[bench_gk]["expected_points"]) / bench_gk_p
        if bench_gk_p > 1e-12 else 0.0
    )
    expected_score += (
        (1.0 - start_gk_p)
        * bench_gk_p
        * bench_gk_conditional_xp
    )

    nominal_outfield = [
        {
            "player_id": int(player_id),
            "position": str(rows[player_id]["position"]),
        }
        for player_id in starter_ids
        if rows[player_id]["position"] != "GK"
    ]

    position_distributions: dict[str, np.ndarray] = {}
    nominal_ids_by_position: dict[str, list[int]] = {}
    for position in ("DEF", "MID", "FWD"):
        ids = [
            int(player_id)
            for player_id in starter_ids
            if rows[player_id]["position"] == position
        ]
        nominal_ids_by_position[position] = ids
        probabilities = [
            float(rows[player_id]["appearance_probability"])
            for player_id in ids
        ]
        position_distributions[position] = (
            _poisson_binomial_count_distribution(probabilities)
        )

    bench_rows = []
    for player_id in bench_order:
        probability = float(rows[player_id]["appearance_probability"])
        bench_rows.append({
            "player_id": player_id,
            "position": str(rows[player_id]["position"]),
            "appearance_probability": probability,
            "conditional_xp": (
                float(rows[player_id]["expected_points"]) / probability
                if probability > 1e-12 else 0.0
            ),
        })

    bench_patterns = []
    for bits in product([False, True], repeat=len(bench_rows)):
        probability = 1.0
        appearing_bench = []
        for row, appears in zip(bench_rows, bits):
            p = row["appearance_probability"]
            probability *= p if appears else 1.0 - p
            if appears:
                appearing_bench.append({
                    "player_id": row["player_id"],
                    "position": row["position"],
                })
        if probability > 1e-15:
            bench_patterns.append((probability, appearing_bench))

    def_dist = position_distributions["DEF"]
    mid_dist = position_distributions["MID"]
    fwd_dist = position_distributions["FWD"]

    autosub_expectation = 0.0
    for n_def, p_def in enumerate(def_dist):
        if p_def <= 0.0:
            continue
        for n_mid, p_mid in enumerate(mid_dist):
            if p_mid <= 0.0:
                continue
            for n_fwd, p_fwd in enumerate(fwd_dist):
                starter_state_probability = float(p_def * p_mid * p_fwd)
                if starter_state_probability <= 1e-15:
                    continue

                appearing_starters = (
                    [{"player_id": -(100 + i), "position": "DEF"} for i in range(n_def)]
                    + [{"player_id": -(200 + i), "position": "MID"} for i in range(n_mid)]
                    + [{"player_id": -(300 + i), "position": "FWD"} for i in range(n_fwd)]
                )

                for bench_probability, appearing_bench in bench_patterns:
                    scenario_probability = starter_state_probability * bench_probability
                    if scenario_probability <= 1e-15:
                        continue
                    final_outfield = apply_outfield_auto_substitutions(
                        appearing_starters,
                        appearing_bench,
                        nominal_starters=nominal_outfield,
                    )
                    selected_bench_ids = {
                        int(player["player_id"])
                        for player in final_outfield
                        if int(player["player_id"]) > 0
                    }
                    added_score = sum(
                        float(row["conditional_xp"])
                        for row in bench_rows
                        if row["player_id"] in selected_bench_ids
                    )
                    autosub_expectation += scenario_probability * added_score

    return float(expected_score + autosub_expectation)

def select_production_captain_pair(starters) -> dict[str, object]:
    """Select captain/vice to maximise expected captain uplift; optional xP-first mode remains configurable."""
    frame = starters.copy() if isinstance(starters, pd.DataFrame) else pd.DataFrame(list(starters))
    if len(frame) < 2:
        raise ValueError("Captain selection requires at least two starters.")

    required = [
        "player_id",
        "expected_points",
        "expected_minutes",
        "appearance_probability",
        "no_appearance_probability",
    ]
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise KeyError(f"Captain selection missing columns: {missing}")

    for column, fill_value in {
        "expected_points": 0.0,
        "expected_minutes": 0.0,
        "appearance_probability": 0.0,
        "no_appearance_probability": 1.0,
    }.items():
        frame[column] = pd.to_numeric(frame[column], errors="coerce").fillna(fill_value)

    if PRODUCTION_CAPTAIN_HIGHEST_XP:
        ranked = frame.sort_values(
            ["expected_points", "appearance_probability", "expected_minutes"],
            ascending=[False, False, False],
        ).reset_index(drop=True)
        captain = ranked.iloc[0]
        vice = (
            ranked.loc[~ranked["player_id"].astype(int).eq(int(captain["player_id"]))]
            .sort_values(
                ["expected_points", "appearance_probability", "expected_minutes"],
                ascending=[False, False, False],
            )
            .iloc[0]
        )
    else:
        # Legacy v2.1 joint captain/vice objective retained as an explicit fallback.
        options = []
        for captain_row in frame.itertuples(index=False):
            for vice_row in frame.itertuples(index=False):
                if int(captain_row.player_id) == int(vice_row.player_id):
                    continue
                uplift = (
                    float(captain_row.expected_points)
                    + float(captain_row.no_appearance_probability)
                    * float(vice_row.expected_points)
                )
                options.append((uplift, captain_row, vice_row))
        _, captain_row, vice_row = max(options, key=lambda row: row[0])
        captain = pd.Series(captain_row._asdict())
        vice = pd.Series(vice_row._asdict())

    captain_extra = float(captain["expected_points"])
    vice_fallback_extra = (
        float(captain["no_appearance_probability"])
        * float(vice["expected_points"])
    )
    return {
        "captain_id": int(captain["player_id"]),
        "vice_captain_id": int(vice["player_id"]),
        "captain_extra": captain_extra,
        "vice_fallback_extra": vice_fallback_extra,
        "expected_captain_uplift": captain_extra + vice_fallback_extra,
    }


def optimise_current_gameweek_details(
    selected_squad: pd.DataFrame,
    projected_lineup: pd.DataFrame,
    gameweek_projection: pd.DataFrame,
) -> dict[str, object]:
    """Choose bench order and the captain/vice-captain pair."""
    squad_projection = (
        selected_squad[
            [
                "player_id",
                "web_name",
                "team_short_name",
                "position",
                "price",
            ]
        ]
        .merge(
            gameweek_projection,
            on="player_id",
            how="left",
            validate="one_to_one",
        )
    )

    for column, fill_value in {
        "expected_points": 0.0,
        "expected_minutes": 0.0,
        "appearance_probability": 0.0,
        "no_appearance_probability": 1.0,
    }.items():
        squad_projection[column] = pd.to_numeric(
            squad_projection[column],
            errors="coerce",
        ).fillna(fill_value)

    starter_ids = set(
        projected_lineup["player_id"].astype(int)
    )
    squad_projection["is_starter"] = (
        squad_projection["player_id"].isin(starter_ids)
    )
    squad_projection["position"] = optimiser_position(
        squad_projection["position"]
    )

    outfield_bench_ids = tuple(
        squad_projection.loc[
            ~squad_projection["is_starter"]
            & ~squad_projection["position"].eq("GK"),
            "player_id",
        ].astype(int)
    )

    bench_order_results = []
    for bench_order in permutations(outfield_bench_ids):
        expected_base_score = expected_score_with_bench_order(
            squad_projection,
            starter_ids,
            bench_order,
        )
        bench_order_results.append({
            "bench_order": bench_order,
            "expected_base_score": expected_base_score,
        })

    best_bench = max(
        bench_order_results,
        key=lambda row: row["expected_base_score"],
    )

    starters = squad_projection.loc[
        squad_projection["is_starter"]
    ].copy()

    best_captain_pair = select_production_captain_pair(starters)

    return {
        "squad_projection": squad_projection,
        "starter_ids": starter_ids,
        "bench_order": best_bench["bench_order"],
        "expected_base_score": best_bench["expected_base_score"],
        **best_captain_pair,
        "expected_total_score": (
            best_bench["expected_base_score"]
            + best_captain_pair["expected_captain_uplift"]
        ),
    }

### v2.2 autosub regression tests

These tests cover the edge case found in the audit: one unrelated starter can remain unreplaced without blocking an otherwise legal bench substitution.

In [ ]:
def _autosub_test_player(pid: int, position: str) -> dict:
    return {"player_id": int(pid), "position": str(position)}

_nominal_343 = (
    [_autosub_test_player(10+i, "DEF") for i in range(3)]
    + [_autosub_test_player(20+i, "MID") for i in range(4)]
    + [_autosub_test_player(30+i, "FWD") for i in range(3)]
)

# Missing one DEF and one MID. A bench MID must still be allowed to replace the
# missing MID even if the absent defender has no replacement.
_appearing = (
    [_autosub_test_player(-(100+i), "DEF") for i in range(2)]
    + [_autosub_test_player(-(200+i), "MID") for i in range(3)]
    + [_autosub_test_player(-(300+i), "FWD") for i in range(3)]
)
_result = apply_outfield_auto_substitutions(
    _appearing,
    [_autosub_test_player(901, "MID")],
    nominal_starters=_nominal_343,
)
assert 901 in {int(p["player_id"]) for p in _result}

# A fourth forward cannot replace either of those missing slots in a nominal
# 3-4-3 because 3-3-4 and 2-4-4 are both illegal.
_result = apply_outfield_auto_substitutions(
    _appearing,
    [_autosub_test_player(902, "FWD")],
    nominal_starters=_nominal_343,
)
assert 902 not in {int(p["player_id"]) for p in _result}

# With a DEF and then a MID available from the bench, both no-shows can be
# legally covered.
_result = apply_outfield_auto_substitutions(
    _appearing,
    [
        _autosub_test_player(903, "DEF"),
        _autosub_test_player(904, "MID"),
    ],
    nominal_starters=_nominal_343,
)
assert {903, 904}.issubset(
    {int(p["player_id"]) for p in _result}
)

print("v2.2 autosub regression tests: PASS")

In [ ]:
optimisation_gameweeks = sorted(
    live_gameweek_expected_points["gameweek"].unique()
)[:PLANNING_HORIZON]

optimisation_candidates, optimisation_projections = (
    prepare_optimisation_inputs(
        live_gameweek_expected_points,
        players,
        optimisation_gameweeks,
    )
)

opening_squad_alternatives = pd.DataFrame()
opening_bench_sensitivity = pd.DataFrame()
if CURRENT_SQUAD_PLAYER_IDS:
    optimisation_result = optimise_squad_lineups_and_captains(
        optimisation_candidates,
        optimisation_projections,
        optimisation_gameweeks,
        fixed_squad_player_ids=CURRENT_SQUAD_PLAYER_IDS,
        budget=INITIAL_SQUAD_BUDGET,
    )
else:
    (
        optimisation_result,
        opening_squad_alternatives,
        opening_bench_sensitivity,
    ) = choose_opening_squad_solution(
        optimisation_candidates,
        optimisation_projections,
        optimisation_gameweeks,
        budget=INITIAL_SQUAD_BUDGET,
    )

optimised_squad = optimisation_result["squad"].copy()
optimised_projected_lineups = optimisation_result[
    "lineups"
].copy()

current_optimisation_gameweek = optimisation_gameweeks[0]
current_projected_lineup = (
    optimised_projected_lineups.loc[
        optimised_projected_lineups["gameweek"].eq(
            current_optimisation_gameweek
        )
    ]
    .copy()
)

current_gameweek_projection = (
    live_gameweek_expected_points.loc[
        live_gameweek_expected_points["gameweek"].eq(
            current_optimisation_gameweek
        ),
        [
            "player_id",
            "expected_points",
            "expected_minutes",
            "appearance_probability",
            "no_appearance_probability",
        ],
    ]
    .copy()
)

current_decision = optimise_current_gameweek_details(
    optimised_squad,
    current_projected_lineup,
    current_gameweek_projection,
)

captain_id = current_decision["captain_id"]
vice_captain_id = current_decision["vice_captain_id"]
starter_ids = current_decision["starter_ids"]
bench_order = current_decision["bench_order"]

current_squad_recommendation = (
    current_decision["squad_projection"]
    .copy()
)
current_squad_recommendation["role"] = "Starter"
current_squad_recommendation.loc[
    current_squad_recommendation["player_id"].eq(captain_id),
    "role",
] = "Captain"
current_squad_recommendation.loc[
    current_squad_recommendation["player_id"].eq(vice_captain_id),
    "role",
] = "Vice-captain"

bench_gk_mask = (
    ~current_squad_recommendation["is_starter"]
    & current_squad_recommendation["position"].eq("GK")
)
current_squad_recommendation.loc[
    bench_gk_mask,
    "role",
] = "Bench goalkeeper"

for bench_rank, player_id in enumerate(bench_order, start=1):
    current_squad_recommendation.loc[
        current_squad_recommendation["player_id"].eq(player_id),
        "role",
    ] = f"Bench {bench_rank}"

role_order = {
    "Captain": 0,
    "Vice-captain": 1,
    "Starter": 2,
    "Bench 1": 3,
    "Bench 2": 4,
    "Bench 3": 5,
    "Bench goalkeeper": 6,
}
current_squad_recommendation["role_order"] = (
    current_squad_recommendation["role"].map(role_order)
)

current_squad_recommendation = (
    current_squad_recommendation
    .sort_values(
        ["role_order", "position", "expected_points"],
        ascending=[True, True, False],
    )
    .drop(columns="role_order")
)

assert len(optimised_squad) == 15
assert np.isclose(
    optimised_squad["price"].sum(),
    optimised_squad["squad_cost"].sum(),
)
if not CURRENT_SQUAD_PLAYER_IDS:
    assert optimised_squad["price"].sum() <= (
        INITIAL_SQUAD_BUDGET + 1e-9
    )
assert (
    optimised_squad.groupby("team_id").size().max()
    <= MAX_PLAYERS_PER_CLUB
)

print(
    "Mode:",
    (
        "Initial squad optimisation"
        if not CURRENT_SQUAD_PLAYER_IDS
        else "Fixed current-squad line-up optimisation"
    ),
)
print(
    f"Squad cost: £{optimised_squad['price'].sum():.1f}m"
)
print(
    "Expected current-Gameweek score including captain and "
    f"automatic substitutions: "
    f"{current_decision['expected_total_score']:.2f}"
)

display(
    current_squad_recommendation[
        [
            "role",
            "web_name",
            "team_short_name",
            "position",
            "price",
            "expected_minutes",
            "appearance_probability",
            "expected_points",
        ]
    ]
)

projected_lineup_summary = (
    optimised_projected_lineups
    .groupby("gameweek", as_index=False)
    .agg(starting_xp=("expected_points", "sum"))
)

captain_rows = optimised_projected_lineups.loc[
    optimised_projected_lineups["is_captain_milp"]
]
captain_by_gameweek = (
    captain_rows.set_index("gameweek")["expected_points"]
)

projected_lineup_summary["captain_xp"] = (
    projected_lineup_summary["gameweek"]
    .map(captain_by_gameweek)
    .fillna(0.0)
)
projected_lineup_summary["projected_score"] = (
    projected_lineup_summary["starting_xp"]
    + projected_lineup_summary["captain_xp"]
)

display(projected_lineup_summary)


if not CURRENT_SQUAD_PLAYER_IDS:
    print("Alternative opening squads — materially different solutions")
    display(opening_squad_alternatives.round(3))
    if RUN_OPENING_BENCH_SENSITIVITY:
        print("Opening-squad bench-cost sensitivity")
        display(opening_bench_sensitivity.replace({np.inf: "unconstrained"}).round(3))


### Result of the optimisation stage

The principal outputs are:

- `optimised_squad`: the selected initial squad, or the fixed current squad;
- `optimised_projected_lineups`: one legal projected XI and captain for each Gameweek in the planning horizon;
- `current_squad_recommendation`: the current Gameweek XI, captain, vice-captain and bench order;
- `current_decision["expected_total_score"]`: expected current-Gameweek score including the captain fallback and automatic substitutions.

This is still a **no-transfer** decision layer. The next stage will take the current squad state, free transfers, money in the bank and selling prices, then compare rolling a transfer with one-transfer, multi-transfer and hit-taking paths.

## 9. Rolling-horizon transfer planning

This stage extends the fixed-squad optimiser into a sequential transfer planner.

For every Gameweek in the planning horizon, the search compares:

- making no transfer and banking another free transfer;
- using one or more free transfers;
- taking a four-point hit when the projected gain justifies it;
- the effect of the resulting squad on later Gameweeks.

The search uses the same receding-horizon principle as the rest of the project: it recommends only the first Gameweek's action, then the complete analysis is rerun before the following deadline.

A full exhaustive search is impractical. The implementation therefore uses a **beam search**:

1. generate legal transfer actions from each retained squad state;
2. score the resulting current-Gameweek squad;
3. add a no-more-transfers estimate for the remaining horizon;
4. retain only the strongest distinct states;
5. repeat for the next projected Gameweek.

The first baseline assumes that current player prices remain fixed across the planning horizon. Enter the current selling price shown by FPL for each owned player when it differs from market price. Forecasting future price changes is deliberately deferred to a later stage.


In [ ]:
# Transfer-state inputs -----------------------------------------------------

# During the season, CURRENT_SQUAD_PLAYER_IDS from the previous section should
# contain the actual 15-player squad. Before Gameweek 1, the optimised initial
# squad is used only as a demonstration state; transfers are unlimited before
# the opening deadline, so its transfer recommendation should not be acted on.

CURRENT_MONEY_IN_BANK = 0.0
CURRENT_FREE_TRANSFERS = 1

# Optional current FPL selling prices, in millions. Any omitted player defaults
# to the player's current market price. Example: {123: 6.4, 456: 7.1}
CURRENT_SQUAD_SELLING_PRICES: dict[int, float] = {}

TRANSFER_HIT_COST = 4.0
MAX_BANKED_FREE_TRANSFERS = 5
MAX_TRANSFERS_PER_GAMEWEEK = 2
# Ordinary search remains capped at two moves for tractability. v2.2 can open
# a deeper repair branch only when the current squad contains several assets
# with virtually zero projected availability.
EMERGENCY_STRUCTURAL_REPAIR_ENABLED = True
EMERGENCY_MAX_TRANSFERS_PER_GAMEWEEK = 5
EMERGENCY_UNAVAILABLE_MIN_COUNT = 3
EMERGENCY_UNAVAILABLE_MAX_HORIZON_MINUTES = 5.0
ALLOW_TRANSFER_HITS = True

# Search controls. Increasing these can improve recall but increases runtime.
TRANSFER_BEAM_WIDTH = 25
TRANSFER_ACTION_BEAM_WIDTH = 35
TRANSFER_TOP_XP_PER_POSITION = 12
TRANSFER_TOP_VALUE_PER_POSITION = 6
TRANSFER_MAX_INCOMING_PER_POSITION = 8

# v2.2.5 adaptive production search.
#
# The old v2.2.4 widths are retained as the explicit "reference" profile.
# Weekly production starts with a compact fast search and only widens when
# distinct first actions are too close to call. Reference-width search is
# benchmark-only by default so a normal weekly run cannot silently regress
# to the ~20+ minute search observed in v2.2.4.
TRANSFER_ADAPTIVE_ENABLED = True
TRANSFER_ADAPTIVE_MIN_FIRST_ACTION_GAP = 0.75
TRANSFER_ADAPTIVE_MIN_DISTINCT_FIRST_ACTIONS = 2
TRANSFER_ADAPTIVE_ALLOW_REFERENCE_ESCALATION = False
TRANSFER_ADAPTIVE_BACKTEST_ENABLED = True

TRANSFER_SEARCH_PROFILES = {
    "fast": {
        "beam_width": 8,
        "action_beam_width": 12,
        "top_xp_per_position": 8,
        "top_value_per_position": 4,
        "max_incoming_per_position": 5,
        "max_swap_candidates_per_partial": 20,
        "protected_first_actions_per_transfer_count": 2,
    },
    "standard": {
        "beam_width": 14,
        "action_beam_width": 20,
        "top_xp_per_position": 10,
        "top_value_per_position": 5,
        "max_incoming_per_position": 6,
        "max_swap_candidates_per_partial": 28,
        "protected_first_actions_per_transfer_count": 2,
    },
    "reference": {
        "beam_width": 25,
        "action_beam_width": 35,
        "top_xp_per_position": 12,
        "top_value_per_position": 6,
        "max_incoming_per_position": 8,
        "max_swap_candidates_per_partial": 48,
        "protected_first_actions_per_transfer_count": 3,
    },
}


# A modest end-of-horizon value reduces the incentive to finish with a squad
# that is useful only for the final explicitly modelled fixture.
TRANSFER_TERMINAL_VALUE_WEIGHT = 0.35

TRANSFER_SEARCH_SIGNATURE = stable_configuration_signature({
    "adaptive_enabled": TRANSFER_ADAPTIVE_ENABLED,
    "profiles": TRANSFER_SEARCH_PROFILES,
    "min_first_action_gap": TRANSFER_ADAPTIVE_MIN_FIRST_ACTION_GAP,
    "min_distinct_first_actions": TRANSFER_ADAPTIVE_MIN_DISTINCT_FIRST_ACTIONS,
    "reference_escalation": TRANSFER_ADAPTIVE_ALLOW_REFERENCE_ESCALATION,
    "dominance_pruning": TRANSFER_DOMINANCE_PRUNING,
    "terminal_value_weight": TRANSFER_TERMINAL_VALUE_WEIGHT,
})


def price_to_tenths(price: float) -> int:
    """Convert an FPL price in millions to exact integer tenths."""
    return int(round(float(price) * 10))


def tenths_to_price(price_tenths: int) -> float:
    """Convert integer tenths back to millions."""
    return float(price_tenths) / 10.0


def update_free_transfers(
    free_transfers_before: int,
    transfers_made: int,
) -> int:
    """Apply the standard post-deadline free-transfer update."""
    unused = max(0, int(free_transfers_before) - int(transfers_made))
    return min(MAX_BANKED_FREE_TRANSFERS, unused + 1)


def transfer_hit_cost(
    free_transfers_before: int,
    transfers_made: int,
) -> float:
    """Return the points deduction for transfers beyond the free allowance."""
    paid_transfers = max(0, int(transfers_made) - int(free_transfers_before))
    return float(TRANSFER_HIT_COST * paid_transfers)


### Candidate pool and fast squad scoring

The transfer search always retains the current squad. Possible incoming players are filtered by position using both total horizon xP and xP per million. This prevents the search from spending most of its time on dominated options while still retaining high-value budget choices.

For path search, a fast legal line-up evaluator is used. It checks every valid FPL formation, chooses the highest-xP players within each position, and selects captain and vice-captain jointly. The final current-Gameweek recommendation is still passed through the more detailed automatic-substitution evaluator from the previous stage.


In [ ]:
def build_transfer_candidate_pool(
    candidates: pd.DataFrame,
    projections: pd.DataFrame,
    gameweeks_to_use: list[int],
    current_squad_ids: list[int],
) -> pd.DataFrame:
    """
    Retain strong xP/value/owned options, then prune only clearly redundant
    same-club/same-position candidates.

    Same-club pruning preserves at least the top three options in each
    club-position group, so it does not remove the ability to stack players
    under the three-per-club rule.
    """
    weights = {
        gameweek: GAMEWEEK_DISCOUNT ** index
        for index, gameweek in enumerate(gameweeks_to_use)
    }

    horizon = projections.copy()
    horizon["discount_weight"] = horizon["gameweek"].map(weights).fillna(0.0)
    horizon["weighted_xp"] = (
        horizon["expected_points"] * horizon["discount_weight"]
    )
    horizon_summary = (
        horizon.groupby("player_id", as_index=False)
        .agg(
            horizon_xp=("expected_points", "sum"),
            weighted_horizon_xp=("weighted_xp", "sum"),
            horizon_minutes=("expected_minutes", "sum"),
            horizon_appearance=(
                "appearance_probability",
                "mean",
            ),
        )
    )

    pool = candidates.merge(
        horizon_summary,
        on="player_id",
        how="left",
        validate="one_to_one",
    )
    for column in [
        "horizon_xp",
        "weighted_horizon_xp",
        "horizon_minutes",
        "horizon_appearance",
    ]:
        pool[column] = pd.to_numeric(
            pool[column], errors="coerce"
        ).fillna(0.0)

    pool["xp_per_million"] = np.where(
        pool["price"].gt(0),
        pool["weighted_horizon_xp"] / pool["price"],
        0.0,
    )

    selected_ids = set(int(player_id) for player_id in current_squad_ids)
    for _, position_group in pool.groupby("position"):
        selected_ids.update(
            position_group.nlargest(
                TRANSFER_TOP_XP_PER_POSITION,
                "weighted_horizon_xp",
            )["player_id"].astype(int)
        )
        selected_ids.update(
            position_group.nlargest(
                TRANSFER_TOP_VALUE_PER_POSITION,
                "xp_per_million",
            )["player_id"].astype(int)
        )

    result = (
        pool.loc[pool["player_id"].isin(selected_ids)]
        .drop_duplicates("player_id")
        .copy()
    )

    if TRANSFER_DOMINANCE_PRUNING and not result.empty:
        keep = set(int(v) for v in current_squad_ids)
        for (_team, _position), group in result.groupby(
            ["team_id", "position"], dropna=False
        ):
            group = group.sort_values(
                ["weighted_horizon_xp", "horizon_minutes", "price"],
                ascending=[False, False, True],
            )
            # Never prune the top three from the same club/position because a
            # legal squad can in principle contain all three.
            keep.update(group.head(3)["player_id"].astype(int).tolist())

            rows = group.to_dict("records")
            for row in rows:
                pid = int(row["player_id"])
                if pid in keep:
                    continue
                dominated = any(
                    (
                        int(other["player_id"]) != pid
                        and float(other["price"]) <= float(row["price"]) + 1e-9
                        and float(other["weighted_horizon_xp"])
                            >= float(row["weighted_horizon_xp"]) - 1e-9
                        and float(other["horizon_minutes"])
                            >= float(row["horizon_minutes"]) - 1e-9
                        and (
                            float(other["price"]) < float(row["price"]) - 1e-9
                            or float(other["weighted_horizon_xp"])
                                > float(row["weighted_horizon_xp"]) + 1e-9
                            or float(other["horizon_minutes"])
                                > float(row["horizon_minutes"]) + 1e-9
                        )
                    )
                    for other in rows
                )
                if not dominated:
                    keep.add(pid)

        result = result.loc[
            result["player_id"].astype(int).isin(keep)
        ].copy()

    return (
        result
        .sort_values(
            ["position", "weighted_horizon_xp"],
            ascending=[True, False],
        )
        .reset_index(drop=True)
    )


def valid_outfield_formations() -> list[dict[str, int]]:
    """Enumerate all legal outfield position counts for an FPL XI."""
    formations = []
    for defenders in range(3, 6):
        for midfielders in range(2, 6):
            for forwards in range(1, 4):
                if defenders + midfielders + forwards == 10:
                    formations.append({
                        "GK": 1,
                        "DEF": defenders,
                        "MID": midfielders,
                        "FWD": forwards,
                    })
    return formations


VALID_OUTFIELD_FORMATIONS = valid_outfield_formations()


def build_projection_lookup(
    projections: pd.DataFrame,
) -> dict[tuple[int, int], dict[str, float]]:
    """Create a compact lookup used repeatedly by the transfer search."""
    lookup = {}
    for row in projections.itertuples(index=False):
        lookup[(int(row.player_id), int(row.gameweek))] = {
            "expected_points": float(row.expected_points),
            "expected_minutes": float(row.expected_minutes),
            "appearance_probability": float(row.appearance_probability),
            "no_appearance_probability": float(row.no_appearance_probability),
        }
    return lookup


def fast_squad_gameweek_decision(
    squad_ids: tuple[int, ...] | list[int],
    gameweek: int,
    candidate_lookup: dict[int, dict[str, object]],
    projection_lookup: dict[tuple[int, int], dict[str, float]],
) -> dict[str, object]:
    """Find the best legal XI and captain pair using lightweight Python lists."""
    squad_ids = tuple(sorted(int(player_id) for player_id in squad_ids))

    by_position: dict[str, list[dict[str, float | int]]] = {
        "GK": [], "DEF": [], "MID": [], "FWD": [],
    }
    all_rows: list[dict[str, float | int | str]] = []

    for player_id in squad_ids:
        player = candidate_lookup[player_id]
        projection = projection_lookup.get(
            (player_id, int(gameweek)),
            {
                "expected_points": 0.0,
                "expected_minutes": 0.0,
                "appearance_probability": 0.0,
                "no_appearance_probability": 1.0,
            },
        )
        row = {
            "player_id": player_id,
            "position": str(player["position"]),
            "expected_points": float(projection["expected_points"]),
            "expected_minutes": float(projection["expected_minutes"]),
            "appearance_probability": float(projection["appearance_probability"]),
            "no_appearance_probability": float(projection["no_appearance_probability"]),
        }
        by_position[row["position"]].append(row)
        all_rows.append(row)

    for position in by_position:
        by_position[position].sort(
            key=lambda row: row["expected_points"],
            reverse=True,
        )

    best = None
    all_xp = {
        int(row["player_id"]): float(row["expected_points"])
        for row in all_rows
    }

    for formation in VALID_OUTFIELD_FORMATIONS:
        starters: list[dict[str, float | int | str]] = []
        legal = True
        for position, number_required in formation.items():
            available = by_position.get(position, [])
            if len(available) < number_required:
                legal = False
                break
            starters.extend(available[:number_required])
        if not legal:
            continue

        starter_ids = tuple(int(row["player_id"]) for row in starters)
        base_score = sum(float(row["expected_points"]) for row in starters)

        captain_pair = select_production_captain_pair(starters)
        captain_uplift = float(captain_pair["expected_captain_uplift"])
        captain_id = int(captain_pair["captain_id"])
        vice_captain_id = int(captain_pair["vice_captain_id"])

        starter_set = set(starter_ids)
        bench_xp = sum(
            xp for player_id, xp in all_xp.items()
            if player_id not in starter_set
        )
        search_score = (
            base_score
            + float(captain_uplift)
            + SQUAD_DEPTH_WEIGHT * bench_xp
        )

        result = {
            "gameweek": int(gameweek),
            "formation": (
                formation["DEF"],
                formation["MID"],
                formation["FWD"],
            ),
            "starter_ids": starter_ids,
            "captain_id": captain_id,
            "vice_captain_id": vice_captain_id,
            "starting_xp": float(base_score),
            "captain_uplift": float(captain_uplift),
            "bench_depth_xp": float(bench_xp),
            "search_score": float(search_score),
        }
        if best is None or result["search_score"] > best["search_score"]:
            best = result

    if best is None:
        raise ValueError("No legal starting formation exists for this squad.")
    return best

### Transfer actions and beam search

A state contains the 15-player squad, bank balance, current selling values, free transfers and the decisions already made. Selling values are represented in integer tenths of a million to avoid floating-point budget errors.

Within one Gameweek, the action generator considers up to two same-position replacements. It enforces squad composition implicitly, and checks budget and the three-player-per-club rule after every move.


In [ ]:
def transfer_state_key(state: dict[str, object]) -> tuple:
    """Key states by all information that can affect later decisions."""
    squad_ids = tuple(sorted(int(player_id) for player_id in state["squad_ids"]))
    selling_prices = tuple(
        sorted(
            (int(player_id), int(price))
            for player_id, price in state["selling_prices_tenths"].items()
            if int(player_id) in squad_ids
        )
    )
    return (
        squad_ids,
        int(state["bank_tenths"]),
        int(state["free_transfers"]),
        selling_prices,
    )


def squad_structure_is_legal(
    squad_ids: set[int] | tuple[int, ...],
    candidate_lookup: dict[int, dict[str, object]],
) -> bool:
    """Fast structural validation without constructing pandas objects."""
    ids = [int(player_id) for player_id in squad_ids]
    if len(ids) != 15 or len(set(ids)) != 15:
        return False
    if any(player_id not in candidate_lookup for player_id in ids):
        return False

    position_counts = {position: 0 for position in SQUAD_POSITION_REQUIREMENTS}
    for player_id in ids:
        position = str(candidate_lookup[player_id]["position"])
        if position not in position_counts:
            return False
        position_counts[position] += 1
    return all(
        int(position_counts[position]) == int(required)
        for position, required in SQUAD_POSITION_REQUIREMENTS.items()
    )


def squad_club_counts(
    squad_ids: set[int] | tuple[int, ...],
    candidate_lookup: dict[int, dict[str, object]],
) -> dict[int, int]:
    """Fast club counts for a structurally legal 15-player squad."""
    if not squad_structure_is_legal(squad_ids, candidate_lookup):
        return {}
    counts: dict[int, int] = {}
    for player_id in squad_ids:
        team_id = int(candidate_lookup[int(player_id)]["team_id"])
        counts[team_id] = counts.get(team_id, 0) + 1
    return counts


def squad_club_excess(
    squad_ids: set[int] | tuple[int, ...],
    candidate_lookup: dict[int, dict[str, object]],
) -> int:
    """Count players above the three-per-club limit across all clubs."""
    return int(sum(
        max(0, int(count) - MAX_PLAYERS_PER_CLUB)
        for count in squad_club_counts(squad_ids, candidate_lookup).values()
    ))


def squad_is_legal(
    squad_ids: set[int] | tuple[int, ...],
    candidate_lookup: dict[int, dict[str, object]],
) -> bool:
    return bool(
        squad_structure_is_legal(squad_ids, candidate_lookup)
        and squad_club_excess(squad_ids, candidate_lookup) == 0
    )

def squad_legality_diagnostics(
    squad_ids: set[int] | tuple[int, ...],
    candidate_lookup: dict[int, dict[str, object]],
) -> dict[str, object]:
    """Describe structural problems and temporary club-limit excess."""
    squad_ids = [int(player_id) for player_id in squad_ids]
    missing_ids = sorted(
        set(squad_ids) - set(candidate_lookup)
    )
    position_counts = {}
    if not missing_ids:
        position_counts = pd.Series([
            candidate_lookup[player_id]["position"]
            for player_id in squad_ids
        ]).value_counts().to_dict()

    club_counts = squad_club_counts(
        squad_ids,
        candidate_lookup,
    )
    overfull_clubs = {
        int(team_id): int(count)
        for team_id, count in club_counts.items()
        if count > MAX_PLAYERS_PER_CLUB
    }
    return {
        "squad_size": len(squad_ids),
        "unique_players": len(set(squad_ids)),
        "missing_player_ids": missing_ids,
        "position_counts": position_counts,
        "overfull_clubs": overfull_clubs,
        "club_excess": squad_club_excess(
            squad_ids,
            candidate_lookup,
        ),
    }


def state_no_transfer_future_score(
    squad_ids: tuple[int, ...],
    remaining_gameweeks: list[int],
    candidate_lookup: dict[int, dict[str, object]],
    projection_lookup: dict[tuple[int, int], dict[str, float]],
    first_horizon_index: int,
    decision_cache: dict[tuple[tuple[int, ...], int], dict[str, object]],
    future_score_cache: dict[tuple, float] | None = None,
) -> float:
    """Estimate the no-more-transfers continuation value, with memoisation."""
    squad_key = tuple(sorted(int(v) for v in squad_ids))
    remaining_key = tuple(int(v) for v in remaining_gameweeks)
    cache_key = (squad_key, remaining_key, int(first_horizon_index))
    if future_score_cache is not None and cache_key in future_score_cache:
        return float(future_score_cache[cache_key])

    total = 0.0
    for offset, gameweek in enumerate(remaining_key):
        decision_key = (squad_key, int(gameweek))
        if decision_key not in decision_cache:
            decision_cache[decision_key] = fast_squad_gameweek_decision(
                squad_key,
                int(gameweek),
                candidate_lookup,
                projection_lookup,
            )
        total += (
            GAMEWEEK_DISCOUNT ** (first_horizon_index + offset)
            * float(decision_cache[decision_key]["search_score"])
        )
    total = float(total)
    if future_score_cache is not None:
        future_score_cache[cache_key] = total
    return total


def _updated_club_state_for_transfer(
    club_counts: dict[int, int],
    club_excess: int,
    outgoing_team: int,
    incoming_team: int,
) -> tuple[dict[int, int], int]:
    """Update club counts/excess in O(1) for one same-position transfer."""
    if outgoing_team == incoming_team:
        return dict(club_counts), int(club_excess)

    updated = dict(club_counts)
    new_excess = int(club_excess)

    for team_id, delta in ((outgoing_team, -1), (incoming_team, 1)):
        before = int(updated.get(team_id, 0))
        after = before + delta
        new_excess -= max(0, before - MAX_PLAYERS_PER_CLUB)
        new_excess += max(0, after - MAX_PLAYERS_PER_CLUB)
        if after <= 0:
            updated.pop(team_id, None)
        else:
            updated[team_id] = after

    return updated, int(new_excess)


def _score_transfer_action_for_ranking(
    action: dict[str, object],
    base_state: dict[str, object],
    gameweek: int,
    remaining_gameweeks: list[int],
    horizon_index: int,
    candidate_lookup: dict[int, dict[str, object]],
    projection_lookup: dict[tuple[int, int], dict[str, float]],
    decision_cache: dict[tuple[tuple[int, ...], int], dict[str, object]],
    future_score_cache: dict[tuple, float] | None,
) -> None:
    squad_key = tuple(sorted(int(v) for v in action["squad_ids"]))
    current_key = (squad_key, int(gameweek))
    if current_key not in decision_cache:
        decision_cache[current_key] = fast_squad_gameweek_decision(
            squad_key, int(gameweek), candidate_lookup, projection_lookup
        )
    current_score = float(decision_cache[current_key]["search_score"])
    hits = transfer_hit_cost(
        int(base_state["free_transfers"]),
        len(action["transfers"]),
    )
    future_score = state_no_transfer_future_score(
        squad_key,
        remaining_gameweeks,
        candidate_lookup,
        projection_lookup,
        horizon_index + 1,
        decision_cache,
        future_score_cache,
    )
    action["action_ranking_score"] = (
        GAMEWEEK_DISCOUNT ** horizon_index * (current_score - hits)
        + future_score
    )


class IllegalForcedTransferAction(RuntimeError):
    """
    Raised only when a robustness/debug forced current-deadline transfer
    package is not legal from the actual opening state.

    This is distinct from an optimiser failure: a legal multi-transfer
    package may have one-transfer subsets that are not individually legal
    because of budget, club-limit, or squad-structure dependencies.
    """
    pass


def generate_transfer_actions(
    base_state: dict[str, object],
    candidate_pool: pd.DataFrame,
    candidate_lookup: dict[int, dict[str, object]],
    projection_lookup: dict[tuple[int, int], dict[str, float]],
    gameweek: int,
    remaining_gameweeks: list[int],
    horizon_index: int,
    decision_cache: dict[tuple[tuple[int, ...], int], dict[str, object]],
    future_score_cache: dict[tuple, float] | None = None,
    pool_by_position: dict[str, list[int]] | None = None,
    forced_transfers: tuple[tuple[int, int], ...] | None = None,
) -> list[dict[str, object]]:
    """Generate/prune legal actions using incremental club and quality state."""
    initial_squad = set(int(player_id) for player_id in base_state["squad_ids"])
    initial_club_counts = squad_club_counts(initial_squad, candidate_lookup)
    initial_club_excess = int(sum(
        max(0, count - MAX_PLAYERS_PER_CLUB)
        for count in initial_club_counts.values()
    ))
    initial_quality = float(sum(
        float(candidate_lookup[player_id].get("weighted_horizon_xp", 0.0))
        for player_id in initial_squad
    ))

    initial_action = {
        "squad_ids": initial_squad,
        "bank_tenths": int(base_state["bank_tenths"]),
        "selling_prices_tenths": dict(base_state["selling_prices_tenths"]),
        "transfers": [],
        "used_out": frozenset(),
        "used_in": frozenset(),
        "club_counts": initial_club_counts,
        "club_excess": initial_club_excess,
        "squad_quality": initial_quality,
    }

    if pool_by_position is None:
        pool_by_position = {
            position: (
                group.sort_values("weighted_horizon_xp", ascending=False)
                .head(TRANSFER_MAX_INCOMING_PER_POSITION)["player_id"]
                .astype(int).tolist()
            )
            for position, group in candidate_pool.groupby("position")
        }

    def apply_one(partial: dict[str, object], outgoing_id: int, incoming_id: int):
        squad_ids = set(partial["squad_ids"])
        outgoing_id = int(outgoing_id)
        incoming_id = int(incoming_id)
        if outgoing_id not in squad_ids or incoming_id in squad_ids:
            return None
        if outgoing_id in partial["used_in"] or incoming_id in partial["used_out"]:
            return None
        if outgoing_id not in candidate_lookup or incoming_id not in candidate_lookup:
            return None
        outgoing = candidate_lookup[outgoing_id]
        incoming = candidate_lookup[incoming_id]
        if str(outgoing["position"]) != str(incoming["position"]):
            return None

        selling_price = int(partial["selling_prices_tenths"].get(
            outgoing_id, price_to_tenths(outgoing["price"])
        ))
        buying_price = price_to_tenths(incoming["price"])
        new_bank = int(partial["bank_tenths"]) + selling_price - buying_price
        if new_bank < 0:
            return None

        new_club_counts, new_club_excess = _updated_club_state_for_transfer(
            partial["club_counts"],
            int(partial["club_excess"]),
            int(outgoing["team_id"]),
            int(incoming["team_id"]),
        )
        current_excess = int(partial["club_excess"])
        if current_excess == 0:
            if new_club_excess != 0:
                return None
        elif new_club_excess >= current_excess:
            return None

        new_selling_prices = dict(partial["selling_prices_tenths"])
        new_selling_prices.pop(outgoing_id, None)
        new_selling_prices[incoming_id] = buying_price
        return {
            "squad_ids": (squad_ids - {outgoing_id}) | {incoming_id},
            "bank_tenths": new_bank,
            "selling_prices_tenths": new_selling_prices,
            "transfers": list(partial["transfers"]) + [(outgoing_id, incoming_id)],
            "used_out": partial["used_out"] | {outgoing_id},
            "used_in": partial["used_in"] | {incoming_id},
            "club_counts": new_club_counts,
            "club_excess": new_club_excess,
            "squad_quality": (
                float(partial["squad_quality"])
                - float(outgoing.get("weighted_horizon_xp", 0.0))
                + float(incoming.get("weighted_horizon_xp", 0.0))
            ),
        }

    # Robustness re-solves know the exact first action. Construct it directly
    # instead of enumerating and ranking every legal current transfer first.
    if forced_transfers is not None:
        forced_transfers = tuple((int(a), int(b)) for a,b in forced_transfers)
        if (
            not ALLOW_TRANSFER_HITS
            and len(forced_transfers) > int(base_state["free_transfers"])
        ):
            raise RuntimeError("Forced first action requires a disallowed hit.")
        partial = initial_action
        for outgoing_id, incoming_id in forced_transfers:
            partial = apply_one(partial, outgoing_id, incoming_id)
            if partial is None:
                raise IllegalForcedTransferAction(
                    "Forced first action was not legal from the "
                    "opening squad/state: "
                    f"{forced_transfers}"
                )
        _score_transfer_action_for_ranking(
            partial, base_state, gameweek, remaining_gameweeks, horizon_index,
            candidate_lookup, projection_lookup, decision_cache,
            future_score_cache,
        )
        return [partial]

    partial_states = [initial_action]
    all_actions = [initial_action]

    effective_max_transfers = int(MAX_TRANSFERS_PER_GAMEWEEK)
    if EMERGENCY_STRUCTURAL_REPAIR_ENABLED:
        unavailable_owned = [
            player_id
            for player_id in initial_squad
            if float(
                candidate_lookup[player_id].get(
                    "horizon_minutes", np.inf
                )
            ) <= float(EMERGENCY_UNAVAILABLE_MAX_HORIZON_MINUTES)
        ]
        if len(unavailable_owned) >= int(EMERGENCY_UNAVAILABLE_MIN_COUNT):
            effective_max_transfers = max(
                effective_max_transfers,
                min(
                    int(EMERGENCY_MAX_TRANSFERS_PER_GAMEWEEK),
                    len(unavailable_owned),
                ),
            )

    for transfer_depth in range(1, effective_max_transfers + 1):
        if (
            not ALLOW_TRANSFER_HITS
            and transfer_depth > int(base_state["free_transfers"])
        ):
            break

        expanded = []
        for partial in partial_states:
            swap_candidates = []
            for outgoing_id in sorted(partial["squad_ids"]):
                if outgoing_id in partial["used_in"]:
                    continue
                position = str(candidate_lookup[outgoing_id]["position"])
                outgoing_quality = float(
                    candidate_lookup[outgoing_id].get(
                        "weighted_horizon_xp", 0.0
                    )
                )
                outgoing_minutes = float(
                    candidate_lookup[outgoing_id].get(
                        "horizon_minutes", 0.0
                    )
                )
                for incoming_id in pool_by_position.get(position, []):
                    if int(incoming_id) in partial["squad_ids"]:
                        continue
                    incoming_quality = float(
                        candidate_lookup[incoming_id].get(
                            "weighted_horizon_xp", 0.0
                        )
                    )
                    # Prioritise xP gain and strongly prioritise removing
                    # structurally unavailable assets. Exact legality/finance
                    # is still checked by apply_one.
                    repair_bonus = (
                        3.0
                        if outgoing_minutes
                        <= float(EMERGENCY_UNAVAILABLE_MAX_HORIZON_MINUTES)
                        else 0.0
                    )
                    swap_candidates.append(
                        (
                            incoming_quality - outgoing_quality + repair_bonus,
                            int(outgoing_id),
                            int(incoming_id),
                        )
                    )

            swap_candidates.sort(reverse=True)
            cap = int(TRANSFER_MAX_SWAP_CANDIDATES_PER_PARTIAL)
            if cap > 0:
                swap_candidates = swap_candidates[:cap]

            for _score, outgoing_id, incoming_id in swap_candidates:
                action = apply_one(partial, outgoing_id, incoming_id)
                if action is not None:
                    expanded.append(action)

        deduplicated = {}
        for action in expanded:
            key = (
                tuple(sorted(action["squad_ids"])),
                int(action["bank_tenths"]),
                tuple(sorted(action["selling_prices_tenths"].items())),
            )
            existing = deduplicated.get(key)
            if (
                existing is None
                or float(action["squad_quality"]) > float(existing["squad_quality"])
            ):
                deduplicated[key] = action

        cheap_ranked = []
        for action in deduplicated.values():
            hits = transfer_hit_cost(
                int(base_state["free_transfers"]),
                len(action["transfers"]),
            )
            action["cheap_ranking_score"] = float(action["squad_quality"]) - hits
            cheap_ranked.append(action)

        partial_states = sorted(
            cheap_ranked,
            key=lambda action: action["cheap_ranking_score"],
            reverse=True,
        )[:TRANSFER_ACTION_BEAM_WIDTH]

        for action in partial_states:
            _score_transfer_action_for_ranking(
                action, base_state, gameweek, remaining_gameweeks, horizon_index,
                candidate_lookup, projection_lookup, decision_cache,
                future_score_cache,
            )
        partial_states.sort(
            key=lambda action: action["action_ranking_score"], reverse=True
        )
        all_actions.extend(partial_states)

    _score_transfer_action_for_ranking(
        initial_action, base_state, gameweek, remaining_gameweeks, horizon_index,
        candidate_lookup, projection_lookup, decision_cache, future_score_cache,
    )
    return all_actions

def optimise_transfer_path(
    current_squad_ids: list[int],
    candidates: pd.DataFrame,
    projections: pd.DataFrame,
    gameweeks_to_use: list[int],
    money_in_bank: float,
    free_transfers: int,
    current_selling_prices: dict[int, float] | None = None,
    forced_first_action_transfers: tuple[tuple[int, int], ...] | None = None,
    prebuilt_transfer_pool: pd.DataFrame | None = None,
    shared_decision_cache: dict | None = None,
    shared_future_score_cache: dict | None = None,
) -> dict[str, object]:
    """Use beam search to optimise transfers across the rolling horizon."""
    current_squad_ids = [int(player_id) for player_id in current_squad_ids]

    forced_first_action_key = (
        tuple(
            sorted(
                (
                    int(out_id),
                    int(in_id),
                )
                for out_id, in_id
                in forced_first_action_transfers
            )
        )
        if forced_first_action_transfers is not None
        else None
    )

    if len(set(current_squad_ids)) != 15:
        raise ValueError("Transfer planning requires exactly 15 unique squad IDs.")
    if not 1 <= int(free_transfers) <= MAX_BANKED_FREE_TRANSFERS:
        raise ValueError("CURRENT_FREE_TRANSFERS must be between 1 and 5.")

    transfer_pool = (
        prebuilt_transfer_pool.copy()
        if prebuilt_transfer_pool is not None
        else build_transfer_candidate_pool(
            candidates, projections, gameweeks_to_use, current_squad_ids
        )
    )

    # Every owned player must remain available as a possible retained asset.
    missing_owned = set(current_squad_ids) - set(transfer_pool["player_id"].astype(int))
    if missing_owned:
        transfer_pool = pd.concat([
            transfer_pool,
            candidates.loc[candidates["player_id"].isin(missing_owned)],
        ], ignore_index=True).drop_duplicates("player_id")

    candidate_lookup = {
        int(row.player_id): row._asdict()
        for row in transfer_pool.itertuples(index=False)
    }
    projection_lookup = build_projection_lookup(projections)

    if not squad_structure_is_legal(
        current_squad_ids,
        candidate_lookup,
    ):
        diagnostics = squad_legality_diagnostics(
            current_squad_ids,
            candidate_lookup,
        )
        raise ValueError(
            "The configured current squad has an invalid 15-player "
            "structure. Diagnostics: "
            f"{diagnostics}"
        )

    # FPL permits a manager to retain more than three players from a club
    # when a real-world player transfer creates the excess. The manager can
    # keep the squad and roll, but any FPL transfer must reduce the excess.
    starting_club_excess = squad_club_excess(
        current_squad_ids,
        candidate_lookup,
    )

    selling_prices_input = current_selling_prices or {}
    selling_prices_tenths = {}
    for player_id in current_squad_ids:
        market_price = candidate_lookup[player_id]["price"]
        selling_prices_tenths[player_id] = price_to_tenths(
            selling_prices_input.get(player_id, market_price)
        )

    initial_state = {
        "squad_ids": tuple(sorted(current_squad_ids)),
        "bank_tenths": price_to_tenths(money_in_bank),
        "selling_prices_tenths": selling_prices_tenths,
        "free_transfers": int(free_transfers),
        "cumulative_expected_points": 0.0,
        "cumulative_hits": 0.0,
        "discounted_objective": 0.0,
        "path": [],
        "starting_club_excess": int(starting_club_excess),
    }

    states = [initial_state]
    decision_cache = (
        shared_decision_cache
        if shared_decision_cache is not None
        else {}
    )
    future_score_cache = (
        shared_future_score_cache
        if shared_future_score_cache is not None
        else {}
    )
    pool_by_position = {
        position: (
            group.sort_values("weighted_horizon_xp", ascending=False)
            .head(TRANSFER_MAX_INCOMING_PER_POSITION)["player_id"]
            .astype(int).tolist()
        )
        for position, group in transfer_pool.groupby("position")
    }

    # Current-deadline actions are the decisions we may actually execute.
    # Keep a small, diverse set of those actions alive throughout the full
    # horizon so the production robustness gate can compare transfer counts
    # using meaningful alternatives rather than whatever branch happens to
    # survive the global beam.
    protected_first_action_keys: set[tuple[tuple[int, int], ...]] = set()

    for horizon_index, gameweek in enumerate(gameweeks_to_use):
        remaining_gameweeks = gameweeks_to_use[horizon_index + 1:]
        next_states = []

        for state in states:
            actions = generate_transfer_actions(
                state,
                transfer_pool,
                candidate_lookup,
                projection_lookup,
                gameweek,
                remaining_gameweeks,
                horizon_index,
                decision_cache,
                future_score_cache=future_score_cache,
                pool_by_position=pool_by_position,
                forced_transfers=(
                    forced_first_action_key
                    if horizon_index == 0
                    else None
                ),
            )

            for action in actions:
                squad_key = tuple(sorted(action["squad_ids"]))
                decision = decision_cache[(squad_key, int(gameweek))]
                transfer_count = len(action["transfers"])
                hit_cost = transfer_hit_cost(
                    int(state["free_transfers"]),
                    transfer_count,
                )
                expected_score_after_hits = decision["search_score"] - hit_cost
                free_transfers_after = update_free_transfers(
                    int(state["free_transfers"]),
                    transfer_count,
                )

                path_row = {
                    "gameweek": int(gameweek),
                    "squad_ids_before": tuple(state["squad_ids"]),
                    "squad_ids_after": squad_key,
                    "transfers": tuple(action["transfers"]),
                    "transfer_count": transfer_count,
                    "free_transfers_before": int(state["free_transfers"]),
                    "free_transfers_after": free_transfers_after,
                    "hit_cost": float(hit_cost),
                    "bank_before_tenths": int(state["bank_tenths"]),
                    "bank_after_tenths": int(action["bank_tenths"]),
                    "expected_score_before_hits": float(decision["search_score"]),
                    "expected_score_after_hits": float(expected_score_after_hits),
                    "lineup_decision": decision,
                }

                next_states.append({
                    "squad_ids": squad_key,
                    "bank_tenths": int(action["bank_tenths"]),
                    "selling_prices_tenths": dict(action["selling_prices_tenths"]),
                    "free_transfers": free_transfers_after,
                    "cumulative_expected_points": (
                        float(state["cumulative_expected_points"])
                        + float(decision["search_score"])
                    ),
                    "cumulative_hits": (
                        float(state["cumulative_hits"])
                        + float(hit_cost)
                    ),
                    "discounted_objective": (
                        float(state["discounted_objective"])
                        + GAMEWEEK_DISCOUNT ** horizon_index
                        * float(expected_score_after_hits)
                    ),
                    "path": list(state["path"]) + [path_row],
                    "starting_club_excess": int(
                        state.get("starting_club_excess", 0)
                    ),
                })

        # Keep the strongest version of each future-relevant state.
        deduplicated_states = {}
        for state in next_states:
            key = transfer_state_key(state)
            current_best = deduplicated_states.get(key)
            if (
                current_best is None
                or state["discounted_objective"]
                > current_best["discounted_objective"]
            ):
                deduplicated_states[key] = state

        ranked_states = []
        for state in deduplicated_states.values():
            future_heuristic = state_no_transfer_future_score(
                tuple(state["squad_ids"]),
                remaining_gameweeks,
                candidate_lookup,
                projection_lookup,
                horizon_index + 1,
                decision_cache,
                future_score_cache,
            )
            state["beam_ranking_score"] = (
                float(state["discounted_objective"])
                + float(future_heuristic)
            )
            ranked_states.append(state)

        ranked_states = sorted(
            ranked_states,
            key=lambda state: state["beam_ranking_score"],
            reverse=True,
        )

        # -----------------------------------------------------------------
        # Protect DISTINCT first actions, not merely one branch per transfer
        # count. A single protected 1-FT branch is insufficient: another
        # one-transfer action can look slightly better to the intermediate
        # heuristic and replace the strategically relevant branch, even when
        # the latter has the better final six-GW objective.
        #
        # At the current deadline we select up to K distinct first actions
        # for every transfer count. On later horizon steps, the strongest
        # continuation of EACH selected first action is explicitly retained.
        # -----------------------------------------------------------------
        if WEEKLY_PROTECT_FIRST_ACTION_TRANSFER_COUNTS:
            if horizon_index == 0:
                protected_first_action_keys.clear()

                for transfer_count in range(
                    0,
                    MAX_TRANSFERS_PER_GAMEWEEK + 1,
                ):
                    seen_for_count = 0
                    seen_keys = set()

                    for state in ranked_states:
                        if not state.get("path"):
                            continue

                        first_row = state["path"][0]
                        if (
                            int(first_row["transfer_count"])
                            != int(transfer_count)
                        ):
                            continue

                        first_key = tuple(
                            (int(out_id), int(in_id))
                            for out_id, in_id
                            in first_row["transfers"]
                        )

                        if first_key in seen_keys:
                            continue

                        seen_keys.add(first_key)
                        protected_first_action_keys.add(
                            first_key
                        )
                        seen_for_count += 1

                        # Roll has only one distinct action. For transfer
                        # counts >0 retain several alternatives.
                        limit = (
                            1
                            if int(transfer_count) == 0
                            else int(
                                WEEKLY_PROTECTED_FIRST_ACTIONS_PER_TRANSFER_COUNT
                            )
                        )
                        if seen_for_count >= limit:
                            break

            protected_states = []
            protected_ids = set()

            for first_key in protected_first_action_keys:
                matches = [
                    state for state in ranked_states
                    if state.get("path")
                    and tuple(
                        (int(out_id), int(in_id))
                        for out_id, in_id
                        in state["path"][0]["transfers"]
                    ) == first_key
                ]

                if matches:
                    best_for_first_action = matches[0]
                    protected_states.append(
                        best_for_first_action
                    )
                    protected_ids.add(
                        id(best_for_first_action)
                    )

            # Keep the protected set itself ordered by the same beam score.
            protected_states.sort(
                key=lambda state: state["beam_ranking_score"],
                reverse=True,
            )

            if len(protected_states) > TRANSFER_BEAM_WIDTH:
                raise RuntimeError(
                    "Configured first-action protection exceeds "
                    "TRANSFER_BEAM_WIDTH. Reduce "
                    "WEEKLY_PROTECTED_FIRST_ACTIONS_PER_TRANSFER_COUNT "
                    "or increase TRANSFER_BEAM_WIDTH."
                )

            states = list(protected_states)
            states.extend(
                state for state in ranked_states
                if id(state) not in protected_ids
            )
        else:
            # Legacy behaviour: explicitly keep Roll if it exists.
            roll_matches = [
                state for state in ranked_states
                if state.get("path")
                and int(
                    state["path"][0]["transfer_count"]
                ) == 0
            ]
            states = (
                [roll_matches[0]]
                if roll_matches
                else []
            )
            states.extend(
                state for state in ranked_states
                if not roll_matches
                or id(state) != id(roll_matches[0])
            )

        states = states[:TRANSFER_BEAM_WIDTH]
        states.sort(
            key=lambda state: state["beam_ranking_score"],
            reverse=True,
        )

        if not states:
            raise RuntimeError(
                f"Transfer search produced no valid states for Gameweek {gameweek}."
            )

    terminal_gameweeks = gameweeks_to_use[-min(2, len(gameweeks_to_use)):]
    for state in states:
        terminal_score = state_no_transfer_future_score(
            tuple(state["squad_ids"]),
            terminal_gameweeks,
            candidate_lookup,
            projection_lookup,
            max(0, len(gameweeks_to_use) - len(terminal_gameweeks)),
            decision_cache,
            future_score_cache,
        ) / max(1, len(terminal_gameweeks))
        state["terminal_value"] = (
            TRANSFER_TERMINAL_VALUE_WEIGHT * terminal_score
        )
        state["final_objective"] = (
            float(state["discounted_objective"])
            + float(state["terminal_value"])
        )

    best_state = max(states, key=lambda state: state["final_objective"])
    return {
        "best_state": best_state,
        "candidate_pool": transfer_pool,
        "decision_cache": decision_cache,
        "future_score_cache": future_score_cache,
        "final_beam": states,
        "protected_first_action_keys": tuple(
            sorted(protected_first_action_keys)
        ),
        "forced_first_action_key": forced_first_action_key,
    }


def _transfer_search_setting_names() -> tuple[str, ...]:
    return (
        "TRANSFER_BEAM_WIDTH",
        "TRANSFER_ACTION_BEAM_WIDTH",
        "TRANSFER_TOP_XP_PER_POSITION",
        "TRANSFER_TOP_VALUE_PER_POSITION",
        "TRANSFER_MAX_INCOMING_PER_POSITION",
        "TRANSFER_MAX_SWAP_CANDIDATES_PER_PARTIAL",
        "WEEKLY_PROTECTED_FIRST_ACTIONS_PER_TRANSFER_COUNT",
    )


def _snapshot_transfer_search_settings() -> dict[str, int]:
    return {
        name: int(globals()[name])
        for name in _transfer_search_setting_names()
    }


def _apply_transfer_search_profile(profile_name: str) -> None:
    if profile_name not in TRANSFER_SEARCH_PROFILES:
        raise ValueError(f"Unknown transfer-search profile: {profile_name!r}")
    profile = TRANSFER_SEARCH_PROFILES[profile_name]
    replacements = {
        "TRANSFER_BEAM_WIDTH": profile["beam_width"],
        "TRANSFER_ACTION_BEAM_WIDTH": profile["action_beam_width"],
        "TRANSFER_TOP_XP_PER_POSITION": profile["top_xp_per_position"],
        "TRANSFER_TOP_VALUE_PER_POSITION": profile["top_value_per_position"],
        "TRANSFER_MAX_INCOMING_PER_POSITION": profile["max_incoming_per_position"],
        "TRANSFER_MAX_SWAP_CANDIDATES_PER_PARTIAL": profile["max_swap_candidates_per_partial"],
        "WEEKLY_PROTECTED_FIRST_ACTIONS_PER_TRANSFER_COUNT": (
            profile["protected_first_actions_per_transfer_count"]
        ),
    }
    for name, value in replacements.items():
        globals()[name] = int(value)


def _restore_transfer_search_settings(snapshot: dict[str, int]) -> None:
    for name, value in snapshot.items():
        globals()[name] = int(value)


def transfer_first_action_scorecard(
    transfer_result: dict[str, object],
    *,
    allow_hits: bool | None = None,
) -> pd.DataFrame:
    """Collapse a terminal beam to the best final objective per first action."""
    rows = []
    allow_hits = ALLOW_TRANSFER_HITS if allow_hits is None else bool(allow_hits)

    for state in transfer_result.get("final_beam", []):
        if not state.get("path"):
            continue
        if not allow_hits and float(state.get("cumulative_hits", 0.0)) > 1e-9:
            continue
        first = state["path"][0]
        action_key = tuple(
            sorted(
                (int(out_id), int(in_id))
                for out_id, in_id in first.get("transfers", ())
            )
        )
        rows.append({
            "first_action": action_key,
            "transfer_count": int(first.get("transfer_count", len(action_key))),
            "final_objective": float(state["final_objective"]),
            "current_hit_cost": float(first.get("hit_cost", 0.0)),
        })

    if not rows:
        return pd.DataFrame(
            columns=[
                "first_action",
                "transfer_count",
                "final_objective",
                "current_hit_cost",
            ]
        )

    frame = pd.DataFrame(rows)
    return (
        frame.sort_values("final_objective", ascending=False)
        .drop_duplicates("first_action", keep="first")
        .reset_index(drop=True)
    )


def _transfer_search_result_summary(
    transfer_result: dict[str, object],
) -> dict[str, object]:
    scorecard = transfer_first_action_scorecard(transfer_result)
    best_action = (
        scorecard.iloc[0]["first_action"]
        if not scorecard.empty else ()
    )
    gap = (
        float(scorecard.iloc[0]["final_objective"] - scorecard.iloc[1]["final_objective"])
        if len(scorecard) >= 2
        else np.nan
    )
    return {
        "best_action": tuple(best_action),
        "best_objective": (
            float(scorecard.iloc[0]["final_objective"])
            if not scorecard.empty else np.nan
        ),
        "first_action_gap": gap,
        "distinct_first_actions": int(len(scorecard)),
        "best_transfer_count": (
            int(scorecard.iloc[0]["transfer_count"])
            if not scorecard.empty else 0
        ),
    }


def run_transfer_search_profile(
    profile_name: str,
    *args,
    **kwargs,
) -> dict[str, object]:
    """Run the existing exact beam-search implementation under one width profile."""
    original = _snapshot_transfer_search_settings()
    started = time.perf_counter()
    try:
        _apply_transfer_search_profile(profile_name)
        result = optimise_transfer_path(*args, **kwargs)
    finally:
        _restore_transfer_search_settings(original)

    result = dict(result)
    result["search_profile"] = str(profile_name)
    result["search_runtime_seconds"] = float(time.perf_counter() - started)
    result["search_summary"] = _transfer_search_result_summary(result)
    return result


def optimise_transfer_path_adaptive(
    current_squad_ids: list[int],
    candidates: pd.DataFrame,
    projections: pd.DataFrame,
    gameweeks_to_use: list[int],
    money_in_bank: float,
    free_transfers: int,
    current_selling_prices: dict[int, float] | None = None,
    forced_first_action_transfers: tuple[tuple[int, int], ...] | None = None,
    prebuilt_transfer_pool: pd.DataFrame | None = None,
    shared_decision_cache: dict | None = None,
    shared_future_score_cache: dict | None = None,
    *,
    force_profile: str | None = None,
    allow_reference_escalation: bool | None = None,
) -> dict[str, object]:
    """
    Two-stage production transfer search.

    Fast search is sufficient when the leading distinct first action has a
    meaningful objective margin. Otherwise the search widens to `standard`.
    The v2.2.4 reference widths are benchmark-only unless explicitly enabled.
    """
    common_kwargs = dict(
        current_squad_ids=current_squad_ids,
        candidates=candidates,
        projections=projections,
        gameweeks_to_use=gameweeks_to_use,
        money_in_bank=money_in_bank,
        free_transfers=free_transfers,
        current_selling_prices=current_selling_prices,
        forced_first_action_transfers=forced_first_action_transfers,
        prebuilt_transfer_pool=prebuilt_transfer_pool,
        shared_decision_cache=shared_decision_cache,
        shared_future_score_cache=shared_future_score_cache,
    )

    # Forced/subset re-solves should be deterministic and should not spend time
    # deciding which search width to use.
    if force_profile is not None:
        result = run_transfer_search_profile(force_profile, **common_kwargs)
        result["adaptive_search_audit"] = {
            "enabled": False,
            "reason": f"force_profile={force_profile}",
            "selected_profile": force_profile,
            "runs": [{
                "profile": force_profile,
                "seconds": float(result["search_runtime_seconds"]),
                **result["search_summary"],
            }],
        }
        return result

    if (
        not TRANSFER_ADAPTIVE_ENABLED
        or forced_first_action_transfers is not None
        or prebuilt_transfer_pool is not None
    ):
        profile_name = "standard"
        result = run_transfer_search_profile(profile_name, **common_kwargs)
        result["adaptive_search_audit"] = {
            "enabled": False,
            "reason": (
                "adaptive search disabled"
                if not TRANSFER_ADAPTIVE_ENABLED
                else "forced/prebuilt search path"
            ),
            "selected_profile": profile_name,
            "runs": [{
                "profile": profile_name,
                "seconds": float(result["search_runtime_seconds"]),
                **result["search_summary"],
            }],
        }
        return result

    runs = []
    fast = run_transfer_search_profile("fast", **common_kwargs)
    fast_summary = dict(fast["search_summary"])
    runs.append({
        "profile": "fast",
        "seconds": float(fast["search_runtime_seconds"]),
        **fast_summary,
    })

    fast_gap = fast_summary["first_action_gap"]
    fast_distinct = int(fast_summary["distinct_first_actions"])
    fast_confident = bool(
        fast_distinct >= int(TRANSFER_ADAPTIVE_MIN_DISTINCT_FIRST_ACTIONS)
        and np.isfinite(fast_gap)
        and float(fast_gap) >= float(TRANSFER_ADAPTIVE_MIN_FIRST_ACTION_GAP)
    )

    if fast_confident:
        fast["adaptive_search_audit"] = {
            "enabled": True,
            "selected_profile": "fast",
            "escalated": False,
            "confidence": "high",
            "reason": (
                f"fast first-action gap {float(fast_gap):.3f} >= "
                f"{float(TRANSFER_ADAPTIVE_MIN_FIRST_ACTION_GAP):.3f}"
            ),
            "runs": runs,
            "fast_standard_agree": None,
        }
        return fast

    # Share exact lineup/future-score caches across widening stages. The
    # widened candidate pool is rebuilt, so no fast-pool candidate restriction
    # leaks into the standard search.
    standard_kwargs = dict(common_kwargs)
    standard_kwargs["shared_decision_cache"] = fast.get("decision_cache")
    standard_kwargs["shared_future_score_cache"] = fast.get("future_score_cache")
    standard = run_transfer_search_profile("standard", **standard_kwargs)
    standard_summary = dict(standard["search_summary"])
    runs.append({
        "profile": "standard",
        "seconds": float(standard["search_runtime_seconds"]),
        **standard_summary,
    })

    agree = (
        tuple(fast_summary["best_action"])
        == tuple(standard_summary["best_action"])
    )
    standard_gap = standard_summary["first_action_gap"]
    standard_confident = bool(
        int(standard_summary["distinct_first_actions"])
        >= int(TRANSFER_ADAPTIVE_MIN_DISTINCT_FIRST_ACTIONS)
        and np.isfinite(standard_gap)
        and float(standard_gap) >= float(TRANSFER_ADAPTIVE_MIN_FIRST_ACTION_GAP)
    )

    permit_reference = (
        TRANSFER_ADAPTIVE_ALLOW_REFERENCE_ESCALATION
        if allow_reference_escalation is None
        else bool(allow_reference_escalation)
    )

    if agree or standard_confident or not permit_reference:
        standard["adaptive_search_audit"] = {
            "enabled": True,
            "selected_profile": "standard",
            "escalated": True,
            "confidence": (
                "high"
                if agree and standard_confident
                else "medium"
                if agree or standard_confident
                else "low"
            ),
            "reason": (
                "fast and standard agree"
                if agree
                else (
                    f"standard first-action gap {float(standard_gap):.3f} >= "
                    f"{float(TRANSFER_ADAPTIVE_MIN_FIRST_ACTION_GAP):.3f}"
                    if standard_confident
                    else "reference escalation disabled; returning standard search"
                )
            ),
            "runs": runs,
            "fast_standard_agree": bool(agree),
        }
        return standard

    reference_kwargs = dict(common_kwargs)
    reference_kwargs["shared_decision_cache"] = standard.get("decision_cache")
    reference_kwargs["shared_future_score_cache"] = standard.get("future_score_cache")
    reference = run_transfer_search_profile("reference", **reference_kwargs)
    reference_summary = dict(reference["search_summary"])
    runs.append({
        "profile": "reference",
        "seconds": float(reference["search_runtime_seconds"]),
        **reference_summary,
    })
    reference["adaptive_search_audit"] = {
        "enabled": True,
        "selected_profile": "reference",
        "escalated": True,
        "confidence": "reference",
        "reason": "fast/standard disagreement with explicit reference escalation enabled",
        "runs": runs,
        "fast_standard_agree": bool(agree),
    }
    return reference

def select_transfer_state_with_marginal_gate(
    transfer_result: dict[str, object],
    *,
    allow_hits: bool,
    min_gain_per_extra_transfer: float,
) -> tuple[dict[str, object], dict[str, object]]:
    """
    Generic, backtestable version of the production marginal-transfer gate.

    This intentionally excludes the separate structural-repair near-tie rule;
    that overlay remains disabled in v2.2 until it has its own historical
    ablation. The selector is usable by both historical validation and live
    production code.
    """
    states = list(transfer_result.get("final_beam", ()))
    eligible = [
        state for state in states
        if allow_hits or float(state.get("cumulative_hits", 0.0)) <= 1e-9
    ]
    if not eligible:
        raise RuntimeError("No eligible terminal transfer states are available.")

    raw = max(eligible, key=lambda state: float(state["final_objective"]))
    selected = raw
    steps = []

    while selected.get("path"):
        selected_count = int(selected["path"][0]["transfer_count"])
        lower_counts = sorted(
            {
                int(state["path"][0]["transfer_count"])
                for state in eligible
                if state.get("path")
                and int(state["path"][0]["transfer_count"]) < selected_count
            },
            reverse=True,
        )
        if not lower_counts:
            break

        # Compare against the best state at the next lower retained count.
        lower_count = lower_counts[0]
        lower = max(
            (
                state for state in eligible
                if state.get("path")
                and int(state["path"][0]["transfer_count"]) == lower_count
            ),
            key=lambda state: float(state["final_objective"]),
        )
        extra = selected_count - lower_count
        gain = (
            float(selected["final_objective"])
            - float(lower["final_objective"])
        )
        required = float(min_gain_per_extra_transfer) * float(extra)

        if gain + 1e-12 < required:
            steps.append({
                "from_transfer_count": selected_count,
                "to_transfer_count": lower_count,
                "incremental_gain": gain,
                "required_gain": required,
            })
            selected = lower
            continue
        break

    audit = {
        "raw_transfer_count": int(raw["path"][0]["transfer_count"]),
        "selected_transfer_count": int(
            selected["path"][0]["transfer_count"]
        ),
        "raw_objective": float(raw["final_objective"]),
        "selected_objective": float(selected["final_objective"]),
        "objective_given_up": (
            float(raw["final_objective"])
            - float(selected["final_objective"])
        ),
        "steps": steps,
        "applied": selected is not raw,
    }
    return selected, audit

### Run the transfer planner

When `CURRENT_SQUAD_PLAYER_IDS` is empty, the section uses the optimised initial squad as a technical demonstration. During the season, enter the actual squad IDs, current bank, free transfers and any selling prices that differ from market price.

The first row of `recommended_transfer_plan` is the only action intended for execution. Later rows explain the current preferred path but will be recalculated before their deadlines.


In [ ]:
if CURRENT_SQUAD_PLAYER_IDS:
    transfer_planner_squad_ids = [
        int(player_id)
        for player_id in CURRENT_SQUAD_PLAYER_IDS
    ]
    transfer_planner_bank = float(CURRENT_MONEY_IN_BANK)
    transfer_planner_mode = "Configured live squad"
else:
    transfer_planner_squad_ids = (
        optimised_squad["player_id"].astype(int).tolist()
    )
    transfer_planner_bank = max(
        0.0,
        INITIAL_SQUAD_BUDGET - float(optimised_squad["price"].sum()),
    )
    transfer_planner_mode = "Preseason demonstration using optimised squad"

transfer_planner_result = optimise_transfer_path(
    current_squad_ids=transfer_planner_squad_ids,
    candidates=optimisation_candidates,
    projections=optimisation_projections,
    gameweeks_to_use=optimisation_gameweeks,
    money_in_bank=transfer_planner_bank,
    free_transfers=CURRENT_FREE_TRANSFERS,
    current_selling_prices=CURRENT_SQUAD_SELLING_PRICES,
)

best_transfer_state = transfer_planner_result["best_state"]
transfer_candidate_pool = transfer_planner_result["candidate_pool"]

candidate_name_lookup = (
    optimisation_candidates
    .set_index("player_id")["web_name"]
    .astype(str)
    .to_dict()
)

plan_rows = []
for path_row in best_transfer_state["path"]:
    transfer_labels = []
    for outgoing_id, incoming_id in path_row["transfers"]:
        transfer_labels.append(
            f"{candidate_name_lookup.get(outgoing_id, outgoing_id)} → "
            f"{candidate_name_lookup.get(incoming_id, incoming_id)}"
        )

    plan_rows.append({
        "gameweek": path_row["gameweek"],
        "transfers": "; ".join(transfer_labels) if transfer_labels else "Roll",
        "transfer_count": path_row["transfer_count"],
        "free_transfers_before": path_row["free_transfers_before"],
        "free_transfers_after": path_row["free_transfers_after"],
        "hit_cost": path_row["hit_cost"],
        "bank_before": tenths_to_price(path_row["bank_before_tenths"]),
        "bank_after": tenths_to_price(path_row["bank_after_tenths"]),
        "expected_score_before_hits": path_row["expected_score_before_hits"],
        "expected_score_after_hits": path_row["expected_score_after_hits"],
    })

recommended_transfer_plan = pd.DataFrame(plan_rows)
recommended_current_transfer = recommended_transfer_plan.iloc[0].to_dict()

# Compare distinct immediate actions after allowing the later path to optimise.
comparison_rows = []
for state in transfer_planner_result["final_beam"]:
    first_row = state["path"][0]
    first_transfers = tuple(first_row["transfers"])
    transfer_labels = [
        f"{candidate_name_lookup.get(outgoing_id, outgoing_id)} → "
        f"{candidate_name_lookup.get(incoming_id, incoming_id)}"
        for outgoing_id, incoming_id in first_transfers
    ]
    comparison_rows.append({
        "first_action_key": first_transfers,
        "current_action": (
            "; ".join(transfer_labels) if transfer_labels else "Roll"
        ),
        "current_transfer_count": first_row["transfer_count"],
        "current_hit_cost": first_row["hit_cost"],
        "bank_after": tenths_to_price(first_row["bank_after_tenths"]),
        "path_objective": state["final_objective"],
    })

transfer_action_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values("path_objective", ascending=False)
    .drop_duplicates("first_action_key")
    .drop(columns="first_action_key")
    .head(10)
    .reset_index(drop=True)
)

roll_rows = transfer_action_comparison.loc[
    transfer_action_comparison["current_action"].eq("Roll")
]
roll_path_objective = (
    float(roll_rows.iloc[0]["path_objective"])
    if not roll_rows.empty
    else np.nan
)
recommended_path_gain_vs_roll = (
    float(best_transfer_state["final_objective"] - roll_path_objective)
    if np.isfinite(roll_path_objective)
    else np.nan
)

# Produce the detailed current-GW team after the first transfer decision.
first_path_row = best_transfer_state["path"][0]
post_transfer_squad_ids = list(first_path_row["squad_ids_after"])
post_transfer_squad = (
    optimisation_candidates.loc[
        optimisation_candidates["player_id"].isin(post_transfer_squad_ids)
    ]
    .copy()
)

first_decision = first_path_row["lineup_decision"]
post_transfer_projected_lineup = (
    post_transfer_squad.loc[
        post_transfer_squad["player_id"].isin(first_decision["starter_ids"])
    ]
    .merge(
        optimisation_projections.loc[
            optimisation_projections["gameweek"].eq(
                first_path_row["gameweek"]
            ),
            ["player_id", "expected_points"],
        ],
        on="player_id",
        how="left",
        validate="one_to_one",
    )
)

post_transfer_gameweek_projection = (
    optimisation_projections.loc[
        optimisation_projections["gameweek"].eq(first_path_row["gameweek"]),
        [
            "player_id",
            "expected_points",
            "expected_minutes",
            "appearance_probability",
            "no_appearance_probability",
        ],
    ]
    .copy()
)

post_transfer_current_decision = optimise_current_gameweek_details(
    post_transfer_squad,
    post_transfer_projected_lineup,
    post_transfer_gameweek_projection,
)

assert len(post_transfer_squad_ids) == 15
post_transfer_candidate_lookup = {
    int(row.player_id): row._asdict()
    for row in optimisation_candidates.itertuples(index=False)
}
assert squad_structure_is_legal(
    post_transfer_squad_ids,
    post_transfer_candidate_lookup,
)
starting_excess = squad_club_excess(
    transfer_planner_squad_ids,
    post_transfer_candidate_lookup,
)
ending_excess = squad_club_excess(
    post_transfer_squad_ids,
    post_transfer_candidate_lookup,
)
if first_path_row["transfer_count"] > 0 and starting_excess > 0:
    assert ending_excess < starting_excess
elif starting_excess == 0:
    assert ending_excess == 0
assert recommended_transfer_plan["free_transfers_after"].between(1, 5).all()
assert recommended_transfer_plan["bank_after"].ge(-1e-9).all()

print("Transfer-planner mode:", transfer_planner_mode)
if not CURRENT_SQUAD_PLAYER_IDS:
    print(
        "Preseason note: transfers are unlimited before Gameweek 1; "
        "this path is a systems test, not an action recommendation."
    )
print(
    "Best discounted transfer-path objective:",
    f"{best_transfer_state['final_objective']:.2f}",
)
print(
    "Projected horizon search score before hits:",
    f"{best_transfer_state['cumulative_expected_points']:.2f}",
)
print(
    "Projected transfer-hit cost:",
    f"{best_transfer_state['cumulative_hits']:.1f}",
)

display(recommended_transfer_plan)

print("Best immediate actions after optimising later Gameweeks:")
display(transfer_action_comparison)
if np.isfinite(recommended_path_gain_vs_roll):
    print(
        "Recommended path advantage over rolling now:",
        f"{recommended_path_gain_vs_roll:+.2f} objective points",
    )

print(
    "Detailed current-Gameweek expected score after transfers, "
    "captaincy and automatic substitutions:",
    f"{post_transfer_current_decision['expected_total_score'] - first_path_row['hit_cost']:.2f}",
)


### Result of the transfer stage

The principal outputs are:

```python
recommended_current_transfer
```

The transfer action to consider at the imminent deadline.

```python
recommended_transfer_plan
```

The current preferred path over the six-Gameweek horizon. Only its first row is actionable; all later decisions must be re-optimised with updated information.

```python
post_transfer_current_decision
```

The detailed starting XI, bench-order and captaincy state after applying the recommended immediate transfer action.

Current limitations:

- future prices are held constant;
- at most two transfers are considered in a single Gameweek;
- chips are not yet included in the transfer state;
- the beam search is approximate and can miss a globally optimal path;
- the path-search score uses a fast bench-depth approximation, while the final immediate recommendation uses the full automatic-substitution calculation.

The next stage is chip evaluation: Triple Captain, Bench Boost, Free Hit and Wildcard uplift relative to the best non-chip path.


## 10. Chip evaluation and timing

This section evaluates each chip by its **incremental expected value relative to the best no-chip path**.

The 2026/27 game provides two sets of four chips:

- Triple Captain;
- Bench Boost;
- Free Hit;
- Wildcard.

The first set expires at the Gameweek 19 deadline and cannot be carried into the second half of the season. Only one chip can be active in a Gameweek. Free Hit and Wildcard are unavailable in Gameweek 1, and Free Hit cannot be used in consecutive Gameweeks.

The calculations below distinguish two kinds of uplift:

1. **One-Gameweek uplift**
   - Triple Captain;
   - Bench Boost;
   - Free Hit.

2. **Remaining-horizon uplift**
   - Wildcard.

The analysis ranks opportunities inside the current planning horizon. Unless the chip-expiry deadline is inside that horizon, it should not be interpreted as proof that the best opportunity of the entire half-season has already appeared.


### Audit of the current output

The executed notebook provides a sound basis for moving to chip evaluation:

- the component expected-points model improves on the rolling baseline on the chronological holdout;
- the team-strength model improves on the home/away scoring baseline;
- the transfer planner correctly preserves a no-transfer comparator;
- in the current pre-season demonstration, rolling the transfer is tied for the strongest immediate action.

There is, however, an important modelling warning. The live ranking is heavily defender-dominated and currently selects a defender as captain. The holdout results show that forwards are underpredicted on average, while defenders are slightly overpredicted. This does not invalidate line-up optimisation, but it makes early Triple Captain and attacking-ceiling conclusions less reliable.

The diagnostics below therefore flag unusual positional concentration and label all chip recommendations as provisional until current-season attacking data becomes available.


In [ ]:
# Chip-state inputs ----------------------------------------------------------
#
# Update these values after playing a chip. They refer to the chip set that is
# currently active: the first-half set before GW19, or the refreshed second-half
# set from GW20 onward.

CURRENT_AVAILABLE_CHIPS = {
    "triple_captain": True,
    "bench_boost": True,
    "free_hit": True,
    "wildcard": True,
}

# Set this to the most recent Gameweek in which Free Hit was played, if any.
# It is used to prevent consecutive Free Hits.
LAST_FREE_HIT_GAMEWEEK = None

FIRST_CHIP_SET_EXPIRY_GAMEWEEK = 19
SECOND_CHIP_SET_EXPIRY_GAMEWEEK = 38

# Candidate-pool controls. Each chip optimiser keeps high-projection players
# plus cheap structural options so the budget problem remains feasible.
CHIP_TOP_PROJECTED_PER_POSITION = 35
CHIP_CHEAP_OPTIONS_PER_POSITION = 15

# Transparent conservative thresholds for an automatic "use now" suggestion.
# The tables are still useful when no threshold is reached.
CHIP_MINIMUM_UPLIFT = {
    "triple_captain": 7.0,
    "bench_boost": 9.0,
    "free_hit": 8.0,
    "wildcard": 10.0,
}
CHIP_CURRENT_VS_HORIZON_RATIO = 0.95

# The Wildcard comparison values a fixed replacement squad over the remaining
# explicit horizon. It does not yet run a second transfer beam after wildcarding.


### Core chip-scoring helpers

The no-chip score uses the same squad, automatic-substitution calculation, captain and vice-captain logic as the preceding optimiser.

- **Triple Captain uplift** is one additional expected copy of the captain or vice-captain score.
- **Bench Boost uplift** is the expected score of all four nominal bench players minus the automatic-substitution value already present in the normal score.
- **Free Hit uplift** compares the best affordable one-week squad with the no-chip path, including any transfer hits avoided.
- **Wildcard uplift** compares an optimised permanent squad over the remaining horizon with the current transfer path over the same horizon.


In [ ]:
def exact_squad_gameweek_decision(
    squad_ids,
    gameweek,
    candidates,
    projections,
):
    """Return the detailed expected score for one fixed legal squad."""
    squad_ids = tuple(sorted(int(player_id) for player_id in squad_ids))

    selected_squad = (
        candidates.loc[candidates["player_id"].isin(squad_ids)]
        .drop_duplicates("player_id")
        .copy()
    )
    if len(selected_squad) != 15:
        raise ValueError(
            f"Expected 15 squad players, found {len(selected_squad)}."
        )

    candidate_lookup = {
        int(row.player_id): row._asdict()
        for row in candidates.itertuples(index=False)
    }
    projection_lookup = build_projection_lookup(projections)

    fast_decision = fast_squad_gameweek_decision(
        squad_ids,
        int(gameweek),
        candidate_lookup,
        projection_lookup,
    )

    gameweek_projection = (
        projections.loc[
            projections["gameweek"].eq(int(gameweek))
            & projections["player_id"].isin(squad_ids),
            [
                "player_id",
                "expected_points",
                "expected_minutes",
                "appearance_probability",
                "no_appearance_probability",
            ],
        ]
        .drop_duplicates("player_id")
        .copy()
    )

    projected_lineup = (
        selected_squad.loc[
            selected_squad["player_id"].isin(
                fast_decision["starter_ids"]
            )
        ]
        .merge(
            gameweek_projection[["player_id", "expected_points"]],
            on="player_id",
            how="left",
            validate="one_to_one",
        )
    )

    detailed = optimise_current_gameweek_details(
        selected_squad,
        projected_lineup,
        gameweek_projection,
    )
    detailed["fast_lineup_decision"] = fast_decision
    detailed["gameweek"] = int(gameweek)
    return detailed


def triple_captain_uplift(detailed_decision):
    """
    A normal captain already contributes one extra copy of their points.
    Triple Captain contributes one additional copy, including vice fallback.
    """
    return float(detailed_decision["expected_captain_uplift"])


def bench_boost_score(detailed_decision):
    """Value all 15 squad members while retaining normal captaincy."""
    all_player_xp = float(
        detailed_decision["squad_projection"]["expected_points"].sum()
    )
    return (
        all_player_xp
        + float(detailed_decision["expected_captain_uplift"])
    )


def bench_boost_uplift(detailed_decision):
    return float(
        bench_boost_score(detailed_decision)
        - detailed_decision["expected_total_score"]
    )


def chip_candidate_subset(
    candidates,
    projections,
    gameweeks_to_use,
    required_player_ids=(),
):
    """
    Keep strong projected options and enough cheap players for budget structure.
    """
    gameweeks_to_use = [int(gameweek) for gameweek in gameweeks_to_use]
    required_player_ids = {
        int(player_id) for player_id in required_player_ids
    }

    weighted = (
        projections.loc[
            projections["gameweek"].isin(gameweeks_to_use),
            ["player_id", "gameweek", "expected_points"],
        ]
        .assign(
            horizon_index=lambda frame: frame["gameweek"].map(
                {
                    gameweek: index
                    for index, gameweek in enumerate(gameweeks_to_use)
                }
            ),
            weighted_xp=lambda frame: (
                frame["expected_points"]
                * GAMEWEEK_DISCOUNT ** frame["horizon_index"]
            ),
        )
        .groupby("player_id", as_index=False)["weighted_xp"]
        .sum()
    )

    ranked = (
        candidates
        .merge(weighted, on="player_id", how="left")
        .assign(weighted_xp=lambda frame: frame["weighted_xp"].fillna(0.0))
    )

    kept_ids = set(required_player_ids)
    for _position, group in ranked.groupby("position"):
        kept_ids.update(
            group.nlargest(
                CHIP_TOP_PROJECTED_PER_POSITION,
                "weighted_xp",
            )["player_id"].astype(int)
        )
        kept_ids.update(
            group.nsmallest(
                CHIP_CHEAP_OPTIONS_PER_POSITION,
                "price",
            )["player_id"].astype(int)
        )

    subset_candidates = (
        ranked.loc[ranked["player_id"].isin(kept_ids)]
        .drop(columns="weighted_xp")
        .drop_duplicates("player_id")
        .reset_index(drop=True)
    )
    subset_projections = projections.loc[
        projections["player_id"].isin(subset_candidates["player_id"])
        & projections["gameweek"].isin(gameweeks_to_use)
    ].copy()

    return subset_candidates, subset_projections


def optimise_chip_squad(
    candidates,
    projections,
    gameweeks_to_use,
    budget,
    required_player_ids=(),
):
    """Optimise a legal 15-player squad for a chip-specific horizon."""
    subset_candidates, subset_projections = chip_candidate_subset(
        candidates,
        projections,
        gameweeks_to_use,
        required_player_ids=required_player_ids,
    )

    result = optimise_squad_lineups_and_captains(
        subset_candidates,
        subset_projections,
        list(gameweeks_to_use),
        budget=float(budget),
    )
    result["subset_candidates"] = subset_candidates
    result["subset_projections"] = subset_projections
    return result


def reconstruct_transfer_path_finances(
    best_state,
    initial_squad_ids,
    initial_bank,
    candidates,
    configured_selling_prices=None,
):
    """
    Reconstruct the selling-price budget available immediately before each
    baseline Gameweek action, assuming the notebook's constant-price baseline.
    """
    configured_selling_prices = configured_selling_prices or {}
    price_lookup = (
        candidates.set_index("player_id")["price"].astype(float).to_dict()
    )

    current_squad = tuple(sorted(int(pid) for pid in initial_squad_ids))
    current_bank_tenths = price_to_tenths(initial_bank)
    selling_prices = {
        int(player_id): price_to_tenths(
            configured_selling_prices.get(
                int(player_id),
                price_lookup[int(player_id)],
            )
        )
        for player_id in current_squad
    }

    contexts = []
    for path_row in best_state["path"]:
        expected_before = tuple(
            sorted(int(pid) for pid in path_row["squad_ids_before"])
        )
        if expected_before != current_squad:
            raise AssertionError(
                "Transfer-path squad state is inconsistent before "
                f"Gameweek {path_row['gameweek']}."
            )

        total_budget_tenths = (
            current_bank_tenths
            + sum(selling_prices[player_id] for player_id in current_squad)
        )

        contexts.append({
            "gameweek": int(path_row["gameweek"]),
            "path_row": path_row,
            "squad_ids_before": current_squad,
            "selling_prices_before_tenths": dict(selling_prices),
            "bank_before_tenths": current_bank_tenths,
            "available_budget": tenths_to_price(total_budget_tenths),
        })

        for outgoing_id, incoming_id in path_row["transfers"]:
            outgoing_id = int(outgoing_id)
            incoming_id = int(incoming_id)
            selling_prices.pop(outgoing_id, None)
            selling_prices[incoming_id] = price_to_tenths(
                price_lookup[incoming_id]
            )

        current_squad = tuple(
            sorted(int(pid) for pid in path_row["squad_ids_after"])
        )
        current_bank_tenths = int(path_row["bank_after_tenths"])

    return contexts


def chip_is_eligible(chip_name, gameweek):
    """Apply the principal chip-availability restrictions."""
    gameweek = int(gameweek)
    if not CURRENT_AVAILABLE_CHIPS.get(chip_name, False):
        return False

    if chip_name in {"free_hit", "wildcard"} and gameweek == 1:
        return False

    if (
        chip_name == "free_hit"
        and LAST_FREE_HIT_GAMEWEEK is not None
        and gameweek == int(LAST_FREE_HIT_GAMEWEEK) + 1
    ):
        return False

    return True


### Fixture-shape and model-readiness diagnostics

Raw expected uplift already reflects blanks and doubles because projections are aggregated by player and Gameweek. The summary below is included so unusually valuable chip weeks can be interpreted rather than treated as unexplained optimiser output.


In [ ]:
def build_fixture_shape_summary(
    fixture_level_predictions,
    teams_table,
    gameweeks_to_use,
):
    """Count blank, single and double-fixture clubs in each Gameweek."""
    team_counts = (
        fixture_level_predictions.loc[
            fixture_level_predictions["gameweek"].isin(gameweeks_to_use)
        ]
        .groupby(["gameweek", "team_id"])["fixture_id"]
        .nunique()
        .rename("fixture_count")
        .reset_index()
    )

    full_index = pd.MultiIndex.from_product(
        [
            [int(gameweek) for gameweek in gameweeks_to_use],
            teams_table["team_id"].astype(int).unique(),
        ],
        names=["gameweek", "team_id"],
    )

    team_counts = (
        team_counts.set_index(["gameweek", "team_id"])
        .reindex(full_index, fill_value=0)
        .reset_index()
        .merge(
            teams_table[
                ["team_id", "team_short_name"]
            ].drop_duplicates("team_id"),
            on="team_id",
            how="left",
            validate="many_to_one",
        )
    )

    rows = []
    for gameweek, group in team_counts.groupby("gameweek"):
        rows.append({
            "gameweek": int(gameweek),
            "blank_teams": int((group["fixture_count"] == 0).sum()),
            "single_fixture_teams": int((group["fixture_count"] == 1).sum()),
            "double_or_more_teams": int((group["fixture_count"] >= 2).sum()),
            "blank_team_names": ", ".join(
                group.loc[
                    group["fixture_count"].eq(0),
                    "team_short_name",
                ].dropna().astype(str)
            ),
            "double_team_names": ", ".join(
                group.loc[
                    group["fixture_count"].ge(2),
                    "team_short_name",
                ].dropna().astype(str)
            ),
        })
    return pd.DataFrame(rows)


fixture_shape_summary = build_fixture_shape_summary(
    live_expected_points,
    teams,
    optimisation_gameweeks,
)

next_gameweek = int(min(optimisation_gameweeks))
next_gameweek_top = (
    live_gameweek_expected_points.loc[
        live_gameweek_expected_points["gameweek"].eq(next_gameweek)
    ]
    .nlargest(20, "expected_points")
)

top_20_defender_share = float(
    next_gameweek_top["position"].eq("DEF").mean()
) if not next_gameweek_top.empty else np.nan

best_projected_player = (
    next_gameweek_top.iloc[0]
    if not next_gameweek_top.empty
    else None
)

current_season_completed_rows = (
    len(current_points_history)
    if "current_points_history" in globals()
    else 0
)

chip_model_warnings = []
if current_season_completed_rows == 0:
    chip_model_warnings.append(
        "No completed current-season player rows are available yet."
    )
if np.isfinite(top_20_defender_share) and top_20_defender_share >= 0.60:
    chip_model_warnings.append(
        "At least 60% of the top-20 live xP projections are defenders."
    )
if (
    best_projected_player is not None
    and str(best_projected_player["position"]) in {"GK", "DEF"}
):
    chip_model_warnings.append(
        "The highest projected next-Gameweek player is a goalkeeper or defender."
    )

display(fixture_shape_summary)

print("Chip-model readiness diagnostics:")
if chip_model_warnings:
    for warning in chip_model_warnings:
        print("-", warning)
else:
    print("- No automated warning was triggered.")


### Evaluate the four chips across the rolling horizon

The baseline is the currently preferred transfer path. For each projected Gameweek:

- Triple Captain and Bench Boost use the baseline permanent squad after that week's planned normal transfers.
- Free Hit uses the selling-value budget available before that week's normal transfers and restores the permanent squad afterwards.
- Wildcard uses the same pre-transfer budget but creates a new permanent squad for the remaining explicit horizon.

The Free Hit and Wildcard calculations are alternatives to that Gameweek's planned transfers, so normal transfer hits in the baseline are avoided.


In [ ]:
baseline_path_contexts = reconstruct_transfer_path_finances(
    best_transfer_state,
    transfer_planner_squad_ids,
    transfer_planner_bank,
    optimisation_candidates,
    configured_selling_prices=CURRENT_SQUAD_SELLING_PRICES,
)

# Exact no-chip scores for the preferred transfer path.
baseline_exact_rows = []
for context in baseline_path_contexts:
    path_row = context["path_row"]
    decision = exact_squad_gameweek_decision(
        path_row["squad_ids_after"],
        context["gameweek"],
        optimisation_candidates,
        optimisation_projections,
    )

    baseline_exact_rows.append({
        **context,
        "baseline_decision": decision,
        "baseline_score_before_hits": float(
            decision["expected_total_score"]
        ),
        "baseline_hit_cost": float(path_row["hit_cost"]),
        "baseline_score_after_hits": float(
            decision["expected_total_score"] - path_row["hit_cost"]
        ),
    })

baseline_exact_lookup = {
    int(row["gameweek"]): row
    for row in baseline_exact_rows
}

chip_rows = []
chip_squad_details = {}

for horizon_index, context in enumerate(baseline_exact_rows):
    gameweek = int(context["gameweek"])
    baseline_decision = context["baseline_decision"]
    baseline_score = float(context["baseline_score_after_hits"])
    baseline_squad_ids = tuple(
        int(pid) for pid in context["path_row"]["squad_ids_after"]
    )

    captain_id = int(baseline_decision["captain_id"])
    vice_id = int(baseline_decision["vice_captain_id"])

    triple_uplift = (
        triple_captain_uplift(baseline_decision)
        if chip_is_eligible("triple_captain", gameweek)
        else np.nan
    )
    bench_uplift = (
        bench_boost_uplift(baseline_decision)
        if chip_is_eligible("bench_boost", gameweek)
        else np.nan
    )

    free_hit_score = np.nan
    free_hit_uplift = np.nan
    free_hit_squad_ids = ()
    free_hit_captain_id = None

    if chip_is_eligible("free_hit", gameweek):
        free_hit_result = optimise_chip_squad(
            optimisation_candidates,
            optimisation_projections,
            [gameweek],
            budget=context["available_budget"],
            required_player_ids=baseline_squad_ids,
        )
        free_hit_squad = free_hit_result["squad"].copy()
        free_hit_squad_ids = tuple(
            free_hit_squad["player_id"].astype(int)
        )
        free_hit_decision = exact_squad_gameweek_decision(
            free_hit_squad_ids,
            gameweek,
            free_hit_result["subset_candidates"],
            free_hit_result["subset_projections"],
        )
        free_hit_score = float(
            free_hit_decision["expected_total_score"]
        )
        free_hit_uplift = free_hit_score - baseline_score
        free_hit_captain_id = int(free_hit_decision["captain_id"])

        chip_squad_details[("free_hit", gameweek)] = {
            "squad": free_hit_squad,
            "decision": free_hit_decision,
            "budget": context["available_budget"],
        }

    wildcard_horizon_score = np.nan
    wildcard_baseline_score = np.nan
    wildcard_uplift = np.nan
    wildcard_current_score = np.nan
    wildcard_squad_ids = ()

    if chip_is_eligible("wildcard", gameweek):
        remaining_gameweeks = optimisation_gameweeks[horizon_index:]
        wildcard_result = optimise_chip_squad(
            optimisation_candidates,
            optimisation_projections,
            remaining_gameweeks,
            budget=context["available_budget"],
            required_player_ids=baseline_squad_ids,
        )
        wildcard_squad = wildcard_result["squad"].copy()
        wildcard_squad_ids = tuple(
            wildcard_squad["player_id"].astype(int)
        )

        wildcard_scores = []
        for remaining_index, remaining_gameweek in enumerate(
            remaining_gameweeks
        ):
            wildcard_decision = exact_squad_gameweek_decision(
                wildcard_squad_ids,
                remaining_gameweek,
                wildcard_result["subset_candidates"],
                wildcard_result["subset_projections"],
            )
            wildcard_scores.append({
                "gameweek": int(remaining_gameweek),
                "score": float(wildcard_decision["expected_total_score"]),
                "discount": GAMEWEEK_DISCOUNT ** remaining_index,
            })

        wildcard_horizon_score = float(sum(
            row["score"] * row["discount"]
            for row in wildcard_scores
        ))
        wildcard_current_score = float(wildcard_scores[0]["score"])

        baseline_remaining_rows = baseline_exact_rows[horizon_index:]
        wildcard_baseline_score = float(sum(
            row["baseline_score_after_hits"]
            * GAMEWEEK_DISCOUNT ** relative_index
            for relative_index, row in enumerate(
                baseline_remaining_rows
            )
        ))
        wildcard_uplift = (
            wildcard_horizon_score - wildcard_baseline_score
        )

        chip_squad_details[("wildcard", gameweek)] = {
            "squad": wildcard_squad,
            "scores": pd.DataFrame(wildcard_scores),
            "budget": context["available_budget"],
        }

    fixture_shape = fixture_shape_summary.loc[
        fixture_shape_summary["gameweek"].eq(gameweek)
    ]
    fixture_shape_row = (
        fixture_shape.iloc[0].to_dict()
        if not fixture_shape.empty
        else {}
    )

    chip_rows.append({
        "gameweek": gameweek,
        "available_budget": context["available_budget"],
        "baseline_score_after_hits": baseline_score,
        "baseline_hit_cost": context["baseline_hit_cost"],
        "captain_id": captain_id,
        "vice_captain_id": vice_id,
        "triple_captain_uplift": triple_uplift,
        "triple_captain_score": (
            baseline_score + triple_uplift
            if np.isfinite(triple_uplift)
            else np.nan
        ),
        "bench_boost_uplift": bench_uplift,
        "bench_boost_score": (
            baseline_score + bench_uplift
            if np.isfinite(bench_uplift)
            else np.nan
        ),
        "free_hit_uplift": free_hit_uplift,
        "free_hit_score": free_hit_score,
        "free_hit_captain_id": free_hit_captain_id,
        "wildcard_horizon_uplift": wildcard_uplift,
        "wildcard_current_score": wildcard_current_score,
        "wildcard_horizon_score": wildcard_horizon_score,
        "baseline_remaining_horizon_score": wildcard_baseline_score,
        "blank_teams": fixture_shape_row.get("blank_teams", np.nan),
        "double_or_more_teams": fixture_shape_row.get(
            "double_or_more_teams",
            np.nan,
        ),
    })

chip_opportunity_table = pd.DataFrame(chip_rows)

player_name_lookup = (
    optimisation_candidates
    .set_index("player_id")["web_name"]
    .astype(str)
    .to_dict()
)
chip_opportunity_table["captain"] = (
    chip_opportunity_table["captain_id"]
    .map(player_name_lookup)
)
chip_opportunity_table["free_hit_captain"] = (
    chip_opportunity_table["free_hit_captain_id"]
    .map(player_name_lookup)
)

display_columns = [
    "gameweek",
    "baseline_score_after_hits",
    "captain",
    "triple_captain_uplift",
    "bench_boost_uplift",
    "free_hit_uplift",
    "wildcard_horizon_uplift",
    "blank_teams",
    "double_or_more_teams",
]

display(
    chip_opportunity_table[display_columns]
    .round(3)
)


### Rank opportunities and make a provisional current decision

The automatic suggestion is deliberately conservative:

1. the chip must be available and legal now;
2. current uplift must reach the configured minimum;
3. current uplift must be at least 95% of that chip's best uplift in the explicit horizon.

If no chip passes these conditions, the recommendation is to save all chips and reconsider next Gameweek.


In [ ]:
CHIP_UPLIFT_COLUMNS = {
    "triple_captain": "triple_captain_uplift",
    "bench_boost": "bench_boost_uplift",
    "free_hit": "free_hit_uplift",
    "wildcard": "wildcard_horizon_uplift",
}

best_chip_rows = []
for chip_name, uplift_column in CHIP_UPLIFT_COLUMNS.items():
    valid = chip_opportunity_table.loc[
        chip_opportunity_table[uplift_column].notna()
    ]
    if valid.empty:
        continue

    best_index = valid[uplift_column].idxmax()
    best_row = valid.loc[best_index]
    best_chip_rows.append({
        "chip": chip_name,
        "best_gameweek_in_horizon": int(best_row["gameweek"]),
        "best_uplift_in_horizon": float(best_row[uplift_column]),
        "available_now": bool(
            chip_is_eligible(chip_name, next_gameweek)
        ),
    })

best_chip_opportunities = (
    pd.DataFrame(best_chip_rows)
    .sort_values("best_uplift_in_horizon", ascending=False)
    .reset_index(drop=True)
)

current_chip_rows = []
current_row = chip_opportunity_table.loc[
    chip_opportunity_table["gameweek"].eq(next_gameweek)
].iloc[0]

for chip_name, uplift_column in CHIP_UPLIFT_COLUMNS.items():
    best_match = best_chip_opportunities.loc[
        best_chip_opportunities["chip"].eq(chip_name)
    ]
    best_uplift = (
        float(best_match.iloc[0]["best_uplift_in_horizon"])
        if not best_match.empty
        else np.nan
    )
    current_uplift = float(current_row[uplift_column])
    threshold = float(CHIP_MINIMUM_UPLIFT[chip_name])

    ratio_to_best = (
        current_uplift / best_uplift
        if np.isfinite(current_uplift)
        and np.isfinite(best_uplift)
        and best_uplift > 1e-9
        else np.nan
    )

    passes = (
        chip_is_eligible(chip_name, next_gameweek)
        and np.isfinite(current_uplift)
        and current_uplift >= threshold
        and np.isfinite(ratio_to_best)
        and ratio_to_best >= CHIP_CURRENT_VS_HORIZON_RATIO
    )

    current_chip_rows.append({
        "chip": chip_name,
        "eligible_now": chip_is_eligible(chip_name, next_gameweek),
        "current_uplift": current_uplift,
        "minimum_uplift": threshold,
        "best_horizon_uplift": best_uplift,
        "current_vs_horizon_best": ratio_to_best,
        "passes_use_now_rule": passes,
        "normalised_strength": (
            current_uplift / threshold
            if np.isfinite(current_uplift) and threshold > 0
            else np.nan
        ),
    })

current_chip_comparison = pd.DataFrame(current_chip_rows)

use_now_candidates = current_chip_comparison.loc[
    current_chip_comparison["passes_use_now_rule"]
].sort_values("normalised_strength", ascending=False)

if use_now_candidates.empty:
    provisional_chip_recommendation = {
        "gameweek": next_gameweek,
        "action": "Save all chips",
        "chip": None,
        "reason": (
            "No available chip clears both the conservative uplift "
            "threshold and the within-horizon timing test."
        ),
    }
else:
    selected = use_now_candidates.iloc[0]
    provisional_chip_recommendation = {
        "gameweek": next_gameweek,
        "action": f"Consider {selected['chip'].replace('_', ' ').title()}",
        "chip": selected["chip"],
        "expected_uplift": float(selected["current_uplift"]),
        "reason": (
            "It clears the configured uplift threshold and is within "
            f"{CHIP_CURRENT_VS_HORIZON_RATIO:.0%} of its best projected "
            "opportunity in the current horizon."
        ),
    }

current_half_expiry = (
    FIRST_CHIP_SET_EXPIRY_GAMEWEEK
    if next_gameweek <= FIRST_CHIP_SET_EXPIRY_GAMEWEEK
    else SECOND_CHIP_SET_EXPIRY_GAMEWEEK
)
expiry_inside_horizon = (
    current_half_expiry in optimisation_gameweeks
)

display(best_chip_opportunities.round(3))
display(current_chip_comparison.round(3))

print("Provisional chip recommendation:")
print(provisional_chip_recommendation)
print(
    "Current chip-set expiry is inside the explicit horizon:",
    expiry_inside_horizon,
)
if not expiry_inside_horizon:
    print(
        "The model has not yet compared these opportunities with all "
        f"Gameweeks up to GW{current_half_expiry}."
    )
if chip_model_warnings:
    print(
        "Caution: model-readiness warnings are active, so chip conclusions "
        "should be treated as provisional."
    )


### Inspect an optimised Free Hit or Wildcard squad

The detailed squads are stored in:

```python
chip_squad_details
```

Examples:

```python
chip_squad_details[("free_hit", gameweek)]["squad"]
chip_squad_details[("wildcard", gameweek)]["squad"]
```

The main outputs of this stage are:

```python
chip_opportunity_table
```

Expected uplift for every chip and projected Gameweek.

```python
best_chip_opportunities
```

The strongest projected use of each available chip inside the rolling horizon.

```python
current_chip_comparison
```

The current-Gameweek decision table.

```python
provisional_chip_recommendation
```

The conservative use-now or save recommendation.

Current limitations:

- chip timing is compared only across the explicit rolling horizon;
- the Wildcard comparison holds the replacement squad fixed after wildcarding rather than launching a second transfer beam;
- Free Hit and Wildcard use constant future prices;
- automatic substitutions assume independent player appearances;
- the live attacking model remains conservative, which particularly affects Triple Captain;
- only one chip is evaluated at a time, while future versions should optimise the full sequence of chip availability and expiry.


In [ ]:
def chip_structural_readiness(
    chip_name: str,
    *,
    current_uplift: float,
    blank_teams: int,
    double_teams: int,
    expiry_pressure: bool,
    normal_transfer_count: int,
    normal_hit_cost: float,
    exceptional_thresholds: dict[str, float],
) -> tuple[bool, str]:
    """Single structural chip rule shared by production and backtests."""
    exceptional = (
        np.isfinite(current_uplift)
        and current_uplift >= float(exceptional_thresholds[chip_name])
    )
    if chip_name == "triple_captain":
        passed = double_teams > 0 or expiry_pressure or exceptional
        reason = "double Gameweek, expiry pressure or exceptional captain uplift"
    elif chip_name == "bench_boost":
        passed = double_teams > 0 or expiry_pressure or exceptional
        reason = "double Gameweek, expiry pressure or exceptional bench uplift"
    elif chip_name == "free_hit":
        passed = blank_teams > 0 or double_teams > 0 or expiry_pressure or exceptional
        reason = "blank/double Gameweek, expiry pressure or exceptional uplift"
    elif chip_name == "wildcard":
        # v2.6: the structural gate is deliberately NOT applied to the
        # wildcard. Its value is squad quality, not fixture structure, and
        # requiring a double/blank/expiry trigger cost about 40 points a
        # season in validation (99.0 -> 59.0 realised, no better than
        # random timing). CHIP_MINIMUM_UPLIFT['wildcard'] remains the gate.
        if globals().get("V26_WILDCARD_THRESHOLD_ONLY", True):
            passed = True
            reason = "uplift threshold alone (v2.6 validated)"
        else:
            passed = (
                expiry_pressure
                or exceptional
                or int(normal_transfer_count) >= 2
                or float(normal_hit_cost) > 0
            )
            reason = "expiry pressure, major squad surgery or exceptional uplift"
    else:
        raise ValueError(f"Unknown chip: {chip_name}")
    return bool(passed), reason


def historical_fixture_shape_counts(
    backtest_table: pd.DataFrame,
    gameweek: int,
) -> tuple[int, int]:
    counts = (
        backtest_table.loc[backtest_table["gameweek"].eq(int(gameweek))]
        .groupby("team_id")["fixture_count"]
        .max()
    )
    # A normal PL Gameweek has 20 clubs with one fixture each.
    blank_teams = max(0, 20 - int(counts.index.nunique())) + int(counts.eq(0).sum())
    double_teams = int(counts.gt(1).sum())
    return blank_teams, double_teams


def preserve_free_transfers_after_unlimited_chip(free_transfers_before: int) -> int:
    """Wildcard and Free Hit preserve banked free transfers in 2026/27."""
    return int(free_transfers_before)


# 10A. Strict deadline-frozen sequential backtest

The previous backtest was chronological for the current Gameweek, but its
six-Gameweek planning matrix selected precomputed rows from each future
Gameweek. Those future rows contained rolling features that would only become
available at those later deadlines. That is look-ahead leakage.

This version rebuilds every planning horizon from one information set frozen
at the current deadline:

- player-history features are frozen before the current Gameweek;
- team strength uses only results completed before the current Gameweek;
- later fixtures may change opponent, venue, Gameweek and scheduled rest;
- players remain attached to their deadline-time club throughout that
  forecast horizon;
- realised outcomes are used only after the decision has been fixed.

The resulting strategy score is expected to be lower than the earlier
2,430-point estimate, but it is the first result that can be interpreted as a
genuine historical simulation.


In [ ]:
BACKTEST_SEASON = HOLDOUT_SEASON
BACKTEST_HORIZON = 6

BACKTEST_INITIAL_BUDGET = 100.0
BACKTEST_INITIAL_FREE_TRANSFERS = 1
BACKTEST_INCLUDE_CHIPS = True
BACKTEST_APPLY_TRANSFER_ROBUSTNESS_GATE = False
BACKTEST_USE_ADAPTIVE_TRANSFER_SEARCH = True

# Reduced search widths make 38 sequential optimisations practical.
BACKTEST_TRANSFER_BEAM_WIDTH = 10
BACKTEST_TRANSFER_ACTION_BEAM_WIDTH = 18
BACKTEST_TRANSFER_TOP_XP_PER_POSITION = 10
BACKTEST_TRANSFER_TOP_VALUE_PER_POSITION = 6

# Expensive one-off squad optimisations are evaluated selectively.
BACKTEST_WILDCARD_REVIEW_FREQUENCY = 3
BACKTEST_FREE_HIT_ONLY_FOR_FIXTURE_DISRUPTION = True

# Keep players from blank clubs selectable using the latest known pre-deadline
# snapshot. None means "no arbitrary age cut-off"; owned and unowned players
# remain available until a newer snapshot supersedes them.
BACKTEST_MAX_PLAYER_SNAPSHOT_AGE = None

def load_historical_availability_snapshots(
    path: Path = HISTORICAL_AVAILABILITY_SNAPSHOT_PATH,
) -> pd.DataFrame:
    """
    Optional deadline-time FPL status/chance archive.

    Expected columns:
      season, gameweek, player_id, status
    Optional:
      chance_of_playing

    When the file is absent, the backtest records availability as unknown
    instead of pretending historical status was observed.
    """
    columns = [
        "season",
        "gameweek",
        "player_id",
        "status",
        "chance_of_playing",
        "availability_known",
    ]
    if not path.exists():
        return pd.DataFrame(columns=columns)

    frame = pd.read_csv(path, compression="infer")
    required = {"season", "gameweek", "player_id", "status"}
    missing = required - set(frame.columns)
    if missing:
        warnings.warn(
            "Ignoring historical availability snapshot file because it is "
            f"missing columns: {sorted(missing)}"
        )
        return pd.DataFrame(columns=columns)

    if "chance_of_playing" not in frame.columns:
        frame["chance_of_playing"] = np.nan
    frame["availability_known"] = True
    frame["gameweek"] = pd.to_numeric(
        frame["gameweek"], errors="coerce"
    ).astype("Int64")
    frame["player_id"] = pd.to_numeric(
        frame["player_id"], errors="coerce"
    ).astype("Int64")
    return (
        frame[columns]
        .dropna(subset=["gameweek", "player_id"])
        .drop_duplicates(
            ["season", "gameweek", "player_id"],
            keep="last",
        )
        .reset_index(drop=True)
    )

HISTORICAL_AVAILABILITY_SNAPSHOTS = (
    load_historical_availability_snapshots()
)

BACKTEST_CHIP_MINIMUM_UPLIFT = {
    "triple_captain": 7.0,
    "bench_boost": 9.0,
    "free_hit": 8.0,
    "wildcard": 10.0,
}


BACKTEST_CHIP_EXCEPTIONAL_UPLIFT = {
    "triple_captain": 12.0,
    "bench_boost": 15.0,
    "free_hit": 15.0,
    "wildcard": 18.0,
}

BACKTEST_CHIPS_PER_HALF = {
    "wildcard": 1,
    "free_hit": 1,
    "triple_captain": 1,
    "bench_boost": 1,
}


SEQUENTIAL_CACHE_SIGNATURE = stable_configuration_signature({
    "forecast_signature": FORECAST_CACHE_SIGNATURE,
    "project_version": PROJECT_VERSION,
    "backtest_horizon": BACKTEST_HORIZON,
    "include_chips": BACKTEST_INCLUDE_CHIPS,
    "transfer_beam": BACKTEST_TRANSFER_BEAM_WIDTH,
    "adaptive_transfer_search": BACKTEST_USE_ADAPTIVE_TRANSFER_SEARCH,
    "transfer_search_signature": TRANSFER_SEARCH_SIGNATURE,
    "action_beam": BACKTEST_TRANSFER_ACTION_BEAM_WIDTH,
    "chip_thresholds": BACKTEST_CHIP_MINIMUM_UPLIFT,
    "chip_exceptional": BACKTEST_CHIP_EXCEPTIONAL_UPLIFT,
    "chip_readiness": {
        "free_hit_fixture_only": BACKTEST_FREE_HIT_ONLY_FOR_FIXTURE_DISRUPTION,
        "wildcard_review_frequency": BACKTEST_WILDCARD_REVIEW_FREQUENCY,
    },
    "availability_snapshot_rows": int(len(HISTORICAL_AVAILABILITY_SNAPSHOTS)),
    "max_player_snapshot_age": BACKTEST_MAX_PLAYER_SNAPSHOT_AGE,
})


In [ ]:
def build_historical_backtest_table(
    fixture_predictions: pd.DataFrame,
) -> pd.DataFrame:
    """Aggregate leakage-safe fixture predictions and outcomes by player-GW."""
    required = {
        "season",
        "gameweek",
        "player_id_season",
        "player_key",
        "player_name",
        "team_id",
        "team_short_name",
        "model_position",
        "price_at_collection",
        "expected_points",
        "expected_minutes",
        "no_appearance_probability",
        "target_fpl_points",
        "target_minutes",
    }
    missing = required - set(fixture_predictions.columns)
    if missing:
        raise KeyError(
            "Backtest input is missing columns: "
            f"{sorted(missing)}"
        )

    data = fixture_predictions.loc[
        fixture_predictions["season"].eq(BACKTEST_SEASON)
    ].copy()

    grouped = (
        data.groupby(
            [
                "season",
                "gameweek",
                "player_id_season",
                "player_key",
                "player_name",
            ],
            dropna=False,
        )
        .agg(
            team_id=("team_id", "last"),
            team_short_name=("team_short_name", "last"),
            position=("model_position", "last"),
            price=("price_at_collection", "last"),
            fixture_count=("fixture_id", "nunique"),
            expected_points=("expected_points", "sum"),
            expected_minutes=("expected_minutes", "sum"),
            no_appearance_probability=(
                "no_appearance_probability",
                "prod",
            ),
            actual_points=("target_fpl_points", "sum"),
            actual_minutes=("target_minutes", "sum"),
        )
        .reset_index()
        .rename(columns={"player_id_season": "player_id"})
    )

    grouped["player_id"] = grouped["player_id"].astype(int)
    grouped["position"] = optimiser_position(grouped["position"])
    grouped["price"] = pd.to_numeric(
        grouped["price"],
        errors="coerce",
    )
    grouped["appearance_probability"] = (
        1.0
        - pd.to_numeric(
            grouped["no_appearance_probability"],
            errors="coerce",
        ).fillna(1.0)
    ).clip(0.0, 1.0)
    grouped["actual_appeared"] = (
        pd.to_numeric(
            grouped["actual_minutes"],
            errors="coerce",
        )
        .fillna(0.0)
        .gt(0)
    )

    # Merge true deadline-time FPL availability only when an archive exists.
    # Otherwise mark it explicitly unknown instead of manufacturing certainty.
    availability = globals().get(
        "HISTORICAL_AVAILABILITY_SNAPSHOTS",
        pd.DataFrame(),
    )
    if (
        isinstance(availability, pd.DataFrame)
        and not availability.empty
    ):
        grouped = grouped.merge(
            availability.loc[
                availability["season"].eq(BACKTEST_SEASON),
                [
                    "season",
                    "gameweek",
                    "player_id",
                    "status",
                    "chance_of_playing",
                    "availability_known",
                ],
            ],
            on=["season", "gameweek", "player_id"],
            how="left",
            validate="one_to_one",
        )
    else:
        grouped["status"] = HISTORICAL_AVAILABILITY_UNKNOWN_STATUS
        grouped["chance_of_playing"] = np.nan
        grouped["availability_known"] = False

    grouped["status"] = (
        grouped["status"]
        .fillna(HISTORICAL_AVAILABILITY_UNKNOWN_STATUS)
        .astype(str)
    )
    grouped["availability_known"] = (
        grouped["availability_known"].fillna(False).astype(bool)
    )

    return grouped.sort_values(
        ["gameweek", "player_id"]
    ).reset_index(drop=True)


backtest_prediction_source = holdout_expected_points.copy()
backtest_fold_premium = None
if premium_attacker_cache is not None:
    backtest_fold_premium = (
        premium_attacker_cache.get("fold_strategy_payloads", {}).get(BACKTEST_SEASON)
    )
backtest_prediction_source = apply_premium_attacker_adjustment(
    backtest_prediction_source,
    backtest_fold_premium,
    position_col="model_position",
    price_col="price_at_collection",
)
backtest_price_prior = build_position_price_prior_table(
    development_points_data
)
backtest_prediction_source = apply_sparse_player_market_prior(
    backtest_prediction_source,
    prior_table=backtest_price_prior,
)
backtest_fold_calibration = None
if xp_calibration_cache is not None:
    backtest_fold_calibration = (
        xp_calibration_cache.get("fold_strategy_payloads", {}).get(BACKTEST_SEASON)
    )
backtest_prediction_source = apply_xp_calibration_payload(
    backtest_prediction_source,
    backtest_fold_calibration,
    position_col="model_position",
    price_col="price_at_collection",
    value_col="expected_points",
)

historical_backtest_table = build_historical_backtest_table(
    backtest_prediction_source
)

backtest_gameweeks = sorted(
    historical_backtest_table["gameweek"].unique()
)
if BACKTEST_MAX_GAMEWEEKS is not None:
    backtest_gameweeks = backtest_gameweeks[
        :int(BACKTEST_MAX_GAMEWEEKS)
    ]

print(
    f"Backtest season: {BACKTEST_SEASON}; "
    f"Gameweeks configured: {len(backtest_gameweeks)}"
)
display(historical_backtest_table.head())

In [ ]:
def historical_candidate_snapshot(
    backtest_table: pd.DataFrame,
    gameweek: int,
    owned_player_ids: tuple[int, ...] = (),
) -> pd.DataFrame:
    """
    Return the latest known pre-deadline player universe.

    v2.2 removes the arbitrary one-Gameweek age cutoff for blank-club players.
    If a historical deadline-time availability archive exists, its status/chance
    fields are preserved; otherwise availability is explicitly marked unknown.
    """
    origin = int(gameweek)
    owned = {int(player_id) for player_id in owned_player_ids}

    history = backtest_table.loc[
        backtest_table["gameweek"].le(origin)
    ].copy()
    if history.empty:
        return pd.DataFrame(columns=[
            "player_id",
            "web_name",
            "team_id",
            "team_short_name",
            "position",
            "price",
            "status",
            "chance_of_playing",
            "availability_known",
        ])

    history["_snapshot_age"] = (
        origin
        - pd.to_numeric(
            history["gameweek"],
            errors="coerce",
        ).fillna(origin)
    )

    latest = (
        history.sort_values(
            ["player_id", "gameweek"],
            ascending=[True, False],
        )
        .drop_duplicates("player_id", keep="first")
    )

    if BACKTEST_MAX_PLAYER_SNAPSHOT_AGE is not None:
        latest = latest.loc[
            latest["_snapshot_age"].le(
                int(BACKTEST_MAX_PLAYER_SNAPSHOT_AGE)
            )
            | latest["player_id"].isin(owned)
        ].copy()

    candidate_columns = [
        "player_id",
        "player_name",
        "team_id",
        "team_short_name",
        "position",
        "price",
    ]
    for optional in [
        "status",
        "chance_of_playing",
        "availability_known",
    ]:
        if optional in latest.columns:
            candidate_columns.append(optional)

    candidates = (
        latest[candidate_columns]
        .rename(columns={"player_name": "web_name"})
        .drop_duplicates("player_id")
        .dropna(subset=["team_id", "position", "price"])
        .copy()
    )
    candidates["player_id"] = candidates["player_id"].astype(int)
    candidates["team_id"] = candidates["team_id"].astype(int)

    if "status" not in candidates.columns:
        candidates["status"] = HISTORICAL_AVAILABILITY_UNKNOWN_STATUS
    candidates["status"] = candidates["status"].fillna(
        HISTORICAL_AVAILABILITY_UNKNOWN_STATUS
    ).astype(str)

    if "chance_of_playing" not in candidates.columns:
        candidates["chance_of_playing"] = np.nan

    if "availability_known" not in candidates.columns:
        candidates["availability_known"] = False
    candidates["availability_known"] = (
        candidates["availability_known"].fillna(False).astype(bool)
    )

    return candidates.reset_index(drop=True)


deadline_frozen_forecast_cache: dict[tuple, pd.DataFrame] = {}
deadline_frozen_forecast_audit: list[dict[str, object]] = []


def _backtest_season_start_year() -> int:
    """Return the first year in BACKTEST_SEASON."""
    return int(str(BACKTEST_SEASON).split("-")[0])


def deadline_team_strength_state(
    forecast_origin_gameweek: int,
) -> pd.DataFrame:
    """
    Calculate team-strength features using only matches completed before
    the forecast-origin deadline.

    Goals, clean sheets and (where archived coverage exists) xG/xGA are
    reconstructed from the frozen information set. This keeps the feature
    schema aligned with the team-goal model used in production.
    """
    origin = int(forecast_origin_gameweek)
    season_start = _backtest_season_start_year()

    history = historical_team_fixtures.loc[
        (
            historical_team_fixtures["season_start_year"]
            .lt(season_start)
        )
        | (
            historical_team_fixtures["season"].eq(BACKTEST_SEASON)
            & historical_team_fixtures["gameweek"].lt(origin)
        )
    ].copy()

    history = history.loc[
        history["team_goals"].notna()
        & history["opponent_goals"].notna()
    ].sort_values(
        ["team_key", "kickoff_time", "fixture_id"]
    )

    has_xg = (
        "team_xg" in history.columns
        and "opponent_xg" in history.columns
        and (
            history["team_xg"].notna().any()
            or history["opponent_xg"].notna().any()
        )
    )

    rows: list[dict[str, object]] = []
    for team_key, team_history in history.groupby(
        "team_key",
        sort=False,
    ):
        goals_for = pd.to_numeric(
            team_history["team_goals"],
            errors="coerce",
        )
        goals_against = pd.to_numeric(
            team_history["opponent_goals"],
            errors="coerce",
        )
        clean_sheets = goals_against.eq(0).astype(float)

        xg_for = (
            pd.to_numeric(team_history["team_xg"], errors="coerce")
            if has_xg else None
        )
        xg_against = (
            pd.to_numeric(team_history["opponent_xg"], errors="coerce")
            if has_xg else None
        )

        row: dict[str, object] = {"team_key": team_key}

        for window in TEAM_ROLLING_WINDOWS:
            row[f"team_goals_for_last_{window}"] = (
                goals_for.tail(window).mean()
            )
            row[f"team_goals_against_last_{window}"] = (
                goals_against.tail(window).mean()
            )
            row[f"team_clean_sheet_rate_last_{window}"] = (
                clean_sheets.tail(window).mean()
            )
            if has_xg:
                row[f"team_xg_for_last_{window}"] = (
                    xg_for.tail(window).mean()
                )
                row[f"team_xg_against_last_{window}"] = (
                    xg_against.tail(window).mean()
                )

        row["team_career_goals_for"] = goals_for.mean()
        row["team_career_goals_against"] = goals_against.mean()
        row["team_career_clean_sheet_rate"] = clean_sheets.mean()
        if has_xg:
            row["team_career_xg_for"] = xg_for.mean()
            row["team_career_xg_against"] = xg_against.mean()

        rows.append(row)

    result = pd.DataFrame(rows)
    # Ensure subject-team features match the trained team model's own-team
    # namespace. Opponent features are generated only after the opponent merge.
    for column in [
        feature
        for feature in TEAM_FEATURE_COLUMNS
        if feature.startswith("team_")
    ]:
        if column not in result.columns:
            result[column] = np.nan
    return result


def deadline_player_feature_snapshot(
    forecast_origin_gameweek: int,
    candidate_ids: set[int] | None = None,
) -> pd.DataFrame:
    """
    Return one player row whose shifted history is valid at the deadline.

    Rows from the forecast-origin Gameweek are safe because all player
    rolling features were shifted by complete Gameweeks. When a player has
    no row in that Gameweek, the most recent earlier row is used.
    """
    origin = int(forecast_origin_gameweek)

    data = holdout_points_data.loc[
        holdout_points_data["season"].eq(BACKTEST_SEASON)
        & holdout_points_data["gameweek"].le(origin)
    ].copy()

    data["player_id"] = pd.to_numeric(
        data["player_id_season"],
        errors="coerce",
    ).astype("Int64")

    if candidate_ids is not None:
        data = data.loc[
            data["player_id"].isin(
                [int(player_id) for player_id in candidate_ids]
            )
        ]

    if data.empty:
        return data

    data["_is_origin_row"] = data["gameweek"].eq(origin)
    data = data.sort_values(
        [
            "player_id",
            "_is_origin_row",
            "gameweek",
            "kickoff_time",
            "fixture_id",
        ],
        ascending=[True, False, False, True, True],
    )

    snapshot = (
        data.drop_duplicates("player_id", keep="first")
        .drop(columns="_is_origin_row")
        .reset_index(drop=True)
    )
    snapshot["player_id"] = snapshot["player_id"].astype(int)
    return snapshot


def deadline_team_fixture_schedule(
    gameweeks: list[int],
) -> pd.DataFrame:
    """Return only schedule information known independently of results."""
    season_schedule = historical_team_fixtures.loc[
        historical_team_fixtures["season"].eq(BACKTEST_SEASON)
    ].copy()

    schedule_columns = [
        "season",
        "season_start_year",
        "gameweek",
        "fixture_id",
        "kickoff_time",
        "team_key",
        "opponent_team_key",
        "opponent_id",
        "was_home",
    ]
    schedule = (
        season_schedule[schedule_columns]
        .drop_duplicates(["team_key", "fixture_id"])
        .sort_values(["team_key", "kickoff_time", "fixture_id"])
        .reset_index(drop=True)
    )

    schedule["previous_league_kickoff"] = (
        schedule.groupby("team_key")["kickoff_time"].shift(1)
    )
    schedule["days_since_previous_league_fixture"] = (
        (
            schedule["kickoff_time"]
            - schedule["previous_league_kickoff"]
        )
        .dt.total_seconds()
        .div(86_400)
    )
    schedule["short_league_rest"] = (
        schedule["days_since_previous_league_fixture"].le(4)
    ).astype("boolean")

    return schedule.loc[
        schedule["gameweek"].isin([int(gw) for gw in gameweeks])
    ].reset_index(drop=True)


def build_deadline_frozen_fixture_predictions(
    forecast_origin_gameweek: int,
    gameweeks: list[int],
    candidate_ids: set[int] | None = None,
) -> pd.DataFrame:
    """
    Project future fixtures using one information set frozen at the deadline.

    Player rolling features and team-strength estimates are frozen at the
    forecast origin. Only information genuinely known for later Gameweeks is
    allowed to change: fixture, opponent, venue, scheduled rest and Gameweek.
    Players remain attached to the club they represent at the deadline, so a
    later real-world transfer cannot leak into an earlier forecast.
    """
    origin = int(forecast_origin_gameweek)
    horizon = sorted(int(gw) for gw in gameweeks)

    player_snapshot = deadline_player_feature_snapshot(
        origin,
        candidate_ids,
    )
    if player_snapshot.empty:
        return pd.DataFrame()

    schedule = deadline_team_fixture_schedule(horizon)
    if schedule.empty:
        return pd.DataFrame()

    schedule = schedule.rename(columns={
        "season": "fixture_season",
        "season_start_year": "fixture_season_start_year",
        "gameweek": "fixture_gameweek",
        "fixture_id": "future_fixture_id",
        "kickoff_time": "future_kickoff_time",
        "team_key": "fixture_team_key",
        "opponent_team_key": "future_opponent_team_key",
        "opponent_id": "future_opponent_id",
        "was_home": "future_was_home",
        "days_since_previous_league_fixture": (
            "future_days_since_previous_league_fixture"
        ),
        "short_league_rest": "future_short_league_rest",
    })

    fixture_rows = player_snapshot.merge(
        schedule,
        left_on="team_key",
        right_on="fixture_team_key",
        how="inner",
        validate="many_to_many",
    )

    fixture_rows["season"] = fixture_rows["fixture_season"]
    fixture_rows["season_start_year"] = (
        fixture_rows["fixture_season_start_year"]
    )
    fixture_rows["gameweek"] = fixture_rows["fixture_gameweek"]
    fixture_rows["fixture_id"] = fixture_rows["future_fixture_id"]
    fixture_rows["kickoff_time"] = (
        fixture_rows["future_kickoff_time"]
    )
    fixture_rows["opponent_team_key"] = (
        fixture_rows["future_opponent_team_key"]
    )
    fixture_rows["opponent_id"] = (
        fixture_rows["future_opponent_id"]
    )
    fixture_rows["was_home"] = normalise_boolean(
        fixture_rows["future_was_home"]
    )
    fixture_rows["days_since_previous_league_fixture"] = (
        fixture_rows[
            "future_days_since_previous_league_fixture"
        ]
    )
    fixture_rows["short_league_rest"] = (
        fixture_rows["future_short_league_rest"]
    )

    # Recalculate team state from outcomes completed strictly before origin.
    team_state = deadline_team_strength_state(origin)
    own_state = team_state.copy()
    opponent_state = team_state.rename(columns={
        "team_key": "opponent_team_key",
        **{
            column: column.replace("team_", "opponent_", 1)
            for column in team_state.columns
            if column.startswith("team_")
            and column != "team_key"
        },
    })

    fixture_rows = fixture_rows.merge(
        own_state,
        on="team_key",
        how="left",
        validate="many_to_one",
    )
    fixture_rows = fixture_rows.merge(
        opponent_state,
        on="opponent_team_key",
        how="left",
        validate="many_to_one",
    )

    # Remove current-row team predictions before producing fixture-specific
    # forecasts for the frozen horizon.
    fixture_rows = fixture_rows.drop(
        columns=[
            "expected_team_goals",
            "expected_team_goals_conceded",
            "team_clean_sheet_probability",
        ],
        errors="ignore",
    ).reset_index(drop=True)

    team_predictions = predict_team_goal_components(
        team_goal_models,
        fixture_rows,
    ).reset_index(drop=True)

    minutes_predictions = predict_minutes_components(
        minutes_models,
        fixture_rows,
    ).reset_index(drop=True)

    event_feature_data = pd.concat(
        [
            fixture_rows.reset_index(drop=True),
            team_predictions,
            minutes_predictions,
        ],
        axis=1,
    )
    event_predictions = predict_player_event_rates(
        player_event_models,
        event_feature_data,
    ).reset_index(drop=True)

    xp_inputs = pd.concat(
        [
            event_feature_data.reset_index(drop=True),
            event_predictions,
        ],
        axis=1,
    )
    v26_set_conversion(development_points_data, label="development")
    xp_components = calculate_component_expected_points(
        xp_inputs,
        development_defensive_action_priors,
        development_rare_event_rates,
        development_defensive_return_priors,
    ).reset_index(drop=True)

    # Defensive-contribution points entered FPL in 2025/26. Earlier
    # walk-forward folds are evaluated against their own historical scoring
    # and must therefore omit this component.
    if _backtest_season_start_year() < 2025:
        xp_components["expected_points"] = (
            xp_components["expected_points"]
            - xp_components["xp_defensive_contribution"]
        )
        xp_components["xp_defensive_contribution"] = 0.0

    predictions = pd.concat(
        [
            xp_inputs.reset_index(drop=True),
            xp_components,
        ],
        axis=1,
    )

    predictions["forecast_origin_gameweek"] = origin
    predictions["player_history_available_through_gw"] = origin - 1
    predictions["team_results_available_through_gw"] = origin - 1

    if not predictions["expected_points"].replace(
        [np.inf, -np.inf],
        np.nan,
    ).notna().all():
        raise ValueError(
            "Deadline-frozen expected-points forecasts contain "
            "non-finite values."
        )

    deadline_frozen_forecast_audit.append({
        "forecast_origin_gameweek": origin,
        "first_projected_gameweek": min(horizon),
        "last_projected_gameweek": max(horizon),
        "candidate_players": int(
            predictions["player_id"].nunique()
        ),
        "fixture_rows": int(len(predictions)),
        "player_history_available_through_gw": origin - 1,
        "team_results_available_through_gw": origin - 1,
    })
    return predictions


def historical_projection_window(
    backtest_table: pd.DataFrame,
    gameweeks: list[int],
    candidate_ids: set[int] | None = None,
) -> pd.DataFrame:
    """
    Build a deadline-frozen optimiser projection matrix.

    The earlier implementation selected already-computed future rows from
    holdout_expected_points. A GW1 six-week plan therefore used GW2-GW6
    rolling features that would not have existed at the GW1 deadline. This
    replacement rebuilds every horizon from the information set available at
    min(gameweeks).
    """
    del backtest_table  # Outcomes are used only later for realised scoring.

    horizon = sorted(int(gw) for gw in gameweeks)
    if not horizon:
        return pd.DataFrame()

    origin = int(min(horizon))
    candidate_key = (
        None
        if candidate_ids is None
        else tuple(sorted(int(pid) for pid in candidate_ids))
    )
    cache_key = (origin, tuple(horizon), candidate_key)

    if cache_key not in deadline_frozen_forecast_cache:
        fixture_predictions = (
            build_deadline_frozen_fixture_predictions(
                origin,
                horizon,
                candidate_ids,
            )
        )

        if fixture_predictions.empty:
            projection = pd.DataFrame(columns=[
                "player_id",
                "gameweek",
                "expected_points",
                "expected_minutes",
                "appearance_probability",
                "no_appearance_probability",
            ])
        else:
            projection = (
                fixture_predictions.groupby(
                    ["player_id", "gameweek"],
                    dropna=False,
                )
                .agg(
                    expected_points=("expected_points", "sum"),
                    expected_minutes=("expected_minutes", "sum"),
                    no_appearance_probability=(
                        "no_appearance_probability",
                        "prod",
                    ),
                )
                .reset_index()
            )
            projection["appearance_probability"] = (
                1.0
                - pd.to_numeric(
                    projection["no_appearance_probability"],
                    errors="coerce",
                ).fillna(1.0)
            ).clip(0.0, 1.0)

        # Apply true archived deadline-time availability only to the current
        # decision GW. Do not assume an injury/suspension persists through the
        # whole planning horizon.
        availability = globals().get(
            "HISTORICAL_AVAILABILITY_SNAPSHOTS",
            pd.DataFrame(),
        )
        if (
            isinstance(availability, pd.DataFrame)
            and not availability.empty
            and not projection.empty
        ):
            snapshot = availability.loc[
                availability["season"].eq(BACKTEST_SEASON)
                & availability["gameweek"].eq(origin),
                [
                    "player_id",
                    "status",
                    "chance_of_playing",
                ],
            ].drop_duplicates("player_id")

            if not snapshot.empty:
                projection = projection.merge(
                    snapshot,
                    on="player_id",
                    how="left",
                    validate="many_to_one",
                )
                chance = pd.to_numeric(
                    projection["chance_of_playing"],
                    errors="coerce",
                )
                multiplier = pd.Series(
                    1.0,
                    index=projection.index,
                    dtype=float,
                )
                known_chance = chance.notna()
                multiplier.loc[known_chance] = (
                    chance.loc[known_chance]
                    .div(100.0)
                    .clip(0.0, 1.0)
                )
                hard_out = (
                    projection["status"].isin(["i", "s", "u"])
                    & chance.isna()
                )
                multiplier.loc[hard_out] = 0.0

                current_mask = projection["gameweek"].eq(origin)
                for column in [
                    "expected_points",
                    "expected_minutes",
                    "appearance_probability",
                ]:
                    projection.loc[current_mask, column] = (
                        pd.to_numeric(
                            projection.loc[current_mask, column],
                            errors="coerce",
                        ).fillna(0.0)
                        * multiplier.loc[current_mask]
                    )

                projection.loc[
                    current_mask,
                    "no_appearance_probability",
                ] = (
                    1.0
                    - projection.loc[
                        current_mask,
                        "appearance_probability",
                    ]
                ).clip(0.0, 1.0)
                projection = projection.drop(
                    columns=["status", "chance_of_playing"]
                )

        deadline_frozen_forecast_cache[cache_key] = projection

    return deadline_frozen_forecast_cache[cache_key].copy()

def fpl_historical_selling_price(
    purchase_price: float,
    current_price: float,
) -> float:
    """Apply the official half-profit selling-price rule in £0.1m units."""
    purchase = price_to_tenths(purchase_price)
    current = price_to_tenths(current_price)
    if current <= purchase:
        return tenths_to_price(current)
    realised_profit = (current - purchase) // 2
    return tenths_to_price(purchase + realised_profit)


def current_selling_prices(
    squad_ids: tuple[int, ...],
    purchase_prices: dict[int, float],
    candidates: pd.DataFrame,
) -> dict[int, float]:
    market = (
        candidates.set_index("player_id")["price"]
        .astype(float)
        .to_dict()
    )
    result = {}
    for player_id in squad_ids:
        current_price = market.get(
            int(player_id),
            purchase_prices[int(player_id)],
        )
        result[int(player_id)] = fpl_historical_selling_price(
            purchase_prices[int(player_id)],
            current_price,
        )
    return result


def validate_detailed_lineup_decision(
    detailed_decision: dict[str, object],
) -> dict[str, object]:
    """
    Validate the structural invariants required by realised FPL scoring.

    The autosub engine must receive the *nominal* starting XI, not merely the
    starters who eventually appeared.  This validator makes that contract
    explicit and gives a local diagnostic if an optimiser/backtest state ever
    becomes malformed.
    """
    if "squad_projection" not in detailed_decision:
        raise ValueError("Decision is missing 'squad_projection'.")

    squad_projection = detailed_decision["squad_projection"]
    if not isinstance(squad_projection, pd.DataFrame):
        raise TypeError(
            "Decision 'squad_projection' must be a pandas DataFrame."
        )

    required_columns = {"player_id", "position"}
    missing_columns = required_columns.difference(
        squad_projection.columns
    )
    if missing_columns:
        raise ValueError(
            "Decision squad_projection is missing required columns: "
            f"{sorted(missing_columns)}"
        )

    squad = squad_projection[["player_id", "position"]].copy()
    squad["player_id"] = pd.to_numeric(
        squad["player_id"],
        errors="raise",
    ).astype(int)
    squad["position"] = normalise_scoring_position(
        squad["position"]
    )

    if squad["player_id"].duplicated().any():
        duplicates = (
            squad.loc[
                squad["player_id"].duplicated(keep=False),
                "player_id",
            ]
            .astype(int)
            .tolist()
        )
        raise ValueError(
            "Decision squad contains duplicate player IDs: "
            f"{sorted(set(duplicates))}"
        )

    if len(squad) != 15:
        raise ValueError(
            "A scored FPL squad must contain exactly 15 players; "
            f"received {len(squad)}."
        )

    squad_ids = set(squad["player_id"].astype(int))
    position_by_id = (
        squad.set_index("player_id")["position"].astype(str).to_dict()
    )

    unknown_positions = sorted(
        {
            str(position)
            for position in position_by_id.values()
            if str(position) not in PLAYER_SCORING_POSITIONS
        }
    )
    if unknown_positions:
        raise ValueError(
            "Decision contains unsupported position labels: "
            f"{unknown_positions}"
        )

    try:
        starter_ids = tuple(
            int(pid)
            for pid in detailed_decision["starter_ids"]
        )
    except KeyError as exc:
        raise ValueError("Decision is missing 'starter_ids'.") from exc

    if len(starter_ids) != 11 or len(set(starter_ids)) != 11:
        raise ValueError(
            "Decision must contain exactly 11 unique starter IDs; "
            f"received {len(starter_ids)} values / "
            f"{len(set(starter_ids))} unique."
        )

    starter_set = set(starter_ids)
    missing_starters = starter_set.difference(squad_ids)
    if missing_starters:
        raise ValueError(
            "Starter IDs are absent from squad_projection: "
            f"{sorted(missing_starters)}"
        )

    starter_positions = [
        position_by_id[player_id]
        for player_id in starter_ids
    ]
    goalkeeper_count = starter_positions.count("GK")
    if goalkeeper_count != 1:
        raise ValueError(
            "Starting XI must contain exactly one goalkeeper; "
            f"received {goalkeeper_count}."
        )

    nominal_outfield_counts = {
        position: starter_positions.count(position)
        for position in ("DEF", "MID", "FWD")
    }
    if not legal_partial_outfield_formation(
        nominal_outfield_counts
    ):
        raise ValueError(
            "Starting XI has an illegal nominal outfield formation: "
            f"{nominal_outfield_counts}"
        )

    nonstarter_ids = squad_ids.difference(starter_set)
    bench_goalkeepers = sorted(
        player_id
        for player_id in nonstarter_ids
        if position_by_id[player_id] == "GK"
    )
    if len(bench_goalkeepers) != 1:
        raise ValueError(
            "Bench must contain exactly one goalkeeper; "
            f"received {len(bench_goalkeepers)}."
        )

    expected_outfield_bench = {
        player_id
        for player_id in nonstarter_ids
        if position_by_id[player_id] != "GK"
    }
    if len(expected_outfield_bench) != 3:
        raise ValueError(
            "Bench must contain exactly three outfield players; "
            f"received {len(expected_outfield_bench)}."
        )

    try:
        bench_order = tuple(
            int(pid)
            for pid in detailed_decision["bench_order"]
        )
    except KeyError as exc:
        raise ValueError("Decision is missing 'bench_order'.") from exc

    if len(bench_order) != 3 or len(set(bench_order)) != 3:
        raise ValueError(
            "bench_order must contain exactly three unique outfield IDs; "
            f"received {bench_order}."
        )

    if set(bench_order) != expected_outfield_bench:
        raise ValueError(
            "bench_order must contain exactly the three non-starting "
            "outfield players. "
            f"expected={sorted(expected_outfield_bench)}, "
            f"received={list(bench_order)}"
        )

    try:
        captain_id = int(detailed_decision["captain_id"])
        vice_id = int(detailed_decision["vice_captain_id"])
    except KeyError as exc:
        raise ValueError(
            "Decision is missing captain_id or vice_captain_id."
        ) from exc

    if captain_id == vice_id:
        raise ValueError("Captain and vice-captain must be different players.")
    if captain_id not in starter_set or vice_id not in starter_set:
        raise ValueError(
            "Captain and vice-captain must both belong to the starting XI."
        )

    nominal_outfield = [
        {
            "player_id": int(player_id),
            "position": str(position_by_id[player_id]),
        }
        for player_id in starter_ids
        if position_by_id[player_id] != "GK"
    ]

    return {
        "squad_ids": tuple(
            int(player_id)
            for player_id in squad["player_id"].tolist()
        ),
        "position_by_id": position_by_id,
        "starter_ids": tuple(starter_ids),
        "starter_set": starter_set,
        "starting_gk": next(
            player_id
            for player_id in starter_ids
            if position_by_id[player_id] == "GK"
        ),
        "bench_gk": int(bench_goalkeepers[0]),
        "bench_order": tuple(bench_order),
        "captain_id": captain_id,
        "vice_captain_id": vice_id,
        "nominal_outfield": nominal_outfield,
        "nominal_outfield_counts": nominal_outfield_counts,
    }


def actual_score_from_decision(
    detailed_decision: dict[str, object],
    actual_gameweek: pd.DataFrame,
    chip: str | None = None,
) -> dict[str, float]:
    """
    Score XI, automatic substitutions and captaincy using realised points.

    v2.2.6 explicitly passes the true nominal starting formation into the
    autosub engine.  The previous realised-score path passed only appearing
    starters, which could incorrectly infer shapes such as 3-6-1 when several
    starters failed to appear.
    """
    state = validate_detailed_lineup_decision(
        detailed_decision
    )

    actual_required = {
        "player_id",
        "actual_points",
        "actual_appeared",
    }
    missing_actual = actual_required.difference(
        actual_gameweek.columns
    )
    if missing_actual:
        raise ValueError(
            "actual_gameweek is missing required scoring columns: "
            f"{sorted(missing_actual)}"
        )

    actual = (
        actual_gameweek[
            [
                "player_id",
                "actual_points",
                "actual_appeared",
            ]
        ]
        .copy()
    )
    actual["player_id"] = pd.to_numeric(
        actual["player_id"],
        errors="coerce",
    ).astype("Int64")
    actual = (
        actual.dropna(subset=["player_id"])
        .drop_duplicates("player_id", keep="last")
        .set_index("player_id")
    )

    squad_ids = list(state["squad_ids"])
    position_by_id = state["position_by_id"]
    starter_ids = state["starter_ids"]
    starter_set = state["starter_set"]
    starting_gk = int(state["starting_gk"])
    bench_gk = int(state["bench_gk"])

    points_by_id = {
        int(player_id): float(
            pd.to_numeric(
                pd.Series(
                    [
                        actual.loc[player_id, "actual_points"]
                        if player_id in actual.index else 0.0
                    ]
                ),
                errors="coerce",
            ).fillna(0.0).iloc[0]
        )
        for player_id in squad_ids
    }
    appeared_by_id = {
        int(player_id): bool(
            actual.loc[player_id, "actual_appeared"]
            if player_id in actual.index
            and pd.notna(actual.loc[player_id, "actual_appeared"])
            else False
        )
        for player_id in squad_ids
    }

    counted_ids: list[int] = []
    if appeared_by_id[starting_gk]:
        counted_ids.append(starting_gk)
    elif appeared_by_id[bench_gk]:
        counted_ids.append(bench_gk)

    starting_outfield = [
        {
            "player_id": int(player_id),
            "position": str(position_by_id[player_id]),
        }
        for player_id in starter_ids
        if position_by_id[player_id] != "GK"
        and appeared_by_id[player_id]
    ]

    appearing_bench = [
        {
            "player_id": int(player_id),
            "position": str(position_by_id[int(player_id)]),
        }
        for player_id in state["bench_order"]
        if appeared_by_id[int(player_id)]
    ]

    final_outfield = apply_outfield_auto_substitutions(
        starting_outfield,
        appearing_bench,
        nominal_starters=state["nominal_outfield"],
    )
    counted_ids.extend(
        int(player["player_id"])
        for player in final_outfield
    )

    # Defensive invariant: autosubs may never score more than ten outfielders
    # or duplicate a player.
    if len(counted_ids) > 11 or len(counted_ids) != len(set(counted_ids)):
        raise RuntimeError(
            "Realised autosub scoring produced an invalid counted-player "
            f"set: {counted_ids}"
        )

    captain_id = int(state["captain_id"])
    vice_id = int(state["vice_captain_id"])
    effective_captain = (
        captain_id
        if appeared_by_id[captain_id]
        else vice_id if appeared_by_id[vice_id] else None
    )

    if chip == "bench_boost":
        base_points = sum(points_by_id.values())
        counted_ids = list(squad_ids)
    else:
        base_points = sum(
            points_by_id[player_id]
            for player_id in counted_ids
        )

    captain_points = (
        points_by_id[effective_captain]
        if effective_captain is not None else 0.0
    )
    captain_multiplier_extra = (
        2.0 if chip == "triple_captain" else 1.0
    )

    return {
        "base_points": float(base_points),
        "captain_points": float(captain_points),
        "captain_extra": float(
            captain_multiplier_extra * captain_points
        ),
        "total_before_hits": float(
            base_points
            + captain_multiplier_extra * captain_points
        ),
        "counted_players": float(len(counted_ids)),
    }
def fixture_disruption_in_gameweek(
    backtest_table: pd.DataFrame,
    gameweek: int,
) -> bool:
    """Detect blank or double clubs from the archived player-GW rows."""
    fixture_counts = (
        backtest_table.loc[
            backtest_table["gameweek"].eq(int(gameweek))
        ]
        .groupby("team_id")["fixture_count"]
        .max()
    )
    return (
        len(fixture_counts) < 20
        or fixture_counts.gt(1).any()
    )


def reset_half_chip_state() -> dict[str, bool]:
    return {
        chip: count > 0
        for chip, count in BACKTEST_CHIPS_PER_HALF.items()
    }

#### v2.2.6 realised-score autosub regression guard

The historical scorer now validates the complete 15-player decision and passes the **true nominal starting XI** into the autosub engine. This prevents no-shows from being mistaken for missing formation slots during policy backtests. The regression cell below exercises every legal outfield formation plus the exact multi-no-show case that previously produced an impossible `3-6-1` reconstruction.

In [ ]:
# v2.2.6 regression: realised scoring must pass the true nominal formation.
# Build a complete 15-player 3-4-3 decision where a DEF and MID both fail to
# appear.  This is the exact class of state that previously produced 3-6-1.
_v226_squad = pd.DataFrame([
    {"player_id": 1, "position": "GK"},
    {"player_id": 2, "position": "GK"},
    *[
        {"player_id": 10 + i, "position": "DEF"}
        for i in range(5)
    ],
    *[
        {"player_id": 20 + i, "position": "MID"}
        for i in range(5)
    ],
    *[
        {"player_id": 30 + i, "position": "FWD"}
        for i in range(3)
    ],
])

_v226_starters = {
    1,
    10, 11, 12,
    20, 21, 22, 23,
    30, 31, 32,
}
_v226_decision = {
    "squad_projection": _v226_squad,
    "starter_ids": _v226_starters,
    "bench_order": (24, 13, 14),
    "captain_id": 20,
    "vice_captain_id": 30,
}
_v226_actual = _v226_squad[["player_id", "position"]].copy()
_v226_actual["actual_points"] = 2.0
_v226_actual["actual_appeared"] = True
_v226_actual.loc[
    _v226_actual["player_id"].isin([10, 20]),
    "actual_appeared",
] = False

_v226_scored = actual_score_from_decision(
    _v226_decision,
    _v226_actual,
)
assert np.isfinite(_v226_scored["total_before_hits"])
assert 0 <= _v226_scored["counted_players"] <= 11

# Exhaust every legal FPL outfield formation with deterministic no-shows.
_v226_legal_formations = [
    {"DEF": d, "MID": m, "FWD": f}
    for d in range(3, 6)
    for m in range(2, 6)
    for f in range(1, 4)
    if d + m + f == 10
]
for _formation in _v226_legal_formations:
    _def_ids = list(range(100, 105))
    _mid_ids = list(range(200, 205))
    _fwd_ids = list(range(300, 303))
    _squad = pd.DataFrame([
        {"player_id": 1, "position": "GK"},
        {"player_id": 2, "position": "GK"},
        *[{"player_id": pid, "position": "DEF"} for pid in _def_ids],
        *[{"player_id": pid, "position": "MID"} for pid in _mid_ids],
        *[{"player_id": pid, "position": "FWD"} for pid in _fwd_ids],
    ])
    _xi = (
        [1]
        + _def_ids[:_formation["DEF"]]
        + _mid_ids[:_formation["MID"]]
        + _fwd_ids[:_formation["FWD"]]
    )
    _bench_outfield = [
        pid
        for pid in (_def_ids + _mid_ids + _fwd_ids)
        if pid not in _xi
    ]
    assert len(_bench_outfield) == 3
    _decision = {
        "squad_projection": _squad,
        "starter_ids": set(_xi),
        "bench_order": tuple(_bench_outfield),
        "captain_id": _xi[1],
        "vice_captain_id": _xi[2],
    }
    _actual = _squad.copy()
    _actual["actual_points"] = 1.0
    _actual["actual_appeared"] = True

    # Deterministic multi-position no-shows stress the nominal-formation path.
    _outfield_xi = [
        pid for pid in _xi if pid != 1
    ]
    if len(_outfield_xi) >= 2:
        _actual.loc[
            _actual["player_id"].isin(_outfield_xi[:2]),
            "actual_appeared",
        ] = False

    _score = actual_score_from_decision(
        _decision,
        _actual,
    )
    assert np.isfinite(_score["total_before_hits"])
    assert 0 <= _score["counted_players"] <= 11

# Invalid decisions fail at the structural contract, before autosub recursion.
_v226_bad = dict(_v226_decision)
_v226_bad["starter_ids"] = {
    1,
    10, 11, 12,
    20, 21, 22, 23, 24,
    30, 31,
}  # 3-5-2 is legal; make bench inconsistent instead.
_v226_bad["bench_order"] = (13, 14, 24)
try:
    validate_detailed_lineup_decision(_v226_bad)
except ValueError as _exc:
    assert "bench_order" in str(_exc)
else:
    raise AssertionError(
        "Malformed decision should fail the structural validator."
    )

print("v2.2.6 realised-score/autosub invariants: PASS")

In [ ]:
def build_backtest_opening_state(
    backtest_table: pd.DataFrame,
    gameweeks: list[int],
    *,
    opening_horizon: int,
) -> dict[str, object]:
    """
    Build a deterministic opening squad independently of a policy profile.

    Section 11 passes the SAME opening_horizon for every policy in a holdout
    season. Generic sequential backtests may still pass their own horizon.
    """
    if not gameweeks:
        raise ValueError("No Gameweeks were supplied for opening-state construction.")

    first_gameweek = int(gameweeks[0])
    available_gameweeks = [
        int(gw)
        for gw in sorted(backtest_table["gameweek"].unique())
        if int(gw) >= first_gameweek
    ]
    opening_gameweeks = available_gameweeks[:max(1, int(opening_horizon))]
    if not opening_gameweeks:
        raise ValueError("No projected Gameweeks are available for the opening squad.")

    initial_candidates = historical_candidate_snapshot(
        backtest_table,
        first_gameweek,
    )
    initial_projections = historical_projection_window(
        backtest_table,
        opening_gameweeks,
        set(initial_candidates["player_id"]),
    )
    subset_candidates, subset_projections = chip_candidate_subset(
        initial_candidates,
        initial_projections,
        opening_gameweeks,
    )
    initial_result = optimise_squad_lineups_and_captains(
        subset_candidates,
        subset_projections,
        opening_gameweeks,
        budget=BACKTEST_INITIAL_BUDGET,
    )
    squad = initial_result["squad"].copy()
    squad_ids = tuple(
        sorted(squad["player_id"].astype(int).tolist())
    )
    price_map = (
        squad.set_index("player_id")["price"]
        .astype(float)
        .to_dict()
    )
    bank = float(
        BACKTEST_INITIAL_BUDGET
        - float(squad["price"].sum())
    )

    signature = stable_configuration_signature({
        "first_gameweek": first_gameweek,
        "opening_gameweeks": opening_gameweeks,
        "budget": float(BACKTEST_INITIAL_BUDGET),
        "squad_ids": squad_ids,
        "prices": {
            str(int(pid)): round(float(price_map[int(pid)]), 4)
            for pid in squad_ids
        },
    })

    return {
        "squad": squad,
        "squad_ids": squad_ids,
        "bank": bank,
        "purchase_prices": price_map,
        "opening_gameweeks": tuple(opening_gameweeks),
        "opening_horizon": int(opening_horizon),
        "signature": signature,
    }


def run_sequential_fpl_backtest(
    backtest_table: pd.DataFrame,
    gameweeks: list[int],
    *,
    opening_state: dict[str, object] | None = None,
) -> dict[str, object]:
    """Run the rolling strategy and a static-squad benchmark."""
    if not gameweeks:
        raise ValueError("No Gameweeks were supplied to the backtest.")

    # Temporarily reduce global beam widths for repeated historical runs.
    global TRANSFER_BEAM_WIDTH
    global TRANSFER_ACTION_BEAM_WIDTH
    global TRANSFER_TOP_XP_PER_POSITION
    global TRANSFER_TOP_VALUE_PER_POSITION

    original_search_settings = (
        TRANSFER_BEAM_WIDTH,
        TRANSFER_ACTION_BEAM_WIDTH,
        TRANSFER_TOP_XP_PER_POSITION,
        TRANSFER_TOP_VALUE_PER_POSITION,
    )
    TRANSFER_BEAM_WIDTH = BACKTEST_TRANSFER_BEAM_WIDTH
    TRANSFER_ACTION_BEAM_WIDTH = (
        BACKTEST_TRANSFER_ACTION_BEAM_WIDTH
    )
    TRANSFER_TOP_XP_PER_POSITION = (
        BACKTEST_TRANSFER_TOP_XP_PER_POSITION
    )
    TRANSFER_TOP_VALUE_PER_POSITION = (
        BACKTEST_TRANSFER_TOP_VALUE_PER_POSITION
    )

    transfer_search_fn = (
        optimise_transfer_path_adaptive
        if BACKTEST_USE_ADAPTIVE_TRANSFER_SEARCH
        else optimise_transfer_path
    )

    try:
        first_gameweek = int(gameweeks[0])
        if opening_state is None:
            opening_state = build_backtest_opening_state(
                backtest_table,
                gameweeks,
                opening_horizon=int(BACKTEST_HORIZON),
            )
        else:
            opening_state = dict(opening_state)

        required_opening_keys = {
            "squad",
            "squad_ids",
            "bank",
            "purchase_prices",
            "signature",
        }
        missing_opening = required_opening_keys - set(opening_state)
        if missing_opening:
            raise ValueError(
                "Opening state is missing required fields: "
                f"{sorted(missing_opening)}"
            )

        initial_squad_frame = opening_state["squad"].copy()
        initial_squad = tuple(
            sorted(int(pid) for pid in opening_state["squad_ids"])
        )
        if len(initial_squad) != 15 or len(set(initial_squad)) != 15:
            raise ValueError(
                "Opening state must contain exactly 15 unique players."
            )

        squad_ids = initial_squad
        static_squad_ids = initial_squad
        bank = float(opening_state["bank"])
        free_transfers = BACKTEST_INITIAL_FREE_TRANSFERS
        purchase_prices = {
            int(pid): float(price)
            for pid, price in dict(
                opening_state["purchase_prices"]
            ).items()
        }
        chips_available = reset_half_chip_state()

        rows = []
        static_rows = []

        all_season_gameweeks = sorted(
            backtest_table["gameweek"].unique()
        )

        for gameweek in gameweeks:
            gameweek = int(gameweek)
            if gameweek == 20:
                chips_available = reset_half_chip_state()

            horizon = [
                gw for gw in all_season_gameweeks
                if gameweek <= gw < gameweek + BACKTEST_HORIZON
            ]

            candidates = historical_candidate_snapshot(
                backtest_table,
                gameweek,
                squad_ids,
            )
            candidate_ids = set(candidates["player_id"].astype(int))
            projections = historical_projection_window(
                backtest_table,
                horizon,
                candidate_ids,
            )
            actual_gameweek = backtest_table.loc[
                backtest_table["gameweek"].eq(gameweek)
            ].copy()

            selling_prices = current_selling_prices(
                squad_ids,
                purchase_prices,
                candidates,
            )

            pre_decision_squad_ids = tuple(int(pid) for pid in squad_ids)
            pre_decision_bank = float(bank)
            pre_decision_free_transfers = int(free_transfers)
            pre_decision_purchase_prices = dict(purchase_prices)

            transfer_result = transfer_search_fn(
                list(squad_ids),
                candidates,
                projections,
                horizon,
                money_in_bank=bank,
                free_transfers=free_transfers,
                current_selling_prices=selling_prices,
            )
            if BACKTEST_APPLY_TRANSFER_ROBUSTNESS_GATE:
                baseline_state, robustness_audit = (
                    select_transfer_state_with_marginal_gate(
                        transfer_result,
                        allow_hits=bool(ALLOW_TRANSFER_HITS),
                        min_gain_per_extra_transfer=float(
                            WEEKLY_MIN_OBJECTIVE_GAIN_PER_EXTRA_TRANSFER
                        ),
                    )
                )
            else:
                baseline_state = transfer_result["best_state"]
                robustness_audit = {
                    "applied": False,
                    "raw_transfer_count": int(
                        baseline_state["path"][0]["transfer_count"]
                    ),
                    "selected_transfer_count": int(
                        baseline_state["path"][0]["transfer_count"]
                    ),
                    "objective_given_up": 0.0,
                }
            baseline_action = baseline_state["path"][0]
            baseline_squad = tuple(
                int(pid)
                for pid in baseline_action["squad_ids_after"]
            )
            baseline_detail = exact_squad_gameweek_decision(
                baseline_squad,
                gameweek,
                candidates,
                projections,
            )
            baseline_expected_after_hits = (
                float(baseline_detail["expected_total_score"])
                - float(baseline_action["hit_cost"])
            )

            hit_counterfactual = {
                "evaluated": False,
                "projected_horizon_gain_vs_no_hit": np.nan,
                "projected_current_gw_gain_vs_no_hit": np.nan,
                "actual_current_gw_gain_vs_no_hit": np.nan,
                "no_hit_transfers": (),
            }
            if (
                HIT_COUNTERFACTUAL_VALIDATION_ENABLED
                and float(baseline_action["hit_cost"]) > 0.0
            ):
                original_allow_hits = bool(ALLOW_TRANSFER_HITS)
                try:
                    globals()["ALLOW_TRANSFER_HITS"] = False
                    no_hit_result = transfer_search_fn(
                        list(pre_decision_squad_ids),
                        candidates,
                        projections,
                        horizon,
                        money_in_bank=pre_decision_bank,
                        free_transfers=pre_decision_free_transfers,
                        current_selling_prices=selling_prices,
                    )
                finally:
                    globals()["ALLOW_TRANSFER_HITS"] = original_allow_hits

                no_hit_state = no_hit_result["best_state"]
                no_hit_action = no_hit_state["path"][0]
                no_hit_squad = tuple(
                    int(pid)
                    for pid in no_hit_action["squad_ids_after"]
                )
                no_hit_detail = exact_squad_gameweek_decision(
                    no_hit_squad,
                    gameweek,
                    candidates,
                    projections,
                )
                no_hit_expected = (
                    float(no_hit_detail["expected_total_score"])
                    - float(no_hit_action["hit_cost"])
                )
                baseline_realised = actual_score_from_decision(
                    baseline_detail,
                    actual_gameweek,
                )["total_before_hits"] - float(
                    baseline_action["hit_cost"]
                )
                no_hit_realised = actual_score_from_decision(
                    no_hit_detail,
                    actual_gameweek,
                )["total_before_hits"] - float(
                    no_hit_action["hit_cost"]
                )
                hit_counterfactual = {
                    "evaluated": True,
                    "projected_horizon_gain_vs_no_hit": (
                        float(baseline_state["final_objective"])
                        - float(no_hit_state["final_objective"])
                    ),
                    "projected_current_gw_gain_vs_no_hit": (
                        float(baseline_expected_after_hits)
                        - float(no_hit_expected)
                    ),
                    "actual_current_gw_gain_vs_no_hit": (
                        float(baseline_realised)
                        - float(no_hit_realised)
                    ),
                    "no_hit_transfers": tuple(no_hit_action["transfers"]),
                }

            half_end = 19 if gameweek <= 19 else 38
            remaining_weeks = half_end - gameweek + 1
            remaining_chips = sum(chips_available.values())
            expiry_pressure = (
                remaining_chips > 0
                and remaining_weeks <= remaining_chips
            )

            chip_candidates = []
            if BACKTEST_INCLUDE_CHIPS:
                if chips_available.get("triple_captain", False):
                    chip_candidates.append({
                        "chip": "triple_captain",
                        "uplift": triple_captain_uplift(
                            baseline_detail
                        ),
                        "detail": baseline_detail,
                        "squad_ids": baseline_squad,
                    })
                if chips_available.get("bench_boost", False):
                    chip_candidates.append({
                        "chip": "bench_boost",
                        "uplift": bench_boost_uplift(
                            baseline_detail
                        ),
                        "detail": baseline_detail,
                        "squad_ids": baseline_squad,
                    })

                disruption = fixture_disruption_in_gameweek(
                    backtest_table,
                    gameweek,
                )
                can_review_fh = (
                    gameweek != 1
                    and chips_available.get("free_hit", False)
                    and (
                        disruption
                        or expiry_pressure
                        or not BACKTEST_FREE_HIT_ONLY_FOR_FIXTURE_DISRUPTION
                    )
                )
                if can_review_fh:
                    available_budget = (
                        bank + sum(selling_prices.values())
                    )
                    fh_result = optimise_chip_squad(
                        candidates,
                        projections,
                        [gameweek],
                        available_budget,
                    )
                    fh_squad = tuple(
                        sorted(
                            fh_result["squad"]["player_id"]
                            .astype(int)
                            .tolist()
                        )
                    )
                    fh_detail = exact_squad_gameweek_decision(
                        fh_squad,
                        gameweek,
                        fh_result["subset_candidates"],
                        fh_result["subset_projections"],
                    )
                    chip_candidates.append({
                        "chip": "free_hit",
                        "uplift": (
                            float(fh_detail["expected_total_score"])
                            - baseline_expected_after_hits
                        ),
                        "detail": fh_detail,
                        "squad_ids": fh_squad,
                    })

                can_review_wc = (
                    gameweek != 1
                    and chips_available.get("wildcard", False)
                    and (
                        gameweek % BACKTEST_WILDCARD_REVIEW_FREQUENCY == 0
                        or disruption
                        or expiry_pressure
                    )
                )
                if can_review_wc:
                    available_budget = (
                        bank + sum(selling_prices.values())
                    )
                    wc_result = optimise_chip_squad(
                        candidates,
                        projections,
                        horizon,
                        available_budget,
                    )
                    wc_squad = tuple(
                        sorted(
                            wc_result["squad"]["player_id"]
                            .astype(int)
                            .tolist()
                        )
                    )
                    wc_discounted_score = 0.0
                    for horizon_index, future_gw in enumerate(horizon):
                        detail = exact_squad_gameweek_decision(
                            wc_squad,
                            future_gw,
                            wc_result["subset_candidates"],
                            wc_result["subset_projections"],
                        )
                        wc_discounted_score += (
                            GAMEWEEK_DISCOUNT ** horizon_index
                            * float(detail["expected_total_score"])
                        )
                    baseline_discounted_score = sum(
                        GAMEWEEK_DISCOUNT ** index
                        * float(row["expected_score_after_hits"])
                        for index, row in enumerate(
                            baseline_state["path"]
                        )
                    )
                    wc_detail = exact_squad_gameweek_decision(
                        wc_squad,
                        gameweek,
                        wc_result["subset_candidates"],
                        wc_result["subset_projections"],
                    )
                    chip_candidates.append({
                        "chip": "wildcard",
                        "uplift": (
                            wc_discounted_score
                            - baseline_discounted_score
                        ),
                        "detail": wc_detail,
                        "squad_ids": wc_squad,
                    })

            blank_teams, double_teams = historical_fixture_shape_counts(
                backtest_table, gameweek
            )
            qualified = []
            for option in chip_candidates:
                threshold = BACKTEST_CHIP_MINIMUM_UPLIFT[option["chip"]]
                clears_uplift = option["uplift"] >= threshold or expiry_pressure
                structural, structural_reason = chip_structural_readiness(
                    option["chip"],
                    current_uplift=float(option["uplift"]),
                    blank_teams=blank_teams,
                    double_teams=double_teams,
                    expiry_pressure=expiry_pressure,
                    normal_transfer_count=int(baseline_action["transfer_count"]),
                    normal_hit_cost=float(baseline_action["hit_cost"]),
                    exceptional_thresholds=BACKTEST_CHIP_EXCEPTIONAL_UPLIFT,
                )
                if clears_uplift and structural:
                    option = dict(option)
                    option["normalised_strength"] = option["uplift"] / threshold
                    option["structural_reason"] = structural_reason
                    qualified.append(option)

            chosen_chip = (
                max(
                    qualified,
                    key=lambda option: option["normalised_strength"],
                )
                if qualified else None
            )

            transfers = tuple(baseline_action["transfers"])
            hit_cost = float(baseline_action["hit_cost"])
            permanent_squad_after = baseline_squad
            decision = baseline_detail
            scoring_chip = None

            if chosen_chip is not None:
                chip_name = chosen_chip["chip"]
                chips_available[chip_name] = False
                decision = chosen_chip["detail"]
                scoring_chip = (
                    chip_name
                    if chip_name in {
                        "triple_captain",
                        "bench_boost",
                    }
                    else None
                )

                if chip_name == "free_hit":
                    # Permanent team, purchase prices and bank are unchanged.
                    permanent_squad_after = squad_ids
                    transfers = ()
                    hit_cost = 0.0
                    free_transfers_after = preserve_free_transfers_after_unlimited_chip(
                        free_transfers
                    )
                elif chip_name == "wildcard":
                    permanent_squad_after = tuple(
                        chosen_chip["squad_ids"]
                    )
                    transfers = ()
                    hit_cost = 0.0
                    market_lookup = (
                        candidates.set_index("player_id")["price"]
                        .astype(float)
                        .to_dict()
                    )
                    purchase_prices = {
                        pid: market_lookup[pid]
                        for pid in permanent_squad_after
                    }
                    bank = (
                        bank + sum(selling_prices.values())
                        - sum(
                            market_lookup[pid]
                            for pid in permanent_squad_after
                        )
                    )
                    free_transfers_after = preserve_free_transfers_after_unlimited_chip(
                        free_transfers
                    )
                else:
                    free_transfers_after = int(
                        baseline_action["free_transfers_after"]
                    )
            else:
                free_transfers_after = int(
                    baseline_action["free_transfers_after"]
                )

            actual_score = actual_score_from_decision(
                decision,
                actual_gameweek,
                chip=scoring_chip,
            )
            realised_points = (
                actual_score["total_before_hits"] - hit_cost
            )

            # Apply permanent transfer finances after scoring decision.
            if chosen_chip is None or chosen_chip["chip"] in {
                "triple_captain",
                "bench_boost",
            }:
                market_lookup = (
                    candidates.set_index("player_id")["price"]
                    .astype(float)
                    .to_dict()
                )
                for outgoing_id, incoming_id in transfers:
                    outgoing_id = int(outgoing_id)
                    incoming_id = int(incoming_id)
                    purchase_prices.pop(outgoing_id, None)
                    purchase_prices[incoming_id] = market_lookup[
                        incoming_id
                    ]
                bank = tenths_to_price(
                    int(baseline_action["bank_after_tenths"])
                )

            squad_ids = tuple(
                sorted(int(pid) for pid in permanent_squad_after)
            )
            free_transfers = int(free_transfers_after)

            predicted_score = (
                float(decision["expected_total_score"])
                - hit_cost
            )
            if scoring_chip == "triple_captain":
                predicted_score += triple_captain_uplift(decision)
            elif scoring_chip == "bench_boost":
                predicted_score += bench_boost_uplift(decision)

            rows.append({
                "gameweek": gameweek,
                "chip": (
                    chosen_chip["chip"]
                    if chosen_chip is not None else None
                ),
                "transfers": len(transfers),
                "hit_cost": hit_cost,
                "free_transfers_after": free_transfers,
                "bank_after": bank,
                "predicted_score": float(predicted_score),
                "actual_score": realised_points,
                "captain_id": int(decision["captain_id"]),
                "vice_captain_id": int(
                    decision["vice_captain_id"]
                ),
                "squad_ids": squad_ids,
                "squad_ids_before": pre_decision_squad_ids,
                "bank_before": pre_decision_bank,
                "free_transfers_before": pre_decision_free_transfers,
                "availability_known_share": float(
                    candidates.get(
                        "availability_known",
                        pd.Series(False, index=candidates.index),
                    ).mean()
                ),
                "hit_counterfactual_evaluated": bool(
                    hit_counterfactual["evaluated"]
                ),
                "hit_projected_horizon_gain_vs_no_hit": (
                    hit_counterfactual[
                        "projected_horizon_gain_vs_no_hit"
                    ]
                ),
                "hit_projected_current_gw_gain_vs_no_hit": (
                    hit_counterfactual[
                        "projected_current_gw_gain_vs_no_hit"
                    ]
                ),
                "hit_actual_current_gw_gain_vs_no_hit": (
                    hit_counterfactual[
                        "actual_current_gw_gain_vs_no_hit"
                    ]
                ),
                "hit_no_hit_transfers": (
                    hit_counterfactual["no_hit_transfers"]
                ),
                "transfer_robustness_applied": bool(
                    robustness_audit.get("applied", False)
                ),
                "transfer_robustness_objective_given_up": float(
                    robustness_audit.get("objective_given_up", 0.0)
                ),
            })

            # Static benchmark: retain the opening squad, but optimise its
            # weekly XI, bench order and captain using the same projections.
            static_candidates = historical_candidate_snapshot(
                backtest_table,
                gameweek,
                static_squad_ids,
            )
            static_projection = historical_projection_window(
                backtest_table,
                [gameweek],
                set(static_candidates["player_id"]),
            )
            static_detail = exact_squad_gameweek_decision(
                static_squad_ids,
                gameweek,
                static_candidates,
                static_projection,
            )
            static_actual = actual_score_from_decision(
                static_detail,
                actual_gameweek,
            )
            static_rows.append({
                "gameweek": gameweek,
                "actual_score": static_actual[
                    "total_before_hits"
                ],
            })

        results = pd.DataFrame(rows)
        results["cumulative_points"] = results[
            "actual_score"
        ].cumsum()
        results["cumulative_predicted"] = results[
            "predicted_score"
        ].cumsum()

        static_results = pd.DataFrame(static_rows)
        static_results["cumulative_points"] = static_results[
            "actual_score"
        ].cumsum()

        static_signature = stable_configuration_signature({
            "opening_squad_signature": str(opening_state["signature"]),
            "gameweeks": static_results["gameweek"].astype(int).tolist(),
            "actual_scores": [
                round(float(value), 6)
                for value in static_results["actual_score"].tolist()
            ],
        })

        return {
            "initial_squad": initial_squad_frame.copy(),
            "initial_squad_signature": str(opening_state["signature"]),
            "opening_horizon": int(
                opening_state.get("opening_horizon", BACKTEST_HORIZON)
            ),
            "weekly_results": results,
            "static_benchmark": static_results,
            "static_benchmark_signature": static_signature,
            "final_squad_ids": squad_ids,
            "final_bank": bank,
            "remaining_chips": chips_available,
        }
    finally:
        (
            TRANSFER_BEAM_WIDTH,
            TRANSFER_ACTION_BEAM_WIDTH,
            TRANSFER_TOP_XP_PER_POSITION,
            TRANSFER_TOP_VALUE_PER_POSITION,
        ) = original_search_settings

In [ ]:
if RUN_SEQUENTIAL_BACKTEST:
    if AUTONOMOUS_REQUIRE_VALIDATION_ARTIFACTS and (
        xp_calibration_cache is None or premium_attacker_cache is None
    ):
        raise RuntimeError(
            "RUN_SEQUENTIAL_BACKTEST=True requires the current signed validation artifacts. "
            "First run Section 11 with RUN_FULL_POLICY_BACKTEST=True, restart "
            "the kernel, and then rerun with only RUN_SEQUENTIAL_BACKTEST=True."
        )
    deadline_frozen_forecast_cache.clear()
    deadline_frozen_forecast_audit.clear()

    backtest_result = run_sequential_fpl_backtest(
        historical_backtest_table,
        backtest_gameweeks,
    )
    sequential_cache_payload = {
        "cache_signature": SEQUENTIAL_CACHE_SIGNATURE,
        "project_version": PROJECT_VERSION,
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "result": backtest_result,
    }
    pd.to_pickle(
        sequential_cache_payload,
        SEQUENTIAL_BACKTEST_CACHE_PATH,
        compression="gzip",
    )
    print("Saved sequential backtest cache:", SEQUENTIAL_BACKTEST_CACHE_PATH)

elif SEQUENTIAL_BACKTEST_CACHE_PATH.exists() and not FORCE_REBUILD_VALIDATION_CACHES:
    sequential_cache_payload = pd.read_pickle(
        SEQUENTIAL_BACKTEST_CACHE_PATH,
        compression="gzip",
    )
    cache_ok = (
        isinstance(sequential_cache_payload, dict)
        and sequential_cache_payload.get("cache_signature") == SEQUENTIAL_CACHE_SIGNATURE
        and "result" in sequential_cache_payload
    )
    if STRICT_CACHE_SIGNATURES and not cache_ok:
        backtest_result = None
        print(
            "Ignoring stale sequential backtest cache. Set "
            "RUN_SEQUENTIAL_BACKTEST=True after Section 11 validation."
        )
    else:
        backtest_result = (
            sequential_cache_payload.get("result")
            if isinstance(sequential_cache_payload, dict)
            else sequential_cache_payload
        )
        print("Loaded sequential backtest cache:", SEQUENTIAL_BACKTEST_CACHE_PATH)
else:
    backtest_result = None
    print(
        "Sequential backtest skipped. Set RUN_SEQUENTIAL_BACKTEST=True only "
        "after the current Section 11 validation artifacts exist."
    )

if backtest_result is not None:
    backtest_weekly_results = backtest_result["weekly_results"]
    backtest_static_benchmark = backtest_result["static_benchmark"]
    backtest_summary = pd.DataFrame([
        {
            "strategy": "Rolling optimiser",
            "gameweeks": len(backtest_weekly_results),
            "total_points": backtest_weekly_results["actual_score"].sum(),
            "total_hits": backtest_weekly_results["hit_cost"].sum(),
            "transfers": backtest_weekly_results["transfers"].sum(),
            "chips_used": backtest_weekly_results["chip"].notna().sum(),
        },
        {
            "strategy": "Static opening squad",
            "gameweeks": len(backtest_static_benchmark),
            "total_points": backtest_static_benchmark["actual_score"].sum(),
            "total_hits": 0.0,
            "transfers": 0,
            "chips_used": 0,
        },
    ])
    display(backtest_summary)
    display(backtest_weekly_results)
else:
    backtest_weekly_results = pd.DataFrame()
    backtest_static_benchmark = pd.DataFrame()
    backtest_summary = pd.DataFrame()


### Backtest interpretation and remaining limitations

This version freezes every multi-Gameweek forecast at the relevant deadline,
so future player form and future team results cannot enter earlier transfer or
chip decisions.

The backtest remains an initial systems test:

- model parameters are fitted once on 2022/23–2024/25 and are not refitted
  during 2025/26;
- fixture dates and opponents are treated as known according to the archived
  schedule;
- future player transfers are not anticipated: each horizon assumes the
  player remains at the club represented at its forecast origin;
- future prices are held constant within each horizon and refreshed at the
  next simulated deadline;
- the transfer search is approximate;
- blank-team players can be underrepresented in the historical candidate pool
  when the archive contains no player-fixture row for that Gameweek;
- chip thresholds are policy parameters rather than historically calibrated
  values;
- defensive-action coverage is concentrated in 2025/26, so early-season
  defensive-contribution estimates rely heavily on sparse priors.

After a successful strict run, the next validation step is a **walk-forward
multi-season backtest**. Each season should be held out in turn, with transfer
and chip thresholds selected only from earlier seasons.


# 11. Walk-forward multi-season validation

A single successful season is encouraging but not enough to establish that the
decision system generalises. This section evaluates the model in expanding
chronological folds:

| Holdout season | Training seasons |
|---|---|
| 2023/24 | 2022/23 |
| 2024/25 | 2022/23–2023/24 |
| 2025/26 | 2022/23–2024/25 |

Every fold refits the minutes, team-strength and player-event models using only
earlier seasons. The held-out season is never used for fitting or model
selection.

There is an important rules distinction:

- defensive-contribution points were introduced in 2025/26;
- five banked free transfers began in 2024/25;
- chip structures differed before 2025/26.

The cross-season strategy comparison therefore runs **without chips**. This
isolates opening-squad selection, line-ups, captaincy, transfers and hits under
the appropriate free-transfer cap. The full eight-chip strategy remains
evaluated separately on 2025/26, whose chip and defensive-contribution rules
most closely match 2026/27.


### v2.1 validation gates

Two forecast corrections are now treated as **candidates**, not assumptions:

- premium-attacker xG/xA correction;
- final xP calibration.

Each is activated in production only when prior chronological folds show an
out-of-sample improvement under the thresholds configured in the first code
cell. Rejected corrections remain visible in the scorecards but are not used by
the optimiser. Validation caches are signed with the current model/configuration
hash and stale caches are ignored automatically.


### v2.2.5 policy-validation workflow

The transfer-policy scorecard is now validated under the same adaptive transfer
search used by weekly production. A full run is intentionally **one-time**:

1. set `RUN_FULL_POLICY_BACKTEST = True`;
2. keep `WALK_FORWARD_MAX_GAMEWEEKS = None`;
3. run Section 11.

Each completed season/profile fold is written immediately to
`POLICY_BACKTEST_CHECKPOINT_PATH`. If the kernel stops, rerun Section 11 and the
matching signed folds are skipped automatically. Only a complete full-season
matrix whose forecast, policy, and transfer-search signatures all match the
current notebook can become the production scorecard. Smoke-test or stale
checkpoints can never overwrite the production policy cache.

For normal weekly use, switch `RUN_FULL_POLICY_BACKTEST` back to `False`. The
validated scorecard is then loaded from disk and Section 12 reports its
provenance explicitly.


In [ ]:
from contextlib import contextmanager

WALK_FORWARD_HOLDOUT_SEASONS = [
    "2023-24",
    "2024-25",
    "2025-26",
]

# These profiles isolate the value of planning horizon and transfer hits.
#
# All profiles may use up to two transfers when enough free transfers are
# available. The no-hit profiles never make more transfers than the current
# free-transfer allowance.
WALK_FORWARD_POLICY_PROFILES = {
    "greedy_1gw_no_hits": {
        "horizon": 1,
        "max_transfers_per_gameweek": 2,
        "allow_hits": False,
        "bank_aware": False,
        "min_gain_per_extra_transfer": 0.0,
    },
    "greedy_1gw_bank_aware": {
        "horizon": 1,
        "max_transfers_per_gameweek": 2,
        "allow_hits": False,
        "bank_aware": True,
        "min_gain_per_extra_transfer": BANK_AWARE_MIN_GAIN_PER_EXTRA_TRANSFER,
    },
    "rolling_3gw_no_hits": {
        "horizon": 3,
        "max_transfers_per_gameweek": 2,
        "allow_hits": False,
        "bank_aware": False,
        "min_gain_per_extra_transfer": 0.0,
    },
    "rolling_6gw_no_hits": {
        "horizon": 6,
        "max_transfers_per_gameweek": 2,
        "allow_hits": False,
        "bank_aware": False,
        "min_gain_per_extra_transfer": 0.0,
    },
    "rolling_6gw_hits": {
        "horizon": 6,
        "max_transfers_per_gameweek": 2,
        "allow_hits": True,
        "bank_aware": False,
        "min_gain_per_extra_transfer": 0.0,
    },
}

WALK_FORWARD_ACTIVE_PROFILES = [
    "greedy_1gw_no_hits",
    "greedy_1gw_bank_aware",
    "rolling_3gw_no_hits",
    "rolling_6gw_no_hits",
    "rolling_6gw_hits",
]

# Cross-season tests disable chips because historical chip rules differ.
WALK_FORWARD_INCLUDE_CHIPS = False


walk_forward_fold_cache: dict[str, dict[str, object]] = {}
walk_forward_shared_opening_cache: dict[tuple[str, str], dict[str, object]] = {}


POLICY_CACHE_SIGNATURE = stable_configuration_signature({
    "forecast_signature": FORECAST_CACHE_SIGNATURE,
    "project_version": PROJECT_VERSION,
    "model_code_version": MODEL_CODE_VERSION,
    "profiles": WALK_FORWARD_POLICY_PROFILES,
    "active_profiles": WALK_FORWARD_ACTIVE_PROFILES,
    "shared_opening_horizon": POLICY_SHARED_OPENING_HORIZON,
    "include_chips": WALK_FORWARD_INCLUDE_CHIPS,
    "gameweek_discount": GAMEWEEK_DISCOUNT,
    "squad_depth_weight": SQUAD_DEPTH_WEIGHT,
    "transfer_hit_cost": TRANSFER_HIT_COST,
    "max_bank_ft": MAX_BANKED_FREE_TRANSFERS,
    "terminal_value_weight": TRANSFER_TERMINAL_VALUE_WEIGHT,
    "transfer_search_signature": TRANSFER_SEARCH_SIGNATURE,
    "backtest_adaptive_search": BACKTEST_USE_ADAPTIVE_TRANSFER_SEARCH,
    "transfer_dominance_pruning": TRANSFER_DOMINANCE_PRUNING,
    "swap_candidate_cap": TRANSFER_MAX_SWAP_CANDIDATES_PER_PARTIAL,
    "emergency_repair_enabled": EMERGENCY_STRUCTURAL_REPAIR_ENABLED,
    "captain_highest_xp": PRODUCTION_CAPTAIN_HIGHEST_XP,
    "max_gameweeks": WALK_FORWARD_MAX_GAMEWEEKS,
    "autosub_rules_version": "nominal_formation_realised_scoring_v3",
})


In [ ]:
def season_start_year(season: str) -> int:
    """Return the first calendar year from an FPL season label."""
    return int(str(season).split("-")[0])


def earlier_historical_seasons(
    holdout_season: str,
) -> list[str]:
    """Return all configured seasons strictly earlier than the holdout."""
    holdout_start = season_start_year(holdout_season)
    return [
        season
        for season in HISTORICAL_SEASONS
        if season_start_year(season) < holdout_start
    ]


def build_walk_forward_fold_bundle(
    holdout_season: str,
) -> dict[str, object]:
    """
    Refit every forecast layer using only seasons before the holdout.

    The returned object contains the fold-specific models, priors, holdout
    rows and expected-point forecasts required by the strategy backtest.
    """
    if holdout_season in walk_forward_fold_cache:
        return walk_forward_fold_cache[holdout_season]

    # v2.5: signed persistent cache for the expensive base forecast fold.
    # The signature deliberately uses the unchanged v2.4 base forecast
    # compatibility hash, so diagnostic/post-processing iterations do not
    # refit minutes/team/event models unnecessarily.
    _fold_cache_path = (
        WALK_FORWARD_FOLD_DISK_CACHE_DIRECTORY
        / f"{str(holdout_season).replace('/', '_')}.pkl.gz"
    )
    if (
        WALK_FORWARD_FOLD_DISK_CACHE_ENABLED
        and not FORCE_REBUILD_VALIDATION_CACHES
        and _fold_cache_path.exists()
    ):
        try:
            _cached_payload = pd.read_pickle(
                _fold_cache_path,
                compression="gzip",
            )
            _cache_ok = bool(
                isinstance(_cached_payload, dict)
                and _cached_payload.get("forecast_signature")
                    == FORECAST_CACHE_SIGNATURE
                and str(_cached_payload.get("holdout_season"))
                    == str(holdout_season)
                and isinstance(_cached_payload.get("bundle"), dict)
            )
            if _cache_ok:
                _cached_bundle = _cached_payload["bundle"]
                walk_forward_fold_cache[str(holdout_season)] = _cached_bundle
                return _cached_bundle
        except Exception as _cache_exc:
            warnings.warn(
                f"Could not load cached walk-forward fold {holdout_season}: "
                f"{_cache_exc}. Rebuilding it."
            )

    development_seasons = earlier_historical_seasons(
        holdout_season
    )
    if not development_seasons:
        raise ValueError(
            f"No earlier training season is available for {holdout_season}."
        )

    fold_minutes_training = historical_model_data.loc[
        historical_model_data["season"].isin(
            development_seasons
        )
    ].copy()
    fold_minutes_holdout = historical_model_data.loc[
        historical_model_data["season"].eq(holdout_season)
    ].copy()

    fold_minutes_models = fit_minutes_model_bundle(
        fold_minutes_training
    )

    fold_team_training = historical_team_fixtures.loc[
        historical_team_fixtures["season"].isin(
            development_seasons
        )
    ].copy()
    fold_team_models = fit_team_goal_models(
        fold_team_training
    )

    fold_seasons = development_seasons + [holdout_season]
    fold_team_rows = historical_team_fixtures.loc[
        historical_team_fixtures["season"].isin(fold_seasons)
    ].copy()

    fold_team_predictions = pd.concat(
        [
            fold_team_rows[
                ["season", "fixture_id", "team_key"]
            ].reset_index(drop=True),
            predict_team_goal_components(
                fold_team_models,
                fold_team_rows,
            ).reset_index(drop=True),
        ],
        axis=1,
    )

    fold_points_data = (
        historical_model_data.loc[
            historical_model_data["season"].isin(fold_seasons)
        ]
        .merge(
            fold_team_predictions,
            on=["season", "fixture_id", "team_key"],
            how="left",
            validate="many_to_one",
        )
    )
    fold_points_data = add_player_per90_history_features(
        fold_points_data
    )
    fold_points_data = add_player_event_targets(
        fold_points_data
    )
    fold_points_data = fold_points_data.loc[
        fold_points_data["model_position"].isin(
            PLAYER_SCORING_POSITIONS
        )
    ].copy()

    fold_development_points = fold_points_data.loc[
        fold_points_data["season"].isin(
            development_seasons
        )
    ].copy()
    fold_holdout_points = fold_points_data.loc[
        fold_points_data["season"].eq(holdout_season)
    ].copy()

    fold_player_event_models = fit_player_event_rate_models(
        fold_development_points
    )

    fold_holdout_minutes = predict_minutes_components(
        fold_minutes_models,
        fold_holdout_points,
    )
    fold_holdout_event_rates = predict_player_event_rates(
        fold_player_event_models,
        fold_holdout_points,
    )

    fold_xp_inputs = pd.concat(
        [
            fold_holdout_points.reset_index(drop=True),
            fold_holdout_minutes.reset_index(drop=True),
            fold_holdout_event_rates.reset_index(drop=True),
        ],
        axis=1,
    )

    fold_defensive_action_priors = (
        defensive_action_position_priors(
            fold_development_points
        )
    )
    fold_defensive_return_priors = (
        defensive_return_position_priors(
            fold_development_points
        )
    )
    fold_rare_event_rates = rare_event_position_rates(
        fold_development_points
    )

    v26_set_conversion(fold_development_points, label=f"fold:{holdout_season}")
    fold_xp_components = calculate_component_expected_points(
        fold_xp_inputs,
        fold_defensive_action_priors,
        fold_rare_event_rates,
        fold_defensive_return_priors,
    )

    # Defensive contributions were not part of FPL scoring before 2025/26.
    if season_start_year(holdout_season) < 2025:
        fold_xp_components["expected_points"] = (
            fold_xp_components["expected_points"]
            - fold_xp_components["xp_defensive_contribution"]
        )
        fold_xp_components["xp_defensive_contribution"] = 0.0

    fold_expected_points = pd.concat(
        [
            fold_xp_inputs.reset_index(drop=True),
            fold_xp_components.reset_index(drop=True),
        ],
        axis=1,
    )

    bundle = {
        "holdout_season": holdout_season,
        "development_seasons": development_seasons,
        "minutes_models": fold_minutes_models,
        "team_goal_models": fold_team_models,
        "player_event_models": fold_player_event_models,
        "defensive_action_priors": (
            fold_defensive_action_priors
        ),
        "defensive_return_priors": (
            fold_defensive_return_priors
        ),
        "rare_event_rates": fold_rare_event_rates,
        "development_points_data": fold_development_points,
        "holdout_points_data": fold_holdout_points,
        "holdout_expected_points": fold_expected_points,
    }
    walk_forward_fold_cache[holdout_season] = bundle

    if WALK_FORWARD_FOLD_DISK_CACHE_ENABLED:
        try:
            _fold_cache_path.parent.mkdir(parents=True, exist_ok=True)
            pd.to_pickle(
                {
                    "forecast_signature": FORECAST_CACHE_SIGNATURE,
                    "holdout_season": str(holdout_season),
                    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
                    "bundle": bundle,
                },
                _fold_cache_path,
                compression="gzip",
            )
        except Exception as _cache_exc:
            warnings.warn(
                f"Could not persist walk-forward fold {holdout_season}: "
                f"{_cache_exc}"
            )

    return bundle


def walk_forward_prediction_metrics(
    bundle: dict[str, object],
) -> dict[str, object]:
    """Calculate prediction metrics for one untouched season."""
    holdout = bundle["holdout_expected_points"]
    development_seasons = bundle["development_seasons"]

    valid = holdout["target_fpl_points"].notna()
    actual = holdout.loc[valid, "target_fpl_points"]
    predicted = holdout.loc[valid, "expected_points"]

    development_rows = historical_points_data.loc[
        historical_points_data["season"].isin(
            development_seasons
        )
    ]
    position_prior = (
        development_rows.groupby("model_position")[
            "target_fpl_points"
        ].mean()
    )
    global_prior = float(
        development_rows["target_fpl_points"].mean()
    )
    baseline = (
        coalesced_history_baseline(
            holdout,
            [
                "points_per_fixture_last_5_gws",
                "points_per_fixture_last_10_gws",
                "career_points_per_fixture",
            ],
            position_prior,
        )
        .fillna(global_prior)
        .clip(lower=-3.0, upper=20.0)
    )

    minutes_actual = pd.to_numeric(
        holdout.loc[valid, "target_minutes"],
        errors="coerce",
    )
    minutes_predicted = pd.to_numeric(
        holdout.loc[valid, "expected_minutes"],
        errors="coerce",
    )

    return {
        "holdout_season": bundle["holdout_season"],
        "training_seasons": ", ".join(development_seasons),
        "rows": int(valid.sum()),
        "xp_mae": mean_absolute_error(actual, predicted),
        "baseline_mae": mean_absolute_error(actual, baseline.loc[valid]),
        "xp_rmse": np.sqrt(
            mean_squared_error(actual, predicted)
        ),
        "xp_spearman": spearmanr(
            actual,
            predicted,
            nan_policy="omit",
        ).statistic,
        "baseline_spearman": spearmanr(
            actual,
            baseline.loc[valid],
            nan_policy="omit",
        ).statistic,
        "xp_bias": float((predicted - actual).mean()),
        "minutes_mae": mean_absolute_error(
            minutes_actual,
            minutes_predicted,
        ),
    }


### Out-of-sample xP calibration

Production xP is calibrated from predictions that were genuinely out of sample. For each historical fold, calibration is fitted only on earlier out-of-sample folds; the live 2026/27 calibration is then fitted on all available out-of-sample seasons. This prevents the calibration layer from learning the same outcomes it is evaluated on.


In [ ]:
from sklearn.isotonic import IsotonicRegression

PRICE_BAND_LABELS = ["lt5", "5to6_5", "6_5to8", "8to10", "ge10"]


def fit_xp_calibration_payload(rows: pd.DataFrame) -> dict | None:
    data = rows.copy()
    if data.empty:
        return None
    data["raw_xp"] = pd.to_numeric(data["expected_points"], errors="coerce")
    data["actual"] = pd.to_numeric(data["target_fpl_points"], errors="coerce")
    price_col = "price_at_collection" if "price_at_collection" in data.columns else "price"
    data["price"] = pd.to_numeric(data[price_col], errors="coerce")
    data = data.loc[data["raw_xp"].notna() & data["actual"].notna()].copy()

    payload = {"positions": {}, "rows": int(len(data))}
    for position in PLAYER_SCORING_POSITIONS:
        subset = data.loc[data["model_position"].eq(position)].copy()
        if len(subset) < int(XP_CALIBRATION_MIN_ROWS_PER_POSITION):
            continue
        model = IsotonicRegression(out_of_bounds="clip")
        model.fit(subset["raw_xp"], subset["actual"])
        base_cal = model.predict(subset["raw_xp"])
        residual = subset["actual"].to_numpy(dtype=float) - base_cal
        band_residual = {}
        for band in PRICE_BAND_LABELS:
            mask = subset["price"].map(_price_band).eq(band).to_numpy(dtype=bool)
            if mask.sum() >= 100:
                mean_residual = float(np.mean(residual[mask]))
                shrink = float(mask.sum() / (mask.sum() + 400.0))
                band_residual[band] = mean_residual * shrink
            else:
                band_residual[band] = 0.0
        payload["positions"][position] = {
            "x": [float(v) for v in model.X_thresholds_],
            "y": [float(v) for v in model.y_thresholds_],
            "rows": int(len(subset)),
            "price_band_residual": band_residual,
        }
    return payload if payload["positions"] else None


def _weighted_metric_mean(metrics: list[dict], column: str) -> float:
    usable = [row for row in metrics if row.get("candidate_evaluated")]
    if not usable:
        return np.nan
    weights = np.array([max(1, int(row.get("rows", 1))) for row in usable], dtype=float)
    values = np.array([float(row[column]) for row in usable], dtype=float)
    return float(np.average(values, weights=weights))


def decide_xp_calibration(metrics: list[dict]) -> dict:
    usable = [row for row in metrics if row.get("candidate_evaluated")]
    raw_mae = _weighted_metric_mean(usable, "raw_mae")
    cal_mae = _weighted_metric_mean(usable, "candidate_mae")
    raw_spearman = _weighted_metric_mean(usable, "raw_spearman")
    cal_spearman = _weighted_metric_mean(usable, "candidate_spearman")
    raw_abs_bias = _weighted_metric_mean(usable, "raw_abs_bias")
    cal_abs_bias = _weighted_metric_mean(usable, "candidate_abs_bias")
    mae_improvement = raw_mae - cal_mae if np.isfinite(raw_mae) and np.isfinite(cal_mae) else np.nan
    spearman_change = cal_spearman - raw_spearman if np.isfinite(raw_spearman) and np.isfinite(cal_spearman) else np.nan
    bias_change = cal_abs_bias - raw_abs_bias if np.isfinite(raw_abs_bias) and np.isfinite(cal_abs_bias) else np.nan

    auto_approved = (
        len(usable) >= int(XP_CALIBRATION_MIN_EVALUATED_FOLDS)
        and np.isfinite(mae_improvement)
        and mae_improvement >= float(XP_CALIBRATION_MIN_MAE_IMPROVEMENT)
        and spearman_change >= -float(XP_CALIBRATION_MAX_SPEARMAN_DEGRADATION)
        and bias_change <= float(XP_CALIBRATION_MAX_ABS_BIAS_WORSENING)
    )
    approved = _validation_mode_allows(XP_CALIBRATION_MODE, auto_approved)
    return {
        "mode": XP_CALIBRATION_MODE,
        "auto_approved": bool(auto_approved),
        "approved": bool(approved),
        "evaluated_folds": len(usable),
        "weighted_raw_mae": raw_mae,
        "weighted_candidate_mae": cal_mae,
        "mae_improvement": mae_improvement,
        "weighted_raw_spearman": raw_spearman,
        "weighted_candidate_spearman": cal_spearman,
        "spearman_change": spearman_change,
        "weighted_raw_abs_bias": raw_abs_bias,
        "weighted_candidate_abs_bias": cal_abs_bias,
        "abs_bias_change": bias_change,
    }


def premium_candidate_metrics(raw_fold: pd.DataFrame, adjusted_fold: pd.DataFrame, season: str) -> dict:
    price_col = "price_at_collection" if "price_at_collection" in raw_fold.columns else "price"
    mask = (
        raw_fold["model_position"].isin(["MID", "FWD"])
        & pd.to_numeric(raw_fold[price_col], errors="coerce").ge(float(PREMIUM_ATTACKER_PRICE_THRESHOLD))
        & raw_fold["target_fpl_points"].notna()
    )
    raw = raw_fold.loc[mask].copy()
    adj = adjusted_fold.loc[mask].copy()
    if raw.empty:
        return {"holdout_season": season, "rows": 0, "candidate_evaluated": False}
    actual = pd.to_numeric(raw["target_fpl_points"], errors="coerce")
    raw_xp = pd.to_numeric(raw["expected_points"], errors="coerce")
    adj_xp = pd.to_numeric(adj["expected_points"], errors="coerce")
    return {
        "holdout_season": season,
        "rows": int(len(raw)),
        "candidate_evaluated": bool((adj_xp - raw_xp).abs().gt(1e-12).any()),
        "raw_mae": mean_absolute_error(actual, raw_xp),
        "candidate_mae": mean_absolute_error(actual, adj_xp),
        "raw_spearman": spearmanr(actual, raw_xp, nan_policy="omit").statistic,
        "candidate_spearman": spearmanr(actual, adj_xp, nan_policy="omit").statistic,
        "mean_raw_xp": float(raw_xp.mean()),
        "mean_candidate_xp": float(adj_xp.mean()),
        "mean_actual": float(actual.mean()),
    }


def decide_premium_adjustment(metrics: list[dict]) -> dict:
    usable = [row for row in metrics if row.get("candidate_evaluated")]
    raw_mae = _weighted_metric_mean(usable, "raw_mae")
    adj_mae = _weighted_metric_mean(usable, "candidate_mae")
    raw_spearman = _weighted_metric_mean(usable, "raw_spearman")
    adj_spearman = _weighted_metric_mean(usable, "candidate_spearman")
    mae_improvement = raw_mae - adj_mae if np.isfinite(raw_mae) and np.isfinite(adj_mae) else np.nan
    spearman_change = adj_spearman - raw_spearman if np.isfinite(raw_spearman) and np.isfinite(adj_spearman) else np.nan
    auto_approved = (
        len(usable) >= int(PREMIUM_ATTACKER_MIN_EVALUATED_FOLDS)
        and np.isfinite(mae_improvement)
        and mae_improvement >= float(PREMIUM_ATTACKER_MIN_XP_MAE_IMPROVEMENT)
        and spearman_change >= -float(PREMIUM_ATTACKER_MAX_SPEARMAN_DEGRADATION)
    )
    approved = _validation_mode_allows(PREMIUM_ATTACKER_ADJUSTMENT_MODE, auto_approved)
    return {
        "mode": PREMIUM_ATTACKER_ADJUSTMENT_MODE,
        "auto_approved": bool(auto_approved),
        "approved": bool(approved),
        "evaluated_folds": len(usable),
        "weighted_raw_mae": raw_mae,
        "weighted_candidate_mae": adj_mae,
        "mae_improvement": mae_improvement,
        "weighted_raw_spearman": raw_spearman,
        "weighted_candidate_spearman": adj_spearman,
        "spearman_change": spearman_change,
    }


premium_oos_rows = []
premium_fold_metrics = []
premium_fold_strategy_payloads = {}
calibration_oos_rows = []
calibration_fold_metrics = []
calibration_fold_strategy_payloads = {}

for fold_season in WALK_FORWARD_HOLDOUT_SEASONS:
    bundle = build_walk_forward_fold_bundle(fold_season)
    raw_fold = bundle["holdout_expected_points"].copy()
    raw_fold["holdout_season"] = fold_season

    # ----- Premium-attacker correction candidate -----------------------------
    prior_premium_rows = (
        pd.concat(premium_oos_rows, ignore_index=True)
        if premium_oos_rows else pd.DataFrame()
    )
    premium_candidate = fit_premium_attacker_adjustment_payload(prior_premium_rows)
    premium_pre_fold_decision = decide_premium_adjustment(premium_fold_metrics)
    premium_strategy_payload = (
        premium_candidate if premium_pre_fold_decision["approved"] else None
    )
    premium_fold_strategy_payloads[fold_season] = premium_strategy_payload

    premium_candidate_fold = apply_premium_attacker_adjustment(
        raw_fold,
        premium_candidate,
        position_col="model_position",
        price_col="price_at_collection",
    )
    premium_fold_metrics.append(
        premium_candidate_metrics(raw_fold, premium_candidate_fold, fold_season)
    )

    strategy_fold = apply_premium_attacker_adjustment(
        raw_fold,
        premium_strategy_payload,
        position_col="model_position",
        price_col="price_at_collection",
    )
    premium_oos_rows.append(raw_fold)

    # ----- Sparse-player fallback --------------------------------------------
    fold_price_prior = build_position_price_prior_table(
        bundle["development_points_data"]
    )
    strategy_fold = apply_sparse_player_market_prior(
        strategy_fold,
        prior_table=fold_price_prior,
    )

    # ----- xP calibration candidate ------------------------------------------
    prior_calibration_rows = (
        pd.concat(calibration_oos_rows, ignore_index=True)
        if calibration_oos_rows else pd.DataFrame()
    )
    calibration_candidate = fit_xp_calibration_payload(prior_calibration_rows)
    calibration_pre_fold_decision = decide_xp_calibration(calibration_fold_metrics)
    calibration_strategy_payload = (
        calibration_candidate if calibration_pre_fold_decision["approved"] else None
    )
    calibration_fold_strategy_payloads[fold_season] = calibration_strategy_payload

    candidate_calibrated = apply_xp_calibration_payload(
        strategy_fold,
        calibration_candidate,
        position_col="model_position",
        price_col="price_at_collection",
        value_col="expected_points",
    )
    valid = candidate_calibrated["target_fpl_points"].notna()
    actual = pd.to_numeric(candidate_calibrated.loc[valid, "target_fpl_points"], errors="coerce")
    raw_xp = pd.to_numeric(strategy_fold.loc[valid, "expected_points"], errors="coerce")
    candidate_xp = pd.to_numeric(candidate_calibrated.loc[valid, "expected_points"], errors="coerce")
    evaluated = calibration_candidate is not None
    calibration_fold_metrics.append({
        "holdout_season": fold_season,
        "rows": int(valid.sum()),
        "candidate_evaluated": bool(evaluated),
        "raw_mae": mean_absolute_error(actual, raw_xp),
        "candidate_mae": mean_absolute_error(actual, candidate_xp),
        "raw_bias": float((raw_xp - actual).mean()),
        "candidate_bias": float((candidate_xp - actual).mean()),
        "raw_abs_bias": float(abs((raw_xp - actual).mean())),
        "candidate_abs_bias": float(abs((candidate_xp - actual).mean())),
        "raw_spearman": spearmanr(actual, raw_xp, nan_policy="omit").statistic,
        "candidate_spearman": spearmanr(actual, candidate_xp, nan_policy="omit").statistic,
    })
    calibration_oos_rows.append(strategy_fold)

# Final production candidates are fitted on every completed OOS fold. Whether
# they are actually activated is decided only by the prior-fold validation.
all_premium_oos_rows = pd.concat(premium_oos_rows, ignore_index=True)
live_premium_candidate = fit_premium_attacker_adjustment_payload(all_premium_oos_rows)
premium_production_decision = decide_premium_adjustment(premium_fold_metrics)

all_calibration_oos_rows = pd.concat(calibration_oos_rows, ignore_index=True)
live_xp_calibration_candidate = fit_xp_calibration_payload(all_calibration_oos_rows)
xp_production_decision = decide_xp_calibration(calibration_fold_metrics)

premium_attacker_artifact = {
    "version": 1,
    "forecast_signature": FORECAST_CACHE_SIGNATURE,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "live_candidate": live_premium_candidate,
    "production_decision": premium_production_decision,
    "fold_strategy_payloads": premium_fold_strategy_payloads,
    "fold_metrics": premium_fold_metrics,
}
premium_attacker_artifact["artifact_id"] = stable_configuration_signature({
    "forecast_signature": FORECAST_CACHE_SIGNATURE,
    "decision": premium_production_decision,
    "metrics": premium_fold_metrics,
})
PREMIUM_ATTACKER_CACHE_PATH.write_text(
    json.dumps(premium_attacker_artifact, indent=2), encoding="utf-8"
)

xp_calibration_artifact = {
    "version": 3,
    "forecast_signature": FORECAST_CACHE_SIGNATURE,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "live_candidate": live_xp_calibration_candidate,
    "production_decision": xp_production_decision,
    "fold_strategy_payloads": calibration_fold_strategy_payloads,
    "fold_metrics": calibration_fold_metrics,
}
xp_calibration_artifact["artifact_id"] = stable_configuration_signature({
    "forecast_signature": FORECAST_CACHE_SIGNATURE,
    "decision": xp_production_decision,
    "metrics": calibration_fold_metrics,
})
XP_CALIBRATION_CACHE_PATH.write_text(
    json.dumps(xp_calibration_artifact, indent=2), encoding="utf-8"
)

# Make the new artifacts visible in this kernel for inspection.
xp_calibration_cache = xp_calibration_artifact
premium_attacker_cache = premium_attacker_artifact


def _v26_refresh_live_after_validation():
    """Rebuild the live post-layers against the just-refitted artifacts.

    Returns True when the live forecast was refreshed. Without this the
    Section 12 artifact-state guard fires on every run that rebuilds an
    artifact, which is every first run after a signature change.
    """
    g = globals()
    if not g.get('V26_ENABLED', False):
        return False
    needed = ('live_expected_points_inputs', 'live_xp_components',
              'LIVE_VALIDATION_ARTIFACT_STATE')
    if any(g.get(name) is None for name in needed):
        return False

    state = g['LIVE_VALIDATION_ARTIFACT_STATE']
    fresh_xp_id = xp_calibration_cache.get('artifact_id') if xp_calibration_cache else None
    fresh_premium_id = (premium_attacker_cache.get('artifact_id')
                        if premium_attacker_cache else None)
    if (state.get('xp_artifact_id') == fresh_xp_id
            and state.get('premium_artifact_id') == fresh_premium_id):
        return False          # nothing changed; the guard would pass anyway

    # Live conversion factors, in case a fold left its own in the globals.
    if 'v26_set_conversion' in g and 'final_historical_points_data' in g:
        v26_set_conversion(g['final_historical_points_data'], label='live')

    rebuilt = pd.concat(
        [
            g['live_expected_points_inputs'].reset_index(drop=True),
            g['live_xp_components'].reset_index(drop=True),
        ],
        axis=1,
    )
    premium_payload = active_premium_attacker_payload(premium_attacker_cache)
    rebuilt = apply_premium_attacker_adjustment(
        rebuilt, premium_payload,
        position_col='model_position', price_col='price',
    )
    rebuilt = apply_sparse_player_market_prior(rebuilt)
    xp_payload = active_xp_calibration_payload(xp_calibration_cache)
    rebuilt = apply_xp_calibration_payload(
        rebuilt, xp_payload,
        position_col='model_position', price_col='price',
        value_col='expected_points',
    )

    g['live_xp_payload'] = xp_payload
    g['live_premium_payload'] = premium_payload
    g['live_expected_points'] = rebuilt
    state['xp_artifact_id'] = fresh_xp_id
    state['premium_artifact_id'] = fresh_premium_id
    state['xp_applied'] = xp_payload is not None
    state['premium_applied'] = premium_payload is not None
    print('[v2.6] live forecast rebuilt against the refitted artifacts '
          '- no kernel restart needed')
    return True


_v26_refresh_live_after_validation()

premium_attacker_scorecard = pd.DataFrame(premium_fold_metrics)
calibration_scorecard = pd.DataFrame(calibration_fold_metrics)

print("Premium-attacker component-correction candidate")
display(premium_attacker_scorecard)
print("Production decision:", premium_production_decision)
print("Saved premium-attacker artifact:", PREMIUM_ATTACKER_CACHE_PATH)

print("Out-of-sample xP calibration candidate")
display(calibration_scorecard)
print("Production decision:", xp_production_decision)
print("Saved xP validation artifact:", XP_CALIBRATION_CACHE_PATH)

if not xp_production_decision["approved"]:
    print(
        "xP calibration was rejected for production. Raw component xP (after "
        "any validated premium/sparse corrections) will be used."
    )
if not premium_production_decision["approved"]:
    print(
        "Premium-attacker rate correction was rejected for production. The "
        "base xG/xA models will be used unchanged."
    )


### v2.4 targeted correction validation

This stage uses only the already-built historical OOS fold forecasts. It does
**not** replay FPL seasons or optimise transfers. The first OOS season seeds the
meta-layer; 2024/25 and 2025/26 are chronological validation folds. Production
activation is automatic only when the targeted metrics improve without harming
overall top-player selection.

In [ ]:
# =============================================================================
# v2.4 targeted forecast candidates — cache-first in v2.5
# =============================================================================

if (
    isinstance(v24_targeted_forecast_cache, dict)
    and not FORCE_REBUILD_VALIDATION_CACHES
):
    v24_targeted_forecast_artifact = v24_targeted_forecast_cache
    _v24_prod = v24_targeted_forecast_artifact.get("production_decision", {})
    _v24_upper_decision = dict(_v24_prod.get("upper_tail", {}))
    _v24_mid_decision = dict(_v24_prod.get("midfield", {}))
    _combined_safe = bool(_v24_prod.get("combined_safe", False))
    _combined_ndcg_change = _v24_prod.get("combined_ndcg_change", np.nan)
    _combined_top10_change = _v24_prod.get("combined_top10_change", np.nan)
    _v24_live_bundle = dict(
        v24_targeted_forecast_artifact.get("live_bundle", {})
    )
    _v24_uncertainty = v24_targeted_forecast_artifact.get("uncertainty")
    _v24_upper_metrics = list(
        v24_targeted_forecast_artifact.get("upper_tail_fold_metrics", [])
    )
    _v24_mid_metrics = list(
        v24_targeted_forecast_artifact.get("midfield_fold_metrics", [])
    )
    _v24_combined_metrics = list(
        v24_targeted_forecast_artifact.get("combined_fold_metrics", [])
    )
    _v24_approved = bool(_v24_prod.get("approved", False))

    live_expected_points = apply_v24_targeted_forecast_layer(
        live_expected_points,
        v24_targeted_forecast_artifact,
        position_col="model_position",
        price_col="price",
        value_col="expected_points",
    )
    live_expected_points = apply_v24_uncertainty_payload(
        live_expected_points,
        v24_targeted_forecast_artifact,
        position_col="model_position",
    )
    LIVE_VALIDATION_ARTIFACT_STATE["v24_artifact_id"] = (
        v24_targeted_forecast_artifact.get("artifact_id")
    )
    LIVE_VALIDATION_ARTIFACT_STATE["v24_applied"] = bool(
        live_expected_points.get(
            "v24_targeted_layer_applied",
            pd.Series(False, index=live_expected_points.index),
        ).any()
    )

    v24_upper_tail_scorecard = pd.DataFrame(_v24_upper_metrics)
    v24_midfield_scorecard = pd.DataFrame(_v24_mid_metrics)
    v24_combined_scorecard = pd.DataFrame(_v24_combined_metrics)

    print(
        "v2.4 targeted forecast validation: loaded reusable signed artifact."
    )
else:

    # =============================================================================
    # v2.4 targeted forecast candidates — chronological OOS validation only
    # NO transfer-policy simulation; NO 2026/27 tuning.
    # =============================================================================

    from scipy.stats import spearmanr as _v24_spearmanr


    def _v24_base_fold_frame(season: str) -> pd.DataFrame:
        bundle = walk_forward_fold_cache.get(str(season))
        if bundle is None:
            if V24_REUSE_WALK_FORWARD_FOLDS_ONLY:
                return pd.DataFrame()
            bundle = build_walk_forward_fold_bundle(str(season))

        frame = bundle["holdout_expected_points"].copy()
        premium_payload = None
        if isinstance(premium_attacker_cache, dict):
            premium_payload = premium_attacker_cache.get("fold_strategy_payloads", {}).get(str(season))
        frame = apply_premium_attacker_adjustment(
            frame, premium_payload,
            position_col="model_position", price_col="price_at_collection",
        )
        frame = apply_sparse_player_market_prior(
            frame,
            prior_table=build_position_price_prior_table(bundle["development_points_data"]),
        )
        calibration_payload = None
        if isinstance(xp_calibration_cache, dict):
            calibration_payload = xp_calibration_cache.get("fold_strategy_payloads", {}).get(str(season))
        frame = apply_xp_calibration_payload(
            frame, calibration_payload,
            position_col="model_position", price_col="price_at_collection",
            value_col="expected_points",
        )
        frame["holdout_season"] = str(season)
        return frame


    def _v24_fit_upper_tail_payload(rows: pd.DataFrame) -> dict | None:
        if rows is None or rows.empty:
            return None
        data = rows.copy()
        valid = (
            data["target_fpl_points"].notna()
            & _v24_numeric(data, "expected_points").notna()
            & _v24_numeric(data, "expected_minutes", 0.0).ge(float(V24_UPPER_TAIL_MIN_EXPECTED_MINUTES))
        )
        data = data.loc[valid].copy()
        if len(data) < 500:
            return None
        X = _v24_upper_tail_design(
            data, price_col="price_at_collection", value_col="expected_points"
        )
        active = np.max(X, axis=1) > 0
        if active.sum() < 300:
            return None
        X = X[active]
        raw = _v24_numeric(data.loc[active], "expected_points").to_numpy(dtype=float)
        actual = _v24_numeric(data.loc[active], "target_fpl_points").to_numpy(dtype=float)
        residual = np.clip(actual - raw, -8.0, 8.0)
        alpha = float(V24_UPPER_TAIL_RIDGE_ALPHA)
        coef = np.linalg.solve(X.T @ X + alpha * np.eye(X.shape[1]), X.T @ residual)
        coef = np.clip(coef, -0.25, float(V24_UPPER_TAIL_MAX_ABS_ADJUSTMENT))
        return {
            "rows": int(active.sum()),
            "coefficients": [float(v) for v in coef],
            "feature_names": ["xp_tail_weight", "premium_price_weight"],
        }


    def _v24_fit_midfield_payload(rows: pd.DataFrame) -> dict | None:
        if rows is None or rows.empty:
            return None
        data = rows.copy()
        price = _v24_price(data, "price_at_collection")
        minutes = _v24_numeric(data, "expected_minutes", 0.0)
        mask = (
            data["model_position"].astype(str).eq("MID")
            & price.ge(float(V24_MID_RANK_MIN_PRICE))
            & minutes.ge(float(V24_MID_RANK_MIN_EXPECTED_MINUTES))
            & data["target_fpl_points"].notna()
        )
        data = data.loc[mask].copy()
        if len(data) < 500:
            return None

        features = _v24_mid_feature_frame(data, price_col="price_at_collection")
        feature_names = [name for name in V24_MID_RANK_FEATURES if name in features.columns]
        X = features[feature_names].copy()
        fills, means, scales = {}, {}, {}
        for name in feature_names:
            col = pd.to_numeric(X[name], errors="coerce")
            fill = float(col.median()) if col.notna().any() else 0.0
            col = col.fillna(fill)
            mean = float(col.mean())
            scale = float(col.std(ddof=0))
            if not np.isfinite(scale) or scale < 1e-8:
                scale = 1.0
            X[name] = (col - mean) / scale
            fills[name], means[name], scales[name] = fill, mean, scale

        raw = _v24_numeric(data, "expected_points").to_numpy(dtype=float)
        actual = _v24_numeric(data, "target_fpl_points").to_numpy(dtype=float)
        residual = np.clip(actual - raw, -8.0, 8.0)
        intercept = float(np.mean(residual))
        y = residual - intercept
        Xm = X.to_numpy(dtype=float)
        alpha = float(V24_MID_RANK_RIDGE_ALPHA)
        coef = np.linalg.solve(Xm.T @ Xm + alpha * np.eye(Xm.shape[1]), Xm.T @ y)
        return {
            "rows": int(len(data)),
            "features": feature_names,
            "fills": {k: float(v) for k,v in fills.items()},
            "means": {k: float(v) for k,v in means.items()},
            "scales": {k: float(v) for k,v in scales.items()},
            "intercept": intercept,
            "coefficients": [float(v) for v in coef],
        }


    def _v24_player_gw(frame: pd.DataFrame, prediction_columns: list[str]) -> pd.DataFrame:
        if frame.empty:
            return pd.DataFrame()
        id_col = "player_key" if "player_key" in frame.columns else "player_id_season"
        keys = ["holdout_season", "gameweek", id_col]
        named = {
            "position": ("model_position", "last"),
            "price": ("price_at_collection", "last"),
            "expected_minutes": ("expected_minutes", "sum"),
            "actual_points": ("target_fpl_points", "sum"),
        }
        for col in prediction_columns:
            named[col] = (col, "sum")
        return frame.groupby(keys, dropna=False).agg(**named).reset_index()


    def _v24_ndcg(group: pd.DataFrame, pred_col: str, k: int = 10) -> float:
        valid = group[[pred_col, "actual_points"]].dropna()
        if valid.empty:
            return np.nan
        k = min(int(k), len(valid))
        ordered = valid.nlargest(k, pred_col)
        oracle = valid.nlargest(k, "actual_points")
        discount = np.log2(np.arange(2, k + 2))
        dcg = float((ordered["actual_points"].clip(lower=0).to_numpy() / discount).sum())
        idcg = float((oracle["actual_points"].clip(lower=0).to_numpy() / discount).sum())
        return dcg / idcg if idcg > 0 else np.nan


    def _v24_rank_summary(player_gw: pd.DataFrame, pred_col: str, *, position: str | None = None, k: int = 10) -> dict:
        data = player_gw.copy()
        data = data.loc[pd.to_numeric(data["expected_minutes"], errors="coerce").ge(45.0)]
        if position is not None:
            data = data.loc[data["position"].eq(position)]
        spears, ndcgs, top_sums = [], [], []
        for _, group in data.groupby(["holdout_season", "gameweek"]):
            if len(group) < 3:
                continue
            rho = _v24_spearmanr(group[pred_col], group["actual_points"], nan_policy="omit").statistic
            if np.isfinite(rho):
                spears.append(float(rho))
            nd = _v24_ndcg(group, pred_col, k=min(k, len(group)))
            if np.isfinite(nd):
                ndcgs.append(float(nd))
            top_sums.append(float(group.nlargest(min(k, len(group)), pred_col)["actual_points"].sum()))
        return {
            "spearman": float(np.mean(spears)) if spears else np.nan,
            "ndcg": float(np.mean(ndcgs)) if ndcgs else np.nan,
            "top_actual": float(np.mean(top_sums)) if top_sums else np.nan,
        }


    def _v24_eval_candidate(base: pd.DataFrame, candidate: pd.DataFrame, *, focus: str) -> dict:
        work = candidate.copy()
        work["v24_raw_eval"] = pd.to_numeric(base["expected_points"], errors="coerce").to_numpy()
        work["v24_candidate_eval"] = pd.to_numeric(candidate["expected_points"], errors="coerce").to_numpy()
        pgw = _v24_player_gw(work, ["v24_raw_eval", "v24_candidate_eval"])

        overall_raw = _v24_rank_summary(pgw, "v24_raw_eval", k=10)
        overall_cand = _v24_rank_summary(pgw, "v24_candidate_eval", k=10)

        if focus == "upper":
            price = pd.to_numeric(pgw["price"], errors="coerce")
            mask = pgw["expected_minutes"].ge(float(V24_UPPER_TAIL_MIN_EXPECTED_MINUTES)) & (
                pgw["v24_raw_eval"].ge(3.0) | price.ge(8.0)
            )
            focused = pgw.loc[mask].copy()
            raw = focused["v24_raw_eval"]
            cand = focused["v24_candidate_eval"]
            actual = focused["actual_points"]
            return {
                "rows": int(len(focused)),
                "raw_mae": float(np.mean(np.abs(raw - actual))) if len(focused) else np.nan,
                "candidate_mae": float(np.mean(np.abs(cand - actual))) if len(focused) else np.nan,
                "raw_abs_bias": float(abs((raw - actual).mean())) if len(focused) else np.nan,
                "candidate_abs_bias": float(abs((cand - actual).mean())) if len(focused) else np.nan,
                "overall_raw_ndcg": overall_raw["ndcg"],
                "overall_candidate_ndcg": overall_cand["ndcg"],
                "overall_raw_top10": overall_raw["top_actual"],
                "overall_candidate_top10": overall_cand["top_actual"],
            }

        if focus == "mid":
            price = pd.to_numeric(pgw["price"], errors="coerce")
            focused = pgw.loc[
                pgw["position"].eq("MID")
                & pgw["expected_minutes"].ge(float(V24_MID_RANK_MIN_EXPECTED_MINUTES))
                & price.ge(float(V24_MID_RANK_MIN_PRICE))
            ].copy()
            raw_rank = _v24_rank_summary(focused, "v24_raw_eval", position=None, k=5)
            cand_rank = _v24_rank_summary(focused, "v24_candidate_eval", position=None, k=5)
            actual = focused["actual_points"]
            raw = focused["v24_raw_eval"]
            cand = focused["v24_candidate_eval"]
            return {
                "rows": int(len(focused)),
                "raw_mae": float(np.mean(np.abs(raw - actual))) if len(focused) else np.nan,
                "candidate_mae": float(np.mean(np.abs(cand - actual))) if len(focused) else np.nan,
                "raw_spearman": raw_rank["spearman"],
                "candidate_spearman": cand_rank["spearman"],
                "raw_ndcg5": raw_rank["ndcg"],
                "candidate_ndcg5": cand_rank["ndcg"],
                "raw_top5": raw_rank["top_actual"],
                "candidate_top5": cand_rank["top_actual"],
                "overall_raw_ndcg": overall_raw["ndcg"],
                "overall_candidate_ndcg": overall_cand["ndcg"],
            }

        return {
            "rows": int(len(pgw)),
            "raw_ndcg": overall_raw["ndcg"],
            "candidate_ndcg": overall_cand["ndcg"],
            "raw_top10": overall_raw["top_actual"],
            "candidate_top10": overall_cand["top_actual"],
        }


    def _v24_mean_metric(rows: list[dict], col: str) -> float:
        vals = [float(r[col]) for r in rows if r.get("candidate_evaluated") and np.isfinite(r.get(col, np.nan))]
        return float(np.mean(vals)) if vals else np.nan


    def _v24_decide_upper(rows: list[dict]) -> dict:
        usable = [r for r in rows if r.get("candidate_evaluated")]
        mae_imp = _v24_mean_metric(usable, "raw_mae") - _v24_mean_metric(usable, "candidate_mae")
        bias_imp = _v24_mean_metric(usable, "raw_abs_bias") - _v24_mean_metric(usable, "candidate_abs_bias")
        ndcg_change = _v24_mean_metric(usable, "overall_candidate_ndcg") - _v24_mean_metric(usable, "overall_raw_ndcg")
        top10_change = _v24_mean_metric(usable, "overall_candidate_top10") - _v24_mean_metric(usable, "overall_raw_top10")
        auto = (
            len(usable) >= int(V24_UPPER_TAIL_MIN_EVALUATED_FOLDS)
            and mae_imp >= float(V24_UPPER_TAIL_MIN_MAE_IMPROVEMENT)
            and bias_imp >= float(V24_UPPER_TAIL_MIN_ABS_BIAS_IMPROVEMENT)
            and ndcg_change >= -float(V24_UPPER_TAIL_MAX_NDCG_DEGRADATION)
            and top10_change >= -float(V24_UPPER_TAIL_MAX_TOP10_DEGRADATION)
        )
        return {
            "auto_approved": bool(auto), "approved": bool(auto), "evaluated_folds": len(usable),
            "mae_improvement": mae_imp, "abs_bias_improvement": bias_imp,
            "overall_ndcg_change": ndcg_change, "overall_top10_change": top10_change,
        }


    def _v24_decide_mid(rows: list[dict]) -> dict:
        usable = [r for r in rows if r.get("candidate_evaluated")]
        spear_change = _v24_mean_metric(usable, "candidate_spearman") - _v24_mean_metric(usable, "raw_spearman")
        ndcg_change = _v24_mean_metric(usable, "candidate_ndcg5") - _v24_mean_metric(usable, "raw_ndcg5")
        mae_change = _v24_mean_metric(usable, "candidate_mae") - _v24_mean_metric(usable, "raw_mae")
        overall_ndcg = _v24_mean_metric(usable, "overall_candidate_ndcg") - _v24_mean_metric(usable, "overall_raw_ndcg")
        auto = (
            len(usable) >= int(V24_MID_RANK_MIN_EVALUATED_FOLDS)
            and spear_change >= float(V24_MID_RANK_MIN_SPEARMAN_IMPROVEMENT)
            and ndcg_change >= float(V24_MID_RANK_MIN_NDCG_IMPROVEMENT)
            and mae_change <= float(V24_MID_RANK_MAX_MAE_DEGRADATION)
            and overall_ndcg >= -float(V24_MID_RANK_MAX_OVERALL_NDCG_DEGRADATION)
        )
        return {
            "auto_approved": bool(auto), "approved": bool(auto), "evaluated_folds": len(usable),
            "mid_spearman_change": spear_change, "mid_ndcg5_change": ndcg_change,
            "mid_mae_change": mae_change, "overall_ndcg_change": overall_ndcg,
        }


    def _v24_fit_uncertainty(rows: pd.DataFrame) -> dict | None:
        if rows is None or rows.empty:
            return None
        data = rows.copy()
        data["_minutes_error"] = (
            _v24_numeric(data, "target_minutes", 0.0) - _v24_numeric(data, "expected_minutes", 0.0)
        ).abs()
        data["_xp_error"] = (
            _v24_numeric(data, "target_fpl_points") - _v24_numeric(data, "expected_points")
        ).abs()
        data["_bucket"] = _v24_minutes_bucket(_v24_numeric(data, "expected_minutes", 0.0))
        data = data.loc[data["target_fpl_points"].notna() & data["target_minutes"].notna()].copy()
        if data.empty:
            return None

        def spec(group):
            return {
                "rows": int(len(group)),
                "minutes_uncertainty_mae": float(group["_minutes_error"].mean()),
                "minutes_uncertainty_p80": float(group["_minutes_error"].quantile(0.80)),
                "forecast_uncertainty_mae": float(group["_xp_error"].mean()),
                "forecast_uncertainty_p80": float(group["_xp_error"].quantile(0.80)),
            }

        groups, global_groups = {}, {}
        for (position, bucket), group in data.groupby(["model_position", "_bucket"], dropna=False):
            if len(group) >= int(V24_UNCERTAINTY_MIN_ROWS):
                groups[f"{position}|{bucket}"] = spec(group)
        for bucket, group in data.groupby("_bucket", dropna=False):
            if len(group) >= int(V24_UNCERTAINTY_MIN_ROWS):
                global_groups[str(bucket)] = spec(group)
        return {"groups": groups, "global_groups": global_groups, "fallback": spec(data)}


    _v24_base_frames = []
    _v24_upper_metrics = []
    _v24_mid_metrics = []
    _v24_combined_metrics = []
    _v24_prior_rows = []
    _v24_fold_candidate_bundles = {}

    for _season in WALK_FORWARD_HOLDOUT_SEASONS:
        _base = _v24_base_fold_frame(str(_season))
        if _base.empty:
            continue
        _v24_base_frames.append(_base)
        _prior = pd.concat(_v24_prior_rows, ignore_index=True) if _v24_prior_rows else pd.DataFrame()
        _upper_payload = _v24_fit_upper_tail_payload(_prior)
        _mid_payload = _v24_fit_midfield_payload(_prior)
        _v24_fold_candidate_bundles[str(_season)] = {
            "upper_tail": _upper_payload,
            "midfield": _mid_payload,
        }

        if not _prior.empty:
            _upper = apply_v24_targeted_forecast_layer(
                _base, {"upper_tail": _upper_payload},
                position_col="model_position", price_col="price_at_collection",
            )
            _mid = apply_v24_targeted_forecast_layer(
                _base, {"midfield": _mid_payload},
                position_col="model_position", price_col="price_at_collection",
            )
            _combined = apply_v24_targeted_forecast_layer(
                _base, {"upper_tail": _upper_payload, "midfield": _mid_payload},
                position_col="model_position", price_col="price_at_collection",
            )
            _um = _v24_eval_candidate(_base, _upper, focus="upper")
            _um.update({"holdout_season": str(_season), "candidate_evaluated": _upper_payload is not None})
            _v24_upper_metrics.append(_um)
            _mm = _v24_eval_candidate(_base, _mid, focus="mid")
            _mm.update({"holdout_season": str(_season), "candidate_evaluated": _mid_payload is not None})
            _v24_mid_metrics.append(_mm)
            _cm = _v24_eval_candidate(_base, _combined, focus="combined")
            _cm.update({"holdout_season": str(_season), "candidate_evaluated": (_upper_payload is not None or _mid_payload is not None)})
            _v24_combined_metrics.append(_cm)
        _v24_prior_rows.append(_base)

    _v24_all_oos = pd.concat(_v24_base_frames, ignore_index=True) if _v24_base_frames else pd.DataFrame()
    _v24_live_upper = _v24_fit_upper_tail_payload(_v24_all_oos)
    _v24_live_mid = _v24_fit_midfield_payload(_v24_all_oos)
    _v24_upper_decision = _v24_decide_upper(_v24_upper_metrics)
    _v24_mid_decision = _v24_decide_mid(_v24_mid_metrics)

    _v24_live_bundle = {}
    if _v24_upper_decision["approved"] and _v24_live_upper:
        _v24_live_bundle["upper_tail"] = _v24_live_upper
    if _v24_mid_decision["approved"] and _v24_live_mid:
        _v24_live_bundle["midfield"] = _v24_live_mid

    # Combined safety is evaluated on the two chronological meta-validation folds.
    _combined_usable = [r for r in _v24_combined_metrics if r.get("candidate_evaluated")]
    _combined_ndcg_change = (
        _v24_mean_metric(_combined_usable, "candidate_ndcg") - _v24_mean_metric(_combined_usable, "raw_ndcg")
        if _combined_usable else np.nan
    )
    _combined_top10_change = (
        _v24_mean_metric(_combined_usable, "candidate_top10") - _v24_mean_metric(_combined_usable, "raw_top10")
        if _combined_usable else np.nan
    )
    _combined_safe = bool(
        not _v24_live_bundle
        or (
            np.isfinite(_combined_ndcg_change)
            and _combined_ndcg_change >= -0.003
            and _combined_top10_change >= -0.25
        )
    )
    if not _combined_safe and len(_v24_live_bundle) > 1:
        # Do not stack two individually promising layers if their combination is not safe.
        upper_score = float(_v24_upper_decision.get("overall_ndcg_change", -np.inf))
        mid_score = float(_v24_mid_decision.get("overall_ndcg_change", -np.inf))
        _v24_live_bundle = (
            {"upper_tail": _v24_live_upper}
            if upper_score >= mid_score and _v24_upper_decision["approved"]
            else ({"midfield": _v24_live_mid} if _v24_mid_decision["approved"] else {})
        )

    _v24_uncertainty = _v24_fit_uncertainty(_v24_all_oos)
    _v24_approved = bool(_v24_live_bundle) and bool(_combined_safe or len(_v24_live_bundle) == 1)
    v24_targeted_forecast_artifact = {
        "version": 1,
        "forecast_signature": FORECAST_CACHE_SIGNATURE,
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "production_decision": {
            "approved": _v24_approved,
            "upper_tail": _v24_upper_decision,
            "midfield": _v24_mid_decision,
            "combined_safe": _combined_safe,
            "combined_ndcg_change": _combined_ndcg_change,
            "combined_top10_change": _combined_top10_change,
            "active_components": sorted(_v24_live_bundle),
        },
        "live_bundle": _v24_live_bundle,
        "uncertainty": _v24_uncertainty,
        "upper_tail_fold_metrics": _v24_upper_metrics,
        "midfield_fold_metrics": _v24_mid_metrics,
        "combined_fold_metrics": _v24_combined_metrics,
        "fold_candidate_bundles": _v24_fold_candidate_bundles,
    }
    v24_targeted_forecast_artifact["artifact_id"] = stable_configuration_signature({
        "forecast_signature": FORECAST_CACHE_SIGNATURE,
        "decision": v24_targeted_forecast_artifact["production_decision"],
        "upper_metrics": _v24_upper_metrics,
        "mid_metrics": _v24_mid_metrics,
        "combined_metrics": _v24_combined_metrics,
    })
    V24_TARGETED_FORECAST_ARTIFACT_PATH.write_text(
        json.dumps(v24_targeted_forecast_artifact, indent=2), encoding="utf-8"
    )
    v24_targeted_forecast_cache = v24_targeted_forecast_artifact

    # Refresh the already-built live table in this same Run-All so Section 12 uses
    # the validated v2.4 layer immediately; application is idempotent.
    live_expected_points = apply_v24_targeted_forecast_layer(
        live_expected_points,
        v24_targeted_forecast_cache,
        position_col="model_position",
        price_col="price",
        value_col="expected_points",
    )
    live_expected_points = apply_v24_uncertainty_payload(
        live_expected_points,
        v24_targeted_forecast_cache,
        position_col="model_position",
    )
    LIVE_VALIDATION_ARTIFACT_STATE["v24_artifact_id"] = v24_targeted_forecast_artifact["artifact_id"]
    LIVE_VALIDATION_ARTIFACT_STATE["v24_applied"] = bool(live_expected_points["v24_targeted_layer_applied"].any())

    v24_upper_tail_scorecard = pd.DataFrame(_v24_upper_metrics)
    v24_midfield_scorecard = pd.DataFrame(_v24_mid_metrics)
    v24_combined_scorecard = pd.DataFrame(_v24_combined_metrics)
    print("v2.4 upper-tail candidate decision:", _v24_upper_decision)
    print("v2.4 midfield reranker decision:", _v24_mid_decision)
    print("v2.4 active components:", sorted(_v24_live_bundle))
    print("v2.4 combined safety:", _combined_safe)
    print("Saved targeted forecast artifact:", V24_TARGETED_FORECAST_ARTIFACT_PATH)


In [ ]:

# =============================================================================
# v2.5 probabilistic minutes validation — cache first, no policy simulation
# =============================================================================

_v25_signature = v25_minutes_layer_signature()
v25_probabilistic_minutes_artifact = load_v25_probabilistic_minutes_artifact()
v25_probabilistic_minutes_scorecard = pd.DataFrame()
v25_probabilistic_minutes_oos = pd.DataFrame()
_v25_cache_source = "artifact"

if (
    v25_probabilistic_minutes_artifact is not None
    and V25_PROBABILISTIC_MINUTES_OOS_CACHE_PATH.exists()
    and not V25_FORCE_REBUILD
):
    try:
        v25_probabilistic_minutes_oos = pd.read_csv(
            V25_PROBABILISTIC_MINUTES_OOS_CACHE_PATH,
            compression="gzip",
        )
    except Exception:
        v25_probabilistic_minutes_oos = pd.DataFrame()

    v25_probabilistic_minutes_scorecard = pd.DataFrame(
        v25_probabilistic_minutes_artifact.get("fold_metrics", [])
    )
    print(
        "v2.5 probabilistic-minutes validation: loaded signed cache",
        V25_PROBABILISTIC_MINUTES_ARTIFACT_PATH,
    )
else:
    _v25_cache_source = "computed"
    _v25_metric_rows = []
    _v25_oos_rows = []

    for _season in WALK_FORWARD_HOLDOUT_SEASONS:
        _season = str(_season)
        _bundle = build_walk_forward_fold_bundle(_season)
        _development_seasons = list(_bundle["development_seasons"])
        _training_rows = historical_model_data.loc[
            historical_model_data["season"].isin(_development_seasons)
        ].copy()

        print(
            "Evaluating v2.5 probabilistic minutes fold",
            _season,
            "using cached/reused base forecast fold...",
        )
        try:
            _role_bundle = fit_v25_probabilistic_minutes_bundle(
                _training_rows
            )
        except Exception as _exc:
            _v25_metric_rows.append({
                "holdout_season": _season,
                "candidate_evaluated": False,
                "reason": str(_exc),
            })
            continue

        _raw_fold = _bundle["holdout_expected_points"].copy()

        # v2.6: build_walk_forward_fold_bundle() short-circuits on a cache hit,
        # so its internal v26_set_conversion() may not have run for this fold.
        # Set the factors from this bundle's own development rows so a cached
        # fold can never be scored with another season's factors.
        v26_set_conversion(
            _bundle["development_points_data"],
            label=f"fold:{_season}",
        )

        _scenario_input = _v25_prepare_fold_for_integration(
            _raw_fold,
            _bundle,
        )
        _candidate_fold = integrate_v25_probabilistic_minutes_xp(
            _scenario_input,
            _role_bundle,
            _bundle["defensive_action_priors"],
            _bundle["rare_event_rates"],
            _bundle["defensive_return_priors"],
            scoring_season=_season,
        )
        _candidate_fold = _v25_apply_existing_post_layers_to_fold(
            _candidate_fold,
            _bundle,
        )

        # v2.6 fix for a pre-existing Section 16 bug. _v25_fold_metrics() calls
        # _v24_player_gw(), which groups by ["holdout_season", "gameweek", id].
        # _v24_base_fold_frame() sets that column on the BASE fold, but nothing
        # sets it on the candidate fold, so the first real v2.5 run dies with
        # KeyError: 'holdout_season'. The audit log records v2.5 as
        # "implemented, pending executed validation", which is why this had
        # never surfaced.
        _candidate_fold["holdout_season"] = str(_season)

        _base_fold = _v24_base_fold_frame(_season)
        if _base_fold.empty:
            # The disk-cached bundle is already in memory now, so this should
            # normally only occur on a malformed/stale base artifact.
            _base_fold = _raw_fold.copy()

        _metrics = _v25_fold_metrics(
            _base_fold,
            _candidate_fold,
            _role_bundle,
        )
        _metrics.update({
            "holdout_season": _season,
            "candidate_evaluated": True,
            "p75_training_rows": int(
                _role_bundle.get("p75_training_rows", 0)
            ),
        })
        _v25_metric_rows.append(_metrics)

        _id_col = (
            "player_key"
            if "player_key" in _base_fold.columns
            else "player_id_season"
        )
        _audit = pd.DataFrame({
            "holdout_season": _season,
            "gameweek": pd.to_numeric(
                _base_fold["gameweek"], errors="coerce"
            ),
            "player_id": _base_fold.get(
                _id_col,
                pd.Series(np.arange(len(_base_fold)), index=_base_fold.index),
            ).astype(str),
            "position": resolve_minutes_model_position(_base_fold),
            "price": pd.to_numeric(
                _base_fold.get(
                    "price_at_collection",
                    pd.Series(np.nan, index=_base_fold.index),
                ),
                errors="coerce",
            ),
            "target_minutes": pd.to_numeric(
                _base_fold["target_minutes"], errors="coerce"
            ),
            "target_started": pd.to_numeric(
                _base_fold["target_started"], errors="coerce"
            ),
            "target_sixty_plus": pd.to_numeric(
                _base_fold["target_60_plus"], errors="coerce"
            ),
            "target_points": pd.to_numeric(
                _base_fold["target_fpl_points"], errors="coerce"
            ),
            "base_appearance_probability": pd.to_numeric(
                _base_fold["appearance_probability"], errors="coerce"
            ),
            "base_start_probability": pd.to_numeric(
                _base_fold["start_probability"], errors="coerce"
            ),
            "base_sixty_plus_probability": pd.to_numeric(
                _base_fold["sixty_plus_probability"], errors="coerce"
            ),
            "base_expected_minutes": pd.to_numeric(
                _base_fold["expected_minutes"], errors="coerce"
            ),
            "candidate_expected_minutes": pd.to_numeric(
                _candidate_fold["expected_minutes"], errors="coerce"
            ),
            "base_expected_points": pd.to_numeric(
                _base_fold["expected_points"], errors="coerce"
            ),
            "candidate_expected_points": pd.to_numeric(
                _candidate_fold["expected_points"], errors="coerce"
            ),
        })
        for _column in [
            "v25_p_no_appearance",
            "v25_p_substitute",
            "v25_p_start_lt60",
            "v25_p_start_60_74",
            "v25_p_start_75_plus",
            "v25_p75_given_60",
            "v25_role_entropy",
            "v25_most_likely_role_probability",
            "v25_most_likely_role_state",
        ]:
            if _column in _candidate_fold.columns:
                _audit[_column] = _candidate_fold[_column].to_numpy()
        _v25_oos_rows.append(_audit)

    v25_probabilistic_minutes_scorecard = pd.DataFrame(
        _v25_metric_rows
    )
    v25_probabilistic_minutes_oos = (
        pd.concat(_v25_oos_rows, ignore_index=True)
        if _v25_oos_rows else pd.DataFrame()
    )
    _v25_decision = _v25_decision_from_scorecard(
        v25_probabilistic_minutes_scorecard
    )

    v25_probabilistic_minutes_artifact = {
        "version": V25_CACHE_SCHEMA_VERSION,
        "layer_signature": _v25_signature,
        "base_forecast_signature": FORECAST_CACHE_SIGNATURE,
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "production_decision": _v25_decision,
        "fold_metrics": (
            v25_probabilistic_minutes_scorecard
            .replace({np.nan: None})
            .to_dict("records")
        ),
        "oos_rows": int(len(v25_probabilistic_minutes_oos)),
    }
    v25_probabilistic_minutes_artifact["artifact_id"] = (
        stable_configuration_signature({
            "layer_signature": _v25_signature,
            "decision": _v25_decision,
            "fold_metrics": v25_probabilistic_minutes_artifact[
                "fold_metrics"
            ],
        })
    )

    V25_PROBABILISTIC_MINUTES_ARTIFACT_PATH.write_text(
        json.dumps(
            v25_probabilistic_minutes_artifact,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )
    if not v25_probabilistic_minutes_oos.empty:
        v25_probabilistic_minutes_oos.to_csv(
            V25_PROBABILISTIC_MINUTES_OOS_CACHE_PATH,
            index=False,
            compression="gzip",
        )

    print(
        "Saved v2.5 probabilistic-minutes validation artifact:",
        V25_PROBABILISTIC_MINUTES_ARTIFACT_PATH,
    )

# Full-history live model: one small classifier, persisted separately.
v25_live_minutes_model = load_v25_live_model_bundle()
if v25_live_minutes_model is None:
    print("Fitting one-time v2.5 live P(75+|60+) model...")
    v25_live_minutes_model = fit_v25_probabilistic_minutes_bundle(
        historical_model_data
    )
    save_v25_live_model_bundle(v25_live_minutes_model)
else:
    print("Loaded cached v2.5 live P(75+|60+) model.")

# Attach role probabilities for diagnostics. Production expected minutes/xP
# only change if the chronological OOS gate approved the layer.
live_expected_points = attach_or_apply_v25_live_minutes(
    live_expected_points,
    v25_probabilistic_minutes_artifact,
    v25_live_minutes_model,
)
live_expected_points = apply_v24_uncertainty_payload(
    live_expected_points,
    v24_targeted_forecast_cache,
    position_col="model_position",
)

# Rebuild only the cheap Gameweek aggregations from the refreshed fixture rows.
# No forecast model, transfer optimiser or season simulation is rerun here.
for _column, _reducer in [
    ("v25_probabilistic_minutes_applied", "max"),
    ("v25_role_entropy", "max"),
    ("v25_most_likely_role_probability", "max"),
    ("v25_minutes_adjustment", "sum"),
    ("v25_probabilistic_xp_adjustment", "sum"),
]:
    if _column in live_expected_points.columns:
        live_gameweek_aggregation[_column] = (_column, _reducer)

live_gameweek_expected_points = (
    live_expected_points
    .groupby(player_gameweek_keys, dropna=False)
    .agg(**live_gameweek_aggregation)
    .reset_index()
)
live_gameweek_expected_points["no_appearance_probability"] = (
    pd.to_numeric(
        live_gameweek_expected_points["no_appearance_probability"],
        errors="coerce",
    )
    .fillna(1.0)
    .clip(0.0, 1.0)
)
live_gameweek_expected_points["appearance_probability"] = (
    1.0
    - live_gameweek_expected_points["no_appearance_probability"]
).clip(0.0, 1.0)

planning_horizon_expected_points = (
    live_gameweek_expected_points
    .groupby(
        [
            "player_id",
            "player_key",
            "web_name",
            "team_short_name",
            "position",
            "price",
        ],
        dropna=False,
    )
    .agg(**planning_horizon_aggregation)
    .reset_index()
)

LIVE_VALIDATION_ARTIFACT_STATE["v25_artifact_id"] = (
    v25_probabilistic_minutes_artifact.get("artifact_id")
    if isinstance(v25_probabilistic_minutes_artifact, dict)
    else None
)
LIVE_VALIDATION_ARTIFACT_STATE["v25_applied"] = bool(
    live_expected_points.get(
        "v25_probabilistic_minutes_applied",
        pd.Series(False, index=live_expected_points.index),
    ).fillna(False).any()
)

print(
    "v2.5 probabilistic-minutes source:",
    _v25_cache_source,
)
print(
    "v2.5 production decision:",
    (
        v25_probabilistic_minutes_artifact.get("production_decision", {})
        if isinstance(v25_probabilistic_minutes_artifact, dict)
        else {}
    ),
)
print(
    "v2.5 applied live:",
    LIVE_VALIDATION_ARTIFACT_STATE["v25_applied"],
)


In [ ]:
walk_forward_prediction_rows = []
for fold_season in WALK_FORWARD_HOLDOUT_SEASONS:
    print(f"Loading/building walk-forward fold {fold_season}...")
    fold_bundle = build_walk_forward_fold_bundle(
        fold_season
    )
    walk_forward_prediction_rows.append(
        walk_forward_prediction_metrics(fold_bundle)
    )

walk_forward_prediction_scorecard = pd.DataFrame(
    walk_forward_prediction_rows
)

display(walk_forward_prediction_scorecard)

walk_forward_prediction_summary = pd.DataFrame([{
    "folds": len(walk_forward_prediction_scorecard),
    "mean_xp_mae": (
        walk_forward_prediction_scorecard["xp_mae"].mean()
    ),
    "mean_baseline_mae": (
        walk_forward_prediction_scorecard["baseline_mae"].mean()
    ),
    "mean_xp_spearman": (
        walk_forward_prediction_scorecard["xp_spearman"].mean()
    ),
    "mean_baseline_spearman": (
        walk_forward_prediction_scorecard[
            "baseline_spearman"
        ].mean()
    ),
}])
display(walk_forward_prediction_summary)


In [ ]:
@contextmanager
def activated_walk_forward_context(
    bundle: dict[str, object],
    policy_profile: dict[str, object],
):
    """Temporarily activate one fold without contaminating other folds."""
    holdout_season = str(bundle["holdout_season"])

    replacement_values = {
        "BACKTEST_SEASON": holdout_season,
        "BACKTEST_HORIZON": int(
            policy_profile["horizon"]
        ),
        "BACKTEST_MAX_GAMEWEEKS": (
            WALK_FORWARD_MAX_GAMEWEEKS
        ),
        "BACKTEST_INCLUDE_CHIPS": (
            WALK_FORWARD_INCLUDE_CHIPS
        ),
        "MAX_TRANSFERS_PER_GAMEWEEK": int(
            policy_profile[
                "max_transfers_per_gameweek"
            ]
        ),
        "ALLOW_TRANSFER_HITS": bool(
            policy_profile["allow_hits"]
        ),
        "BACKTEST_APPLY_TRANSFER_ROBUSTNESS_GATE": bool(
            policy_profile.get("bank_aware", False)
        ),
        "WEEKLY_MIN_OBJECTIVE_GAIN_PER_EXTRA_TRANSFER": float(
            policy_profile.get(
                "min_gain_per_extra_transfer",
                BANK_AWARE_MIN_GAIN_PER_EXTRA_TRANSFER,
            )
        ),
        "MAX_BANKED_FREE_TRANSFERS": (
            2 if season_start_year(holdout_season) < 2024 else 5
        ),
        "minutes_models": bundle["minutes_models"],
        "team_goal_models": bundle["team_goal_models"],
        "player_event_models": (
            bundle["player_event_models"]
        ),
        "development_defensive_action_priors": (
            bundle["defensive_action_priors"]
        ),
        "development_defensive_return_priors": (
            bundle["defensive_return_priors"]
        ),
        "development_rare_event_rates": (
            bundle["rare_event_rates"]
        ),
        "holdout_points_data": (
            bundle["holdout_points_data"]
        ),
    }

    original_values = {
        name: globals()[name]
        for name in replacement_values
    }

    globals().update(replacement_values)
    deadline_frozen_forecast_cache.clear()
    deadline_frozen_forecast_audit.clear()

    try:
        yield
    finally:
        globals().update(original_values)
        deadline_frozen_forecast_cache.clear()
        deadline_frozen_forecast_audit.clear()


def run_walk_forward_strategy_fold(
    bundle: dict[str, object],
    profile_name: str,
) -> dict[str, object]:
    """Run one chipless, strict-deadline strategy fold."""
    profile = WALK_FORWARD_POLICY_PROFILES[
        profile_name
    ]

    with activated_walk_forward_context(
        bundle,
        profile,
    ):
        strategy_forecast = bundle["holdout_expected_points"].copy()
        fold_premium = premium_fold_strategy_payloads.get(
            bundle["holdout_season"]
        )
        strategy_forecast = apply_premium_attacker_adjustment(
            strategy_forecast,
            fold_premium,
            position_col="model_position",
            price_col="price_at_collection",
        )
        fold_price_prior = build_position_price_prior_table(
            bundle["development_points_data"]
        )
        strategy_forecast = apply_sparse_player_market_prior(
            strategy_forecast,
            prior_table=fold_price_prior,
        )
        fold_calibration = calibration_fold_strategy_payloads.get(
            bundle["holdout_season"]
        )
        strategy_forecast = apply_xp_calibration_payload(
            strategy_forecast,
            fold_calibration,
            position_col="model_position",
            price_col="price_at_collection",
            value_col="expected_points",
        )
        backtest_table = build_historical_backtest_table(
            strategy_forecast
        )
        gameweeks = sorted(
            backtest_table["gameweek"].unique()
        )
        if WALK_FORWARD_MAX_GAMEWEEKS is not None:
            gameweeks = gameweeks[
                :int(WALK_FORWARD_MAX_GAMEWEEKS)
            ]

        shared_opening_key = (
            str(bundle["holdout_season"]),
            str(POLICY_CACHE_SIGNATURE),
        )
        if shared_opening_key not in walk_forward_shared_opening_cache:
            walk_forward_shared_opening_cache[shared_opening_key] = (
                build_backtest_opening_state(
                    backtest_table,
                    gameweeks,
                    opening_horizon=int(POLICY_SHARED_OPENING_HORIZON),
                )
            )
        shared_opening_state = walk_forward_shared_opening_cache[
            shared_opening_key
        ]

        result = run_sequential_fpl_backtest(
            backtest_table,
            gameweeks,
            opening_state=shared_opening_state,
        )

        audit = pd.DataFrame(
            deadline_frozen_forecast_audit
        )
        if not audit.empty:
            assert (
                audit["player_history_available_through_gw"]
                .lt(audit["first_projected_gameweek"])
                .all()
            )
            assert (
                audit["team_results_available_through_gw"]
                .lt(audit["first_projected_gameweek"])
                .all()
            )

        weekly = result["weekly_results"]
        static = result["static_benchmark"]

        return {
            "holdout_season": bundle["holdout_season"],
            "profile": profile_name,
            "horizon": int(profile["horizon"]),
            "allow_hits": bool(profile["allow_hits"]),
            "bank_aware": bool(profile.get("bank_aware", False)),
            "min_gain_per_extra_transfer": float(
                profile.get("min_gain_per_extra_transfer", 0.0)
            ),
            "shared_opening_horizon": int(POLICY_SHARED_OPENING_HORIZON),
            "initial_squad_signature": str(
                result["initial_squad_signature"]
            ),
            "static_benchmark_signature": str(
                result["static_benchmark_signature"]
            ),
            "opening_squad_price": float(
                result["initial_squad"]["price"].sum()
            ),
            "gameweeks": len(weekly),
            "strategy_points": float(
                weekly["actual_score"].sum()
            ),
            "static_points": float(
                static["actual_score"].sum()
            ),
            "gain_vs_static": float(
                weekly["actual_score"].sum()
                - static["actual_score"].sum()
            ),
            "hit_points": float(weekly["hit_cost"].sum()),
            "paid_transfers": int(
                round(
                    float(weekly["hit_cost"].sum())
                    / float(TRANSFER_HIT_COST)
                )
            ),
            "transfers": int(
                weekly["transfers"].sum()
            ),
            "prediction_bias": float(
                (
                    weekly["predicted_score"]
                    - weekly["actual_score"]
                ).mean()
            ),
            "weekly_results": weekly,
            "initial_squad": result["initial_squad"],
        }


In [ ]:
walk_forward_strategy_results = []
walk_forward_strategy_details = {}


def policy_baseline_fairness_status(
    scorecard: pd.DataFrame,
) -> tuple[bool, str]:
    """
    Verify that policy comparisons are genuinely like-for-like within season.

    Every profile must start from the same 15-player opening squad and must
    therefore produce the same static benchmark. A mismatch invalidates the
    entire production scorecard rather than merely warning.
    """
    if scorecard is None or scorecard.empty:
        return False, "no scorecard available for baseline-fairness audit"

    required = {
        "holdout_season",
        "profile",
        "initial_squad_signature",
        "static_benchmark_signature",
        "static_points",
        "shared_opening_horizon",
    }
    missing = sorted(required - set(scorecard.columns))
    if missing:
        return False, (
            "scorecard missing baseline-fairness columns: "
            f"{missing}"
        )

    issues = []
    for season, group in scorecard.groupby("holdout_season", sort=False):
        initial_signatures = (
            group["initial_squad_signature"]
            .dropna()
            .astype(str)
            .unique()
        )
        static_signatures = (
            group["static_benchmark_signature"]
            .dropna()
            .astype(str)
            .unique()
        )
        static_values = pd.to_numeric(
            group["static_points"],
            errors="coerce",
        ).dropna()
        opening_horizons = pd.to_numeric(
            group["shared_opening_horizon"],
            errors="coerce",
        ).dropna().astype(int).unique()

        if len(initial_signatures) != 1:
            issues.append(
                f"{season}: {len(initial_signatures)} opening-squad signatures"
            )
        if len(static_signatures) != 1:
            issues.append(
                f"{season}: {len(static_signatures)} static-benchmark signatures"
            )
        if (
            len(static_values) != len(group)
            or (
                not static_values.empty
                and float(static_values.max() - static_values.min()) > 1e-9
            )
        ):
            issues.append(
                f"{season}: static_points differ across policy profiles"
            )
        if (
            len(opening_horizons) != 1
            or int(opening_horizons[0]) != int(POLICY_SHARED_OPENING_HORIZON)
        ):
            issues.append(
                f"{season}: shared opening horizon is not consistently "
                f"{int(POLICY_SHARED_OPENING_HORIZON)}"
            )

    if issues:
        return False, "; ".join(issues)
    return True, "shared opening squad and static benchmark confirmed within every season"


def production_policy_scorecard_status(
    scorecard: pd.DataFrame,
) -> tuple[bool, str]:
    """Validate that a policy scorecard is complete enough for production provenance."""
    if scorecard is None or scorecard.empty:
        return False, "no valid scorecard loaded"

    required_columns = {
        "holdout_season",
        "profile",
        "strategy_points",
        "cache_signature",
        "transfer_search_signature",
        "validation_scope",
        "initial_squad_signature",
        "static_benchmark_signature",
        "shared_opening_horizon",
    }
    missing = sorted(required_columns - set(scorecard.columns))
    if missing:
        return False, f"scorecard missing required columns: {missing}"

    fairness_ok, fairness_reason = policy_baseline_fairness_status(scorecard)
    if not fairness_ok:
        return False, f"policy baseline fairness failed: {fairness_reason}"

    signature_ok = scorecard["cache_signature"].astype(str).eq(
        str(POLICY_CACHE_SIGNATURE)
    ).all()
    if not signature_ok:
        return False, "scorecard signature does not match the current model/config"

    search_signature_ok = scorecard["transfer_search_signature"].astype(str).eq(
        str(TRANSFER_SEARCH_SIGNATURE)
    ).all()
    if not search_signature_ok:
        return False, "scorecard transfer-search signature does not match v2.2.7 production search"

    scope_ok = scorecard["validation_scope"].astype(str).eq("full_season").all()
    if not scope_ok:
        return False, "scorecard is not a full-season production validation"

    seasons = set(scorecard["holdout_season"].astype(str))
    missing_seasons = sorted(
        set(map(str, WALK_FORWARD_HOLDOUT_SEASONS)) - seasons
    )
    if missing_seasons:
        return False, f"scorecard missing holdout seasons: {missing_seasons}"

    profiles = set(scorecard["profile"].astype(str))
    missing_profiles = sorted(
        set(map(str, WALK_FORWARD_ACTIVE_PROFILES)) - profiles
    )
    if missing_profiles:
        return False, f"scorecard missing active profiles: {missing_profiles}"

    expected_pairs = {
        (str(season), str(profile))
        for season in WALK_FORWARD_HOLDOUT_SEASONS
        for profile in WALK_FORWARD_ACTIVE_PROFILES
    }
    actual_pairs = set(
        zip(
            scorecard["holdout_season"].astype(str),
            scorecard["profile"].astype(str),
        )
    )
    missing_pairs = sorted(expected_pairs - actual_pairs)
    if missing_pairs:
        return False, (
            "scorecard is incomplete; missing season/profile pairs: "
            f"{missing_pairs[:8]}"
        )

    if WALK_FORWARD_MAX_GAMEWEEKS is not None:
        return False, (
            "current kernel is in walk-forward smoke-test mode "
            f"(WALK_FORWARD_MAX_GAMEWEEKS={WALK_FORWARD_MAX_GAMEWEEKS})"
        )

    return True, "complete signed full-season walk-forward scorecard"


if RUN_FULL_POLICY_BACKTEST:
    print(
        "Running/resuming the full walk-forward policy backtest. "
        "Completed season/profile folds are checkpointed after every run."
    )

    checkpoint_rows = []
    if (
        POLICY_BACKTEST_RESUME
        and POLICY_BACKTEST_CHECKPOINT_PATH.exists()
        and not FORCE_REBUILD_VALIDATION_CACHES
    ):
        try:
            checkpoint = pd.read_csv(
                POLICY_BACKTEST_CHECKPOINT_PATH,
                compression="gzip",
            )
            checkpoint_ok = bool(
                not checkpoint.empty
                and "cache_signature" in checkpoint.columns
                and checkpoint["cache_signature"].astype(str).eq(
                    str(POLICY_CACHE_SIGNATURE)
                ).all()
                and "transfer_search_signature" in checkpoint.columns
                and checkpoint["transfer_search_signature"].astype(str).eq(
                    str(TRANSFER_SEARCH_SIGNATURE)
                ).all()
            )
            if checkpoint_ok:
                checkpoint_rows = checkpoint.to_dict("records")
                print(
                    "Resuming policy validation from checkpoint:",
                    POLICY_BACKTEST_CHECKPOINT_PATH,
                    f"({len(checkpoint_rows)} completed folds)",
                )
            else:
                print(
                    "Ignoring stale/incompatible policy checkpoint and starting "
                    "the current validation signature from scratch."
                )
        except Exception as exc:
            warnings.warn(
                f"Could not load the policy-validation checkpoint: {exc}. "
                "Starting from scratch."
            )

    completed_pairs = {
        (str(row["holdout_season"]), str(row["profile"]))
        for row in checkpoint_rows
        if row.get("holdout_season") is not None
        and row.get("profile") is not None
    }
    walk_forward_strategy_results = list(checkpoint_rows)

    expected_pairs = [
        (str(season), str(profile))
        for season in WALK_FORWARD_HOLDOUT_SEASONS
        for profile in WALK_FORWARD_ACTIVE_PROFILES
    ]

    for fold_season in WALK_FORWARD_HOLDOUT_SEASONS:
        bundle = None
        for profile_name in WALK_FORWARD_ACTIVE_PROFILES:
            pair = (str(fold_season), str(profile_name))
            if pair in completed_pairs:
                print(
                    f"Skipping completed policy fold {fold_season} / "
                    f"{profile_name}."
                )
                continue

            if bundle is None:
                bundle = build_walk_forward_fold_bundle(fold_season)

            print(
                f"Running {fold_season} strategy fold "
                f"with profile '{profile_name}'..."
            )
            fold_result = run_walk_forward_strategy_fold(
                bundle,
                profile_name,
            )
            walk_forward_strategy_details[(fold_season, profile_name)] = fold_result

            row = {
                key: value
                for key, value in fold_result.items()
                if key not in {"weekly_results", "initial_squad"}
            }
            row["cache_signature"] = POLICY_CACHE_SIGNATURE
            row["transfer_search_signature"] = TRANSFER_SEARCH_SIGNATURE
            row["project_version"] = PROJECT_VERSION
            row["validation_scope"] = (
                "full_season"
                if WALK_FORWARD_MAX_GAMEWEEKS is None
                else f"smoke_{int(WALK_FORWARD_MAX_GAMEWEEKS)}gw"
            )
            walk_forward_strategy_results.append(row)
            completed_pairs.add(pair)

            # Fail fast if two profiles in the same holdout season ever stop
            # being like-for-like. This catches baseline contamination after
            # the second affected fold rather than after an expensive full run.
            _season_rows = pd.DataFrame(
                walk_forward_strategy_results
            )
            _season_rows = _season_rows.loc[
                _season_rows["holdout_season"].astype(str).eq(
                    str(fold_season)
                )
            ]
            if len(_season_rows) >= 2:
                _fair_now, _fair_reason = policy_baseline_fairness_status(
                    _season_rows
                )
                if not _fair_now:
                    raise RuntimeError(
                        "Section 11 baseline fairness failed during validation: "
                        f"{_fair_reason}"
                    )

            checkpoint = (
                pd.DataFrame(walk_forward_strategy_results)
                .drop_duplicates(
                    ["holdout_season", "profile"],
                    keep="last",
                )
                .reset_index(drop=True)
            )
            POLICY_BACKTEST_CHECKPOINT_PATH.parent.mkdir(
                parents=True,
                exist_ok=True,
            )
            checkpoint.to_csv(
                POLICY_BACKTEST_CHECKPOINT_PATH,
                index=False,
                compression="gzip",
            )
            print(
                "Checkpointed policy fold:",
                pair,
                f"({len(checkpoint)}/{len(expected_pairs)} complete)",
            )

    walk_forward_strategy_scorecard = (
        pd.DataFrame(walk_forward_strategy_results)
        .drop_duplicates(
            ["holdout_season", "profile"],
            keep="last",
        )
        .sort_values(
            ["holdout_season", "strategy_points"],
            ascending=[True, False],
        )
        .reset_index(drop=True)
    )

    # Re-materialise provenance fields in case resumed checkpoint rows came
    # from an older compatible partial write.
    walk_forward_strategy_scorecard["cache_signature"] = POLICY_CACHE_SIGNATURE
    walk_forward_strategy_scorecard["transfer_search_signature"] = (
        TRANSFER_SEARCH_SIGNATURE
    )
    walk_forward_strategy_scorecard["project_version"] = PROJECT_VERSION
    walk_forward_strategy_scorecard["validation_scope"] = (
        "full_season"
        if WALK_FORWARD_MAX_GAMEWEEKS is None
        else f"smoke_{int(WALK_FORWARD_MAX_GAMEWEEKS)}gw"
    )

    production_valid, production_reason = production_policy_scorecard_status(
        walk_forward_strategy_scorecard
    )

    if WALK_FORWARD_MAX_GAMEWEEKS is None:
        if not production_valid:
            raise RuntimeError(
                "The full policy backtest finished but did not satisfy the "
                f"production scorecard contract: {production_reason}"
            )

        WALK_FORWARD_POLICY_SCORECARD_CACHE.parent.mkdir(
            parents=True,
            exist_ok=True,
        )
        walk_forward_strategy_scorecard.to_csv(
            WALK_FORWARD_POLICY_SCORECARD_CACHE,
            index=False,
            compression="gzip",
        )
        print(
            "Saved validated production policy scorecard to:",
            WALK_FORWARD_POLICY_SCORECARD_CACHE,
        )

        if (
            POLICY_BACKTEST_DELETE_CHECKPOINT_ON_SUCCESS
            and POLICY_BACKTEST_CHECKPOINT_PATH.exists()
        ):
            POLICY_BACKTEST_CHECKPOINT_PATH.unlink()
    else:
        print(
            "Walk-forward smoke-test result NOT saved as the production "
            "policy scorecard."
        )

elif (
    WALK_FORWARD_POLICY_SCORECARD_CACHE.exists()
    and not FORCE_REBUILD_VALIDATION_CACHES
):
    candidate_scorecard = pd.read_csv(
        WALK_FORWARD_POLICY_SCORECARD_CACHE,
        compression="gzip",
    )
    candidate_valid, candidate_reason = production_policy_scorecard_status(
        candidate_scorecard
    )
    if STRICT_CACHE_SIGNATURES and not candidate_valid:
        walk_forward_strategy_scorecard = pd.DataFrame()
        print(
            "Ignoring stale/incomplete policy scorecard:",
            candidate_reason,
            "Set RUN_FULL_POLICY_BACKTEST=True in the first cell.",
        )
    else:
        walk_forward_strategy_scorecard = candidate_scorecard
        print(
            "Loaded policy scorecard candidate:",
            WALK_FORWARD_POLICY_SCORECARD_CACHE,
        )
else:
    walk_forward_strategy_scorecard = pd.DataFrame()
    print(
        "No valid policy scorecard is available. Set RUN_FULL_POLICY_BACKTEST=True "
        "and run Section 11 once. Until then the production fallback is: "
        f"{POLICY_RECOMMENDATION_FALLBACK}."
    )

(
    walk_forward_policy_validation_available,
    walk_forward_policy_validation_source,
) = production_policy_scorecard_status(
    walk_forward_strategy_scorecard
)

print(
    "Policy validation provenance:",
    "VALIDATED"
    if walk_forward_policy_validation_available
    else "NOT VALIDATED",
    "—",
    walk_forward_policy_validation_source,
)

if not walk_forward_strategy_scorecard.empty:
    display(walk_forward_strategy_scorecard)

if WALK_FORWARD_MAX_GAMEWEEKS is not None:
    print(
        "Walk-forward smoke-test mode is active. Set WALK_FORWARD_MAX_GAMEWEEKS=None "
        "before generating the production policy scorecard."
    )


### Historical club-transfer edge case

If a real-world Premier League transfer temporarily creates more than three owned players from one club, FPL allows the squad to be held. The next transfer must reduce the excess. The transfer planner models this transitional state rather than rejecting the squad.


### Interpreting the policy comparison

This comparison separates five questions:

1. Does looking beyond the next Gameweek improve decisions?
2. Is a three-Gameweek horizon sufficient?
3. Does extending the horizon from three to six Gameweeks help?
4. Does a one-Gameweek policy improve when it prices the option value of banking an extra free transfer?
5. Do paid transfers add enough realised points to justify their deductions?

The one-Gameweek no-hit profile is the strongest simple benchmark because it
uses the same forecasts, line-up logic and current squad state, but does not
attempt multi-Gameweek planning.

Profile selection should not rely on the highest score in one season. Prefer a
policy that performs consistently across folds, avoids a poor worst season and
does not depend heavily on transfer hits.


v2.2.7 fairness rule: all policy profiles in a holdout season start from the
same independently constructed opening squad and static benchmark. The
scorecard is rejected if those signatures differ within a season.


In [ ]:
recommended_policy_profile = None
recommended_policy_profile_validated = False
recommended_policy_profile_source = str(
    globals().get(
        "walk_forward_policy_validation_source",
        "no validation provenance",
    )
)
diagnostic_best_policy_profile = None

if walk_forward_strategy_scorecard.empty:
    profile_season_pivot = pd.DataFrame()
    profile_summary = pd.DataFrame()
    policy_increment_table = pd.DataFrame()

    print(
        "Policy comparison unavailable in this kernel. "
        "No policy is being labelled as validated. "
        "Section 12 will use the configured fallback:",
        POLICY_RECOMMENDATION_FALLBACK,
    )
else:
    profile_season_pivot = (
        walk_forward_strategy_scorecard
        .pivot(
            index="holdout_season",
            columns="profile",
            values="strategy_points",
        )
    )

    print("Realised points by season and policy")
    display(profile_season_pivot)

    policy_baseline_fairness_audit = (
        walk_forward_strategy_scorecard
        .groupby("holdout_season", as_index=False)
        .agg(
            profiles=("profile", "nunique"),
            opening_squad_signatures=(
                "initial_squad_signature",
                "nunique",
            ),
            static_benchmark_signatures=(
                "static_benchmark_signature",
                "nunique",
            ),
            static_points_min=("static_points", "min"),
            static_points_max=("static_points", "max"),
            shared_opening_horizons=(
                "shared_opening_horizon",
                "nunique",
            ),
        )
    )
    policy_baseline_fairness_audit["static_points_range"] = (
        policy_baseline_fairness_audit["static_points_max"]
        - policy_baseline_fairness_audit["static_points_min"]
    )
    fairness_ok, fairness_reason = policy_baseline_fairness_status(
        walk_forward_strategy_scorecard
    )
    print(
        "Policy baseline fairness:",
        "PASS" if fairness_ok else "FAIL",
        "—",
        fairness_reason,
    )
    display(policy_baseline_fairness_audit)
    if not fairness_ok:
        raise RuntimeError(
            "Policy comparison is not like-for-like: "
            f"{fairness_reason}"
        )

    profile_summary = (
        walk_forward_strategy_scorecard
        .groupby(
            [
                "profile",
                "horizon",
                "allow_hits",
                "bank_aware",
                "min_gain_per_extra_transfer",
            ],
            as_index=False,
        )
        .agg(
            seasons=("holdout_season", "nunique"),
            mean_points=("strategy_points", "mean"),
            median_points=("strategy_points", "median"),
            minimum_points=("strategy_points", "min"),
            maximum_points=("strategy_points", "max"),
            points_std=("strategy_points", "std"),
            mean_gain_vs_static=("gain_vs_static", "mean"),
            total_hit_points=("hit_points", "sum"),
            total_paid_transfers=("paid_transfers", "sum"),
            mean_transfers=("transfers", "mean"),
            mean_prediction_bias=("prediction_bias", "mean"),
        )
    )

    # Rank each policy within each holdout season, then average the ranks.
    season_rank = (
        walk_forward_strategy_scorecard
        .assign(
            season_rank=lambda frame: (
                frame.groupby("holdout_season")[
                    "strategy_points"
                ]
                .rank(
                    method="average",
                    ascending=False,
                )
            )
        )
        .groupby("profile", as_index=False)
        .agg(
            mean_season_rank=("season_rank", "mean"),
            worst_season_rank=("season_rank", "max"),
        )
    )

    profile_summary = (
        profile_summary
        .merge(
            season_rank,
            on="profile",
            how="left",
            validate="one_to_one",
        )
        .sort_values(
            [
                "mean_season_rank",
                "minimum_points",
                "mean_points",
            ],
            ascending=[True, False, False],
        )
        .reset_index(drop=True)
    )

    print("Cross-season policy summary")
    display(profile_summary)

    greedy_profile = "greedy_1gw_no_hits"
    policy_increment_table = (
        profile_season_pivot
        .subtract(
            profile_season_pivot[greedy_profile],
            axis=0,
        )
        .rename_axis(
            index="holdout_season",
            columns="profile",
        )
    )

    print("Realised gain versus the one-Gameweek no-hit policy")
    display(policy_increment_table)

    if "greedy_1gw_bank_aware" in profile_season_pivot.columns:
        bank_aware_audit = pd.DataFrame({
            "greedy_1gw_no_hits": profile_season_pivot[
                "greedy_1gw_no_hits"
            ],
            "greedy_1gw_bank_aware": profile_season_pivot[
                "greedy_1gw_bank_aware"
            ],
        })
        bank_aware_audit["gain_from_bank_awareness"] = (
            bank_aware_audit["greedy_1gw_bank_aware"]
            - bank_aware_audit["greedy_1gw_no_hits"]
        )
        print("One-Gameweek bank-awareness audit")
        display(bank_aware_audit)

    diagnostic_best_policy_profile = str(
        profile_summary.iloc[0]["profile"]
    )

    if bool(
        globals().get(
            "walk_forward_policy_validation_available",
            False,
        )
    ):
        recommended_policy_profile = diagnostic_best_policy_profile
        recommended_policy_profile_validated = True
        recommended_policy_profile_source = str(
            globals().get(
                "walk_forward_policy_validation_source",
                "validated Section 11 scorecard",
            )
        )
        print(
            "Validated production-policy candidate:",
            recommended_policy_profile,
        )
    else:
        recommended_policy_profile = None
        recommended_policy_profile_validated = False
        print(
            "Diagnostic best policy:",
            diagnostic_best_policy_profile,
        )
        print(
            "This scorecard is not production-valid in the current kernel, "
            "so Section 12 will use the configured fallback and label it "
            "accordingly."
        )

    print(
        "Policy ranking uses mean within-season rank, then worst-season "
        "performance and mean points. Even a validated historical policy "
        "choice is not an unbiased estimate of future performance."
    )

    # Practical hit audit for the paid-transfer profile.
    hit_profile_rows = walk_forward_strategy_scorecard.loc[
        walk_forward_strategy_scorecard["allow_hits"]
    ].copy()

    if not hit_profile_rows.empty:
        hit_profile_name = str(
            hit_profile_rows.iloc[0]["profile"]
        )
        no_hit_comparator = "rolling_6gw_no_hits"

        if (
            no_hit_comparator
            in set(walk_forward_strategy_scorecard["profile"])
        ):
            hit_comparison = (
                walk_forward_strategy_scorecard.loc[
                    walk_forward_strategy_scorecard["profile"].isin(
                        [hit_profile_name, no_hit_comparator]
                    ),
                    [
                        "holdout_season",
                        "profile",
                        "strategy_points",
                        "hit_points",
                        "paid_transfers",
                    ],
                ]
                .pivot(
                    index="holdout_season",
                    columns="profile",
                    values="strategy_points",
                )
                .reset_index()
            )
            hit_comparison["gain_from_allowing_hits"] = (
                hit_comparison[hit_profile_name]
                - hit_comparison[no_hit_comparator]
            )

            hit_costs = (
                hit_profile_rows[
                    [
                        "holdout_season",
                        "hit_points",
                        "paid_transfers",
                    ]
                ]
            )
            hit_comparison = hit_comparison.merge(
                hit_costs,
                on="holdout_season",
                how="left",
                validate="one_to_one",
            )

            print("Six-Gameweek hit-policy audit")
            display(hit_comparison)


### Readiness decision

The system is ready for the final weekly interface when:

- the expected-points model beats its rolling baseline across folds;
- one planning policy performs robustly across seasons;
- longer horizons outperform or at least match the one-Gameweek benchmark;
- paid transfers improve realised points after their deductions, rather than
  merely increasing activity;
- no single season determines the selected policy.

After selecting the production policy, the final notebook stage will turn the
model outputs into one concise weekly recommendation:

- transfers or roll;
- starting XI and formation;
- bench order;
- captain and vice-captain;
- chip recommendation;
- expected score and uncertainty;
- alternative actions and their projected cost.


# 11B. Cache-only hindsight regret and elite benchmarks

This section answers a different question from the forecast-validation and
policy-selection sections:

> **Where did the historical points gap actually come from?**

It is deliberately **diagnostic only**. It does not modify forecasts, the
selected transfer policy, chip rules or the production recommendation.

With both expensive flags set to `False`, the preceding sections load the
signed v2.1 policy and sequential caches. This section then reuses those
results rather than rerunning either expensive backtest.

The diagnostics have three levels:

1. **Same-squad hindsight oracle** — keeps the exact 15-man squad the model
   owned in each Gameweek, but retrospectively chooses the best legal XI and
   captain from realised points. This isolates execution regret due to
   line-up/captain decisions from the squad itself.
2. **One-Gameweek free-rebuild hindsight oracle** — retrospectively builds the
   best legal squad for each individual Gameweek under an approximate budget
   equal to the model squad's current market value plus bank. This is an
   intentionally unattainable ceiling: it ignores transfer continuity and
   therefore measures the scale of the *squad opportunity gap*, not actionable
   transfer regret.
3. **Elite benchmark** — compares cached historical strategy scores with
   winner totals and sparse public rank anchors. Rank estimates are labelled
   approximate and are only interpolated inside the available anchor range.

`Free Hit` weeks cannot be decomposed into same-squad line-up/captain regret
from the existing sequential cache because v2.1 stores the permanent squad
rather than the temporary Free Hit squad. Those weeks are explicitly marked
instead of being guessed.

Triple Captain and Bench Boost timing diagnostics are available from cached
realised points. Wildcard and Free Hit timing require counterfactual squad
re-optimisation and are therefore intentionally not reconstructed in this
cache-only section.


In [ ]:
# Cache-only regret/oracle diagnostics -----------------------------------------

HISTORICAL_WINNER_POINTS = {
    # Official Premier League reports give 2,799 and 2,810 for the first two
    # seasons. The 2025/26 winner total (2,582) is included as a configurable
    # benchmark from the published final standings/history.
    "2023-24": 2799,
    "2024-25": 2810,
    "2025-26": 2582,
}

# Sparse public FPL manager-history anchors. These are NOT exact rank cutoffs.
# They are only used to interpolate an indicative rank when the model score
# falls between observed score/rank pairs. Outside the anchor range, the code
# reports a bound instead of extrapolating.
HISTORICAL_RANK_ANCHORS = {
    "2023-24": [
        (2799, 1),
        (2543, 26813),
        (2494, 86170),
        (2474, 124907),
        (2393, 394214),
        (2352, 625943),
        (2314, 916872),
        (2259, 1460148),
        (2089, 3678986),
        (2017, 4587875),
    ],
    "2024-25": [
        (2810, 1),
        (2697, 155),
        (2570, 24629),
        (2521, 81835),
        (2452, 283124),
        (2335, 1122924),
        (2252, 2113595),
        (2239, 2298753),
        (2143, 3683973),
    ],
    "2025-26": [
        (2582, 1),
        (2399, 9575),
        (2337, 75615),
        (2240, 527869),
        (2212, 774352),
    ],
}


def _normalise_chip_name(value) -> str | None:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    text = str(value).strip().lower()
    return None if text in {"", "none", "nan"} else text


def _actual_gameweek_player_table(
    backtest_table: pd.DataFrame,
    gameweek: int,
    required_player_ids=(),
) -> pd.DataFrame:
    """Return candidate metadata plus realised points for one historical GW."""
    required = tuple(int(v) for v in required_player_ids)
    candidates = historical_candidate_snapshot(
        backtest_table,
        int(gameweek),
        required,
    ).copy()

    actual = (
        backtest_table.loc[
            backtest_table["gameweek"].eq(int(gameweek)),
            ["player_id", "actual_points", "actual_appeared"],
        ]
        .drop_duplicates("player_id")
        .copy()
    )
    actual["player_id"] = actual["player_id"].astype(int)

    data = candidates.merge(
        actual,
        on="player_id",
        how="left",
        validate="one_to_one",
    )
    data["actual_points"] = pd.to_numeric(
        data["actual_points"],
        errors="coerce",
    ).fillna(0.0)
    data["actual_appeared"] = (
        data["actual_appeared"].fillna(False).astype(bool)
    )
    data["position"] = optimiser_position(data["position"])
    data["price"] = pd.to_numeric(data["price"], errors="coerce")
    return data


def _best_hindsight_lineup_from_squad(
    squad_actual: pd.DataFrame,
    *,
    bench_boost: bool = False,
) -> dict[str, object]:
    """Best legal XI/captain from one fixed 15-man squad using realised points."""
    squad = squad_actual.drop_duplicates("player_id").copy()

    if len(squad) != 15:
        raise ValueError(
            f"Hindsight squad oracle expected 15 players, found {len(squad)}."
        )

    by_position = {
        position: (
            squad.loc[squad["position"].eq(position)]
            .sort_values(
                ["actual_points", "player_id"],
                ascending=[False, True],
            )
            .reset_index(drop=True)
        )
        for position in SQUAD_POSITION_REQUIREMENTS
    }

    best = None
    for defenders in range(3, 6):
        for midfielders in range(2, 6):
            for forwards in range(1, 4):
                if defenders + midfielders + forwards != 10:
                    continue
                if (
                    len(by_position["GK"]) < 1
                    or len(by_position["DEF"]) < defenders
                    or len(by_position["MID"]) < midfielders
                    or len(by_position["FWD"]) < forwards
                ):
                    continue

                selected = pd.concat(
                    [
                        by_position["GK"].head(1),
                        by_position["DEF"].head(defenders),
                        by_position["MID"].head(midfielders),
                        by_position["FWD"].head(forwards),
                    ],
                    ignore_index=True,
                )
                base = float(selected["actual_points"].sum())
                if best is None or base > best["base_points"] + 1e-12:
                    best = {
                        "lineup": selected,
                        "base_points": base,
                        "formation": (
                            f"{defenders}-{midfielders}-{forwards}"
                        ),
                    }

    if best is None:
        raise RuntimeError("No legal hindsight XI could be formed.")

    lineup = best["lineup"]
    captain_row = (
        lineup.sort_values(
            ["actual_points", "player_id"],
            ascending=[False, True],
        ).iloc[0]
    )
    captain_points = float(captain_row["actual_points"])

    if bench_boost:
        base_points = float(squad["actual_points"].sum())
    else:
        base_points = float(best["base_points"])

    return {
        "base_points": base_points,
        "best_xi_base_points": float(best["base_points"]),
        "all_15_points": float(squad["actual_points"].sum()),
        "lineup_ids": tuple(lineup["player_id"].astype(int)),
        "formation": best["formation"],
        "captain_id": int(captain_row["player_id"]),
        "captain_name": str(captain_row["web_name"]),
        "captain_points": captain_points,
    }


def _effective_model_captain(
    actual_players: pd.DataFrame,
    captain_id: int,
    vice_captain_id: int,
) -> tuple[int | None, str | None, float]:
    lookup = actual_players.set_index("player_id")
    captain_id = int(captain_id)
    vice_captain_id = int(vice_captain_id)

    captain_appeared = (
        captain_id in lookup.index
        and bool(lookup.loc[captain_id, "actual_appeared"])
    )
    vice_appeared = (
        vice_captain_id in lookup.index
        and bool(lookup.loc[vice_captain_id, "actual_appeared"])
    )

    effective = (
        captain_id
        if captain_appeared
        else vice_captain_id if vice_appeared else None
    )
    if effective is None:
        return None, None, 0.0

    row = lookup.loc[effective]
    return (
        int(effective),
        str(row["web_name"]),
        float(row["actual_points"]),
    )


def _one_week_hindsight_oracle(
    actual_players: pd.DataFrame,
    *,
    budget: float,
    chip: str | None,
) -> dict[str, object]:
    """
    Perfect-hindsight one-GW squad MILP.

    This is deliberately a ceiling rather than a realistic transfer strategy:
    it may rebuild the entire squad for free every Gameweek.
    """
    candidates = (
        actual_players.dropna(
            subset=["team_id", "position", "price"]
        )
        .drop_duplicates("player_id")
        .reset_index(drop=True)
        .copy()
    )
    if candidates.empty:
        raise RuntimeError("No candidates are available for hindsight oracle.")

    player_ids = candidates["player_id"].astype(int).tolist()
    n = len(candidates)

    # x = 15-man squad, y = XI, c = captain.
    x0 = 0
    y0 = n
    c0 = 2 * n
    variables = 3 * n

    def x(i): return x0 + i
    def y(i): return y0 + i
    def c(i): return c0 + i

    chip = _normalise_chip_name(chip)
    captain_extra_multiplier = 2.0 if chip == "triple_captain" else 1.0
    bench_boost = chip == "bench_boost"

    points = candidates["actual_points"].to_numpy(dtype=float)
    objective = np.zeros(variables, dtype=float)

    if bench_boost:
        for i in range(n):
            objective[x(i)] = -points[i]
            objective[c(i)] = -points[i]
    else:
        for i in range(n):
            objective[y(i)] = -points[i]
            objective[c(i)] = -captain_extra_multiplier * points[i]

    rows = []
    lower = []
    upper = []

    def add(terms, lo=-np.inf, hi=np.inf):
        rows.append(terms)
        lower.append(float(lo))
        upper.append(float(hi))

    add({x(i): 1.0 for i in range(n)}, 15, 15)

    for position, required in SQUAD_POSITION_REQUIREMENTS.items():
        indices = np.flatnonzero(
            candidates["position"].eq(position).to_numpy()
        )
        add({x(int(i)): 1.0 for i in indices}, required, required)

    add(
        {
            x(i): float(candidates.loc[i, "price"])
            for i in range(n)
        },
        hi=float(budget),
    )

    for team_id in candidates["team_id"].dropna().unique():
        indices = np.flatnonzero(
            candidates["team_id"].eq(team_id).to_numpy()
        )
        add(
            {x(int(i)): 1.0 for i in indices},
            hi=float(MAX_PLAYERS_PER_CLUB),
        )

    add({y(i): 1.0 for i in range(n)}, 11, 11)
    for position, (minimum, maximum) in LINEUP_POSITION_BOUNDS.items():
        indices = np.flatnonzero(
            candidates["position"].eq(position).to_numpy()
        )
        add(
            {y(int(i)): 1.0 for i in indices},
            minimum,
            maximum,
        )

    add({c(i): 1.0 for i in range(n)}, 1, 1)

    for i in range(n):
        add({y(i): 1.0, x(i): -1.0}, hi=0.0)
        add({c(i): 1.0, y(i): -1.0}, hi=0.0)

    row_idx = []
    col_idx = []
    coeff = []
    for r, terms in enumerate(rows):
        for variable, value in terms.items():
            row_idx.append(r)
            col_idx.append(variable)
            coeff.append(float(value))

    matrix = coo_matrix(
        (coeff, (row_idx, col_idx)),
        shape=(len(rows), variables),
    ).tocsr()

    result = milp(
        c=objective,
        integrality=np.ones(variables, dtype=int),
        bounds=Bounds(
            np.zeros(variables),
            np.ones(variables),
        ),
        constraints=LinearConstraint(
            matrix,
            np.asarray(lower),
            np.asarray(upper),
        ),
        options={"time_limit": 30},
    )
    if not result.success or result.x is None:
        raise RuntimeError(
            "One-Gameweek hindsight oracle failed: "
            f"{result.message}"
        )

    solution = result.x
    squad_mask = np.array([solution[x(i)] > 0.5 for i in range(n)])
    lineup_mask = np.array([solution[y(i)] > 0.5 for i in range(n)])
    captain_mask = np.array([solution[c(i)] > 0.5 for i in range(n)])

    squad = candidates.loc[squad_mask].copy()
    lineup = candidates.loc[lineup_mask].copy()
    captain = candidates.loc[captain_mask].iloc[0]

    return {
        "score": float(-result.fun),
        "budget": float(budget),
        "squad_cost": float(squad["price"].sum()),
        "captain_id": int(captain["player_id"]),
        "captain_name": str(captain["web_name"]),
        "squad_ids": tuple(squad["player_id"].astype(int)),
        "lineup_ids": tuple(lineup["player_id"].astype(int)),
    }


def _approximate_rank_from_anchors(
    season: str,
    points: float,
) -> tuple[float | None, str]:
    """Log-rank interpolation between sparse observed public score/rank anchors."""
    anchors = sorted(
        HISTORICAL_RANK_ANCHORS.get(str(season), []),
        key=lambda item: item[0],
        reverse=True,
    )
    if not anchors:
        return None, "No rank anchors configured."

    score = float(points)
    best_points, best_rank = anchors[0]
    worst_points, worst_rank = anchors[-1]

    if score > best_points:
        return 1.0, f"Above best configured anchor ({best_points} pts)."
    if score == best_points:
        return float(best_rank), "Exact configured anchor."
    if score < worst_points:
        return None, (
            f"Below lowest configured anchor: >{worst_rank:,} "
            f"at {worst_points} points."
        )
    if score == worst_points:
        return float(worst_rank), "Exact configured anchor."

    for (high_points, high_rank), (low_points, low_rank) in zip(
        anchors[:-1],
        anchors[1:],
    ):
        if high_points >= score >= low_points:
            if high_points == low_points:
                estimate = float(max(high_rank, low_rank))
            else:
                fraction = (
                    (high_points - score)
                    / (high_points - low_points)
                )
                log_rank = (
                    np.log(max(1.0, high_rank))
                    + fraction
                    * (
                        np.log(max(1.0, low_rank))
                        - np.log(max(1.0, high_rank))
                    )
                )
                estimate = float(np.exp(log_rank))
            return estimate, (
                "Approximate log-rank interpolation between sparse "
                "public FPL history anchors."
            )

    return None, "Rank interpolation unavailable."


def build_cache_only_regret_diagnostics() -> dict[str, object]:
    if backtest_result is None or backtest_weekly_results.empty:
        if REGRET_REQUIRE_CACHED_SEQUENTIAL:
            raise RuntimeError(
                "Regret diagnostics require the signed v2.1 sequential cache, "
                "but no valid cached sequential result was loaded. Do NOT turn "
                "on the expensive backtest automatically. Confirm that "
                "data/sequential_backtest_v2_1.pkl.gz from Run B is present "
                "and that PROJECT_VERSION remains 2.1.0."
            )
        return {}

    weekly = backtest_weekly_results.copy()
    if REGRET_MAX_GAMEWEEKS is not None:
        weekly = weekly.head(int(REGRET_MAX_GAMEWEEKS)).copy()

    rows = []

    for record in weekly.itertuples(index=False):
        gameweek = int(record.gameweek)
        chip = _normalise_chip_name(record.chip)
        squad_ids = tuple(int(v) for v in record.squad_ids)

        full_actual = _actual_gameweek_player_table(
            historical_backtest_table,
            gameweek,
            squad_ids,
        )
        squad_actual = full_actual.loc[
            full_actual["player_id"].isin(squad_ids)
        ].copy()

        effective_cap_id, effective_cap_name, effective_cap_points = (
            _effective_model_captain(
                full_actual,
                int(record.captain_id),
                int(record.vice_captain_id),
            )
        )

        cap_multiplier_extra = (
            2.0 if chip == "triple_captain" else 1.0
        )
        model_cap_bonus = (
            cap_multiplier_extra * effective_cap_points
        )
        model_before_hits = (
            float(record.actual_score) + float(record.hit_cost)
        )
        model_base_score = (
            model_before_hits - model_cap_bonus
        )

        same_squad_available = chip != "free_hit"
        same_squad = None
        if same_squad_available:
            same_squad = _best_hindsight_lineup_from_squad(
                squad_actual,
                bench_boost=(chip == "bench_boost"),
            )
            oracle_cap_bonus = (
                cap_multiplier_extra
                * float(same_squad["captain_points"])
            )
            same_squad_before_hits = (
                float(same_squad["base_points"])
                + oracle_cap_bonus
            )
            same_squad_after_hits = (
                same_squad_before_hits - float(record.hit_cost)
            )

            lineup_regret = (
                float(same_squad["base_points"])
                - model_base_score
            )
            captain_regret = (
                oracle_cap_bonus - model_cap_bonus
            )
            total_execution_regret = (
                same_squad_after_hits - float(record.actual_score)
            )

            # TC timing: one *additional* captain copy relative to normal.
            tc_hindsight_uplift = float(
                same_squad["captain_points"]
            )
            # BB timing proxy: all-15 points minus hindsight best-XI base.
            bb_hindsight_surplus = float(
                same_squad["all_15_points"]
                - same_squad["best_xi_base_points"]
            )
        else:
            same_squad_after_hits = np.nan
            lineup_regret = np.nan
            captain_regret = np.nan
            total_execution_regret = np.nan
            tc_hindsight_uplift = np.nan
            bb_hindsight_surplus = np.nan

        free_rebuild_score = np.nan
        free_rebuild_budget = np.nan
        free_rebuild_captain = None

        if REGRET_BUILD_WEEKLY_FREE_REBUILD_ORACLE:
            universe = _actual_gameweek_player_table(
                historical_backtest_table,
                gameweek,
            )

            if REGRET_ORACLE_BUDGET_MODE == "fixed_100":
                oracle_budget = float(BACKTEST_INITIAL_BUDGET)
            elif REGRET_ORACLE_BUDGET_MODE == "market_value_plus_bank":
                squad_market_value = float(
                    squad_actual["price"].sum()
                )
                oracle_budget = max(
                    float(BACKTEST_INITIAL_BUDGET),
                    squad_market_value + float(record.bank_after),
                )
            else:
                raise ValueError(
                    "REGRET_ORACLE_BUDGET_MODE must be "
                    "'market_value_plus_bank' or 'fixed_100'."
                )

            oracle = _one_week_hindsight_oracle(
                universe,
                budget=oracle_budget,
                chip=chip,
            )
            free_rebuild_score = float(oracle["score"])
            free_rebuild_budget = float(oracle["budget"])
            free_rebuild_captain = oracle["captain_name"]

        rows.append({
            "gameweek": gameweek,
            "chip": chip,
            "model_actual_after_hits": float(record.actual_score),
            "model_actual_before_hits": model_before_hits,
            "hit_cost": float(record.hit_cost),
            "model_captain": effective_cap_name,
            "model_captain_points": effective_cap_points,
            "same_squad_oracle_available": same_squad_available,
            "same_squad_oracle_after_hits": same_squad_after_hits,
            "same_squad_oracle_captain": (
                same_squad["captain_name"]
                if same_squad is not None else None
            ),
            "lineup_regret": lineup_regret,
            "captain_regret": captain_regret,
            "execution_regret": total_execution_regret,
            "tc_hindsight_uplift": tc_hindsight_uplift,
            "bb_hindsight_surplus": bb_hindsight_surplus,
            "free_rebuild_oracle_score": free_rebuild_score,
            "free_rebuild_oracle_budget": free_rebuild_budget,
            "free_rebuild_oracle_captain": free_rebuild_captain,
            "squad_opportunity_ceiling": (
                max(
                    0.0,
                    free_rebuild_score - same_squad_after_hits,
                )
                if np.isfinite(free_rebuild_score)
                and np.isfinite(same_squad_after_hits)
                else np.nan
            ),
        })

    weekly_regret = pd.DataFrame(rows)

    valid_execution = weekly_regret.loc[
        weekly_regret["same_squad_oracle_available"]
    ].copy()

    model_points_all = float(
        weekly_regret["model_actual_after_hits"].sum()
    )
    model_points_comparable = float(
        valid_execution["model_actual_after_hits"].sum()
    )
    same_squad_points = float(
        valid_execution["same_squad_oracle_after_hits"].sum()
    )

    regret_summary = pd.DataFrame([{
        "season": BACKTEST_SEASON,
        "model_points_all_gws": model_points_all,
        "comparable_non_fh_model_points": model_points_comparable,
        "same_squad_hindsight_points_non_fh": same_squad_points,
        "lineup_regret_non_fh": float(
            valid_execution["lineup_regret"].sum()
        ),
        "captain_regret_non_fh": float(
            valid_execution["captain_regret"].sum()
        ),
        "execution_regret_non_fh": float(
            valid_execution["execution_regret"].sum()
        ),
        "free_hit_weeks_excluded_from_execution_regret": int(
            (~weekly_regret["same_squad_oracle_available"]).sum()
        ),
        "free_rebuild_oracle_total": (
            float(weekly_regret["free_rebuild_oracle_score"].sum())
            if weekly_regret["free_rebuild_oracle_score"].notna().any()
            else np.nan
        ),
        "squad_opportunity_ceiling_non_fh": (
            float(weekly_regret["squad_opportunity_ceiling"].sum())
            if weekly_regret["squad_opportunity_ceiling"].notna().any()
            else np.nan
        ),
    }])

    # TC / BB timing: cache-only hindsight diagnostics.
    chip_timing_rows = []
    for half_name, start, end in [
        ("first_half", 1, 19),
        ("second_half", 20, 38),
    ]:
        half = weekly_regret.loc[
            weekly_regret["gameweek"].between(start, end)
            & weekly_regret["same_squad_oracle_available"]
        ].copy()

        for chip_name, value_col in [
            ("triple_captain", "tc_hindsight_uplift"),
            ("bench_boost", "bb_hindsight_surplus"),
        ]:
            usable = half.loc[
                pd.to_numeric(
                    half[value_col],
                    errors="coerce",
                ).notna()
            ].copy()
            if usable.empty:
                continue

            best = usable.loc[
                usable[value_col].astype(float).idxmax()
            ]
            used = half.loc[half["chip"].eq(chip_name)]

            chip_timing_rows.append({
                "half": half_name,
                "chip": chip_name,
                "used_gameweek": (
                    int(used.iloc[0]["gameweek"])
                    if not used.empty else None
                ),
                "used_hindsight_uplift": (
                    float(used.iloc[0][value_col])
                    if not used.empty else np.nan
                ),
                "best_hindsight_gameweek": int(best["gameweek"]),
                "best_hindsight_uplift": float(best[value_col]),
                "timing_regret": (
                    float(best[value_col] - used.iloc[0][value_col])
                    if not used.empty else np.nan
                ),
                "scope": (
                    "Same-squad hindsight timing only; does not "
                    "counterfactually combine chips."
                ),
            })

    chip_timing = pd.DataFrame(chip_timing_rows)

    # Cross-season elite benchmark from the already-cached policy scorecard.
    policy_benchmark_rows = []
    if (
        not walk_forward_strategy_scorecard.empty
        and recommended_policy_profile
        in set(walk_forward_strategy_scorecard["profile"])
    ):
        selected_policy = walk_forward_strategy_scorecard.loc[
            walk_forward_strategy_scorecard["profile"].eq(
                recommended_policy_profile
            )
        ].copy()

        for row in selected_policy.itertuples(index=False):
            season = str(row.holdout_season)
            score = float(row.strategy_points)
            winner = HISTORICAL_WINNER_POINTS.get(season)
            estimate, rank_note = _approximate_rank_from_anchors(
                season,
                score,
            )
            policy_benchmark_rows.append({
                "season": season,
                "source": "cached selected transfer policy",
                "profile": recommended_policy_profile,
                "points": score,
                "winner_points": winner,
                "gap_to_winner": (
                    float(winner - score)
                    if winner is not None else np.nan
                ),
                "winner_score_fraction": (
                    float(score / winner)
                    if winner else np.nan
                ),
                "approximate_rank": (
                    int(round(estimate))
                    if estimate is not None else None
                ),
                "rank_note": rank_note,
                "important_scope_note": (
                    "Cross-season policy validation excludes chips."
                ),
            })

    if backtest_result is not None:
        season = str(BACKTEST_SEASON)
        score = float(
            backtest_weekly_results["actual_score"].sum()
        )
        winner = HISTORICAL_WINNER_POINTS.get(season)
        estimate, rank_note = _approximate_rank_from_anchors(
            season,
            score,
        )
        policy_benchmark_rows.append({
            "season": season,
            "source": "cached full sequential strategy",
            "profile": "full sequential + chips",
            "points": score,
            "winner_points": winner,
            "gap_to_winner": (
                float(winner - score)
                if winner is not None else np.nan
            ),
            "winner_score_fraction": (
                float(score / winner)
                if winner else np.nan
            ),
            "approximate_rank": (
                int(round(estimate))
                if estimate is not None else None
            ),
            "rank_note": rank_note,
            "important_scope_note": (
                "Includes the model's historical chip schedule."
            ),
        })

    elite_benchmark = pd.DataFrame(policy_benchmark_rows)

    return {
        "weekly_regret": weekly_regret,
        "summary": regret_summary,
        "chip_timing": chip_timing,
        "elite_benchmark": elite_benchmark,
    }


if RUN_REGRET_DIAGNOSTICS:
    regret_diagnostics = build_cache_only_regret_diagnostics()

    if regret_diagnostics:
        historical_regret_weekly = regret_diagnostics["weekly_regret"]
        historical_regret_summary = regret_diagnostics["summary"]
        historical_chip_timing_regret = regret_diagnostics["chip_timing"]
        historical_elite_benchmark = regret_diagnostics["elite_benchmark"]

        print("=" * 72)
        print("CACHE-ONLY HISTORICAL REGRET DIAGNOSTICS")
        print("=" * 72)

        print("Season-level regret summary")
        display(historical_regret_summary.round(2))

        print("Largest same-squad execution-regret Gameweeks")
        display(
            historical_regret_weekly.loc[
                historical_regret_weekly[
                    "execution_regret"
                ].notna()
            ]
            .sort_values(
                "execution_regret",
                ascending=False,
            )
            .head(12)
            .round(2)
        )

        print("Captaincy regret leaders")
        display(
            historical_regret_weekly.loc[
                historical_regret_weekly[
                    "captain_regret"
                ].notna()
            ][
                [
                    "gameweek",
                    "chip",
                    "model_captain",
                    "model_captain_points",
                    "same_squad_oracle_captain",
                    "captain_regret",
                    "lineup_regret",
                    "execution_regret",
                ]
            ]
            .sort_values(
                "captain_regret",
                ascending=False,
            )
            .head(12)
            .round(2)
        )

        if REGRET_BUILD_WEEKLY_FREE_REBUILD_ORACLE:
            print(
                "Largest one-Gameweek squad-opportunity ceilings "
                "(perfect hindsight; not actionable transfer regret)"
            )
            display(
                historical_regret_weekly.loc[
                    historical_regret_weekly[
                        "squad_opportunity_ceiling"
                    ].notna()
                ][
                    [
                        "gameweek",
                        "chip",
                        "model_actual_after_hits",
                        "same_squad_oracle_after_hits",
                        "free_rebuild_oracle_score",
                        "free_rebuild_oracle_budget",
                        "squad_opportunity_ceiling",
                    ]
                ]
                .sort_values(
                    "squad_opportunity_ceiling",
                    ascending=False,
                )
                .head(12)
                .round(2)
            )

        print("Cache-only TC / BB timing regret")
        if historical_chip_timing_regret.empty:
            print("No TC/BB timing rows available.")
        else:
            display(historical_chip_timing_regret.round(2))

        print("Historical elite / winner benchmark")
        if historical_elite_benchmark.empty:
            print("No valid cached policy scorecard was available.")
        else:
            display(historical_elite_benchmark.round(3))

        # Helpful direct interpretation.
        summary_row = historical_regret_summary.iloc[0]
        print("\nInterpretation")
        print(
            "- Same-squad execution regret measures points that were available "
            "without changing a single owned player."
        )
        print(
            "- Captain regret + lineup regret decompose that same-squad gap."
        )
        print(
            "- The free-rebuild oracle is intentionally impossible in real FPL; "
            "its extra gap is a ceiling on squad/transfer opportunity, not a "
            "claim that those points were achievable."
        )
        print(
            "- Free Hit weeks are excluded from same-squad decomposition because "
            "the v2.1 cache stores the permanent squad, not the temporary FH squad."
        )
        print(
            "- Rank estimates use sparse public anchors and should be treated as "
            "indicative, never as exact historical cutoffs."
        )

        if REGRET_SAVE_OUTPUT:
            REGRET_DIAGNOSTICS_OUTPUT_PATH.parent.mkdir(
                parents=True,
                exist_ok=True,
            )

            payload = {
                "notebook_release": NOTEBOOK_RELEASE,
                "model_project_version": PROJECT_VERSION,
                "backtest_season": BACKTEST_SEASON,
                "generated_at_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
                "summary": (
                    historical_regret_summary
                    .replace({np.nan: None})
                    .to_dict("records")
                ),
                "weekly_regret": (
                    historical_regret_weekly
                    .replace({np.nan: None})
                    .to_dict("records")
                ),
                "chip_timing": (
                    historical_chip_timing_regret
                    .replace({np.nan: None})
                    .to_dict("records")
                ),
                "elite_benchmark": (
                    historical_elite_benchmark
                    .replace({np.nan: None})
                    .to_dict("records")
                ),
            }

            with REGRET_DIAGNOSTICS_OUTPUT_PATH.open(
                "w",
                encoding="utf-8",
            ) as file:
                json.dump(
                    payload,
                    file,
                    indent=2,
                    default=str,
                )

            print(
                "Saved regret diagnostics to:",
                REGRET_DIAGNOSTICS_OUTPUT_PATH,
            )
else:
    historical_regret_weekly = pd.DataFrame()
    historical_regret_summary = pd.DataFrame()
    historical_chip_timing_regret = pd.DataFrame()
    historical_elite_benchmark = pd.DataFrame()
    print(
        "Regret diagnostics skipped. Set RUN_REGRET_DIAGNOSTICS=True "
        "in the first code cell to enable them."
    )


### How to interpret the regret section

The most useful number is **same-squad execution regret**.

If that number is large, the model is leaving many points on the table through
line-up/captain choices even when its player ownership is held fixed. That
would point directly toward improving captaincy, expected minutes, auto-sub
handling or XI optimisation.

If same-squad regret is small but the one-week free-rebuild ceiling is much
higher, the larger opportunity is in **which players the model owns**, i.e.
player forecasts, opening-squad construction and transfer decisions.

The free-rebuild figure is intentionally an upper bound. It is *not* a valid
backtest strategy because it uses realised points and can replace all 15
players every week for free.

The elite benchmark uses the already-saved cross-season policy scorecard and
the saved full sequential result. The cross-season policy score excludes chips,
so its gap to the winner is contextual rather than a like-for-like rank claim.
The full sequential benchmark includes the model's historical chip schedule.

This section changes **no production decision logic** and therefore does not
invalidate the v2.1 signed caches.


# 11C. Cheap execution-policy validation and realistic transfer regret

The v2.1 regret analysis showed that the 2025/26 cached squad contained far
more realised points than the final score extracted from it. This section
therefore targets the **execution layer** rather than changing the validated
forecasting or transfer model.

It does two cheap jobs:

1. **Captain / vice / XI policy validation.** The exact 15-player squads from
   the signed 2025/26 sequential cache are held fixed. Several conservative
   pre-deadline rules are re-scored using the leakage-safe expected-points,
   expected-minutes and appearance forecasts that were available for each
   Gameweek. A candidate policy may become the 2026/27 production execution
   overlay only if it beats the baseline by the configured margin and does not
   materially fail in either half of the season.
2. **One-/two-transfer hindsight ceiling.** For ordinary historical
   Gameweeks, a small MILP starts from the permanent squad held before the
   deadline and allows at most one or two position-preserving transfers. It
   uses realised points, so it is still a hindsight ceiling rather than a
   deployable strategy, but it is vastly more realistic than rebuilding all 15
   players every week.

The expensive transfer-policy and sequential backtests remain **off**. The
core `PROJECT_VERSION` intentionally stays at `2.1.0`, so their signed caches
are not invalidated. The newly selected execution overlay affects current-week
XI/captain/vice decisions only; the six-week transfer search remains the
validated v2.1 core.


In [ ]:
# Execution-layer validation ---------------------------------------------------

# Memoization is critical here. The exact bench-order evaluator enumerates
# appearance states, so recomputing the same GW/squad/lineup for every captain
# tie-break candidate is unnecessarily expensive.
_EXECUTION_CONTEXT_CACHE: dict[tuple, tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]] = {}
_EXECUTION_LINEUP_DETAIL_CACHE: dict[tuple, dict[str, object]] = {}
_EXECUTION_BASELINE_DETAIL_CACHE: dict[tuple, dict[str, object]] = {}


def _execution_projection_for_squad(
    backtest_table: pd.DataFrame,
    gameweek: int,
    squad_ids,
) -> pd.DataFrame:
    """Cheap one-GW projection table from the already-built leakage-safe holdout."""
    ids = {int(v) for v in squad_ids}
    cols = [
        "player_id",
        "gameweek",
        "expected_points",
        "expected_minutes",
        "appearance_probability",
        "no_appearance_probability",
    ]
    projection = (
        backtest_table.loc[
            backtest_table["gameweek"].eq(int(gameweek))
            & backtest_table["player_id"].astype(int).isin(ids),
            cols,
        ]
        .drop_duplicates(["player_id", "gameweek"])
        .copy()
    )
    return projection


def _captain_confidence_from_gap(gap: float) -> str:
    if not np.isfinite(gap):
        return "Not assessed"
    if gap <= CAPTAIN_CONFIDENCE_VERY_LOW_GAP:
        return "Very low"
    if gap <= CAPTAIN_CONFIDENCE_LOW_GAP:
        return "Low"
    if gap <= CAPTAIN_CONFIDENCE_MEDIUM_GAP:
        return "Medium"
    return "High"


def _captain_alternative_table(
    starters: pd.DataFrame,
    selected_captain_id: int,
    *,
    tie_band: float,
) -> pd.DataFrame:
    table = starters[
        [
            "player_id",
            "web_name",
            "expected_points",
            "expected_minutes",
            "appearance_probability",
            "no_appearance_probability",
        ]
    ].copy()
    table["expected_points"] = pd.to_numeric(
        table["expected_points"], errors="coerce"
    ).fillna(0.0)
    table["expected_minutes"] = pd.to_numeric(
        table["expected_minutes"], errors="coerce"
    ).fillna(0.0)
    table["appearance_probability"] = pd.to_numeric(
        table["appearance_probability"], errors="coerce"
    ).fillna(0.0)

    table = table.sort_values(
        ["expected_points", "expected_minutes", "appearance_probability"],
        ascending=[False, False, False],
    ).reset_index(drop=True)

    best_xp = float(table.iloc[0]["expected_points"]) if not table.empty else np.nan
    table["xp_gap_to_top"] = best_xp - table["expected_points"]
    table["within_captain_tie_band"] = (
        table["xp_gap_to_top"].le(float(tie_band) + 1e-12)
    )
    table["selected"] = table["player_id"].astype(int).eq(
        int(selected_captain_id)
    )
    return table.head(int(CAPTAIN_ALTERNATIVES_TO_SHOW))


def _select_overlay_lineup(
    squad_projection: pd.DataFrame,
    minutes_weight: float,
) -> tuple[int, ...]:
    """Choose a legal XI with xP primary and a deliberately tiny minutes bonus."""
    data = squad_projection.copy()
    data["_execution_utility"] = (
        pd.to_numeric(data["expected_points"], errors="coerce").fillna(0.0)
        + float(minutes_weight)
        * (
            pd.to_numeric(data["expected_minutes"], errors="coerce")
            .fillna(0.0)
            .clip(0.0, 90.0)
            / 90.0
        )
    )

    by_position = {
        position: (
            data.loc[data["position"].eq(position)]
            .sort_values(
                ["_execution_utility", "expected_points", "expected_minutes"],
                ascending=[False, False, False],
            )
            .reset_index(drop=True)
        )
        for position in ["GK", "DEF", "MID", "FWD"]
    }

    best = None
    for formation in VALID_OUTFIELD_FORMATIONS:
        selected_parts = []
        legal = True
        for position, required in formation.items():
            available = by_position.get(position, pd.DataFrame())
            if len(available) < int(required):
                legal = False
                break
            selected_parts.append(available.head(int(required)))
        if not legal:
            continue

        selected = pd.concat(selected_parts, ignore_index=True)
        utility = float(selected["_execution_utility"].sum())
        xp = float(selected["expected_points"].sum())
        minutes = float(selected["expected_minutes"].sum())
        candidate = {
            "starter_ids": tuple(selected["player_id"].astype(int)),
            "utility": utility,
            "xp": xp,
            "minutes": minutes,
        }
        if (
            best is None
            or candidate["utility"] > best["utility"] + 1e-12
            or (
                abs(candidate["utility"] - best["utility"]) <= 1e-12
                and candidate["xp"] > best["xp"] + 1e-12
            )
        ):
            best = candidate

    if best is None:
        raise RuntimeError("No legal execution-overlay XI exists.")
    return best["starter_ids"]


def _select_overlay_captain(
    starters: pd.DataFrame,
    *,
    tie_band: float,
    mode: str,
) -> int:
    data = starters.copy()
    data["expected_points"] = pd.to_numeric(
        data["expected_points"], errors="coerce"
    ).fillna(0.0)
    data["expected_minutes"] = pd.to_numeric(
        data["expected_minutes"], errors="coerce"
    ).fillna(0.0)
    data["appearance_probability"] = pd.to_numeric(
        data["appearance_probability"], errors="coerce"
    ).fillna(0.0)

    max_xp = float(data["expected_points"].max())
    eligible = data.loc[
        data["expected_points"].ge(max_xp - float(tie_band) - 1e-12)
    ].copy()

    if mode == "minutes_tiebreak":
        eligible = eligible.sort_values(
            ["expected_minutes", "appearance_probability", "expected_points"],
            ascending=[False, False, False],
        )
    elif mode == "appearance_tiebreak":
        eligible = eligible.sort_values(
            ["appearance_probability", "expected_minutes", "expected_points"],
            ascending=[False, False, False],
        )
    elif mode == "ceiling_tiebreak":
        eligible["_ceiling_rate"] = (
            eligible["expected_points"]
            / eligible["expected_minutes"].clip(lower=30.0)
            * 90.0
        )
        eligible = eligible.sort_values(
            ["_ceiling_rate", "expected_points", "expected_minutes"],
            ascending=[False, False, False],
        )
    elif mode == "xp":
        eligible = eligible.sort_values(
            ["expected_points", "expected_minutes"],
            ascending=[False, False],
        )
    else:
        raise ValueError(f"Unknown captain execution mode: {mode}")

    return int(eligible.iloc[0]["player_id"])


def _select_overlay_vice(
    starters: pd.DataFrame,
    captain_id: int,
    *,
    tie_band: float,
) -> int:
    data = starters.loc[
        ~starters["player_id"].astype(int).eq(int(captain_id))
    ].copy()
    data["expected_points"] = pd.to_numeric(
        data["expected_points"], errors="coerce"
    ).fillna(0.0)
    data["expected_minutes"] = pd.to_numeric(
        data["expected_minutes"], errors="coerce"
    ).fillna(0.0)
    data["appearance_probability"] = pd.to_numeric(
        data["appearance_probability"], errors="coerce"
    ).fillna(0.0)

    max_xp = float(data["expected_points"].max())
    eligible = data.loc[
        data["expected_points"].ge(max_xp - float(tie_band) - 1e-12)
    ].copy()
    eligible = eligible.sort_values(
        ["appearance_probability", "expected_minutes", "expected_points"],
        ascending=[False, False, False],
    )
    return int(eligible.iloc[0]["player_id"])


def execution_policy_squad_gameweek_decision(
    squad_ids,
    gameweek,
    candidates,
    projections,
    policy: dict[str, object] | None = None,
):
    """
    Exact current-GW decision with an optional validated marginal execution overlay.

    Transfer planning continues to use the original v2.1 decision objective.
    """
    if policy is None:
        policy = globals().get(
            "ACTIVE_EXECUTION_POLICY",
            {
                "name": "baseline",
                "lineup_minutes_weight": 0.0,
                "captain_tie_band": 0.0,
                "captain_mode": "xp",
                "vice_tie_band": 0.0,
            },
        )

    if (
        not APPLY_EXECUTION_POLICY_TO_PRODUCTION
        or policy.get("name") == "baseline"
    ):
        baseline_cache_key = (
            int(gameweek),
            tuple(sorted(int(v) for v in squad_ids)),
        )
        cached_baseline = _EXECUTION_BASELINE_DETAIL_CACHE.get(
            baseline_cache_key
        )
        if cached_baseline is None:
            cached_baseline = exact_squad_gameweek_decision(
                squad_ids,
                gameweek,
                candidates,
                projections,
            )
            _EXECUTION_BASELINE_DETAIL_CACHE[
                baseline_cache_key
            ] = cached_baseline
        detailed = dict(cached_baseline)
        starters = detailed["squad_projection"].loc[
            detailed["squad_projection"]["player_id"].isin(
                detailed["starter_ids"]
            )
        ].copy()
        alternatives = _captain_alternative_table(
            starters,
            int(detailed["captain_id"]),
            tie_band=0.0,
        )
        selected_row = alternatives.loc[alternatives["selected"]]
        selected_xp = (
            float(selected_row.iloc[0]["expected_points"])
            if not selected_row.empty else np.nan
        )
        other = alternatives.loc[~alternatives["selected"]]
        next_xp = (
            float(other["expected_points"].max())
            if not other.empty else np.nan
        )
        gap = (
            selected_xp - next_xp
            if np.isfinite(selected_xp) and np.isfinite(next_xp)
            else np.nan
        )
        detailed["execution_policy_name"] = "baseline"
        detailed["captain_alternatives"] = alternatives
        detailed["captain_xp_gap"] = float(gap) if np.isfinite(gap) else np.nan
        detailed["captain_confidence"] = _captain_confidence_from_gap(
            abs(float(gap)) if np.isfinite(gap) else np.nan
        )
        return detailed

    selected_squad = (
        candidates.loc[candidates["player_id"].isin(
            [int(v) for v in squad_ids]
        )]
        .drop_duplicates("player_id")
        .copy()
    )
    gameweek_projection = (
        projections.loc[
            projections["gameweek"].eq(int(gameweek))
            & projections["player_id"].isin(
                selected_squad["player_id"].astype(int)
            ),
            [
                "player_id",
                "expected_points",
                "expected_minutes",
                "appearance_probability",
                "no_appearance_probability",
            ],
        ]
        .drop_duplicates("player_id")
        .copy()
    )

    # Build the same projection frame used by the original exact decision.
    squad_projection = (
        selected_squad[
            [
                "player_id",
                "web_name",
                "team_short_name",
                "position",
                "price",
            ]
        ]
        .merge(
            gameweek_projection,
            on="player_id",
            how="left",
            validate="one_to_one",
        )
    )
    for column, fill_value in {
        "expected_points": 0.0,
        "expected_minutes": 0.0,
        "appearance_probability": 0.0,
        "no_appearance_probability": 1.0,
    }.items():
        squad_projection[column] = pd.to_numeric(
            squad_projection[column], errors="coerce"
        ).fillna(fill_value)
    squad_projection["position"] = optimiser_position(
        squad_projection["position"]
    )

    starter_ids = _select_overlay_lineup(
        squad_projection,
        float(policy["lineup_minutes_weight"]),
    )
    projected_lineup = (
        selected_squad.loc[
            selected_squad["player_id"].astype(int).isin(starter_ids)
        ]
        .merge(
            gameweek_projection[["player_id", "expected_points"]],
            on="player_id",
            how="left",
            validate="one_to_one",
        )
    )

    # Reuse the proven bench-order/autosub expectation code, but memoize it.
    # Captain mode/tie-band candidates sharing the same lineup weight have the
    # same XI and bench-order problem, so this expensive calculation should run
    # once per (GW, squad, lineup_weight), not once per captain candidate.
    lineup_cache_key = (
        int(gameweek),
        tuple(sorted(int(v) for v in squad_ids)),
        round(float(policy["lineup_minutes_weight"]), 6),
    )
    cached_detail = _EXECUTION_LINEUP_DETAIL_CACHE.get(lineup_cache_key)
    if cached_detail is None:
        cached_detail = optimise_current_gameweek_details(
            selected_squad,
            projected_lineup,
            gameweek_projection,
        )
        _EXECUTION_LINEUP_DETAIL_CACHE[lineup_cache_key] = cached_detail
    detailed = dict(cached_detail)
    starters = detailed["squad_projection"].loc[
        detailed["squad_projection"]["player_id"].astype(int).isin(
            set(int(v) for v in starter_ids)
        )
    ].copy()

    captain_id = _select_overlay_captain(
        starters,
        tie_band=float(policy["captain_tie_band"]),
        mode=str(policy["captain_mode"]),
    )
    vice_id = _select_overlay_vice(
        starters,
        captain_id,
        tie_band=float(policy["vice_tie_band"]),
    )

    lookup = starters.set_index("player_id")
    captain = lookup.loc[captain_id]
    vice = lookup.loc[vice_id]
    captain_extra = float(captain["expected_points"])
    vice_fallback_extra = (
        float(captain["no_appearance_probability"])
        * float(vice["expected_points"])
    )
    uplift = captain_extra + vice_fallback_extra

    detailed["starter_ids"] = set(int(v) for v in starter_ids)
    detailed["captain_id"] = int(captain_id)
    detailed["vice_captain_id"] = int(vice_id)
    detailed["captain_extra"] = captain_extra
    detailed["vice_fallback_extra"] = vice_fallback_extra
    detailed["expected_captain_uplift"] = float(uplift)
    detailed["expected_total_score"] = (
        float(detailed["expected_base_score"]) + float(uplift)
    )
    detailed["gameweek"] = int(gameweek)
    detailed["execution_policy_name"] = str(policy["name"])

    alternatives = _captain_alternative_table(
        starters,
        captain_id,
        tie_band=float(policy["captain_tie_band"]),
    )
    selected_xp = float(lookup.loc[captain_id, "expected_points"])
    best_other_xp = float(
        starters.loc[
            ~starters["player_id"].astype(int).eq(captain_id),
            "expected_points",
        ].max()
    )
    gap = selected_xp - best_other_xp
    detailed["captain_alternatives"] = alternatives
    detailed["captain_xp_gap"] = float(gap)
    detailed["captain_confidence"] = _captain_confidence_from_gap(
        abs(float(gap))
    )
    return detailed


def _execution_candidate_policies() -> list[dict[str, object]]:
    candidates = [{
        "name": "baseline",
        "lineup_minutes_weight": 0.0,
        "captain_tie_band": 0.0,
        "captain_mode": "xp",
        "vice_tie_band": 0.0,
    }]

    # Keep the grid intentionally conservative: all changes are limited to
    # marginal xP calls, with no arbitrary player/club overrides.
    for lineup_weight in EXECUTION_LINEUP_MINUTES_WEIGHTS:
        for captain_band in EXECUTION_CAPTAIN_TIE_BANDS:
            if lineup_weight == 0.0 and captain_band == 0.0:
                continue
            for captain_mode in EXECUTION_CAPTAIN_MODES:
                candidates.append({
                    "name": (
                        f"lw{lineup_weight:.3f}_"
                        f"cb{captain_band:.2f}_"
                        f"{captain_mode}"
                    ),
                    "lineup_minutes_weight": float(lineup_weight),
                    "captain_tie_band": float(captain_band),
                    "captain_mode": str(captain_mode),
                    "vice_tie_band": float(EXECUTION_VICE_XP_TIE_BAND),
                })
    return candidates


def _score_execution_policy_on_cached_squads(
    policy: dict[str, object],
) -> pd.DataFrame:
    rows = []
    weekly = backtest_weekly_results.copy()

    for record in weekly.itertuples(index=False):
        gw = int(record.gameweek)
        chip = _normalise_chip_name(record.chip)

        # v2.1 stores the permanent squad rather than the temporary FH squad.
        # Preserve the cached FH result instead of inventing a reconstruction.
        if chip == "free_hit":
            rows.append({
                "gameweek": gw,
                "chip": chip,
                "score": float(record.actual_score),
                "cached_score": float(record.actual_score),
                "changed_captain": False,
                "changed_vice": False,
                "captain_id": int(record.captain_id),
                "vice_captain_id": int(record.vice_captain_id),
                "scope": "cached_free_hit",
            })
            continue

        squad_ids = tuple(int(v) for v in record.squad_ids)
        context_key = (
            gw,
            tuple(sorted(squad_ids)),
        )
        cached_context = _EXECUTION_CONTEXT_CACHE.get(context_key)
        if cached_context is None:
            candidates = historical_candidate_snapshot(
                historical_backtest_table,
                gw,
                squad_ids,
            )
            projections = _execution_projection_for_squad(
                historical_backtest_table,
                gw,
                squad_ids,
            )
            actual = historical_backtest_table.loc[
                historical_backtest_table["gameweek"].eq(gw)
            ].copy()
            cached_context = (candidates, projections, actual)
            _EXECUTION_CONTEXT_CACHE[context_key] = cached_context
        candidates, projections, actual = cached_context

        decision = execution_policy_squad_gameweek_decision(
            squad_ids,
            gw,
            candidates,
            projections,
            policy=policy,
        )

        scoring_chip = (
            chip
            if chip in {"triple_captain", "bench_boost"}
            else None
        )
        realised = actual_score_from_decision(
            decision,
            actual,
            chip=scoring_chip,
        )
        score = float(realised["total_before_hits"]) - float(record.hit_cost)

        rows.append({
            "gameweek": gw,
            "chip": chip,
            "score": score,
            "cached_score": float(record.actual_score),
            "changed_captain": int(decision["captain_id"]) != int(record.captain_id),
            "changed_vice": int(decision["vice_captain_id"]) != int(record.vice_captain_id),
            "captain_id": int(decision["captain_id"]),
            "vice_captain_id": int(decision["vice_captain_id"]),
            "scope": "rescored_same_squad",
        })

    return pd.DataFrame(rows)


def validate_execution_policies_from_cache() -> dict[str, object]:
    if backtest_result is None or backtest_weekly_results.empty:
        if EXECUTION_REQUIRE_CACHED_SEQUENTIAL:
            raise RuntimeError(
                "Execution-policy validation needs the signed v2.1 sequential "
                "cache. Keep RUN_SEQUENTIAL_BACKTEST=False and confirm that "
                "data/sequential_backtest_v2_1.pkl.gz is present."
            )
        return {
            "approved": False,
            "reason": "No sequential cache available.",
            "selected_policy": {"name": "baseline"},
        }

    results = []
    detailed = {}

    for policy in _execution_candidate_policies():
        scored = _score_execution_policy_on_cached_squads(policy)
        detailed[policy["name"]] = scored

        first_half = scored.loc[scored["gameweek"].le(19)]
        second_half = scored.loc[scored["gameweek"].ge(20)]
        results.append({
            **policy,
            "total_score": float(scored["score"].sum()),
            "first_half_score": float(first_half["score"].sum()),
            "second_half_score": float(second_half["score"].sum()),
            "captain_changes": int(scored["changed_captain"].sum()),
            "vice_changes": int(scored["changed_vice"].sum()),
        })

    scorecard = pd.DataFrame(results)

    # Use a pre-derived baseline only for the subtraction itself.
    baseline_raw = scorecard.loc[
        scorecard["name"].eq("baseline")
    ].iloc[0]

    scorecard["gain_vs_baseline"] = (
        scorecard["total_score"] - float(baseline_raw["total_score"])
    )
    scorecard["first_half_gain"] = (
        scorecard["first_half_score"] - float(baseline_raw["first_half_score"])
    )
    scorecard["second_half_gain"] = (
        scorecard["second_half_score"] - float(baseline_raw["second_half_score"])
    )

    # IMPORTANT: re-select the baseline AFTER derived columns exist.  The old
    # v2.1.2 code kept the earlier Series object, so `selected = baseline` in
    # the no-eligible-candidate branch had no gain_vs_baseline fields.
    baseline = scorecard.loc[
        scorecard["name"].eq("baseline")
    ].iloc[0]

    cached_total = float(backtest_weekly_results["actual_score"].sum())
    reconstruction_error = float(baseline["total_score"] - cached_total)

    eligible = scorecard.loc[
        ~scorecard["name"].eq("baseline")
        & scorecard["gain_vs_baseline"].ge(float(EXECUTION_MIN_HISTORICAL_GAIN))
        & scorecard["first_half_gain"].ge(
            -float(EXECUTION_MAX_HALF_SEASON_DEGRADATION)
        )
        & scorecard["second_half_gain"].ge(
            -float(EXECUTION_MAX_HALF_SEASON_DEGRADATION)
        )
    ].copy()

    # Prefer the best score, but if several are essentially tied choose the
    # smaller intervention (narrower captain band, lower lineup bonus).
    if eligible.empty:
        selected = baseline
        approved = False
        reason = (
            "No execution overlay cleared the conservative historical gate."
        )
    else:
        best_gain = float(eligible["gain_vs_baseline"].max())
        near_best = eligible.loc[
            eligible["gain_vs_baseline"].ge(best_gain - 1.0)
        ].copy()
        near_best["_complexity"] = (
            near_best["lineup_minutes_weight"].astype(float)
            + near_best["captain_tie_band"].astype(float)
        )
        selected = near_best.sort_values(
            ["_complexity", "gain_vs_baseline"],
            ascending=[True, False],
        ).iloc[0]
        approved = (
            abs(reconstruction_error)
            <= float(EXECUTION_MAX_BASELINE_RECONSTRUCTION_ERROR)
        )
        reason = (
            "Approved: prior-season fixed-squad execution improved enough "
            "without a material half-season failure."
            if approved
            else (
                "Rejected because the cheap baseline reconstruction differed "
                "too much from the cached sequential score."
            )
        )

    selected_policy = {
        "name": str(selected["name"]),
        "lineup_minutes_weight": float(selected["lineup_minutes_weight"]),
        "captain_tie_band": float(selected["captain_tie_band"]),
        "captain_mode": str(selected["captain_mode"]),
        "vice_tie_band": float(selected["vice_tie_band"]),
    }

    return {
        "approved": bool(approved),
        "reason": reason,
        "baseline_cached_total": cached_total,
        "baseline_reconstructed_total": float(baseline["total_score"]),
        "baseline_reconstruction_error": reconstruction_error,
        "selected_policy": selected_policy,
        "selected_gain_vs_baseline": float(
            selected["gain_vs_baseline"]
        ),
        "selected_first_half_gain": float(
            selected["first_half_gain"]
        ),
        "selected_second_half_gain": float(
            selected["second_half_gain"]
        ),
        "scorecard": scorecard.sort_values(
            ["gain_vs_baseline", "total_score"],
            ascending=[False, False],
        ).reset_index(drop=True),
        "selected_weekly": detailed[str(selected["name"])],
        "baseline_weekly": detailed["baseline"],
    }


EXECUTION_BASELINE_POLICY = {
    "name": "baseline",
    "lineup_minutes_weight": 0.0,
    "captain_tie_band": 0.0,
    "captain_mode": "xp",
    "vice_tie_band": 0.0,
}

if RUN_EXECUTION_POLICY_VALIDATION:
    execution_policy_validation = validate_execution_policies_from_cache()
else:
    execution_policy_validation = {
        "approved": False,
        "reason": (
            "Execution-policy validation disabled for production. "
            "The prior v2.1.2 run reached the no-eligible-candidate branch, "
            "so no tested overlay cleared the historical activation gate."
        ),
        "selected_policy": EXECUTION_BASELINE_POLICY,
        "selected_gain_vs_baseline": 0.0,
        "selected_first_half_gain": 0.0,
        "selected_second_half_gain": 0.0,
        "scorecard": pd.DataFrame(),
    }

if EXECUTION_POLICY_MODE == "off":
    ACTIVE_EXECUTION_POLICY = dict(EXECUTION_BASELINE_POLICY)
    execution_policy_source = "forced off"
elif EXECUTION_POLICY_MODE == "force_best":
    ACTIVE_EXECUTION_POLICY = dict(
        execution_policy_validation.get(
            "selected_policy",
            EXECUTION_BASELINE_POLICY,
        )
    )
    execution_policy_source = "forced best cached candidate"
elif (
    EXECUTION_POLICY_MODE == "auto"
    and bool(execution_policy_validation.get("approved", False))
):
    ACTIVE_EXECUTION_POLICY = dict(
        execution_policy_validation["selected_policy"]
    )
    execution_policy_source = "approved prior-season cache validation"
else:
    ACTIVE_EXECUTION_POLICY = dict(EXECUTION_BASELINE_POLICY)
    execution_policy_source = "baseline after validation gate"

print("=" * 72)
print("EXECUTION POLICY VALIDATION")
print("=" * 72)
print("Active policy:", ACTIVE_EXECUTION_POLICY)
print("Source:", execution_policy_source)
print("Validation reason:", execution_policy_validation.get("reason"))
if "baseline_reconstruction_error" in execution_policy_validation:
    print(
        "Baseline reconstruction error:",
        f"{execution_policy_validation['baseline_reconstruction_error']:+.2f}",
    )
if "selected_gain_vs_baseline" in execution_policy_validation:
    print(
        "Selected cached-season gain:",
        f"{execution_policy_validation['selected_gain_vs_baseline']:+.1f}",
    )
    print(
        "Half-season gains:",
        f"{execution_policy_validation['selected_first_half_gain']:+.1f} / "
        f"{execution_policy_validation['selected_second_half_gain']:+.1f}",
    )

execution_scorecard = execution_policy_validation.get(
    "scorecard",
    pd.DataFrame(),
)
if isinstance(execution_scorecard, pd.DataFrame) and not execution_scorecard.empty:
    display(
        execution_scorecard[
            [
                "name",
                "lineup_minutes_weight",
                "captain_tie_band",
                "captain_mode",
                "total_score",
                "gain_vs_baseline",
                "first_half_gain",
                "second_half_gain",
                "captain_changes",
                "vice_changes",
            ]
        ].head(20).round(3)
    )

# Save the cheap validation artifact for auditability.
if RUN_EXECUTION_POLICY_VALIDATION:
    payload = {
        "notebook_release": NOTEBOOK_RELEASE,
        "core_project_version": PROJECT_VERSION,
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "approved": bool(execution_policy_validation.get("approved", False)),
        "reason": execution_policy_validation.get("reason"),
        "active_policy": ACTIVE_EXECUTION_POLICY,
        "policy_source": execution_policy_source,
        "baseline_cached_total": execution_policy_validation.get(
            "baseline_cached_total"
        ),
        "baseline_reconstructed_total": execution_policy_validation.get(
            "baseline_reconstructed_total"
        ),
        "baseline_reconstruction_error": execution_policy_validation.get(
            "baseline_reconstruction_error"
        ),
        "selected_gain_vs_baseline": execution_policy_validation.get(
            "selected_gain_vs_baseline"
        ),
        "selected_first_half_gain": execution_policy_validation.get(
            "selected_first_half_gain"
        ),
        "selected_second_half_gain": execution_policy_validation.get(
            "selected_second_half_gain"
        ),
        "scorecard": (
            execution_scorecard.replace({np.nan: None}).to_dict("records")
            if isinstance(execution_scorecard, pd.DataFrame)
            else []
        ),
    }
    with EXECUTION_POLICY_OUTPUT_PATH.open("w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, default=str)
    print("Saved execution-policy validation:", EXECUTION_POLICY_OUTPUT_PATH)


# Realistic one-/two-transfer hindsight ceiling -------------------------------

def _limited_transfer_hindsight_oracle(
    actual_players: pd.DataFrame,
    starting_squad_ids,
    *,
    budget: float,
    max_transfers: int,
) -> dict[str, object]:
    """
    Perfect-hindsight one-GW MILP with at most N transfers from a fixed squad.

    Transfers are implicitly position-preserving because the normal 2/5/5/3
    squad constraints remain enforced. Budget uses current market value rather
    than exact historical selling value, so this is an optimistic ceiling.
    """
    candidates = (
        actual_players.dropna(subset=["team_id", "position", "price"])
        .drop_duplicates("player_id")
        .reset_index(drop=True)
        .copy()
    )
    start_ids = {int(v) for v in starting_squad_ids}
    n = len(candidates)

    x0, y0, c0 = 0, n, 2 * n
    variables = 3 * n
    def x(i): return x0 + i
    def y(i): return y0 + i
    def c(i): return c0 + i

    points = pd.to_numeric(
        candidates["actual_points"], errors="coerce"
    ).fillna(0.0).to_numpy(dtype=float)

    objective = np.zeros(variables, dtype=float)
    for i in range(n):
        objective[y(i)] = -points[i]
        objective[c(i)] = -points[i]

    rows, lower, upper = [], [], []
    def add(terms, lo=-np.inf, hi=np.inf):
        rows.append(terms)
        lower.append(float(lo))
        upper.append(float(hi))

    add({x(i): 1.0 for i in range(n)}, 15, 15)

    for position, required in SQUAD_POSITION_REQUIREMENTS.items():
        indices = np.flatnonzero(
            candidates["position"].eq(position).to_numpy()
        )
        add({x(int(i)): 1.0 for i in indices}, required, required)

    add(
        {x(i): float(candidates.loc[i, "price"]) for i in range(n)},
        hi=float(budget),
    )

    for team_id in candidates["team_id"].dropna().unique():
        indices = np.flatnonzero(
            candidates["team_id"].eq(team_id).to_numpy()
        )
        add(
            {x(int(i)): 1.0 for i in indices},
            hi=float(MAX_PLAYERS_PER_CLUB),
        )

    # At most N transfers means at least 15-N original players retained.
    original_indices = [
        i for i in range(n)
        if int(candidates.loc[i, "player_id"]) in start_ids
    ]
    add(
        {x(int(i)): 1.0 for i in original_indices},
        lo=float(15 - int(max_transfers)),
    )

    add({y(i): 1.0 for i in range(n)}, 11, 11)
    for position, (minimum, maximum) in LINEUP_POSITION_BOUNDS.items():
        indices = np.flatnonzero(
            candidates["position"].eq(position).to_numpy()
        )
        add(
            {y(int(i)): 1.0 for i in indices},
            minimum,
            maximum,
        )

    add({c(i): 1.0 for i in range(n)}, 1, 1)
    for i in range(n):
        add({y(i): 1.0, x(i): -1.0}, hi=0.0)
        add({c(i): 1.0, y(i): -1.0}, hi=0.0)

    row_idx, col_idx, coeff = [], [], []
    for r, terms in enumerate(rows):
        for variable, value in terms.items():
            row_idx.append(r)
            col_idx.append(variable)
            coeff.append(float(value))

    matrix = coo_matrix(
        (coeff, (row_idx, col_idx)),
        shape=(len(rows), variables),
    ).tocsr()

    result = milp(
        c=objective,
        integrality=np.ones(variables, dtype=int),
        bounds=Bounds(np.zeros(variables), np.ones(variables)),
        constraints=LinearConstraint(
            matrix,
            np.asarray(lower),
            np.asarray(upper),
        ),
        options={"time_limit": 20},
    )
    if not result.success or result.x is None:
        raise RuntimeError(
            f"Limited-transfer hindsight MILP failed: {result.message}"
        )

    solution = result.x
    squad_ids = tuple(
        candidates.loc[
            [solution[x(i)] > 0.5 for i in range(n)],
            "player_id",
        ].astype(int)
    )
    incoming = sorted(set(squad_ids) - start_ids)
    outgoing = sorted(start_ids - set(squad_ids))

    return {
        "score": float(-result.fun),
        "transfers_used": int(len(incoming)),
        "incoming_ids": tuple(incoming),
        "outgoing_ids": tuple(outgoing),
        "budget": float(budget),
    }


def build_realistic_transfer_regret() -> pd.DataFrame:
    if not RUN_REALISTIC_TRANSFER_REGRET:
        return pd.DataFrame()
    if backtest_result is None or backtest_weekly_results.empty:
        if EXECUTION_REQUIRE_CACHED_SEQUENTIAL:
            raise RuntimeError(
                "Realistic transfer regret requires the signed sequential cache."
            )
        return pd.DataFrame()

    weekly = backtest_weekly_results.copy()
    if TRANSFER_REGRET_MAX_GAMEWEEKS is not None:
        weekly = weekly.head(int(TRANSFER_REGRET_MAX_GAMEWEEKS))

    rows = []
    previous_squad = None
    previous_bank = 0.0

    initial_squad_ids = tuple(
        backtest_result["initial_squad"]["player_id"].astype(int)
    )

    for record in weekly.itertuples(index=False):
        gw = int(record.gameweek)
        chip = _normalise_chip_name(record.chip)

        if gw == int(weekly["gameweek"].min()):
            starting_squad = initial_squad_ids
            bank_before = float(
                BACKTEST_INITIAL_BUDGET
                - backtest_result["initial_squad"]["price"].astype(float).sum()
            )
        else:
            starting_squad = tuple(int(v) for v in previous_squad)
            bank_before = float(previous_bank)

        # Skip unlimited-transfer and temporary-squad chip weeks. Ordinary,
        # TC and BB weeks retain meaningful one-/two-transfer comparisons.
        if chip in {"wildcard", "free_hit"}:
            rows.append({
                "gameweek": gw,
                "chip": chip,
                "roll_hindsight_score": np.nan,
                "one_transfer_hindsight_score": np.nan,
                "two_transfer_hindsight_score": np.nan,
                "one_transfer_gain": np.nan,
                "two_transfer_gain": np.nan,
                "scope": "skipped_unlimited_or_temporary_chip",
            })
            previous_squad = tuple(int(v) for v in record.squad_ids)
            previous_bank = float(record.bank_after)
            continue

        actual_universe = _actual_gameweek_player_table(
            historical_backtest_table,
            gw,
            starting_squad,
        )
        start_actual = actual_universe.loc[
            actual_universe["player_id"].astype(int).isin(
                set(int(v) for v in starting_squad)
            )
        ].copy()

        roll = _best_hindsight_lineup_from_squad(start_actual)
        roll_score = float(
            roll["base_points"] + roll["captain_points"]
        )

        # Approximate available budget using current market value + bank.
        budget = max(
            float(BACKTEST_INITIAL_BUDGET),
            float(start_actual["price"].astype(float).sum()) + bank_before,
        )

        one = _limited_transfer_hindsight_oracle(
            actual_universe,
            starting_squad,
            budget=budget,
            max_transfers=1,
        )
        two = _limited_transfer_hindsight_oracle(
            actual_universe,
            starting_squad,
            budget=budget,
            max_transfers=min(
                2,
                int(TRANSFER_REGRET_MAX_TRANSFERS),
            ),
        )

        rows.append({
            "gameweek": gw,
            "chip": chip,
            "roll_hindsight_score": roll_score,
            "one_transfer_hindsight_score": float(one["score"]),
            "two_transfer_hindsight_score": float(two["score"]),
            "one_transfer_gain": float(one["score"] - roll_score),
            "two_transfer_gain": float(two["score"] - roll_score),
            "one_transfer_used": int(one["transfers_used"]),
            "two_transfers_used": int(two["transfers_used"]),
            "budget_approx": budget,
            "scope": (
                "perfect hindsight, max 1/2 transfers, current market-value "
                "budget; execution also hindsight-perfect"
            ),
        })

        previous_squad = tuple(int(v) for v in record.squad_ids)
        previous_bank = float(record.bank_after)

    return pd.DataFrame(rows)


historical_realistic_transfer_regret = build_realistic_transfer_regret()

if not historical_realistic_transfer_regret.empty:
    valid_transfer_regret = historical_realistic_transfer_regret.loc[
        historical_realistic_transfer_regret["one_transfer_gain"].notna()
    ].copy()

    print("\nRealistic transfer-opportunity ceiling")
    display(
        valid_transfer_regret.sort_values(
            "two_transfer_gain",
            ascending=False,
        ).head(15).round(2)
    )

    if not valid_transfer_regret.empty:
        print(
            "Mean hindsight gain available with <=1 transfer:",
            f"{valid_transfer_regret['one_transfer_gain'].mean():.2f}",
        )
        print(
            "Mean hindsight gain available with <=2 transfers:",
            f"{valid_transfer_regret['two_transfer_gain'].mean():.2f}",
        )

    with TRANSFER_REGRET_OUTPUT_PATH.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            {
                "notebook_release": NOTEBOOK_RELEASE,
                "core_project_version": PROJECT_VERSION,
                "generated_at_utc": datetime.now(timezone.utc).isoformat(),
                "rows": (
                    historical_realistic_transfer_regret
                    .replace({np.nan: None})
                    .to_dict("records")
                ),
            },
            file,
            indent=2,
            default=str,
        )
    print("Saved realistic transfer regret:", TRANSFER_REGRET_OUTPUT_PATH)


### Why the production overlay is deliberately narrow

The transfer optimiser and player xP model remain unchanged. The overlay is
only allowed to resolve **marginal current-week decisions** differently:

- a small expected-minutes bonus can alter an XI choice only when projected
  points are already close;
- captaincy can use minutes, appearance probability or per-90 ceiling only
  among players inside a small xP band around the top forecast;
- vice-captain selection becomes explicitly availability-aware.

If no candidate clears the prior-season activation gate, `auto` falls back to
the original v2.1 execution logic.

The captain-confidence label is informational. A very-low-confidence captain
means the leading options are essentially tied by the model; it does not cause
a manual override.


# 11D. v2.2 production-alignment gates

The live manager must not apply a decision overlay that has not been evaluated historically. v2.2 therefore ships early-season shrinkage and structural/marginal transfer overrides **off by default**. The helpers below make the transfer-count robustness gate directly ablatable on the same sequential backtest.

In [ ]:
def production_alignment_status() -> pd.DataFrame:
    rows = [
        {
            "layer": "early_season_shrinkage",
            "production_active": bool(EARLY_SEASON_SHRINKAGE_ENABLED),
            "historically_validated_here": False,
            "status": (
                "OFF / aligned"
                if not EARLY_SEASON_SHRINKAGE_ENABLED
                else "UNVALIDATED - disable before production"
            ),
        },
        {
            "layer": "marginal_transfer_robustness",
            "production_active": bool(WEEKLY_TRANSFER_ROBUSTNESS_GATE),
            "historically_validated_here": bool(
                PRODUCTION_WRAPPER_VALIDATION_ENABLED
            ),
            "status": (
                "OFF / aligned"
                if not WEEKLY_TRANSFER_ROBUSTNESS_GATE
                else "VALIDATION REQUIRED"
            ),
        },
        {
            "layer": "same_count_structural_repair",
            "production_active": bool(WEEKLY_STRUCTURAL_REPAIR_GATE),
            "historically_validated_here": False,
            "status": (
                "OFF / aligned"
                if not WEEKLY_STRUCTURAL_REPAIR_GATE
                else "UNVALIDATED - disable before production"
            ),
        },
    ]
    return pd.DataFrame(rows)


def run_transfer_robustness_ablation(
    bundle: dict[str, object],
    profile_name: str = "rolling_6gw_hits",
) -> pd.DataFrame:
    """
    Compare the exact same fold/state machinery with and without the marginal
    extra-transfer gate. This isolates the wrapper instead of comparing two
    unrelated season paths from different configurations.
    """
    profile = WALK_FORWARD_POLICY_PROFILES[profile_name]
    results = []

    with activated_walk_forward_context(bundle, profile):
        strategy_forecast = bundle["holdout_expected_points"].copy()
        fold_price_prior = build_position_price_prior_table(
            bundle["development_points_data"]
        )
        strategy_forecast = apply_sparse_player_market_prior(
            strategy_forecast,
            prior_table=fold_price_prior,
        )
        backtest_table = build_historical_backtest_table(
            strategy_forecast
        )
        gameweeks = sorted(backtest_table["gameweek"].unique())
        if WALK_FORWARD_MAX_GAMEWEEKS is not None:
            gameweeks = gameweeks[:int(WALK_FORWARD_MAX_GAMEWEEKS)]

        original = bool(BACKTEST_APPLY_TRANSFER_ROBUSTNESS_GATE)
        try:
            for enabled in (False, True):
                globals()["BACKTEST_APPLY_TRANSFER_ROBUSTNESS_GATE"] = enabled
                run = run_sequential_fpl_backtest(
                    backtest_table,
                    gameweeks,
                )
                weekly = run["weekly_results"]
                results.append({
                    "holdout_season": bundle["holdout_season"],
                    "profile": profile_name,
                    "robustness_gate": enabled,
                    "actual_points": float(weekly["actual_score"].sum()),
                    "predicted_points": float(weekly["predicted_score"].sum()),
                    "hit_points": float(weekly["hit_cost"].sum()),
                    "transfers": int(weekly["transfers"].sum()),
                    "gate_applications": int(
                        weekly.get(
                            "transfer_robustness_applied",
                            pd.Series(False, index=weekly.index),
                        ).sum()
                    ),
                })
        finally:
            globals()["BACKTEST_APPLY_TRANSFER_ROBUSTNESS_GATE"] = original

    return pd.DataFrame(results)


production_alignment_audit = production_alignment_status()
display(production_alignment_audit)

_unvalidated_active = production_alignment_audit.loc[
    production_alignment_audit["production_active"]
    & production_alignment_audit["status"].str.contains(
        "UNVALIDATED|REQUIRED",
        regex=True,
    )
]
if not _unvalidated_active.empty and AUTONOMOUS_MODE:
    raise RuntimeError(
        "v2.2 production-alignment guard: an unvalidated live overlay is active. "
        "Disable it or run/record the corresponding historical ablation first."
    )

### Same-state hit validation

Set `HIT_COUNTERFACTUAL_VALIDATION_ENABLED=True` before a sequential/walk-forward run to record, at every paid-transfer decision, the no-hit optimum from the **same pre-deadline squad, bank, free-transfer state and forecasts**. The weekly output then contains projected horizon gain, projected current-GW gain and realised current-GW gain versus that no-hit counterfactual.

In [ ]:
def transfer_search_fidelity_ablation(
    current_squad_ids: list[int],
    candidates: pd.DataFrame,
    projections: pd.DataFrame,
    gameweeks_to_use: list[int],
    money_in_bank: float,
    free_transfers: int,
    selling_prices: dict[int, float] | None = None,
    *,
    include_reference: bool = True,
) -> pd.DataFrame:
    """
    Compare v2.2.6 fast/standard search with the v2.2.4 reference widths.

    This is deliberately opt-in: the reference profile is the expensive search
    we are trying not to run every week. Use sampled historical/live states to
    establish first-action fidelity before tightening profiles further.
    """
    profiles = ["fast", "standard"]
    if include_reference:
        profiles.append("reference")

    rows = []
    shared_decision_cache = {}
    shared_future_score_cache = {}

    for profile in profiles:
        result = run_transfer_search_profile(
            profile,
            current_squad_ids=current_squad_ids,
            candidates=candidates,
            projections=projections,
            gameweeks_to_use=gameweeks_to_use,
            money_in_bank=money_in_bank,
            free_transfers=free_transfers,
            current_selling_prices=selling_prices,
            shared_decision_cache=shared_decision_cache,
            shared_future_score_cache=shared_future_score_cache,
        )
        summary = result["search_summary"]
        rows.append({
            "profile": profile,
            "seconds": float(result["search_runtime_seconds"]),
            "objective": float(summary["best_objective"]),
            "first_action": tuple(summary["best_action"]),
            "first_action_gap": float(summary["first_action_gap"])
                if np.isfinite(summary["first_action_gap"]) else np.nan,
            "distinct_first_actions": int(summary["distinct_first_actions"]),
        })

    frame = pd.DataFrame(rows)
    if "reference" in set(frame["profile"]):
        reference = frame.loc[frame["profile"].eq("reference")].iloc[0]
        frame["same_first_action_as_reference"] = frame["first_action"].map(
            lambda action: tuple(action) == tuple(reference["first_action"])
        )
        frame["objective_loss_vs_reference"] = (
            float(reference["objective"]) - frame["objective"]
        )
    else:
        standard = frame.loc[frame["profile"].eq("standard")].iloc[0]
        frame["same_first_action_as_standard"] = frame["first_action"].map(
            lambda action: tuple(action) == tuple(standard["first_action"])
        )
        frame["objective_loss_vs_standard"] = (
            float(standard["objective"]) - frame["objective"]
        )
    return frame


# Backward-compatible alias for previous notebooks/scripts.
transfer_search_runtime_ablation = transfer_search_fidelity_ablation


# 12. Autonomous weekly manager

This is the production interface. In `AUTONOMOUS_MODE`, the notebook:

- reconciles the previous pending recommendation against public FPL picks after the deadline;
- loads squad, bank, purchase prices, free transfers and chip state from the local ledger;
- refuses stale live projections;
- uses the validated/cached transfer policy;
- uses calibrated xP in every downstream decision;
- applies the same structural chip gate used in historical validation;
- outputs exactly one immediate instruction;
- saves the recommendation as a pending ledger state for verification after the deadline.

Before GW1 the ledger is empty and the optimiser chooses the opening squad. After the team exists, set `FPL_TEAM_ID` once.


In [ ]:
# Weekly production state ----------------------------------------------------
#
# In autonomous mode these values are overwritten from the committed local
# ledger after GW1. Manual values remain available as a recovery/debug fallback.

WEEKLY_CURRENT_SQUAD = []
WEEKLY_MONEY_IN_BANK = 0.0
WEEKLY_FREE_TRANSFERS = 1
WEEKLY_PURCHASE_PRICES = {}
WEEKLY_SELLING_PRICES = {}
WEEKLY_AVAILABLE_CHIPS = {
    "triple_captain": True,
    "bench_boost": True,
    "free_hit": True,
    "wildcard": True,
}
WEEKLY_LAST_FREE_HIT_GAMEWEEK = None

WEEKLY_POLICY_PROFILES = {
    "greedy_1gw_no_hits": {
        "horizon": 1,
        "max_transfers_per_gameweek": 2,
        "allow_hits": False,
        "bank_aware": False,
        "min_gain_per_extra_transfer": 0.0,
    },
    "greedy_1gw_bank_aware": {
        "horizon": 1,
        "max_transfers_per_gameweek": 2,
        "allow_hits": False,
        "bank_aware": True,
        "min_gain_per_extra_transfer": BANK_AWARE_MIN_GAIN_PER_EXTRA_TRANSFER,
    },
    "rolling_3gw_no_hits": {
        "horizon": 3,
        "max_transfers_per_gameweek": 2,
        "allow_hits": False,
        "bank_aware": False,
        "min_gain_per_extra_transfer": 0.0,
    },
    "rolling_6gw_no_hits": {
        "horizon": 6,
        "max_transfers_per_gameweek": 2,
        "allow_hits": False,
        "bank_aware": False,
        "min_gain_per_extra_transfer": 0.0,
    },
    "rolling_6gw_hits": {
        "horizon": 6,
        "max_transfers_per_gameweek": 2,
        "allow_hits": True,
        "bank_aware": False,
        "min_gain_per_extra_transfer": 0.0,
    },
}

# Autonomous mode relies on FPL API availability/news. Manual overrides are
# disabled by default so the system can make the decision without human edits.
WEEKLY_CURRENT_GW_PROJECTION_MULTIPLIERS = {}
WEEKLY_HORIZON_PROJECTION_MULTIPLIERS = {}

WEEKLY_CHIP_EXCEPTIONAL_UPLIFT = {
    "triple_captain": 12.0,
    "bench_boost": 15.0,
    "free_hit": 15.0,
    "wildcard": 18.0,
}



In [ ]:
from contextlib import contextmanager
from datetime import datetime, timezone


def weekly_resolve_player_reference(
    reference,
    player_table: pd.DataFrame,
) -> int:
    """Resolve one official player ID or exact current player name."""
    valid_ids = set(
        pd.to_numeric(
            player_table["id"],
            errors="coerce",
        ).dropna().astype(int)
    )

    if isinstance(reference, (int, np.integer)):
        player_id = int(reference)
        if player_id not in valid_ids:
            raise ValueError(
                f"Player ID {player_id} is not in the current FPL data."
            )
        return player_id

    text = str(reference).strip()
    if text.isdigit():
        player_id = int(text)
        if player_id in valid_ids:
            return player_id

    normalised = text.casefold()
    matches = player_table.loc[
        player_table["web_name"].fillna("").astype(str)
        .str.strip().str.casefold().eq(normalised)
        |
        player_table["full_name"].fillna("").astype(str)
        .str.strip().str.casefold().eq(normalised)
    ]

    if matches.empty:
        raise ValueError(
            f"Could not resolve player reference {reference!r}. "
            "Use an official player ID or an exact current player name."
        )
    if matches["id"].nunique() != 1:
        options = (
            matches[
                ["id", "web_name", "team_short_name", "position"]
            ]
            .drop_duplicates()
            .to_dict("records")
        )
        raise ValueError(
            f"Player reference {reference!r} is ambiguous: {options}"
        )
    return int(matches.iloc[0]["id"])


def weekly_resolve_squad(
    squad_references,
    player_table: pd.DataFrame,
    next_gameweek: int,
) -> list[int]:
    """Resolve and validate the configured live squad."""
    references = list(squad_references)
    if not references:
        if int(next_gameweek) == 1:
            return []
        raise ValueError(
            "WEEKLY_CURRENT_SQUAD must contain the 15 current players "
            "after Gameweek 1."
        )

    squad_ids = [
        weekly_resolve_player_reference(reference, player_table)
        for reference in references
    ]
    if len(squad_ids) != 15 or len(set(squad_ids)) != 15:
        raise ValueError(
            "WEEKLY_CURRENT_SQUAD must resolve to exactly 15 unique players."
        )
    return squad_ids


def weekly_resolve_price_mapping(
    configured_mapping: dict,
    player_table: pd.DataFrame,
) -> dict[int, float]:
    """Resolve a name/ID keyed price mapping to official player IDs."""
    resolved = {}
    for reference, price in dict(configured_mapping).items():
        player_id = weekly_resolve_player_reference(
            reference,
            player_table,
        )
        numeric_price = float(price)
        if not np.isfinite(numeric_price) or numeric_price <= 0:
            raise ValueError(
                f"Invalid price {price!r} for player {reference!r}."
            )
        resolved[player_id] = numeric_price
    return resolved


def weekly_calculate_selling_price(
    purchase_price: float,
    current_price: float,
) -> float:
    """Apply FPL's half-profit selling-price rule in integer tenths."""
    purchase_tenths = price_to_tenths(purchase_price)
    current_tenths = price_to_tenths(current_price)

    if current_tenths <= purchase_tenths:
        selling_tenths = current_tenths
    else:
        selling_tenths = (
            purchase_tenths
            + (current_tenths - purchase_tenths) // 2
        )
    return tenths_to_price(selling_tenths)


def weekly_build_selling_prices(
    squad_ids: list[int],
    candidates: pd.DataFrame,
    purchase_prices: dict[int, float],
    explicit_selling_prices: dict[int, float],
) -> tuple[dict[int, float], list[str]]:
    """Return current selling prices and any assumptions that were required."""
    market_lookup = (
        candidates
        .set_index("player_id")["price"]
        .astype(float)
        .to_dict()
    )

    selling_prices = {}
    warnings = []
    for player_id in squad_ids:
        current_price = float(market_lookup[player_id])

        if player_id in explicit_selling_prices:
            selling_price = float(
                explicit_selling_prices[player_id]
            )
        elif player_id in purchase_prices:
            selling_price = weekly_calculate_selling_price(
                purchase_prices[player_id],
                current_price,
            )
        else:
            selling_price = current_price
            warnings.append(
                f"No purchase/selling price was supplied for player ID "
                f"{player_id}; current market price was used."
            )

        selling_prices[player_id] = selling_price

    return selling_prices, warnings


def weekly_apply_projection_multipliers(
    gameweek_projections: pd.DataFrame,
    player_table: pd.DataFrame,
    next_gameweek: int,
    current_gameweek_multipliers: dict,
    horizon_multipliers: dict,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Apply transparent manual late-news adjustments."""
    adjusted = gameweek_projections.copy()
    audit_rows = []

    def apply_one(reference, factor, gameweek_scope):
        player_id = weekly_resolve_player_reference(
            reference,
            player_table,
        )
        factor = float(factor)
        if not np.isfinite(factor) or not 0.0 <= factor <= 1.0:
            raise ValueError(
                f"Projection multiplier for {reference!r} must be "
                "between 0 and 1."
            )

        mask = adjusted["player_id"].eq(player_id)
        if gameweek_scope is not None:
            mask &= adjusted["gameweek"].eq(
                int(gameweek_scope)
            )

        affected_rows = int(mask.sum())
        if affected_rows == 0:
            raise ValueError(
                f"Projection override for {reference!r} matched no "
                "player–Gameweek rows."
            )

        old_appearance = pd.to_numeric(
            adjusted.loc[mask, "appearance_probability"],
            errors="coerce",
        ).fillna(0.0)

        for column in [
            "expected_points",
            "expected_minutes",
            "projected_goals",
            "projected_assists",
        ] + [
            column
            for column in XP_COMPONENT_COLUMNS
            if column in adjusted.columns
        ]:
            if column in adjusted.columns:
                adjusted.loc[mask, column] = (
                    pd.to_numeric(
                        adjusted.loc[mask, column],
                        errors="coerce",
                    ).fillna(0.0)
                    * factor
                )

        adjusted_appearance = (
            old_appearance * factor
        ).clip(0.0, 1.0)
        adjusted.loc[
            mask,
            "appearance_probability",
        ] = adjusted_appearance
        adjusted.loc[
            mask,
            "no_appearance_probability",
        ] = 1.0 - adjusted_appearance

        player_row = player_table.loc[
            player_table["id"].eq(player_id)
        ].iloc[0]
        audit_rows.append({
            "player_id": player_id,
            "player": player_row["web_name"],
            "scope": (
                f"GW{int(gameweek_scope)}"
                if gameweek_scope is not None
                else "Full horizon"
            ),
            "multiplier": factor,
            "affected_rows": affected_rows,
        })

    for reference, factor in dict(
        horizon_multipliers
    ).items():
        apply_one(reference, factor, None)

    for reference, factor in dict(
        current_gameweek_multipliers
    ).items():
        apply_one(reference, factor, int(next_gameweek))

    return adjusted, pd.DataFrame(audit_rows)


def weekly_projection_freshness_audit(
    in_memory_players: pd.DataFrame,
    in_memory_fixtures: pd.DataFrame,
    in_memory_gameweeks: pd.DataFrame,
    in_memory_gameweek_xp: pd.DataFrame,
) -> dict[str, object]:
    """Compare the live API with the data used to create the projections."""
    latest_bootstrap = get_fpl_data("bootstrap-static/")
    latest_fixtures_raw = get_fpl_data("fixtures/")

    latest_events = pd.DataFrame(
        latest_bootstrap["events"]
    )
    latest_next_gameweek = get_first_planning_gameweek(
        latest_events,
    )

    projected_gameweeks = sorted(
        pd.to_numeric(
            in_memory_gameweek_xp["gameweek"],
            errors="coerce",
        ).dropna().astype(int).unique()
    )
    projected_next_gameweek = (
        int(projected_gameweeks[0])
        if projected_gameweeks
        else None
    )

    latest_players = pd.DataFrame(
        latest_bootstrap["elements"]
    )
    latest_player_state = latest_players[
        [
            "id",
            "now_cost",
            "status",
            "chance_of_playing_next_round",
            "news",
        ]
    ].copy()
    latest_player_state["latest_price"] = (
        pd.to_numeric(
            latest_player_state["now_cost"],
            errors="coerce",
        ) / 10.0
    )

    memory_player_state = in_memory_players[
        [
            "id",
            "price",
            "status",
            "chance_of_playing_next_round",
            "news",
        ]
    ].copy().rename(columns={
        "price": "memory_price",
        "status": "memory_status",
        "chance_of_playing_next_round": "memory_chance",
        "news": "memory_news",
    })

    player_comparison = latest_player_state.merge(
        memory_player_state,
        on="id",
        how="outer",
    )

    price_changes = (
        pd.to_numeric(
            player_comparison["latest_price"],
            errors="coerce",
        ).round(1)
        !=
        pd.to_numeric(
            player_comparison["memory_price"],
            errors="coerce",
        ).round(1)
    )
    status_changes = (
        player_comparison["status"].fillna("").astype(str)
        !=
        player_comparison["memory_status"].fillna("").astype(str)
    )
    chance_changes = (
        pd.to_numeric(
            player_comparison[
                "chance_of_playing_next_round"
            ],
            errors="coerce",
        ).fillna(-1)
        !=
        pd.to_numeric(
            player_comparison["memory_chance"],
            errors="coerce",
        ).fillna(-1)
    )
    news_changes = (
        player_comparison["news"].fillna("").astype(str).str.strip()
        !=
        player_comparison["memory_news"].fillna("").astype(str).str.strip()
    )
    player_changed_mask = (
        price_changes
        | status_changes
        | chance_changes
        | news_changes
    )
    changed_player_rows = int(player_changed_mask.sum())

    player_name_lookup = (
        in_memory_players.set_index("id")["web_name"].astype(str).to_dict()
        if "web_name" in in_memory_players.columns else {}
    )
    changed_player_details = []
    for row_index in player_comparison.index[player_changed_mask]:
        row = player_comparison.loc[row_index]
        player_id = row.get("id")
        changed_fields = []
        if bool(price_changes.loc[row_index]): changed_fields.append("price")
        if bool(status_changes.loc[row_index]): changed_fields.append("status")
        if bool(chance_changes.loc[row_index]): changed_fields.append("chance")
        if bool(news_changes.loc[row_index]): changed_fields.append("news")
        changed_player_details.append({
            "player_id": int(player_id) if pd.notna(player_id) else None,
            "player": player_name_lookup.get(int(player_id), str(player_id)) if pd.notna(player_id) else None,
            "changed_fields": changed_fields,
            "memory_price": row.get("memory_price"),
            "latest_price": row.get("latest_price"),
            "memory_status": row.get("memory_status"),
            "latest_status": row.get("status"),
            "memory_chance": row.get("memory_chance"),
            "latest_chance": row.get("chance_of_playing_next_round"),
            "memory_news": row.get("memory_news"),
            "latest_news": row.get("news"),
        })

    latest_fixture_state = pd.DataFrame(
        latest_fixtures_raw
    )[
        ["id", "event", "team_h", "team_a", "kickoff_time"]
    ].copy()
    latest_fixture_state["kickoff_time"] = pd.to_datetime(
        latest_fixture_state["kickoff_time"],
        utc=True,
        errors="coerce",
    )

    memory_fixture_state = in_memory_fixtures[
        ["id", "event", "team_h", "team_a", "kickoff_time"]
    ].copy()
    memory_fixture_state["kickoff_time"] = pd.to_datetime(
        memory_fixture_state["kickoff_time"],
        utc=True,
        errors="coerce",
    )

    fixture_comparison = latest_fixture_state.merge(
        memory_fixture_state,
        on="id",
        how="outer",
        suffixes=("_latest", "_memory"),
    )
    fixture_changed = (
        fixture_comparison["event_latest"].fillna(-1)
        != fixture_comparison["event_memory"].fillna(-1)
    ) | (
        fixture_comparison["team_h_latest"].fillna(-1)
        != fixture_comparison["team_h_memory"].fillna(-1)
    ) | (
        fixture_comparison["team_a_latest"].fillna(-1)
        != fixture_comparison["team_a_memory"].fillna(-1)
    ) | (
        fixture_comparison["kickoff_time_latest"].fillna(
            pd.Timestamp("1970-01-01", tz="UTC")
        )
        !=
        fixture_comparison["kickoff_time_memory"].fillna(
            pd.Timestamp("1970-01-01", tz="UTC")
        )
    )
    changed_fixture_rows = int(fixture_changed.sum())
    changed_fixture_details = (
        fixture_comparison.loc[fixture_changed]
        .replace({np.nan: None})
        .to_dict("records")
    )

    is_fresh = (
        latest_next_gameweek == projected_next_gameweek
        and changed_player_rows == 0
        and changed_fixture_rows == 0
    )

    return {
        "checked_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "latest_next_gameweek": latest_next_gameweek,
        "projected_next_gameweek": projected_next_gameweek,
        "changed_player_rows": changed_player_rows,
        "changed_fixture_rows": changed_fixture_rows,
        "changed_players": changed_player_details,
        "changed_fixtures": changed_fixture_details,
        "is_fresh": bool(is_fresh),
    }


_WEEKLY_MISSING = object()


@contextmanager
def weekly_runtime_context(
    policy_profile: dict[str, object],
    available_chips: dict[str, bool],
    last_free_hit_gameweek,
):
    """Temporarily activate weekly policy and chip settings."""
    replacements = {
        "MAX_TRANSFERS_PER_GAMEWEEK": int(
            policy_profile["max_transfers_per_gameweek"]
        ),
        "MAX_BANKED_FREE_TRANSFERS": 5,
        "CURRENT_AVAILABLE_CHIPS": dict(available_chips),
        "LAST_FREE_HIT_GAMEWEEK": last_free_hit_gameweek,
    }
    originals = {
        key: globals().get(key, _WEEKLY_MISSING)
        for key in replacements
    }
    globals().update(replacements)

    try:
        yield
    finally:
        for key, value in originals.items():
            if value is _WEEKLY_MISSING:
                globals().pop(key, None)
            else:
                globals()[key] = value


In [ ]:
def _json_safe_int_dict(mapping: dict[int, float]) -> dict[str, float]:
    return {str(int(k)): float(v) for k, v in mapping.items()}


def load_autonomous_ledger() -> dict:
    if not AUTONOMOUS_STATE_PATH.exists():
        return {"version": 2, "entry_id": FPL_TEAM_ID, "committed": None, "pending": None}
    ledger = json.loads(AUTONOMOUS_STATE_PATH.read_text(encoding="utf-8"))
    if int(ledger.get("version", 0)) != 2:
        raise RuntimeError("Autonomous ledger version mismatch. Move the old ledger aside and initialise v2.")
    return ledger


def save_autonomous_ledger(ledger: dict) -> None:
    AUTONOMOUS_STATE_PATH.write_text(json.dumps(ledger, indent=2), encoding="utf-8")


def _chip_api_name_to_internal(name: str | None) -> str | None:
    mapping = {
        "3xc": "triple_captain",
        "bboost": "bench_boost",
        "freehit": "free_hit",
        "wildcard": "wildcard",
    }
    return mapping.get(str(name).lower()) if name is not None else None


def autonomous_latest_completed_gameweek(gameweeks_table: pd.DataFrame) -> int | None:
    """Resolve the latest deadline whose manager state is already public."""
    candidates = []
    finished = gameweeks_table.loc[
        gameweeks_table.get("finished", False).fillna(False),
        "id",
    ]
    if not finished.empty:
        candidates.append(int(finished.max()))

    # FPL can publish manager history/picks before events.finished propagates.
    # For autonomous state reconciliation, published manager state is enough.
    if FPL_TEAM_ID is not None:
        try:
            history = get_fpl_data(f"entry/{int(FPL_TEAM_ID)}/history/")
            events = [
                int(row.get("event"))
                for row in history.get("current", [])
                if row.get("event") is not None
            ]
            if events:
                candidates.append(max(events))
        except Exception:
            # The later public-state call remains authoritative and will raise
            # under strict reconciliation if manager data is genuinely missing.
            pass

    return max(candidates) if candidates else None


def autonomous_public_state(entry_id: int, completed_gw: int | None) -> dict:
    result = {
        "entry": get_fpl_data(f"entry/{int(entry_id)}/"),
        "history": get_fpl_data(f"entry/{int(entry_id)}/history/"),
        "transfers": get_fpl_data(f"entry/{int(entry_id)}/transfers/"),
        "picks": None,
    }
    if completed_gw is not None:
        result["picks"] = get_fpl_data(
            f"entry/{int(entry_id)}/event/{int(completed_gw)}/picks/"
        )
    return result


def chips_available_from_public_history(history_payload: dict, next_gameweek: int) -> tuple[dict[str, bool], int | None]:
    half_start = 1 if int(next_gameweek) <= 19 else 20
    half_end = 19 if int(next_gameweek) <= 19 else 38
    used = set()
    last_fh = None
    for row in history_payload.get("chips", []):
        event = int(row.get("event", -1))
        internal = _chip_api_name_to_internal(row.get("name"))
        if internal == "free_hit" and event < int(next_gameweek):
            last_fh = event if last_fh is None else max(last_fh, event)
        if internal and half_start <= event <= half_end:
            used.add(internal)
    available = {
        chip: chip not in used
        for chip in ["triple_captain", "bench_boost", "free_hit", "wildcard"]
    }
    return available, last_fh


def infer_free_transfers_from_public_history(history_payload: dict, next_gameweek: int) -> int:
    """Reconstruct banked FTs from completed deadlines under 2026/27 rules."""
    ft = 1
    chips_by_event = {
        int(row.get("event")): _chip_api_name_to_internal(row.get("name"))
        for row in history_payload.get("chips", [])
    }
    current_rows = {int(row.get("event")): row for row in history_payload.get("current", [])}
    # GW1 creates the normal single FT for GW2. Process completed GW2+ deadlines.
    for event in range(2, int(next_gameweek)):
        row = current_rows.get(event, {})
        transfers = int(row.get("event_transfers", 0) or 0)
        chip = chips_by_event.get(event)
        if chip in {"wildcard", "free_hit"}:
            ft = preserve_free_transfers_after_unlimited_chip(ft)
        else:
            ft = update_free_transfers(ft, transfers)
    return int(max(1, min(MAX_BANKED_FREE_TRANSFERS, ft)))


def bootstrap_purchase_prices_from_public(
    squad_ids: list[int],
    transfers_payload: list[dict],
    player_table: pd.DataFrame,
) -> tuple[dict[int, float], list[str]]:
    """Best-effort mid-season bootstrap. Exact autonomous ledgers avoid this path."""
    current_price = player_table.set_index("id")["price"].astype(float).to_dict()
    incoming = {}
    for transfer in sorted(transfers_payload, key=lambda r: str(r.get("time", ""))):
        pid = transfer.get("element_in")
        price = transfer.get("element_in_cost")
        if pid is not None and price is not None:
            incoming[int(pid)] = float(price) / 10.0
    prices = {}
    warnings_out = []
    for pid in squad_ids:
        if int(pid) in incoming:
            prices[int(pid)] = incoming[int(pid)]
        else:
            prices[int(pid)] = float(current_price[int(pid)])
            warnings_out.append(
                f"Purchase price for player {pid} was bootstrapped from current market price."
            )
    return prices, warnings_out


def reconcile_committed_state_to_public(
    committed: dict | None,
    public: dict,
    player_table: pd.DataFrame,
    completed_gw: int,
    next_gw: int,
) -> tuple[dict, list[str]]:
    """Build the next committed ledger state from what the manager actually did."""
    picks = public.get("picks") or {}
    active_chip = _chip_api_name_to_internal(picks.get("active_chip"))
    public_squad = sorted(
        int(row["element"])
        for row in picks.get("picks", [])
        if row.get("element") is not None
    )
    warnings_out = []

    old_purchase = {}
    if committed is not None:
        old_purchase = {
            int(k): float(v)
            for k, v in committed.get("purchase_prices", {}).items()
        }

    if active_chip == "free_hit":
        # Completed-GW public picks are the temporary FH squad. The permanent
        # squad and purchase prices therefore remain the pre-deadline ones.
        if committed is None:
            raise RuntimeError(
                "Cannot reconstruct the permanent squad from a historical Free Hit "
                "without an existing committed ledger."
            )
        permanent_squad = sorted(int(v) for v in committed["squad_ids"])
        purchase_prices = {
            pid: old_purchase[pid]
            for pid in permanent_squad
            if pid in old_purchase
        }
        if len(purchase_prices) != len(permanent_squad):
            raise RuntimeError(
                "Permanent Free Hit squad is missing stored purchase prices."
            )
        bank = float(committed.get("bank", 0.0))
    else:
        if len(public_squad) != 15:
            raise RuntimeError(
                f"Expected 15 public players after GW{completed_gw}, "
                f"found {len(public_squad)}."
            )
        permanent_squad = public_squad
        new_player_ids = [pid for pid in permanent_squad if pid not in old_purchase]
        inferred_new_prices, bootstrap_warnings = bootstrap_purchase_prices_from_public(
            new_player_ids,
            public.get("transfers", []),
            player_table,
        )
        warnings_out.extend(bootstrap_warnings)
        purchase_prices = {
            pid: (
                old_purchase[pid]
                if pid in old_purchase
                else float(inferred_new_prices[pid])
            )
            for pid in permanent_squad
        }
        public_bank = (picks.get("entry_history") or {}).get("bank")
        if public_bank is not None:
            bank = float(public_bank) / 10.0
        elif committed is not None:
            bank = float(committed.get("bank", 0.0))
            warnings_out.append(
                "Public bank was unavailable; preserved the previous committed bank."
            )
        else:
            bank = 0.0
            warnings_out.append(
                "Public bank was unavailable during bootstrap; assumed £0.0m."
            )

    chips_available, last_fh = chips_available_from_public_history(
        public["history"],
        next_gw,
    )
    free_transfers = infer_free_transfers_from_public_history(
        public["history"],
        next_gw,
    )

    return {
        "through_gameweek": int(completed_gw),
        "squad_ids": permanent_squad,
        "purchase_prices": _json_safe_int_dict(purchase_prices),
        "bank": float(bank),
        "free_transfers": int(free_transfers),
        "chips_available": chips_available,
        "last_free_hit_gameweek": last_fh,
    }, warnings_out


def autonomous_resolve_weekly_state(
    player_table: pd.DataFrame,
    gameweeks_table: pd.DataFrame,
) -> tuple[dict, list[str]]:
    """Reconcile the ledger to public FPL reality, then prepare the next deadline."""
    global WEEKLY_CURRENT_SQUAD, WEEKLY_MONEY_IN_BANK, WEEKLY_FREE_TRANSFERS
    global WEEKLY_PURCHASE_PRICES, WEEKLY_SELLING_PRICES
    global WEEKLY_AVAILABLE_CHIPS, WEEKLY_LAST_FREE_HIT_GAMEWEEK

    warnings_out = []
    ledger = load_autonomous_ledger()
    next_rows = gameweeks_table.loc[gameweeks_table.get("is_next", False).fillna(False)]
    next_gw = int(next_rows.iloc[0]["id"]) if not next_rows.empty else int(
        gameweeks_table.loc[~gameweeks_table.get("finished", False).fillna(False), "id"].min()
    )
    completed_gw = autonomous_latest_completed_gameweek(gameweeks_table)

    if next_gw == 1 and ledger.get("committed") is None:
        WEEKLY_CURRENT_SQUAD = []
        WEEKLY_MONEY_IN_BANK = 0.0
        WEEKLY_FREE_TRANSFERS = 1
        WEEKLY_PURCHASE_PRICES = {}
        WEEKLY_SELLING_PRICES = {}
        WEEKLY_AVAILABLE_CHIPS = {k: True for k in ["triple_captain", "bench_boost", "free_hit", "wildcard"]}
        WEEKLY_LAST_FREE_HIT_GAMEWEEK = None
        return ledger, warnings_out

    if FPL_TEAM_ID is None:
        raise RuntimeError(
            "Set FPL_TEAM_ID once your FPL entry exists. Autonomous in-season "
            "state cannot be verified without it."
        )

    ledger["entry_id"] = int(FPL_TEAM_ID)
    public = autonomous_public_state(int(FPL_TEAM_ID), completed_gw)
    picks = public.get("picks") or {}
    public_squad = sorted(
        int(row["element"])
        for row in picks.get("picks", [])
        if row.get("element") is not None
    )
    active_chip = _chip_api_name_to_internal(picks.get("active_chip"))

    pending = ledger.get("pending")
    committed = ledger.get("committed")
    pending_due = bool(
        pending
        and completed_gw is not None
        and int(pending.get("gameweek", 999)) <= int(completed_gw)
    )

    if pending_due:
        expected_verification = sorted(
            int(v) for v in pending.get("verification_squad_ids", [])
        )
        squad_matches = (not expected_verification) or public_squad == expected_verification
        chip_matches = (
            active_chip == pending.get("chip")
            or (active_chip is None and pending.get("chip") is None)
        )
        if not (squad_matches and chip_matches):
            warnings_out.append(
                "Previous autonomous recommendation was not executed exactly. "
                "Reconciled to public FPL reality instead of assuming the pending plan. "
                f"Expected squad={expected_verification}, public squad={public_squad}, "
                f"expected chip={pending.get('chip')}, public chip={active_chip}."
            )

    committed_through = int(committed.get("through_gameweek", 0)) if committed else 0
    needs_public_reconciliation = bool(
        completed_gw is not None
        and (
            committed is None
            or int(completed_gw) > committed_through
            or pending_due
        )
    )

    if needs_public_reconciliation:
        try:
            committed, reconcile_warnings = reconcile_committed_state_to_public(
                committed,
                public,
                player_table,
                int(completed_gw),
                next_gw,
            )
            warnings_out.extend(reconcile_warnings)
            ledger["committed"] = committed
            if pending_due:
                ledger["pending"] = None
            save_autonomous_ledger(ledger)
        except Exception as exc:
            message = (
                "Could not safely reconcile the autonomous ledger to public FPL "
                f"state after GW{completed_gw}: {exc}"
            )
            if AUTONOMOUS_STRICT_RECONCILIATION:
                raise RuntimeError(message) from exc
            warnings_out.append(message)

    committed = ledger.get("committed")

    # The second chip set becomes available for GW20. Public history remains
    # authoritative, so refresh the half-season availability on the boundary.
    if (
        committed is not None
        and int(next_gw) >= 20
        and int(committed.get("through_gameweek", 0)) <= 19
    ):
        public_available, public_last_fh = chips_available_from_public_history(
            public["history"], next_gw
        )
        committed["chips_available"] = public_available
        committed["last_free_hit_gameweek"] = public_last_fh
        ledger["committed"] = committed
        save_autonomous_ledger(ledger)

    if committed is None:
        if not public_squad:
            raise RuntimeError("Unable to bootstrap autonomous state from public FPL picks.")
        committed, bootstrap_warnings = reconcile_committed_state_to_public(
            None,
            public,
            player_table,
            int(completed_gw or 0),
            next_gw,
        )
        warnings_out.extend(bootstrap_warnings)
        ledger["committed"] = committed
        save_autonomous_ledger(ledger)

    WEEKLY_CURRENT_SQUAD = [int(v) for v in committed["squad_ids"]]
    WEEKLY_MONEY_IN_BANK = float(committed["bank"])
    WEEKLY_FREE_TRANSFERS = int(committed["free_transfers"])
    WEEKLY_PURCHASE_PRICES = {
        int(k): float(v) for k, v in committed["purchase_prices"].items()
    }
    WEEKLY_SELLING_PRICES = {}
    WEEKLY_AVAILABLE_CHIPS = {
        k: bool(v) for k, v in committed["chips_available"].items()
    }
    WEEKLY_LAST_FREE_HIT_GAMEWEEK = committed.get("last_free_hit_gameweek")
    return ledger, warnings_out


def autonomous_save_pending_decision(
    ledger: dict,
    *,
    gameweek: int,
    current_squad_ids: list[int],
    current_purchase_prices: dict[int, float],
    current_bank: float,
    current_free_transfers: int,
    normal_transfer_pairs: list[tuple[int, int]],
    normal_squad_ids_after: list[int],
    normal_bank_after: float,
    normal_free_transfers_after: int,
    final_squad: pd.DataFrame,
    selected_chip: str | None,
    candidates: pd.DataFrame,
    chips_available: dict[str, bool],
    last_free_hit_gameweek: int | None,
) -> dict:
    market = candidates.set_index("player_id")["price"].astype(float).to_dict()
    purchase = {int(k): float(v) for k, v in current_purchase_prices.items()}

    if int(gameweek) == 1 and not current_squad_ids:
        permanent = sorted(final_squad["player_id"].astype(int).tolist())
        purchase = {pid: float(market[pid]) for pid in permanent}
        bank_after = float(INITIAL_SQUAD_BUDGET - sum(market[pid] for pid in permanent))
        ft_after = 1
        verification = permanent
    elif selected_chip == "free_hit":
        permanent = sorted(int(v) for v in current_squad_ids)
        bank_after = float(current_bank)
        ft_after = preserve_free_transfers_after_unlimited_chip(current_free_transfers)
        verification = sorted(final_squad["player_id"].astype(int).tolist())
    elif selected_chip == "wildcard":
        permanent = sorted(final_squad["player_id"].astype(int).tolist())
        purchase = {pid: float(market[pid]) for pid in permanent}
        selling_now, _ = weekly_build_selling_prices(
            current_squad_ids, candidates, current_purchase_prices, {}
        )
        bank_after = float(current_bank + sum(selling_now.values()) - sum(market[pid] for pid in permanent))
        ft_after = preserve_free_transfers_after_unlimited_chip(current_free_transfers)
        verification = permanent
    else:
        permanent = sorted(int(v) for v in normal_squad_ids_after)
        for outgoing, incoming in normal_transfer_pairs:
            purchase.pop(int(outgoing), None)
            purchase[int(incoming)] = float(market[int(incoming)])
        bank_after = float(normal_bank_after)
        ft_after = int(normal_free_transfers_after)
        verification = permanent

    chip_state = dict(chips_available)
    if selected_chip is not None:
        chip_state[selected_chip] = False
    last_fh_after = int(gameweek) if selected_chip == "free_hit" else last_free_hit_gameweek

    pending = {
        "gameweek": int(gameweek),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "chip": selected_chip,
        "verification_squad_ids": verification,
        "permanent_squad_ids_after": permanent,
        "purchase_prices_after": _json_safe_int_dict(purchase),
        "bank_after": bank_after,
        "free_transfers_after": ft_after,
        "chips_available_after": chip_state,
        "last_free_hit_gameweek_after": last_fh_after,
    }
    ledger["pending"] = pending
    save_autonomous_ledger(ledger)
    return pending



def weekly_attach_exact_raw_action_subsets(
    transfer_result: dict[str, object],
    current_squad_ids: list[int],
    candidates: pd.DataFrame,
    projections: pd.DataFrame,
    gameweeks_to_use: list[int],
    money_in_bank: float,
    free_transfers: int,
    current_selling_prices: dict[int, float],
    allow_hits: bool,
) -> dict[str, object]:
    """
    Re-optimise the full horizon for each one-transfer subset of the raw
    optimal current double transfer.

    MAX_TRANSFERS_PER_GAMEWEEK is 2 in production, so this requires at most
    two extra rolling-horizon solves and gives the robustness gate the exact
    comparison it needs.
    """
    result = dict(transfer_result)
    result["robustness_comparator_states"] = []
    result["robustness_subset_audit"] = []
    result["robustness_subset_attempted_count"] = 0
    result["robustness_subset_legal_count"] = 0
    result["robustness_subset_illegal_count"] = 0
    result["robustness_subset_runtime_seconds"] = 0.0
    result["robustness_subset_skipped_reason"] = None

    # These re-solves exist solely to support the marginal transfer-count
    # robustness gate. When that gate is OFF, running them cannot affect the
    # selected action and only wastes several rolling-horizon solves.
    if not WEEKLY_TRANSFER_ROBUSTNESS_GATE:
        result["robustness_subset_skipped_reason"] = (
            "WEEKLY_TRANSFER_ROBUSTNESS_GATE is disabled"
        )
        return result

    if not WEEKLY_EXACT_RAW_ACTION_SUBSET_CHECK:
        result["robustness_subset_skipped_reason"] = (
            "WEEKLY_EXACT_RAW_ACTION_SUBSET_CHECK is disabled"
        )
        return result

    base_eligible = [
        state
        for state in transfer_result["final_beam"]
        if allow_hits
        or float(state["cumulative_hits"]) <= 1e-9
    ]
    if not base_eligible:
        return result

    raw_best = max(
        base_eligible,
        key=lambda state: state["final_objective"],
    )
    if not raw_best.get("path"):
        return result

    raw_transfers = tuple(
        (
            int(out_id),
            int(in_id),
        )
        for out_id, in_id
        in raw_best["path"][0]["transfers"]
    )

    # The production transfer generator is capped at two current moves.
    # Only a raw two-transfer optimum needs exact 1-FT subset checks.
    if len(raw_transfers) != 2:
        return result

    comparator_states = []
    audit_rows = []
    subset_started = time.perf_counter()
    shared_decision_cache = (
        transfer_result.get("decision_cache")
        if WEEKLY_SHARE_TRANSFER_CACHES
        else None
    )
    shared_future_score_cache = (
        transfer_result.get("future_score_cache")
        if WEEKLY_SHARE_TRANSFER_CACHES
        else None
    )
    shared_transfer_pool = (
        transfer_result.get("candidate_pool")
        if WEEKLY_SHARE_TRANSFER_CACHES
        else None
    )

    for forced_transfer in raw_transfers:
        forced_key = (forced_transfer,)

        try:
            forced_result = optimise_transfer_path(
                current_squad_ids=current_squad_ids,
                candidates=candidates,
                projections=projections,
                gameweeks_to_use=gameweeks_to_use,
                money_in_bank=money_in_bank,
                free_transfers=free_transfers,
                current_selling_prices=current_selling_prices,
                forced_first_action_transfers=forced_key,
                prebuilt_transfer_pool=shared_transfer_pool,
                shared_decision_cache=shared_decision_cache,
                shared_future_score_cache=shared_future_score_cache,
            )
        except IllegalForcedTransferAction as exc:
            # A legal 2-FT package can contain a 1-FT subset that is not
            # legal in isolation (e.g. it needs cash/club-limit relief from
            # the other move). That subset is not a valid robustness
            # comparator, so record it and continue.
            audit_rows.append({
                "forced_first_action": forced_key,
                "transfer_count": 1,
                "legal": False,
                "path_objective": None,
                "reason": str(exc),
            })
            continue

        forced_eligible = [
            state
            for state in forced_result["final_beam"]
            if allow_hits
            or float(state["cumulative_hits"]) <= 1e-9
        ]
        if not forced_eligible:
            audit_rows.append({
                "forced_first_action": forced_key,
                "transfer_count": 1,
                "legal": True,
                "path_objective": None,
                "reason": (
                    "Legal opening transfer, but no full-horizon path "
                    "survived the active hit/policy constraints."
                ),
            })
            continue

        forced_best = max(
            forced_eligible,
            key=lambda state: state["final_objective"],
        )
        forced_best = dict(forced_best)
        forced_best["_exact_subset_comparator"] = True
        comparator_states.append(forced_best)

        audit_rows.append({
            "forced_first_action": forced_key,
            "transfer_count": 1,
            "legal": True,
            "path_objective": float(
                forced_best["final_objective"]
            ),
            "reason": None,
        })

    result["robustness_comparator_states"] = comparator_states
    result["robustness_subset_audit"] = audit_rows
    result["robustness_subset_attempted_count"] = int(
        len(raw_transfers)
    )
    result["robustness_subset_legal_count"] = int(
        sum(bool(row.get("legal", False)) for row in audit_rows)
    )
    result["robustness_subset_illegal_count"] = int(
        sum(row.get("legal") is False for row in audit_rows)
    )
    result["robustness_subset_runtime_seconds"] = float(
        time.perf_counter() - subset_started
    )
    return result



def weekly_attach_structural_unavailable_context(
    transfer_result: dict[str, object],
    current_squad_ids: list[int],
    candidates: pd.DataFrame,
    projections: pd.DataFrame,
) -> dict[str, object]:
    """
    Identify materially priced current-squad players with essentially zero
    projected availability throughout the explicit planning horizon.

    This is not a transfer mandate. It is only used as a tie-breaker when
    same-transfer-count actions are already within the configured objective
    robustness band.
    """
    result = dict(transfer_result)

    squad_ids = {
        int(player_id)
        for player_id in current_squad_ids
    }

    player_context = (
        candidates.loc[
            candidates["player_id"].isin(squad_ids),
            [
                "player_id",
                "web_name",
                "price",
                "status",
            ],
        ]
        .drop_duplicates("player_id")
        .copy()
    )

    horizon = (
        projections.loc[
            projections["player_id"].isin(squad_ids)
        ]
        .groupby("player_id", as_index=False)
        .agg(
            horizon_minutes=("expected_minutes", "sum"),
            horizon_max_appearance=(
                "appearance_probability",
                "max",
            ),
            horizon_xp=("expected_points", "sum"),
        )
    )

    context = player_context.merge(
        horizon,
        on="player_id",
        how="left",
        validate="one_to_one",
    )

    for column in [
        "horizon_minutes",
        "horizon_max_appearance",
        "horizon_xp",
    ]:
        context[column] = pd.to_numeric(
            context[column],
            errors="coerce",
        ).fillna(0.0)

    context["price"] = pd.to_numeric(
        context["price"],
        errors="coerce",
    ).fillna(0.0)

    unavailable = context.loc[
        context["price"].ge(
            float(
                WEEKLY_STRUCTURAL_UNAVAILABLE_MIN_PRICE
            )
        )
        & context["horizon_minutes"].le(
            float(
                WEEKLY_STRUCTURAL_UNAVAILABLE_MAX_HORIZON_MINUTES
            )
        )
        & context["horizon_max_appearance"].le(
            float(
                WEEKLY_STRUCTURAL_UNAVAILABLE_MAX_APPEARANCE
            )
        )
    ].copy()

    unavailable_map = {
        int(row.player_id): {
            "web_name": str(row.web_name),
            "price": float(row.price),
            "horizon_minutes": float(
                row.horizon_minutes
            ),
            "horizon_xp": float(row.horizon_xp),
        }
        for row in unavailable.itertuples(index=False)
    }

    result["structural_unavailable_players"] = (
        unavailable_map
    )
    return result


def _weekly_first_action_deadwood_cost(
    state: dict[str, object],
    structural_unavailable_players: dict[int, dict[str, object]],
) -> float:
    """£m of materially unavailable current players retained after GW action."""
    if not state.get("path"):
        return 0.0

    squad_after = {
        int(player_id)
        for player_id
        in state["path"][0]["squad_ids_after"]
    }

    return float(
        sum(
            float(info["price"])
            for player_id, info
            in structural_unavailable_players.items()
            if int(player_id) in squad_after
        )
    )


def _weekly_choose_robust_same_count_state(
    states: list[dict[str, object]],
    transfer_count: int,
    structural_unavailable_players: dict[int, dict[str, object]],
) -> tuple[dict[str, object] | None, dict[str, object] | None]:
    """
    Choose the objective-best state for this transfer count, except that
    within a small objective band we prefer lower retained structural deadwood.
    """
    same_count = [
        state
        for state in states
        if state.get("path")
        and int(
            state["path"][0]["transfer_count"]
        ) == int(transfer_count)
    ]
    if not same_count:
        return None, None

    objective_best = max(
        same_count,
        key=lambda state: state["final_objective"],
    )

    if (
        not WEEKLY_STRUCTURAL_REPAIR_GATE
        or not structural_unavailable_players
    ):
        return objective_best, None

    best_objective = float(
        objective_best["final_objective"]
    )
    near_tied = [
        state
        for state in same_count
        if (
            best_objective
            - float(state["final_objective"])
        )
        <= float(WEEKLY_SAME_COUNT_OBJECTIVE_BAND) + 1e-12
    ]

    if len(near_tied) <= 1:
        return objective_best, None

    costs = {
        id(state): _weekly_first_action_deadwood_cost(
            state,
            structural_unavailable_players,
        )
        for state in near_tied
    }
    minimum_cost = min(costs.values())

    structurally_best = [
        state
        for state in near_tied
        if abs(costs[id(state)] - minimum_cost) <= 1e-12
    ]
    chosen = max(
        structurally_best,
        key=lambda state: state["final_objective"],
    )

    if chosen is objective_best:
        return objective_best, None

    objective_cost = (
        float(objective_best["final_objective"])
        - float(chosen["final_objective"])
    )
    deadwood_before = costs[id(objective_best)]
    deadwood_after = costs[id(chosen)]

    return chosen, {
        "transfer_count": int(transfer_count),
        "objective_best": float(
            objective_best["final_objective"]
        ),
        "chosen_objective": float(
            chosen["final_objective"]
        ),
        "objective_cost": float(objective_cost),
        "deadwood_before": float(deadwood_before),
        "deadwood_after": float(deadwood_after),
        "deadwood_reduction": float(
            deadwood_before - deadwood_after
        ),
        "objective_band": float(
            WEEKLY_SAME_COUNT_OBJECTIVE_BAND
        ),
    }


def weekly_choose_transfer_state(
    transfer_result: dict[str, object],
    allow_hits: bool,
    *,
    marginal_gate_enabled: bool | None = None,
    min_gain_per_extra_transfer: float | None = None,
) -> dict[str, object]:
    """
    Production transfer selector.

    A bank-aware policy may activate the marginal-extra-transfer gate as part
    of the VALIDATED policy itself. The legacy global robustness switch remains
    available for separate experiments, but is not required for a validated
    bank-aware profile.
    """
    effective_marginal_gate = (
        bool(WEEKLY_TRANSFER_ROBUSTNESS_GATE)
        if marginal_gate_enabled is None
        else bool(marginal_gate_enabled)
    )
    effective_threshold = (
        float(WEEKLY_MIN_OBJECTIVE_GAIN_PER_EXTRA_TRANSFER)
        if min_gain_per_extra_transfer is None
        else float(min_gain_per_extra_transfer)
    )

    comparison_states = list(
        transfer_result["final_beam"]
    ) + list(
        transfer_result.get(
            "robustness_comparator_states",
            [],
        )
    )

    eligible = [
        state
        for state in comparison_states
        if allow_hits
        or float(state["cumulative_hits"]) <= 1e-9
    ]
    if not eligible:
        raise RuntimeError(
            "The transfer beam retained no path compatible "
            "with the selected hit policy."
        )

    structural_players = dict(
        transfer_result.get(
            "structural_unavailable_players",
            {},
        )
    )

    raw_best = max(
        eligible,
        key=lambda state: state["final_objective"],
    )
    raw_count = int(
        raw_best["path"][0]["transfer_count"]
    )

    selected, structural_step = (
        _weekly_choose_robust_same_count_state(
            eligible,
            raw_count,
            structural_players,
        )
    )
    if selected is None:
        selected = raw_best

    structural_steps = []
    if structural_step is not None:
        structural_steps.append(structural_step)

    transfer_count_steps = []
    marginal_comparison_available = True

    if effective_marginal_gate:
        while selected.get("path"):
            selected_count = int(
                selected["path"][0]["transfer_count"]
            )
            if selected_count <= 0:
                break

            lower_counts = sorted(
                {
                    int(state["path"][0]["transfer_count"])
                    for state in eligible
                    if state.get("path")
                    and int(state["path"][0]["transfer_count"]) < selected_count
                },
                reverse=True,
            )
            if not lower_counts:
                marginal_comparison_available = False
                break

            lower_representatives = []
            for lower_count in lower_counts:
                representative, step = (
                    _weekly_choose_robust_same_count_state(
                        eligible,
                        lower_count,
                        structural_players,
                    )
                )
                if representative is not None:
                    lower_representatives.append(
                        representative
                    )
                    if step is not None:
                        structural_steps.append(step)

            if not lower_representatives:
                marginal_comparison_available = False
                break

            lower_best = max(
                lower_representatives,
                key=lambda state: state["final_objective"],
            )
            lower_count = int(
                lower_best["path"][0]["transfer_count"]
            )

            extra_transfers = selected_count - lower_count
            incremental_gain = (
                float(selected["final_objective"])
                - float(lower_best["final_objective"])
            )
            required_gain = (
                effective_threshold * float(extra_transfers)
            )

            if incremental_gain + 1e-12 < required_gain:
                transfer_count_steps.append({
                    "from_transfer_count": selected_count,
                    "to_transfer_count": lower_count,
                    "incremental_gain": float(incremental_gain),
                    "required_gain": float(required_gain),
                })
                selected = lower_best
                continue
            break

    selected_count = int(
        selected["path"][0]["transfer_count"]
    )
    objective_given_up = (
        float(raw_best["final_objective"])
        - float(selected["final_objective"])
    )

    raw_key = tuple(
        raw_best["path"][0]["transfers"]
    )
    selected_key = tuple(
        selected["path"][0]["transfers"]
    )

    transfer_count_gate_applied = selected_count != raw_count
    structural_gate_applied = (
        selected_key != raw_key
        and bool(structural_steps)
    )
    gate_applied = (
        transfer_count_gate_applied
        or structural_gate_applied
    )

    reason_parts = []

    if structural_steps:
        relevant = [
            step
            for step in structural_steps
            if int(step["transfer_count"]) == selected_count
        ]
        if relevant:
            step = relevant[-1]
            reason_parts.append(
                "Within the same-transfer-count near-tie band, "
                f"preferred the structurally cleaner action: "
                f"retained unavailable squad value fell from "
                f"£{step['deadwood_before']:.1f}m to "
                f"£{step['deadwood_after']:.1f}m for only "
                f"{step['objective_cost']:.3f} horizon objective points."
            )

    if transfer_count_steps:
        last_step = transfer_count_steps[-1]
        reason_parts.append(
            f"Preserved {raw_count - selected_count} current transfer(s): "
            f"the marginal extra move(s) added only "
            f"{last_step['incremental_gain']:+.3f} objective points, below "
            f"the {last_step['required_gain']:+.3f} bank-aware threshold."
        )
    elif effective_marginal_gate and not marginal_comparison_available:
        reason_parts.append(
            "Bank-aware gate was active but no lower-transfer-count state "
            "survived the beam; treat this action as lower-confidence."
        )
    elif effective_marginal_gate and not transfer_count_gate_applied:
        reason_parts.append(
            "The selected transfer count cleared the validated bank-aware "
            "marginal-value threshold."
        )

    if not reason_parts:
        reason_parts.append(
            "Transfer robustness gates disabled or not applicable."
        )

    result = dict(selected)
    result["_robustness_gate"] = {
        "enabled": bool(
            effective_marginal_gate
            or WEEKLY_STRUCTURAL_REPAIR_GATE
        ),
        "marginal_gate_enabled": bool(effective_marginal_gate),
        "marginal_comparison_available": bool(
            marginal_comparison_available
        ),
        "applied": bool(gate_applied),
        "raw_transfer_count": raw_count,
        "selected_transfer_count": selected_count,
        "raw_objective": float(raw_best["final_objective"]),
        "selected_objective": float(selected["final_objective"]),
        "objective_given_up": float(objective_given_up),
        "threshold_per_extra_transfer": float(effective_threshold),
        "same_count_objective_band": float(
            WEEKLY_SAME_COUNT_OBJECTIVE_BAND
        ),
        "structural_repair_gate_enabled": bool(
            WEEKLY_STRUCTURAL_REPAIR_GATE
        ),
        "structural_repair_gate_applied": bool(
            structural_gate_applied
        ),
        "structural_unavailable_players": structural_players,
        "structural_steps": structural_steps,
        "protected_first_action_count": int(
            len(
                transfer_result.get(
                    "protected_first_action_keys",
                    (),
                )
            )
        ),
        "exact_subset_comparator_count": int(
            len(
                transfer_result.get(
                    "robustness_comparator_states",
                    (),
                )
            )
        ),
        "exact_subset_attempted_count": int(
            transfer_result.get(
                "robustness_subset_attempted_count",
                0,
            )
        ),
        "exact_subset_illegal_count": int(
            transfer_result.get(
                "robustness_subset_illegal_count",
                0,
            )
        ),
        "exact_subset_audit": list(
            transfer_result.get(
                "robustness_subset_audit",
                (),
            )
        ),
        "steps": transfer_count_steps,
        "reason": " ".join(reason_parts),
    }
    return result




def weekly_transfer_evidence_audit(
    selected_state: dict[str, object],
    transfer_result: dict[str, object] | None,
    candidates: pd.DataFrame,
    gameweek_expected_points: pd.DataFrame,
    new_player_audit: pd.DataFrame,
    *,
    gameweek: int,
    allow_hits: bool,
) -> tuple[pd.DataFrame, dict[str, object]]:
    """
    Diagnose whether incoming transfers are supported by enough evidence.

    This is deliberately diagnostic rather than a hard veto. The validated
    bank-aware policy decides whether an extra FT is worth spending; this audit
    explains when a chosen incoming player is sparse-history, DC-driven, or
    nearly replaceable by a materially cheaper alternative.
    """
    empty_summary = {
        "selected_transfer_count": 0,
        "best_lower_transfer_count": None,
        "incremental_objective_gain_vs_lower_count": None,
        "low_evidence_incoming_count": 0,
        "action_extra_transfer_fragile": False,
        "action_low_evidence": False,
        "reason": "No incoming transfers to audit.",
    }

    if not selected_state.get("path"):
        return pd.DataFrame(), empty_summary

    first_action = selected_state["path"][0]
    transfers = tuple(first_action.get("transfers", ()))
    selected_count = int(first_action.get("transfer_count", len(transfers)))
    if not transfers:
        return pd.DataFrame(), empty_summary

    selected_final_ids = {
        int(pid)
        for pid in first_action.get(
            "squad_ids_after",
            selected_state.get("squad_ids", ()),
        )
    }

    # Same-state comparison against the strongest surviving action using fewer
    # current transfers. This is diagnostic even when a bank-aware gate already
    # selected the lower-count action.
    lower_count = None
    incremental_gain = np.nan
    if transfer_result is not None:
        states = (
            list(transfer_result.get("final_beam", ()))
            + list(transfer_result.get("robustness_comparator_states", ()))
        )
        eligible_states = [
            state for state in states
            if state.get("path")
            and (
                allow_hits
                or float(state.get("cumulative_hits", 0.0)) <= 1e-9
            )
        ]
        lower_states = [
            state for state in eligible_states
            if int(state["path"][0].get("transfer_count", 0)) < selected_count
        ]
        if lower_states:
            lower_best = max(
                lower_states,
                key=lambda state: float(state.get("final_objective", -np.inf)),
            )
            lower_count = int(lower_best["path"][0].get("transfer_count", 0))
            incremental_gain = (
                float(selected_state.get("final_objective", np.nan))
                - float(lower_best.get("final_objective", np.nan))
            )

    candidate_frame = candidates.copy()
    if "player_id" not in candidate_frame.columns and "id" in candidate_frame.columns:
        candidate_frame = candidate_frame.rename(columns={"id": "player_id"})
    candidate_frame["player_id"] = pd.to_numeric(
        candidate_frame["player_id"], errors="coerce"
    ).astype("Int64")

    xp = gameweek_expected_points.copy()
    if "player_id" not in xp.columns and "id" in xp.columns:
        xp = xp.rename(columns={"id": "player_id"})
    if "gameweek" in xp.columns:
        xp = xp.loc[
            pd.to_numeric(xp["gameweek"], errors="coerce").eq(int(gameweek))
        ].copy()
    xp["player_id"] = pd.to_numeric(xp["player_id"], errors="coerce").astype("Int64")
    xp = xp.sort_values(
        "expected_points" if "expected_points" in xp.columns else "player_id",
        ascending=False,
    ).drop_duplicates("player_id", keep="first")

    audit = new_player_audit.copy()
    if not audit.empty:
        if "player_id" not in audit.columns and "id" in audit.columns:
            audit = audit.rename(columns={"id": "player_id"})
        audit["player_id"] = pd.to_numeric(
            audit["player_id"], errors="coerce"
        ).astype("Int64")
        audit = audit.drop_duplicates("player_id", keep="first")

    candidate_lookup = (
        candidate_frame.dropna(subset=["player_id"])
        .set_index("player_id")
        .to_dict("index")
    )
    xp_lookup = (
        xp.dropna(subset=["player_id"])
        .set_index("player_id")
        .to_dict("index")
    )
    audit_lookup = (
        audit.dropna(subset=["player_id"])
        .set_index("player_id")
        .to_dict("index")
        if not audit.empty and "player_id" in audit.columns
        else {}
    )

    rows = []
    for outgoing_id, incoming_id in transfers:
        outgoing_id = int(outgoing_id)
        incoming_id = int(incoming_id)
        incoming_meta = dict(candidate_lookup.get(incoming_id, {}))
        outgoing_meta = dict(candidate_lookup.get(outgoing_id, {}))
        incoming_xp = dict(xp_lookup.get(incoming_id, {}))
        sparse_meta = dict(audit_lookup.get(incoming_id, {}))

        name = str(
            incoming_meta.get(
                "web_name",
                incoming_xp.get("web_name", incoming_id),
            )
        )
        position = str(
            incoming_meta.get(
                "position",
                incoming_xp.get("position", ""),
            )
        )
        price = pd.to_numeric(
            pd.Series([incoming_meta.get("price", np.nan)]),
            errors="coerce",
        ).iloc[0]
        expected_points = pd.to_numeric(
            pd.Series([incoming_xp.get("expected_points", np.nan)]),
            errors="coerce",
        ).iloc[0]
        dc_xp = pd.to_numeric(
            pd.Series([
                incoming_xp.get("xp_defensive_contribution", np.nan)
            ]),
            errors="coerce",
        ).iloc[0]
        dc_share = (
            max(float(dc_xp), 0.0) / max(abs(float(expected_points)), 1e-9)
            if pd.notna(dc_xp) and pd.notna(expected_points)
            else np.nan
        )

        career_fixtures = pd.to_numeric(
            pd.Series([
                sparse_meta.get(
                    "prior_career_fixture_count",
                    incoming_xp.get("prior_career_fixture_count", np.nan),
                )
            ]),
            errors="coerce",
        ).iloc[0]
        season_fixtures = pd.to_numeric(
            pd.Series([
                sparse_meta.get(
                    "prior_season_fixture_count",
                    incoming_xp.get("prior_season_fixture_count", np.nan),
                )
            ]),
            errors="coerce",
        ).iloc[0]
        external_available = bool(
            pd.to_numeric(
                pd.Series([sparse_meta.get("external_prior_available", 0)]),
                errors="coerce",
            ).fillna(0).iloc[0] > 0
        )
        external_event_weight = float(
            pd.to_numeric(
                pd.Series([
                    sparse_meta.get("external_event_prior_weight", 0.0)
                ]),
                errors="coerce",
            ).fillna(0.0).iloc[0]
        )
        market_fallback_weight = float(
            pd.to_numeric(
                pd.Series([
                    sparse_meta.get(
                        "market_fallback_weight",
                        incoming_xp.get("market_fallback_weight", 0.0),
                    )
                ]),
                errors="coerce",
            ).fillna(0.0).iloc[0]
        )

        cheaper_alt = None
        if pd.notna(price) and position:
            same_position = candidate_frame.loc[
                candidate_frame.get(
                    "position",
                    pd.Series("", index=candidate_frame.index),
                ).astype(str).eq(position)
                & ~candidate_frame["player_id"].isin(selected_final_ids)
            ].copy()
            same_position["price_num"] = pd.to_numeric(
                same_position.get("price"), errors="coerce"
            )
            same_position = same_position.loc[
                same_position["price_num"].le(
                    float(price) - LOW_EVIDENCE_CHEAPER_ALT_MIN_SAVING + 1e-9
                )
            ]
            if not same_position.empty:
                alt = same_position.merge(
                    xp[
                        [
                            column
                            for column in [
                                "player_id",
                                "expected_points",
                            ]
                            if column in xp.columns
                        ]
                    ],
                    on="player_id",
                    how="left",
                )
                alt["expected_points"] = pd.to_numeric(
                    alt.get("expected_points"), errors="coerce"
                )
                alt = alt.dropna(subset=["expected_points"]).sort_values(
                    ["expected_points", "price_num"],
                    ascending=[False, True],
                )
                if not alt.empty:
                    best_alt = alt.iloc[0]
                    xp_gap = (
                        float(expected_points) - float(best_alt["expected_points"])
                        if pd.notna(expected_points)
                        else np.nan
                    )
                    saving = float(price) - float(best_alt["price_num"])
                    if (
                        np.isfinite(xp_gap)
                        and xp_gap <= LOW_EVIDENCE_CHEAPER_ALT_XP_GAP + 1e-9
                        and saving >= LOW_EVIDENCE_CHEAPER_ALT_MIN_SAVING - 1e-9
                    ):
                        cheaper_alt = {
                            "name": str(best_alt.get("web_name", best_alt["player_id"])),
                            "price": float(best_alt["price_num"]),
                            "expected_points": float(best_alt["expected_points"]),
                            "xp_gap": float(xp_gap),
                            "saving": float(saving),
                        }

        reasons = []
        sparse_without_event_prior = (
            pd.notna(career_fixtures)
            and float(career_fixtures) < float(LOW_EVIDENCE_MAX_PL_FIXTURES)
            and external_event_weight <= 0.05
        )
        dc_driven_sparse = (
            pd.notna(season_fixtures)
            and float(season_fixtures) < float(LOW_EVIDENCE_MAX_PL_FIXTURES)
            and np.isfinite(dc_share)
            and float(dc_share) >= float(LOW_EVIDENCE_DC_SHARE_THRESHOLD)
        )
        fragile_extra_transfer = (
            selected_count >= 2
            and lower_count is not None
            and np.isfinite(incremental_gain)
            and float(incremental_gain)
                < float(LOW_EVIDENCE_EXTRA_TRANSFER_GAIN_THRESHOLD)
        )

        if sparse_without_event_prior:
            reasons.append(
                f"sparse PL evidence ({float(career_fixtures):.0f} prior fixture(s)) "
                "without an external event-rate prior"
            )
        if dc_driven_sparse:
            reasons.append(
                f"{float(dc_share):.0%} of xP comes from defensive contributions "
                "on sparse current-season evidence"
            )
        if cheaper_alt is not None:
            reasons.append(
                f"cheaper same-position alternative {cheaper_alt['name']} is only "
                f"{cheaper_alt['xp_gap']:+.3f} xP behind while saving "
                f"£{cheaper_alt['saving']:.1f}m"
            )
        if market_fallback_weight > 0.05:
            reasons.append(
                f"market/sparse-history shrinkage is active "
                f"(weight {market_fallback_weight:.2f})"
            )

        hard_low_evidence = bool(
            sparse_without_event_prior
            or dc_driven_sparse
        )
        evidence_level = (
            "LOW"
            if hard_low_evidence
            else (
                "MEDIUM"
                if reasons
                else "HIGH"
            )
        )

        rows.append({
            "outgoing": str(outgoing_meta.get("web_name", outgoing_id)),
            "incoming": name,
            "position": position,
            "price": float(price) if pd.notna(price) else np.nan,
            "expected_points": (
                float(expected_points) if pd.notna(expected_points) else np.nan
            ),
            "xp_defensive_contribution": (
                float(dc_xp) if pd.notna(dc_xp) else np.nan
            ),
            "dc_share_of_xp": float(dc_share) if np.isfinite(dc_share) else np.nan,
            "prior_pl_fixtures": (
                float(career_fixtures) if pd.notna(career_fixtures) else np.nan
            ),
            "prior_season_fixtures": (
                float(season_fixtures) if pd.notna(season_fixtures) else np.nan
            ),
            "external_prior_available": bool(external_available),
            "external_event_prior_weight": float(external_event_weight),
            "external_prior_match_method": sparse_meta.get(
                "external_prior_match_method"
            ),
            "external_source_leagues": sparse_meta.get(
                "external_source_leagues"
            ),
            "external_prior_source_note": sparse_meta.get(
                "external_prior_source_note"
            ),
            "market_fallback_weight": float(market_fallback_weight),
            "selected_transfer_count": selected_count,
            "best_lower_transfer_count": lower_count,
            "incremental_objective_gain_vs_lower_count": (
                float(incremental_gain)
                if np.isfinite(incremental_gain)
                else np.nan
            ),
            "cheaper_alternative": (
                cheaper_alt["name"] if cheaper_alt is not None else None
            ),
            "cheaper_alternative_price": (
                cheaper_alt["price"] if cheaper_alt is not None else np.nan
            ),
            "cheaper_alternative_xp": (
                cheaper_alt["expected_points"] if cheaper_alt is not None else np.nan
            ),
            "cheaper_alternative_xp_gap": (
                cheaper_alt["xp_gap"] if cheaper_alt is not None else np.nan
            ),
            "cheaper_alternative_saving": (
                cheaper_alt["saving"] if cheaper_alt is not None else np.nan
            ),
            "action_extra_transfer_fragile": bool(fragile_extra_transfer),
            "evidence_level": evidence_level,
            "evidence_reasons": "; ".join(reasons) if reasons else "No low-evidence flags.",
        })

    frame = pd.DataFrame(rows)
    low_count = (
        int(frame["evidence_level"].eq("LOW").sum())
        if not frame.empty
        else 0
    )
    action_fragile = bool(
        selected_count >= 2
        and lower_count is not None
        and np.isfinite(incremental_gain)
        and float(incremental_gain)
            < float(LOW_EVIDENCE_EXTRA_TRANSFER_GAIN_THRESHOLD)
    )
    if low_count and action_fragile:
        reason = (
            "At least one incoming player has sparse/fragile evidence and the "
            "extra-transfer action has a small marginal objective gain."
        )
    elif low_count:
        reason = "At least one incoming player triggered the low-evidence rules."
    elif action_fragile:
        reason = (
            "The multi-transfer action has a small marginal objective gain "
            "versus a lower-transfer-count state."
        )
    else:
        reason = "No incoming transfer triggered the low-evidence rules."

    summary = {
        "selected_transfer_count": selected_count,
        "best_lower_transfer_count": lower_count,
        "incremental_objective_gain_vs_lower_count": (
            float(incremental_gain)
            if np.isfinite(incremental_gain)
            else None
        ),
        "low_evidence_incoming_count": low_count,
        "action_extra_transfer_fragile": action_fragile,
        "action_low_evidence": bool(low_count > 0 or action_fragile),
        "reason": reason,
    }
    return frame, summary

def weekly_build_opening_state(
    candidates: pd.DataFrame,
    projections: pd.DataFrame,
    gameweeks_to_use: list[int],
) -> tuple[dict[str, object], list[int], float]:
    """Create a no-transfer path from the optimised Gameweek 1 squad."""
    (
        opening_result,
        opening_alternative_table,
        opening_bench_table,
    ) = choose_opening_squad_solution(
        candidates,
        projections,
        gameweeks_to_use,
        budget=INITIAL_SQUAD_BUDGET,
    )
    opening_squad = opening_result["squad"].copy()
    squad_ids = opening_squad[
        "player_id"
    ].astype(int).tolist()
    bank = max(
        0.0,
        INITIAL_SQUAD_BUDGET
        - float(opening_squad["price"].sum()),
    )

    candidate_lookup = {
        int(row.player_id): row._asdict()
        for row in candidates.itertuples(index=False)
    }
    projection_lookup = build_projection_lookup(
        projections
    )

    path = []
    free_transfers = 1
    cumulative_points = 0.0
    discounted_objective = 0.0

    for horizon_index, gameweek in enumerate(
        gameweeks_to_use
    ):
        decision = fast_squad_gameweek_decision(
            squad_ids,
            int(gameweek),
            candidate_lookup,
            projection_lookup,
        )
        free_transfers_after = update_free_transfers(
            free_transfers,
            0,
        )
        row = {
            "gameweek": int(gameweek),
            "squad_ids_before": tuple(sorted(squad_ids)),
            "squad_ids_after": tuple(sorted(squad_ids)),
            "transfers": (),
            "transfer_count": 0,
            "free_transfers_before": int(free_transfers),
            "free_transfers_after": int(
                free_transfers_after
            ),
            "hit_cost": 0.0,
            "bank_before_tenths": price_to_tenths(bank),
            "bank_after_tenths": price_to_tenths(bank),
            "expected_score_before_hits": float(
                decision["search_score"]
            ),
            "expected_score_after_hits": float(
                decision["search_score"]
            ),
            "lineup_decision": decision,
        }
        path.append(row)
        cumulative_points += float(
            decision["search_score"]
        )
        discounted_objective += (
            GAMEWEEK_DISCOUNT ** horizon_index
            * float(decision["search_score"])
        )
        free_transfers = free_transfers_after

    state = {
        "squad_ids": tuple(sorted(squad_ids)),
        "bank_tenths": price_to_tenths(bank),
        "selling_prices_tenths": {
            int(row.player_id): price_to_tenths(
                float(row.price)
            )
            for row in opening_squad.itertuples(index=False)
        },
        "free_transfers": int(free_transfers),
        "cumulative_expected_points": cumulative_points,
        "cumulative_hits": 0.0,
        "discounted_objective": discounted_objective,
        "terminal_value": 0.0,
        "final_objective": discounted_objective,
        "path": path,
    }
    return state, squad_ids, bank, opening_alternative_table, opening_bench_table


def weekly_state_plan_table(
    state: dict[str, object],
    candidates: pd.DataFrame,
) -> pd.DataFrame:
    """Format the rolling transfer path for display."""
    name_lookup = (
        candidates
        .set_index("player_id")["web_name"]
        .astype(str)
        .to_dict()
    )
    rows = []
    for path_row in state["path"]:
        labels = [
            (
                f"{name_lookup.get(outgoing_id, outgoing_id)}"
                f" → "
                f"{name_lookup.get(incoming_id, incoming_id)}"
            )
            for outgoing_id, incoming_id
            in path_row["transfers"]
        ]
        lineup_decision = path_row["lineup_decision"]
        rows.append({
            "gameweek": int(path_row["gameweek"]),
            "transfers": (
                "; ".join(labels) if labels else "Roll"
            ),
            "transfer_count": int(
                path_row["transfer_count"]
            ),
            "free_transfers_before": int(
                path_row["free_transfers_before"]
            ),
            "free_transfers_after": int(
                path_row["free_transfers_after"]
            ),
            "hit_cost": float(path_row["hit_cost"]),
            "bank_after": tenths_to_price(
                int(path_row["bank_after_tenths"])
            ),
            "captain": name_lookup.get(
                int(lineup_decision["captain_id"]),
                lineup_decision["captain_id"],
            ),
            "projected_score_after_hits": float(
                path_row["expected_score_after_hits"]
            ),
        })
    return pd.DataFrame(rows)


def weekly_transfer_action_comparison(
    transfer_result: dict[str, object],
    candidates: pd.DataFrame,
    allow_hits: bool,
    selected_state: dict[str, object] | None = None,
) -> pd.DataFrame:
    """Compare immediate actions and always expose the best 0/1/2-FT option."""
    name_lookup = (
        candidates
        .set_index("player_id")["web_name"]
        .astype(str)
        .to_dict()
    )
    comparison_states = list(
        transfer_result["final_beam"]
    ) + list(
        transfer_result.get(
            "robustness_comparator_states",
            [],
        )
    )

    eligible_states = [
        state for state in comparison_states
        if allow_hits or float(state["cumulative_hits"]) <= 1e-9
    ]
    if not eligible_states:
        return pd.DataFrame()

    raw_best = max(eligible_states, key=lambda state: state["final_objective"])
    raw_key = tuple(raw_best["path"][0]["transfers"])
    selected_key = (
        tuple(selected_state["path"][0]["transfers"])
        if selected_state is not None and selected_state.get("path")
        else raw_key
    )

    rows = []
    for state in eligible_states:
        first_row = state["path"][0]
        first_transfers = tuple(first_row["transfers"])
        labels = [
            (
                f"{name_lookup.get(outgoing_id, outgoing_id)}"
                f" → "
                f"{name_lookup.get(incoming_id, incoming_id)}"
            )
            for outgoing_id, incoming_id in first_transfers
        ]
        structural_players = dict(
            transfer_result.get(
                "structural_unavailable_players",
                {},
            )
        )
        rows.append({
            "first_action_key": first_transfers,
            "current_action": "; ".join(labels) if labels else "Roll",
            "transfer_count": int(first_row["transfer_count"]),
            "current_hit_cost": float(first_row["hit_cost"]),
            "bank_after": tenths_to_price(int(first_row["bank_after_tenths"])),
            "path_objective": float(state["final_objective"]),
            "structural_deadwood_after": (
                _weekly_first_action_deadwood_cost(
                    state,
                    structural_players,
                )
            ),
            "raw_best": first_transfers == raw_key,
            "selected": first_transfers == selected_key,
            "exact_subset_comparator": bool(
                state.get(
                    "_exact_subset_comparator",
                    False,
                )
            ),
        })

    frame = (
        pd.DataFrame(rows)
        .sort_values("path_objective", ascending=False)
        .drop_duplicates("first_action_key")
        .reset_index(drop=True)
    )
    frame["best_for_transfer_count"] = False
    if not frame.empty:
        best_count_indices = (
            frame.sort_values("path_objective", ascending=False)
            .groupby("transfer_count", sort=True)
            .head(1)
            .index
        )
        frame.loc[best_count_indices, "best_for_transfer_count"] = True

    must_show = frame.loc[
        frame["best_for_transfer_count"] | frame["selected"] | frame["raw_best"]
    ]
    top = frame.head(10)
    result = (
        pd.concat([top, must_show], ignore_index=False)
        .drop_duplicates("first_action_key")
        .sort_values("path_objective", ascending=False)
        .drop(columns="first_action_key")
        .reset_index(drop=True)
    )
    return result


def weekly_exact_current_decision(
    state: dict[str, object],
    candidates: pd.DataFrame,
    projections: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[str, object]]:
    """Return the exact current squad and detailed current-GW decision."""
    first_row = state["path"][0]
    squad_ids = [
        int(player_id)
        for player_id in first_row["squad_ids_after"]
    ]
    squad = candidates.loc[
        candidates["player_id"].isin(squad_ids)
    ].copy()
    decision = execution_policy_squad_gameweek_decision(
        squad_ids,
        int(first_row["gameweek"]),
        candidates,
        projections,
    )
    return squad, decision


def weekly_evaluate_chips(
    best_state: dict[str, object],
    initial_squad_ids: list[int],
    initial_bank: float,
    candidates: pd.DataFrame,
    projections: pd.DataFrame,
    gameweeks_to_use: list[int],
    selling_prices: dict[int, float],
    fixture_level_predictions: pd.DataFrame,
    teams_table: pd.DataFrame,
    normal_transfer_count: int = 0,
    normal_hit_cost: float = 0.0,
) -> dict[str, object]:
    """
    Evaluate chips with production-lazy expensive optimisation.

    Triple Captain and Bench Boost remain cheap and are evaluated across the
    full horizon. Free Hit and Wildcard are evaluated for the current GW first.
    Future FH/WC MILPs are solved only if the current opportunity can still
    pass both its minimum uplift threshold and the structural-readiness gate.
    """
    evaluation_started = time.perf_counter()
    baseline_contexts = reconstruct_transfer_path_finances(
        best_state,
        initial_squad_ids,
        initial_bank,
        candidates,
        configured_selling_prices=selling_prices,
    )

    baseline_rows = []
    for context in baseline_contexts:
        path_row = context["path_row"]
        decision = execution_policy_squad_gameweek_decision(
            path_row["squad_ids_after"], context["gameweek"], candidates, projections
        )
        baseline_rows.append({
            **context,
            "baseline_decision": decision,
            "baseline_score_before_hits": float(decision["expected_total_score"]),
            "baseline_hit_cost": float(path_row["hit_cost"]),
            "baseline_score_after_hits": float(
                decision["expected_total_score"] - path_row["hit_cost"]
            ),
        })

    fixture_summary = build_fixture_shape_summary(
        fixture_level_predictions, teams_table, gameweeks_to_use
    )
    details: dict[tuple[str, int], dict[str, object]] = {}
    expensive_solve_count = 0
    skipped_future_expensive_chips: list[str] = []
    chip_squad_cache: dict[tuple, dict[str, object]] = {}

    def cached_chip_squad(gameweeks, budget, required_player_ids):
        nonlocal expensive_solve_count
        key = (
            tuple(int(gw) for gw in gameweeks),
            round(float(budget), 3),
            tuple(sorted(int(pid) for pid in required_player_ids)),
        )
        if key not in chip_squad_cache:
            expensive_solve_count += 1
            chip_squad_cache[key] = optimise_chip_squad(
                candidates,
                projections,
                list(key[0]),
                budget=float(budget),
                required_player_ids=key[2],
            )
        return chip_squad_cache[key]

    rows = []
    for horizon_index, context in enumerate(baseline_rows):
        gameweek = int(context["gameweek"])
        baseline_decision = context["baseline_decision"]
        baseline_score = float(context["baseline_score_after_hits"])
        shape_match = fixture_summary.loc[fixture_summary["gameweek"].eq(gameweek)]
        shape = shape_match.iloc[0].to_dict() if not shape_match.empty else {}
        rows.append({
            "gameweek": gameweek,
            "baseline_score_after_hits": baseline_score,
            "captain_id": int(baseline_decision["captain_id"]),
            "vice_captain_id": int(baseline_decision["vice_captain_id"]),
            "triple_captain_uplift": (
                triple_captain_uplift(baseline_decision)
                if chip_is_eligible("triple_captain", gameweek) else np.nan
            ),
            "bench_boost_uplift": (
                bench_boost_uplift(baseline_decision)
                if chip_is_eligible("bench_boost", gameweek) else np.nan
            ),
            "free_hit_uplift": np.nan,
            "free_hit_score": np.nan,
            "free_hit_captain_id": None,
            "wildcard_horizon_uplift": np.nan,
            "wildcard_current_score": np.nan,
            "wildcard_horizon_score": np.nan,
            "baseline_remaining_horizon_score": np.nan,
            "blank_teams": shape.get("blank_teams", np.nan),
            "double_or_more_teams": shape.get("double_or_more_teams", np.nan),
            "blank_team_names": shape.get("blank_team_names", ""),
            "double_team_names": shape.get("double_team_names", ""),
        })

    opportunities = pd.DataFrame(rows)
    next_gameweek = int(gameweeks_to_use[0])

    def fixture_shape_for(gameweek: int) -> tuple[int, int]:
        match = opportunities.loc[opportunities["gameweek"].eq(int(gameweek))]
        if match.empty:
            return 0, 0
        row = match.iloc[0]
        blank = pd.to_numeric(row.get("blank_teams", 0), errors="coerce")
        double = pd.to_numeric(row.get("double_or_more_teams", 0), errors="coerce")
        return (
            int(blank) if pd.notna(blank) else 0,
            int(double) if pd.notna(double) else 0,
        )

    def expiry_pressure_at(gameweek: int) -> bool:
        expiry = (
            FIRST_CHIP_SET_EXPIRY_GAMEWEEK
            if int(gameweek) <= FIRST_CHIP_SET_EXPIRY_GAMEWEEK
            else SECOND_CHIP_SET_EXPIRY_GAMEWEEK
        )
        # v2.6 P21 — start the expiry cascade with one week of slack.
        #
        # `remaining_chips >= remaining_weeks` is the must-start-now boundary, not
        # a margin: with 4 chips it first fires at GW16, leaving exactly four weeks
        # for four chips and no recovery room. One missed or disrupted week then
        # loses a chip permanently — and a free hit blocks the following gameweek,
        # so a late free hit can strand another chip by itself.
        #
        # With zero doubles and blanks on the calendar, chip value is roughly flat
        # across GW15-19, so buying a spare week costs almost nothing.
        remaining_weeks = max(1, int(expiry) - int(gameweek) + 1)
        remaining_chips = sum(bool(v) for v in WEEKLY_AVAILABLE_CHIPS.values())
        _p21_slack = int(globals().get("V26_CHIP_EXPIRY_SLACK", 1))
        return bool(remaining_chips >= remaining_weeks - _p21_slack)

    def evaluate_free_hit_at(horizon_index: int):
        gameweek = int(gameweeks_to_use[horizon_index])
        if not chip_is_eligible("free_hit", gameweek):
            return np.nan
        context = baseline_rows[horizon_index]
        baseline_score = float(context["baseline_score_after_hits"])
        baseline_squad_ids = tuple(
            int(pid) for pid in context["path_row"]["squad_ids_after"]
        )
        result = cached_chip_squad(
            [gameweek], context["available_budget"], baseline_squad_ids
        )
        squad = result["squad"].copy()
        ids = squad["player_id"].astype(int).tolist()
        decision = execution_policy_squad_gameweek_decision(
            ids, gameweek, result["subset_candidates"], result["subset_projections"]
        )
        score = float(decision["expected_total_score"])
        uplift = score - baseline_score
        mask = opportunities["gameweek"].eq(gameweek)
        opportunities.loc[mask, "free_hit_uplift"] = uplift
        opportunities.loc[mask, "free_hit_score"] = score
        opportunities.loc[mask, "free_hit_captain_id"] = int(decision["captain_id"])
        details[("free_hit", gameweek)] = {
            "squad": squad, "decision": decision, "budget": context["available_budget"]
        }
        return float(uplift)

    def evaluate_wildcard_at(horizon_index: int):
        gameweek = int(gameweeks_to_use[horizon_index])
        if not chip_is_eligible("wildcard", gameweek):
            return np.nan
        context = baseline_rows[horizon_index]
        baseline_squad_ids = tuple(
            int(pid) for pid in context["path_row"]["squad_ids_after"]
        )
        remaining_gameweeks = gameweeks_to_use[horizon_index:]
        result = cached_chip_squad(
            remaining_gameweeks, context["available_budget"], baseline_squad_ids
        )
        squad = result["squad"].copy()
        ids = squad["player_id"].astype(int).tolist()
        score_rows = []
        current_decision = None
        for relative_index, gw in enumerate(remaining_gameweeks):
            decision = execution_policy_squad_gameweek_decision(
                ids, int(gw), result["subset_candidates"], result["subset_projections"]
            )
            if relative_index == 0:
                current_decision = decision
            score_rows.append({
                "gameweek": int(gw),
                "score": float(decision["expected_total_score"]),
                "discount": GAMEWEEK_DISCOUNT ** relative_index,
            })
        horizon_score = float(sum(r["score"] * r["discount"] for r in score_rows))
        baseline_remaining = baseline_rows[horizon_index:]
        baseline_horizon = float(sum(
            row["baseline_score_after_hits"] * GAMEWEEK_DISCOUNT ** relative_index
            for relative_index, row in enumerate(baseline_remaining)
        ))
        uplift = horizon_score - baseline_horizon
        mask = opportunities["gameweek"].eq(gameweek)
        opportunities.loc[mask, "wildcard_horizon_uplift"] = uplift
        opportunities.loc[mask, "wildcard_current_score"] = float(score_rows[0]["score"])
        opportunities.loc[mask, "wildcard_horizon_score"] = horizon_score
        opportunities.loc[mask, "baseline_remaining_horizon_score"] = baseline_horizon
        details[("wildcard", gameweek)] = {
            "squad": squad,
            "decision": current_decision,
            "scores": pd.DataFrame(score_rows),
            "budget": context["available_budget"],
        }
        return float(uplift)

    # Evaluate only the current expensive opportunities first.
    current_fh = evaluate_free_hit_at(0) if chip_is_eligible("free_hit", next_gameweek) else np.nan
    current_wc = evaluate_wildcard_at(0) if chip_is_eligible("wildcard", next_gameweek) else np.nan

    # Future FH/WC timing matters only if the current option is still capable
    # of being recommended now. Otherwise the minimum threshold or structural
    # gate already makes Save deterministic.
    expensive_current = {
        "free_hit": current_fh,
        "wildcard": current_wc,
    }
    for chip_name, current_uplift in expensive_current.items():
        if not np.isfinite(current_uplift):
            continue
        blank_teams, double_teams = fixture_shape_for(next_gameweek)
        structural, _ = chip_structural_readiness(
            chip_name,
            current_uplift=float(current_uplift),
            blank_teams=blank_teams,
            double_teams=double_teams,
            expiry_pressure=expiry_pressure_at(next_gameweek),
            normal_transfer_count=int(normal_transfer_count),
            normal_hit_cost=float(normal_hit_cost),
            exceptional_thresholds=WEEKLY_CHIP_EXCEPTIONAL_UPLIFT,
        )
        can_still_use_now = (
            float(current_uplift) >= float(CHIP_MINIMUM_UPLIFT[chip_name])
            and (
                structural
                or not WEEKLY_CHIP_REQUIRE_STRUCTURAL_TRIGGER
            )
        )
        if WEEKLY_LAZY_CHIP_EVALUATION and not can_still_use_now:
            skipped_future_expensive_chips.append(chip_name)
            continue
        for horizon_index in range(1, len(gameweeks_to_use)):
            if chip_name == "free_hit":
                evaluate_free_hit_at(horizon_index)
            else:
                evaluate_wildcard_at(horizon_index)

    name_lookup = candidates.set_index("player_id")["web_name"].astype(str).to_dict()
    opportunities["captain"] = opportunities["captain_id"].map(name_lookup)
    opportunities["free_hit_captain"] = opportunities["free_hit_captain_id"].map(name_lookup)

    uplift_columns = {
        "triple_captain": "triple_captain_uplift",
        "bench_boost": "bench_boost_uplift",
        "free_hit": "free_hit_uplift",
        "wildcard": "wildcard_horizon_uplift",
    }
    best_rows = []
    for chip_name, column in uplift_columns.items():
        valid = opportunities.loc[opportunities[column].notna()]
        if valid.empty:
            continue
        best = valid.loc[valid[column].idxmax()]
        best_rows.append({
            "chip": chip_name,
            "best_gameweek_in_horizon": int(best["gameweek"]),
            "best_uplift_in_horizon": float(best[column]),
        })
    best_opportunities = pd.DataFrame(best_rows, columns=[
        "chip", "best_gameweek_in_horizon", "best_uplift_in_horizon"
    ])

    current = opportunities.loc[opportunities["gameweek"].eq(next_gameweek)].iloc[0]
    comparison_rows = []
    for chip_name, column in uplift_columns.items():
        best_match = best_opportunities.loc[best_opportunities["chip"].eq(chip_name)]
        best_uplift = float(best_match.iloc[0]["best_uplift_in_horizon"]) if not best_match.empty else np.nan
        current_uplift = float(current[column]) if pd.notna(current[column]) else np.nan
        threshold = float(CHIP_MINIMUM_UPLIFT[chip_name])

        # v2.6 P19 — the wildcard uplift is on a different scale from the others.
        #
        # triple_captain_uplift, bench_boost_uplift and the free-hit uplift are all
        # SINGLE-GAMEWEEK quantities, so their 7.0 / 9.0 / 8.0 bars are per-gameweek
        # and correct at any horizon. wildcard_horizon_uplift is a DISCOUNTED SUM
        # over the remaining horizon (cell 146: horizon_score - baseline_horizon).
        #
        # The identical constants {7,9,8,10} appear as BACKTEST_CHIP_MINIMUM_UPLIFT
        # where BACKTEST_HORIZON = 6, so the wildcard's 10.0 was exercised against a
        # six-gameweek sum (discount weights sum to 5.431). The live policy
        # greedy_1gw_no_hits runs at horizon 1, so the same 10.0 is being applied to
        # a one-gameweek number — roughly 5.4x too strict.
        #
        # Rescaling it is NOT the right fix: at horizon 1 the wildcard "uplift" just
        # measures a one-week squad reshuffle and ignores that you keep the squad,
        # so a scaled bar would fire the wildcard on a few points of one-week gain.
        # The honest position is that a wildcard cannot be judged at horizon 1 at
        # all. It is therefore suppressed explicitly and visibly, instead of being
        # suppressed by accident via a mis-scaled constant — and the P18 expiry
        # override still forces it out before it can expire unused.
        _p19_suppressed = False
        if chip_name == "wildcard" and len(gameweeks_to_use) < int(
            globals().get("V26_WILDCARD_MIN_HORIZON", 3)
        ):
            _p19_suppressed = not expiry_pressure_at(next_gameweek)
            if _p19_suppressed:
                threshold = float("inf")
        ratio = (
            current_uplift / best_uplift
            if np.isfinite(current_uplift) and np.isfinite(best_uplift) and best_uplift > 1e-9
            else np.nan
        )
        # v2.6 P18 — expiry-pressure override, mirroring the backtest.
        #
        # The sequential backtest (cell 166) gates on
        #     clears_uplift = option["uplift"] >= threshold or expiry_pressure
        # but this weekly production path had no such override: `passes` required
        # the uplift threshold unconditionally. `expiry_pressure_at()` was used
        # only further down, where it can REMOVE a candidate (the coarse-lookahead
        # veto) and never admit one. Net effect: if no chip cleared its threshold,
        # no chip could ever be played — including in the final week before the
        # chips expire. That is the ~92 points/season "chips expire unused"
        # failure mode, and it was live.
        #
        # Under expiry pressure the structural-readiness gate downstream already
        # returns True (it contains `or expiry_pressure`), so admitting the chip
        # here is consistent with the rest of the chain rather than bypassing it.
        _p18_expiry = expiry_pressure_at(next_gameweek)
        passes = (
            chip_is_eligible(chip_name, next_gameweek)
            and np.isfinite(current_uplift)
            # Positive-uplift floor: playing a chip worth <= 0 is strictly
            # worse than letting it expire. FH/WC uplifts really can go
            # negative (an executed run showed wildcard -0.276), because the
            # chip squad is rebuilt at current market prices while the held
            # squad was bought at purchase prices.
            and (
                current_uplift >= threshold
                or (_p18_expiry and current_uplift > 0.0)
            )
            and (
                _p18_expiry
                or (np.isfinite(ratio) and ratio >= CHIP_CURRENT_VS_HORIZON_RATIO)
            )
        )
        comparison_rows.append({
            "chip": chip_name,
            "eligible_now": chip_is_eligible(chip_name, next_gameweek),
            "current_uplift": current_uplift,
            "minimum_uplift": threshold,
            "best_horizon_uplift": best_uplift,
            "current_vs_horizon_best": ratio,
            "passes_use_now_rule": passes,
            "normalised_strength": (
                current_uplift / threshold
                if np.isfinite(current_uplift) and threshold > 0 else np.nan
            ),
        })

    current_comparison = pd.DataFrame(comparison_rows)

    # v2.2 coarse chip option-value horizon.
    # Detailed xP remains limited to the transfer-planning horizon, but chip
    # timing also inspects fixture structure through the relevant chip expiry
    # (GW19 or GW38). A clearly stronger future blank/double can reserve a chip
    # unless today's opportunity is exceptional.
    current_comparison["coarse_future_best_gameweek"] = np.nan
    current_comparison["coarse_current_signal"] = np.nan
    current_comparison["coarse_future_signal"] = np.nan
    current_comparison["coarse_future_stronger"] = False

    if CHIP_COARSE_LOOKAHEAD_ENABLED:
        fixture_table = globals().get("fixtures", pd.DataFrame())
        detailed_end = int(max(gameweeks_to_use))
        half_end = (
            int(CHIP_FIRST_HALF_END_GW)
            if next_gameweek <= int(CHIP_FIRST_HALF_END_GW)
            else int(CHIP_SEASON_END_GW)
        )

        structural_rows = []
        if (
            isinstance(fixture_table, pd.DataFrame)
            and not fixture_table.empty
            and {"event", "team_h", "team_a"}.issubset(
                fixture_table.columns
            )
        ):
            n_teams = int(
                max(
                    fixture_table["team_h"].nunique(),
                    fixture_table["team_a"].nunique(),
                )
            )
            for gw in range(next_gameweek, half_end + 1):
                gw_fx = fixture_table.loc[
                    pd.to_numeric(
                        fixture_table["event"],
                        errors="coerce",
                    ).eq(gw)
                ]
                counts = pd.concat(
                    [
                        gw_fx["team_h"],
                        gw_fx["team_a"],
                    ],
                    ignore_index=True,
                ).value_counts()
                teams_with_fixture = int((counts > 0).sum())
                blank_teams = max(0, n_teams - teams_with_fixture)
                double_teams = int((counts > 1).sum())
                structural_rows.append({
                    "gameweek": gw,
                    "blank_teams": blank_teams,
                    "double_teams": double_teams,
                })

        structural = pd.DataFrame(structural_rows)
        if not structural.empty:
            signal_map = {
                "triple_captain": lambda row: float(row["double_teams"]),
                "bench_boost": lambda row: float(row["double_teams"]),
                "free_hit": lambda row: (
                    float(row["blank_teams"])
                    + 2.0 * float(row["double_teams"])
                ),
                "wildcard": lambda row: (
                    0.5 * float(row["blank_teams"])
                    + 0.5 * float(row["double_teams"])
                ),
            }

            for row_index, row in current_comparison.iterrows():
                chip = str(row["chip"])
                signal_fn = signal_map[chip]
                current_shape = structural.loc[
                    structural["gameweek"].eq(next_gameweek)
                ]
                current_signal = (
                    signal_fn(current_shape.iloc[0])
                    if not current_shape.empty
                    else 0.0
                )

                future_shape = structural.loc[
                    structural["gameweek"].gt(detailed_end)
                ].copy()
                if not future_shape.empty:
                    future_shape["_signal"] = future_shape.apply(
                        signal_fn,
                        axis=1,
                    )
                    best_future = future_shape.sort_values(
                        ["_signal", "gameweek"],
                        ascending=[False, True],
                    ).iloc[0]
                    future_signal = float(best_future["_signal"])
                    future_gw = int(best_future["gameweek"])
                else:
                    future_signal = 0.0
                    future_gw = np.nan

                stronger = bool(
                    future_signal > current_signal + 1e-12
                )
                current_comparison.at[
                    row_index,
                    "coarse_future_best_gameweek",
                ] = future_gw
                current_comparison.at[
                    row_index,
                    "coarse_current_signal",
                ] = current_signal
                current_comparison.at[
                    row_index,
                    "coarse_future_signal",
                ] = future_signal
                current_comparison.at[
                    row_index,
                    "coarse_future_stronger",
                ] = stronger

                # Do not spend a structurally sensitive chip into an obviously
                # stronger future fixture shape unless today's xP edge is
                # exceptional or expiry pressure makes waiting unsafe.
                if (
                    chip in {"triple_captain", "bench_boost", "free_hit"}
                    and stronger
                    and not expiry_pressure_at(next_gameweek)
                ):
                    exceptional = float(
                        WEEKLY_CHIP_EXCEPTIONAL_UPLIFT.get(
                            chip,
                            np.inf,
                        )
                    )
                    current_uplift = pd.to_numeric(
                        row["current_uplift"],
                        errors="coerce",
                    )
                    if (
                        not np.isfinite(current_uplift)
                        or float(current_uplift) < exceptional
                    ):
                        current_comparison.at[
                            row_index,
                            "passes_use_now_rule",
                        ] = False

    # v2.6 P20 — order forced chips by what is lost by waiting, not by
    # normalised_strength.
    #
    # At horizon 1 evaluate_wildcard_at and evaluate_free_hit_at solve the same
    # one-gameweek problem, so their uplifts are identical by construction
    # (confirmed in production: both 3.8910851937150426). normalised_strength is
    # uplift / threshold, so the wildcard's larger bar (10.0 vs 8.0) always sorts
    # it BELOW the free hit — and with P19 suppressing it the ratio is 0.0, so it
    # sorts dead last. Under an expiry cascade that schedules the wildcard into the
    # final week before the chip set resets, which is the worst possible week for
    # the one chip whose value scales with the number of weeks remaining.
    #
    # Only under expiry pressure does ordering matter (otherwise a single chip
    # qualifies on merit). There, play the wildcard first: it buys a better squad
    # for every remaining week, and it makes the single-gameweek chips that follow
    # worth more. Normal merit-based ordering is untouched.
    _p20_forced = expiry_pressure_at(next_gameweek)
    candidates_now = current_comparison.loc[
        current_comparison["passes_use_now_rule"]
    ].copy()
    if _p20_forced and len(candidates_now) > 1:
        candidates_now["_p20_priority"] = (
            candidates_now["chip"].eq("wildcard").astype(int)
        )
        candidates_now = candidates_now.sort_values(
            ["_p20_priority", "normalised_strength"], ascending=[False, False]
        )
    else:
        candidates_now = candidates_now.sort_values(
            "normalised_strength", ascending=False
        )

    if candidates_now.empty:
        recommendation = {
            "gameweek": next_gameweek,
            "action": "Save all chips",
            "chip": None,
            "reason": (
                "No available chip clears both the configured uplift threshold "
                "and the within-horizon timing test."
            ),
        }
    else:
        selected = candidates_now.iloc[0]
        recommendation = {
            "gameweek": next_gameweek,
            "action": "Play " + str(selected["chip"]).replace("_", " ").title(),
            "chip": str(selected["chip"]),
            "expected_uplift": float(selected["current_uplift"]),
            "reason": (
                "The chip clears its uplift threshold and is sufficiently close "
                "to its best opportunity in the current horizon."
            ),
        }

    return {
        "opportunities": opportunities,
        "best_opportunities": best_opportunities,
        "current_comparison": current_comparison,
        "recommendation": recommendation,
        "details": details,
        "baseline_rows": baseline_rows,
        "fixture_summary": fixture_summary,
        "runtime": {
            "lazy": bool(WEEKLY_LAZY_CHIP_EVALUATION),
            "expensive_squad_solves": int(expensive_solve_count),
            "skipped_future_expensive_chips": skipped_future_expensive_chips,
            "seconds": float(time.perf_counter() - evaluation_started),
        },
    }

def weekly_format_lineup(
    decision: dict[str, object],
    fixture_level_predictions: pd.DataFrame,
    gameweek: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Format starters and substitutes with fixtures and roles."""
    squad_projection = decision[
        "squad_projection"
    ].copy()
    squad_projection["player_id"] = (
        squad_projection["player_id"].astype(int)
    )

    fixture_rows = fixture_level_predictions.loc[
        fixture_level_predictions["gameweek"].eq(
            int(gameweek)
        ),
        [
            "player_id",
            "opponent_short_name",
            "venue",
        ],
    ].copy()
    fixture_rows["fixture_label"] = (
        fixture_rows[
            "opponent_short_name"
        ].fillna("BLANK").astype(str)
        + " ("
        + fixture_rows["venue"].fillna("-").astype(str)
        + ")"
    )
    fixture_labels = (
        fixture_rows
        .groupby("player_id")["fixture_label"]
        .agg(", ".join)
        .rename("fixtures")
        .reset_index()
    )
    squad_projection = squad_projection.merge(
        fixture_labels,
        on="player_id",
        how="left",
    )
    squad_projection["fixtures"] = (
        squad_projection["fixtures"]
        .fillna("BLANK")
    )

    squad_projection["role"] = ""
    squad_projection.loc[
        squad_projection["player_id"].eq(
            int(decision["captain_id"])
        ),
        "role",
    ] = "C"
    squad_projection.loc[
        squad_projection["player_id"].eq(
            int(decision["vice_captain_id"])
        ),
        "role",
    ] = "VC"

    position_order = {
        "GK": 0,
        "DEF": 1,
        "MID": 2,
        "FWD": 3,
    }
    squad_projection["_position_order"] = (
        squad_projection["position"]
        .map(position_order)
        .fillna(9)
    )

    starters = (
        squad_projection.loc[
            squad_projection["is_starter"]
        ]
        .sort_values(
            [
                "_position_order",
                "expected_points",
            ],
            ascending=[True, False],
        )
        [
            [
                "position",
                "web_name",
                "team_short_name",
                "fixtures",
                "expected_minutes",
                "appearance_probability",
                "expected_points",
                "role",
            ]
        ]
        .reset_index(drop=True)
    )

    bench_rows = []
    player_lookup = (
        squad_projection
        .set_index("player_id")
        .to_dict("index")
    )
    for order, player_id in enumerate(
        decision["bench_order"],
        start=1,
    ):
        row = dict(player_lookup[int(player_id)])
        row["bench_slot"] = order
        bench_rows.append(row)

    goalkeeper_bench = squad_projection.loc[
        ~squad_projection["is_starter"]
        & squad_projection["position"].eq("GK")
    ]
    if not goalkeeper_bench.empty:
        row = goalkeeper_bench.iloc[0].to_dict()
        row["bench_slot"] = "GK"
        bench_rows.append(row)

    bench = pd.DataFrame(bench_rows)
    if not bench.empty:
        bench = bench[
            [
                "bench_slot",
                "position",
                "web_name",
                "team_short_name",
                "fixtures",
                "expected_minutes",
                "appearance_probability",
                "expected_points",
            ]
        ].reset_index(drop=True)

    return starters, bench



def weekly_validate_state_inputs() -> None:
    """Validate manually configured production inputs before optimisation."""
    if not np.isfinite(float(WEEKLY_MONEY_IN_BANK)) or float(WEEKLY_MONEY_IN_BANK) < 0:
        raise ValueError("WEEKLY_MONEY_IN_BANK must be a finite non-negative number.")

    free_transfers = int(WEEKLY_FREE_TRANSFERS)
    if free_transfers < 1 or free_transfers > int(MAX_BANKED_FREE_TRANSFERS):
        raise ValueError(
            "WEEKLY_FREE_TRANSFERS must be between 1 and "
            f"{MAX_BANKED_FREE_TRANSFERS}."
        )

    required_chips = {
        "triple_captain",
        "bench_boost",
        "free_hit",
        "wildcard",
    }
    if set(WEEKLY_AVAILABLE_CHIPS) != required_chips:
        raise ValueError(
            "WEEKLY_AVAILABLE_CHIPS must contain exactly: "
            f"{sorted(required_chips)}"
        )
    if not all(isinstance(value, (bool, np.bool_)) for value in WEEKLY_AVAILABLE_CHIPS.values()):
        raise TypeError("Every WEEKLY_AVAILABLE_CHIPS value must be True or False.")

    if WEEKLY_CHIP_DECISION_MODE not in {"conservative", "raw"}:
        raise ValueError(
            "WEEKLY_CHIP_DECISION_MODE must be 'conservative' or 'raw'."
        )


def weekly_resolve_policy_profile() -> tuple[str, str, bool]:
    """
    Resolve the weekly policy with explicit validation provenance.

    A profile name alone is never enough to imply validation. The Section 11
    selection must carry a positive `recommended_policy_profile_validated`
    flag generated from a complete, signed production scorecard.
    """
    configured = str(WEEKLY_POLICY_PROFILE)
    validated_profile = globals().get("recommended_policy_profile")
    validated_flag = bool(
        globals().get(
            "recommended_policy_profile_validated",
            False,
        )
    )
    validation_source = str(
        globals().get(
            "recommended_policy_profile_source",
            "no Section 11 validation provenance",
        )
    )

    if configured != "auto":
        if configured not in WEEKLY_POLICY_PROFILES:
            raise ValueError(
                f"Unknown WEEKLY_POLICY_PROFILE: {configured!r}"
            )
        scorecard_profiles = (
            set(
                walk_forward_strategy_scorecard[
                    "profile"
                ].astype(str)
            )
            if (
                isinstance(
                    globals().get(
                        "walk_forward_strategy_scorecard"
                    ),
                    pd.DataFrame,
                )
                and not walk_forward_strategy_scorecard.empty
            )
            else set()
        )
        explicit_validated = bool(
            globals().get(
                "walk_forward_policy_validation_available",
                False,
            )
            and configured in scorecard_profiles
        )
        return (
            configured,
            (
                "explicit configuration; profile present in validated "
                "Section 11 scorecard"
                if explicit_validated
                else "explicit configuration; not validated by current Section 11 scorecard"
            ),
            explicit_validated,
        )

    if (
        validated_flag
        and validated_profile in WEEKLY_POLICY_PROFILES
    ):
        return (
            str(validated_profile),
            f"validated Section 11 walk-forward selection — {validation_source}",
            True,
        )

    fallback = str(WEEKLY_POLICY_FALLBACK)
    if fallback not in WEEKLY_POLICY_PROFILES:
        raise ValueError(
            f"Unknown WEEKLY_POLICY_FALLBACK: {fallback!r}"
        )
    return (
        fallback,
        "fallback — no production-valid Section 11 policy scorecard",
        False,
    )


def weekly_apply_chip_readiness_gate(
    chip_results: dict[str, object],
    next_gameweek: int,
    gameweeks_to_use: list[int],
    normal_transfer_count: int,
    normal_hit_cost: float,
) -> dict[str, object]:
    """
    Prevent a short explicit horizon from spending a chip prematurely.

    The raw chip tables are retained. Only the use-now recommendation is
    changed when the current opportunity lacks a structural or exceptional
    reason to act.
    """
    raw = dict(chip_results["recommendation"])
    result = dict(chip_results)
    result["raw_recommendation"] = raw

    chip_name = raw.get("chip")
    if chip_name is None or WEEKLY_CHIP_DECISION_MODE == "raw":
        result["recommendation"] = raw
        result["readiness_gate"] = {
            "passed": chip_name is not None,
            "reason": "Raw chip policy was used." if chip_name else raw.get("reason"),
        }
        return result

    opportunities = chip_results["opportunities"]
    current_match = opportunities.loc[
        opportunities["gameweek"].eq(int(next_gameweek))
    ]
    current_shape = current_match.iloc[0] if not current_match.empty else pd.Series(dtype=float)

    blank_value = pd.to_numeric(
        current_shape.get("blank_teams", 0),
        errors="coerce",
    )
    double_value = pd.to_numeric(
        current_shape.get("double_or_more_teams", 0),
        errors="coerce",
    )
    blank_teams = int(blank_value) if pd.notna(blank_value) else 0
    double_teams = int(double_value) if pd.notna(double_value) else 0

    expiry = (
        FIRST_CHIP_SET_EXPIRY_GAMEWEEK
        if int(next_gameweek) <= FIRST_CHIP_SET_EXPIRY_GAMEWEEK
        else SECOND_CHIP_SET_EXPIRY_GAMEWEEK
    )
    remaining_weeks = max(1, int(expiry) - int(next_gameweek) + 1)
    remaining_available_chips = sum(bool(v) for v in WEEKLY_AVAILABLE_CHIPS.values())
    expiry_pressure = remaining_available_chips >= remaining_weeks

    comparison = chip_results["current_comparison"]
    chip_row = comparison.loc[comparison["chip"].eq(chip_name)]
    current_uplift = (
        float(chip_row.iloc[0]["current_uplift"])
        if not chip_row.empty
        else float("nan")
    )
    exceptional = (
        np.isfinite(current_uplift)
        and current_uplift >= float(WEEKLY_CHIP_EXCEPTIONAL_UPLIFT[chip_name])
    )

    structural, structural_reason = chip_structural_readiness(
        chip_name,
        current_uplift=current_uplift,
        blank_teams=blank_teams,
        double_teams=double_teams,
        expiry_pressure=expiry_pressure,
        normal_transfer_count=normal_transfer_count,
        normal_hit_cost=normal_hit_cost,
        exceptional_thresholds=WEEKLY_CHIP_EXCEPTIONAL_UPLIFT,
    )

    if WEEKLY_CHIP_REQUIRE_STRUCTURAL_TRIGGER and not structural:
        result["recommendation"] = {
            "gameweek": int(next_gameweek),
            "action": "Save all chips",
            "chip": None,
            "candidate_chip": chip_name,
            "candidate_uplift": current_uplift,
            "reason": (
                f"{chip_name.replace('_', ' ').title()} is the strongest current "
                "six-Gameweek candidate, but the explicit horizon does not show "
                f"the required {structural_reason}. Preserve the chip and rerun "
                "before the next deadline."
            ),
        }
        result["readiness_gate"] = {
            "passed": False,
            "reason": result["recommendation"]["reason"],
        }
    else:
        result["recommendation"] = raw
        result["readiness_gate"] = {
            "passed": True,
            "reason": structural_reason,
        }

    return result


def weekly_decision_confidence(
    alternatives: pd.DataFrame,
    initial_squad_mode: bool,
) -> tuple[str, float | None]:
    """Return an interpretable confidence label from the objective gap."""
    if initial_squad_mode or alternatives.empty or len(alternatives) < 2:
        return "Not assessed", None

    ordered = alternatives.sort_values("path_objective", ascending=False)
    gap = float(ordered.iloc[0]["path_objective"] - ordered.iloc[1]["path_objective"])
    if gap >= 2.0:
        label = "High"
    elif gap >= 0.75:
        label = "Medium"
    else:
        label = "Low"
    return label, gap



def weekly_opening_squad_confidence(
    alternatives: pd.DataFrame,
) -> tuple[str, float | None]:
    """Confidence from the gap between materially different opening squads."""
    if alternatives is None or alternatives.empty or len(alternatives) < 2:
        return "Not assessed", None
    ordered = alternatives.sort_values("objective", ascending=False)
    gap = float(ordered.iloc[0]["objective"] - ordered.iloc[1]["objective"])
    if gap >= 2.0:
        label = "High"
    elif gap >= 0.75:
        label = "Medium"
    else:
        label = "Low"
    return label, gap

In [ ]:
# GW completion / public-picks hotfix -----------------------------------------

# v2.2: duplicate autonomous_latest_completed_gameweek hotfix removed.
# The canonical implementation is defined in the preceding autonomous helpers cell.


# Verify before rerunning Section 12.
_debug_gw = autonomous_latest_completed_gameweek(gameweeks)

print("FPL_TEAM_ID:", FPL_TEAM_ID)
print("Detected completed GW:", _debug_gw)

if _debug_gw is not None:
    _debug_picks = get_fpl_data(
        f"entry/{int(FPL_TEAM_ID)}/event/{int(_debug_gw)}/picks/"
    )

    print(
        "Public picks returned:",
        len(_debug_picks.get("picks", [])),
    )

    _debug_entry = get_fpl_data(
        f"entry/{int(FPL_TEAM_ID)}/"
    )
    print(
        "Entry found:",
        bool(_debug_entry),
    )

In [ ]:
# Latest-Gameweek statistics audit -------------------------------------------

def weekly_latest_gameweek_stats_audit():
    # 1. Determine the latest GW that has genuinely been published/completed.
    published_gws = []

    # Normal bootstrap signal.
    if "finished" in gameweeks.columns:
        finished = gameweeks.loc[
            gameweeks["finished"].fillna(False),
            "id",
        ]
        if not finished.empty:
            published_gws.append(int(finished.max()))

    # More robust manager-history signal.
    if FPL_TEAM_ID is not None:
        history = get_fpl_data(
            f"entry/{int(FPL_TEAM_ID)}/history/"
        )
        history_gws = [
            int(row["event"])
            for row in history.get("current", [])
            if row.get("event") is not None
        ]
        if history_gws:
            published_gws.append(max(history_gws))

    latest_completed_gw = (
        max(published_gws)
        if published_gws
        else None
    )

    # 2. Determine the latest GW actually present in the model's
    # current-season player history.
    if (
        "current_season_player_gameweeks" not in globals()
        or current_season_player_gameweeks.empty
    ):
        latest_stats_gw = None
        latest_stats_rows = 0
    else:
        latest_stats_gw = int(
            current_season_player_gameweeks[
                "gameweek"
            ].max()
        )

        latest_stats_rows = int(
            current_season_player_gameweeks[
                "gameweek"
            ].eq(latest_stats_gw).sum()
        )

    # 3. The newest completed GW must be represented in the history
    # that generated the live forecasts.
    latest_used = (
        latest_completed_gw is None
        or (
            latest_stats_gw is not None
            and latest_stats_gw >= latest_completed_gw
        )
    )

    return {
        "latest_completed_gameweek": latest_completed_gw,
        "latest_stats_gameweek_loaded": latest_stats_gw,
        "latest_stats_rows": latest_stats_rows,
        "latest_gameweek_stats_used": bool(latest_used),
    }


weekly_gameweek_data_audit = (
    weekly_latest_gameweek_stats_audit()
)

print("=" * 72)
print("DATA RECENCY AUDIT")
print("=" * 72)

print(
    "Latest completed/published GW:",
    weekly_gameweek_data_audit[
        "latest_completed_gameweek"
    ],
)
print(
    "Latest GW in player-stat history:",
    weekly_gameweek_data_audit[
        "latest_stats_gameweek_loaded"
    ],
)
print(
    "Rows from latest loaded GW:",
    weekly_gameweek_data_audit[
        "latest_stats_rows"
    ],
)
print(
    "Latest GW incorporated:",
    (
        "YES ✓"
        if weekly_gameweek_data_audit[
            "latest_gameweek_stats_used"
        ]
        else "NO ✗"
    ),
)

if not weekly_gameweek_data_audit[
    "latest_gameweek_stats_used"
]:
    raise RuntimeError(
        "The latest completed FPL Gameweek has NOT been "
        "incorporated into the player-history inputs used by "
        "the live forecasts. Rerun the current-season history "
        "and live projection sections before using Section 12. "
        f"Audit: {weekly_gameweek_data_audit}"
    )

def weekly_early_season_shrinkage_audit():
    required_player_columns = {
        "early_season_player_shrinkage_active",
        "early_season_player_current_weight",
    }
    required_team_columns = {
        "team_early_season_shrinkage_active",
        "team_early_season_current_weight",
    }

    player_ready = (
        isinstance(
            globals().get("current_as_of_history"),
            pd.DataFrame,
        )
        and required_player_columns.issubset(
            current_as_of_history.columns
        )
    )
    team_ready = (
        isinstance(
            globals().get("live_team_as_of"),
            pd.DataFrame,
        )
        and required_team_columns.issubset(
            live_team_as_of.columns
        )
    )

    active_players = (
        int(
            current_as_of_history[
                "early_season_player_shrinkage_active"
            ].fillna(False).sum()
        )
        if player_ready else 0
    )
    active_teams = (
        int(
            live_team_as_of[
                "team_early_season_shrinkage_active"
            ].fillna(False).sum()
        )
        if team_ready else 0
    )

    return {
        "enabled": bool(EARLY_SEASON_SHRINKAGE_ENABLED),
        "live_feature_signature": LIVE_FEATURE_SIGNATURE,
        "player_layer_ready": bool(player_ready),
        "team_layer_ready": bool(team_ready),
        "active_sparse_players": active_players,
        "active_sparse_teams": active_teams,
        "ready": bool(
            (not EARLY_SEASON_SHRINKAGE_ENABLED)
            or (player_ready and team_ready)
        ),
    }


weekly_early_season_audit = weekly_early_season_shrinkage_audit()

print("=" * 72)
print("EARLY-SEASON STABILISATION AUDIT")
print("=" * 72)
print("Enabled:", weekly_early_season_audit["enabled"])
print(
    "Player/team layers ready:",
    weekly_early_season_audit["player_layer_ready"],
    "/",
    weekly_early_season_audit["team_layer_ready"],
)
print(
    "Sparse players/teams actively shrunk:",
    weekly_early_season_audit["active_sparse_players"],
    "/",
    weekly_early_season_audit["active_sparse_teams"],
)
print(
    "Live feature signature:",
    weekly_early_season_audit["live_feature_signature"],
)

if (
    EARLY_SEASON_STRICT_AUDIT
    and not weekly_early_season_audit["ready"]
):
    raise RuntimeError(
        "The early-season stabilisation layer is enabled but the live "
        "player/team shrinkage tables were not generated. Restart the kernel "
        "and rerun the notebook before using Section 12."
    )


### v2.2.7 release regression guards

These lightweight guards run before the autonomous weekly manager. They protect
the five changes that materially affect v2.2.7 production behaviour: fair policy
baselines, bank-aware transfer selection, sparse defensive-action shrinkage,
smooth sparse-player evidence decay, and abbreviated external-prior matching.


In [ ]:
# v2.2.7 lightweight regression guards ---------------------------------------
# 1) Historical and weekly policy catalogs must describe the same candidates.
assert set(WALK_FORWARD_ACTIVE_PROFILES) == set(WEEKLY_POLICY_PROFILES)
assert set(WALK_FORWARD_ACTIVE_PROFILES) == set(WALK_FORWARD_POLICY_PROFILES)
assert "greedy_1gw_bank_aware" in WALK_FORWARD_ACTIVE_PROFILES
assert bool(WALK_FORWARD_POLICY_PROFILES["greedy_1gw_bank_aware"]["bank_aware"])
assert np.isclose(
    float(
        WALK_FORWARD_POLICY_PROFILES[
            "greedy_1gw_bank_aware"
        ]["min_gain_per_extra_transfer"]
    ),
    float(BANK_AWARE_MIN_GAIN_PER_EXTRA_TRANSFER),
)

# 2) Marginal transfer gate must bank a weak second FT but preserve a strong
#    first FT.  The synthetic objective gaps mirror the intended semantics.
def _v227_state(transfer_count: int, objective: float) -> dict[str, object]:
    return {
        "final_objective": float(objective),
        "cumulative_hits": 0.0,
        "path": [{
            "transfer_count": int(transfer_count),
            "transfers": tuple(
                (100 + index, 200 + index)
                for index in range(int(transfer_count))
            ),
        }],
    }

_v227_transfer_result = {
    "final_beam": [
        _v227_state(0, 7.50),
        _v227_state(1, 9.00),
        _v227_state(2, 10.00),
    ]
}
_v227_selected, _v227_gate = select_transfer_state_with_marginal_gate(
    _v227_transfer_result,
    allow_hits=False,
    min_gain_per_extra_transfer=1.25,
)
assert int(_v227_selected["path"][0]["transfer_count"]) == 1
assert bool(_v227_gate["applied"])

# 3) Static baseline fairness must pass like-for-like rows and reject a
#    deliberately corrupted comparator.
_v227_fair = pd.DataFrame([
    {
        "holdout_season": "A",
        "profile": "p1",
        "initial_squad_signature": "s",
        "static_benchmark_signature": "b",
        "static_points": 100.0,
        "shared_opening_horizon": POLICY_SHARED_OPENING_HORIZON,
    },
    {
        "holdout_season": "A",
        "profile": "p2",
        "initial_squad_signature": "s",
        "static_benchmark_signature": "b",
        "static_points": 100.0,
        "shared_opening_horizon": POLICY_SHARED_OPENING_HORIZON,
    },
])
assert policy_baseline_fairness_status(_v227_fair)[0]
_v227_unfair = _v227_fair.copy()
_v227_unfair.loc[1, "static_points"] = 99.0
assert not policy_baseline_fairness_status(_v227_unfair)[0]

# 4) Sparse defensive-action evidence must shrink more than established
#    evidence for the same raw action rate.
_v227_dc = pd.DataFrame({
    "model_position": ["DEF", "DEF"],
    "tackles_per90_last_5_gws": [5.0, 5.0],
    "cbi_per90_last_5_gws": [10.0, 10.0],
    "career_tackles_per90": [5.0, 5.0],
    "career_cbi_per90": [10.0, 10.0],
    "prior_career_fixture_count": [3.0, 100.0],
    "prior_season_fixture_count": [3.0, 30.0],
})
_v227_dc_rate = estimate_defensive_action_rate(
    _v227_dc,
    pd.Series({"DEF": 8.0}),
)
assert float(_v227_dc_rate.iloc[0]) < float(_v227_dc_rate.iloc[1])
assert 8.0 < float(_v227_dc_rate.iloc[0]) < 15.0

# 5) Sparse-player market evidence must decay smoothly and be zero at/after
#    the configured fade horizon.
_v227_counts = pd.Series([0.0, 1.0, 3.0, 8.0])
_v227_fade = (
    1.0
    - _v227_counts.div(float(SPARSE_PLAYER_MARKET_PRIOR_FADE_FIXTURES))
).clip(0.0, 1.0)
assert np.all(np.diff(_v227_fade.to_numpy()) <= 1e-12)
assert np.isclose(float(_v227_fade.iloc[-1]), 0.0)

# 6) Unique initial+surname matching must resolve the Khalaili-style
#    abbreviated-name case without weakening general fuzzy-match thresholds.
_v227_external = pd.DataFrame([
    {
        "external_name_key": "anan khalaili",
        "external_position": "DEF",
        "external_player": "Anan Khalaili",
    },
    {
        "external_name_key": "john smith",
        "external_position": "DEF",
        "external_player": "John Smith",
    },
])
_v227_match, _v227_match_score = _initial_surname_external_match(
    "a khalaili",
    "DEF",
    _v227_external,
)
assert _v227_match is not None
assert str(_v227_match["external_player"]) == "Anan Khalaili"
assert float(_v227_match_score) >= float(
    EXTERNAL_INITIAL_SURNAME_MATCH_THRESHOLD
)

print("v2.2.7 fair-policy / sparse-evidence regression guards: PASS")


In [ ]:
# Run the weekly live decision ------------------------------------------------
weekly_runtime_started = time.perf_counter()
weekly_runtime_profile = {}
required_weekly_objects = [
    "players",
    "fixtures",
    "gameweeks",
    "live_expected_points",
    "live_gameweek_expected_points",
    "current_new_player_audit",
    "prepare_optimisation_inputs",
    "optimise_transfer_path_adaptive",
    "exact_squad_gameweek_decision",
    "optimise_chip_squad",
]
missing_weekly_objects = [
    name
    for name in required_weekly_objects
    if name not in globals()
]
if missing_weekly_objects:
    raise RuntimeError(
        "Run the notebook through Section 10 before Section 12. "
        f"Missing objects: {missing_weekly_objects}"
    )

weekly_autonomous_ledger = None
weekly_autonomous_state_warnings = []
if AUTONOMOUS_MODE:
    weekly_autonomous_ledger, weekly_autonomous_state_warnings = (
        autonomous_resolve_weekly_state(players, gameweeks)
    )
    if AUTONOMOUS_REQUIRE_VALIDATION_ARTIFACTS and (
        xp_calibration_cache is None or premium_attacker_cache is None
    ):
        raise RuntimeError(
            "Autonomous mode requires the current signed validation artifacts. "
            "Run Section 11 once with RUN_FULL_POLICY_BACKTEST=True, restart "
            "the kernel, then rerun Sections 1–10 and Section 12."
        )
    current_artifact_state = {
        "forecast_signature": FORECAST_CACHE_SIGNATURE,
        "live_feature_signature": LIVE_FEATURE_SIGNATURE,
        "xp_artifact_id": (
            xp_calibration_cache.get("artifact_id") if xp_calibration_cache else None
        ),
        "premium_artifact_id": (
            premium_attacker_cache.get("artifact_id") if premium_attacker_cache else None
        ),
    }
    if any(
        LIVE_VALIDATION_ARTIFACT_STATE.get(key) != value
        for key, value in current_artifact_state.items()
    ):
        raise RuntimeError(
            "Validation artifacts changed after the live projections were generated. "
            "Restart the kernel and rerun Sections 1–10 before Section 12."
        )
    if (
        not AUTONOMOUS_ALLOW_MANUAL_PROJECTION_OVERRIDES
        and (WEEKLY_CURRENT_GW_PROJECTION_MULTIPLIERS or WEEKLY_HORIZON_PROJECTION_MULTIPLIERS)
    ):
        raise RuntimeError(
            "Manual projection overrides are disabled in autonomous mode."
        )

weekly_validate_state_inputs()
(
    weekly_policy_name,
    weekly_policy_source,
    weekly_policy_validated,
) = weekly_resolve_policy_profile()
weekly_policy = dict(
    WEEKLY_POLICY_PROFILES[weekly_policy_name]
)
weekly_policy_bank_aware = bool(
    weekly_policy.get("bank_aware", False)
)
weekly_effective_marginal_gate = bool(
    weekly_policy_bank_aware
    or WEEKLY_TRANSFER_ROBUSTNESS_GATE
)
weekly_effective_min_gain_per_extra_transfer = float(
    weekly_policy.get(
        "min_gain_per_extra_transfer",
        WEEKLY_MIN_OBJECTIVE_GAIN_PER_EXTRA_TRANSFER,
    )
    if weekly_policy_bank_aware
    else WEEKLY_MIN_OBJECTIVE_GAIN_PER_EXTRA_TRANSFER
)

weekly_freshness = weekly_projection_freshness_audit(
    players,
    fixtures,
    gameweeks,
    live_gameweek_expected_points,
)


if "weekly_gameweek_data_audit" not in globals():
    weekly_gameweek_data_audit = weekly_latest_gameweek_stats_audit()

if not weekly_gameweek_data_audit[
    "latest_gameweek_stats_used"
]:
    raise RuntimeError(
        "The latest completed/published FPL Gameweek has not been "
        "incorporated into the player-history inputs. "
        f"Audit: {weekly_gameweek_data_audit}"
    )

if "weekly_early_season_audit" not in globals():
    weekly_early_season_audit = weekly_early_season_shrinkage_audit()

if (
    EARLY_SEASON_STRICT_AUDIT
    and not weekly_early_season_audit["ready"]
):
    raise RuntimeError(
        "Early-season stabilisation is not ready. "
        f"Audit: {weekly_early_season_audit}"
    )

_all_comp_status_payload = dict(
    globals().get(
        "live_all_comp_schedule_status",
        {},
    )
)
_fbref_coverage_payload = dict(
    globals().get(
        "current_fbref_live_coverage_dict",
        {},
    )
)

weekly_live_context_audit = {
    "all_comp_enabled": bool(LIVE_ALL_COMP_WORKLOAD_ENABLED),
    "all_comp_status": (
        _all_comp_status_payload.get("status", "UNAVAILABLE")
        if LIVE_ALL_COMP_WORKLOAD_ENABLED
        else "OFF"
    ),
    "all_comp_fixture_rows": int(
        len(live_all_comp_schedule)
        if isinstance(globals().get("live_all_comp_schedule"), pd.DataFrame)
        else 0
    ),
    "all_comp_unique_events": int(
        _all_comp_status_payload.get("unique_events", 0)
    ),
    "all_comp_successful_requests": int(
        _all_comp_status_payload.get("successful_requests", 0)
    ),
    "all_comp_failed_requests": int(
        _all_comp_status_payload.get("failed_requests", 0)
    ),
    "all_comp_total_requests": int(
        _all_comp_status_payload.get("total_requests", 0)
    ),
    "all_comp_circuit_open": bool(
        _all_comp_status_payload.get("circuit_open", False)
    ),
    "all_comp_remaining_requests_skipped": (
        _all_comp_status_payload.get("remaining_requests_skipped")
    ),
    "all_comp_failure_cache_active": bool(
        _all_comp_status_payload.get("failure_cache_active", False)
    ),
    "all_comp_failure_cache_expires_at_utc": (
        _all_comp_status_payload.get("failure_cache_expires_at_utc")
    ),
    "all_comp_player_minute_rows": int(
        len(live_all_comp_player_minutes)
        if isinstance(globals().get("live_all_comp_player_minutes"), pd.DataFrame)
        else 0
    ),
    "fbref_enabled": bool(LIVE_CURRENT_FBREF_ENABLED),
    "fbref_status": (
        str(globals().get("current_fbref_live_status", "UNAVAILABLE"))
        if LIVE_CURRENT_FBREF_ENABLED
        else "OFF"
    ),
    "fbref_matched_players": int(
        len(current_fbref_live_context)
        if isinstance(globals().get("current_fbref_live_context"), pd.DataFrame)
        else 0
    ),
    "fbref_coverage": _fbref_coverage_payload,
    "fbref_refresh_source": str(
        globals().get(
            "current_fbref_refresh_audit",
            {},
        ).get("source", "unknown")
    ),
    "fbref_refresh_fallback_used": bool(
        globals().get(
            "current_fbref_refresh_audit",
            {},
        ).get("fallback_used", False)
    ),
    "set_piece_enabled": bool(LIVE_SET_PIECE_ROLE_ENABLED),
}

if (
    LIVE_ALL_COMP_WORKLOAD_ENABLED
    and LIVE_ALL_COMP_STRICT
    and weekly_live_context_audit["all_comp_status"]
    != "ACTIVE"
):
    raise RuntimeError(
        "All-competition workload is strict but its live status is "
        f"{weekly_live_context_audit['all_comp_status']}."
    )
if (
    LIVE_CURRENT_FBREF_ENABLED
    and LIVE_CURRENT_FBREF_STRICT
    and weekly_live_context_audit["fbref_status"]
    != "ACTIVE"
):
    raise RuntimeError(
        "Current FBref is strict but its live status is "
        f"{weekly_live_context_audit['fbref_status']}."
    )
def _v26_triage_freshness(freshness, minutes_threshold=30.0):
    """Split API changes into material and immaterial, and refresh prices.

    Material   = an availability change (status / chance / news) for a player
                 the model projects at >= `minutes_threshold` next gameweek,
                 i.e. someone who could already be owned or could be bought.
    Immaterial = a price-only move, or an availability change for a player with
                 no projected role.

    Price moves are refreshed into the live frame rather than ignored, so
    affordability stays correct. A 0.1 move does not cross a calibration price
    band (5.0 / 6.5 / 8.0 / 10.0), so the applied calibration stays valid.
    """
    changed = freshness.get("changed_players") or []
    live = globals().get("live_expected_points")
    target_gw = freshness.get("projected_next_gameweek")

    minutes_by_player = {}
    if isinstance(live, pd.DataFrame) and "player_id" in live.columns:
        nxt = live[pd.to_numeric(live.get("gameweek"), errors="coerce") == target_gw]
        if nxt.empty:
            nxt = live
        minutes_by_player = nxt.groupby("player_id")["expected_minutes"].max().to_dict()

    material, immaterial, repriced = [], [], 0
    for row in changed:
        fields = set(row.get("changed_fields") or [])
        pid = row.get("player_id")
        mins = float(minutes_by_player.get(pid, 0.0) or 0.0)

        if "price" in fields and isinstance(live, pd.DataFrame) and \
                "player_id" in live.columns and "price" in live.columns:
            latest = row.get("latest_price")
            try:
                latest = float(latest)
            except (TypeError, ValueError):
                latest = None
            if latest is not None and np.isfinite(latest):
                mask = live["player_id"] == pid
                if mask.any():
                    live.loc[mask, "price"] = latest
                    repriced += 1

        if (fields & {"status", "chance", "news"}) and mins >= minutes_threshold:
            material.append((row, mins))
        else:
            immaterial.append((row, mins))

    return material, immaterial, repriced


if WEEKLY_STRICT_FRESHNESS_CHECK and not weekly_freshness["is_fresh"]:
    # These two can never be triaged away: the whole projection set is wrong.
    if weekly_freshness.get("latest_next_gameweek") != \
            weekly_freshness.get("projected_next_gameweek"):
        raise RuntimeError(
            "The gameweek rolled over while the notebook was running. "
            "Rerun the notebook through Section 10, then run Section 12. "
            "Freshness audit: " + str(weekly_freshness)
        )
    if weekly_freshness.get("changed_fixture_rows", 0):
        raise RuntimeError(
            "Fixture scheduling changed since the projections were built. "
            "Rerun the notebook through Section 10, then run Section 12. "
            "Changed fixtures: " + str(weekly_freshness.get("changed_fixtures"))
        )

    if globals().get("V26_ENABLED", False):
        _v26_material, _v26_immaterial, _v26_repriced = _v26_triage_freshness(
            weekly_freshness
        )
    else:
        _v26_material = [
            (r, float("nan"))
            for r in (weekly_freshness.get("changed_players") or [])
        ]
        _v26_immaterial, _v26_repriced = [], 0

    print(
        "[v2.6] FPL API moved since the projections were built: "
        + str(weekly_freshness.get("changed_player_rows"))
        + " player row(s)"
    )
    for _row, _mins in _v26_immaterial:
        print(
            "       immaterial: "
            + str(_row.get("player"))
            + " "
            + str(_row.get("changed_fields"))
            + " (projected " + format(_mins, ".0f") + " min)"
        )
    if _v26_repriced:
        print(
            "       refreshed " + str(_v26_repriced)
            + " price(s) into the live frame"
        )

    if _v26_material:
        for _row, _mins in _v26_material:
            print(
                "       MATERIAL: "
                + str(_row.get("player"))
                + " "
                + str(_row.get("changed_fields"))
                + " (projected " + format(_mins, ".0f") + " min)"
            )
        raise RuntimeError(
            "Availability changed for players the model projects to play. "
            "Rerun the notebook through Section 10, then run Section 12. "
            "Material changes: " + str([r for r, _ in _v26_material])
        )

    print("[v2.6] no material change - proceeding without a rerun")

available_live_gameweeks = sorted(
    live_gameweek_expected_points[
        "gameweek"
    ].astype(int).unique()
)
weekly_horizon = min(
    int(weekly_policy["horizon"]),
    len(available_live_gameweeks),
)
weekly_gameweeks = available_live_gameweeks[
    :weekly_horizon
]
if not weekly_gameweeks:
    raise RuntimeError(
        "No upcoming projected Gameweeks are available."
    )
weekly_next_gameweek = int(
    weekly_gameweeks[0]
)

weekly_adjusted_gameweek_xp, weekly_override_audit = (
    weekly_apply_projection_multipliers(
        live_gameweek_expected_points,
        players,
        weekly_next_gameweek,
        WEEKLY_CURRENT_GW_PROJECTION_MULTIPLIERS,
        WEEKLY_HORIZON_PROJECTION_MULTIPLIERS,
    )
)

weekly_candidates, weekly_projections = (
    prepare_optimisation_inputs(
        weekly_adjusted_gameweek_xp,
        players,
        weekly_gameweeks,
    )
)

weekly_squad_ids = weekly_resolve_squad(
    WEEKLY_CURRENT_SQUAD,
    players,
    weekly_next_gameweek,
)
weekly_purchase_prices = (
    weekly_resolve_price_mapping(
        WEEKLY_PURCHASE_PRICES,
        players,
    )
)
weekly_explicit_selling_prices = (
    weekly_resolve_price_mapping(
        WEEKLY_SELLING_PRICES,
        players,
    )
)

with weekly_runtime_context(
    weekly_policy,
    WEEKLY_AVAILABLE_CHIPS,
    WEEKLY_LAST_FREE_HIT_GAMEWEEK,
):
    if not weekly_squad_ids:
        (
            weekly_best_state,
            weekly_initial_squad_ids,
            weekly_initial_bank,
            weekly_opening_squad_alternatives,
            weekly_opening_bench_sensitivity,
        ) = weekly_build_opening_state(
            weekly_candidates,
            weekly_projections,
            weekly_gameweeks,
        )
        weekly_mode = "Initial squad selection"
        weekly_selling_prices = {
            int(row.player_id): float(row.price)
            for row in weekly_candidates.loc[
                weekly_candidates["player_id"].isin(
                    weekly_initial_squad_ids
                )
            ].itertuples(index=False)
        }
        weekly_price_warnings = []
        weekly_transfer_result = None
        weekly_transfer_alternatives = (
            pd.DataFrame([{
                "current_action": "Select optimised initial squad",
                "transfer_count": 0,
                "current_hit_cost": 0.0,
                "bank_after": weekly_initial_bank,
                "path_objective": weekly_best_state[
                    "final_objective"
                ],
            }])
        )
    else:
        weekly_mode = "Live in-season squad"
        weekly_opening_squad_alternatives = pd.DataFrame()
        weekly_opening_bench_sensitivity = pd.DataFrame()
        (
            weekly_selling_prices,
            weekly_price_warnings,
        ) = weekly_build_selling_prices(
            weekly_squad_ids,
            weekly_candidates,
            weekly_purchase_prices,
            weekly_explicit_selling_prices,
        )
        weekly_initial_squad_ids = list(
            weekly_squad_ids
        )
        weekly_initial_bank = float(
            WEEKLY_MONEY_IN_BANK
        )

        _weekly_transfer_started = time.perf_counter()
        weekly_transfer_result = optimise_transfer_path_adaptive(
            current_squad_ids=weekly_squad_ids,
            candidates=weekly_candidates,
            projections=weekly_projections,
            gameweeks_to_use=weekly_gameweeks,
            money_in_bank=WEEKLY_MONEY_IN_BANK,
            free_transfers=WEEKLY_FREE_TRANSFERS,
            current_selling_prices=weekly_selling_prices,
        )
        weekly_runtime_profile["adaptive_transfer_search"] = dict(
            weekly_transfer_result.get("adaptive_search_audit", {})
        )
        weekly_runtime_profile["base_transfer_search_seconds"] = float(
            time.perf_counter() - _weekly_transfer_started
        )
        _adaptive_audit = weekly_transfer_result.get(
            "adaptive_search_audit", {}
        )
        if _adaptive_audit:
            print(
                "Adaptive transfer search:",
                _adaptive_audit.get("selected_profile", "unknown"),
                "| confidence:",
                _adaptive_audit.get("confidence", "n/a"),
                "| escalated:",
                bool(_adaptive_audit.get("escalated", False)),
            )
            for _run in _adaptive_audit.get("runs", []):
                print(
                    "  ",
                    _run.get("profile"),
                    f"{float(_run.get('seconds', 0.0)):.1f}s",
                    "| first-action gap:",
                    (
                        f"{float(_run.get('first_action_gap')):.3f}"
                        if np.isfinite(_run.get("first_action_gap", np.nan))
                        else "n/a"
                    ),
                )
        weekly_transfer_result = (
            weekly_attach_exact_raw_action_subsets(
                transfer_result=weekly_transfer_result,
                current_squad_ids=weekly_squad_ids,
                candidates=weekly_candidates,
                projections=weekly_projections,
                gameweeks_to_use=weekly_gameweeks,
                money_in_bank=WEEKLY_MONEY_IN_BANK,
                free_transfers=WEEKLY_FREE_TRANSFERS,
                current_selling_prices=weekly_selling_prices,
                allow_hits=bool(
                    weekly_policy["allow_hits"]
                ),
            )
        )

        weekly_transfer_result = (
            weekly_attach_structural_unavailable_context(
                transfer_result=weekly_transfer_result,
                current_squad_ids=weekly_squad_ids,
                candidates=weekly_candidates,
                projections=weekly_projections,
            )
        )

        weekly_best_state = (
            weekly_choose_transfer_state(
                weekly_transfer_result,
                bool(weekly_policy["allow_hits"]),
                marginal_gate_enabled=weekly_effective_marginal_gate,
                min_gain_per_extra_transfer=(
                    weekly_effective_min_gain_per_extra_transfer
                ),
            )
        )
        weekly_transfer_alternatives = (
            weekly_transfer_action_comparison(
                weekly_transfer_result,
                weekly_candidates,
                bool(weekly_policy["allow_hits"]),
                selected_state=weekly_best_state,
            )
        )

    weekly_transfer_plan = weekly_state_plan_table(
        weekly_best_state,
        weekly_candidates,
    )
    _weekly_exact_started = time.perf_counter()
    (
        weekly_normal_squad,
        weekly_normal_decision,
    ) = weekly_exact_current_decision(
        weekly_best_state,
        weekly_candidates,
        weekly_projections,
    )

    weekly_runtime_profile["current_exact_decision_seconds"] = float(
        time.perf_counter() - _weekly_exact_started
    )
    _weekly_chip_started = time.perf_counter()
    weekly_first_action_for_chips = weekly_best_state["path"][0]
    weekly_chip_results = weekly_evaluate_chips(
        weekly_best_state,
        weekly_initial_squad_ids,
        weekly_initial_bank,
        weekly_candidates,
        weekly_projections,
        weekly_gameweeks,
        weekly_selling_prices,
        live_expected_points,
        teams,
        normal_transfer_count=int(weekly_first_action_for_chips["transfer_count"]),
        normal_hit_cost=float(weekly_first_action_for_chips["hit_cost"]),
    )
    weekly_runtime_profile["chip_evaluation_seconds"] = float(
        time.perf_counter() - _weekly_chip_started
    )

weekly_runtime_profile["subset_robustness_seconds"] = float(
    (weekly_transfer_result or {}).get(
        "robustness_subset_runtime_seconds", 0.0
    )
)
weekly_runtime_profile["total_seconds_so_far"] = float(
    time.perf_counter() - weekly_runtime_started
)

weekly_transfer_robustness = weekly_best_state.get(
    "_robustness_gate",
    {
        "enabled": False,
        "applied": False,
        "raw_transfer_count": None,
        "selected_transfer_count": None,
        "raw_objective": None,
        "selected_objective": float(weekly_best_state.get("final_objective", np.nan)),
        "objective_given_up": 0.0,
        "threshold_per_extra_transfer": None,
        "steps": [],
        "reason": "Not applicable to opening-squad selection.",
    },
)

weekly_first_path_row = (
    weekly_best_state["path"][0]
)
weekly_chip_results = weekly_apply_chip_readiness_gate(
    weekly_chip_results,
    weekly_next_gameweek,
    weekly_gameweeks,
    int(weekly_first_path_row["transfer_count"]),
    float(weekly_first_path_row["hit_cost"]),
)
weekly_chip_recommendation = (
    weekly_chip_results["recommendation"]
)
weekly_selected_chip = (
    weekly_chip_recommendation.get("chip")
)
weekly_normal_score_after_hits = float(
    weekly_normal_decision[
        "expected_total_score"
    ]
    - weekly_first_path_row["hit_cost"]
)

weekly_final_squad = weekly_normal_squad
weekly_final_decision = weekly_normal_decision
weekly_expected_score = (
    weekly_normal_score_after_hits
)

if weekly_selected_chip in {
    "free_hit",
    "wildcard",
}:
    weekly_chip_detail = (
        weekly_chip_results["details"][
            (
                weekly_selected_chip,
                weekly_next_gameweek,
            )
        ]
    )
    weekly_final_squad = (
        weekly_chip_detail["squad"].copy()
    )
    weekly_final_decision = (
        weekly_chip_detail["decision"]
    )
    weekly_expected_score = float(
        weekly_final_decision[
            "expected_total_score"
        ]
    )
elif weekly_selected_chip in {
    "triple_captain",
    "bench_boost",
}:
    current_chip_row = (
        weekly_chip_results[
            "current_comparison"
        ].loc[
            weekly_chip_results[
                "current_comparison"
            ]["chip"].eq(
                weekly_selected_chip
            )
        ].iloc[0]
    )
    weekly_expected_score = (
        weekly_normal_score_after_hits
        + float(
            current_chip_row[
                "current_uplift"
            ]
        )
    )

(
    weekly_starting_xi,
    weekly_bench,
) = weekly_format_lineup(
    weekly_final_decision,
    live_expected_points,
    weekly_next_gameweek,
)

weekly_name_lookup = (
    weekly_candidates
    .set_index("player_id")["web_name"]
    .astype(str)
    .to_dict()
)
weekly_captain = weekly_name_lookup.get(
    int(weekly_final_decision["captain_id"]),
    weekly_final_decision["captain_id"],
)
weekly_vice_captain = weekly_name_lookup.get(
    int(
        weekly_final_decision[
            "vice_captain_id"
        ]
    ),
    weekly_final_decision[
        "vice_captain_id"
    ],
)

weekly_captain_confidence = weekly_final_decision.get(
    "captain_confidence",
    "Not assessed",
)
weekly_captain_xp_gap = weekly_final_decision.get(
    "captain_xp_gap",
    np.nan,
)
weekly_captain_alternatives = weekly_final_decision.get(
    "captain_alternatives",
    pd.DataFrame(),
)
weekly_execution_policy_name = weekly_final_decision.get(
    "execution_policy_name",
    ACTIVE_EXECUTION_POLICY.get("name", "baseline"),
)

if weekly_mode == "Initial squad selection":
    weekly_primary_transfer_action = "Select optimised initial squad"
    weekly_transfer_plan = weekly_transfer_plan.copy()
    weekly_transfer_plan.loc[
        weekly_transfer_plan.index[0],
        "transfers",
    ] = "Initial squad selected"
else:
    weekly_primary_transfer_action = str(
        weekly_transfer_plan.iloc[0]["transfers"]
    )
if weekly_selected_chip in {
    "free_hit",
    "wildcard",
}:
    weekly_primary_action = (
        weekly_chip_recommendation["action"]
        + "; do not execute the normal-transfer path."
    )
elif weekly_selected_chip is not None:
    weekly_primary_action = (
        weekly_primary_transfer_action
        + "; "
        + weekly_chip_recommendation["action"]
    )
else:
    weekly_primary_action = (
        weekly_primary_transfer_action
    )

roll_match = weekly_transfer_alternatives.loc[
    weekly_transfer_alternatives[
        "current_action"
    ].eq("Roll")
] if not weekly_transfer_alternatives.empty else pd.DataFrame()
best_objective = float(
    weekly_best_state["final_objective"]
)
weekly_gain_vs_roll = (
    best_objective
    - float(roll_match.iloc[0]["path_objective"])
    if weekly_mode != "Initial squad selection" and not roll_match.empty
    else np.nan
)
weekly_confidence, weekly_objective_gap = weekly_decision_confidence(
    weekly_transfer_alternatives,
    weekly_mode == "Initial squad selection",
)
if (
    weekly_transfer_robustness.get("applied", False)
    and not weekly_policy_bank_aware
):
    # A legacy unvalidated production overlay still lowers confidence.
    # A bank-aware gate that is part of the selected/validated policy does not:
    # preserving a transfer is the policy's intended action, not an override.
    weekly_confidence = "Low"
    weekly_objective_gap = float(
        weekly_transfer_robustness.get("objective_given_up", np.nan)
    )
if weekly_mode == "Initial squad selection":
    weekly_confidence, weekly_objective_gap = weekly_opening_squad_confidence(
        weekly_opening_squad_alternatives
    )

deadline_matches = gameweeks.loc[
    gameweeks["id"].eq(
        weekly_next_gameweek
    )
]
weekly_deadline = (
    deadline_matches.iloc[0][
        "deadline_time_spain"
    ]
    if not deadline_matches.empty
    else pd.NaT
)

weekly_model_warnings = list(weekly_autonomous_state_warnings)

# v2.4 uncertainty-aware captain context. This does not replace the optimiser's
# expected-value choice; it prevents tiny xP gaps from being presented as strong
# evidence when historical forecast error is much larger than the edge.
weekly_forecast_uncertainty = pd.DataFrame()
weekly_captain_uncertainty_ratio = np.nan
if "forecast_uncertainty_p80" in live_expected_points.columns:
    _unc = live_expected_points.loc[
        live_expected_points["gameweek"].eq(int(weekly_next_gameweek)),
        ["player_id", "expected_points", "forecast_uncertainty_p80", "minutes_uncertainty_p80"],
    ].copy()
    if not _unc.empty:
        _unc["forecast_uncertainty_sq"] = pd.to_numeric(
            _unc["forecast_uncertainty_p80"], errors="coerce"
        ).fillna(0.0).pow(2)
        _unc["minutes_uncertainty_sq"] = pd.to_numeric(
            _unc["minutes_uncertainty_p80"], errors="coerce"
        ).fillna(0.0).pow(2)
        weekly_forecast_uncertainty = _unc.groupby("player_id", as_index=False).agg(
            expected_points=("expected_points", "sum"),
            forecast_uncertainty_sq=("forecast_uncertainty_sq", "sum"),
            minutes_uncertainty_sq=("minutes_uncertainty_sq", "sum"),
        )
        weekly_forecast_uncertainty["forecast_uncertainty_p80"] = np.sqrt(
            weekly_forecast_uncertainty.pop("forecast_uncertainty_sq")
        )
        weekly_forecast_uncertainty["minutes_uncertainty_p80"] = np.sqrt(
            weekly_forecast_uncertainty.pop("minutes_uncertainty_sq")
        )

        _starter_ids = {int(v) for v in weekly_final_decision.get("starter_ids", ())}
        _starters = weekly_forecast_uncertainty.loc[
            weekly_forecast_uncertainty["player_id"].astype(int).isin(_starter_ids)
        ].sort_values("expected_points", ascending=False)
        if len(_starters) >= 2:
            _cap_row = _starters.loc[
                _starters["player_id"].astype(int).eq(int(weekly_final_decision["captain_id"]))
            ]
            _other = _starters.loc[
                ~_starters["player_id"].astype(int).eq(int(weekly_final_decision["captain_id"]))
            ].head(1)
            if not _cap_row.empty and not _other.empty:
                _cap_row = _cap_row.iloc[0]
                _other = _other.iloc[0]
                _edge = float(_cap_row["expected_points"] - _other["expected_points"])
                _scale = float(np.hypot(
                    _cap_row["forecast_uncertainty_p80"],
                    _other["forecast_uncertainty_p80"],
                ))
                if _scale > 1e-9:
                    weekly_captain_uncertainty_ratio = _edge / _scale
                    if weekly_captain_uncertainty_ratio < float(V24_UNCERTAINTY_EDGE_RATIO_LOW):
                        weekly_captain_confidence = "Very low"
                        weekly_model_warnings.append(
                            "Captain xP edge is tiny relative to historical forecast uncertainty; "
                            "treat the leading captain options as effectively tied."
                        )
                    elif weekly_captain_uncertainty_ratio < float(V24_UNCERTAINTY_EDGE_RATIO_MEDIUM):
                        if weekly_captain_confidence == "High":
                            weekly_captain_confidence = "Medium"
                        weekly_model_warnings.append(
                            "Captain edge is small relative to historical forecast uncertainty."
                        )

# v2.5 role-state uncertainty context. This remains a warning/confidence aid;
# it never subtracts arbitrary points from a player.
if "v25_role_entropy" in live_expected_points.columns:
    _role_unc = (
        live_expected_points.loc[
            live_expected_points["gameweek"].eq(int(weekly_next_gameweek)),
            ["player_id", "v25_role_entropy", "v25_most_likely_role_probability"],
        ]
        .groupby("player_id", as_index=False)
        .agg(
            role_entropy=("v25_role_entropy", "max"),
            most_likely_role_probability=(
                "v25_most_likely_role_probability", "min"
            ),
        )
    )
    _starter_ids_for_role = {
        int(v) for v in weekly_final_decision.get("starter_ids", ())
    }
    _uncertain_starters = _role_unc.loc[
        _role_unc["player_id"].astype(int).isin(_starter_ids_for_role)
        & pd.to_numeric(
            _role_unc["role_entropy"], errors="coerce"
        ).ge(float(V25_ROLE_ENTROPY_HIGH))
    ]
    if not _uncertain_starters.empty:
        weekly_model_warnings.append(
            f"{len(_uncertain_starters)} selected starter(s) have high "
            "probabilistic role/minutes uncertainty. Treat small xP edges "
            "involving those players as non-decisive."
        )

if not weekly_policy_validated:
    weekly_model_warnings.append(
        "Transfer policy is running from the configured fallback because no "
        "production-valid Section 11 policy scorecard is available. "
        "Treat the recommendation as provisional with respect to policy choice."
    )

if (
    LIVE_ALL_COMP_WORKLOAD_ENABLED
    and weekly_live_context_audit["all_comp_status"] != "ACTIVE"
):
    weekly_model_warnings.append(
        "All-competition workload status is "
        f"{weekly_live_context_audit['all_comp_status']}; "
        "expected minutes may fall back partly or fully to the "
        "Premier-League-only baseline."
    )
if (
    LIVE_CURRENT_FBREF_ENABLED
    and weekly_live_context_audit["fbref_status"] != "ACTIVE"
):
    weekly_model_warnings.append(
        "Current FBref enrichment status is "
        f"{weekly_live_context_audit['fbref_status']}; "
        "FPL live xG/xA remains the production underlying-data source."
    )

weekly_final_player_ids = set(
    weekly_final_squad["player_id"].astype(int)
)
weekly_external_prior_audit = (
    current_new_player_audit.loc[
        current_new_player_audit["id"].astype(int).isin(
            weekly_final_player_ids
        )
    ]
    .copy()
)
if "external_event_prior_weight" in weekly_external_prior_audit.columns:
    weekly_external_prior_audit = weekly_external_prior_audit.loc[
        pd.to_numeric(
            weekly_external_prior_audit[
                "external_event_prior_weight"
            ],
            errors="coerce",
        ).fillna(0.0).gt(0.05)
        | pd.to_numeric(
            weekly_external_prior_audit.get(
                "external_minutes_prior_weight"
            ),
            errors="coerce",
        ).fillna(0.0).gt(0.05)
    ]

weekly_unmatched_priority_players = (
    current_new_player_audit.loc[
        current_new_player_audit["new_or_returning_signing_model"]
        & ~current_new_player_audit[
            "external_prior_available"
        ].fillna(0).eq(1)
        & pd.to_numeric(
            current_new_player_audit["price"],
            errors="coerce",
        ).ge(6.0)
    ]
    .copy()
)

if weekly_mode != "Initial squad selection":
    (
        weekly_transfer_evidence,
        weekly_transfer_evidence_summary,
    ) = weekly_transfer_evidence_audit(
        weekly_best_state,
        weekly_transfer_result,
        weekly_candidates,
        weekly_adjusted_gameweek_xp,
        current_new_player_audit,
        gameweek=weekly_next_gameweek,
        allow_hits=bool(weekly_policy["allow_hits"]),
    )
else:
    weekly_transfer_evidence = pd.DataFrame()
    weekly_transfer_evidence_summary = {
        "selected_transfer_count": 0,
        "best_lower_transfer_count": None,
        "incremental_objective_gain_vs_lower_count": None,
        "low_evidence_incoming_count": 0,
        "action_extra_transfer_fragile": False,
        "action_low_evidence": False,
        "reason": "Opening-squad selection does not use transfer evidence warnings.",
    }

if weekly_transfer_evidence_summary.get("action_low_evidence", False):
    weekly_model_warnings.append(
        "Low-evidence transfer warning: "
        + str(weekly_transfer_evidence_summary.get("reason"))
    )
    weekly_confidence = "Low"

next_week_top = (
    weekly_adjusted_gameweek_xp.loc[
        weekly_adjusted_gameweek_xp[
            "gameweek"
        ].eq(weekly_next_gameweek)
    ]
    .nlargest(20, "expected_points")
)
if not next_week_top.empty:
    defender_share = float(
        next_week_top["position"].eq(
            "DEF"
        ).mean()
    )
    if defender_share >= 0.60:
        weekly_model_warnings.append(
            "At least 60% of the top-20 projections are defenders."
        )
    if str(
        next_week_top.iloc[0]["position"]
    ) in {"GK", "DEF"}:
        weekly_model_warnings.append(
            "The highest projected player is a goalkeeper or defender."
        )
if weekly_price_warnings:
    weekly_model_warnings.append(
        "Some selling prices defaulted to market price; "
        "enter purchase or selling prices for exact affordability."
    )
if not weekly_freshness["is_fresh"]:
    weekly_model_warnings.append(
        "The live API differs from the in-memory projection data."
    )
if weekly_override_audit.empty is False:
    weekly_model_warnings.append(
        "Manual projection multipliers were applied."
    )
if weekly_policy_source == "conservative fallback":
    weekly_model_warnings.append(
        "Section 11 policy comparison was not available; the configured "
        "autonomous fallback policy was used."
    )
if not weekly_chip_results.get("readiness_gate", {}).get("passed", True):
    weekly_model_warnings.append(
        "The strongest raw chip candidate was blocked by the production "
        "readiness gate."
    )
if not weekly_external_prior_audit.empty:
    weekly_model_warnings.append(
        "At least one selected player still relies materially on a "
        "previous-season external new-signing prior; review the prior audit."
    )
if not weekly_unmatched_priority_players.empty:
    weekly_model_warnings.append(
        "One or more >=£6.0m sparse-history players lack an external prior; "
        "the empirical position/price fallback is being used automatically."
    )

_weekly_adaptive_search_audit = (
    dict((weekly_transfer_result or {}).get("adaptive_search_audit", {}))
    if weekly_mode != "Initial squad selection"
    else {}
)
if _weekly_adaptive_search_audit.get("confidence") == "low":
    weekly_model_warnings.append(
        "Fast and standard transfer searches did not establish a stable "
        "first-action margin. The standard result is shown, but consider "
        "running the explicit reference fidelity benchmark before the deadline."
    )

# Explain expensive selections by comparing them with the strongest omitted
# same-position alternatives over the active planning horizon.
weekly_selected_vs_omitted_rows = []
selected_ids = set(weekly_final_squad["player_id"].astype(int))
current_xp_lookup = (
    weekly_adjusted_gameweek_xp.loc[
        weekly_adjusted_gameweek_xp["gameweek"].eq(weekly_next_gameweek)
    ].set_index("player_id")["expected_points"].astype(float).to_dict()
)
horizon_xp_lookup = (
    weekly_projections.groupby("player_id")["expected_points"].sum().astype(float).to_dict()
)
for selected in weekly_final_squad.itertuples(index=False):
    alternatives = weekly_candidates.loc[
        weekly_candidates["position"].eq(selected.position)
        & ~weekly_candidates["player_id"].isin(selected_ids)
        & weekly_candidates["price"].le(float(selected.price) + 1e-9)
    ].copy()
    if alternatives.empty:
        continue
    alternatives["_horizon_xp"] = alternatives["player_id"].map(horizon_xp_lookup).fillna(0.0)
    best_alt = alternatives.sort_values(["_horizon_xp", "price"], ascending=[False, True]).iloc[0]
    weekly_selected_vs_omitted_rows.append({
        "selected": selected.web_name,
        "position": selected.position,
        "selected_price": float(selected.price),
        "selected_gw_xp": float(current_xp_lookup.get(int(selected.player_id), 0.0)),
        "selected_horizon_xp": float(horizon_xp_lookup.get(int(selected.player_id), 0.0)),
        "best_omitted": str(best_alt["web_name"]),
        "omitted_price": float(best_alt["price"]),
        "omitted_gw_xp": float(current_xp_lookup.get(int(best_alt["player_id"]), 0.0)),
        "omitted_horizon_xp": float(best_alt["_horizon_xp"]),
    })
weekly_selected_vs_omitted = pd.DataFrame(weekly_selected_vs_omitted_rows)

weekly_pending_state = None
if AUTONOMOUS_MODE:
    current_purchase_map = weekly_purchase_prices
    normal_transfer_pairs = [
        (int(a), int(b)) for a, b in weekly_first_path_row.get("transfers", ())
    ]
    normal_squad_after_ids = weekly_normal_squad["player_id"].astype(int).tolist()
    weekly_pending_state = autonomous_save_pending_decision(
        weekly_autonomous_ledger,
        gameweek=weekly_next_gameweek,
        current_squad_ids=[int(v) for v in weekly_initial_squad_ids] if weekly_mode != "Initial squad selection" else [],
        current_purchase_prices=current_purchase_map,
        current_bank=float(weekly_initial_bank),
        current_free_transfers=int(WEEKLY_FREE_TRANSFERS),
        normal_transfer_pairs=normal_transfer_pairs,
        normal_squad_ids_after=normal_squad_after_ids,
        normal_bank_after=float(weekly_transfer_plan.iloc[0]["bank_after"]),
        normal_free_transfers_after=int(weekly_first_path_row.get("free_transfers_after", WEEKLY_FREE_TRANSFERS)),
        final_squad=weekly_final_squad,
        selected_chip=weekly_selected_chip,
        candidates=weekly_candidates,
        chips_available=WEEKLY_AVAILABLE_CHIPS,
        last_free_hit_gameweek=WEEKLY_LAST_FREE_HIT_GAMEWEEK,
    )

# Production-readiness audit -------------------------------------------------
def _weekly_readiness_row(
    component: str,
    status: str,
    detail: str,
    *,
    critical: bool,
) -> dict[str, object]:
    return {
        "component": component,
        "status": status,
        "critical": bool(critical),
        "detail": str(detail),
    }


weekly_production_readiness_rows = []

_artifacts_loaded = bool(
    xp_calibration_cache is not None
    and premium_attacker_cache is not None
)
weekly_production_readiness_rows.append(
    _weekly_readiness_row(
        "forecast_validation_artifacts",
        "PASS" if _artifacts_loaded else "FAIL",
        (
            "Signed forecasting validation artifacts are loaded."
            if _artifacts_loaded
            else "Required signed forecasting validation artifacts are missing."
        ),
        critical=True,
    )
)

weekly_production_readiness_rows.append(
    _weekly_readiness_row(
        "transfer_policy_provenance",
        "PASS" if weekly_policy_validated else "WARN",
        (
            f"{weekly_policy_name}: {weekly_policy_source}"
        ),
        critical=False,
    )
)

_transfer_evidence_warn = bool(
    weekly_transfer_evidence_summary.get("action_low_evidence", False)
)
weekly_production_readiness_rows.append(
    _weekly_readiness_row(
        "transfer_evidence",
        "WARN" if _transfer_evidence_warn else "PASS",
        str(weekly_transfer_evidence_summary.get("reason")),
        critical=False,
    )
)

_adaptive_confidence = str(
    _weekly_adaptive_search_audit.get("confidence", "n/a")
)
_adaptive_profile = str(
    _weekly_adaptive_search_audit.get("selected_profile", "n/a")
)
_adaptive_warn = (
    weekly_mode != "Initial squad selection"
    and _adaptive_confidence == "low"
)
weekly_production_readiness_rows.append(
    _weekly_readiness_row(
        "adaptive_transfer_search",
        "WARN" if _adaptive_warn else "PASS",
        (
            "Initial-squad mode; transfer search not applicable."
            if weekly_mode == "Initial squad selection"
            else (
                f"profile={_adaptive_profile}; confidence={_adaptive_confidence}; "
                f"escalated={bool(_weekly_adaptive_search_audit.get('escalated', False))}; "
                f"search_signature={TRANSFER_SEARCH_SIGNATURE}"
            )
        ),
        critical=False,
    )
)

weekly_production_readiness_rows.append(
    _weekly_readiness_row(
        "official_fpl_freshness",
        "PASS" if bool(weekly_freshness["is_fresh"]) else "FAIL",
        str(weekly_freshness),
        critical=True,
    )
)

_latest_gw_ok = bool(
    weekly_gameweek_data_audit.get(
        "latest_gameweek_stats_used",
        False,
    )
)
weekly_production_readiness_rows.append(
    _weekly_readiness_row(
        "latest_completed_gw_ingested",
        "PASS" if _latest_gw_ok else "FAIL",
        (
            f"latest completed={weekly_gameweek_data_audit.get('latest_completed_gameweek')}, "
            f"stats loaded={weekly_gameweek_data_audit.get('latest_stats_gameweek_loaded')}"
        ),
        critical=True,
    )
)

_schema_repairs = int(len(MODEL_SCHEMA_REPAIR_LOG))
weekly_production_readiness_rows.append(
    _weekly_readiness_row(
        "model_schema_contracts",
        "PASS" if _schema_repairs == 0 else "WARN",
        (
            "No automatic schema repairs were required."
            if _schema_repairs == 0
            else f"{_schema_repairs} automatic schema repair event(s) were required."
        ),
        critical=False,
    )
)

_all_comp_status = str(
    weekly_live_context_audit["all_comp_status"]
)
_all_comp_ok = (
    not LIVE_ALL_COMP_WORKLOAD_ENABLED
    or _all_comp_status in {"ACTIVE", "EMPTY"}
)
weekly_production_readiness_rows.append(
    _weekly_readiness_row(
        "optional_all_comp_workload",
        "PASS" if _all_comp_ok else "WARN",
        (
            f"status={_all_comp_status}; "
            f"requests={weekly_live_context_audit['all_comp_successful_requests']}/"
            f"{weekly_live_context_audit['all_comp_total_requests']} successful; "
            f"circuit_open={weekly_live_context_audit['all_comp_circuit_open']}; "
            f"failure_cache_active={weekly_live_context_audit['all_comp_failure_cache_active']}"
        ),
        critical=False,
    )
)

_fbref_status = str(
    weekly_live_context_audit["fbref_status"]
)
_fbref_ok = (
    not LIVE_CURRENT_FBREF_ENABLED
    or _fbref_status == "ACTIVE"
)
weekly_production_readiness_rows.append(
    _weekly_readiness_row(
        "optional_fbref_context",
        "PASS" if _fbref_ok else "WARN",
        (
            f"status={_fbref_status}; "
            f"refresh_source={weekly_live_context_audit['fbref_refresh_source']}; "
            "FPL xG/xA remains authoritative when optional key fields are unavailable."
        ),
        critical=False,
    )
)

_alignment_bad = (
    isinstance(
        globals().get("production_alignment_audit"),
        pd.DataFrame,
    )
    and not production_alignment_audit.empty
    and bool(
        (
            production_alignment_audit["production_active"]
            & production_alignment_audit["status"].astype(str).str.contains(
                "UNVALIDATED|REQUIRED",
                regex=True,
            )
        ).any()
    )
)
weekly_production_readiness_rows.append(
    _weekly_readiness_row(
        "production_wrapper_alignment",
        "FAIL" if _alignment_bad else "PASS",
        (
            "No active unvalidated production-only wrapper."
            if not _alignment_bad
            else "An active production-only wrapper lacks historical validation."
        ),
        critical=True,
    )
)

weekly_production_readiness_audit = pd.DataFrame(
    weekly_production_readiness_rows
)

_critical_fail = bool(
    (
        weekly_production_readiness_audit["critical"]
        & weekly_production_readiness_audit["status"].eq("FAIL")
    ).any()
)
_any_warn = bool(
    weekly_production_readiness_audit["status"].eq("WARN").any()
)

if _critical_fail:
    weekly_production_readiness_status = "BLOCKED"
elif not weekly_policy_validated and _any_warn:
    weekly_production_readiness_status = (
        "PROVISIONAL_POLICY_WITH_DEGRADED_OPTIONAL_CONTEXT"
    )
elif not weekly_policy_validated:
    weekly_production_readiness_status = "PROVISIONAL_POLICY"
elif _any_warn:
    weekly_production_readiness_status = (
        "READY_WITH_DEGRADED_OPTIONAL_CONTEXT"
    )
else:
    weekly_production_readiness_status = "READY"

weekly_recommendation_summary = {
    "generated_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "mode": weekly_mode,
    "gameweek": weekly_next_gameweek,
    "deadline_spain": (
        weekly_deadline.isoformat()
        if pd.notna(weekly_deadline)
        else None
    ),
    "policy": weekly_policy_name,
    "policy_source": weekly_policy_source,
    "policy_validated": bool(weekly_policy_validated),
    "transfer_search_profile": _weekly_adaptive_search_audit.get(
        "selected_profile"
    ),
    "transfer_search_confidence": _weekly_adaptive_search_audit.get(
        "confidence"
    ),
    "transfer_search_escalated": _weekly_adaptive_search_audit.get(
        "escalated"
    ),
    "transfer_search_signature": TRANSFER_SEARCH_SIGNATURE,
    "production_readiness": weekly_production_readiness_status,
    "planning_horizon": weekly_horizon,
    "primary_action": weekly_primary_action,
    "normal_transfer_action": (
        weekly_primary_transfer_action
    ),
    "chip_action": (
        weekly_chip_recommendation[
            "action"
        ]
    ),
    "captain": weekly_captain,
    "vice_captain": weekly_vice_captain,
    "captain_confidence": weekly_captain_confidence,
    "captain_xp_gap_vs_best_other": (
        float(weekly_captain_xp_gap)
        if np.isfinite(weekly_captain_xp_gap)
        else None
    ),
    "execution_policy": weekly_execution_policy_name,
    "execution_policy_source": execution_policy_source,
    "expected_current_score": (
        weekly_expected_score
    ),
    "current_hit_cost": float(
        weekly_first_path_row["hit_cost"]
    ),
    "bank_after_normal_action": float(
        weekly_transfer_plan.iloc[0][
            "bank_after"
        ]
    ),
    "gain_vs_roll_objective": (
        float(weekly_gain_vs_roll)
        if np.isfinite(weekly_gain_vs_roll)
        else None
    ),
    "decision_confidence": weekly_confidence,
    "objective_gap_to_second_action": weekly_objective_gap,
    "transfer_robustness_gate_applied": bool(
        weekly_transfer_robustness.get("applied", False)
    ),
    "transfer_robustness_objective_given_up": weekly_transfer_robustness.get(
        "objective_given_up"
    ),
    "transfer_robustness_reason": weekly_transfer_robustness.get("reason"),
    "policy_bank_aware": bool(weekly_policy_bank_aware),
    "marginal_transfer_gate_enabled": bool(
        weekly_effective_marginal_gate
    ),
    "min_gain_per_extra_transfer": float(
        weekly_effective_min_gain_per_extra_transfer
    ),
    "transfer_evidence_low": bool(
        weekly_transfer_evidence_summary.get("action_low_evidence", False)
    ),
    "low_evidence_incoming_count": int(
        weekly_transfer_evidence_summary.get("low_evidence_incoming_count", 0)
    ),
    "incremental_objective_gain_vs_lower_transfer_count": (
        weekly_transfer_evidence_summary.get(
            "incremental_objective_gain_vs_lower_count"
        )
    ),
    "data_fresh": bool(
        weekly_freshness["is_fresh"]
    ),
    "all_comp_workload_enabled": bool(
        LIVE_ALL_COMP_WORKLOAD_ENABLED
    ),
    "all_comp_status": weekly_live_context_audit[
        "all_comp_status"
    ],
    "all_comp_unique_events": int(
        weekly_live_context_audit["all_comp_unique_events"]
    ),
    "all_comp_fixture_rows": int(
        weekly_live_context_audit["all_comp_fixture_rows"]
    ),
    "all_comp_circuit_open": bool(
        weekly_live_context_audit["all_comp_circuit_open"]
    ),
    "all_comp_remaining_requests_skipped": (
        weekly_live_context_audit[
            "all_comp_remaining_requests_skipped"
        ]
    ),
    "all_comp_failure_cache_active": bool(
        weekly_live_context_audit[
            "all_comp_failure_cache_active"
        ]
    ),
    "all_comp_player_minute_rows": int(
        weekly_live_context_audit["all_comp_player_minute_rows"]
    ),
    "current_fbref_status": weekly_live_context_audit[
        "fbref_status"
    ],
    "current_fbref_matched_players": int(
        weekly_live_context_audit["fbref_matched_players"]
    ),
    "current_fbref_refresh_source": weekly_live_context_audit[
        "fbref_refresh_source"
    ],
    "current_fbref_npxg_coverage": float(
        weekly_live_context_audit[
            "fbref_coverage"
        ].get(
            "fbref_npxg_per90",
            {},
        ).get("coverage", 0.0)
    ),
    "current_fbref_xag_coverage": float(
        weekly_live_context_audit[
            "fbref_coverage"
        ].get(
            "fbref_xag_per90",
            {},
        ).get("coverage", 0.0)
    ),
    "set_piece_role_enabled": bool(
        LIVE_SET_PIECE_ROLE_ENABLED
    ),
    "autonomous_mode": bool(AUTONOMOUS_MODE),
    "xp_validation_artifact_loaded": bool(xp_calibration_cache is not None),
    "xp_calibration_applied": bool(active_xp_calibration_payload(xp_calibration_cache) is not None),
    "premium_attacker_adjustment_applied": bool(active_premium_attacker_payload(premium_attacker_cache) is not None),
    "warnings": weekly_model_warnings,
    "latest_completed_gameweek": (
        weekly_gameweek_data_audit[
            "latest_completed_gameweek"
        ]
    ),
    "latest_stats_gameweek_loaded": (
        weekly_gameweek_data_audit[
            "latest_stats_gameweek_loaded"
        ]
    ),
    "latest_gameweek_stats_used": (
        weekly_gameweek_data_audit[
            "latest_gameweek_stats_used"
        ]
    ),
    "early_season_shrinkage_enabled": (
        weekly_early_season_audit["enabled"]
    ),
    "early_season_shrinkage_ready": (
        weekly_early_season_audit["ready"]
    ),
    "early_season_sparse_players_active": (
        weekly_early_season_audit["active_sparse_players"]
    ),
    "early_season_sparse_teams_active": (
        weekly_early_season_audit["active_sparse_teams"]
    ),
    "live_feature_signature": LIVE_FEATURE_SIGNATURE,
}

weekly_summary_table = pd.DataFrame(
    [
        {
            "field": key,
            "value": value,
        }
        for key, value
        in weekly_recommendation_summary.items()
        if key != "warnings"
    ]
)

print("=" * 72)
print(
    f"FPL GAMEWEEK {weekly_next_gameweek} RECOMMENDATION"
)
print("=" * 72)
print(
    "Latest GW data used:",
    (
        "YES"
        if weekly_gameweek_data_audit[
            "latest_gameweek_stats_used"
        ]
        else "NO"
    ),
    f"(completed/published GW "
    f"{weekly_gameweek_data_audit['latest_completed_gameweek']}; "
    f"stats through GW "
    f"{weekly_gameweek_data_audit['latest_stats_gameweek_loaded']})",
)
print(
    "Early-season stabilisation:",
    (
        "ACTIVE"
        if EARLY_SEASON_SHRINKAGE_ENABLED
        else "OFF"
    ),
    f"({weekly_early_season_audit['active_sparse_players']} sparse players, "
    f"{weekly_early_season_audit['active_sparse_teams']} sparse teams)",
)
print(
    "All-competition workload:",
    (
        f"{weekly_live_context_audit['all_comp_status']} "
        f"({weekly_live_context_audit['all_comp_unique_events']} events; "
        f"{weekly_live_context_audit['all_comp_fixture_rows']} team-fixture rows; "
        f"{weekly_live_context_audit['all_comp_player_minute_rows']} player-minute rows; "
        f"{weekly_live_context_audit['all_comp_successful_requests']}/"
        f"{weekly_live_context_audit['all_comp_total_requests']} requests successful)"
        if LIVE_ALL_COMP_WORKLOAD_ENABLED
        else "OFF"
    ),
)

_fbref_npxg = weekly_live_context_audit["fbref_coverage"].get(
    "fbref_npxg_per90",
    {},
)
_fbref_xag = weekly_live_context_audit["fbref_coverage"].get(
    "fbref_xag_per90",
    {},
)
_fbref_shots = weekly_live_context_audit["fbref_coverage"].get(
    "fbref_shots_per90",
    {},
)
_fbref_pkatt = weekly_live_context_audit["fbref_coverage"].get(
    "fbref_pkatt_per90",
    {},
)
print(
    "Current FBref enrichment:",
    (
        f"{weekly_live_context_audit['fbref_status']} "
        f"({weekly_live_context_audit['fbref_matched_players']} matched; "
        f"npxG {_fbref_npxg.get('populated_players', 0)}, "
        f"xAG {_fbref_xag.get('populated_players', 0)}, "
        f"shots {_fbref_shots.get('populated_players', 0)}, "
        f"PKatt {_fbref_pkatt.get('populated_players', 0)})"
        if LIVE_CURRENT_FBREF_ENABLED
        else "OFF"
    ),
)
print(
    "Explicit set-piece roles:",
    "ACTIVE" if LIVE_SET_PIECE_ROLE_ENABLED else "OFF",
)
print("Primary action:", weekly_primary_action)
print(
    "Expected current score:",
    f"{weekly_expected_score:.2f}",
)
print(
    "Captain / vice:",
    f"{weekly_captain} / {weekly_vice_captain}",
)
print(
    "Captain confidence:",
    weekly_captain_confidence,
    (
        f"(xP gap {weekly_captain_xp_gap:+.3f})"
        if np.isfinite(weekly_captain_xp_gap)
        else ""
    ),
)
print(
    "Execution policy:",
    f"{weekly_execution_policy_name} ({execution_policy_source})",
)
print(
    "Policy:",
    f"{weekly_policy_name} ({weekly_policy_source})",
)
print(
    "Decision confidence:",
    weekly_confidence,
)
print(
    "Transfer robustness:",
    weekly_transfer_robustness.get("reason"),
)
print(
    "Bank-aware marginal gate:",
    (
        f"ACTIVE ({weekly_effective_min_gain_per_extra_transfer:.2f} "
        "objective points per extra transfer)"
        if weekly_effective_marginal_gate
        else "OFF"
    ),
)
if weekly_transfer_evidence_summary.get("action_low_evidence", False):
    print(
        "⚠ LOW-EVIDENCE TRANSFER:",
        weekly_transfer_evidence_summary.get("reason"),
    )
    if not weekly_transfer_evidence.empty:
        for _evidence_row in weekly_transfer_evidence.loc[
            weekly_transfer_evidence["evidence_level"].eq("LOW")
        ].itertuples(index=False):
            print(
                f"  - {_evidence_row.incoming}: "
                f"{_evidence_row.evidence_reasons}"
            )
    if weekly_transfer_evidence_summary.get(
        "action_extra_transfer_fragile", False
    ):
        _incremental = weekly_transfer_evidence_summary.get(
            "incremental_objective_gain_vs_lower_count"
        )
        print(
            "  - Multi-transfer fragility:",
            (
                f"{float(_incremental):+.3f} objective points versus "
                "the best lower-transfer-count state."
                if _incremental is not None
                else "lower-transfer-count comparator unavailable."
            ),
        )
_subset_skip_reason = (
    (weekly_transfer_result or {}).get(
        "robustness_subset_skipped_reason"
    )
)
print(
    "Exact raw-action subset checks:",
    (
        f"SKIPPED — {_subset_skip_reason}"
        if _subset_skip_reason
        else (
            f"{weekly_transfer_robustness.get('exact_subset_comparator_count', 0)} "
            f"legal comparator(s)"
            + (
                f"; {weekly_transfer_robustness.get('exact_subset_illegal_count', 0)} "
                "subset(s) illegal in isolation"
                if weekly_transfer_robustness.get(
                    "exact_subset_illegal_count",
                    0,
                )
                else ""
            )
        )
    ),
)
structural_unavailable_display = (
    weekly_transfer_robustness.get(
        "structural_unavailable_players",
        {},
    )
)
if structural_unavailable_display:
    print(
        "Structural unavailable:",
        ", ".join(
            f"{info['web_name']} (£{info['price']:.1f}m)"
            for info in structural_unavailable_display.values()
        ),
    )
if WEEKLY_RUNTIME_TIMING:
    print(
        "Runtime profile:",
        f"base transfer {weekly_runtime_profile.get('base_transfer_search_seconds', 0.0):.1f}s | ",
        f"subset checks {weekly_runtime_profile.get('subset_robustness_seconds', 0.0):.1f}s | ",
        f"exact current {weekly_runtime_profile.get('current_exact_decision_seconds', 0.0):.1f}s | ",
        f"chips {weekly_runtime_profile.get('chip_evaluation_seconds', 0.0):.1f}s",
    )
    print(
        "Chip optimisation:",
        weekly_chip_results.get("runtime", {}).get(
            "expensive_squad_solves",
            "n/a",
        ),
        "expensive squad solve(s); future skipped for",
        weekly_chip_results.get("runtime", {}).get(
            "skipped_future_expensive_chips",
            [],
        ),
    )
if np.isfinite(weekly_gain_vs_roll):
    print(
        "Selected path advantage over rolling:",
        f"{weekly_gain_vs_roll:+.2f} objective points",
    )
print(
    "Chip conclusion:",
    weekly_chip_recommendation[
        "reason"
    ],
)

display(weekly_summary_table)
print("Starting XI")
display(
    weekly_starting_xi.round({
        "expected_minutes": 1,
        "appearance_probability": 3,
        "expected_points": 2,
    })
)
print("Bench order")
display(
    weekly_bench.round({
        "expected_minutes": 1,
        "appearance_probability": 3,
        "expected_points": 2,
    })
)
print("Captain alternatives")
if isinstance(weekly_captain_alternatives, pd.DataFrame) and not weekly_captain_alternatives.empty:
    display(
        weekly_captain_alternatives[
            [
                column for column in [
                    "web_name",
                    "expected_points",
                    "expected_minutes",
                    "appearance_probability",
                    "xp_gap_to_top",
                    "within_captain_tie_band",
                    "selected",
                ]
                if column in weekly_captain_alternatives.columns
            ]
        ].round(3)
    )

if not weekly_transfer_evidence.empty:
    print("Transfer evidence audit")
    display(
        weekly_transfer_evidence.round({
            "price": 1,
            "expected_points": 3,
            "xp_defensive_contribution": 3,
            "dc_share_of_xp": 3,
            "prior_pl_fixtures": 0,
            "prior_season_fixtures": 0,
            "external_event_prior_weight": 3,
            "market_fallback_weight": 3,
            "incremental_objective_gain_vs_lower_count": 3,
            "cheaper_alternative_price": 1,
            "cheaper_alternative_xp": 3,
            "cheaper_alternative_xp_gap": 3,
            "cheaper_alternative_saving": 1,
        })
    )

print("Current transfer alternatives")
display(
    weekly_transfer_alternatives.round(3)
)
print("Provisional rolling plan — rerun before every deadline")
display(
    weekly_transfer_plan.round(3)
)
print("Current chip comparison")
display(
    weekly_chip_results[
        "current_comparison"
    ].round(3)
)
print("Best chip opportunities inside the explicit horizon")
display(
    weekly_chip_results[
        "best_opportunities"
    ].round(3)
)

if not weekly_override_audit.empty:
    print("Manual projection overrides")
    display(weekly_override_audit)


weekly_live_context_columns = [
    column
    for column in [
        "web_name",
        "team_short_name",
        "expected_minutes",
        "expected_points",
        "all_comp_start_multiplier",
        "previous_non_pl_competition",
        "days_since_previous_all_comp_fixture",
        "previous_non_pl_player_minutes",
        "penalties_order",
        "corners_and_indirect_freekicks_order",
        "direct_freekicks_order",
        "set_piece_role_score",
        "set_piece_penalty_xg_overlay_per90",
        "fbref_npxg_per90",
        "fbref_xag_per90",
        "fbref_shots_per90",
        "fbref_pkatt_per90",
    ]
    if column in weekly_adjusted_gameweek_xp.columns
]
weekly_live_context_selected = (
    weekly_adjusted_gameweek_xp.loc[
        weekly_adjusted_gameweek_xp["gameweek"].eq(
            weekly_next_gameweek
        )
        & weekly_adjusted_gameweek_xp[
            "player_id"
        ].astype(int).isin(weekly_final_player_ids),
        weekly_live_context_columns,
    ]
    .sort_values("expected_points", ascending=False)
)

if not weekly_live_context_selected.empty:
    print("Selected-squad all-competition / set-piece / FBref context")
    display(weekly_live_context_selected.round(3))

if not weekly_external_prior_audit.empty:
    print("Selected-squad external new-signing priors")
    selected_prior_columns = [
        column
        for column in [
            "web_name",
            "team_short_name",
            "position",
            "price",
            "prior_career_fixture_count",
            "external_source_leagues",
            "external_prior_match_method",
            "external_prior_source_note",
            "external_prior_reliability",
            "external_minutes_prior_weight",
            "external_event_prior_weight",
            "market_fallback_weight",
            "market_fallback_evidence_fade",
            "expected_minutes",
            "expected_points",
        ]
        if column in weekly_external_prior_audit.columns
    ]
    display(
        weekly_external_prior_audit[
            selected_prior_columns
        ]
    )

if not weekly_unmatched_priority_players.empty:
    print("Priority new players missing external priors")
    display(
        weekly_unmatched_priority_players[
            [
                "web_name",
                "team_short_name",
                "position",
                "price",
                "prior_career_fixture_count",
            ]
        ]
    )

print("Selected-vs-best-omitted audit")
if not weekly_selected_vs_omitted.empty:
    display(weekly_selected_vs_omitted.round(3))

if weekly_mode == "Initial squad selection":
    print("Alternative opening squads")
    display(weekly_opening_squad_alternatives.round(3))
    if RUN_OPENING_BENCH_SENSITIVITY:
        print("Opening bench-cost sensitivity")
        display(
            weekly_opening_bench_sensitivity
            .replace({np.inf: "unconstrained"})
            .round(3)
        )

print("Production readiness:", weekly_production_readiness_status)
display(weekly_production_readiness_audit)

if weekly_model_warnings:
    print("Warnings")
    for warning in weekly_model_warnings:
        print("-", warning)

if WEEKLY_SAVE_OUTPUT:
    WEEKLY_OUTPUT_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    weekly_output_payload = {
        "summary": weekly_recommendation_summary,
        "production_readiness_audit": (
            weekly_production_readiness_audit.to_dict("records")
        ),
        "autonomous_pending_state": weekly_pending_state,
        "selected_vs_omitted": (
            weekly_selected_vs_omitted.replace({np.nan: None}).to_dict("records")
        ),
        "opening_squad_alternatives": (
            weekly_opening_squad_alternatives.replace({np.nan: None, np.inf: None}).to_dict("records")
        ),
        "opening_bench_sensitivity": (
            weekly_opening_bench_sensitivity.replace({np.nan: None, np.inf: None}).to_dict("records")
        ),
        "starting_xi": (
            weekly_starting_xi.to_dict(
                "records"
            )
        ),
        "bench": weekly_bench.to_dict(
            "records"
        ),
        "captain_alternatives": (
            weekly_captain_alternatives
            .replace({np.nan: None})
            .to_dict("records")
            if isinstance(weekly_captain_alternatives, pd.DataFrame)
            else []
        ),
        "execution_policy_validation": {
            "approved": bool(execution_policy_validation.get("approved", False)),
            "reason": execution_policy_validation.get("reason"),
            "active_policy": ACTIVE_EXECUTION_POLICY,
            "source": execution_policy_source,
            "selected_gain_vs_baseline": execution_policy_validation.get(
                "selected_gain_vs_baseline"
            ),
        },
        "transfer_alternatives": (
            weekly_transfer_alternatives
            .to_dict("records")
        ),
        "transfer_robustness": weekly_transfer_robustness,
        "transfer_evidence_summary": weekly_transfer_evidence_summary,
        "transfer_evidence_audit": (
            weekly_transfer_evidence
            .replace({np.nan: None})
            .to_dict("records")
        ),
        "rolling_plan": (
            weekly_transfer_plan
            .to_dict("records")
        ),
        "chip_comparison": (
            weekly_chip_results[
                "current_comparison"
            ].to_dict("records")
        ),
        "raw_chip_recommendation": weekly_chip_results.get(
            "raw_recommendation"
        ),
        "chip_readiness_gate": weekly_chip_results.get(
            "readiness_gate"
        ),
        "best_chip_opportunities": (
            weekly_chip_results[
                "best_opportunities"
            ].to_dict("records")
        ),
        "freshness_audit": weekly_freshness,
        "projection_overrides": (
            weekly_override_audit
            .to_dict("records")
        ),
        "external_prior_audit": (
            weekly_external_prior_audit
            .replace({np.nan: None})
            .to_dict("records")
        ),
        "unmatched_priority_new_players": (
            weekly_unmatched_priority_players[
                [
                    "web_name",
                    "team_short_name",
                    "position",
                    "price",
                    "prior_career_fixture_count",
                ]
            ]
            .replace({np.nan: None})
            .to_dict("records")
        ),
    }
    with WEEKLY_OUTPUT_PATH.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            weekly_output_payload,
            file,
            indent=2,
            default=str,
        )
    print(
        "Saved latest weekly recommendation to:",
        WEEKLY_OUTPUT_PATH,
    )


In [ ]:
# =============================================================================
# v2.6 EXECUTION CARD — the single instruction for this deadline
# =============================================================================
# Everything below is already computed by Section 12; this cell only
# consolidates it into one unambiguous block so the recommendation can be
# executed without reading the diagnostics above.
#
# Every lookup is defensive: if a variable is missing the card says so rather
# than failing, because a half-printed instruction is worse than a loud gap.

def _v26_card() -> None:
    import pandas as _pd
    import numpy as _np

    G = globals()
    missing = []

    def need(name, default=None):
        if name not in G or G[name] is None:
            missing.append(name)
            return default
        return G[name]

    gw = need("weekly_next_gameweek", "?")
    deadline = G.get("weekly_deadline")
    chip = need("weekly_chip_recommendation", {}) or {}
    xi = need("weekly_starting_xi", _pd.DataFrame())
    bench = need("weekly_bench", _pd.DataFrame())
    captain = need("weekly_captain", "?")
    vice = need("weekly_vice_captain", "?")
    action = need("weekly_primary_action", "?")
    score = G.get("weekly_expected_score", float("nan"))

    bar = "=" * 70
    print(bar)
    deadline_text = ""
    if deadline is not None and _pd.notna(deadline):
        deadline_text = f"  —  deadline {deadline:%a %d %b %H:%M}"
    print(f"  GAMEWEEK {gw} — EXECUTE THIS{deadline_text}")
    print(bar)

    # 1. chip
    chip_name = chip.get("chip") or "NONE"
    chip_action = chip.get("action", "?")
    print(f"  1. CHIP       : {str(chip_name).replace('_', ' ').upper()}")
    print(f"                  {chip_action}")
    if chip.get("reason"):
        print(f"                  why: {chip['reason']}")

    # 2. transfers
    print(f"  2. TRANSFERS  : {action}")
    moves = G.get("weekly_transfer_moves")
    if isinstance(moves, _pd.DataFrame) and not moves.empty:
        for row in moves.itertuples(index=False):
            out_name = getattr(row, "outgoing", "?")
            in_name = getattr(row, "incoming", "?")
            print(f"                  OUT {out_name}   ->   IN {in_name}")
    hit = G.get("weekly_hit_cost")
    if hit is not None and _pd.notna(hit):
        print(f"                  points hit: {float(hit):.0f}")

    # 3. starting XI, grouped by position
    print("  3. STARTING XI:")
    if isinstance(xi, _pd.DataFrame) and not xi.empty:
        name_col = next((c for c in ("web_name", "player_name", "name")
                         if c in xi.columns), None)
        pos_col = next((c for c in ("model_position", "position") if c in xi.columns), None)
        if name_col and pos_col:
            shape = []
            for pos in ("GK", "DEF", "MID", "FWD"):
                members = xi.loc[xi[pos_col].astype(str).str.upper()
                                 .str.startswith(pos[:3]), name_col].tolist()
                if pos != "GK":
                    shape.append(str(len(members)))
                tagged = []
                for m in members:
                    suffix = " (C)" if m == captain else (" (V)" if m == vice else "")
                    tagged.append(f"{m}{suffix}")
                print(f"                  {pos:4s} {', '.join(tagged) if tagged else '-'}")
            print(f"                  formation {'-'.join(shape)}")
        else:
            print("                  (name/position columns not found)")
    else:
        print("                  (no starting XI available)")

    # 4. bench, in autosub order
    print("  4. BENCH ORDER: (autosubs happen in this order)")
    if isinstance(bench, _pd.DataFrame) and not bench.empty:
        name_col = next((c for c in ("web_name", "player_name", "name")
                         if c in bench.columns), None)
        pos_col = next((c for c in ("model_position", "position")
                        if c in bench.columns), None)
        for i, row in enumerate(bench.itertuples(index=False), start=1):
            nm = getattr(row, name_col) if name_col else "?"
            ps = getattr(row, pos_col) if pos_col else ""
            print(f"                  {i}. {nm} ({ps})")
    else:
        print("                  (no bench available)")

    # 5. captaincy
    print(f"  5. CAPTAIN    : {captain}")
    print(f"     VICE       : {vice}")

    print("-" * 70)
    if score is not None and _pd.notna(score):
        print(f"  Projected points for GW{gw}: {float(score):.1f}")

    gap = G.get("weekly_captain_xp_gap")
    conf = G.get("weekly_captain_confidence")
    if gap is not None and _np.isfinite(gap):
        tie = "  <- near-tie, treat as a coin flip" if abs(float(gap)) < 0.35 else ""
        print(f"  Captain confidence: {conf} (xP gap {float(gap):+.2f}){tie}")
    if G.get("weekly_confidence"):
        print(f"  Decision confidence: {G['weekly_confidence']}")

    warnings_out = []
    ev = G.get("weekly_transfer_evidence_summary") or {}
    if ev.get("action_low_evidence"):
        warnings_out.append(f"LOW-EVIDENCE TRANSFER: {ev.get('reason')}")
    readiness = G.get("weekly_production_readiness_status")
    if readiness and str(readiness).upper() not in ("OK", "READY", "TRUE"):
        warnings_out.append(f"production readiness: {readiness}")
    if G.get("V26_APPLY_ASSIST_CONVERSION"):
        warnings_out.append(
            "v2.6 assist-conversion ACTIVE — confirm the Section 11 gate passed"
        )
    if G.get("V26_RELAX_UPPER_TAIL_MAE_GATE"):
        _v24 = G.get("v24_targeted_forecast_cache") or {}
        _ut = (_v24.get("production_decision") or {}).get("upper_tail") or {}
        warnings_out.append(
            "v2.4 upper-tail MAE gate relaxed; layer approved="
            + str(_ut.get("approved"))
        )
    if missing:
        warnings_out.append("missing variables: " + ", ".join(sorted(set(missing))))
    for w in warnings_out:
        print(f"  ! {w}")

    print(bar)
    print("  Reminder: this is the ONLY action for this deadline. The rolling")
    print("  plan beyond this gameweek is provisional — rerun before next week.")
    print(bar)


_v26_card()


### Interpreting Section 12

- **Primary action** is the only action intended for the imminent deadline.
- The later transfer plan is provisional and should not be followed without
  rerunning the model next week.
- A Free Hit or Wildcard recommendation supersedes the normal transfer path.
- Triple Captain and Bench Boost are combined with the normal transfer action.
- `gain_vs_roll_objective` is a discounted planning-objective difference, not a
  guarantee of realised points.
- In autonomous mode, purchase prices and bank are maintained by the local ledger and reconciled after each deadline. A mid-season bootstrap without an existing ledger is marked approximate.
- The freshness check deliberately stops the run if prices, availability news
  or fixture scheduling changed after the projections were built.
- Manual projection multipliers are disabled by default in autonomous mode; the system relies on the official FPL availability/news state and fails closed on stale data.

- In initial-squad mode, transfer gain versus rolling is reported as not applicable.
- A blocked raw chip candidate is still shown in the chip tables, but the primary action preserves the chip until the structural readiness gate passes.
- `decision_confidence` reflects the planning-objective gap between the two best immediate transfer paths; it is not a probability of being correct.
- The external-prior audit flags selected players whose projections still depend materially on previous-season non-FPL evidence.
- Sparse-history players without an external prior receive a weak empirical position/price fallback rather than requiring a manual estimate.


## Autonomous production checklist

### v2.1.4.1 normal weekly run

The first code cell is already configured for production:

```python
RUN_FULL_POLICY_BACKTEST = False
RUN_SEQUENTIAL_BACKTEST = False

RUN_REGRET_DIAGNOSTICS = False
RUN_EXECUTION_POLICY_VALIDATION = False
RUN_REALISTIC_TRANSFER_REGRET = False

EARLY_SEASON_SHRINKAGE_ENABLED = True
EARLY_SEASON_STRICT_AUDIT = True
```

Keep the notebook in the same project folder as the existing `data/` caches,
restart the kernel and **Run All**.

The expensive historical policy and sequential backtests remain off.
`PROJECT_VERSION` stays `2.1.0` so the signed v2.1 validation artifacts remain
loadable. A separate `LIVE_FEATURE_SIGNATURE` protects the new v2.1.4 live
stabilisation settings.

Before Section 12 gives an actionable recommendation it now verifies and
prints:

```text
Latest completed/published GW
Latest GW in player-stat history
Latest GW data used: YES
Early-season stabilisation: ACTIVE
Sparse players/teams actively shrunk
```

If the latest completed GW is missing or the stabilisation layer was not
generated while enabled, Section 12 stops instead of recommending a transfer.

### Shrinkage behaviour

For a new/sparse PL player, current observations receive weight
`n / EARLY_SEASON_PLAYER_FADE_FIXTURES`. With the default fade of eight
fixtures, a single GW receives 12.5% weight in the rolling/career history
features; the remainder comes from archived PL history when available or a
historical position prior.

For promoted/new-to-model clubs the same principle stabilises goals for,
goals against and clean-sheet history. Archived PL club history is preferred;
otherwise the historical Premier League average is used.

The literal last-GW minutes/start observation remains unshrunk by default so
genuine role information is retained.


In [ ]:
# ============================================================
# GENERIC EARLY-SEASON SHRINKAGE AUDIT
# ============================================================

def early_season_shrinkage_audit_tables():
    player_audit = pd.DataFrame()
    team_audit = pd.DataFrame()
    projection_audit = pd.DataFrame()

    if (
        "current_as_of_history" in globals()
        and isinstance(current_as_of_history, pd.DataFrame)
        and not current_as_of_history.empty
    ):
        player_context = players[
            [
                "player_key",
                "web_name",
                "team_short_name",
                "position",
                "price",
            ]
        ].drop_duplicates("player_key")

        player_audit = (
            current_as_of_history
            .merge(
                player_context,
                on="player_key",
                how="left",
                validate="one_to_one",
            )
            .loc[
                lambda frame: frame[
                    "early_season_player_shrinkage_active"
                ].fillna(False)
            ]
            .sort_values(
                [
                    "early_season_player_current_weight",
                    "price",
                ],
                ascending=[True, False],
            )
        )

    if (
        "live_team_as_of" in globals()
        and isinstance(live_team_as_of, pd.DataFrame)
        and not live_team_as_of.empty
    ):
        # FPL bootstrap `short_name` is normalised earlier in this notebook to
        # `team_short_name`. Keep this diagnostic tolerant of either schema so
        # a display-only audit can never stop the production recommendation.
        team_name_column = next(
            (
                column
                for column in [
                    "team_short_name",
                    "short_name",
                    "team_name",
                    "name",
                ]
                if column in teams.columns
            ),
            None,
        )

        if team_name_column is None:
            team_names = (
                teams[["team_id"]]
                .drop_duplicates("team_id")
                .assign(team_short_name=lambda frame: frame["team_id"].astype(str))
            )
        else:
            team_names = (
                teams[
                    ["team_id", team_name_column]
                ]
                .drop_duplicates("team_id")
                .rename(
                    columns={
                        team_name_column: "team_short_name"
                    }
                )
            )

        team_audit = (
            live_team_as_of
            .merge(
                team_names,
                on="team_id",
                how="left",
                validate="one_to_one",
            )
            .loc[
                lambda frame: frame[
                    "team_early_season_shrinkage_active"
                ].fillna(False)
            ]
            .sort_values(
                "team_early_season_current_weight"
            )
        )

    if (
        "live_gameweek_expected_points" in globals()
        and isinstance(live_gameweek_expected_points, pd.DataFrame)
        and not live_gameweek_expected_points.empty
    ):
        next_gw = int(
            live_gameweek_expected_points["gameweek"].min()
        )
        columns = [
            column
            for column in [
                "gameweek",
                "web_name",
                "team_short_name",
                "position",
                "price",
                "expected_minutes",
                "expected_points",
                "early_season_player_shrinkage_active",
                "early_season_player_current_fixtures",
                "early_season_player_archived_pl_fixtures",
                "early_season_player_current_weight",
                "early_season_player_prior_source",
                "team_early_season_shrinkage_active",
                "team_early_season_current_matches",
                "team_early_season_archived_pl_matches",
                "team_early_season_current_weight",
                "team_early_season_prior_source",
                "opponent_early_season_shrinkage_active",
                "opponent_early_season_current_weight",
            ]
            if column in live_gameweek_expected_points.columns
        ]
        projection_audit = (
            live_gameweek_expected_points.loc[
                live_gameweek_expected_points["gameweek"].eq(next_gw),
                columns,
            ]
            .loc[
                lambda frame: (
                    frame.get(
                        "early_season_player_shrinkage_active",
                        pd.Series(False, index=frame.index),
                    ).fillna(False)
                    | frame.get(
                        "team_early_season_shrinkage_active",
                        pd.Series(False, index=frame.index),
                    ).fillna(False)
                )
            ]
            .sort_values("expected_points", ascending=False)
        )

    return player_audit, team_audit, projection_audit


try:
    (
        early_season_player_audit,
        early_season_team_audit,
        early_season_projection_audit,
    ) = early_season_shrinkage_audit_tables()
except Exception as exc:
    warnings.warn(
        "Early-season display audit could not be rendered. "
        "This does not invalidate the already-generated live forecasts. "
        f"Diagnostic error: {exc}"
    )
    early_season_player_audit = pd.DataFrame()
    early_season_team_audit = pd.DataFrame()
    early_season_projection_audit = pd.DataFrame()

print("=" * 72)
print("EARLY-SEASON SHRINKAGE AUDIT")
print("=" * 72)
print("Enabled:", EARLY_SEASON_SHRINKAGE_ENABLED)
print("Live feature signature:", LIVE_FEATURE_SIGNATURE)
print("Sparse player histories stabilised:", len(early_season_player_audit))
print("Sparse/new team histories stabilised:", len(early_season_team_audit))

if not early_season_team_audit.empty:
    print("\nTeams currently receiving shrinkage")
    display(
        early_season_team_audit[
            [
                column
                for column in [
                    "team_short_name",
                    "team_early_season_current_matches",
                    "team_early_season_archived_pl_matches",
                    "team_early_season_current_weight",
                    "team_early_season_prior_source",
                    "team_goals_for_last_5",
                    "team_goals_against_last_5",
                    "team_clean_sheet_rate_last_5",
                ]
                if column in early_season_team_audit.columns
            ]
        ].round(3)
    )

if not early_season_projection_audit.empty:
    print("\nHighest next-GW projections affected by early-season shrinkage")
    display(
        early_season_projection_audit.head(30).round(3)
    )


# 13. v2.3 observational xP audit

This section deliberately **does not rerun season simulations, policy sweeps, transfer
optimisation, or historical walk-forward folds**.

It answers a narrower question: *are the deadline forecasts we actually made useful for
FPL decisions?*

The automatic audit uses:

1. official realised FPL player outcomes;
2. official manager Gameweek points and overall Gameweek average;
3. forecasts frozen before the deadline;
4. FPL `ep_next`, form and points-per-game from the existing pre-deadline bootstrap
   snapshots when available.

Two preserved executed notebooks are recovered as partial forecast snapshots for GW4 and
GW5. They contain the displayed top-30 player forecasts and the actual recommendation
tables from those deadline runs. From v2.3 onward, the notebook stores the **full**
pre-deadline forecast table automatically, so coverage grows every week.

`ONLINE_DIAGNOSTIC_COUNTERFACTUALS` is `False` by default. Therefore this section does
not ask "what if we had made another transfer?" or simulate an alternative season path.
Those checks should only be enabled later for a specific diagnostic hypothesis.


In [ ]:

# ============================================================================
# v2.3 OBSERVATIONAL FORECAST ARCHIVE
# ============================================================================

# Recovered from the executed deadline notebooks supplied during this project.
# These are intentionally marked partial because only the displayed top-30
# player forecast table was preserved in notebook output.
_V23_RECOVERED_DEADLINE_SNAPSHOTS = json.loads(r"""{"4":{"source":"recovered_from_executed_notebook_output","partial_player_pool":true,"player_pool_scope":"displayed_top30","gameweek":4,"summary":{"generated_at_utc":"2026-09-11T15:10:21.709791+00:00","mode":"Live in-season squad","gameweek":"4","deadline_spain":"2026-09-12T14:30:00+02:00","policy":"greedy_1gw_no_hits","policy_source":"validated Section 11 walk-forward selection — ...","policy_validated":"True","transfer_search_profile":"standard","transfer_search_confidence":"medium","transfer_search_escalated":"True","transfer_search_signature":"f2b963340451175c2ecc","production_readiness":"READY_WITH_DEGRADED_OPTIONAL_CONTEXT","planning_horizon":"1","primary_action":"Maguire → Richards; Palmer → Saka","normal_transfer_action":"Maguire → Richards; Palmer → Saka","chip_action":"Save all chips","captain":"B.Fernandes","vice_captain":"Saka","captain_confidence":"Medium","captain_xp_gap_vs_best_other":"0.18876","execution_policy":"baseline","execution_policy_source":"forced off","expected_current_score":"46.89784","current_hit_cost":"0.0","bank_after_normal_action":"0.6","gain_vs_roll_objective":"1.489049","decision_confidence":"Low","objective_gap_to_second_action":"0.030644","transfer_robustness_gate_applied":"False","transfer_robustness_objective_given_up":"0.0","transfer_robustness_reason":"Transfer robustness gates disabled or not appl...","policy_bank_aware":"False","marginal_transfer_gate_enabled":"False","min_gain_per_extra_transfer":"1.0","transfer_evidence_low":"True","low_evidence_incoming_count":"1","incremental_objective_gain_vs_lower_transfer_c...":"0.73606","data_fresh":"True","all_comp_workload_enabled":"True","all_comp_status":"CIRCUIT_OPEN","all_comp_unique_events":"0","all_comp_fixture_rows":"0","all_comp_circuit_open":"True","all_comp_remaining_requests_skipped":null,"all_comp_failure_cache_active":"True","all_comp_player_minute_rows":"0","current_fbref_status":"PARTIAL","current_fbref_matched_players":"259","current_fbref_refresh_source":"fresh_cache","current_fbref_npxg_coverage":"0.0","current_fbref_xag_coverage":"0.0","set_piece_role_enabled":"True","autonomous_mode":"True","xp_validation_artifact_loaded":"True","xp_calibration_applied":"False","premium_attacker_adjustment_applied":"False","latest_completed_gameweek":"3","latest_stats_gameweek_loaded":"3","latest_gameweek_stats_used":"True","early_season_shrinkage_enabled":"False","early_season_shrinkage_ready":"True","early_season_sparse_players_active":"0","early_season_sparse_teams_active":"0","live_feature_signature":"4a722404f8bd59fd4fc3"},"player_forecasts":[{"player_id":426,"web_name":"B.Fernandes","team_short_name":"MUN","position":"MID","price":12.0,"gameweek":4,"expected_minutes":82.834053,"appearance_probability":0.957858,"projected_goals":0.251307,"projected_assists":0.211651,"expected_points":4.219599,"xp_appearance":1.851034,"xp_goals":1.256533,"xp_assists":0.634954,"xp_clean_sheet":0.180935,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.067334,"xp_bonus":0.385896,"xp_yellow_cards":-0.144399,"xp_red_cards":-0.004651,"xp_own_goals":-0.0037,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.004338,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":411,"web_name":"Haaland","team_short_name":"MCI","position":"FWD","price":15.5,"gameweek":4,"expected_minutes":78.027883,"appearance_probability":0.946674,"projected_goals":0.478022,"projected_assists":0.060814,"expected_points":4.14395,"xp_appearance":1.808619,"xp_goals":1.912088,"xp_assists":0.182443,"xp_clean_sheet":0.0,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.002699,"xp_bonus":0.39375,"xp_yellow_cards":-0.133175,"xp_red_cards":-0.004381,"xp_own_goals":-0.002068,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.016025,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":12,"web_name":"Saka","team_short_name":"ARS","position":"MID","price":9.5,"gameweek":4,"expected_minutes":73.223738,"appearance_probability":0.940136,"projected_goals":0.226596,"projected_assists":0.171928,"expected_points":4.030839,"xp_appearance":1.750751,"xp_goals":1.132981,"xp_assists":0.515783,"xp_clean_sheet":0.30285,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.118626,"xp_bonus":0.347913,"xp_yellow_cards":-0.126849,"xp_red_cards":-0.004111,"xp_own_goals":-0.003271,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.003835,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":368,"web_name":"Szoboszlai","team_short_name":"LIV","position":"MID","price":7.0,"gameweek":4,"expected_minutes":81.199429,"appearance_probability":0.956125,"projected_goals":0.173611,"projected_assists":0.162114,"expected_points":3.901513,"xp_appearance":1.837125,"xp_goals":0.868057,"xp_assists":0.486342,"xp_clean_sheet":0.29297,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.254664,"xp_bonus":0.318154,"xp_yellow_cards":-0.143359,"xp_red_cards":-0.00456,"xp_own_goals":-0.003627,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.004252,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":356,"web_name":"Virgil","team_short_name":"LIV","position":"DEF","price":6.5,"gameweek":4,"expected_minutes":83.739993,"appearance_probability":0.95326,"projected_goals":0.054748,"projected_assists":0.047617,"expected_points":3.743059,"xp_appearance":1.865429,"xp_goals":0.328488,"xp_assists":0.142852,"xp_clean_sheet":1.183715,"xp_saves":0.0,"xp_goals_conceded":-0.307489,"xp_defensive_contribution":0.439045,"xp_bonus":0.263324,"xp_yellow_cards":-0.14997,"xp_red_cards":-0.004704,"xp_own_goals":-0.017633,"xp_penalty_saves":0.0,"xp_penalty_misses":0.0,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":202,"web_name":"Richards","team_short_name":"CRY","position":"DEF","price":5.0,"gameweek":4,"expected_minutes":79.264643,"appearance_probability":0.930918,"projected_goals":0.046073,"projected_assists":0.043173,"expected_points":3.731201,"xp_appearance":1.789336,"xp_goals":0.276436,"xp_assists":0.12952,"xp_clean_sheet":1.16351,"xp_saves":0.0,"xp_goals_conceded":-0.272244,"xp_defensive_contribution":0.568655,"xp_bonus":0.239926,"xp_yellow_cards":-0.142795,"xp_red_cards":-0.004453,"xp_own_goals":-0.01669,"xp_penalty_saves":0.0,"xp_penalty_misses":0.0,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":427,"web_name":"Mbeumo","team_short_name":"MUN","position":"MID","price":7.9,"gameweek":4,"expected_minutes":77.63828,"appearance_probability":0.960229,"projected_goals":0.215325,"projected_assists":0.14066,"expected_points":3.719228,"xp_appearance":1.820301,"xp_goals":1.076625,"xp_assists":0.421981,"xp_clean_sheet":0.188806,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.04163,"xp_bonus":0.317643,"xp_yellow_cards":-0.135865,"xp_red_cards":-0.00436,"xp_own_goals":-0.003468,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.004066,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":481,"web_name":"Anderson","team_short_name":"MCI","position":"MID","price":6.3,"gameweek":4,"expected_minutes":75.845234,"appearance_probability":0.953777,"projected_goals":0.116833,"projected_assists":0.108775,"expected_points":3.71237,"xp_appearance":1.789527,"xp_goals":0.584167,"xp_assists":0.326326,"xp_clean_sheet":0.231392,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.677251,"xp_bonus":0.252182,"xp_yellow_cards":-0.136856,"xp_red_cards":-0.00426,"xp_own_goals":-0.003388,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.003972,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":116,"web_name":"Dunk","team_short_name":"BHA","position":"DEF","price":4.5,"gameweek":4,"expected_minutes":79.268736,"appearance_probability":0.930598,"projected_goals":0.04723,"projected_assists":0.044686,"expected_points":3.682648,"xp_appearance":1.7955,"xp_goals":0.283382,"xp_assists":0.134057,"xp_clean_sheet":1.395858,"xp_saves":0.0,"xp_goals_conceded":-0.20832,"xp_defensive_contribution":0.203102,"xp_bonus":0.24337,"xp_yellow_cards":-0.143156,"xp_red_cards":-0.004453,"xp_own_goals":-0.016691,"xp_penalty_saves":0.0,"xp_penalty_misses":0.0,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":13,"web_name":"Rice","team_short_name":"ARS","position":"MID","price":7.4,"gameweek":4,"expected_minutes":79.140766,"appearance_probability":0.957224,"projected_goals":0.138823,"projected_assists":0.134945,"expected_points":3.677735,"xp_appearance":1.830372,"xp_goals":0.694115,"xp_assists":0.404834,"xp_clean_sheet":0.314032,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.313878,"xp_bonus":0.274581,"xp_yellow_cards":-0.141952,"xp_red_cards":-0.004445,"xp_own_goals":-0.003535,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.004144,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":68,"web_name":"Tavernier","team_short_name":"BOU","position":"MID","price":6.0,"gameweek":4,"expected_minutes":78.702405,"appearance_probability":0.943475,"projected_goals":0.18095,"projected_assists":0.145964,"expected_points":3.652574,"xp_appearance":1.799198,"xp_goals":0.904749,"xp_assists":0.437893,"xp_clean_sheet":0.213477,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.142086,"xp_bonus":0.306262,"xp_yellow_cards":-0.139035,"xp_red_cards":-0.00442,"xp_own_goals":-0.003515,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.004122,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":362,"web_name":"Jacquet","team_short_name":"LIV","position":"DEF","price":5.0,"gameweek":4,"expected_minutes":80.725314,"appearance_probability":0.96373,"projected_goals":0.042543,"projected_assists":0.040285,"expected_points":3.615611,"xp_appearance":1.883123,"xp_goals":0.255256,"xp_assists":0.120856,"xp_clean_sheet":1.262969,"xp_saves":0.0,"xp_goals_conceded":-0.290906,"xp_defensive_contribution":0.315272,"xp_bonus":0.236533,"xp_yellow_cards":-0.145959,"xp_red_cards":-0.004535,"xp_own_goals":-0.016998,"xp_penalty_saves":0.0,"xp_penalty_misses":0.0,"external_prior_available":1.0,"external_prior_reliability":0.928333,"external_event_prior_weight":0.348125,"market_fallback_weight":0.0},{"player_id":4,"web_name":"Gabriel","team_short_name":"ARS","position":"DEF","price":8.0,"gameweek":4,"expected_minutes":83.043369,"appearance_probability":0.951122,"projected_goals":0.049547,"projected_assists":0.049722,"expected_points":3.601215,"xp_appearance":1.84699,"xp_goals":0.297283,"xp_assists":0.149165,"xp_clean_sheet":1.226758,"xp_saves":0.0,"xp_goals_conceded":-0.281974,"xp_defensive_contribution":0.274735,"xp_bonus":0.260221,"xp_yellow_cards":-0.149812,"xp_red_cards":-0.004664,"xp_own_goals":-0.017486,"xp_penalty_saves":0.0,"xp_penalty_misses":0.0,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":109,"web_name":"Verbruggen","team_short_name":"BHA","position":"GK","price":4.5,"gameweek":4,"expected_minutes":82.203459,"appearance_probability":0.924681,"projected_goals":0.0,"projected_assists":0.0,"expected_points":3.600444,"xp_appearance":1.833477,"xp_goals":0.0,"xp_assists":0.0,"xp_clean_sheet":1.442257,"xp_saves":0.377341,"xp_goals_conceded":-0.220633,"xp_defensive_contribution":0.0,"xp_bonus":0.246283,"xp_yellow_cards":-0.137367,"xp_red_cards":-0.004614,"xp_own_goals":-0.011418,"xp_penalty_saves":0.075119,"xp_penalty_misses":0.0,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":115,"web_name":"De Cuyper","team_short_name":"BHA","position":"DEF","price":4.8,"gameweek":4,"expected_minutes":68.17322,"appearance_probability":0.912703,"projected_goals":0.049996,"projected_assists":0.104764,"expected_points":3.585673,"xp_appearance":1.648632,"xp_goals":0.299975,"xp_assists":0.314293,"xp_clean_sheet":1.277754,"xp_saves":0.0,"xp_goals_conceded":-0.163479,"xp_defensive_contribution":0.072569,"xp_bonus":0.27424,"xp_yellow_cards":-0.120127,"xp_red_cards":-0.003828,"xp_own_goals":-0.014355,"xp_penalty_saves":0.0,"xp_penalty_misses":0.0,"external_prior_available":1.0,"external_prior_reliability":0.785556,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":586,"web_name":"Mendy","team_short_name":"HUL","position":"DEF","price":4.1,"gameweek":4,"expected_minutes":74.186004,"appearance_probability":0.944796,"projected_goals":0.070721,"projected_assists":0.029684,"expected_points":3.582205,"xp_appearance":1.757989,"xp_goals":0.424324,"xp_assists":0.089053,"xp_clean_sheet":1.017916,"xp_saves":0.0,"xp_goals_conceded":-0.311097,"xp_defensive_contribution":0.532914,"xp_bonus":0.224045,"xp_yellow_cards":-0.133152,"xp_red_cards":-0.004167,"xp_own_goals":-0.015621,"xp_penalty_saves":0.0,"xp_penalty_misses":0.0,"external_prior_available":1.0,"external_prior_reliability":0.998333,"external_event_prior_weight":0.374375,"market_fallback_weight":0.0},{"player_id":69,"web_name":"Scott","team_short_name":"BOU","position":"MID","price":6.1,"gameweek":4,"expected_minutes":74.853491,"appearance_probability":0.935521,"projected_goals":0.111783,"projected_assists":0.10337,"expected_points":3.581911,"xp_appearance":1.75675,"xp_goals":0.558916,"xp_assists":0.31011,"xp_clean_sheet":0.214362,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.64353,"xp_bonus":0.244256,"xp_yellow_cards":-0.134544,"xp_red_cards":-0.004204,"xp_own_goals":-0.003344,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.00392,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":124,"web_name":"Groß","team_short_name":"BHA","position":"MID","price":5.6,"gameweek":4,"expected_minutes":78.649103,"appearance_probability":0.944799,"projected_goals":0.130251,"projected_assists":0.159798,"expected_points":3.533503,"xp_appearance":1.79658,"xp_goals":0.651253,"xp_assists":0.479393,"xp_clean_sheet":0.347339,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.105075,"xp_bonus":0.305423,"xp_yellow_cards":-0.139511,"xp_red_cards":-0.004417,"xp_own_goals":-0.003513,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.004119,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.375,"market_fallback_weight":0.0},{"player_id":588,"web_name":"Khalaili","team_short_name":"CRY","position":"DEF","price":5.0,"gameweek":4,"expected_minutes":76.670878,"appearance_probability":0.958983,"projected_goals":0.053731,"projected_assists":0.050519,"expected_points":3.49922,"xp_appearance":1.805649,"xp_goals":0.322383,"xp_assists":0.151556,"xp_clean_sheet":1.209898,"xp_saves":0.0,"xp_goals_conceded":-0.258671,"xp_defensive_contribution":0.561907,"xp_bonus":0.236572,"xp_yellow_cards":-0.138108,"xp_red_cards":-0.004306,"xp_own_goals":-0.016144,"xp_penalty_saves":0.0,"xp_penalty_misses":0.0,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.375,"market_fallback_weight":0.1875},{"player_id":154,"web_name":"Palmer","team_short_name":"CHE","position":"MID","price":9.7,"gameweek":4,"expected_minutes":76.318858,"appearance_probability":0.938472,"projected_goals":0.189976,"projected_assists":0.123628,"expected_points":3.49106,"xp_appearance":1.776871,"xp_goals":0.949879,"xp_assists":0.370885,"xp_clean_sheet":0.194277,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.044165,"xp_bonus":0.301053,"xp_yellow_cards":-0.134378,"xp_red_cards":-0.004286,"xp_own_goals":-0.003409,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.003997,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":203,"web_name":"Canvot","team_short_name":"CRY","position":"DEF","price":4.9,"gameweek":4,"expected_minutes":77.670436,"appearance_probability":0.922167,"projected_goals":0.044689,"projected_assists":0.041761,"expected_points":3.488505,"xp_appearance":1.770149,"xp_goals":0.268134,"xp_assists":0.125284,"xp_clean_sheet":1.168187,"xp_saves":0.0,"xp_goals_conceded":-0.263884,"xp_defensive_contribution":0.342484,"xp_bonus":0.238341,"xp_yellow_cards":-0.139474,"xp_red_cards":-0.004363,"xp_own_goals":-0.016355,"xp_penalty_saves":0.0,"xp_penalty_misses":0.0,"external_prior_available":1.0,"external_prior_reliability":0.742222,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":338,"web_name":"Ampadu","team_short_name":"LEE","position":"MID","price":5.5,"gameweek":4,"expected_minutes":76.931563,"appearance_probability":0.919658,"projected_goals":0.096174,"projected_assists":0.082727,"expected_points":3.452204,"xp_appearance":1.747398,"xp_goals":0.480871,"xp_assists":0.248181,"xp_clean_sheet":0.185193,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.717476,"xp_bonus":0.224723,"xp_yellow_cards":-0.139851,"xp_red_cards":-0.004322,"xp_own_goals":-0.003436,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.004029,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":26,"web_name":"Havertz","team_short_name":"ARS","position":"FWD","price":7.5,"gameweek":4,"expected_minutes":70.313946,"appearance_probability":0.927322,"projected_goals":0.352571,"projected_assists":0.057218,"expected_points":3.447415,"xp_appearance":1.686128,"xp_goals":1.410284,"xp_assists":0.171655,"xp_clean_sheet":0.0,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.002381,"xp_bonus":0.319063,"xp_yellow_cards":-0.121845,"xp_red_cards":-0.003948,"xp_own_goals":-0.001863,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.014441,"external_prior_available":1.0,"external_prior_reliability":0.322778,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":397,"web_name":"Semenyo","team_short_name":"MCI","position":"MID","price":8.4,"gameweek":4,"expected_minutes":81.648465,"appearance_probability":0.963454,"projected_goals":0.159169,"projected_assists":0.112486,"expected_points":3.42706,"xp_appearance":1.865265,"xp_goals":0.795846,"xp_assists":0.337459,"xp_clean_sheet":0.23743,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.058157,"xp_bonus":0.291205,"xp_yellow_cards":-0.145793,"xp_red_cards":-0.004586,"xp_own_goals":-0.003647,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.004276,"external_prior_available":1.0,"external_prior_reliability":0.998889,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":366,"web_name":"Wirtz","team_short_name":"LIV","position":"MID","price":7.4,"gameweek":4,"expected_minutes":72.154163,"appearance_probability":0.958373,"projected_goals":0.161405,"projected_assists":0.120911,"expected_points":3.426552,"xp_appearance":1.768476,"xp_goals":0.807023,"xp_assists":0.362734,"xp_clean_sheet":0.294083,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.048958,"xp_bonus":0.283274,"xp_yellow_cards":-0.126941,"xp_red_cards":-0.004052,"xp_own_goals":-0.003223,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.003779,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":544,"web_name":"Xhaka","team_short_name":"SUN","position":"MID","price":5.5,"gameweek":4,"expected_minutes":79.584053,"appearance_probability":0.940347,"projected_goals":0.109541,"projected_assists":0.103519,"expected_points":3.423704,"xp_appearance":1.797644,"xp_goals":0.547706,"xp_assists":0.310557,"xp_clean_sheet":0.150709,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.530459,"xp_bonus":0.243427,"xp_yellow_cards":-0.144606,"xp_red_cards":-0.004471,"xp_own_goals":-0.003555,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.004168,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":210,"web_name":"Wharton","team_short_name":"CRY","position":"MID","price":5.5,"gameweek":4,"expected_minutes":73.339112,"appearance_probability":0.949252,"projected_goals":0.117894,"projected_assists":0.120551,"expected_points":3.414269,"xp_appearance":1.749287,"xp_goals":0.589468,"xp_assists":0.361653,"xp_clean_sheet":0.29056,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.311723,"xp_bonus":0.253869,"xp_yellow_cards":-0.131056,"xp_red_cards":-0.004119,"xp_own_goals":-0.003276,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.003841,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":229,"web_name":"Tarkowski","team_short_name":"EVE","position":"DEF","price":6.0,"gameweek":4,"expected_minutes":83.637573,"appearance_probability":0.95505,"projected_goals":0.047392,"projected_assists":0.048543,"expected_points":3.40878,"xp_appearance":1.864997,"xp_goals":0.284355,"xp_assists":0.145628,"xp_clean_sheet":0.844823,"xp_saves":0.0,"xp_goals_conceded":-0.451296,"xp_defensive_contribution":0.650497,"xp_bonus":0.244366,"xp_yellow_cards":-0.15228,"xp_red_cards":-0.004698,"xp_own_goals":-0.017611,"xp_penalty_saves":0.0,"xp_penalty_misses":0.0,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":237,"web_name":"Ndiaye","team_short_name":"MCI","position":"MID","price":5.9,"gameweek":4,"expected_minutes":76.508787,"appearance_probability":0.932512,"projected_goals":0.125073,"projected_assists":0.098494,"expected_points":3.399903,"xp_appearance":1.763548,"xp_goals":0.625367,"xp_assists":0.295483,"xp_clean_sheet":0.223727,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.38635,"xp_bonus":0.254736,"xp_yellow_cards":-0.137587,"xp_red_cards":-0.004297,"xp_own_goals":-0.003417,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.004007,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":480,"web_name":"Gibbs-White","team_short_name":"NFO","position":"MID","price":7.9,"gameweek":4,"expected_minutes":79.693742,"appearance_probability":0.954806,"projected_goals":0.170733,"projected_assists":0.114645,"expected_points":3.395237,"xp_appearance":1.821505,"xp_goals":0.853663,"xp_assists":0.343935,"xp_clean_sheet":0.194466,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.043528,"xp_bonus":0.292091,"xp_yellow_cards":-0.141742,"xp_red_cards":-0.004476,"xp_own_goals":-0.00356,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.004173,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0}],"starting_xi":[{"position":"GK","web_name":"Verbruggen","team_short_name":"BHA","fixtures":"COV (A)","expected_minutes":82.2,"appearance_probability":0.925,"expected_points":3.6,"role":null},{"position":"DEF","web_name":"Virgil","team_short_name":"LIV","fixtures":"FUL (H)","expected_minutes":83.7,"appearance_probability":0.953,"expected_points":3.74,"role":null},{"position":"DEF","web_name":"Richards","team_short_name":"CRY","fixtures":"IPS (H)","expected_minutes":79.3,"appearance_probability":0.931,"expected_points":3.73,"role":null},{"position":"DEF","web_name":"Botman","team_short_name":"NEW","fixtures":"LEE (A)","expected_minutes":78.5,"appearance_probability":0.928,"expected_points":3.38,"role":null},{"position":"DEF","web_name":"Thiaw","team_short_name":"NEW","fixtures":"LEE (A)","expected_minutes":82.4,"appearance_probability":0.955,"expected_points":3.21,"role":null},{"position":"MID","web_name":"B.Fernandes","team_short_name":"MUN","fixtures":"MCI (H)","expected_minutes":82.8,"appearance_probability":0.958,"expected_points":4.22,"role":"C"},{"position":"MID","web_name":"Saka","team_short_name":"ARS","fixtures":"SUN (A)","expected_minutes":73.2,"appearance_probability":0.94,"expected_points":4.03,"role":"VC"},{"position":"MID","web_name":"Szoboszlai","team_short_name":"LIV","fixtures":"FUL (H)","expected_minutes":81.2,"appearance_probability":0.956,"expected_points":3.9,"role":null},{"position":"MID","web_name":"Tavernier","team_short_name":"BOU","fixtures":"BRE (H)","expected_minutes":78.7,"appearance_probability":0.943,"expected_points":3.65,"role":null},{"position":"FWD","web_name":"Havertz","team_short_name":"ARS","fixtures":"SUN (A)","expected_minutes":70.3,"appearance_probability":0.927,"expected_points":3.45,"role":null},{"position":"FWD","web_name":"Calvert-Lewin","team_short_name":"LEE","fixtures":"NEW (H)","expected_minutes":70.8,"appearance_probability":0.918,"expected_points":3.38,"role":null}],"bench":[{"bench_slot":"1","position":"FWD","web_name":"João Pedro","team_short_name":"CHE","fixtures":"HUL (H)","expected_minutes":74.1,"appearance_probability":0.915,"expected_points":3.21},{"bench_slot":"2","position":"MID","web_name":"Enzo","team_short_name":"MCI","fixtures":"MUN (A)","expected_minutes":65.8,"appearance_probability":0.881,"expected_points":2.93},{"bench_slot":"3","position":"DEF","web_name":"Milenković","team_short_name":"NFO","fixtures":"AVL (A)","expected_minutes":83.7,"appearance_probability":0.958,"expected_points":3.03},{"bench_slot":"GK","position":"GK","web_name":"Lammens","team_short_name":"MUN","fixtures":"MCI (H)","expected_minutes":83.4,"appearance_probability":0.932,"expected_points":2.73}],"captain_alternatives":[{"web_name":"B.Fernandes","expected_points":4.22,"expected_minutes":82.834,"appearance_probability":0.958,"xp_gap_to_top":0.0,"within_captain_tie_band":true,"selected":true},{"web_name":"Saka","expected_points":4.031,"expected_minutes":73.224,"appearance_probability":0.94,"xp_gap_to_top":0.189,"within_captain_tie_band":false,"selected":false},{"web_name":"Szoboszlai","expected_points":3.902,"expected_minutes":81.199,"appearance_probability":0.956,"xp_gap_to_top":0.318,"within_captain_tie_band":false,"selected":false},{"web_name":"Virgil","expected_points":3.743,"expected_minutes":83.74,"appearance_probability":0.953,"xp_gap_to_top":0.477,"within_captain_tie_band":false,"selected":false},{"web_name":"Richards","expected_points":3.731,"expected_minutes":79.265,"appearance_probability":0.931,"xp_gap_to_top":0.488,"within_captain_tie_band":false,"selected":false},{"web_name":"Tavernier","expected_points":3.653,"expected_minutes":78.702,"appearance_probability":0.943,"xp_gap_to_top":0.567,"within_captain_tie_band":false,"selected":false},{"web_name":"Verbruggen","expected_points":3.6,"expected_minutes":82.203,"appearance_probability":0.925,"xp_gap_to_top":0.619,"within_captain_tie_band":false,"selected":false},{"web_name":"Havertz","expected_points":3.447,"expected_minutes":70.314,"appearance_probability":0.927,"xp_gap_to_top":0.772,"within_captain_tie_band":false,"selected":false}],"transfer_evidence":[{"outgoing":"Maguire","incoming":"Richards","position":"DEF","price":5.0,"expected_points":3.731,"xp_defensive_contribution":0.569,"dc_share_of_xp":0.152,"prior_pl_fixtures":155.0,"prior_season_fixtures":3.0,"external_prior_available":true,"external_event_prior_weight":0.0,"external_prior_match_method":"exact_name","external_source_leagues":"ENG-Premier League","external_prior_source_note":null,"market_fallback_weight":0.0,"selected_transfer_count":2,"best_lower_transfer_count":1,"incremental_objective_gain_vs_lower_count":0.736,"cheaper_alternative":"Dunk","cheaper_alternative_price":4.5,"cheaper_alternative_xp":3.683,"cheaper_alternative_xp_gap":0.049,"cheaper_alternative_saving":0.5,"action_extra_transfer_fragile":true,"evidence_level":"LOW","evidence_reasons":"15% of xP comes from defensive contributions o..."},{"outgoing":"Palmer","incoming":"Saka","position":"MID","price":9.5,"expected_points":4.031,"xp_defensive_contribution":0.119,"dc_share_of_xp":0.029,"prior_pl_fixtures":155.0,"prior_season_fixtures":3.0,"external_prior_available":true,"external_event_prior_weight":0.0,"external_prior_match_method":"exact_name","external_source_leagues":"ENG-Premier League","external_prior_source_note":null,"market_fallback_weight":0.0,"selected_transfer_count":2,"best_lower_transfer_count":1,"incremental_objective_gain_vs_lower_count":0.736,"cheaper_alternative":null,"cheaper_alternative_price":null,"cheaper_alternative_xp":null,"cheaper_alternative_xp_gap":null,"cheaper_alternative_saving":null,"action_extra_transfer_fragile":true,"evidence_level":"HIGH","evidence_reasons":"No low-evidence flags."}],"transfer_alternatives":[{"current_action":"Maguire → Richards; Palmer → Saka","transfer_count":2,"current_hit_cost":0.0,"bank_after":0.6,"path_objective":61.62,"structural_deadwood_after":0.0,"raw_best":true,"selected":true,"exact_subset_comparator":false,"best_for_transfer_count":true},{"current_action":"Maguire → Richards; Enzo → Anderson","transfer_count":2,"current_hit_cost":0.0,"bank_after":1.1,"path_objective":61.59,"structural_deadwood_after":0.0,"raw_best":false,"selected":false,"exact_subset_comparator":false,"best_for_transfer_count":false},{"current_action":"Milenković → Richards; Enzo → Anderson","transfer_count":2,"current_hit_cost":0.0,"bank_after":1.7,"path_objective":61.564,"structural_deadwood_after":0.0,"raw_best":false,"selected":false,"exact_subset_comparator":false,"best_for_transfer_count":false},{"current_action":"Palmer → Saka; Maguire → Dunk","transfer_count":2,"current_hit_cost":0.0,"bank_after":1.1,"path_objective":61.555,"structural_deadwood_after":0.0,"raw_best":false,"selected":false,"exact_subset_comparator":false,"best_for_transfer_count":false},{"current_action":"Maguire → Richards; Enzo → Rice","transfer_count":2,"current_hit_cost":0.0,"bank_after":0.0,"path_objective":61.543,"structural_deadwood_after":0.0,"raw_best":false,"selected":false,"exact_subset_comparator":false,"best_for_transfer_count":false},{"current_action":"Maguire → Richards; Milenković → Dunk","transfer_count":2,"current_hit_cost":0.0,"bank_after":1.5,"path_objective":61.54,"structural_deadwood_after":0.0,"raw_best":false,"selected":false,"exact_subset_comparator":false,"best_for_transfer_count":false},{"current_action":"Maguire → Dunk; Enzo → Mbeumo","transfer_count":2,"current_hit_cost":0.0,"bank_after":0.0,"path_objective":61.534,"structural_deadwood_after":0.0,"raw_best":false,"selected":false,"exact_subset_comparator":false,"best_for_transfer_count":false},{"current_action":"Enzo → Anderson; Maguire → Dunk","transfer_count":2,"current_hit_cost":0.0,"bank_after":1.6,"path_objective":61.524,"structural_deadwood_after":0.0,"raw_best":false,"selected":false,"exact_subset_comparator":false,"best_for_transfer_count":false},{"current_action":"Milenković → Richards; Enzo → Rice","transfer_count":2,"current_hit_cost":0.0,"bank_after":0.6,"path_objective":61.517,"structural_deadwood_after":0.0,"raw_best":false,"selected":false,"exact_subset_comparator":false,"best_for_transfer_count":false},{"current_action":"Enzo → Rice; Maguire → Dunk","transfer_count":2,"current_hit_cost":0.0,"bank_after":0.5,"path_objective":61.478,"structural_deadwood_after":0.0,"raw_best":false,"selected":false,"exact_subset_comparator":false,"best_for_transfer_count":false},{"current_action":"Maguire → Richards","transfer_count":1,"current_hit_cost":0.0,"bank_after":0.5,"path_objective":60.884,"structural_deadwood_after":0.0,"raw_best":false,"selected":false,"exact_subset_comparator":false,"best_for_transfer_count":true},{"current_action":"Roll","transfer_count":0,"current_hit_cost":0.0,"bank_after":0.6,"path_objective":60.131,"structural_deadwood_after":0.0,"raw_best":false,"selected":false,"exact_subset_comparator":false,"best_for_transfer_count":true}]},"5":{"source":"recovered_from_executed_notebook_output","partial_player_pool":true,"player_pool_scope":"displayed_top30","gameweek":5,"summary":{"generated_at_utc":"2026-09-18T06:46:02.227757+00:00","mode":"Live in-season squad","gameweek":"5","deadline_spain":"2026-09-18T19:30:00+02:00","policy":"greedy_1gw_no_hits","policy_source":"validated Section 11 walk-forward selection — ...","policy_validated":"True","transfer_search_profile":"standard","transfer_search_confidence":"medium","transfer_search_escalated":"True","transfer_search_signature":"f2b963340451175c2ecc","production_readiness":"READY_WITH_DEGRADED_OPTIONAL_CONTEXT","planning_horizon":"1","primary_action":"Milenković → N.Williams; Thiaw → Tarkowski","normal_transfer_action":"Milenković → N.Williams; Thiaw → Tarkowski","chip_action":"Save all chips","captain":"B.Fernandes","vice_captain":"N.Williams","captain_confidence":"Low","captain_xp_gap_vs_best_other":"0.079127","execution_policy":"baseline","execution_policy_source":"forced off","expected_current_score":"46.791879","current_hit_cost":"0.0","bank_after_normal_action":"0.4","gain_vs_roll_objective":"2.855396","decision_confidence":"Low","objective_gap_to_second_action":"0.007115","transfer_robustness_gate_applied":"False","transfer_robustness_objective_given_up":"0.0","transfer_robustness_reason":"Transfer robustness gates disabled or not appl...","policy_bank_aware":"False","marginal_transfer_gate_enabled":"False","min_gain_per_extra_transfer":"1.0","transfer_evidence_low":"True","low_evidence_incoming_count":"1","incremental_objective_gain_vs_lower_transfer_c...":"1.045405","data_fresh":"True","all_comp_workload_enabled":"True","all_comp_status":"CIRCUIT_OPEN","all_comp_unique_events":"0","all_comp_fixture_rows":"0","all_comp_circuit_open":"True","all_comp_remaining_requests_skipped":"58","all_comp_failure_cache_active":"True","all_comp_player_minute_rows":"0","current_fbref_status":"PARTIAL","current_fbref_matched_players":"259","current_fbref_refresh_source":"soccerdata_refresh","current_fbref_npxg_coverage":"0.0","current_fbref_xag_coverage":"0.0","set_piece_role_enabled":"True","autonomous_mode":"True","xp_validation_artifact_loaded":"True","xp_calibration_applied":"False","premium_attacker_adjustment_applied":"False","latest_completed_gameweek":"4","latest_stats_gameweek_loaded":"4","latest_gameweek_stats_used":"True","early_season_shrinkage_enabled":"False","early_season_shrinkage_ready":"True","early_season_sparse_players_active":"0","early_season_sparse_teams_active":"0","live_feature_signature":"4a722404f8bd59fd4fc3"},"player_forecasts":[{"player_id":411,"web_name":"Haaland","team_short_name":"MCI","position":"FWD","price":15.6,"gameweek":5,"expected_minutes":79.312968,"appearance_probability":0.945617,"projected_goals":0.612427,"projected_assists":0.068218,"expected_points":4.760076,"xp_appearance":1.81355,"xp_goals":2.449706,"xp_assists":0.204655,"xp_clean_sheet":0.0,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.002506,"xp_bonus":0.445336,"xp_yellow_cards":-0.132835,"xp_red_cards":-0.004452,"xp_own_goals":-0.002102,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.016289,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":426,"web_name":"B.Fernandes","team_short_name":"MUN","position":"MID","price":12.0,"gameweek":5,"expected_minutes":82.291906,"appearance_probability":0.958721,"projected_goals":0.24758,"projected_assists":0.211237,"expected_points":4.264058,"xp_appearance":1.846877,"xp_goals":1.237901,"xp_assists":0.633712,"xp_clean_sheet":0.251914,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.065881,"xp_bonus":0.384071,"xp_yellow_cards":-0.143692,"xp_red_cards":-0.004621,"xp_own_goals":-0.003676,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.00431,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":469,"web_name":"N.Williams","team_short_name":"NFO","position":"DEF","price":5.0,"gameweek":5,"expected_minutes":80.828186,"appearance_probability":0.945331,"projected_goals":0.05008,"projected_assists":0.067101,"expected_points":4.184932,"xp_appearance":1.831352,"xp_goals":0.30048,"xp_assists":0.201303,"xp_clean_sheet":1.61015,"xp_saves":0.0,"xp_goals_conceded":-0.171567,"xp_defensive_contribution":0.300204,"xp_bonus":0.278298,"xp_yellow_cards":-0.143729,"xp_red_cards":-0.00454,"xp_own_goals":-0.017019,"xp_penalty_saves":0.0,"xp_penalty_misses":0.0,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":480,"web_name":"Gibbs-White","team_short_name":"NFO","position":"MID","price":8.0,"gameweek":5,"expected_minutes":80.408265,"appearance_probability":0.953739,"projected_goals":0.220172,"projected_assists":0.155125,"expected_points":4.040208,"xp_appearance":1.827909,"xp_goals":1.100861,"xp_assists":0.465376,"xp_clean_sheet":0.399734,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.039663,"xp_bonus":0.358142,"xp_yellow_cards":-0.13916,"xp_red_cards":-0.004515,"xp_own_goals":-0.003592,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.004211,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":229,"web_name":"Tarkowski","team_short_name":"EVE","position":"DEF","price":6.1,"gameweek":5,"expected_minutes":83.908576,"appearance_probability":0.954248,"projected_goals":0.050907,"projected_assists":0.05397,"expected_points":4.018156,"xp_appearance":1.868232,"xp_goals":0.305442,"xp_assists":0.161911,"xp_clean_sheet":1.258119,"xp_saves":0.0,"xp_goals_conceded":-0.284037,"xp_defensive_contribution":0.607599,"xp_bonus":0.273247,"xp_yellow_cards":-0.149975,"xp_red_cards":-0.004713,"xp_own_goals":-0.017668,"xp_penalty_saves":0.0,"xp_penalty_misses":0.0,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":481,"web_name":"Anderson","team_short_name":"MCI","position":"MID","price":6.3,"gameweek":5,"expected_minutes":77.640553,"appearance_probability":0.9542,"projected_goals":0.133458,"projected_assists":0.123507,"expected_points":4.010141,"xp_appearance":1.805169,"xp_goals":0.667292,"xp_assists":0.370521,"xp_clean_sheet":0.333473,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.705724,"xp_bonus":0.277787,"xp_yellow_cards":-0.13793,"xp_red_cards":-0.004361,"xp_own_goals":-0.003468,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.004066,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":12,"web_name":"Saka","team_short_name":"ARS","position":"MID","price":9.5,"gameweek":5,"expected_minutes":75.813068,"appearance_probability":0.949555,"projected_goals":0.23314,"projected_assists":0.171134,"expected_points":3.978633,"xp_appearance":1.783458,"xp_goals":1.165698,"xp_assists":0.513401,"xp_clean_sheet":0.260287,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.048596,"xp_bonus":0.350342,"xp_yellow_cards":-0.131536,"xp_red_cards":-0.004257,"xp_own_goals":-0.003386,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.00397,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":472,"web_name":"Murillo","team_short_name":"NFO","position":"DEF","price":5.5,"gameweek":5,"expected_minutes":75.117183,"appearance_probability":0.891918,"projected_goals":0.046333,"projected_assists":0.046217,"expected_points":3.97661,"xp_appearance":1.7116,"xp_goals":0.278,"xp_assists":0.13865,"xp_clean_sheet":1.500894,"xp_saves":0.0,"xp_goals_conceded":-0.152238,"xp_defensive_contribution":0.42165,"xp_bonus":0.233569,"xp_yellow_cards":-0.135479,"xp_red_cards":-0.004219,"xp_own_goals":-0.015817,"xp_penalty_saves":0.0,"xp_penalty_misses":0.0,"external_prior_available":0.0,"external_prior_reliability":null,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":467,"web_name":"Sels","team_short_name":"NFO","position":"GK","price":5.0,"gameweek":5,"expected_minutes":82.642378,"appearance_probability":0.930054,"projected_goals":0.0,"projected_assists":0.0,"expected_points":3.85455,"xp_appearance":1.845315,"xp_goals":0.0,"xp_assists":0.0,"xp_clean_sheet":1.641327,"xp_saves":0.362033,"xp_goals_conceded":-0.177839,"xp_defensive_contribution":0.0,"xp_bonus":0.258945,"xp_yellow_cards":-0.134633,"xp_red_cards":-0.004638,"xp_own_goals":-0.011479,"xp_penalty_saves":0.07552,"xp_penalty_misses":0.0,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":239,"web_name":"Garner","team_short_name":"EVE","position":"MID","price":6.0,"gameweek":5,"expected_minutes":77.23175,"appearance_probability":0.955048,"projected_goals":0.126099,"projected_assists":0.125192,"expected_points":3.84287,"xp_appearance":1.810829,"xp_goals":0.630494,"xp_assists":0.375577,"xp_clean_sheet":0.311953,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.609499,"xp_bonus":0.255106,"xp_yellow_cards":-0.138755,"xp_red_cards":-0.004338,"xp_own_goals":-0.00345,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.004045,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":201,"web_name":"Muñoz","team_short_name":"NFO","position":"DEF","price":5.4,"gameweek":5,"expected_minutes":77.560146,"appearance_probability":0.916798,"projected_goals":0.04818,"projected_assists":0.057493,"expected_points":3.785153,"xp_appearance":1.761459,"xp_goals":0.289079,"xp_assists":0.172479,"xp_clean_sheet":1.534468,"xp_saves":0.0,"xp_goals_conceded":-0.160426,"xp_defensive_contribution":0.087173,"xp_bonus":0.260536,"xp_yellow_cards":-0.138929,"xp_red_cards":-0.004356,"xp_own_goals":-0.016331,"xp_penalty_saves":0.0,"xp_penalty_misses":0.0,"external_prior_available":0.0,"external_prior_reliability":null,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":427,"web_name":"Mbeumo","team_short_name":"MUN","position":"MID","price":7.9,"gameweek":5,"expected_minutes":78.188578,"appearance_probability":0.959053,"projected_goals":0.208722,"projected_assists":0.142114,"expected_points":3.757754,"xp_appearance":1.821623,"xp_goals":1.043612,"xp_assists":0.426341,"xp_clean_sheet":0.257512,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.038634,"xp_bonus":0.319409,"xp_yellow_cards":-0.137399,"xp_red_cards":-0.004391,"xp_own_goals":-0.003492,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.004095,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":334,"web_name":"Muharemović","team_short_name":"LEE","position":"DEF","price":5.0,"gameweek":5,"expected_minutes":84.248309,"appearance_probability":0.958975,"projected_goals":0.061235,"projected_assists":0.041165,"expected_points":3.752452,"xp_appearance":1.883243,"xp_goals":0.367411,"xp_assists":0.123494,"xp_clean_sheet":1.00244,"xp_saves":0.0,"xp_goals_conceded":-0.391051,"xp_defensive_contribution":0.700274,"xp_bonus":0.241438,"xp_yellow_cards":-0.152326,"xp_red_cards":-0.004732,"xp_own_goals":-0.01774,"xp_penalty_saves":0.0,"xp_penalty_misses":0.0,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.3,"market_fallback_weight":0.0},{"player_id":388,"web_name":"Guéhi","team_short_name":"MCI","position":"DEF","price":6.0,"gameweek":5,"expected_minutes":82.541126,"appearance_probability":0.944398,"projected_goals":0.054845,"projected_assists":0.056879,"expected_points":3.735851,"xp_appearance":1.843292,"xp_goals":0.329071,"xp_assists":0.170637,"xp_clean_sheet":1.346682,"xp_saves":0.0,"xp_goals_conceded":-0.245273,"xp_defensive_contribution":0.175353,"xp_bonus":0.284594,"xp_yellow_cards":-0.146489,"xp_red_cards":-0.004636,"xp_own_goals":-0.01738,"xp_penalty_saves":0.0,"xp_penalty_misses":0.0,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":397,"web_name":"Semenyo","team_short_name":"MCI","position":"MID","price":8.4,"gameweek":5,"expected_minutes":81.844723,"appearance_probability":0.961396,"projected_goals":0.181481,"projected_assists":0.128898,"expected_points":3.707884,"xp_appearance":1.858267,"xp_goals":0.907406,"xp_assists":0.386694,"xp_clean_sheet":0.342727,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.050009,"xp_bonus":0.319122,"xp_yellow_cards":-0.143803,"xp_red_cards":-0.004596,"xp_own_goals":-0.003656,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.004286,"external_prior_available":1.0,"external_prior_reliability":0.998889,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":368,"web_name":"Szoboszlai","team_short_name":"LIV","position":"MID","price":7.0,"gameweek":5,"expected_minutes":80.69725,"appearance_probability":0.957068,"projected_goals":0.156724,"projected_assists":0.148556,"expected_points":3.699958,"xp_appearance":1.832584,"xp_goals":0.783618,"xp_assists":0.445667,"xp_clean_sheet":0.24075,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.258641,"xp_bonus":0.295308,"xp_yellow_cards":-0.144247,"xp_red_cards":-0.004532,"xp_own_goals":-0.003605,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.004226,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":233,"web_name":"Mykolenko","team_short_name":"EVE","position":"DEF","price":4.5,"gameweek":5,"expected_minutes":82.775109,"appearance_probability":0.948019,"projected_goals":0.044468,"projected_assists":0.050372,"expected_points":3.678234,"xp_appearance":1.851216,"xp_goals":0.266809,"xp_assists":0.151116,"xp_clean_sheet":1.249494,"xp_saves":0.0,"xp_goals_conceded":-0.278205,"xp_defensive_contribution":0.352798,"xp_bonus":0.256438,"xp_yellow_cards":-0.149352,"xp_red_cards":-0.004649,"xp_own_goals":-0.017429,"xp_penalty_saves":0.0,"xp_penalty_misses":0.0,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":565,"web_name":"M.Sangaré","team_short_name":"BRE","position":"MID","price":5.7,"gameweek":5,"expected_minutes":77.588459,"appearance_probability":0.966681,"projected_goals":0.127073,"projected_assists":0.107393,"expected_points":3.6724,"xp_appearance":1.834885,"xp_goals":0.635365,"xp_assists":0.32218,"xp_clean_sheet":0.24456,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.505233,"xp_bonus":0.279764,"xp_yellow_cards":-0.137701,"xp_red_cards":-0.004357,"xp_own_goals":-0.003466,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.004063,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.3,"market_fallback_weight":0.0},{"player_id":473,"web_name":"Aina","team_short_name":"NFO","position":"DEF","price":4.5,"gameweek":5,"expected_minutes":76.884426,"appearance_probability":0.900578,"projected_goals":0.042558,"projected_assists":0.04926,"expected_points":3.670825,"xp_appearance":1.729773,"xp_goals":0.255345,"xp_assists":0.14778,"xp_clean_sheet":1.512978,"xp_saves":0.0,"xp_goals_conceded":-0.158149,"xp_defensive_contribution":0.093397,"xp_bonus":0.247675,"xp_yellow_cards":-0.137468,"xp_red_cards":-0.004319,"xp_own_goals":-0.016189,"xp_penalty_saves":0.0,"xp_penalty_misses":0.0,"external_prior_available":1.0,"external_prior_reliability":0.882222,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":338,"web_name":"Ampadu","team_short_name":"LEE","position":"MID","price":5.4,"gameweek":5,"expected_minutes":76.954551,"appearance_probability":0.919794,"projected_goals":0.102004,"projected_assists":0.087967,"expected_points":3.661602,"xp_appearance":1.747428,"xp_goals":0.510018,"xp_assists":0.263902,"xp_clean_sheet":0.229654,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.826901,"xp_bonus":0.23448,"xp_yellow_cards":-0.138991,"xp_red_cards":-0.004323,"xp_own_goals":-0.003437,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.00403,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":346,"web_name":"Calvert-Lewin","team_short_name":"LEE","position":"FWD","price":6.0,"gameweek":5,"expected_minutes":71.912281,"appearance_probability":0.929473,"projected_goals":0.3959,"projected_assists":0.054702,"expected_points":3.642091,"xp_appearance":1.745687,"xp_goals":1.583599,"xp_assists":0.164107,"xp_clean_sheet":0.0,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.002471,"xp_bonus":0.292316,"xp_yellow_cards":-0.125375,"xp_red_cards":-0.004038,"xp_own_goals":-0.001906,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.014769,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":69,"web_name":"Scott","team_short_name":"BOU","position":"MID","price":6.1,"gameweek":5,"expected_minutes":75.007768,"appearance_probability":0.935859,"projected_goals":0.110863,"projected_assists":0.102083,"expected_points":3.558704,"xp_appearance":1.759197,"xp_goals":0.554313,"xp_assists":0.306249,"xp_clean_sheet":0.187955,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.656191,"xp_bonus":0.241393,"xp_yellow_cards":-0.1351,"xp_red_cards":-0.004213,"xp_own_goals":-0.00335,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.003928,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":68,"web_name":"Tavernier","team_short_name":"BOU","position":"MID","price":6.1,"gameweek":5,"expected_minutes":76.672964,"appearance_probability":0.934668,"projected_goals":0.177448,"projected_assists":0.142875,"expected_points":3.546983,"xp_appearance":1.773573,"xp_goals":0.887239,"xp_assists":0.428626,"xp_clean_sheet":0.185705,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.119805,"xp_bonus":0.299229,"xp_yellow_cards":-0.135447,"xp_red_cards":-0.004306,"xp_own_goals":-0.003425,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.004015,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":26,"web_name":"Havertz","team_short_name":"ARS","position":"FWD","price":7.6,"gameweek":5,"expected_minutes":74.647016,"appearance_probability":0.943139,"projected_goals":0.352266,"projected_assists":0.058435,"expected_points":3.504718,"xp_appearance":1.746987,"xp_goals":1.409064,"xp_assists":0.175305,"xp_clean_sheet":0.0,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.00227,"xp_bonus":0.322726,"xp_yellow_cards":-0.130133,"xp_red_cards":-0.004191,"xp_own_goals":-0.001978,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.015331,"external_prior_available":1.0,"external_prior_reliability":0.322778,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":236,"web_name":"Dewsbury-Hall","team_short_name":"EVE","position":"MID","price":6.5,"gameweek":5,"expected_minutes":78.058513,"appearance_probability":0.941434,"projected_goals":0.144006,"projected_assists":0.130445,"expected_points":3.502742,"xp_appearance":1.789335,"xp_goals":0.720031,"xp_assists":0.391334,"xp_clean_sheet":0.303562,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.159432,"xp_bonus":0.289259,"xp_yellow_cards":-0.138253,"xp_red_cards":-0.004384,"xp_own_goals":-0.003487,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.004088,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":384,"web_name":"Donnarumma","team_short_name":"MCI","position":"GK","price":5.5,"gameweek":5,"expected_minutes":83.047462,"appearance_probability":0.930172,"projected_goals":0.0,"projected_assists":0.0,"expected_points":3.499277,"xp_appearance":1.844798,"xp_goals":0.0,"xp_assists":0.0,"xp_clean_sheet":1.361782,"xp_saves":0.367659,"xp_goals_conceded":-0.247612,"xp_defensive_contribution":0.0,"xp_bonus":0.251533,"xp_yellow_cards":-0.138575,"xp_red_cards":-0.004662,"xp_own_goals":-0.011535,"xp_penalty_saves":0.07589,"xp_penalty_misses":0.0,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":4,"web_name":"Gabriel","team_short_name":"ARS","position":"DEF","price":8.0,"gameweek":5,"expected_minutes":83.086982,"appearance_probability":0.951242,"projected_goals":0.049177,"projected_assists":0.049427,"expected_points":3.451757,"xp_appearance":1.847919,"xp_goals":0.295062,"xp_assists":0.148281,"xp_clean_sheet":1.029149,"xp_saves":0.0,"xp_goals_conceded":-0.354637,"xp_defensive_contribution":0.401666,"xp_bonus":0.256711,"xp_yellow_cards":-0.150232,"xp_red_cards":-0.004667,"xp_own_goals":-0.017495,"xp_penalty_saves":0.0,"xp_penalty_misses":0.0,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":391,"web_name":"Gvardiol","team_short_name":"MCI","position":"DEF","price":5.7,"gameweek":5,"expected_minutes":75.279253,"appearance_probability":0.909958,"projected_goals":0.050587,"projected_assists":0.055982,"expected_points":3.449641,"xp_appearance":1.731558,"xp_goals":0.303524,"xp_assists":0.167946,"xp_clean_sheet":1.268521,"xp_saves":0.0,"xp_goals_conceded":-0.212304,"xp_defensive_contribution":0.077254,"xp_bonus":0.266345,"xp_yellow_cards":-0.133124,"xp_red_cards":-0.004228,"xp_own_goals":-0.015851,"xp_penalty_saves":0.0,"xp_penalty_misses":0.0,"external_prior_available":1.0,"external_prior_reliability":0.762222,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":154,"web_name":"Palmer","team_short_name":"CHE","position":"MID","price":9.7,"gameweek":5,"expected_minutes":76.214588,"appearance_probability":0.94009,"projected_goals":0.184345,"projected_assists":0.121765,"expected_points":3.440629,"xp_appearance":1.772936,"xp_goals":0.921727,"xp_assists":0.365295,"xp_clean_sheet":0.191011,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.039724,"xp_bonus":0.29651,"xp_yellow_cards":-0.134897,"xp_red_cards":-0.00428,"xp_own_goals":-0.003404,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.003991,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0},{"player_id":13,"web_name":"Rice","team_short_name":"ARS","position":"MID","price":7.4,"gameweek":5,"expected_minutes":77.030813,"appearance_probability":0.946135,"projected_goals":0.130928,"projected_assists":0.127548,"expected_points":3.423623,"xp_appearance":1.797479,"xp_goals":0.654642,"xp_assists":0.382644,"xp_clean_sheet":0.261953,"xp_saves":0.0,"xp_goals_conceded":0.0,"xp_defensive_contribution":0.218114,"xp_bonus":0.259342,"xp_yellow_cards":-0.138748,"xp_red_cards":-0.004327,"xp_own_goals":-0.003441,"xp_penalty_saves":0.0,"xp_penalty_misses":-0.004034,"external_prior_available":1.0,"external_prior_reliability":1.0,"external_event_prior_weight":0.0,"market_fallback_weight":0.0}],"starting_xi":[{"position":"GK","web_name":"Lammens","team_short_name":"MUN","fixtures":"FUL (A)","expected_minutes":83.1,"appearance_probability":0.933,"expected_points":3.17,"role":null},{"position":"DEF","web_name":"N.Williams","team_short_name":"NFO","fixtures":"COV (H)","expected_minutes":80.8,"appearance_probability":0.945,"expected_points":4.18,"role":"VC"},{"position":"DEF","web_name":"Tarkowski","team_short_name":"EVE","fixtures":"IPS (H)","expected_minutes":83.9,"appearance_probability":0.954,"expected_points":4.02,"role":null},{"position":"DEF","web_name":"Virgil","team_short_name":"LIV","fixtures":"BOU (A)","expected_minutes":83.5,"appearance_probability":0.954,"expected_points":3.4,"role":null},{"position":"MID","web_name":"B.Fernandes","team_short_name":"MUN","fixtures":"FUL (A)","expected_minutes":82.3,"appearance_probability":0.959,"expected_points":4.26,"role":"C"},{"position":"MID","web_name":"Szoboszlai","team_short_name":"LIV","fixtures":"BOU (A)","expected_minutes":80.7,"appearance_probability":0.957,"expected_points":3.7,"role":null},{"position":"MID","web_name":"Tavernier","team_short_name":"BOU","fixtures":"LIV (H)","expected_minutes":76.7,"appearance_probability":0.935,"expected_points":3.55,"role":null},{"position":"MID","web_name":"Palmer","team_short_name":"CHE","fixtures":"BRE (A)","expected_minutes":76.2,"appearance_probability":0.94,"expected_points":3.44,"role":null},{"position":"MID","web_name":"Enzo","team_short_name":"MCI","fixtures":"SUN (H)","expected_minutes":70.1,"appearance_probability":0.903,"expected_points":3.37,"role":null},{"position":"FWD","web_name":"Calvert-Lewin","team_short_name":"LEE","fixtures":"CRY (H)","expected_minutes":71.9,"appearance_probability":0.929,"expected_points":3.64,"role":null},{"position":"FWD","web_name":"Havertz","team_short_name":"ARS","fixtures":"BHA (A)","expected_minutes":74.6,"appearance_probability":0.943,"expected_points":3.5,"role":null}],"bench":[{"bench_slot":"1","position":"DEF","web_name":"Botman","team_short_name":"NEW","fixtures":"HUL (H)","expected_minutes":78.7,"appearance_probability":0.926,"expected_points":3.26},{"bench_slot":"2","position":"FWD","web_name":"João Pedro","team_short_name":"CHE","fixtures":"BRE (A)","expected_minutes":54.9,"appearance_probability":0.686,"expected_points":2.34},{"bench_slot":"3","position":"DEF","web_name":"Dunk","team_short_name":"BHA","fixtures":"ARS (H)","expected_minutes":79.5,"appearance_probability":0.929,"expected_points":2.67},{"bench_slot":"GK","position":"GK","web_name":"Verbruggen","team_short_name":"BHA","fixtures":"ARS (H)","expected_minutes":82.5,"appearance_probability":0.923,"expected_points":2.65}],"captain_alternatives":[{"web_name":"B.Fernandes","expected_points":4.264,"expected_minutes":82.292,"appearance_probability":0.959,"xp_gap_to_top":0.0,"within_captain_tie_band":true,"selected":true},{"web_name":"N.Williams","expected_points":4.185,"expected_minutes":80.828,"appearance_probability":0.945,"xp_gap_to_top":0.079,"within_captain_tie_band":false,"selected":false},{"web_name":"Tarkowski","expected_points":4.018,"expected_minutes":83.909,"appearance_probability":0.954,"xp_gap_to_top":0.246,"within_captain_tie_band":false,"selected":false},{"web_name":"Szoboszlai","expected_points":3.7,"expected_minutes":80.697,"appearance_probability":0.957,"xp_gap_to_top":0.564,"within_captain_tie_band":false,"selected":false},{"web_name":"Calvert-Lewin","expected_points":3.642,"expected_minutes":71.912,"appearance_probability":0.929,"xp_gap_to_top":0.622,"within_captain_tie_band":false,"selected":false},{"web_name":"Tavernier","expected_points":3.547,"expected_minutes":76.673,"appearance_probability":0.935,"xp_gap_to_top":0.717,"within_captain_tie_band":false,"selected":false},{"web_name":"Havertz","expected_points":3.505,"expected_minutes":74.647,"appearance_probability":0.943,"xp_gap_to_top":0.759,"within_captain_tie_band":false,"selected":false},{"web_name":"Palmer","expected_points":3.441,"expected_minutes":76.215,"appearance_probability":0.94,"xp_gap_to_top":0.823,"within_captain_tie_band":false,"selected":false}],"transfer_evidence":[{"outgoing":"Milenković","incoming":"N.Williams","position":"DEF","price":5.0,"expected_points":4.185,"xp_defensive_contribution":0.3,"dc_share_of_xp":0.072,"prior_pl_fixtures":156.0,"prior_season_fixtures":4.0,"external_prior_available":true,"external_event_prior_weight":0.0,"external_prior_match_method":"exact_name","external_source_leagues":"ENG-Premier League","external_prior_source_note":null,"market_fallback_weight":0.0,"selected_transfer_count":2,"best_lower_transfer_count":1,"incremental_objective_gain_vs_lower_count":1.045,"cheaper_alternative":null,"cheaper_alternative_price":null,"cheaper_alternative_xp":null,"cheaper_alternative_xp_gap":null,"cheaper_alternative_saving":null,"action_extra_transfer_fragile":true,"evidence_level":"HIGH","evidence_reasons":"No low-evidence flags."},{"outgoing":"Thiaw","incoming":"Tarkowski","position":"DEF","price":6.1,"expected_points":4.018,"xp_defensive_contribution":0.608,"dc_share_of_xp":0.151,"prior_pl_fixtures":156.0,"prior_season_fixtures":4.0,"external_prior_available":true,"external_event_prior_weight":0.0,"external_prior_match_method":"exact_name","external_source_leagues":"ENG-Premier League","external_prior_source_note":null,"market_fallback_weight":0.0,"selected_transfer_count":2,"best_lower_transfer_count":1,"incremental_objective_gain_vs_lower_count":1.045,"cheaper_alternative":"Murillo","cheaper_alternative_price":5.5,"cheaper_alternative_xp":3.977,"cheaper_alternative_xp_gap":0.042,"cheaper_alternative_saving":0.6,"action_extra_transfer_fragile":true,"evidence_level":"LOW","evidence_reasons":"15% of xP comes from defensive contributions o..."}],"transfer_alternatives":[{"current_action":"Milenković → N.Williams; Thiaw → Tarkowski","transfer_count":2,"current_hit_cost":0.0,"bank_after":0.4,"path_objective":61.499,"structural_deadwood_after":0.0,"raw_best":true,"selected":true,"exact_subset_comparator":false,"best_for_transfer_count":true},{"current_action":"Milenković → N.Williams; Dunk → Murillo","transfer_count":2,"current_hit_cost":0.0,"bank_after":0.5,"path_objective":61.492,"structural_deadwood_after":0.0,"raw_best":false,"selected":false,"exact_subset_comparator":false,"best_for_transfer_count":false},{"current_action":"Milenković → N.Williams; Thiaw → Murillo","transfer_count":2,"current_hit_cost":0.0,"bank_after":1.0,"path_objective":61.444,"structural_deadwood_after":0.0,"raw_best":false,"selected":false,"exact_subset_comparator":false,"best_for_transfer_count":false},{"current_action":"Milenković → N.Williams; Verbruggen → Sels","transfer_count":2,"current_hit_cost":0.0,"bank_after":1.0,"path_objective":61.434,"structural_deadwood_after":0.0,"raw_best":false,"selected":false,"exact_subset_comparator":false,"best_for_transfer_count":false},{"current_action":"Milenković → N.Williams; Dunk → Muñoz","transfer_count":2,"current_hit_cost":0.0,"bank_after":0.6,"path_objective":61.233,"structural_deadwood_after":0.0,"raw_best":false,"selected":false,"exact_subset_comparator":false,"best_for_transfer_count":false},{"current_action":"Milenković → N.Williams; Dunk → Muharemović","transfer_count":2,"current_hit_cost":0.0,"bank_after":1.0,"path_objective":61.188,"structural_deadwood_after":0.0,"raw_best":false,"selected":false,"exact_subset_comparator":false,"best_for_transfer_count":false},{"current_action":"Milenković → N.Williams; Dunk → Guéhi","transfer_count":2,"current_hit_cost":0.0,"bank_after":0.0,"path_objective":61.166,"structural_deadwood_after":0.0,"raw_best":false,"selected":false,"exact_subset_comparator":false,"best_for_transfer_count":false},{"current_action":"Milenković → Murillo; Verbruggen → Sels","transfer_count":2,"current_hit_cost":0.0,"bank_after":0.5,"path_objective":61.154,"structural_deadwood_after":0.0,"raw_best":false,"selected":false,"exact_subset_comparator":false,"best_for_transfer_count":false},{"current_action":"Milenković → N.Williams; Verbruggen → Donnarumma","transfer_count":2,"current_hit_cost":0.0,"bank_after":0.5,"path_objective":60.954,"structural_deadwood_after":0.0,"raw_best":false,"selected":false,"exact_subset_comparator":false,"best_for_transfer_count":false},{"current_action":"Milenković → Murillo; Dunk → Muñoz","transfer_count":2,"current_hit_cost":0.0,"bank_after":0.1,"path_objective":60.952,"structural_deadwood_after":0.0,"raw_best":false,"selected":false,"exact_subset_comparator":false,"best_for_transfer_count":false},{"current_action":"Milenković → N.Williams","transfer_count":1,"current_hit_cost":0.0,"bank_after":1.5,"path_objective":60.454,"structural_deadwood_after":0.0,"raw_best":false,"selected":false,"exact_subset_comparator":false,"best_for_transfer_count":true},{"current_action":"Roll","transfer_count":0,"current_hit_cost":0.0,"bank_after":1.0,"path_objective":58.644,"structural_deadwood_after":5.5,"raw_best":false,"selected":false,"exact_subset_comparator":false,"best_for_transfer_count":true}]}}""")


def _diag_jsonable(value):
    """Convert pandas/numpy objects to ordinary JSON-safe Python values."""
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        if not np.isfinite(value):
            return None
        return float(value)
    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()
    if pd.isna(value) if not isinstance(value, (dict, list, tuple, set)) else False:
        return None
    return value


def _diag_records(frame: pd.DataFrame) -> list[dict]:
    if not isinstance(frame, pd.DataFrame) or frame.empty:
        return []
    clean = frame.replace({np.nan: None, np.inf: None, -np.inf: None})
    return [
        {str(k): _diag_jsonable(v) for k, v in row.items()}
        for row in clean.to_dict("records")
    ]


def _diag_load_archive(path: Path = ONLINE_DIAGNOSTIC_ARCHIVE_PATH) -> dict:
    if path.exists():
        with gzip.open(path, "rt", encoding="utf-8") as file:
            archive = json.load(file)
    else:
        archive = {"format_version": 1, "snapshots": {}}
    archive.setdefault("format_version", 1)
    archive.setdefault("snapshots", {})
    return archive


def _diag_save_archive(
    archive: dict,
    path: Path = ONLINE_DIAGNOSTIC_ARCHIVE_PATH,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(path, "wt", encoding="utf-8") as file:
        json.dump(archive, file, ensure_ascii=False, default=str)


def _diag_snapshot_key(gameweek: int) -> str:
    return f"{CURRENT_SEASON}_GW{int(gameweek):02d}"


def _diag_seed_recovered_snapshots(archive: dict) -> int:
    if not ONLINE_DIAGNOSTIC_SEED_RECOVERED_GWS:
        return 0

    inserted = 0
    for gw_text, payload in _V23_RECOVERED_DEADLINE_SNAPSHOTS.items():
        gw = int(gw_text)
        key = _diag_snapshot_key(gw)
        bucket = archive["snapshots"].setdefault(
            key,
            {
                "season": CURRENT_SEASON,
                "gameweek": gw,
                "versions": [],
            },
        )
        versions = bucket.setdefault("versions", [])
        if versions:
            continue

        version = dict(payload)
        version["captured_at_utc"] = None
        version["deadline_time_utc"] = None
        version["diagnostic_release"] = DIAGNOSTIC_RELEASE
        version["forecast_signature"] = FORECAST_CACHE_SIGNATURE
        versions.append(version)
        inserted += 1

    return inserted


def _diag_aggregate_current_forecasts(gameweek: int) -> pd.DataFrame:
    """Aggregate fixture-level xP into one player row for one FPL event."""
    source = weekly_adjusted_gameweek_xp.loc[
        weekly_adjusted_gameweek_xp["gameweek"].eq(int(gameweek))
    ].copy()

    if source.empty:
        return source

    sum_columns = [
        column
        for column in [
            "expected_points",
            "expected_minutes",
            "projected_goals",
            "projected_assists",
            "xp_appearance",
            "xp_goals",
            "xp_assists",
            "xp_clean_sheet",
            "xp_saves",
            "xp_goals_conceded",
            "xp_defensive_contribution",
            "xp_bonus",
            "xp_yellow_cards",
            "xp_red_cards",
            "xp_own_goals",
            "xp_penalty_saves",
            "xp_penalty_misses",
        ]
        if column in source.columns
    ]

    first_columns = [
        column
        for column in [
            "player_key",
            "web_name",
            "team_id",
            "team_short_name",
            "position",
            "price",
            "external_prior_available",
            "external_prior_reliability",
            "external_event_prior_weight",
            "market_fallback_weight",
        ]
        if column in source.columns
    ]

    aggregations = {column: "sum" for column in sum_columns}
    aggregations.update({column: "first" for column in first_columns})

    result = (
        source.groupby("player_id", as_index=False)
        .agg(aggregations)
    )
    result["gameweek"] = int(gameweek)
    return result


def save_observational_forecast_snapshot() -> dict:
    """Freeze the forecast that existed before the deadline.

    This performs no optimisation or simulation. It serialises objects that
    Section 12 has already produced.
    """
    archive = _diag_load_archive()
    seeded = _diag_seed_recovered_snapshots(archive)

    if not ONLINE_DIAGNOSTICS_ENABLED:
        if seeded:
            _diag_save_archive(archive)
        return {"saved": False, "reason": "diagnostics disabled", "seeded": seeded}

    required = [
        "weekly_next_gameweek",
        "weekly_adjusted_gameweek_xp",
        "weekly_recommendation_summary",
        "weekly_starting_xi",
        "weekly_bench",
    ]
    missing = [name for name in required if name not in globals()]
    if missing:
        if seeded:
            _diag_save_archive(archive)
        return {
            "saved": False,
            "reason": f"Section 12 objects missing: {missing}",
            "seeded": seeded,
        }

    gw = int(weekly_next_gameweek)
    event_row = gameweeks.loc[gameweeks["id"].eq(gw)]
    if len(event_row) != 1:
        if seeded:
            _diag_save_archive(archive)
        return {
            "saved": False,
            "reason": f"Could not identify GW{gw} deadline uniquely",
            "seeded": seeded,
        }

    deadline = pd.to_datetime(
        event_row.iloc[0]["deadline_time"],
        utc=True,
        errors="coerce",
    )
    captured = datetime.now(timezone.utc)

    if pd.isna(deadline) or captured >= deadline.to_pydatetime():
        if seeded:
            _diag_save_archive(archive)
        return {
            "saved": False,
            "reason": "deadline already passed or unavailable",
            "seeded": seeded,
        }

    forecasts = _diag_aggregate_current_forecasts(gw)

    version = {
        "source": "v2.3_full_predeadline_archive",
        "partial_player_pool": False,
        "player_pool_scope": "all_live_forecasts",
        "season": CURRENT_SEASON,
        "gameweek": gw,
        "captured_at_utc": captured.isoformat(),
        "deadline_time_utc": deadline.isoformat(),
        "diagnostic_release": DIAGNOSTIC_RELEASE,
        "forecast_signature": FORECAST_CACHE_SIGNATURE,
        "live_feature_signature": LIVE_FEATURE_SIGNATURE,
        "summary": {
            str(key): _diag_jsonable(value)
            for key, value in weekly_recommendation_summary.items()
        },
        "player_forecasts": _diag_records(forecasts),
        "starting_xi": _diag_records(weekly_starting_xi),
        "bench": _diag_records(weekly_bench),
        "captain_alternatives": _diag_records(
            weekly_captain_alternatives
            if isinstance(weekly_captain_alternatives, pd.DataFrame)
            else pd.DataFrame()
        ),
        "transfer_evidence": _diag_records(
            weekly_transfer_evidence
            if isinstance(weekly_transfer_evidence, pd.DataFrame)
            else pd.DataFrame()
        ),
        "transfer_alternatives": _diag_records(
            weekly_transfer_alternatives
            if isinstance(weekly_transfer_alternatives, pd.DataFrame)
            else pd.DataFrame()
        ),
    }

    # Hash only the substantive forecast/decision payload so repeated Run All
    # calls do not create duplicate versions.
    signature_payload = {
        "player_forecasts": version["player_forecasts"],
        "summary": version["summary"],
        "starting_xi": version["starting_xi"],
        "bench": version["bench"],
    }
    version_hash = hashlib.sha256(
        json.dumps(
            signature_payload,
            sort_keys=True,
            default=str,
        ).encode("utf-8")
    ).hexdigest()[:20]
    version["snapshot_hash"] = version_hash

    key = _diag_snapshot_key(gw)
    bucket = archive["snapshots"].setdefault(
        key,
        {"season": CURRENT_SEASON, "gameweek": gw, "versions": []},
    )
    versions = bucket.setdefault("versions", [])

    if any(row.get("snapshot_hash") == version_hash for row in versions):
        _diag_save_archive(archive)
        return {
            "saved": False,
            "reason": "identical pre-deadline forecast already archived",
            "seeded": seeded,
            "gameweek": gw,
        }

    versions.append(version)
    versions.sort(key=lambda row: str(row.get("captured_at_utc") or ""))
    _diag_save_archive(archive)

    return {
        "saved": True,
        "seeded": seeded,
        "gameweek": gw,
        "players": len(forecasts),
        "path": str(ONLINE_DIAGNOSTIC_ARCHIVE_PATH),
    }


v23_snapshot_save_audit = save_observational_forecast_snapshot()
print("v2.3 pre-deadline forecast archive:", v23_snapshot_save_audit)


In [ ]:

# ============================================================================
# v2.3 OBSERVATIONAL ONLINE VALIDATION
# ============================================================================

def _diag_latest_version(bucket: dict) -> dict | None:
    versions = list(bucket.get("versions", []))
    if not versions:
        return None
    # Recovered legacy snapshots have no timestamp. Full v2.3 versions have
    # timestamps and should supersede legacy data for the same event.
    versions.sort(
        key=lambda row: (
            row.get("captured_at_utc") is not None,
            str(row.get("captured_at_utc") or ""),
        )
    )
    return versions[-1]


def _diag_event_actuals(gameweek: int) -> pd.DataFrame:
    payload = get_fpl_data(f"event/{int(gameweek)}/live/")
    rows = []
    for element in payload.get("elements", []):
        stats = dict(element.get("stats", {}))
        rows.append({
            "player_id": int(element["id"]),
            "actual_points": float(stats.get("total_points", 0) or 0),
            "actual_minutes": float(stats.get("minutes", 0) or 0),
            "actual_goals": float(stats.get("goals_scored", 0) or 0),
            "actual_assists": float(stats.get("assists", 0) or 0),
            "actual_bonus": float(stats.get("bonus", 0) or 0),
            "actual_clean_sheets": float(stats.get("clean_sheets", 0) or 0),
            "actual_saves": float(stats.get("saves", 0) or 0),
            "actual_goals_conceded": float(stats.get("goals_conceded", 0) or 0),
        })
    return pd.DataFrame(rows)


def _diag_manager_history() -> pd.DataFrame:
    if FPL_TEAM_ID is None:
        return pd.DataFrame()
    payload = get_fpl_data(f"entry/{int(FPL_TEAM_ID)}/history/")
    rows = []
    for row in payload.get("current", []):
        rows.append({
            "gameweek": int(row.get("event")),
            "manager_points": float(row.get("points", 0) or 0),
            "manager_total_points": float(row.get("total_points", 0) or 0),
            "overall_rank": row.get("overall_rank"),
            "gameweek_rank": row.get("rank"),
            "transfers": int(row.get("event_transfers", 0) or 0),
            "transfer_cost": float(row.get("event_transfers_cost", 0) or 0),
            "points_on_bench": float(row.get("points_on_bench", 0) or 0),
        })
    return pd.DataFrame(rows)


def _diag_manager_transfers() -> pd.DataFrame:
    if FPL_TEAM_ID is None:
        return pd.DataFrame()

    payload = get_fpl_data(f"entry/{int(FPL_TEAM_ID)}/transfers/")
    element_lookup = {
        int(row["id"]): str(row.get("web_name", row["id"]))
        for row in bootstrap.get("elements", [])
    } if isinstance(bootstrap, dict) else {}

    rows = []
    for row in payload:
        event = row.get("event")
        player_in = row.get("element_in")
        player_out = row.get("element_out")
        if event is None or player_in is None or player_out is None:
            continue
        rows.append({
            "gameweek": int(event),
            "outgoing": element_lookup.get(int(player_out), str(player_out)),
            "incoming": element_lookup.get(int(player_in), str(player_in)),
        })
    return pd.DataFrame(rows)


def _diag_event_averages() -> pd.DataFrame:
    event_rows = bootstrap.get("events", []) if isinstance(bootstrap, dict) else []
    if not event_rows:
        return pd.DataFrame()
    return pd.DataFrame([
        {
            "gameweek": int(row["id"]),
            "league_average": (
                float(row["average_entry_score"])
                if row.get("average_entry_score") is not None
                else np.nan
            ),
            "finished": bool(row.get("finished", False)),
        }
        for row in event_rows
    ])


def _diag_predeadline_baseline(gameweek: int) -> pd.DataFrame:
    """Recover simple FPL baselines from the existing raw pre-deadline archive."""
    if not LIVE_SNAPSHOT_PATH.exists():
        return pd.DataFrame()

    try:
        with gzip.open(LIVE_SNAPSHOT_PATH, "rt", encoding="utf-8") as file:
            raw_archive = json.load(file)
    except Exception:
        return pd.DataFrame()

    candidates = []
    for snapshot in raw_archive.get("snapshots", {}).values():
        try:
            if int(snapshot.get("gameweek")) == int(gameweek):
                candidates.append(snapshot)
        except Exception:
            continue

    if not candidates:
        return pd.DataFrame()

    # save_predeadline_snapshot refreshes a GW key, so normally only one row
    # exists. If there are several, take the latest capture.
    candidates.sort(key=lambda row: str(row.get("captured_at_utc") or ""))
    snapshot = candidates[-1]
    elements = snapshot.get("bootstrap", {}).get("elements", [])

    rows = []
    for row in elements:
        def _num(name):
            value = row.get(name)
            try:
                return float(value)
            except (TypeError, ValueError):
                return np.nan

        rows.append({
            "player_id": int(row["id"]),
            "fpl_ep_next": _num("ep_next"),
            "fpl_form": _num("form"),
            "fpl_points_per_game": _num("points_per_game"),
        })
    return pd.DataFrame(rows)


def _diag_spearman(frame: pd.DataFrame, predicted: str, actual: str) -> float:
    valid = frame[[predicted, actual]].dropna()
    if (
        len(valid) < int(ONLINE_DIAGNOSTIC_MIN_ROWS_FOR_RANKING)
        or valid[predicted].nunique() < 2
        or valid[actual].nunique() < 2
    ):
        return np.nan
    return float(valid[predicted].corr(valid[actual], method="spearman"))


def _diag_error_metrics(
    frame: pd.DataFrame,
    predicted: str,
    actual: str = "actual_points",
) -> dict:
    valid = frame[[predicted, actual]].dropna()
    if valid.empty:
        return {
            "rows": 0,
            "mae": np.nan,
            "rmse": np.nan,
            "bias_pred_minus_actual": np.nan,
            "spearman": np.nan,
        }

    error = valid[predicted] - valid[actual]
    return {
        "rows": int(len(valid)),
        "mae": float(error.abs().mean()),
        "rmse": float(np.sqrt(np.square(error).mean())),
        "bias_pred_minus_actual": float(error.mean()),
        "spearman": _diag_spearman(valid, predicted, actual),
    }


def _diag_top_k_quality(
    frame: pd.DataFrame,
    predicted: str,
    k: int,
) -> dict:
    valid = frame[[predicted, "actual_points"]].dropna()
    if valid.empty:
        return {
            "actual_sum_selected": np.nan,
            "actual_sum_oracle": np.nan,
            "oracle_efficiency": np.nan,
        }

    k = min(int(k), len(valid))
    selected = valid.nlargest(k, predicted)
    oracle = valid.nlargest(k, "actual_points")
    selected_sum = float(selected["actual_points"].sum())
    oracle_sum = float(oracle["actual_points"].sum())

    return {
        "actual_sum_selected": selected_sum,
        "actual_sum_oracle": oracle_sum,
        "oracle_efficiency": (
            selected_sum / oracle_sum
            if oracle_sum > 0
            else np.nan
        ),
    }


def _diag_captain_result(snapshot: dict, forecast_actual: pd.DataFrame) -> dict:
    rows = pd.DataFrame(snapshot.get("captain_alternatives", []))
    if rows.empty or "web_name" not in rows.columns:
        return {}

    name_actual = (
        forecast_actual[
            ["web_name", "actual_points"]
        ]
        .dropna(subset=["web_name"])
        .drop_duplicates("web_name")
    )
    rows = rows.merge(name_actual, on="web_name", how="left")
    selected_mask = (
        rows["selected"].fillna(False).astype(bool)
        if "selected" in rows.columns
        else pd.Series(False, index=rows.index)
    )
    selected = rows.loc[selected_mask]

    if selected.empty:
        selected_name = snapshot.get("summary", {}).get("captain")
        selected = rows.loc[rows["web_name"].eq(selected_name)]

    if selected.empty or rows["actual_points"].dropna().empty:
        return {}

    captain_row = selected.iloc[0]
    best_actual = float(rows["actual_points"].max())
    captain_actual = float(captain_row["actual_points"])

    return {
        "captain": captain_row["web_name"],
        "captain_actual_points": captain_actual,
        "best_candidate_actual_points": best_actual,
        "captain_regret_raw_points": best_actual - captain_actual,
    }


def run_observational_xp_audit() -> dict:
    """Evaluate frozen forecasts against realised outcomes.

    No historical strategy simulation, transfer optimisation, or season
    counterfactual is invoked here.
    """
    archive = _diag_load_archive()
    _diag_seed_recovered_snapshots(archive)
    _diag_save_archive(archive)

    manager = _diag_manager_history()
    manager_transfers = _diag_manager_transfers()
    event_averages = _diag_event_averages()

    if manager.empty:
        manager_vs_average = pd.DataFrame()
    elif event_averages.empty:
        manager_vs_average = manager.copy()
        manager_vs_average["league_average"] = np.nan
        manager_vs_average["finished"] = False
    else:
        manager_vs_average = manager.merge(
            event_averages,
            on="gameweek",
            how="left",
        )

    if not manager_vs_average.empty:
        manager_vs_average["manager_minus_average"] = (
            manager_vs_average["manager_points"]
            - manager_vs_average["league_average"]
        )
        manager_vs_average["beat_league_average"] = (
            manager_vs_average["manager_minus_average"] > 0
        )

    per_gw_rows = []
    position_rows = []
    captain_rows = []
    merged_forecasts = []

    finished_lookup = (
        event_averages.set_index("gameweek")["finished"].to_dict()
        if not event_averages.empty
        else {}
    )

    for bucket in archive.get("snapshots", {}).values():
        gw = int(bucket.get("gameweek"))
        if finished_lookup and not bool(finished_lookup.get(gw, False)):
            continue

        snapshot = _diag_latest_version(bucket)
        if not snapshot:
            continue

        forecasts = pd.DataFrame(snapshot.get("player_forecasts", []))
        if forecasts.empty or "player_id" not in forecasts.columns:
            continue

        actual = _diag_event_actuals(gw)
        if actual.empty:
            continue

        forecasts["player_id"] = pd.to_numeric(
            forecasts["player_id"], errors="coerce"
        ).astype("Int64")
        actual["player_id"] = actual["player_id"].astype("Int64")

        merged = forecasts.merge(actual, on="player_id", how="left")
        baseline = _diag_predeadline_baseline(gw)
        if not baseline.empty:
            baseline["player_id"] = baseline["player_id"].astype("Int64")
            merged = merged.merge(baseline, on="player_id", how="left")

        merged["gameweek"] = gw
        merged["forecast_source"] = snapshot.get("source")
        merged["partial_player_pool"] = bool(
            snapshot.get("partial_player_pool", False)
        )
        merged["prediction_error"] = (
            pd.to_numeric(merged["expected_points"], errors="coerce")
            - pd.to_numeric(merged["actual_points"], errors="coerce")
        )

        merged_forecasts.append(merged)

        model = _diag_error_metrics(merged, "expected_points")
        fpl = (
            _diag_error_metrics(merged, "fpl_ep_next")
            if "fpl_ep_next" in merged.columns
            else {}
        )
        form = (
            _diag_error_metrics(merged, "fpl_form")
            if "fpl_form" in merged.columns
            else {}
        )
        ppg = (
            _diag_error_metrics(merged, "fpl_points_per_game")
            if "fpl_points_per_game" in merged.columns
            else {}
        )

        minutes_valid = merged[
            ["expected_minutes", "actual_minutes"]
        ].dropna() if {
            "expected_minutes", "actual_minutes"
        }.issubset(merged.columns) else pd.DataFrame()

        top_model = _diag_top_k_quality(
            merged,
            "expected_points",
            ONLINE_DIAGNOSTIC_TOP_K,
        )
        top_fpl = (
            _diag_top_k_quality(
                merged,
                "fpl_ep_next",
                ONLINE_DIAGNOSTIC_TOP_K,
            )
            if "fpl_ep_next" in merged.columns
            else {}
        )

        premium = merged.loc[
            pd.to_numeric(merged.get("price"), errors="coerce")
            >= float(ONLINE_DIAGNOSTIC_PREMIUM_PRICE)
        ]
        premium_model = _diag_error_metrics(
            premium,
            "expected_points",
        )

        predicted = pd.to_numeric(
            merged["expected_points"],
            errors="coerce",
        )
        cutoff = predicted.quantile(0.75) if predicted.notna().any() else np.nan
        top_quartile = (
            merged.loc[predicted.ge(cutoff)]
            if np.isfinite(cutoff)
            else pd.DataFrame()
        )
        top_quartile_metrics = _diag_error_metrics(
            top_quartile,
            "expected_points",
        )

        manager_row = (
            manager_vs_average.loc[
                manager_vs_average["gameweek"].eq(gw)
            ].iloc[0]
            if not manager_vs_average.empty
            and manager_vs_average["gameweek"].eq(gw).any()
            else None
        )

        model_transfer_rows = pd.DataFrame(
            snapshot.get("transfer_evidence", [])
        )
        model_pairs = set()
        if {
            "outgoing", "incoming"
        }.issubset(model_transfer_rows.columns):
            model_pairs = {
                (
                    str(row.outgoing).strip().casefold(),
                    str(row.incoming).strip().casefold(),
                )
                for row in model_transfer_rows[
                    ["outgoing", "incoming"]
                ].dropna().itertuples(index=False)
            }

        actual_transfer_rows = (
            manager_transfers.loc[
                manager_transfers["gameweek"].eq(gw)
            ]
            if not manager_transfers.empty
            else pd.DataFrame()
        )
        actual_pairs = set()
        if not actual_transfer_rows.empty:
            actual_pairs = {
                (
                    str(row.outgoing).strip().casefold(),
                    str(row.incoming).strip().casefold(),
                )
                for row in actual_transfer_rows[
                    ["outgoing", "incoming"]
                ].dropna().itertuples(index=False)
            }

        recommendation_followed_transfers = (
            bool(model_pairs == actual_pairs)
            if model_pairs or actual_pairs
            else None
        )

        summary = snapshot.get("summary", {})
        per_gw_rows.append({
            "gameweek": gw,
            "forecast_scope": snapshot.get("player_pool_scope"),
            "forecast_rows": model.get("rows", 0),
            "model_mae": model.get("mae"),
            "model_rmse": model.get("rmse"),
            "model_bias_pred_minus_actual": model.get(
                "bias_pred_minus_actual"
            ),
            "model_spearman": model.get("spearman"),
            "minutes_mae": (
                float(
                    (
                        pd.to_numeric(
                            minutes_valid["expected_minutes"],
                            errors="coerce",
                        )
                        - pd.to_numeric(
                            minutes_valid["actual_minutes"],
                            errors="coerce",
                        )
                    ).abs().mean()
                )
                if not minutes_valid.empty
                else np.nan
            ),
            "fpl_ep_next_mae": fpl.get("mae"),
            "fpl_ep_next_spearman": fpl.get("spearman"),
            "form_mae": form.get("mae"),
            "form_spearman": form.get("spearman"),
            "ppg_mae": ppg.get("mae"),
            "ppg_spearman": ppg.get("spearman"),
            "model_topk_actual_sum": top_model.get("actual_sum_selected"),
            "fpl_topk_actual_sum": top_fpl.get("actual_sum_selected"),
            "oracle_topk_actual_sum_within_scope": top_model.get(
                "actual_sum_oracle"
            ),
            "model_topk_oracle_efficiency": top_model.get(
                "oracle_efficiency"
            ),
            "premium_rows": premium_model.get("rows", 0),
            "premium_mae": premium_model.get("mae"),
            "premium_bias_pred_minus_actual": premium_model.get(
                "bias_pred_minus_actual"
            ),
            "top_quartile_mae": top_quartile_metrics.get("mae"),
            "top_quartile_bias_pred_minus_actual": (
                top_quartile_metrics.get("bias_pred_minus_actual")
            ),
            "model_expected_team_score": pd.to_numeric(
                summary.get("expected_current_score"),
                errors="coerce",
            ),
            "recommended_transfer_count": int(len(model_pairs)),
            "actual_transfer_count": int(len(actual_pairs)),
            "recommendation_followed_transfers": (
                recommendation_followed_transfers
            ),
            "manager_actual_points": (
                float(manager_row["manager_points"])
                if manager_row is not None
                else np.nan
            ),
            "league_average": (
                float(manager_row["league_average"])
                if manager_row is not None
                and pd.notna(manager_row["league_average"])
                else np.nan
            ),
            "manager_minus_average": (
                float(manager_row["manager_minus_average"])
                if manager_row is not None
                and pd.notna(manager_row["manager_minus_average"])
                else np.nan
            ),
        })

        if "position" in merged.columns:
            for position, group in merged.groupby("position", dropna=False):
                metrics = _diag_error_metrics(group, "expected_points")
                position_rows.append({
                    "gameweek": gw,
                    "position": position,
                    **metrics,
                })

        captain = _diag_captain_result(snapshot, merged)
        if captain:
            captain_rows.append({"gameweek": gw, **captain})

    forecast_metrics = pd.DataFrame(per_gw_rows).sort_values(
        "gameweek"
    ) if per_gw_rows else pd.DataFrame()
    position_metrics = pd.DataFrame(position_rows)
    captain_metrics = pd.DataFrame(captain_rows)
    player_audit = (
        pd.concat(merged_forecasts, ignore_index=True)
        if merged_forecasts
        else pd.DataFrame()
    )

    # Direct model-vs-simple-baseline comparison; no simulated decisions.
    comparison_rows = []
    if not forecast_metrics.empty:
        comparisons = [
            ("MAE", "model_mae", "fpl_ep_next_mae", "lower"),
            (
                "Spearman",
                "model_spearman",
                "fpl_ep_next_spearman",
                "higher",
            ),
            (
                f"Top-{ONLINE_DIAGNOSTIC_TOP_K} realised sum",
                "model_topk_actual_sum",
                "fpl_topk_actual_sum",
                "higher",
            ),
        ]
        for label, model_col, baseline_col, direction in comparisons:
            paired = forecast_metrics[[model_col, baseline_col]].dropna()
            if paired.empty:
                continue
            model_value = float(paired[model_col].mean())
            baseline_value = float(paired[baseline_col].mean())
            better = (
                model_value < baseline_value
                if direction == "lower"
                else model_value > baseline_value
            )
            comparison_rows.append({
                "metric": label,
                "gameweeks_compared": int(len(paired)),
                "model": model_value,
                "fpl_ep_next": baseline_value,
                "model_better": bool(better),
            })

    baseline_comparison = pd.DataFrame(comparison_rows)

    return {
        "forecast_metrics": forecast_metrics,
        "manager_vs_average": manager_vs_average,
        "position_metrics": position_metrics,
        "captain_metrics": captain_metrics,
        "player_audit": player_audit,
        "baseline_comparison": baseline_comparison,
    }


v23_online_audit = run_observational_xp_audit()

online_forecast_metrics = v23_online_audit["forecast_metrics"]
online_manager_vs_average = v23_online_audit["manager_vs_average"]
online_position_metrics = v23_online_audit["position_metrics"]
online_captain_metrics = v23_online_audit["captain_metrics"]
online_player_audit = v23_online_audit["player_audit"]
online_baseline_comparison = v23_online_audit["baseline_comparison"]

print("=" * 72)
print("v2.3 OBSERVATIONAL xP AUDIT")
print("=" * 72)
print("Historical/policy simulations run by this section: 0")
print(
    "Counterfactual decision simulations:",
    "ON" if ONLINE_DIAGNOSTIC_COUNTERFACTUALS else "OFF",
)
print(
    "Completed GWs with preserved xP forecasts:",
    (
        online_forecast_metrics["gameweek"].astype(int).tolist()
        if not online_forecast_metrics.empty
        else []
    ),
)

if not online_manager_vs_average.empty:
    completed_manager = online_manager_vs_average.loc[
        online_manager_vs_average["finished"].fillna(False)
    ]
    if not completed_manager.empty:
        beaten = int(
            completed_manager["beat_league_average"]
            .fillna(False)
            .sum()
        )
        total = int(len(completed_manager))
        cumulative_delta = float(
            completed_manager["manager_minus_average"]
            .dropna()
            .sum()
        )
        print(
            f"Manager vs overall GW average: {beaten}/{total} GWs above "
            f"average; cumulative delta {cumulative_delta:+.1f} points."
        )

print("\nForecast-level metrics")
print(
    "Note: manager actual points are not treated as the model's realised "
    "team forecast unless the real transfers/lineup matched the recommendation."
)
display(
    online_forecast_metrics.round(3)
    if not online_forecast_metrics.empty
    else pd.DataFrame({"status": ["No completed archived forecasts yet"]})
)

print("\nModel vs FPL ep_next on the exact same archived player rows")
display(
    online_baseline_comparison.round(3)
    if not online_baseline_comparison.empty
    else pd.DataFrame({
        "status": [
            "No paired pre-deadline FPL ep_next baseline available yet"
        ]
    })
)

print("\nManager vs overall FPL Gameweek average")
display(
    online_manager_vs_average.round(3)
    if not online_manager_vs_average.empty
    else pd.DataFrame({"status": ["Manager history unavailable"]})
)

if not online_position_metrics.empty:
    print("\nForecast error by position")
    display(online_position_metrics.round(3))

if not online_captain_metrics.empty:
    print("\nCaptain diagnostic (within the model's displayed candidate set)")
    display(online_captain_metrics.round(3))

if not online_player_audit.empty:
    print("\nLargest absolute xP misses in preserved forecasts")
    largest = online_player_audit.assign(
        absolute_error=lambda frame: frame["prediction_error"].abs()
    ).sort_values(
        ["gameweek", "absolute_error"],
        ascending=[True, False],
    )
    columns = [
        column
        for column in [
            "gameweek",
            "web_name",
            "team_short_name",
            "position",
            "price",
            "expected_points",
            "actual_points",
            "prediction_error",
            "expected_minutes",
            "actual_minutes",
            "xp_goals",
            "xp_assists",
            "xp_clean_sheet",
            "xp_defensive_contribution",
            "xp_bonus",
        ]
        if column in largest.columns
    ]
    display(largest[columns].groupby("gameweek").head(10).round(3))

# Lightweight report file; it contains observed metrics only.
if not online_forecast_metrics.empty:
    ONLINE_DIAGNOSTIC_REPORT_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    online_forecast_metrics.to_csv(
        ONLINE_DIAGNOSTIC_REPORT_PATH,
        index=False,
    )
    print(
        "\nSaved observational diagnostic report:",
        ONLINE_DIAGNOSTIC_REPORT_PATH,
    )


### How to interpret the v2.3 audit

This section intentionally separates **team performance** from **forecast quality**.

- `manager_minus_average < 0` confirms that the FPL team underperformed the overall
  Gameweek average, but it does not by itself prove the xP model is wrong.
- `model_mae`, `model_spearman`, `minutes_mae`, premium-player error and top-quartile
  bias diagnose the forecast itself.
- The `FPL ep_next` comparison is the key sanity benchmark: a complex model should earn
  its complexity by beating a simple pre-deadline public baseline on the *same players*.
- For recovered GW4/GW5 snapshots, the player pool is only the displayed top 30, so
  top-K/oracle metrics are explicitly **within that preserved pool**.
- From the first v2.3 deadline onward, the archive stores the full live player forecast
  table and the comparison becomes stronger.

Do not tune the xP equations after one bad Gameweek. Accumulate several frozen,
pre-deadline observations, identify a repeated failure mode (minutes, premium attackers,
defenders, top-xP ranking, etc.), and then change only that component.

`ONLINE_DIAGNOSTIC_COUNTERFACTUALS=False` is intentional. Turn it on only when testing a
specific question such as "was transfer X worse than rolling for that single deadline?"
It should not be used to launch broad season simulations.


# 14. v2.3.1 decision-quality diagnosis

This section answers the question that aggregate player-level MAE cannot: **does the model rank the players who matter for FPL decisions well enough?**

It deliberately performs **no policy simulation, no transfer optimiser run, and no model refit**. The historical audit reuses the three walk-forward fold bundles that Section 11 already built in this kernel. If those bundles are absent and `HISTORICAL_DECISION_AUDIT_REUSE_FOLDS_ONLY=True`, the audit skips instead of triggering expensive fitting.

The audit covers:

- per-Gameweek top-10 ranking quality, oracle efficiency, NDCG and simple-history-baseline lift;
- premium MID/FWD ranking and captain-like top-player regret;
- error/calibration by position, price and predicted-xP bucket;
- minutes MAE plus appearance/start/60+ probability Brier scores;
- attacking-rate diagnosis by comparing projected goals/assists with realised FPL xG/xA, which removes finishing noise from the diagnosis;
- team-goal model error;
- an archived public FPL xP baseline for 2026/27 GW1–GW5 when available from the public `vaastav/Fantasy-Premier-League` dataset;
- execution provenance, distinguishing the genuine GW2 deviation from later assistant-advised conservative overrides.

The public GW1–GW3 data are used only as a **baseline/context**. We do not possess genuine frozen pre-deadline forecasts from our own model for those Gameweeks, so the notebook does not pretend that we do.


In [ ]:
# =============================================================================
# v2.3.1 HISTORICAL DECISION-QUALITY AUDIT — NO NEW FITTING / NO SIMULATION
# =============================================================================

def _v231_production_fold_frame(holdout_season: str) -> pd.DataFrame:
    """Return the production-semantics OOS fold already built in Section 11."""
    bundle = walk_forward_fold_cache.get(str(holdout_season))
    if bundle is None:
        if HISTORICAL_DECISION_AUDIT_REUSE_FOLDS_ONLY:
            return pd.DataFrame()
        bundle = build_walk_forward_fold_bundle(str(holdout_season))

    frame = bundle["holdout_expected_points"].copy()

    premium_payload = None
    if isinstance(globals().get("premium_attacker_cache"), dict):
        premium_payload = premium_attacker_cache.get(
            "fold_strategy_payloads", {}
        ).get(str(holdout_season))
    frame = apply_premium_attacker_adjustment(
        frame,
        premium_payload,
        position_col="model_position",
        price_col="price_at_collection",
    )

    price_prior = build_position_price_prior_table(
        bundle["development_points_data"]
    )
    frame = apply_sparse_player_market_prior(
        frame,
        prior_table=price_prior,
    )

    calibration_payload = None
    if isinstance(globals().get("xp_calibration_cache"), dict):
        calibration_payload = xp_calibration_cache.get(
            "fold_strategy_payloads", {}
        ).get(str(holdout_season))
    frame = apply_xp_calibration_payload(
        frame,
        calibration_payload,
        position_col="model_position",
        price_col="price_at_collection",
        value_col="expected_points",
    )

    # On a later run, include the production-approved v2.4 correction so the
    # diagnostic tables reflect the live forecast semantics. The dedicated
    # Section 15 scorecards remain the leakage-safe approval evidence.
    frame = apply_v24_targeted_forecast_layer(
        frame,
        v24_targeted_forecast_cache,
        position_col="model_position",
        price_col="price_at_collection",
        value_col="expected_points",
    )

    development = bundle["development_points_data"]
    position_prior = (
        development.groupby("model_position")["target_fpl_points"].mean()
    )
    global_prior = float(development["target_fpl_points"].mean())
    frame["simple_history_baseline"] = (
        coalesced_history_baseline(
            frame,
            [
                "points_per_fixture_last_5_gws",
                "points_per_fixture_last_10_gws",
                "career_points_per_fixture",
            ],
            position_prior,
        )
        .fillna(global_prior)
        .clip(lower=-3.0, upper=20.0)
    )
    frame["holdout_season"] = str(holdout_season)
    return frame


def _v231_player_gw(frame: pd.DataFrame) -> pd.DataFrame:
    """Aggregate fixture-level forecasts/outcomes to one player-Gameweek row."""
    if frame.empty:
        return pd.DataFrame()

    id_col = "player_key" if "player_key" in frame.columns else "player_id_season"
    keys = ["holdout_season", "gameweek", id_col]
    agg = {
        "player_name": ("player_name", "last"),
        "team_short_name": ("team_short_name", "last"),
        "position": ("model_position", "last"),
        "price": ("price_at_collection", "last"),
        "expected_points": ("expected_points", "sum"),
        "baseline_points": ("simple_history_baseline", "sum"),
        "actual_points": ("target_fpl_points", "sum"),
        "expected_minutes": ("expected_minutes", "sum"),
        "actual_minutes": ("target_minutes", "sum"),
    }
    optional_sums = {
        "projected_goals": "projected_goals",
        "projected_assists": "projected_assists",
        "actual_xg": "expected_goals",
        "actual_xa": "expected_assists",
        "actual_goals": "goals_scored",
        "actual_assists": "assists",
        "xp_appearance": "xp_appearance",
        "xp_goals": "xp_goals",
        "xp_assists": "xp_assists",
        "xp_clean_sheet": "xp_clean_sheet",
        "xp_defensive_contribution": "xp_defensive_contribution",
        "xp_bonus": "xp_bonus",
    }
    for output, source in optional_sums.items():
        if source in frame.columns:
            agg[output] = (source, "sum")

    result = frame.groupby(keys, dropna=False).agg(**agg).reset_index()
    for column in [
        "expected_points", "baseline_points", "actual_points",
        "expected_minutes", "actual_minutes", "price",
        "projected_goals", "projected_assists", "actual_xg", "actual_xa",
    ]:
        if column in result.columns:
            result[column] = pd.to_numeric(result[column], errors="coerce")
    return result


def _v231_ndcg(frame: pd.DataFrame, predicted: str, k: int) -> float:
    valid = frame[[predicted, "actual_points"]].dropna()
    if valid.empty:
        return np.nan
    k = min(int(k), len(valid))
    if k <= 0:
        return np.nan
    # Linear, non-negative relevance avoids allowing one haul to dominate the
    # metric exponentially while keeping the ordering objective explicit.
    ordered = valid.sort_values(predicted, ascending=False).head(k)
    oracle = valid.sort_values("actual_points", ascending=False).head(k)
    discount = np.log2(np.arange(2, k + 2))
    dcg = float((ordered["actual_points"].clip(lower=0).to_numpy() / discount).sum())
    idcg = float((oracle["actual_points"].clip(lower=0).to_numpy() / discount).sum())
    return dcg / idcg if idcg > 0 else np.nan


def _v231_gw_rank_row(group: pd.DataFrame, k: int) -> dict:
    pool = group.loc[
        pd.to_numeric(group["expected_minutes"], errors="coerce")
        .ge(float(HISTORICAL_DECISION_MIN_EXPECTED_MINUTES))
    ].copy()
    if len(pool) < k:
        pool = group.loc[
            pd.to_numeric(group["expected_minutes"], errors="coerce").gt(0)
        ].copy()
    if pool.empty:
        return {}

    k = min(int(k), len(pool))
    model_top = pool.nlargest(k, "expected_points")
    baseline_top = pool.nlargest(k, "baseline_points")
    oracle_top = pool.nlargest(k, "actual_points")
    oracle_sum = float(oracle_top["actual_points"].sum())

    model_ids = set(model_top.index)
    oracle_ids = set(oracle_top.index)
    model_sum = float(model_top["actual_points"].sum())
    baseline_sum = float(baseline_top["actual_points"].sum())

    captain_pool = pool.nlargest(
        min(int(HISTORICAL_DECISION_CAPTAIN_POOL), len(pool)),
        "expected_points",
    )
    model_captain = model_top.iloc[0]
    best_candidate_actual = float(captain_pool["actual_points"].max())

    premium = pool.loc[
        pool["position"].isin(["MID", "FWD"])
        & pd.to_numeric(pool["price"], errors="coerce").ge(8.0)
    ]
    premium_k = min(3, len(premium))
    if premium_k:
        premium_model = premium.nlargest(premium_k, "expected_points")
        premium_base = premium.nlargest(premium_k, "baseline_points")
        premium_oracle = premium.nlargest(premium_k, "actual_points")
        premium_model_sum = float(premium_model["actual_points"].sum())
        premium_base_sum = float(premium_base["actual_points"].sum())
        premium_oracle_sum = float(premium_oracle["actual_points"].sum())
    else:
        premium_model_sum = premium_base_sum = premium_oracle_sum = np.nan

    return {
        "players_in_pool": int(len(pool)),
        "model_spearman": _diag_spearman(pool, "expected_points", "actual_points"),
        "baseline_spearman": _diag_spearman(pool, "baseline_points", "actual_points"),
        "model_ndcg10": _v231_ndcg(pool, "expected_points", min(10, k)),
        "baseline_ndcg10": _v231_ndcg(pool, "baseline_points", min(10, k)),
        "model_topk_actual": model_sum,
        "baseline_topk_actual": baseline_sum,
        "oracle_topk_actual": oracle_sum,
        "model_topk_oracle_efficiency": model_sum / oracle_sum if oracle_sum > 0 else np.nan,
        "baseline_topk_oracle_efficiency": baseline_sum / oracle_sum if oracle_sum > 0 else np.nan,
        "model_minus_baseline_topk": model_sum - baseline_sum,
        "topk_oracle_overlap": len(model_ids & oracle_ids) / k,
        "model_captain_actual": float(model_captain["actual_points"]),
        "best_actual_within_model_top_candidates": best_candidate_actual,
        "captain_regret_within_candidate_pool": best_candidate_actual - float(model_captain["actual_points"]),
        "premium_model_top3_actual": premium_model_sum,
        "premium_baseline_top3_actual": premium_base_sum,
        "premium_oracle_top3_actual": premium_oracle_sum,
    }


def _v231_probability_brier(frame: pd.DataFrame, predicted: str, target: str) -> float:
    if predicted not in frame.columns or target not in frame.columns:
        return np.nan
    valid = frame[[predicted, target]].dropna()
    if valid.empty:
        return np.nan
    p = pd.to_numeric(valid[predicted], errors="coerce").clip(0, 1)
    y = pd.to_numeric(valid[target], errors="coerce").clip(0, 1)
    keep = p.notna() & y.notna()
    return float(np.square(p[keep] - y[keep]).mean()) if keep.any() else np.nan


historical_decision_fixture_frames = []
historical_decision_player_gws = []
missing_fold_cache = []
if HISTORICAL_DECISION_AUDIT_ENABLED:
    for season in WALK_FORWARD_HOLDOUT_SEASONS:
        fold = _v231_production_fold_frame(season)
        if fold.empty:
            missing_fold_cache.append(season)
            continue
        historical_decision_fixture_frames.append(fold)
        historical_decision_player_gws.append(_v231_player_gw(fold))

historical_decision_fixture_frame = (
    pd.concat(historical_decision_fixture_frames, ignore_index=True)
    if historical_decision_fixture_frames else pd.DataFrame()
)
historical_decision_player_gw = (
    pd.concat(historical_decision_player_gws, ignore_index=True)
    if historical_decision_player_gws else pd.DataFrame()
)

if missing_fold_cache:
    print(
        "Historical decision audit skipped cached folds:", missing_fold_cache,
        "— no refitting was triggered."
    )

# Per-GW ranking/selection diagnostics.
historical_rank_rows = []
if not historical_decision_player_gw.empty:
    for (season, gw), group in historical_decision_player_gw.groupby(
        ["holdout_season", "gameweek"], sort=True
    ):
        metrics = _v231_gw_rank_row(group, HISTORICAL_DECISION_TOP_K)
        if metrics:
            historical_rank_rows.append({
                "season": season,
                "gameweek": int(gw),
                **metrics,
            })
historical_rank_audit = pd.DataFrame(historical_rank_rows)

# Position and price diagnostics on the full player-GW OOS sample.
historical_position_rows = []
if not historical_decision_player_gw.empty:
    for position, group in historical_decision_player_gw.groupby("position"):
        model = _diag_error_metrics(group, "expected_points")
        base = _diag_error_metrics(group, "baseline_points")
        historical_position_rows.append({
            "position": position,
            "rows": model.get("rows"),
            "model_mae": model.get("mae"),
            "baseline_mae": base.get("mae"),
            "model_bias": model.get("bias_pred_minus_actual"),
            "model_spearman": model.get("spearman"),
            "baseline_spearman": base.get("spearman"),
        })
historical_position_audit = pd.DataFrame(historical_position_rows)

historical_price_audit = pd.DataFrame()
if not historical_decision_player_gw.empty:
    price_work = historical_decision_player_gw.copy()
    price_work["price_bucket"] = pd.cut(
        pd.to_numeric(price_work["price"], errors="coerce"),
        bins=[0, 5.0, 6.5, 8.0, 10.0, np.inf],
        labels=["<5.0", "5.0-6.4", "6.5-7.9", "8.0-9.9", "10.0+"],
        right=False,
    )
    rows = []
    for bucket, group in price_work.groupby("price_bucket", observed=False):
        if group.empty:
            continue
        m = _diag_error_metrics(group, "expected_points")
        b = _diag_error_metrics(group, "baseline_points")
        rows.append({
            "price_bucket": str(bucket), "rows": m["rows"],
            "model_mae": m["mae"], "baseline_mae": b["mae"],
            "model_spearman": m["spearman"], "baseline_spearman": b["spearman"],
            "model_bias": m["bias_pred_minus_actual"],
        })
    historical_price_audit = pd.DataFrame(rows)

# Calibration by predicted xP bucket.
historical_xp_calibration = pd.DataFrame()
if not historical_decision_player_gw.empty:
    cal = historical_decision_player_gw.copy()
    cal["xp_bucket"] = pd.cut(
        pd.to_numeric(cal["expected_points"], errors="coerce"),
        bins=[-np.inf, 1, 2, 3, 4, 5, 6, np.inf],
        labels=["<1", "1-2", "2-3", "3-4", "4-5", "5-6", "6+"],
        right=False,
    )
    historical_xp_calibration = (
        cal.groupby("xp_bucket", observed=False)
        .agg(
            rows=("actual_points", "size"),
            mean_xp=("expected_points", "mean"),
            mean_actual=("actual_points", "mean"),
        )
        .reset_index()
    )
    historical_xp_calibration["bias"] = (
        historical_xp_calibration["mean_xp"]
        - historical_xp_calibration["mean_actual"]
    )

# Minutes diagnostics and probability calibration.
historical_minutes_audit = pd.DataFrame()
historical_probability_audit = pd.DataFrame()
if not historical_decision_fixture_frame.empty:
    mins = historical_decision_fixture_frame.copy()
    mins["minutes_error"] = (
        pd.to_numeric(mins["expected_minutes"], errors="coerce")
        - pd.to_numeric(mins["target_minutes"], errors="coerce")
    )
    mins["expected_minutes_bucket"] = pd.cut(
        pd.to_numeric(mins["expected_minutes"], errors="coerce"),
        bins=[-0.1, 15, 30, 45, 60, 75, 90.1],
        labels=["0-14", "15-29", "30-44", "45-59", "60-74", "75-90"],
        right=False,
    )
    historical_minutes_audit = (
        mins.groupby(["model_position", "expected_minutes_bucket"], observed=False)
        .agg(
            rows=("target_minutes", "size"),
            expected_minutes=("expected_minutes", "mean"),
            actual_minutes=("target_minutes", "mean"),
            minutes_mae=("minutes_error", lambda s: s.abs().mean()),
            minutes_bias=("minutes_error", "mean"),
        )
        .reset_index()
    )
    probability_rows = []
    for season, group in mins.groupby("holdout_season"):
        probability_rows.append({
            "season": season,
            "appearance_brier": _v231_probability_brier(group, "appearance_probability", "target_appeared"),
            "start_brier": _v231_probability_brier(group, "start_probability", "target_started"),
            "sixty_plus_brier": _v231_probability_brier(group, "sixty_plus_probability", "target_60_plus"),
            "minutes_mae": float(pd.to_numeric(group["minutes_error"], errors="coerce").abs().mean()),
        })
    historical_probability_audit = pd.DataFrame(probability_rows)

# Attack-model diagnosis against realised FPL xG/xA rather than goals/assists.
historical_attack_audit = pd.DataFrame()
if not historical_decision_player_gw.empty:
    attack_rows = []
    for position, group in historical_decision_player_gw.groupby("position"):
        row = {"position": position, "rows": len(group)}
        if {"projected_goals", "actual_xg"}.issubset(group.columns):
            m = _diag_error_metrics(
                group.rename(columns={"actual_xg": "_actual_xg"}),
                "projected_goals", "_actual_xg"
            )
            row.update({"xg_mae": m["mae"], "xg_bias": m["bias_pred_minus_actual"], "xg_spearman": m["spearman"]})
        if {"projected_assists", "actual_xa"}.issubset(group.columns):
            m = _diag_error_metrics(
                group.rename(columns={"actual_xa": "_actual_xa"}),
                "projected_assists", "_actual_xa"
            )
            row.update({"xa_mae": m["mae"], "xa_bias": m["bias_pred_minus_actual"], "xa_spearman": m["spearman"]})
        attack_rows.append(row)
    historical_attack_audit = pd.DataFrame(attack_rows)

# Team-model diagnosis on unique fixture/team rows.
historical_team_audit = pd.DataFrame()
if not historical_decision_fixture_frame.empty:
    team_rows = []
    for season, frame in historical_decision_fixture_frame.groupby("holdout_season"):
        needed = ["fixture_id", "team_key", "expected_team_goals", "expected_team_goals_conceded"]
        if not set(needed).issubset(frame.columns):
            continue
        pred = frame[needed].drop_duplicates(["fixture_id", "team_key"])
        actual_cols = ["fixture_id", "team_key", "team_goals", "opponent_goals"]
        if "team_xg" in historical_team_fixtures.columns:
            actual_cols += ["team_xg", "opponent_xg"]
        actual = historical_team_fixtures.loc[
            historical_team_fixtures["season"].eq(season), actual_cols
        ].drop_duplicates(["fixture_id", "team_key"])
        merged = pred.merge(actual, on=["fixture_id", "team_key"], how="inner")
        if merged.empty:
            continue
        gf = _diag_error_metrics(
            merged.rename(columns={"team_goals": "actual_points"}),
            "expected_team_goals"
        )
        ga = _diag_error_metrics(
            merged.rename(columns={"opponent_goals": "actual_points"}),
            "expected_team_goals_conceded"
        )
        row = {
            "season": season, "team_fixture_rows": len(merged),
            "goals_for_mae": gf["mae"], "goals_for_bias": gf["bias_pred_minus_actual"],
            "goals_against_mae": ga["mae"], "goals_against_bias": ga["bias_pred_minus_actual"],
        }
        if "team_xg" in merged.columns and merged["team_xg"].notna().any():
            xgf = _diag_error_metrics(
                merged.rename(columns={"team_xg": "actual_points"}),
                "expected_team_goals"
            )
            xga = _diag_error_metrics(
                merged.rename(columns={"opponent_xg": "actual_points"}),
                "expected_team_goals_conceded"
            )
            row.update({"xg_for_mae": xgf["mae"], "xg_against_mae": xga["mae"]})
        team_rows.append(row)
    historical_team_audit = pd.DataFrame(team_rows)

print("=" * 80)
print("v2.3.1 HISTORICAL DECISION-QUALITY AUDIT")
print("=" * 80)
print("New model fits triggered by this section: 0")
print("Policy/transfer simulations triggered by this section: 0")
if not historical_rank_audit.empty:
    display(
        historical_rank_audit.groupby("season", as_index=False).agg(
            gameweeks=("gameweek", "nunique"),
            mean_model_spearman=("model_spearman", "mean"),
            mean_baseline_spearman=("baseline_spearman", "mean"),
            mean_model_ndcg10=("model_ndcg10", "mean"),
            mean_baseline_ndcg10=("baseline_ndcg10", "mean"),
            mean_model_top10_actual=("model_topk_actual", "mean"),
            mean_baseline_top10_actual=("baseline_topk_actual", "mean"),
            mean_oracle_top10_actual=("oracle_topk_actual", "mean"),
            mean_top10_lift_vs_baseline=("model_minus_baseline_topk", "mean"),
            top10_win_rate_vs_baseline=("model_minus_baseline_topk", lambda s: (s > 0).mean()),
            mean_captain_regret=("captain_regret_within_candidate_pool", "mean"),
            mean_premium_top3_model=("premium_model_top3_actual", "mean"),
            mean_premium_top3_baseline=("premium_baseline_top3_actual", "mean"),
        ).round(3)
    )
print("\nBy position")
display(historical_position_audit.round(3))
print("\nBy price")
display(historical_price_audit.round(3))
print("\nxP calibration")
display(historical_xp_calibration.round(3))
print("\nMinutes / availability")
display(historical_probability_audit.round(4))
display(historical_minutes_audit.round(2))
print("\nUnderlying attacking-rate model vs realised FPL xG/xA")
display(historical_attack_audit.round(4))
print("\nTeam model")
display(historical_team_audit.round(4))


In [ ]:
# =============================================================================
# v2.3.1 CURRENT-SEASON CONTEXT: PUBLIC FPL xP + EXECUTION PROVENANCE
# =============================================================================

def _v231_cached_public_csv(url: str, cache_name: str) -> pd.DataFrame:
    path = PUBLIC_EARLY_SEASON_BASELINE_CACHE / cache_name
    if path.exists():
        try:
            return pd.read_csv(path)
        except Exception:
            pass
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    path.write_bytes(response.content)
    return pd.read_csv(path)


def _v231_public_fpl_baseline(gameweek: int) -> pd.DataFrame:
    base = "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/2026-27/gws"
    xp = _v231_cached_public_csv(
        f"{base}/xP{int(gameweek)}.csv",
        f"xP{int(gameweek)}.csv",
    )
    actual = _v231_cached_public_csv(
        f"{base}/gw{int(gameweek)}.csv",
        f"gw{int(gameweek)}.csv",
    )
    xp = xp.rename(columns={"id": "player_id", "xP": "archived_fpl_xp"})
    actual = actual.rename(columns={"element": "player_id"})
    actual["player_id"] = pd.to_numeric(actual["player_id"], errors="coerce").astype("Int64")
    actual["total_points"] = pd.to_numeric(actual["total_points"], errors="coerce")
    group_cols = ["player_id"]
    agg = {"actual_points": ("total_points", "sum")}
    for column in ["name", "position", "team"]:
        if column in actual.columns:
            agg[column] = (column, "last")
    actual = actual.groupby(group_cols, dropna=False).agg(**agg).reset_index()
    xp["player_id"] = pd.to_numeric(xp["player_id"], errors="coerce").astype("Int64")
    xp["archived_fpl_xp"] = pd.to_numeric(xp["archived_fpl_xp"], errors="coerce")
    return xp.merge(actual, on="player_id", how="inner")


public_fpl_early_season_rows = []
if PUBLIC_EARLY_SEASON_BASELINE_ENABLED:
    max_completed = max(resolved_completed_gameweek_ids(gameweeks, fixtures) or [0])
    public_max = min(int(PUBLIC_EARLY_SEASON_BASELINE_MAX_GW), int(max_completed))
    for gw in range(1, public_max + 1):
        try:
            frame = _v231_public_fpl_baseline(gw)
            metrics = _diag_error_metrics(frame, "archived_fpl_xp")
            top = _diag_top_k_quality(frame, "archived_fpl_xp", ONLINE_DIAGNOSTIC_TOP_K)
            public_fpl_early_season_rows.append({
                "gameweek": gw,
                "rows": metrics["rows"],
                "fpl_xp_mae": metrics["mae"],
                "fpl_xp_bias": metrics["bias_pred_minus_actual"],
                "fpl_xp_spearman": metrics["spearman"],
                "fpl_top10_actual": top["actual_sum_selected"],
                "oracle_top10_actual": top["actual_sum_oracle"],
                "fpl_top10_oracle_efficiency": top["oracle_efficiency"],
            })
        except Exception as exc:
            warnings.warn(f"Could not load archived public FPL baseline for GW{gw}: {exc}")
public_fpl_early_season_audit = pd.DataFrame(public_fpl_early_season_rows)


def _v231_pairs(frame: pd.DataFrame) -> set[tuple[str, str]]:
    if frame is None or frame.empty or not {"outgoing", "incoming"}.issubset(frame.columns):
        return set()
    return {
        (str(r.outgoing).strip().casefold(), str(r.incoming).strip().casefold())
        for r in frame[["outgoing", "incoming"]].dropna().itertuples(index=False)
    }


def _v231_execution_provenance() -> pd.DataFrame:
    archive = _diag_load_archive()
    actual_transfers = _diag_manager_transfers()
    rows = []
    buckets = list(archive.get("snapshots", {}).values())
    buckets.sort(key=lambda bucket: int(bucket.get("gameweek", 999)))
    for bucket in buckets:
        gw = int(bucket.get("gameweek"))
        snapshot = _diag_latest_version(bucket)
        if not snapshot:
            continue
        model = pd.DataFrame(snapshot.get("transfer_evidence", []))
        actual = actual_transfers.loc[
            actual_transfers["gameweek"].eq(gw)
        ] if not actual_transfers.empty else pd.DataFrame()
        model_pairs = _v231_pairs(model)
        actual_pairs = _v231_pairs(actual)
        exact = model_pairs == actual_pairs
        if exact:
            provenance = "model_recommendation_followed"
        elif gw in set(ONLINE_GENUINE_MANAGER_DEVIATION_GWS):
            provenance = "genuine_manager_deviation"
        else:
            provenance = "assistant_advised_conservative_override"
        rows.append({
            "gameweek": gw,
            "model_transfer_count": len(model_pairs),
            "actual_transfer_count": len(actual_pairs),
            "exact_model_action_followed": exact,
            "execution_provenance": provenance,
            "model_action": "; ".join(f"{a} -> {b}" for a, b in sorted(model_pairs)) or "Roll",
            "actual_action": "; ".join(f"{a} -> {b}" for a, b in sorted(actual_pairs)) or "Roll",
        })
    return pd.DataFrame(rows)


online_execution_provenance = _v231_execution_provenance()

print("=" * 80)
print("CURRENT-SEASON CONTEXT")
print("=" * 80)
print("Archived public FPL xP baseline — useful context for GW1-GW3, not our model")
display(public_fpl_early_season_audit.round(3))
print("\nExecution provenance")
display(online_execution_provenance)


In [ ]:
# =============================================================================
# v2.3.1 DIAGNOSTIC VERDICT + EXPORTS
# =============================================================================

def _v231_safe_mean(frame: pd.DataFrame, column: str) -> float:
    if frame is None or frame.empty or column not in frame.columns:
        return np.nan
    return float(pd.to_numeric(frame[column], errors="coerce").mean())


def _v231_historical_position_spearman(position: str) -> float:
    if historical_rank_audit.empty or historical_decision_player_gw.empty:
        return np.nan
    values = []
    subset = historical_decision_player_gw.loc[
        historical_decision_player_gw["position"].eq(position)
        & pd.to_numeric(historical_decision_player_gw["expected_minutes"], errors="coerce").ge(
            float(HISTORICAL_DECISION_MIN_EXPECTED_MINUTES)
        )
    ]
    for (_, _), group in subset.groupby(["holdout_season", "gameweek"]):
        value = _diag_spearman(group, "expected_points", "actual_points")
        if np.isfinite(value):
            values.append(value)
    return float(np.mean(values)) if values else np.nan


hist_top10_lift = _v231_safe_mean(historical_rank_audit, "model_minus_baseline_topk")
hist_top10_win_rate = (
    float((historical_rank_audit["model_minus_baseline_topk"] > 0).mean())
    if not historical_rank_audit.empty else np.nan
)
hist_model_ndcg = _v231_safe_mean(historical_rank_audit, "model_ndcg10")
hist_baseline_ndcg = _v231_safe_mean(historical_rank_audit, "baseline_ndcg10")
hist_mid_spearman = _v231_historical_position_spearman("MID")
hist_fwd_spearman = _v231_historical_position_spearman("FWD")
hist_premium_lift = (
    _v231_safe_mean(historical_rank_audit, "premium_model_top3_actual")
    - _v231_safe_mean(historical_rank_audit, "premium_baseline_top3_actual")
)
hist_captain_regret = _v231_safe_mean(historical_rank_audit, "captain_regret_within_candidate_pool")
hist_minutes_mae = _v231_safe_mean(historical_probability_audit, "minutes_mae")

online_gws = (
    int(online_forecast_metrics["gameweek"].nunique())
    if isinstance(globals().get("online_forecast_metrics"), pd.DataFrame)
    and not online_forecast_metrics.empty else 0
)
online_model_top10 = _v231_safe_mean(online_forecast_metrics, "model_topk_actual_sum") if online_gws else np.nan
online_fpl_top10 = _v231_safe_mean(online_forecast_metrics, "fpl_topk_actual_sum") if online_gws else np.nan

priority_rows = []

def _add_priority(priority, area, finding, evidence, next_test):
    priority_rows.append({
        "priority": priority,
        "area": area,
        "finding": finding,
        "evidence": evidence,
        "next_test": next_test,
    })

if np.isfinite(hist_top10_lift):
    if hist_top10_lift <= 0 or hist_top10_win_rate < 0.50:
        _add_priority(
            "P0", "Ranking / selection",
            "Historical OOS top-player selection does not reliably beat the simple history baseline.",
            f"Mean top-10 lift {hist_top10_lift:+.2f}; GW win rate {hist_top10_win_rate:.1%}; model NDCG {hist_model_ndcg:.3f} vs baseline {hist_baseline_ndcg:.3f}.",
            "Redesign ranking/event model before any further transfer-policy optimisation.",
        )
    else:
        _add_priority(
            "Monitor", "Ranking / selection",
            "Historical OOS ranking adds value over the simple history baseline.",
            f"Mean top-10 lift {hist_top10_lift:+.2f}; GW win rate {hist_top10_win_rate:.1%}; model NDCG {hist_model_ndcg:.3f} vs baseline {hist_baseline_ndcg:.3f}.",
            "Check whether 2026/27 underperformance is an early-season regime shift rather than a structural ranking failure.",
        )

if np.isfinite(hist_mid_spearman):
    _add_priority(
        "P1" if hist_mid_spearman < 0.35 else "Monitor",
        "Midfield ranking",
        "MID ranking is a decision-critical diagnostic because premium midfielders drive transfers and captaincy.",
        f"Mean historical per-GW MID Spearman among decision candidates: {hist_mid_spearman:.3f}; FWD {hist_fwd_spearman:.3f}.",
        "If weak, inspect predicted xG/xA and minutes separately for MID/FWD before changing total xP.",
    )

if np.isfinite(hist_premium_lift):
    _add_priority(
        "P1" if hist_premium_lift <= 0 else "Monitor",
        "Premium attackers",
        "Premium MID/FWD ranking is the highest-leverage subset for captaincy and transfers.",
        f"Mean realised top-3 premium lift vs simple baseline: {hist_premium_lift:+.2f} points/GW.",
        "Prefer component-level xG/xA/minutes fixes over blanket xP calibration.",
    )

if np.isfinite(hist_minutes_mae):
    _add_priority(
        "P1" if hist_minutes_mae > 14 else "P2",
        "Minutes",
        "Minutes error propagates into every xP component.",
        f"Historical OOS fixture-level minutes MAE: {hist_minutes_mae:.2f}.",
        "Inspect 60+ starters and unexpected non-appearances/benchings; add current workload/team-news features only if the error is concentrated there.",
    )

if online_gws:
    _add_priority(
        "Evidence", "2026/27 online sample",
        "The live archive is still too small to justify parameter tuning by itself.",
        f"True preserved model GWs: {online_gws}; mean model top-10 realised {online_model_top10:.1f} vs archived FPL baseline {online_fpl_top10:.1f}.",
        "Use online data as confirmation/shift detection, not as a tuning set.",
    )

v231_diagnostic_priorities = pd.DataFrame(priority_rows)

# Export only diagnostic tables. No model or policy state is mutated.
exports = {
    "historical_rank_audit_v231.csv": historical_rank_audit,
    "historical_position_audit_v231.csv": historical_position_audit,
    "historical_price_audit_v231.csv": historical_price_audit,
    "historical_xp_calibration_v231.csv": historical_xp_calibration,
    "historical_minutes_audit_v231.csv": historical_minutes_audit,
    "historical_probability_audit_v231.csv": historical_probability_audit,
    "historical_attack_audit_v231.csv": historical_attack_audit,
    "historical_team_audit_v231.csv": historical_team_audit,
    "public_fpl_early_season_audit_v231.csv": public_fpl_early_season_audit,
    "online_execution_provenance_v231.csv": online_execution_provenance,
    "diagnostic_priorities_v231.csv": v231_diagnostic_priorities,
}
for filename, frame in exports.items():
    if isinstance(frame, pd.DataFrame) and not frame.empty:
        frame.to_csv(OUTPUT_DIRECTORY / filename, index=False)

print("=" * 80)
print("v2.3.1 DIAGNOSTIC PRIORITIES")
print("=" * 80)
display(v231_diagnostic_priorities)
print(
    "No forecast parameters, transfer-policy parameters, or model artifacts "
    "were changed by Sections 13–14."
)


### Interpreting v2.3.1

The historical tables are the primary evidence because they contain roughly three full untouched seasons. The 2026/27 archived live sample is a **shift detector**, not a tuning dataset.

A weak historical top-10 lift/NDCG result means the aggregate MAE was masking a ranking problem and the next version should redesign the forecast model. A strong historical ranking result combined with weak GW4–GW5 live ranking instead points toward an early-season/current-data problem: minutes, promoted/transferred players, team-strength priors, or current availability.

The GW1–GW3 public rows are an external comparator only. They answer whether the opening weeks were intrinsically difficult for FPL's own archived xP; they are not retroactively labelled as forecasts from this notebook.

Finally, execution provenance is deliberately asymmetric: GW2 remains the genuine manager deviation you identified. Other archived action mismatches are labelled as assistant-advised conservative overrides, so they are not counted as failures to follow the model.


# 15. v2.4 targeted forecast validation

This section reports the two chronological meta-validation folds used to gate
the v2.4 corrections. It does **not** run transfers, policies or season
simulations. If a component is rejected, the live model falls back to the
v2.3.1 production semantics for that component. Uncertainty is always reported
as confidence context and never deducted from xP.

In [ ]:

print("=" * 80)
print("v2.4 TARGETED FORECAST VALIDATION")
print("=" * 80)
print("Transfer/policy simulations run by this section: 0")
print("2026/27 rows used for tuning: 0")
print("Production decision:")
display(pd.DataFrame([
    {"component": "upper_tail", **_v24_upper_decision},
    {"component": "midfield_reranker", **_v24_mid_decision},
    {"component": "combined", "approved": bool(v24_targeted_forecast_artifact["production_decision"]["approved"]),
     "combined_safe": bool(v24_targeted_forecast_artifact["production_decision"]["combined_safe"]),
     "overall_ndcg_change": v24_targeted_forecast_artifact["production_decision"]["combined_ndcg_change"],
     "top10_change": v24_targeted_forecast_artifact["production_decision"]["combined_top10_change"]},
]).round(4))

print("\nUpper-tail chronological folds")
display(v24_upper_tail_scorecard.round(4) if not v24_upper_tail_scorecard.empty else pd.DataFrame({"status":["not evaluated"]}))
print("\nMID reranker chronological folds")
display(v24_midfield_scorecard.round(4) if not v24_midfield_scorecard.empty else pd.DataFrame({"status":["not evaluated"]}))
print("\nCombined safety folds")
display(v24_combined_scorecard.round(4) if not v24_combined_scorecard.empty else pd.DataFrame({"status":["not evaluated"]}))

if _v24_uncertainty:
    _unc_rows = []
    for key, spec in _v24_uncertainty.get("groups", {}).items():
        position, bucket = key.split("|", 1)
        _unc_rows.append({"position":position, "minutes_bucket":bucket, **spec})
    v24_uncertainty_table = pd.DataFrame(_unc_rows)
    print("\nHistorical uncertainty by position/minutes bucket")
    display(v24_uncertainty_table.sort_values(["position","minutes_bucket"]).round(3))
else:
    v24_uncertainty_table = pd.DataFrame()

if isinstance(live_expected_points, pd.DataFrame) and not live_expected_points.empty:
    _gw = int(pd.to_numeric(live_expected_points["gameweek"], errors="coerce").min())
    _live = live_expected_points.loc[live_expected_points["gameweek"].eq(_gw)].copy()
    _cols = [c for c in [
        "web_name", "team_short_name", "model_position", "price",
        "pre_v24_expected_points", "v24_upper_tail_adjustment",
        "v24_midfield_rank_adjustment", "v24_total_adjustment",
        "expected_points", "expected_minutes", "minutes_uncertainty_p80",
        "forecast_uncertainty_p80",
    ] if c in _live.columns]
    print(f"\nLargest live v2.4 changes for GW{_gw}")
    display(
        _live.assign(_abs_adj=_live["v24_total_adjustment"].abs())
        .sort_values("_abs_adj", ascending=False)
        .head(20)[_cols]
        .round(3)
    )

# Export cheap diagnostic scorecards only.
for _name, _frame in {
    "v24_upper_tail_scorecard.csv": v24_upper_tail_scorecard,
    "v24_midfield_scorecard.csv": v24_midfield_scorecard,
    "v24_combined_scorecard.csv": v24_combined_scorecard,
    "v24_uncertainty_table.csv": v24_uncertainty_table,
}.items():
    if isinstance(_frame, pd.DataFrame) and not _frame.empty:
        _frame.to_csv(OUTPUT_DIRECTORY / _name, index=False)


## 16. v2.5 probabilistic minutes / role validation

This section is deliberately **cache-first**. It does not run transfer-policy
backtests or season simulations.

The v2.5 layer reuses the existing, historically validated hierarchy for
`P(appear)`, `P(start)` and `P(60+)`. It fits only one additional conditional
classifier, `P(75+ | starter and 60+)`, and converts the minutes forecast into
five mutually-exclusive states:

- no appearance;
- substitute appearance;
- start under 60;
- start 60–74;
- start 75+.

FPL scoring is then evaluated conditionally in each state and probability
weighted. This matters because appearance points, clean sheets, saves, goals
conceded and defensive-contribution thresholds are nonlinear in minutes.

The layer is activated only if chronological OOS tests improve role/minutes
quality **without degrading the FPL ranking objective**. All fold predictions,
the live model and the approval decision are persisted under `data/` and reused
on subsequent runs.

In [ ]:

print("=" * 80)
print("v2.5 PROBABILISTIC MINUTES / ROLE VALIDATION")
print("=" * 80)
print("Transfer/policy simulations run by this section: 0")
print("Base walk-forward forecast folds: disk-cached and reusable")
print("Validation source:", globals().get("_v25_cache_source", "unknown"))

_v25_decision_display = (
    v25_probabilistic_minutes_artifact.get("production_decision", {})
    if isinstance(globals().get("v25_probabilistic_minutes_artifact"), dict)
    else {}
)
display(pd.DataFrame([_v25_decision_display]).round(5))

print("\nChronological fold scorecard")
display(
    v25_probabilistic_minutes_scorecard.round(5)
    if isinstance(globals().get("v25_probabilistic_minutes_scorecard"), pd.DataFrame)
    and not v25_probabilistic_minutes_scorecard.empty
    else pd.DataFrame({"status": ["no scorecard available"]})
)

if isinstance(globals().get("v25_probabilistic_minutes_oos"), pd.DataFrame) and not v25_probabilistic_minutes_oos.empty:
    _v25_oos = v25_probabilistic_minutes_oos.copy()
    _v25_oos["base_minutes_abs_error"] = (
        pd.to_numeric(_v25_oos["target_minutes"], errors="coerce")
        - pd.to_numeric(_v25_oos["base_expected_minutes"], errors="coerce")
    ).abs()
    _v25_oos["candidate_minutes_abs_error"] = (
        pd.to_numeric(_v25_oos["target_minutes"], errors="coerce")
        - pd.to_numeric(_v25_oos["candidate_expected_minutes"], errors="coerce")
    ).abs()

    print("\nMinutes error by realised/forecast-relevant position")
    display(
        _v25_oos.groupby("position", as_index=False).agg(
            rows=("target_minutes", "size"),
            base_minutes_mae=("base_minutes_abs_error", "mean"),
            candidate_minutes_mae=("candidate_minutes_abs_error", "mean"),
            mean_role_entropy=("v25_role_entropy", "mean"),
        ).round(3)
    )

    print("\nMost uncertain OOS role states")
    _uncertain_cols = [
        "holdout_season", "gameweek", "player_id", "position", "price",
        "base_expected_minutes", "candidate_expected_minutes", "target_minutes",
        "v25_role_entropy", "v25_most_likely_role_state",
        "v25_most_likely_role_probability",
    ]
    display(
        _v25_oos.sort_values("v25_role_entropy", ascending=False)
        .head(20)[[c for c in _uncertain_cols if c in _v25_oos.columns]]
        .round(3)
    )

if isinstance(globals().get("live_expected_points"), pd.DataFrame) and not live_expected_points.empty:
    _v25_live_gw = int(pd.to_numeric(
        live_expected_points["gameweek"], errors="coerce"
    ).min())
    _v25_live = live_expected_points.loc[
        live_expected_points["gameweek"].eq(_v25_live_gw)
    ].copy()
    _v25_live_cols = [
        "web_name", "team_short_name", "model_position", "price",
        "pre_v25_expected_minutes", "expected_minutes",
        "v25_minutes_adjustment",
        "v25_p_no_appearance", "v25_p_substitute",
        "v25_p_start_lt60", "v25_p_start_60_74", "v25_p_start_75_plus",
        "v25_role_entropy", "v25_most_likely_role_state",
        "pre_v25_expected_points", "expected_points",
        "v25_probabilistic_xp_adjustment",
        "v25_probabilistic_minutes_applied",
    ]
    print(f"\nHighest-role-uncertainty live players for GW{_v25_live_gw}")
    display(
        _v25_live.sort_values("v25_role_entropy", ascending=False)
        .head(25)[[c for c in _v25_live_cols if c in _v25_live.columns]]
        .round(3)
    )

print("\nReusable caches")
display(pd.DataFrame([
    {
        "artifact": "base walk-forward folds",
        "path": str(WALK_FORWARD_FOLD_DISK_CACHE_DIRECTORY),
        "exists": WALK_FORWARD_FOLD_DISK_CACHE_DIRECTORY.exists(),
    },
    {
        "artifact": "v2.5 OOS row audit",
        "path": str(V25_PROBABILISTIC_MINUTES_OOS_CACHE_PATH),
        "exists": V25_PROBABILISTIC_MINUTES_OOS_CACHE_PATH.exists(),
    },
    {
        "artifact": "v2.5 validation decision",
        "path": str(V25_PROBABILISTIC_MINUTES_ARTIFACT_PATH),
        "exists": V25_PROBABILISTIC_MINUTES_ARTIFACT_PATH.exists(),
    },
    {
        "artifact": "v2.5 full-history live model",
        "path": str(V25_PROBABILISTIC_MINUTES_MODEL_CACHE_PATH),
        "exists": V25_PROBABILISTIC_MINUTES_MODEL_CACHE_PATH.exists(),
    },
]))
